In [1]:
# Step 1: Project initialization and raw-source integrity.

from pathlib import Path
import os
import sys
import json
import random
import hashlib
import platform
import datetime as dt
import logging
import numpy as np
import pandas as pd
RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
RAW_DATA_FILE = Path('E:\\BDHS_ANC_FINAL\\00_raw_data\\BDHS 2022 IR.DTA')
PROJECT_ROOT = Path('E:\\BDHS_ANC_Q1_FINAL_REBUILD_2026')
DIRS = {'raw_reference': PROJECT_ROOT / '00_raw_reference', 'processed': PROJECT_ROOT / '01_processed_data', 'qc': PROJECT_ROOT / '02_quality_control', 'protocol': PROJECT_ROOT / '03_protocol', 'splits': PROJECT_ROOT / '04_cv_splits', 'models': PROJECT_ROOT / '05_models', 'predictions': PROJECT_ROOT / '06_predictions', 'calibration': PROJECT_ROOT / '07_calibration', 'bootstrap': PROJECT_ROOT / '08_bootstrap', 'geographic': PROJECT_ROOT / '09_geographic_validation', 'xai': PROJECT_ROOT / '10_explainability', 'sensitivity': PROJECT_ROOT / '11_sensitivity', 'subgroup': PROJECT_ROOT / '12_subgroup_analysis', 'tables': PROJECT_ROOT / '13_tables', 'figures': PROJECT_ROOT / '14_figures', 'logs': PROJECT_ROOT / '15_logs', 'reporting': PROJECT_ROOT / '16_reporting'}
if not Path('E:\\\\').exists():
    raise FileNotFoundError('E: drive is not accessible. Confirm that the external/local drive is mounted.')
if not RAW_DATA_FILE.exists():
    raise FileNotFoundError(f'Raw BDHS file was not found:\n{RAW_DATA_FILE}')
if not RAW_DATA_FILE.is_file():
    raise RuntimeError(f'The raw-data path is not a file:\n{RAW_DATA_FILE}')
if RAW_DATA_FILE.suffix.lower() != '.dta':
    raise RuntimeError('The specified BDHS source file is not a Stata .DTA file.')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)
LOG_FILE = DIRS['logs'] / 'BDHS_ANC_FINAL_PIPELINE.log'
logger = logging.getLogger('BDHS_ANC_FINAL')
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()
file_handler = logging.FileHandler(LOG_FILE, mode='a', encoding='utf-8')
console_handler = logging.StreamHandler()
log_format = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')
file_handler.setFormatter(log_format)
console_handler.setFormatter(log_format)
logger.addHandler(file_handler)
logger.addHandler(console_handler)
logger.info('STEP 1 started: project initialization.')

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()
    with open(file_path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)
    return sha256.hexdigest()
RAW_DATA_SHA256 = calculate_sha256(RAW_DATA_FILE)
raw_stat = RAW_DATA_FILE.stat()
RAW_FILE_SIZE_BYTES = int(raw_stat.st_size)
RAW_FILE_SIZE_MB = RAW_FILE_SIZE_BYTES / 1024 ** 2
RAW_FILE_MODIFIED = dt.datetime.fromtimestamp(raw_stat.st_mtime).isoformat(timespec='seconds')
environment_info = {'project_name': 'BDHS ANC Q1 Final Rebuild 2026', 'project_root': str(PROJECT_ROOT), 'raw_data_file': str(RAW_DATA_FILE), 'raw_data_sha256': RAW_DATA_SHA256, 'raw_file_size_bytes': RAW_FILE_SIZE_BYTES, 'raw_file_size_mb': RAW_FILE_SIZE_MB, 'raw_file_last_modified': RAW_FILE_MODIFIED, 'random_seed': RANDOM_SEED, 'python_version': sys.version, 'python_executable': sys.executable, 'operating_system': platform.platform(), 'numpy_version': np.__version__, 'pandas_version': pd.__version__, 'step_1_execution_time': dt.datetime.now().isoformat(timespec='seconds')}
MANIFEST_FILE = DIRS['protocol'] / 'STEP01_project_manifest.json'
with open(MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(environment_info, f, indent=4, ensure_ascii=False)
directory_manifest = pd.DataFrame([{'Directory_Key': key, 'Directory_Path': str(path), 'Exists': path.exists()} for key, path in DIRS.items()])
DIRECTORY_MANIFEST_FILE = DIRS['protocol'] / 'STEP01_directory_manifest.csv'
directory_manifest.to_csv(DIRECTORY_MANIFEST_FILE, index=False)
if not all((path.exists() for path in DIRS.values())):
    raise RuntimeError('One or more project directories were not created.')
if len(RAW_DATA_SHA256) != 64:
    raise RuntimeError('Unexpected SHA-256 hash length.')
if RAW_FILE_SIZE_BYTES <= 0:
    raise RuntimeError('Raw BDHS file appears to be empty.')
if not MANIFEST_FILE.exists():
    raise RuntimeError('Project manifest was not saved.')
logger.info('STEP 1 completed successfully.')
print('\n' + '=' * 78)
print('STEP 1 — FINAL PROJECT INITIALIZATION + RAW SOURCE INTEGRITY')
print('=' * 78)
print(f'Project root      : {PROJECT_ROOT}')
print(f'Raw source        : {RAW_DATA_FILE}')
print(f'Raw file exists   : {RAW_DATA_FILE.exists()}')
print(f'Raw file size     : {RAW_FILE_SIZE_MB:,.2f} MB')
print(f'Raw SHA-256       : {RAW_DATA_SHA256}')
print(f'Random seed       : {RANDOM_SEED}')
print(f'Python            : {sys.version.split()[0]}')
print(f'NumPy             : {np.__version__}')
print(f'Pandas            : {pd.__version__}')
print(f'Directories made  : {sum((path.exists() for path in DIRS.values()))}/{len(DIRS)}')
print(f'Manifest          : {MANIFEST_FILE}')
print(f'Log file          : {LOG_FILE}')
print('-' * 78)
print('IMPORTANT:')
print('- No participant was selected or excluded in Step 1.')
print('- No DHS variable was recoded in Step 1.')
print('- No previous processed/model output was reused.')
print('- The original BDHS .DTA file is the only source dataset.')
print('=' * 78)
print('STATUS: STEP 1 PASS')
print('=' * 78)


2026-09-16 21:04:05,048 | INFO | STEP 1 started: project initialization.
2026-09-16 21:04:05,637 | INFO | STEP 1 completed successfully.



STEP 1 — FINAL PROJECT INITIALIZATION + RAW SOURCE INTEGRITY
Project root      : E:\BDHS_ANC_Q1_FINAL_REBUILD_2026
Raw source        : E:\BDHS_ANC_FINAL\00_raw_data\BDHS 2022 IR.DTA
Raw file exists   : True
Raw file size     : 172.72 MB
Raw SHA-256       : 81676772f734e43b33e09085a2f3232c35f5c04dfdf898cdb551957a3278b56f
Random seed       : 2026
Python            : 3.13.9
NumPy             : 2.3.5
Pandas            : 3.0.5
Directories made  : 17/17
Manifest          : E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\03_protocol\STEP01_project_manifest.json
Log file          : E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\15_logs\BDHS_ANC_FINAL_PIPELINE.log
------------------------------------------------------------------------------
IMPORTANT:
- No participant was selected or excluded in Step 1.
- No DHS variable was recoded in Step 1.
- No previous processed/model output was reused.
- The original BDHS .DTA file is the only source dataset.
STATUS: STEP 1 PASS


In [2]:
# Step 2: Raw BDHS import and metadata/schema audit.

import re
try:
    import pyreadstat
except ImportError as exc:
    raise ImportError('pyreadstat is required for the final pipeline.\nInstall it in the current environment with:\npip install pyreadstat') from exc
STEP1_MANIFEST = DIRS['protocol'] / 'STEP01_project_manifest.json'
if not STEP1_MANIFEST.exists():
    raise FileNotFoundError(f'Step-1 manifest not found:\n{STEP1_MANIFEST}')
with open(STEP1_MANIFEST, 'r', encoding='utf-8') as f:
    step1_manifest = json.load(f)
expected_raw_hash = step1_manifest['raw_data_sha256']
current_raw_hash = calculate_sha256(RAW_DATA_FILE)
if current_raw_hash != expected_raw_hash:
    raise RuntimeError('RAW DATA HASH MISMATCH.\nThe source file differs from the file locked in Step 1.\nSTOP before continuing.')
logger.info('STEP 2 started: importing raw BDHS .DTA file.')
raw_original, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
EXPECTED_RAW_ROWS = 30078
EXPECTED_RAW_COLUMNS = 5425
observed_raw_rows = len(raw_original)
observed_raw_columns = raw_original.shape[1]
if observed_raw_rows != EXPECTED_RAW_ROWS:
    raise RuntimeError(f'Unexpected number of source rows.\nExpected : {EXPECTED_RAW_ROWS:,}\nObserved : {observed_raw_rows:,}')
if observed_raw_columns != EXPECTED_RAW_COLUMNS:
    raise RuntimeError(f'Unexpected number of source columns.\nExpected : {EXPECTED_RAW_COLUMNS:,}\nObserved : {observed_raw_columns:,}')
original_columns = list(raw_original.columns)
normalized_columns = [str(col).strip().upper() for col in original_columns]
if len(set(normalized_columns)) != len(normalized_columns):
    duplicated_normalized = pd.Series(normalized_columns).loc[pd.Series(normalized_columns).duplicated(keep=False)].tolist()
    raise RuntimeError(f'Variable-name collision detected after uppercase normalization:\n{duplicated_normalized[:20]}')
column_rename_map = dict(zip(original_columns, normalized_columns))
raw = raw_original.rename(columns=column_rename_map).reset_index(drop=True).copy()
raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if not raw['_ROW_ID'].is_unique:
    raise RuntimeError('_ROW_ID is not unique.')
if raw['_ROW_ID'].isna().any():
    raise RuntimeError('_ROW_ID contains missing values.')
if 'CASEID' not in raw.columns:
    raise RuntimeError('CASEID was not found in the BDHS IR file.')
n_caseid_missing = int(raw['CASEID'].isna().sum())
n_caseid_duplicate = int(raw['CASEID'].duplicated().sum())
if n_caseid_missing > 0:
    raise RuntimeError(f'CASEID contains {n_caseid_missing} missing values.')
if n_caseid_duplicate > 0:
    raise RuntimeError(f'CASEID contains {n_caseid_duplicate} duplicates.')
raw_variable_labels = getattr(stata_meta, 'column_names_to_labels', {}) or {}
variable_label_map = {}
for original_name in original_columns:
    normalized_name = str(original_name).strip().upper()
    variable_label_map[normalized_name] = raw_variable_labels.get(original_name, None)
raw_variable_value_labels = getattr(stata_meta, 'variable_value_labels', {}) or {}
VARIABLE_VALUE_LABELS = {}
for original_name, mapping in raw_variable_value_labels.items():
    normalized_name = str(original_name).strip().upper()
    VARIABLE_VALUE_LABELS[normalized_name] = dict(mapping)
variable_dictionary_rows = []
for original_name in original_columns:
    normalized_name = str(original_name).strip().upper()
    variable_dictionary_rows.append({'Original_Name': original_name, 'Normalized_Name': normalized_name, 'Variable_Label': variable_label_map.get(normalized_name), 'Pandas_Dtype': str(raw[normalized_name].dtype), 'Has_Value_Labels': normalized_name in VARIABLE_VALUE_LABELS})
variable_dictionary = pd.DataFrame(variable_dictionary_rows)
value_label_rows = []
for variable, mapping in VARIABLE_VALUE_LABELS.items():
    for raw_value, value_label in mapping.items():
        value_label_rows.append({'Variable': variable, 'Variable_Label': variable_label_map.get(variable), 'Raw_Value': raw_value, 'Value_Label': value_label})
value_label_crosswalk = pd.DataFrame(value_label_rows)
if not value_label_crosswalk.empty:
    value_label_crosswalk = value_label_crosswalk.sort_values(by=['Variable', 'Raw_Value'], kind='stable').reset_index(drop=True)

def numbered_family(columns, prefix):
    pattern = re.compile(f'^{re.escape(prefix)}_(\\d+)$')
    matched = []
    for col in columns:
        match = pattern.fullmatch(col)
        if match:
            matched.append((int(match.group(1)), col))
    return [col for _, col in sorted(matched)]
M80_COLS = numbered_family(raw.columns, 'M80')
M14_COLS = numbered_family(raw.columns, 'M14')
M10_COLS = numbered_family(raw.columns, 'M10')
BORD_COLS = numbered_family(raw.columns, 'BORD')
B3_COLS = numbered_family(raw.columns, 'B3')
BIDX_COLS = numbered_family(raw.columns, 'BIDX')
CRITICAL_STATIC_VARIABLES = ['CASEID', 'V005', 'V021', 'V023', 'V011', 'V012', 'V201', 'V024', 'V025', 'V106', 'V190', 'V157', 'V158', 'V159', 'V714', 'V130', 'V501', 'V701', 'V743A', 'V136', 'V151', 'V467D']
critical_static_audit = pd.DataFrame({'Variable': CRITICAL_STATIC_VARIABLES, 'Present': [variable in raw.columns for variable in CRITICAL_STATIC_VARIABLES], 'Variable_Label': [variable_label_map.get(variable) for variable in CRITICAL_STATIC_VARIABLES]})
missing_critical = critical_static_audit.loc[~critical_static_audit['Present'], 'Variable'].tolist()
if missing_critical:
    raise RuntimeError(f'Critical BDHS source variables are missing:\n{missing_critical}')
required_families = {'M80': M80_COLS, 'M14': M14_COLS, 'M10': M10_COLS, 'BORD': BORD_COLS, 'B3': B3_COLS, 'BIDX': BIDX_COLS}
for family_name, family_columns in required_families.items():
    if len(family_columns) == 0:
        raise RuntimeError(f'No {family_name}_* variables were found.')
family_audit = pd.DataFrame([{'Variable_Family': family, 'N_Variables': len(columns), 'First_Variable': columns[0] if columns else None, 'Last_Variable': columns[-1] if columns else None} for family, columns in required_families.items()])
VARIABLE_DICTIONARY_FILE = DIRS['raw_reference'] / 'STEP02_variable_dictionary.csv'
VALUE_LABEL_FILE = DIRS['raw_reference'] / 'STEP02_value_label_crosswalk.csv'
CRITICAL_AUDIT_FILE = DIRS['qc'] / 'STEP02_critical_variable_audit.csv'
FAMILY_AUDIT_FILE = DIRS['qc'] / 'STEP02_variable_family_audit.csv'
variable_dictionary.to_csv(VARIABLE_DICTIONARY_FILE, index=False)
value_label_crosswalk.to_csv(VALUE_LABEL_FILE, index=False)
critical_static_audit.to_csv(CRITICAL_AUDIT_FILE, index=False)
family_audit.to_csv(FAMILY_AUDIT_FILE, index=False)
step2_manifest = {'step': 'STEP 2', 'description': 'Raw BDHS import and source metadata audit', 'raw_sha256': current_raw_hash, 'source_rows': int(observed_raw_rows), 'source_columns': int(observed_raw_columns), 'working_columns_with_row_id': int(raw.shape[1]), 'caseid_missing': n_caseid_missing, 'caseid_duplicates': n_caseid_duplicate, 'm80_variables': M80_COLS, 'm14_variables': M14_COLS, 'm10_variables': M10_COLS, 'bord_variables': BORD_COLS, 'b3_variables': B3_COLS, 'bidx_variables': BIDX_COLS, 'n_variables_with_value_labels': int(len(VARIABLE_VALUE_LABELS)), 'n_value_label_rows': int(len(value_label_crosswalk)), 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP2_MANIFEST_FILE = DIRS['protocol'] / 'STEP02_raw_import_manifest.json'
with open(STEP2_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step2_manifest, f, indent=4, ensure_ascii=False, default=str)
assert len(raw) == 30078
assert raw.shape[1] == 5426
assert raw['_ROW_ID'].is_unique
assert raw['CASEID'].is_unique
assert current_raw_hash == expected_raw_hash
assert critical_static_audit['Present'].all()
assert len(M80_COLS) > 0
assert len(M14_COLS) > 0
assert len(M10_COLS) > 0
assert len(BORD_COLS) > 0
assert len(B3_COLS) > 0
assert len(BIDX_COLS) > 0
logger.info(f'STEP 2 completed successfully: {len(raw):,} rows x {raw.shape[1]:,} working columns.')
print('\n' + '=' * 78)
print('STEP 2 — RAW BDHS IMPORT + METADATA / SCHEMA AUDIT')
print('=' * 78)
print(f'Raw SHA-256 verified       : {current_raw_hash == expected_raw_hash}')
print(f'Source rows                : {observed_raw_rows:,}')
print(f'Source columns             : {observed_raw_columns:,}')
print(f'Working columns (+ROW_ID)  : {raw.shape[1]:,}')
print(f'Missing CASEID             : {n_caseid_missing:,}')
print(f'Duplicate CASEID           : {n_caseid_duplicate:,}')
print(f'Variables with value labels: {len(VARIABLE_VALUE_LABELS):,}')
print(f'Value-label crosswalk rows : {len(value_label_crosswalk):,}')
print('\nMATERNITY / BIRTH-HISTORY VARIABLE FAMILIES:')
display(family_audit)
print('\nCRITICAL STATIC SOURCE VARIABLES:')
display(critical_static_audit)
print('\nFIRST 20 NORMALIZED VARIABLES:')
print(raw.columns[:20].tolist())
print('\nSOURCE METADATA SAVED:')
print(f'- {VARIABLE_DICTIONARY_FILE}')
print(f'- {VALUE_LABEL_FILE}')
print(f'- {CRITICAL_AUDIT_FILE}')
print(f'- {FAMILY_AUDIT_FILE}')
print(f'- {STEP2_MANIFEST_FILE}')
print('\nIMPORTANT:')
print('- Numeric DHS source codes remain unmodified.')
print('- Stata variable/value labels were preserved separately.')
print('- No cohort restriction has occurred.')
print('- No outcome or predictor has been constructed.')
print('- _ROW_ID is only a stable source-row identifier.')
print('=' * 78)
print('STATUS: STEP 2 PASS')
print('=' * 78)


2026-09-16 21:04:07,985 | INFO | STEP 2 started: importing raw BDHS .DTA file.
2026-09-16 21:05:30,396 | INFO | STEP 2 completed successfully: 30,078 rows x 5,426 working columns.



STEP 2 — RAW BDHS IMPORT + METADATA / SCHEMA AUDIT
Raw SHA-256 verified       : True
Source rows                : 30,078
Source columns             : 5,425
Working columns (+ROW_ID)  : 5,426
Missing CASEID             : 0
Duplicate CASEID           : 0
Variables with value labels: 4,690
Value-label crosswalk rows : 15,272

MATERNITY / BIRTH-HISTORY VARIABLE FAMILIES:


,Variable_Family,N_Variables,First_Variable,Last_Variable
0,M80,6,M80_1,M80_6
1,M14,6,M14_1,M14_6
2,M10,6,M10_1,M10_6
3,BORD,20,BORD_01,BORD_20
4,B3,20,B3_01,B3_20
5,BIDX,20,BIDX_01,BIDX_20



CRITICAL STATIC SOURCE VARIABLES:


,Variable,Present,Variable_Label
0,CASEID,True,case identification
1,V005,True,women's individual sample weight (6 decimals)
2,V021,True,primary sampling unit
3,V023,True,stratification used in sample design
4,V011,True,date of birth (cmc)
5,V012,True,respondent's current age
6,V201,True,total children ever born
7,V024,True,division
8,V025,True,type of place of residence
9,V106,True,highest educational level



FIRST 20 NORMALIZED VARIABLES:
['_ROW_ID', 'CASEID', 'V000', 'V001', 'V002', 'V003', 'V004', 'V005', 'V006', 'V007', 'V008', 'V008A', 'V009', 'V010', 'V011', 'V012', 'V013', 'V014', 'V015', 'V016']

SOURCE METADATA SAVED:
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\00_raw_reference\STEP02_variable_dictionary.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\00_raw_reference\STEP02_value_label_crosswalk.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP02_critical_variable_audit.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP02_variable_family_audit.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\03_protocol\STEP02_raw_import_manifest.json

IMPORTANT:
- Numeric DHS source codes remain unmodified.
- Stata variable/value labels were preserved separately.
- No cohort restriction has occurred.
- No outcome or predictor has been constructed.
- _ROW_ID is only a stable source-row identifier.
STATUS: STEP 2 PASS


In [3]:
# Step 3: Survey-design construction and quality control.

if 'raw' not in globals():
    logger.info('Raw dataframe not found in memory. Reloading Step-2 source.')
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if len(raw) != 30078:
    raise RuntimeError(f'Unexpected raw row count: {len(raw):,}')
if '_ROW_ID' not in raw.columns:
    raise RuntimeError('_ROW_ID missing from raw dataframe.')
if not raw['_ROW_ID'].is_unique:
    raise RuntimeError('_ROW_ID is not unique.')
required_design_sources = ['V005', 'V021', 'V023', 'V024', 'V025']
missing_design_sources = [x for x in required_design_sources if x not in raw.columns]
if missing_design_sources:
    raise RuntimeError(f'Missing survey-design source variables:\n{missing_design_sources}')
raw = raw.copy()
existing_design_columns = [col for col in ['SURVEY_WEIGHT', 'PSU', 'STRATUM'] if col in raw.columns]
if existing_design_columns:
    raw = raw.drop(columns=existing_design_columns)
v005_numeric = pd.to_numeric(raw['V005'], errors='coerce')
v021_numeric = pd.to_numeric(raw['V021'], errors='coerce')
v023_numeric = pd.to_numeric(raw['V023'], errors='coerce')
design_source_missing = {'V005': int(v005_numeric.isna().sum()), 'V021': int(v021_numeric.isna().sum()), 'V023': int(v023_numeric.isna().sum())}
if any((value > 0 for value in design_source_missing.values())):
    raise RuntimeError(f'Missing survey-design source values detected:\n{design_source_missing}')
if not np.isfinite(v005_numeric.to_numpy(dtype=float)).all():
    raise RuntimeError('V005 contains non-finite values.')
n_nonpositive_v005 = int((v005_numeric <= 0).sum())
if n_nonpositive_v005 > 0:
    raise RuntimeError(f'{n_nonpositive_v005:,} non-positive V005 values detected.')

def assert_integer_like(series, variable_name):
    values = series.to_numpy(dtype=float)
    rounded = np.round(values)
    if not np.allclose(values, rounded, rtol=0, atol=1e-10):
        bad_mask = ~np.isclose(values, rounded, rtol=0, atol=1e-10)
        bad_values = pd.Series(values[bad_mask]).drop_duplicates().head(20).tolist()
        raise RuntimeError(f'{variable_name} contains non-integer-like values:\n{bad_values}')
assert_integer_like(v021_numeric, 'V021')
assert_integer_like(v023_numeric, 'V023')
survey_design_variables = pd.DataFrame({'SURVEY_WEIGHT': v005_numeric.astype(float) / 1000000.0, 'PSU': np.round(v021_numeric).astype(np.int64), 'STRATUM': np.round(v023_numeric).astype(np.int64)}, index=raw.index)
raw = pd.concat([raw, survey_design_variables], axis=1)
n_missing_weight = int(raw['SURVEY_WEIGHT'].isna().sum())
n_nonpositive_weight = int((raw['SURVEY_WEIGHT'] <= 0).sum())
n_missing_psu = int(raw['PSU'].isna().sum())
n_missing_stratum = int(raw['STRATUM'].isna().sum())
n_unique_psu = int(raw['PSU'].nunique())
n_unique_strata = int(raw['STRATUM'].nunique())
if n_missing_weight != 0:
    raise RuntimeError('Missing survey weights remain.')
if n_nonpositive_weight != 0:
    raise RuntimeError('Non-positive survey weights remain.')
if n_missing_psu != 0:
    raise RuntimeError('Missing PSU values remain.')
if n_missing_stratum != 0:
    raise RuntimeError('Missing stratum values remain.')
EXPECTED_RAW_PSU = 674
EXPECTED_RAW_STRATA = 16
if n_unique_psu != EXPECTED_RAW_PSU:
    raise RuntimeError(f'Unexpected number of PSUs.\nExpected : {EXPECTED_RAW_PSU}\nObserved : {n_unique_psu}')
if n_unique_strata != EXPECTED_RAW_STRATA:
    raise RuntimeError(f'Unexpected number of sampling strata.\nExpected : {EXPECTED_RAW_STRATA}\nObserved : {n_unique_strata}')
psu_stratum_nunique = raw.groupby('PSU', observed=True)['STRATUM'].nunique()
inconsistent_psu = psu_stratum_nunique[psu_stratum_nunique > 1]
if len(inconsistent_psu) > 0:
    raise RuntimeError(f'Some PSUs occur in more than one stratum.\nProblem PSUs:\n{inconsistent_psu.head(20)}')
n_inconsistent_psu = int(len(inconsistent_psu))
v024_numeric = pd.to_numeric(raw['V024'], errors='coerce')
v025_numeric = pd.to_numeric(raw['V025'], errors='coerce')
if v024_numeric.isna().any() or v025_numeric.isna().any():
    raise RuntimeError('Missing V024 or V025 values detected.')
assert_integer_like(v024_numeric, 'V024')
assert_integer_like(v025_numeric, 'V025')
stratum_pair_table = raw[['STRATUM', 'V024', 'V025', 'PSU']].groupby(['STRATUM', 'V024', 'V025'], observed=True).agg(N_Rows=('PSU', 'size'), N_PSU=('PSU', 'nunique')).reset_index().sort_values(['STRATUM', 'V024', 'V025']).reset_index(drop=True)
pairs_per_stratum = stratum_pair_table.groupby('STRATUM', observed=True).size()
strata_per_pair = stratum_pair_table.groupby(['V024', 'V025'], observed=True)['STRATUM'].nunique()
if not (pairs_per_stratum == 1).all():
    raise RuntimeError('At least one V023 stratum maps to multiple division/residence combinations.')
if not (strata_per_pair == 1).all():
    raise RuntimeError('At least one division/residence combination maps to multiple V023 strata.')
n_division_residence_pairs = int(stratum_pair_table[['V024', 'V025']].drop_duplicates().shape[0])
if n_division_residence_pairs != 16:
    raise RuntimeError(f'Unexpected number of division x residence combinations.\nExpected : 16\nObserved : {n_division_residence_pairs}')

def safe_value_label(variable, raw_value):
    mapping = VARIABLE_VALUE_LABELS.get(variable, {}) if 'VARIABLE_VALUE_LABELS' in globals() else {}
    candidates = [raw_value]
    try:
        candidates.append(int(raw_value))
    except Exception:
        pass
    try:
        candidates.append(float(raw_value))
    except Exception:
        pass
    for candidate in candidates:
        if candidate in mapping:
            return mapping[candidate]
    return None
stratum_pair_table['Division_Label'] = stratum_pair_table['V024'].map(lambda x: safe_value_label('V024', x))
stratum_pair_table['Residence_Label'] = stratum_pair_table['V025'].map(lambda x: safe_value_label('V025', x))
psu_per_stratum = raw.groupby('STRATUM', observed=True).agg(N_Rows=('_ROW_ID', 'size'), N_PSU=('PSU', 'nunique'), Survey_Weight_Sum=('SURVEY_WEIGHT', 'sum')).reset_index().sort_values('STRATUM').reset_index(drop=True)
n_singleton_strata = int((psu_per_stratum['N_PSU'] <= 1).sum())
if n_singleton_strata > 0:
    raise RuntimeError(f'{n_singleton_strata} singleton sampling strata detected.')
weight_quantiles = raw['SURVEY_WEIGHT'].quantile([0.0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0])
survey_weight_qc = pd.DataFrame({'Metric': ['N', 'Missing', 'Non_positive', 'Minimum', 'P01', 'P05', 'P25', 'Median', 'Mean', 'P75', 'P95', 'P99', 'Maximum', 'Sum'], 'Value': [len(raw), n_missing_weight, n_nonpositive_weight, weight_quantiles.loc[0.0], weight_quantiles.loc[0.01], weight_quantiles.loc[0.05], weight_quantiles.loc[0.25], weight_quantiles.loc[0.5], raw['SURVEY_WEIGHT'].mean(), weight_quantiles.loc[0.75], weight_quantiles.loc[0.95], weight_quantiles.loc[0.99], weight_quantiles.loc[1.0], raw['SURVEY_WEIGHT'].sum()]})
survey_design_qc = pd.DataFrame({'Check': ['Raw respondents', 'Missing survey weight', 'Non-positive survey weight', 'Missing PSU', 'Missing stratum', 'Unique PSU', 'Unique strata', 'PSU assigned to >1 stratum', 'Division x residence combinations', 'Singleton strata'], 'Observed': [len(raw), n_missing_weight, n_nonpositive_weight, n_missing_psu, n_missing_stratum, n_unique_psu, n_unique_strata, n_inconsistent_psu, n_division_residence_pairs, n_singleton_strata], 'Expected': [30078, 0, 0, 0, 0, 674, 16, 0, 16, 0]})
survey_design_qc['PASS'] = survey_design_qc['Observed'] == survey_design_qc['Expected']
if not survey_design_qc['PASS'].all():
    raise RuntimeError('One or more survey-design QC checks failed.')
survey_design_reference = raw[['_ROW_ID', 'CASEID', 'V005', 'SURVEY_WEIGHT', 'PSU', 'STRATUM', 'V024', 'V025']].copy()
STEP3_DESIGN_REFERENCE_FILE = DIRS['processed'] / 'STEP03_raw_survey_design_reference.csv'
STEP3_QC_FILE = DIRS['qc'] / 'STEP03_survey_design_qc.csv'
STEP3_WEIGHT_QC_FILE = DIRS['qc'] / 'STEP03_survey_weight_diagnostics.csv'
STEP3_PSU_STRATUM_FILE = DIRS['qc'] / 'STEP03_psu_per_stratum.csv'
STEP3_STRATUM_CROSSCHECK_FILE = DIRS['qc'] / 'STEP03_stratum_division_residence_crosscheck.csv'
survey_design_reference.to_csv(STEP3_DESIGN_REFERENCE_FILE, index=False)
survey_design_qc.to_csv(STEP3_QC_FILE, index=False)
survey_weight_qc.to_csv(STEP3_WEIGHT_QC_FILE, index=False)
psu_per_stratum.to_csv(STEP3_PSU_STRATUM_FILE, index=False)
stratum_pair_table.to_csv(STEP3_STRATUM_CROSSCHECK_FILE, index=False)
step3_manifest = {'step': 'STEP 3', 'description': 'Survey design construction and hard QC', 'survey_weight_source': 'V005', 'survey_weight_formula': 'V005 / 1,000,000', 'psu_source': 'V021', 'stratum_source': 'V023', 'division_source': 'V024', 'residence_source': 'V025', 'raw_n': int(len(raw)), 'unique_psu': n_unique_psu, 'unique_strata': n_unique_strata, 'psu_stratum_inconsistencies': n_inconsistent_psu, 'division_residence_pairs': n_division_residence_pairs, 'singleton_strata': n_singleton_strata, 'survey_design_variables_are_predictors': False, 'global_weight_rescaling_performed': False, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP3_MANIFEST_FILE = DIRS['protocol'] / 'STEP03_survey_design_manifest.json'
with open(STEP3_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step3_manifest, f, indent=4, ensure_ascii=False)
logger.info(f'STEP 3 completed successfully: N={len(raw):,}, PSU={n_unique_psu}, strata={n_unique_strata}.')
print('\n' + '=' * 78)
print('STEP 3 — SURVEY DESIGN CONSTRUCTION + HARD QC')
print('=' * 78)
print(f'Raw respondents             : {len(raw):,}')
print(f'Missing survey weights      : {n_missing_weight:,}')
print(f'Non-positive survey weights : {n_nonpositive_weight:,}')
print(f'Missing PSU                 : {n_missing_psu:,}')
print(f'Missing stratum             : {n_missing_stratum:,}')
print(f'Unique PSU                  : {n_unique_psu:,}')
print(f'Unique strata               : {n_unique_strata:,}')
print(f'PSU->stratum inconsistencies: {n_inconsistent_psu:,}')
print(f'Division x residence pairs  : {n_division_residence_pairs:,}')
print(f'Singleton strata            : {n_singleton_strata:,}')
print('\nOVERALL SURVEY-DESIGN QC:')
display(survey_design_qc)
print('\nSURVEY-WEIGHT DIAGNOSTICS:')
display(survey_weight_qc)
print('\nPSU DISTRIBUTION BY STRATUM:')
display(psu_per_stratum)
print('\nV023 STRATUM vs DIVISION x RESIDENCE:')
display(stratum_pair_table)
print('\nFILES SAVED:')
print(f'- {STEP3_DESIGN_REFERENCE_FILE}')
print(f'- {STEP3_QC_FILE}')
print(f'- {STEP3_WEIGHT_QC_FILE}')
print(f'- {STEP3_PSU_STRATUM_FILE}')
print(f'- {STEP3_STRATUM_CROSSCHECK_FILE}')
print(f'- {STEP3_MANIFEST_FILE}')
print('\nIMPORTANT:')
print('- No participant was selected or excluded.')
print('- V005/1,000,000 is retained as the original DHS analysis weight.')
print('- No global ML-training weight normalization was performed.')
print('- PSU and stratum are design variables, not predictors.')
print('- ANC outcome construction has not started yet.')
print('=' * 78)
print('STATUS: STEP 3 PASS')
print('=' * 78)


2026-09-16 21:05:41,521 | INFO | STEP 3 completed successfully: N=30,078, PSU=674, strata=16.



STEP 3 — SURVEY DESIGN CONSTRUCTION + HARD QC
Raw respondents             : 30,078
Missing survey weights      : 0
Non-positive survey weights : 0
Missing PSU                 : 0
Missing stratum             : 0
Unique PSU                  : 674
Unique strata               : 16
PSU->stratum inconsistencies: 0
Division x residence pairs  : 16
Singleton strata            : 0

OVERALL SURVEY-DESIGN QC:


,Check,Observed,Expected,PASS
0,Raw respondents,30078,30078,True
1,Missing survey weight,0,0,True
2,Non-positive survey weight,0,0,True
3,Missing PSU,0,0,True
4,Missing stratum,0,0,True
5,Unique PSU,674,674,True
6,Unique strata,16,16,True
7,PSU assigned to >1 stratum,0,0,True
8,Division x residence combinations,16,16,True
9,Singleton strata,0,0,True



SURVEY-WEIGHT DIAGNOSTICS:


,Metric,Value
0,N,30078.000000
1,Missing,0.000000
2,Non_positive,0.000000
3,Minimum,0.091325
4,P01,0.134405
5,P05,0.247309
6,P25,0.659415
7,Median,0.962460
8,Mean,1.000000
9,P75,1.295068



PSU DISTRIBUTION BY STRATUM:


,STRATUM,N_Rows,N_PSU,Survey_Weight_Sum
0,1,1095,24,280.613447
1,2,2137,47,1544.860376
2,3,1847,39,1441.620905
3,4,2614,55,4143.006036
4,5,2217,52,4046.758565
5,6,2337,53,3590.587006
6,7,1248,28,792.292452
7,8,2680,58,2810.128034
8,9,779,17,347.588828
9,10,2476,58,1957.552631



V023 STRATUM vs DIVISION x RESIDENCE:


,STRATUM,V024,V025,N_Rows,N_PSU,Division_Label,Residence_Label
0,1,1,1,1095,24,barishal,urban
1,2,1,2,2137,47,barishal,rural
2,3,2,1,1847,39,chattogram,urban
3,4,2,2,2614,55,chattogram,rural
4,5,3,1,2217,52,dhaka,urban
5,6,3,2,2337,53,dhaka,rural
6,7,4,1,1248,28,khulna,urban
7,8,4,2,2680,58,khulna,rural
8,9,5,1,779,17,mymensingh,urban
9,10,5,2,2476,58,mymensingh,rural



FILES SAVED:
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP03_raw_survey_design_reference.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP03_survey_design_qc.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP03_survey_weight_diagnostics.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP03_psu_per_stratum.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP03_stratum_division_residence_crosscheck.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\03_protocol\STEP03_survey_design_manifest.json

IMPORTANT:
- No participant was selected or excluded.
- V005/1,000,000 is retained as the original DHS analysis weight.
- No global ML-training weight normalization was performed.
- PSU and stratum are design variables, not predictors.
- ANC outcome construction has not started yet.
STATUS: STEP 3 PASS


In [4]:
# Step 4: Maternity-slot and ANC metadata verification.

import re
if 'raw' not in globals():
    logger.info('Raw dataframe not found in memory. Reloading source data.')
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'stata_meta' not in globals():
    _, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), metadataonly=True)
raw_variable_labels = getattr(stata_meta, 'column_names_to_labels', {}) or {}
variable_label_map = {str(variable).strip().upper(): label for variable, label in raw_variable_labels.items()}
raw_variable_value_labels = getattr(stata_meta, 'variable_value_labels', {}) or {}
VARIABLE_VALUE_LABELS = {str(variable).strip().upper(): dict(mapping) for variable, mapping in raw_variable_value_labels.items()}

def numbered_family(columns, prefix):
    pattern = re.compile(f'^{re.escape(prefix)}_(\\d+)$')
    matches = []
    for variable in columns:
        match = pattern.fullmatch(variable)
        if match:
            matches.append((int(match.group(1)), variable))
    return [variable for _, variable in sorted(matches)]

def extract_slot(variable_name):
    match = re.search('_(\\d+)$', variable_name)
    if match is None:
        raise ValueError(f'Cannot determine maternity slot for {variable_name}')
    return int(match.group(1))

def normalize_label_text(value):
    if value is None:
        return ''
    return str(value).strip().lower().replace('’', "'").replace('`', "'").replace('–', '-').replace('—', '-')
M80_COLS = numbered_family(raw.columns, 'M80')
M14_COLS = numbered_family(raw.columns, 'M14')
M10_COLS = numbered_family(raw.columns, 'M10')
EXPECTED_SLOTS = [1, 2, 3, 4, 5, 6]
m80_slots = [extract_slot(x) for x in M80_COLS]
m14_slots = [extract_slot(x) for x in M14_COLS]
m10_slots = [extract_slot(x) for x in M10_COLS]
if m80_slots != EXPECTED_SLOTS:
    raise RuntimeError(f'Unexpected M80 maternity slots: {m80_slots}')
if m14_slots != EXPECTED_SLOTS:
    raise RuntimeError(f'Unexpected M14 maternity slots: {m14_slots}')
if m10_slots != EXPECTED_SLOTS:
    raise RuntimeError(f'Unexpected M10 maternity slots: {m10_slots}')
slot_alignment = pd.DataFrame({'MATERNITY_SLOT': EXPECTED_SLOTS, 'M80_VARIABLE': [f'M80_{slot}' for slot in EXPECTED_SLOTS], 'M14_VARIABLE': [f'M14_{slot}' for slot in EXPECTED_SLOTS], 'M10_VARIABLE': [f'M10_{slot}' for slot in EXPECTED_SLOTS]})
slot_alignment['M80_PRESENT'] = slot_alignment['M80_VARIABLE'].isin(raw.columns)
slot_alignment['M14_PRESENT'] = slot_alignment['M14_VARIABLE'].isin(raw.columns)
slot_alignment['M10_PRESENT'] = slot_alignment['M10_VARIABLE'].isin(raw.columns)
if not slot_alignment[['M80_PRESENT', 'M14_PRESENT', 'M10_PRESENT']].all().all():
    raise RuntimeError('M80/M14/M10 maternity-slot alignment failed.')
maternity_variable_rows = []
for family_name, variable_list in {'M80': M80_COLS, 'M14': M14_COLS, 'M10': M10_COLS}.items():
    for variable in variable_list:
        maternity_variable_rows.append({'Family': family_name, 'Variable': variable, 'Slot': extract_slot(variable), 'Variable_Label': variable_label_map.get(variable), 'Has_Value_Labels': variable in VARIABLE_VALUE_LABELS})
maternity_variable_audit = pd.DataFrame(maternity_variable_rows)

def make_family_value_label_table(variables):
    rows = []
    for variable in variables:
        mapping = VARIABLE_VALUE_LABELS.get(variable, {})
        for raw_value, value_label in mapping.items():
            rows.append({'Variable': variable, 'Slot': extract_slot(variable), 'Raw_Value': raw_value, 'Value_Label': value_label})
    return pd.DataFrame(rows)
m80_value_labels = make_family_value_label_table(M80_COLS)
m14_value_labels = make_family_value_label_table(M14_COLS)
m10_value_labels = make_family_value_label_table(M10_COLS)
m80_code1_rows = []
for variable in M80_COLS:
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    code1_label = None
    for candidate in [1, 1.0]:
        if candidate in mapping:
            code1_label = mapping[candidate]
            break
    normalized_label = normalize_label_text(code1_label)
    verified = normalized_label == 'most recent live birth'
    m80_code1_rows.append({'Variable': variable, 'Slot': extract_slot(variable), 'Raw_Code': 1, 'Value_Label': code1_label, 'Verified_Most_Recent_Live_Birth': verified})
m80_code1_audit = pd.DataFrame(m80_code1_rows)
if not m80_code1_audit['Verified_Most_Recent_Live_Birth'].all():
    raise RuntimeError("M80=1 was NOT verified as 'most recent live birth' in every maternity slot. Do not continue to cohort construction.")
m80_mapping_signatures = []
for variable in M80_COLS:
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    signature = tuple(sorted([(str(key), normalize_label_text(value)) for key, value in mapping.items()]))
    m80_mapping_signatures.append(signature)
m80_mappings_identical = len(set(m80_mapping_signatures)) == 1
if not m80_mappings_identical:
    raise RuntimeError('M80 value-label mappings differ across maternity slots.')
m80_observed_parts = []
for variable in M80_COLS:
    counts = raw[variable].value_counts(dropna=False).rename_axis('Raw_Value').reset_index(name='N')
    counts.insert(0, 'Variable', variable)
    counts.insert(1, 'Slot', extract_slot(variable))
    m80_observed_parts.append(counts)
m80_observed_distribution = pd.concat(m80_observed_parts, ignore_index=True)
m80_observed_distribution = m80_observed_distribution.sort_values(['Slot', 'Raw_Value'], na_position='last').reset_index(drop=True)
SPECIAL_LABEL_PATTERNS = ["don't know", 'dont know', 'unknown', 'missing', 'not applicable', 'not asked', 'inconsistent']

def is_special_label(label):
    text = normalize_label_text(label)
    return any((pattern in text for pattern in SPECIAL_LABEL_PATTERNS))
if m14_value_labels.empty:
    raise RuntimeError('No M14 value-label metadata found. Cannot safely define ANC special codes.')
m14_value_labels['Is_Special_Code'] = m14_value_labels['Value_Label'].map(is_special_label)
m14_special_codes = m14_value_labels.loc[m14_value_labels['Is_Special_Code']].copy()
m14_special_code_summary = m14_special_codes[['Raw_Value', 'Value_Label']].drop_duplicates().sort_values('Raw_Value').reset_index(drop=True)
if m14_special_code_summary.empty:
    raise RuntimeError('No M14 special ANC code was identified from metadata. Inspect the Stata labels before proceeding.')

def lookup_value_label(variable, raw_value):
    if pd.isna(raw_value):
        return None
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    candidate_keys = [raw_value]
    try:
        candidate_keys.append(int(raw_value))
    except Exception:
        pass
    try:
        candidate_keys.append(float(raw_value))
    except Exception:
        pass
    for candidate in candidate_keys:
        if candidate in mapping:
            return mapping[candidate]
    return None
m14_long_parts = []
for variable in M14_COLS:
    temp = pd.DataFrame({'_ROW_ID': raw['_ROW_ID'], 'M14_VARIABLE': variable, 'MATERNITY_SLOT': extract_slot(variable), 'M14_RAW': pd.to_numeric(raw[variable], errors='coerce')})
    m14_long_parts.append(temp)
m14_long = pd.concat(m14_long_parts, ignore_index=True)
m14_long['VALUE_LABEL'] = [lookup_value_label(variable, raw_value) for variable, raw_value in zip(m14_long['M14_VARIABLE'], m14_long['M14_RAW'])]
m14_long['SPECIAL_CODE_BY_LABEL'] = m14_long['VALUE_LABEL'].map(is_special_label)
m14_observed_distribution = m14_long.groupby(['M14_RAW', 'VALUE_LABEL', 'SPECIAL_CODE_BY_LABEL'], dropna=False, observed=True).size().reset_index(name='N_Observed_Across_All_Slots').sort_values('M14_RAW', na_position='last').reset_index(drop=True)
m14_distribution_by_slot = m14_long.groupby(['MATERNITY_SLOT', 'M14_VARIABLE', 'M14_RAW', 'VALUE_LABEL', 'SPECIAL_CODE_BY_LABEL'], dropna=False, observed=True).size().reset_index(name='N').sort_values(['MATERNITY_SLOT', 'M14_RAW'], na_position='last').reset_index(drop=True)
observed_m14_special_summary = m14_long.loc[m14_long['SPECIAL_CODE_BY_LABEL']].groupby(['M14_RAW', 'VALUE_LABEL'], dropna=False, observed=True).size().reset_index(name='N_Observed_Across_All_Slots').sort_values('M14_RAW').reset_index(drop=True)
m14_nonmissing = m14_long['M14_RAW'].dropna()
m14_numeric_summary = pd.DataFrame({'Metric': ['Non-missing M14 cells across six slots', 'Minimum observed M14', 'Maximum observed M14', 'Observed metadata-defined special M14 cells'], 'Value': [int(len(m14_nonmissing)), float(m14_nonmissing.min()), float(m14_nonmissing.max()), int(m14_long['SPECIAL_CODE_BY_LABEL'].sum())]})
if m10_value_labels.empty:
    raise RuntimeError('No M10 value-label metadata found.')
m10_label_summary = m10_value_labels[['Raw_Value', 'Value_Label']].drop_duplicates().sort_values('Raw_Value').reset_index(drop=True)
m10_mapping_signatures = []
for variable in M10_COLS:
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    signature = tuple(sorted([(str(key), normalize_label_text(value)) for key, value in mapping.items()]))
    m10_mapping_signatures.append(signature)
m10_mappings_identical = len(set(m10_mapping_signatures)) == 1
if not m10_mappings_identical:
    raise RuntimeError('M10 value-label mappings differ across maternity slots.')
STEP4_SLOT_FILE = DIRS['qc'] / 'STEP04_maternity_slot_alignment.csv'
STEP4_VARIABLE_FILE = DIRS['raw_reference'] / 'STEP04_maternity_variable_labels.csv'
STEP4_M80_LABEL_FILE = DIRS['raw_reference'] / 'STEP04_M80_value_labels.csv'
STEP4_M80_VERIFY_FILE = DIRS['qc'] / 'STEP04_M80_code1_verification.csv'
STEP4_M80_OBSERVED_FILE = DIRS['qc'] / 'STEP04_M80_observed_distribution.csv'
STEP4_M14_LABEL_FILE = DIRS['raw_reference'] / 'STEP04_M14_value_labels.csv'
STEP4_M14_SPECIAL_FILE = DIRS['qc'] / 'STEP04_M14_special_code_summary.csv'
STEP4_M14_OBSERVED_FILE = DIRS['qc'] / 'STEP04_M14_observed_distribution.csv'
STEP4_M14_SLOT_FILE = DIRS['qc'] / 'STEP04_M14_distribution_by_slot.csv'
STEP4_M14_SPECIAL_OBS_FILE = DIRS['qc'] / 'STEP04_M14_observed_special_values.csv'
STEP4_M10_LABEL_FILE = DIRS['raw_reference'] / 'STEP04_M10_value_labels.csv'
slot_alignment.to_csv(STEP4_SLOT_FILE, index=False)
maternity_variable_audit.to_csv(STEP4_VARIABLE_FILE, index=False)
m80_value_labels.to_csv(STEP4_M80_LABEL_FILE, index=False)
m80_code1_audit.to_csv(STEP4_M80_VERIFY_FILE, index=False)
m80_observed_distribution.to_csv(STEP4_M80_OBSERVED_FILE, index=False)
m14_value_labels.to_csv(STEP4_M14_LABEL_FILE, index=False)
m14_special_code_summary.to_csv(STEP4_M14_SPECIAL_FILE, index=False)
m14_observed_distribution.to_csv(STEP4_M14_OBSERVED_FILE, index=False)
m14_distribution_by_slot.to_csv(STEP4_M14_SLOT_FILE, index=False)
observed_m14_special_summary.to_csv(STEP4_M14_SPECIAL_OBS_FILE, index=False)
m10_value_labels.to_csv(STEP4_M10_LABEL_FILE, index=False)
step4_manifest = {'step': 'STEP 4', 'description': 'Maternity-slot and M80/M14/M10 metadata verification', 'maternity_slots': EXPECTED_SLOTS, 'm80_variables': M80_COLS, 'm14_variables': M14_COLS, 'm10_variables': M10_COLS, 'm80_code_1_verified_as_most_recent_live_birth': bool(m80_code1_audit['Verified_Most_Recent_Live_Birth'].all()), 'm80_value_labels_identical_across_slots': bool(m80_mappings_identical), 'm10_value_labels_identical_across_slots': bool(m10_mappings_identical), 'm14_metadata_special_codes': m14_special_code_summary.to_dict(orient='records'), 'cohort_restriction_performed': False, 'outcome_constructed': False, 'predictors_constructed': False, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP4_MANIFEST_FILE = DIRS['protocol'] / 'STEP04_maternity_metadata_manifest.json'
with open(STEP4_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step4_manifest, f, indent=4, ensure_ascii=False, default=str)
assert len(M80_COLS) == 6
assert len(M14_COLS) == 6
assert len(M10_COLS) == 6
assert m80_slots == EXPECTED_SLOTS
assert m14_slots == EXPECTED_SLOTS
assert m10_slots == EXPECTED_SLOTS
assert m80_code1_audit['Verified_Most_Recent_Live_Birth'].all()
assert m80_mappings_identical
assert m10_mappings_identical
assert not m14_special_code_summary.empty
logger.info('STEP 4 completed successfully: M80/M14/M10 maternity metadata verified.')
print('\n' + '=' * 78)
print('STEP 4 — MATERNITY SLOT + M80/M14/M10 METADATA VERIFICATION')
print('=' * 78)
print('\nMATERNITY SLOT ALIGNMENT:')
display(slot_alignment)
print('\nM80 = 1 VERIFICATION:')
display(m80_code1_audit)
print('\nM80 VALUE-LABEL MAPPINGS IDENTICAL ACROSS SLOTS:')
print(m80_mappings_identical)
print('\nM14 VARIABLE LABELS:')
display(maternity_variable_audit.loc[maternity_variable_audit['Family'] == 'M14'].reset_index(drop=True))
print('\nM14 METADATA-DEFINED SPECIAL CODE(S):')
display(m14_special_code_summary)
print('\nM14 NUMERIC / SPECIAL-CODE SUMMARY:')
display(m14_numeric_summary)
print('\nOBSERVED M14 SPECIAL VALUE(S) ACROSS ALL SIX SLOTS:')
display(observed_m14_special_summary)
print('\nM10 PREGNANCY-INTENTION VALUE LABELS:')
display(m10_label_summary)
print('\nM10 VALUE-LABEL MAPPINGS IDENTICAL ACROSS SLOTS:')
print(m10_mappings_identical)
print('\nFILES SAVED:')
for file_path in [STEP4_SLOT_FILE, STEP4_VARIABLE_FILE, STEP4_M80_LABEL_FILE, STEP4_M80_VERIFY_FILE, STEP4_M80_OBSERVED_FILE, STEP4_M14_LABEL_FILE, STEP4_M14_SPECIAL_FILE, STEP4_M14_OBSERVED_FILE, STEP4_M14_SLOT_FILE, STEP4_M14_SPECIAL_OBS_FILE, STEP4_M10_LABEL_FILE, STEP4_MANIFEST_FILE]:
    print(f'- {file_path}')
print('\nIMPORTANT:')
print('- M80=1 has been checked against the original Stata value labels.')
print('- M14 special codes were identified from metadata, not guessed numerically.')
print('- M10 metadata is preserved for exact maternity-slot matching later.')
print('- No woman has yet been selected or excluded.')
print('- No ANC outcome has yet been created.')
print('=' * 78)
print('STATUS: STEP 4 PASS')
print('=' * 78)


2026-09-16 21:05:46,088 | INFO | STEP 4 completed successfully: M80/M14/M10 maternity metadata verified.



STEP 4 — MATERNITY SLOT + M80/M14/M10 METADATA VERIFICATION

MATERNITY SLOT ALIGNMENT:


,MATERNITY_SLOT,M80_VARIABLE,M14_VARIABLE,M10_VARIABLE,M80_PRESENT,M14_PRESENT,M10_PRESENT
0,1,M80_1,M14_1,M10_1,True,True,True
1,2,M80_2,M14_2,M10_2,True,True,True
2,3,M80_3,M14_3,M10_3,True,True,True
3,4,M80_4,M14_4,M10_4,True,True,True
4,5,M80_5,M14_5,M10_5,True,True,True
5,6,M80_6,M14_6,M10_6,True,True,True



M80 = 1 VERIFICATION:


,Variable,Slot,Raw_Code,Value_Label,Verified_Most_Recent_Live_Birth
0,M80_1,1,1,most recent live birth,True
1,M80_2,2,1,most recent live birth,True
2,M80_3,3,1,most recent live birth,True
3,M80_4,4,1,most recent live birth,True
4,M80_5,5,1,most recent live birth,True
5,M80_6,6,1,most recent live birth,True



M80 VALUE-LABEL MAPPINGS IDENTICAL ACROSS SLOTS:
True

M14 VARIABLE LABELS:


... (6 outputs left out)

,M14_RAW,VALUE_LABEL,N_Observed_Across_All_Slots
0,98.0,don't know,1



M10 PREGNANCY-INTENTION VALUE LABELS:


,Raw_Value,Value_Label
0,1,then
1,2,later
2,3,no more



M10 VALUE-LABEL MAPPINGS IDENTICAL ACROSS SLOTS:
True

FILES SAVED:
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP04_maternity_slot_alignment.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\00_raw_reference\STEP04_maternity_variable_labels.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\00_raw_reference\STEP04_M80_value_labels.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP04_M80_code1_verification.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP04_M80_observed_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\00_raw_reference\STEP04_M14_value_labels.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP04_M14_special_code_summary.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP04_M14_observed_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP04_M14_distribution_by_slot.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP04_M14_observed_special_values.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\00_

In [5]:
# Step 5: Analytic cohort and ANC outcome construction.

if 'raw' not in globals():
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'SURVEY_WEIGHT' not in raw.columns:
    raw['SURVEY_WEIGHT'] = pd.to_numeric(raw['V005'], errors='coerce') / 1000000.0
if 'PSU' not in raw.columns:
    raw['PSU'] = pd.to_numeric(raw['V021'], errors='raise').round().astype(np.int64)
if 'STRATUM' not in raw.columns:
    raw['STRATUM'] = pd.to_numeric(raw['V023'], errors='raise').round().astype(np.int64)
if 'stata_meta' not in globals():
    _, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), metadataonly=True)
raw_variable_value_labels = getattr(stata_meta, 'variable_value_labels', {}) or {}
VARIABLE_VALUE_LABELS = {str(variable).strip().upper(): dict(mapping) for variable, mapping in raw_variable_value_labels.items()}

def numbered_family(columns, prefix):
    pattern = re.compile(f'^{re.escape(prefix)}_(\\d+)$')
    matches = []
    for variable in columns:
        match = pattern.fullmatch(variable)
        if match:
            matches.append((int(match.group(1)), variable))
    return [variable for _, variable in sorted(matches)]

def extract_slot(variable_name):
    match = re.search('_(\\d+)$', variable_name)
    if match is None:
        raise ValueError(f'Unable to identify slot: {variable_name}')
    return int(match.group(1))

def normalize_label_text(value):
    if value is None:
        return ''
    return str(value).strip().lower().replace('’', "'").replace('`', "'")

def lookup_value_label(variable, raw_value):
    if pd.isna(raw_value):
        return None
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    candidates = [raw_value]
    try:
        candidates.append(int(raw_value))
    except Exception:
        pass
    try:
        candidates.append(float(raw_value))
    except Exception:
        pass
    for candidate in candidates:
        if candidate in mapping:
            return mapping[candidate]
    return None
M80_COLS = numbered_family(raw.columns, 'M80')
M14_COLS = numbered_family(raw.columns, 'M14')
M10_COLS = numbered_family(raw.columns, 'M10')
EXPECTED_SLOTS = [1, 2, 3, 4, 5, 6]
if [extract_slot(x) for x in M80_COLS] != EXPECTED_SLOTS:
    raise RuntimeError('Unexpected M80 slot structure.')
if [extract_slot(x) for x in M14_COLS] != EXPECTED_SLOTS:
    raise RuntimeError('Unexpected M14 slot structure.')
if [extract_slot(x) for x in M10_COLS] != EXPECTED_SLOTS:
    raise RuntimeError('Unexpected M10 slot structure.')
for variable in M80_COLS:
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    label_for_one = None
    for key in [1, 1.0]:
        if key in mapping:
            label_for_one = mapping[key]
            break
    if normalize_label_text(label_for_one) != 'most recent live birth':
        raise RuntimeError(f'M80=1 verification failed for {variable}.')
SPECIAL_LABEL_PATTERNS = ["don't know", 'dont know', 'unknown', 'missing', 'not applicable', 'not asked', 'inconsistent']

def is_special_label(value):
    text = normalize_label_text(value)
    return any((pattern in text for pattern in SPECIAL_LABEL_PATTERNS))
m14_special_code_records = []
for variable in M14_COLS:
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    for raw_code, value_label in mapping.items():
        if is_special_label(value_label):
            m14_special_code_records.append({'Variable': variable, 'Raw_Code': raw_code, 'Value_Label': value_label})
m14_special_code_table = pd.DataFrame(m14_special_code_records).drop_duplicates().reset_index(drop=True)
if m14_special_code_table.empty:
    raise RuntimeError('No metadata-defined M14 special code found.')
M14_SPECIAL_CODES = set(pd.to_numeric(m14_special_code_table['Raw_Code'], errors='raise').astype(float).tolist())
m80_numeric = raw[M80_COLS].apply(pd.to_numeric, errors='coerce')
m80_is_selected = m80_numeric.eq(1)
raw_m80_selected_count = m80_is_selected.sum(axis=1)
m80_selection_audit = raw_m80_selected_count.value_counts(dropna=False).sort_index().rename_axis('N_M80_EQ_1_PER_WOMAN').reset_index(name='N_WOMEN')
n_zero_m80 = int((raw_m80_selected_count == 0).sum())
n_exactly_one_m80 = int((raw_m80_selected_count == 1).sum())
n_multiple_m80 = int((raw_m80_selected_count > 1).sum())
max_m80_flags_per_woman = int(raw_m80_selected_count.max())
if n_multiple_m80 > 0:
    raise RuntimeError(f'{n_multiple_m80} women have more than one M80=1 slot.')
EXPECTED_M80_SELECTED_N = 5064
if n_exactly_one_m80 != EXPECTED_M80_SELECTED_N:
    raise RuntimeError(f'Unexpected number of women with exactly one M80=1.\nExpected : {EXPECTED_M80_SELECTED_N:,}\nObserved : {n_exactly_one_m80:,}')
selected_mask = raw_m80_selected_count == 1
selected_positions = np.flatnonzero(selected_mask.to_numpy())
selected_m80_matrix = m80_is_selected.to_numpy()[selected_positions, :]
selected_slot_zero_based = np.argmax(selected_m80_matrix, axis=1)
selected_slot = selected_slot_zero_based + 1
m80_numeric_matrix = m80_numeric.to_numpy(dtype=float)
selected_m80_values = np.take_along_axis(m80_numeric_matrix[selected_positions, :], selected_slot_zero_based[:, None], axis=1).ravel()
if not np.all(selected_m80_values == 1):
    raise RuntimeError('Selected M80 slot does not contain M80=1 for every woman.')
m14_numeric = raw[M14_COLS].apply(pd.to_numeric, errors='coerce')
m14_numeric_matrix = m14_numeric.to_numpy(dtype=float)
selected_m14 = np.take_along_axis(m14_numeric_matrix[selected_positions, :], selected_slot_zero_based[:, None], axis=1).ravel()
m10_numeric = raw[M10_COLS].apply(pd.to_numeric, errors='coerce')
m10_numeric_matrix = m10_numeric.to_numpy(dtype=float)
selected_m10 = np.take_along_axis(m10_numeric_matrix[selected_positions, :], selected_slot_zero_based[:, None], axis=1).ravel()
selected_source_rows = raw.iloc[selected_positions].copy().reset_index(drop=True)
index_birth_cohort = pd.DataFrame({'_ROW_ID': selected_source_rows['_ROW_ID'].astype(np.int64), 'CASEID': selected_source_rows['CASEID'], 'INDEX_MATERNITY_SLOT': selected_slot.astype(np.int64), 'M80_SELECTED_VALUE': selected_m80_values, 'ANC_VISITS_RAW': selected_m14, 'PREG_INTENTION_RAW': selected_m10, 'SURVEY_WEIGHT': selected_source_rows['SURVEY_WEIGHT'].astype(float), 'PSU': selected_source_rows['PSU'].astype(np.int64), 'STRATUM': selected_source_rows['STRATUM'].astype(np.int64)})
index_birth_cohort['M80_SOURCE_VARIABLE'] = [f'M80_{slot}' for slot in index_birth_cohort['INDEX_MATERNITY_SLOT']]
index_birth_cohort['M14_SOURCE_VARIABLE'] = [f'M14_{slot}' for slot in index_birth_cohort['INDEX_MATERNITY_SLOT']]
index_birth_cohort['M10_SOURCE_VARIABLE'] = [f'M10_{slot}' for slot in index_birth_cohort['INDEX_MATERNITY_SLOT']]
index_birth_cohort['ANC_VISITS_VALUE_LABEL'] = [lookup_value_label(variable, raw_value) for variable, raw_value in zip(index_birth_cohort['M14_SOURCE_VARIABLE'], index_birth_cohort['ANC_VISITS_RAW'])]
index_birth_cohort['PREG_INTENTION_VALUE_LABEL'] = [lookup_value_label(variable, raw_value) for variable, raw_value in zip(index_birth_cohort['M10_SOURCE_VARIABLE'], index_birth_cohort['PREG_INTENTION_RAW'])]
index_birth_cohort['ANC_MISSING'] = index_birth_cohort['ANC_VISITS_RAW'].isna()
index_birth_cohort['ANC_SPECIAL_CODE'] = index_birth_cohort['ANC_VISITS_RAW'].astype(float).isin(M14_SPECIAL_CODES)
index_birth_cohort['ANC_VALID'] = ~index_birth_cohort['ANC_MISSING'] & ~index_birth_cohort['ANC_SPECIAL_CODE']
valid_anc_values = index_birth_cohort.loc[index_birth_cohort['ANC_VALID'], 'ANC_VISITS_RAW'].astype(float)
if (valid_anc_values < 0).any():
    raise RuntimeError('Negative valid ANC visit count detected.')
if not np.allclose(valid_anc_values.to_numpy(), np.round(valid_anc_values.to_numpy()), rtol=0, atol=1e-10):
    raise RuntimeError('Non-integer ANC visit count detected.')
invalid_anc_records = index_birth_cohort.loc[~index_birth_cohort['ANC_VALID']].copy().reset_index(drop=True)
n_missing_selected_m14 = int(index_birth_cohort['ANC_MISSING'].sum())
n_special_selected_m14 = int(index_birth_cohort['ANC_SPECIAL_CODE'].sum())
n_invalid_selected_m14 = int((~index_birth_cohort['ANC_VALID']).sum())
analytic_cohort = index_birth_cohort.loc[index_birth_cohort['ANC_VALID']].copy().reset_index(drop=True)
analytic_cohort['ANC_VISITS'] = analytic_cohort['ANC_VISITS_RAW'].round().astype(np.int64)
analytic_cohort['ANC_CLASS'] = np.select([analytic_cohort['ANC_VISITS'].eq(0), analytic_cohort['ANC_VISITS'].between(1, 7, inclusive='both'), analytic_cohort['ANC_VISITS'].ge(8)], [0, 1, 2], default=-1).astype(np.int8)
if (analytic_cohort['ANC_CLASS'] < 0).any():
    raise RuntimeError('At least one valid ANC record could not be classified.')
ANC_CLASS_LABEL_MAP = {0: 'No ANC', 1: '1-7 ANC visits', 2: '>=8 ANC visits'}
analytic_cohort['ANC_CLASS_LABEL'] = analytic_cohort['ANC_CLASS'].map(ANC_CLASS_LABEL_MAP)
EXPECTED_FINAL_N = 5063
EXPECTED_CLASS_COUNTS = {0: 403, 1: 4407, 2: 253}
if len(analytic_cohort) != EXPECTED_FINAL_N:
    raise RuntimeError(f'Unexpected final analytic cohort size.\nExpected : {EXPECTED_FINAL_N:,}\nObserved : {len(analytic_cohort):,}')
observed_class_counts = analytic_cohort['ANC_CLASS'].value_counts().sort_index().to_dict()
if observed_class_counts != EXPECTED_CLASS_COUNTS:
    raise RuntimeError(f'ANC outcome counts differ from the locked expected counts.\nExpected : {EXPECTED_CLASS_COUNTS}\nObserved : {observed_class_counts}')
if not analytic_cohort['_ROW_ID'].is_unique:
    raise RuntimeError('Duplicate _ROW_ID in final analytic cohort.')
if not analytic_cohort['CASEID'].is_unique:
    raise RuntimeError('Duplicate CASEID in final analytic cohort.')
final_n_psu = int(analytic_cohort['PSU'].nunique())
final_n_strata = int(analytic_cohort['STRATUM'].nunique())
final_psu_per_stratum = analytic_cohort.groupby('STRATUM', observed=True).agg(N=('CASEID', 'size'), N_PSU=('PSU', 'nunique'), Survey_Weight_Sum=('SURVEY_WEIGHT', 'sum')).reset_index().sort_values('STRATUM').reset_index(drop=True)
final_singleton_strata = int((final_psu_per_stratum['N_PSU'] <= 1).sum())
if final_n_psu != 672:
    raise RuntimeError(f'Unexpected final analytic PSU count.\nExpected : 672\nObserved : {final_n_psu}')
if final_n_strata != 16:
    raise RuntimeError('Unexpected final analytic stratum count.')
if final_singleton_strata != 0:
    raise RuntimeError('Singleton stratum detected in final analytic cohort.')
cohort_flow = pd.DataFrame({'Stage': ['Raw BDHS interviewed women', 'No M80=1 most-recent-live-birth record', 'Exactly one M80=1 most-recent-live-birth record', 'Multiple M80=1 records', 'Selected records with missing M14', 'Selected records with metadata-defined special M14', 'Final analytic cohort with valid ANC count'], 'N': [int(len(raw)), n_zero_m80, n_exactly_one_m80, n_multiple_m80, n_missing_selected_m14, n_special_selected_m14, int(len(analytic_cohort))]})
outcome_distribution = analytic_cohort.groupby(['ANC_CLASS', 'ANC_CLASS_LABEL'], observed=True).agg(Unweighted_N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index().sort_values('ANC_CLASS').reset_index(drop=True)
outcome_distribution['Unweighted_Percent'] = outcome_distribution['Unweighted_N'] / len(analytic_cohort) * 100
outcome_distribution['Weighted_Percent'] = outcome_distribution['Weighted_N'] / outcome_distribution['Weighted_N'].sum() * 100
selected_slot_distribution = index_birth_cohort['INDEX_MATERNITY_SLOT'].value_counts().sort_index().rename_axis('MATERNITY_SLOT').reset_index(name='N_WOMEN')
m10_selected_distribution = analytic_cohort.groupby(['PREG_INTENTION_RAW', 'PREG_INTENTION_VALUE_LABEL'], dropna=False, observed=True).size().reset_index(name='N').sort_values('PREG_INTENTION_RAW', na_position='last').reset_index(drop=True)
final_anc_summary = pd.DataFrame({'Metric': ['Final N', 'Minimum ANC visits', 'Median ANC visits', 'Maximum ANC visits', 'Unique PSU', 'Unique strata', 'Singleton strata'], 'Value': [len(analytic_cohort), analytic_cohort['ANC_VISITS'].min(), analytic_cohort['ANC_VISITS'].median(), analytic_cohort['ANC_VISITS'].max(), final_n_psu, final_n_strata, final_singleton_strata]})
STEP5_PREOUTCOME_FILE = DIRS['processed'] / 'STEP05_index_birth_selected_pre_outcome.csv'
STEP5_FINAL_COHORT_FILE = DIRS['processed'] / 'STEP05_analytic_cohort_anc_outcome.csv'
STEP5_FLOW_FILE = DIRS['qc'] / 'STEP05_cohort_flow.csv'
STEP5_M80_AUDIT_FILE = DIRS['qc'] / 'STEP05_M80_selection_audit.csv'
STEP5_INVALID_ANC_FILE = DIRS['qc'] / 'STEP05_invalid_anc_records.csv'
STEP5_OUTCOME_FILE = DIRS['qc'] / 'STEP05_anc_outcome_distribution.csv'
STEP5_SLOT_DIST_FILE = DIRS['qc'] / 'STEP05_selected_maternity_slot_distribution.csv'
STEP5_M10_DIST_FILE = DIRS['qc'] / 'STEP05_selected_M10_distribution.csv'
STEP5_FINAL_DESIGN_FILE = DIRS['qc'] / 'STEP05_final_psu_per_stratum.csv'
index_birth_cohort.to_csv(STEP5_PREOUTCOME_FILE, index=False)
analytic_cohort.to_csv(STEP5_FINAL_COHORT_FILE, index=False)
cohort_flow.to_csv(STEP5_FLOW_FILE, index=False)
m80_selection_audit.to_csv(STEP5_M80_AUDIT_FILE, index=False)
invalid_anc_records.to_csv(STEP5_INVALID_ANC_FILE, index=False)
outcome_distribution.to_csv(STEP5_OUTCOME_FILE, index=False)
selected_slot_distribution.to_csv(STEP5_SLOT_DIST_FILE, index=False)
m10_selected_distribution.to_csv(STEP5_M10_DIST_FILE, index=False)
final_psu_per_stratum.to_csv(STEP5_FINAL_DESIGN_FILE, index=False)
STEP5_FINAL_COHORT_SHA256 = calculate_sha256(STEP5_FINAL_COHORT_FILE)
step5_manifest = {'step': 'STEP 5', 'description': 'Exact most-recent-live-birth cohort and ANC outcome', 'raw_n': int(len(raw)), 'women_with_zero_M80_eq_1': n_zero_m80, 'women_with_exactly_one_M80_eq_1': n_exactly_one_m80, 'women_with_multiple_M80_eq_1': n_multiple_m80, 'selected_M14_missing': n_missing_selected_m14, 'selected_M14_special': n_special_selected_m14, 'M14_special_codes': sorted(list(M14_SPECIAL_CODES)), 'final_analytic_n': int(len(analytic_cohort)), 'ANC_class_counts': {str(key): int(value) for key, value in observed_class_counts.items()}, 'final_unique_psu': final_n_psu, 'final_unique_strata': final_n_strata, 'final_singleton_strata': final_singleton_strata, 'final_cohort_sha256': STEP5_FINAL_COHORT_SHA256, 'predictor_recode_performed': False, 'imputation_performed': False, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP5_MANIFEST_FILE = DIRS['protocol'] / 'STEP05_cohort_outcome_manifest.json'
with open(STEP5_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step5_manifest, f, indent=4, ensure_ascii=False, default=str)
assert len(raw) == 30078
assert n_exactly_one_m80 == 5064
assert n_multiple_m80 == 0
assert n_missing_selected_m14 == 0
assert n_special_selected_m14 == 1
assert len(analytic_cohort) == 5063
assert observed_class_counts == {0: 403, 1: 4407, 2: 253}
assert final_n_psu == 672
assert final_n_strata == 16
assert final_singleton_strata == 0
assert analytic_cohort['CASEID'].is_unique
assert analytic_cohort['_ROW_ID'].is_unique
logger.info(f'STEP 5 completed successfully: selected={n_exactly_one_m80:,}, final N={len(analytic_cohort):,}, PSU={final_n_psu}, strata={final_n_strata}.')
print('\n' + '=' * 78)
print('STEP 5 — EXACT MOST-RECENT-LIVE-BIRTH COHORT + ANC OUTCOME')
print('=' * 78)
print('\nM80 SELECTION AUDIT:')
display(m80_selection_audit)
print('\nCOHORT FLOW:')
display(cohort_flow)
print('\nSELECTED MATERNITY SLOT DISTRIBUTION:')
display(selected_slot_distribution)
print('\nINVALID / SPECIAL ANC RECORD(S):')
display(invalid_anc_records[['CASEID', 'INDEX_MATERNITY_SLOT', 'M14_SOURCE_VARIABLE', 'ANC_VISITS_RAW', 'ANC_VISITS_VALUE_LABEL', 'ANC_MISSING', 'ANC_SPECIAL_CODE']])
print('\nFINAL ANC OUTCOME DISTRIBUTION:')
display(outcome_distribution)
print('\nFINAL ANC / DESIGN SUMMARY:')
display(final_anc_summary)
print('\nSLOT-MATCHED M10 DISTRIBUTION:')
display(m10_selected_distribution)
print('\nFINAL ANALYTIC FILE SHA-256:')
print(STEP5_FINAL_COHORT_SHA256)
print('\nFILES SAVED:')
for path in [STEP5_PREOUTCOME_FILE, STEP5_FINAL_COHORT_FILE, STEP5_FLOW_FILE, STEP5_M80_AUDIT_FILE, STEP5_INVALID_ANC_FILE, STEP5_OUTCOME_FILE, STEP5_SLOT_DIST_FILE, STEP5_M10_DIST_FILE, STEP5_FINAL_DESIGN_FILE, STEP5_MANIFEST_FILE]:
    print(f'- {path}')
print('\nIMPORTANT:')
print('- M14 and M10 were taken from the exact M80=1 maternity slot.')
print('- No arbitrary M14 >=90 exclusion rule was used.')
print('- Only missing or metadata-defined special ANC values were excluded.')
print('- ANC classes are: 0=No ANC, 1=1-7 visits, 2=>=8 visits.')
print('- >=8 is a visit-count category; interpretation as WHO contact completion will be handled cautiously in the manuscript.')
print('- No predictor has yet been recoded or modeled.')
print('=' * 78)
print('STATUS: STEP 5 PASS')
print('=' * 78)


2026-09-16 21:05:49,333 | INFO | STEP 5 completed successfully: selected=5,064, final N=5,063, PSU=672, strata=16.



STEP 5 — EXACT MOST-RECENT-LIVE-BIRTH COHORT + ANC OUTCOME

M80 SELECTION AUDIT:


,N_M80_EQ_1_PER_WOMAN,N_WOMEN
0,0,25014
1,1,5064



COHORT FLOW:


,Stage,N
0,Raw BDHS interviewed women,30078
1,No M80=1 most-recent-live-birth record,25014
2,Exactly one M80=1 most-recent-live-birth record,5064
3,Multiple M80=1 records,0
4,Selected records with missing M14,0
5,Selected records with metadata-defined special...,1
6,Final analytic cohort with valid ANC count,5063



SELECTED MATERNITY SLOT DISTRIBUTION:


... (6 outputs left out)

,Metric,Value
0,Final N,5063.0
1,Minimum ANC visits,0.0
2,Median ANC visits,3.0
3,Maximum ANC visits,20.0
4,Unique PSU,672.0
5,Unique strata,16.0
6,Singleton strata,0.0



SLOT-MATCHED M10 DISTRIBUTION:


,PREG_INTENTION_RAW,PREG_INTENTION_VALUE_LABEL,N
0,1.0,then,4076
1,2.0,later,617
2,3.0,no more,370



FINAL ANALYTIC FILE SHA-256:
9f2aacec3b046316d5bcd23203c90c8fcad9b67e21dfa975c6680a6588eac0c6

FILES SAVED:
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP05_index_birth_selected_pre_outcome.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP05_analytic_cohort_anc_outcome.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP05_cohort_flow.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP05_M80_selection_audit.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP05_invalid_anc_records.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP05_anc_outcome_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP05_selected_maternity_slot_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP05_selected_M10_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP05_final_psu_per_stratum.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\03_protocol\STEP05_cohort_outcome_manife

In [6]:
# Step 6: Exact maternity-to-birth-history linkage.

import re
import json
import datetime as dt
import numpy as np
import pandas as pd
if 'analytic_cohort' not in globals():
    STEP5_FINAL_COHORT_FILE = DIRS['processed'] / 'STEP05_analytic_cohort_anc_outcome.csv'
    if not STEP5_FINAL_COHORT_FILE.exists():
        raise FileNotFoundError(f'Step-5 analytic cohort not found:\n{STEP5_FINAL_COHORT_FILE}')
    analytic_cohort = pd.read_csv(STEP5_FINAL_COHORT_FILE)
if len(analytic_cohort) != 5063:
    raise RuntimeError(f'Unexpected Step-5 cohort N: {len(analytic_cohort):,}')
if not analytic_cohort['_ROW_ID'].is_unique:
    raise RuntimeError('Step-5 _ROW_ID is not unique.')
if not analytic_cohort['CASEID'].is_unique:
    raise RuntimeError('Step-5 CASEID is not unique.')
if 'raw' not in globals():
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'stata_meta' not in globals():
    _, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), metadataonly=True)
raw_variable_labels = getattr(stata_meta, 'column_names_to_labels', {}) or {}
variable_label_map = {str(variable).strip().upper(): label for variable, label in raw_variable_labels.items()}

def numbered_family(columns, prefix):
    pattern = re.compile(f'^{re.escape(prefix)}_(\\d+)$')
    matches = []
    for variable in columns:
        match = pattern.fullmatch(variable)
        if match:
            matches.append((int(match.group(1)), variable))
    return [variable for _, variable in sorted(matches)]

def suffix_number(variable):
    match = re.search('_(\\d+)$', variable)
    if match is None:
        raise ValueError(f'Cannot identify suffix: {variable}')
    return int(match.group(1))
MIDX_COLS = numbered_family(raw.columns, 'MIDX')
BIDX_COLS = numbered_family(raw.columns, 'BIDX')
B3_COLS = numbered_family(raw.columns, 'B3')
BORD_COLS = numbered_family(raw.columns, 'BORD')
B0_COLS = numbered_family(raw.columns, 'B0')
print('Detected family sizes:')
print(f'MIDX : {len(MIDX_COLS)}')
print(f'BIDX : {len(BIDX_COLS)}')
print(f'B3   : {len(B3_COLS)}')
print(f'BORD : {len(BORD_COLS)}')
print(f'B0   : {len(B0_COLS)}')
if len(MIDX_COLS) != 6:
    raise RuntimeError(f'Expected 6 MIDX variables; found {len(MIDX_COLS)}.')
EXPECTED_MATERNITY_SLOTS = [1, 2, 3, 4, 5, 6]
if [suffix_number(x) for x in MIDX_COLS] != EXPECTED_MATERNITY_SLOTS:
    raise RuntimeError('Unexpected MIDX maternity-slot structure.')
EXPECTED_BIRTH_HISTORY_SLOTS = list(range(1, 21))
for family_name, columns in {'BIDX': BIDX_COLS, 'B3': B3_COLS, 'BORD': BORD_COLS, 'B0': B0_COLS}.items():
    observed_suffixes = [suffix_number(x) for x in columns]
    if observed_suffixes != EXPECTED_BIRTH_HISTORY_SLOTS:
        raise RuntimeError(f'{family_name} birth-history slots are not 01-20.\nObserved: {observed_suffixes}')
linkage_variable_audit = pd.DataFrame([{'Family': family, 'First_Variable': columns[0], 'First_Variable_Label': variable_label_map.get(columns[0]), 'N_Variables': len(columns)} for family, columns in {'MIDX': MIDX_COLS, 'BIDX': BIDX_COLS, 'B3': B3_COLS, 'BORD': BORD_COLS, 'B0': B0_COLS}.items()])
raw_by_rowid = raw.set_index('_ROW_ID', drop=False)
missing_rowids = set(analytic_cohort['_ROW_ID'].astype(int)) - set(raw_by_rowid.index)
if missing_rowids:
    raise RuntimeError(f'{len(missing_rowids)} Step-5 row IDs are absent from the raw dataset.')
source = raw_by_rowid.loc[analytic_cohort['_ROW_ID'].astype(int).tolist()].reset_index(drop=True)
if len(source) != len(analytic_cohort):
    raise RuntimeError('Raw-to-analytic row alignment failed.')
if not np.array_equal(source['_ROW_ID'].to_numpy(), analytic_cohort['_ROW_ID'].astype(int).to_numpy()):
    raise RuntimeError('_ROW_ID order mismatch after raw alignment.')
index_maternity_slot = analytic_cohort['INDEX_MATERNITY_SLOT'].astype(int).to_numpy()
if not np.isin(index_maternity_slot, EXPECTED_MATERNITY_SLOTS).all():
    raise RuntimeError('Invalid INDEX_MATERNITY_SLOT detected.')
midx_matrix = source[MIDX_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
selected_midx = np.take_along_axis(midx_matrix, (index_maternity_slot - 1)[:, None], axis=1).ravel()
n_missing_selected_midx = int(np.isnan(selected_midx).sum())
if n_missing_selected_midx > 0:
    raise RuntimeError(f'{n_missing_selected_midx} selected MIDX values are missing.')
if not np.allclose(selected_midx, np.round(selected_midx), rtol=0, atol=1e-10):
    raise RuntimeError('Selected MIDX contains non-integer values.')
selected_midx = np.round(selected_midx).astype(np.int64)
if (selected_midx <= 0).any():
    raise RuntimeError('M80=1 live-birth records contain non-positive MIDX.')
bidx_matrix = source[BIDX_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
b3_matrix = source[B3_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
bord_matrix = source[BORD_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
b0_matrix = source[B0_COLS].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
bidx_match_matrix = bidx_matrix == selected_midx[:, None]
bidx_match_count = bidx_match_matrix.sum(axis=1)
n_zero_bidx_matches = int((bidx_match_count == 0).sum())
n_multiple_bidx_matches = int((bidx_match_count > 1).sum())
if n_zero_bidx_matches > 0:
    raise RuntimeError(f'{n_zero_bidx_matches} women have no BIDX match for selected MIDX.')
if n_multiple_bidx_matches > 0:
    raise RuntimeError(f'{n_multiple_bidx_matches} women have multiple BIDX matches for selected MIDX.')
matched_birth_position_zero_based = np.argmax(bidx_match_matrix, axis=1)
matched_bidx = np.take_along_axis(bidx_matrix, matched_birth_position_zero_based[:, None], axis=1).ravel()
index_birth_cmc = np.take_along_axis(b3_matrix, matched_birth_position_zero_based[:, None], axis=1).ravel()
matched_bord = np.take_along_axis(bord_matrix, matched_birth_position_zero_based[:, None], axis=1).ravel()
matched_b0 = np.take_along_axis(b0_matrix, matched_birth_position_zero_based[:, None], axis=1).ravel()
if not np.array_equal(selected_midx, matched_bidx.astype(np.int64)):
    raise RuntimeError('Selected MIDX and matched BIDX are not identical.')
n_missing_b3 = int(np.isnan(index_birth_cmc).sum())
n_missing_bord = int(np.isnan(matched_bord).sum())
n_missing_b0 = int(np.isnan(matched_b0).sum())
if n_missing_b3 > 0:
    raise RuntimeError(f'{n_missing_b3} matched index births have missing B3.')
if n_missing_bord > 0:
    raise RuntimeError(f'{n_missing_bord} matched index births have missing BORD.')
if n_missing_b0 > 0:
    raise RuntimeError(f'{n_missing_b0} matched index births have missing B0.')
same_birth_cmc_matrix = np.isclose(b3_matrix, index_birth_cmc[:, None], rtol=0, atol=1e-10, equal_nan=False)
same_cmc_birth_count = same_birth_cmc_matrix.sum(axis=1).astype(np.int64)
if (same_cmc_birth_count < 1).any():
    raise RuntimeError('At least one matched index birth has no same-CMC birth-history entry.')
multiple_birth_by_cmc = same_cmc_birth_count > 1
multiple_birth_by_b0 = matched_b0 > 0
multiple_birth_disagreement = int((multiple_birth_by_cmc != multiple_birth_by_b0).sum())
if multiple_birth_disagreement > 0:
    raise RuntimeError('Multiple-birth identification disagrees between B3 same-CMC structure and matched B0.')
births_later_than_index = b3_matrix > index_birth_cmc[:, None]
n_later_birth_entries_per_woman = births_later_than_index.sum(axis=1)
n_women_with_later_birth = int((n_later_birth_entries_per_woman > 0).sum())
if n_women_with_later_birth > 0:
    raise RuntimeError(f'{n_women_with_later_birth} women have a live birth with B3 later than the M80=1 linked birth.')
v201 = pd.to_numeric(source['V201'], errors='coerce').to_numpy(dtype=float)
if np.isnan(v201).any():
    raise RuntimeError('V201 contains missing values in analytic cohort.')
prior_live_births_v201_candidate = v201 - same_cmc_birth_count
if (prior_live_births_v201_candidate < 0).any():
    raise RuntimeError('Negative candidate prior-live-birth count detected.')
prior_live_births_b3_count = (b3_matrix < index_birth_cmc[:, None]).sum(axis=1).astype(np.int64)
eligible_full_birth_history_check = v201 <= 20
prior_parity_disagreement_full_history = int((prior_live_births_b3_count[eligible_full_birth_history_check] != prior_live_births_v201_candidate[eligible_full_birth_history_check]).sum())
if prior_parity_disagreement_full_history > 0:
    raise RuntimeError('V201-based candidate prior live births disagree with the complete B3 birth-history count.')
old_bord_minus_one = matched_bord - 1
bord_minus_one_disagreement = old_bord_minus_one != prior_live_births_v201_candidate
n_bord_minus_one_disagreement = int(bord_minus_one_disagreement.sum())
maternal_dob_cmc = pd.to_numeric(source['V011'], errors='coerce').to_numpy(dtype=float)
current_age_v012 = pd.to_numeric(source['V012'], errors='coerce').to_numpy(dtype=float)
if np.isnan(maternal_dob_cmc).any():
    raise RuntimeError('V011 contains missing values.')
candidate_age_at_index_birth_years = (index_birth_cmc - maternal_dob_cmc) / 12.0
n_age_below_10 = int((candidate_age_at_index_birth_years < 10).sum())
n_age_above_50 = int((candidate_age_at_index_birth_years >= 50).sum())
if n_age_below_10 > 0 or n_age_above_50 > 0:
    raise RuntimeError(f'Implausible candidate maternal age at index birth detected.\n<10 years: {n_age_below_10}\n>=50 years: {n_age_above_50}')
matched_birth_history_position = matched_birth_position_zero_based + 1
matched_bidx_variable = [BIDX_COLS[position] for position in matched_birth_position_zero_based]
matched_b3_variable = [B3_COLS[position] for position in matched_birth_position_zero_based]
matched_bord_variable = [BORD_COLS[position] for position in matched_birth_position_zero_based]
matched_b0_variable = [B0_COLS[position] for position in matched_birth_position_zero_based]
birth_linkage_audit = pd.DataFrame({'_ROW_ID': analytic_cohort['_ROW_ID'].astype(int), 'CASEID': analytic_cohort['CASEID'], 'INDEX_MATERNITY_SLOT': index_maternity_slot, 'MIDX_SOURCE_VARIABLE': [f'MIDX_{slot}' for slot in index_maternity_slot], 'SELECTED_MIDX': selected_midx, 'MATCHED_BIRTH_HISTORY_POSITION': matched_birth_history_position, 'MATCHED_BIDX_VARIABLE': matched_bidx_variable, 'MATCHED_BIDX': matched_bidx, 'MATCHED_B3_VARIABLE': matched_b3_variable, 'INDEX_BIRTH_CMC': index_birth_cmc, 'MATCHED_BORD_VARIABLE': matched_bord_variable, 'MATCHED_BORD': matched_bord, 'MATCHED_B0_VARIABLE': matched_b0_variable, 'MATCHED_B0': matched_b0, 'SAME_CMC_BIRTH_COUNT': same_cmc_birth_count, 'MULTIPLE_BIRTH': multiple_birth_by_cmc, 'V201_TOTAL_LIVE_BIRTHS': v201, 'PRIOR_LIVE_BIRTHS_V201_CANDIDATE': prior_live_births_v201_candidate, 'PRIOR_LIVE_BIRTHS_B3_COUNT': prior_live_births_b3_count, 'OLD_BORD_MINUS_ONE': old_bord_minus_one, 'BORD_MINUS_ONE_DISAGREES': bord_minus_one_disagreement, 'MATERNAL_DOB_CMC_V011': maternal_dob_cmc, 'CURRENT_AGE_V012': current_age_v012, 'CANDIDATE_AGE_AT_INDEX_BIRTH_YEARS': candidate_age_at_index_birth_years, 'N_LATER_BIRTH_ENTRIES': n_later_birth_entries_per_woman})
midx_distribution = birth_linkage_audit['SELECTED_MIDX'].value_counts().sort_index().rename_axis('SELECTED_MIDX').reset_index(name='N')
birth_history_position_distribution = birth_linkage_audit['MATCHED_BIRTH_HISTORY_POSITION'].value_counts().sort_index().rename_axis('BIRTH_HISTORY_POSITION').reset_index(name='N')
multiple_birth_distribution = birth_linkage_audit['SAME_CMC_BIRTH_COUNT'].value_counts().sort_index().rename_axis('N_CHILDREN_SAME_INDEX_BIRTH_CMC').reset_index(name='N_WOMEN')
candidate_age_summary = birth_linkage_audit['CANDIDATE_AGE_AT_INDEX_BIRTH_YEARS'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).rename('Value').reset_index().rename(columns={'index': 'Statistic'})
step6_qc = pd.DataFrame({'Check': ['Analytic cohort N', 'Missing selected MIDX', 'MIDX with no BIDX match', 'MIDX with multiple BIDX matches', 'Missing matched B3', 'Missing matched BORD', 'Missing matched B0', 'Women with birth later than linked index birth', 'B3/B0 multiple-birth disagreements', 'Prior-live-birth disagreements where V201 <=20', 'Candidate age <10', 'Candidate age >=50'], 'Observed': [len(birth_linkage_audit), n_missing_selected_midx, n_zero_bidx_matches, n_multiple_bidx_matches, n_missing_b3, n_missing_bord, n_missing_b0, n_women_with_later_birth, multiple_birth_disagreement, prior_parity_disagreement_full_history, n_age_below_10, n_age_above_50], 'Expected': [5063, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]})
step6_qc['PASS'] = step6_qc['Observed'] == step6_qc['Expected']
if not step6_qc['PASS'].all():
    raise RuntimeError('One or more Step-6 linkage QC checks failed.')
STEP6_LINKAGE_FILE = DIRS['processed'] / 'STEP06_birth_history_linkage_audit.csv'
STEP6_QC_FILE = DIRS['qc'] / 'STEP06_birth_history_linkage_qc.csv'
STEP6_VARIABLE_FILE = DIRS['raw_reference'] / 'STEP06_birth_history_variable_audit.csv'
STEP6_MIDX_FILE = DIRS['qc'] / 'STEP06_selected_MIDX_distribution.csv'
STEP6_POSITION_FILE = DIRS['qc'] / 'STEP06_matched_birth_history_position_distribution.csv'
STEP6_MULTIPLE_FILE = DIRS['qc'] / 'STEP06_multiple_birth_distribution.csv'
STEP6_BORD_MISMATCH_FILE = DIRS['qc'] / 'STEP06_BORD_minus_one_disagreements.csv'
birth_linkage_audit.to_csv(STEP6_LINKAGE_FILE, index=False)
step6_qc.to_csv(STEP6_QC_FILE, index=False)
linkage_variable_audit.to_csv(STEP6_VARIABLE_FILE, index=False)
midx_distribution.to_csv(STEP6_MIDX_FILE, index=False)
birth_history_position_distribution.to_csv(STEP6_POSITION_FILE, index=False)
multiple_birth_distribution.to_csv(STEP6_MULTIPLE_FILE, index=False)
birth_linkage_audit.loc[birth_linkage_audit['BORD_MINUS_ONE_DISAGREES']].to_csv(STEP6_BORD_MISMATCH_FILE, index=False)
step6_manifest = {'step': 'STEP 6', 'description': 'Exact MIDX to BIDX/B3/BORD/B0 birth-history linkage audit', 'analytic_n': int(len(birth_linkage_audit)), 'MIDX_variables': MIDX_COLS, 'BIDX_variables': BIDX_COLS, 'B3_variables': B3_COLS, 'BORD_variables': BORD_COLS, 'B0_variables': B0_COLS, 'MIDX_without_BIDX_match': n_zero_bidx_matches, 'MIDX_multiple_BIDX_matches': n_multiple_bidx_matches, 'women_with_multiple_index_birth': int(multiple_birth_by_cmc.sum()), 'BORD_minus_one_disagreements': n_bord_minus_one_disagreement, 'prior_parity_consistency_disagreements_V201_le_20': prior_parity_disagreement_full_history, 'predictor_frozen': False, 'new_exclusion_performed': False, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP6_MANIFEST_FILE = DIRS['protocol'] / 'STEP06_birth_history_linkage_manifest.json'
with open(STEP6_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step6_manifest, f, indent=4, ensure_ascii=False, default=str)
logger.info(f'STEP 6 completed successfully: N={len(birth_linkage_audit):,}, MIDX/BIDX mismatches=0, multiple births={int(multiple_birth_by_cmc.sum()):,}.')
print('\n' + '=' * 78)
print('STEP 6 — EXACT MIDX -> BIRTH-HISTORY LINKAGE AUDIT')
print('=' * 78)
print('\nVARIABLE FAMILY AUDIT:')
display(linkage_variable_audit)
print('\nOVERALL LINKAGE QC:')
display(step6_qc)
print('\nSELECTED MIDX DISTRIBUTION:')
display(midx_distribution)
print('\nMATCHED BIRTH-HISTORY POSITION DISTRIBUTION:')
display(birth_history_position_distribution)
print('\nINDEX-PREGNANCY MULTIPLE-BIRTH DISTRIBUTION:')
display(multiple_birth_distribution)
print('\nBORD-1 vs MULTIPLE-BIRTH-AWARE PRIOR-LIVE-BIRTH AUDIT:')
print(f'BORD-1 disagreements : {n_bord_minus_one_disagreement:,}')
if n_bord_minus_one_disagreement > 0:
    display(birth_linkage_audit.loc[birth_linkage_audit['BORD_MINUS_ONE_DISAGREES'], ['CASEID', 'SELECTED_MIDX', 'MATCHED_BIDX', 'MATCHED_BORD', 'MATCHED_B0', 'SAME_CMC_BIRTH_COUNT', 'V201_TOTAL_LIVE_BIRTHS', 'OLD_BORD_MINUS_ONE', 'PRIOR_LIVE_BIRTHS_V201_CANDIDATE']].head(20))
print('\nCANDIDATE MATERNAL AGE AT INDEX BIRTH:')
display(candidate_age_summary)
print('\nKEY COUNTS:')
print(f'Analytic N                         : {len(birth_linkage_audit):,}')
print(f'Missing selected MIDX             : {n_missing_selected_midx:,}')
print(f'No BIDX match                     : {n_zero_bidx_matches:,}')
print(f'Multiple BIDX matches             : {n_multiple_bidx_matches:,}')
print(f'Women with later live birth       : {n_women_with_later_birth:,}')
print(f'Women with multiple index birth   : {int(multiple_birth_by_cmc.sum()):,}')
print(f'B3/B0 multiple-birth disagreement : {multiple_birth_disagreement:,}')
print(f'Prior parity consistency mismatch : {prior_parity_disagreement_full_history:,}')
print(f'BORD-1 shortcut disagreements     : {n_bord_minus_one_disagreement:,}')
print('\nFILES SAVED:')
for path in [STEP6_LINKAGE_FILE, STEP6_QC_FILE, STEP6_VARIABLE_FILE, STEP6_MIDX_FILE, STEP6_POSITION_FILE, STEP6_MULTIPLE_FILE, STEP6_BORD_MISMATCH_FILE, STEP6_MANIFEST_FILE]:
    print(f'- {path}')
print('\nIMPORTANT:')
print('- MIDX was taken from the exact M80=1 maternity slot.')
print('- MIDX was matched by searching BIDX values; no suffix assumption was used.')
print('- B3/BORD/B0 were extracted from that exact matched birth-history entry.')
print('- Multiple births were explicitly audited before defining prior parity.')
print('- Candidate age/parity values are QC only; they are NOT frozen predictors yet.')
print('- No participant was excluded in Step 6.')
print('=' * 78)
print('STATUS: STEP 6 PASS')
print('=' * 78)


Detected family sizes:
MIDX : 6
BIDX : 20
B3   : 20
BORD : 20
B0   : 20


2026-09-16 21:05:52,202 | INFO | STEP 6 completed successfully: N=5,063, MIDX/BIDX mismatches=0, multiple births=53.



STEP 6 — EXACT MIDX -> BIRTH-HISTORY LINKAGE AUDIT

VARIABLE FAMILY AUDIT:


,Family,First_Variable,First_Variable_Label,N_Variables
0,MIDX,MIDX_1,index to birth history,6
1,BIDX,BIDX_01,birth column number,20
2,B3,B3_01,date of birth (cmc),20
3,BORD,BORD_01,birth order number,20
4,B0,B0_01,child is twin,20



OVERALL LINKAGE QC:


,Check,Observed,Expected,PASS
0,Analytic cohort N,5063,5063,True
1,Missing selected MIDX,0,0,True
2,MIDX with no BIDX match,0,0,True
3,MIDX with multiple BIDX matches,0,0,True
4,Missing matched B3,0,0,True
5,Missing matched BORD,0,0,True
6,Missing matched B0,0,0,True
7,Women with birth later than linked index birth,0,0,True
8,B3/B0 multiple-birth disagreements,0,0,True
9,Prior-live-birth disagreements where V201 <=20,0,0,True


... (7 outputs left out)

,CASEID,SELECTED_MIDX,MATCHED_BIDX,MATCHED_BORD,MATCHED_B0,SAME_CMC_BIRTH_COUNT,V201_TOTAL_LIVE_BIRTHS,OLD_BORD_MINUS_ONE,PRIOR_LIVE_BIRTHS_V201_CANDIDATE
78,11 36 2,1,1.0,3.0,2.0,2,3.0,2.0,1.0
86,13 5 4,1,1.0,3.0,2.0,2,3.0,2.0,1.0
139,20 90 3,1,1.0,3.0,2.0,2,3.0,2.0,1.0
309,43 110 1,1,1.0,4.0,2.0,2,4.0,3.0,2.0
313,44 18 2,1,1.0,2.0,2.0,2,2.0,1.0,0.0
586,79 8 3,1,1.0,2.0,2.0,2,2.0,1.0,0.0
615,82 15 6,1,1.0,2.0,2.0,2,2.0,1.0,0.0
682,88 19 2,1,1.0,3.0,2.0,2,3.0,2.0,1.0
787,99 119 8,1,1.0,2.0,2.0,2,2.0,1.0,0.0
793,100 113 4,1,1.0,2.0,2.0,2,2.0,1.0,0.0



CANDIDATE MATERNAL AGE AT INDEX BIRTH:


,Statistic,Value
0,count,5063.000000
1,mean,24.837679
2,std,5.635341
3,min,13.583333
4,1%,15.583333
5,5%,17.083333
6,25%,20.333333
7,50%,24.083333
8,75%,28.583333
9,95%,35.000000



KEY COUNTS:
Analytic N                         : 5,063
Missing selected MIDX             : 0
No BIDX match                     : 0
Multiple BIDX matches             : 0
Women with later live birth       : 0
Women with multiple index birth   : 53
B3/B0 multiple-birth disagreement : 0
Prior parity consistency mismatch : 0
BORD-1 shortcut disagreements     : 53

FILES SAVED:
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP06_birth_history_linkage_audit.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP06_birth_history_linkage_qc.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\00_raw_reference\STEP06_birth_history_variable_audit.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP06_selected_MIDX_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP06_matched_birth_history_position_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_control\STEP06_multiple_birth_distribution.csv
- E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\02_quality_con

In [7]:
# Step 7: Index-birth age, prior live births, and clean core dataset.

import json
import hashlib
import datetime as dt
import numpy as np
import pandas as pd
STEP5_FINAL_COHORT_FILE = DIRS['processed'] / 'STEP05_analytic_cohort_anc_outcome.csv'
STEP6_LINKAGE_FILE = DIRS['processed'] / 'STEP06_birth_history_linkage_audit.csv'
if not STEP5_FINAL_COHORT_FILE.exists():
    raise FileNotFoundError(f'Step-5 file not found:\n{STEP5_FINAL_COHORT_FILE}')
if not STEP6_LINKAGE_FILE.exists():
    raise FileNotFoundError(f'Step-6 file not found:\n{STEP6_LINKAGE_FILE}')
EXPECTED_STEP5_SHA256 = '9f2aacec3b046316d5bcd23203c90c8fcad9b67e21dfa975c6680a6588eac0c6'
observed_step5_sha256 = calculate_sha256(STEP5_FINAL_COHORT_FILE)
if observed_step5_sha256 != EXPECTED_STEP5_SHA256:
    raise RuntimeError('Step-5 analytic cohort hash mismatch.\nThe locked outcome file has changed.\nSTOP.')
step5 = pd.read_csv(STEP5_FINAL_COHORT_FILE)
step6 = pd.read_csv(STEP6_LINKAGE_FILE)
if len(step5) != 5063:
    raise RuntimeError(f'Unexpected Step-5 N: {len(step5):,}')
if len(step6) != 5063:
    raise RuntimeError(f'Unexpected Step-6 N: {len(step6):,}')
for data_name, data in {'STEP5': step5, 'STEP6': step6}.items():
    if data['_ROW_ID'].isna().any():
        raise RuntimeError(f'{data_name}: missing _ROW_ID.')
    if not data['_ROW_ID'].is_unique:
        raise RuntimeError(f'{data_name}: duplicate _ROW_ID.')
    if data['CASEID'].isna().any():
        raise RuntimeError(f'{data_name}: missing CASEID.')
    if not data['CASEID'].is_unique:
        raise RuntimeError(f'{data_name}: duplicate CASEID.')
step5_key = step5[['_ROW_ID', 'CASEID']].copy()
step6_key = step6[['_ROW_ID', 'CASEID']].copy()
key_check = step5_key.merge(step6_key, on='_ROW_ID', how='outer', suffixes=('_STEP5', '_STEP6'), indicator=True, validate='one_to_one')
if not (key_check['_merge'] == 'both').all():
    raise RuntimeError('Step-5 and Step-6 _ROW_ID sets differ.')
caseid_mismatch = key_check['CASEID_STEP5'].astype(str) != key_check['CASEID_STEP6'].astype(str)
if caseid_mismatch.any():
    raise RuntimeError('CASEID mismatch between Step 5 and Step 6.')
linkage_columns = ['_ROW_ID', 'SELECTED_MIDX', 'MATCHED_BIDX', 'INDEX_BIRTH_CMC', 'MATCHED_BORD', 'MATCHED_B0', 'SAME_CMC_BIRTH_COUNT', 'MULTIPLE_BIRTH', 'V201_TOTAL_LIVE_BIRTHS', 'PRIOR_LIVE_BIRTHS_V201_CANDIDATE', 'PRIOR_LIVE_BIRTHS_B3_COUNT', 'MATERNAL_DOB_CMC_V011', 'CURRENT_AGE_V012', 'CANDIDATE_AGE_AT_INDEX_BIRTH_YEARS', 'N_LATER_BIRTH_ENTRIES']
merged = step5.merge(step6[linkage_columns], on='_ROW_ID', how='left', validate='one_to_one')
if len(merged) != 5063:
    raise RuntimeError('Unexpected row count after Step5-Step6 merge.')
if 'raw' not in globals():
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'V008' not in raw.columns:
    raise RuntimeError('V008 interview-date CMC not found.')
raw_timing = raw[['_ROW_ID', 'V008']].copy()
raw_timing['INTERVIEW_CMC'] = pd.to_numeric(raw_timing['V008'], errors='coerce')
raw_timing = raw_timing[['_ROW_ID', 'INTERVIEW_CMC']]
merged = merged.merge(raw_timing, on='_ROW_ID', how='left', validate='one_to_one')
merged['MATERNAL_AGE_INDEX_BIRTH_YEARS'] = (pd.to_numeric(merged['INDEX_BIRTH_CMC'], errors='coerce') - pd.to_numeric(merged['MATERNAL_DOB_CMC_V011'], errors='coerce')) / 12.0
age_difference = np.abs(merged['MATERNAL_AGE_INDEX_BIRTH_YEARS'].to_numpy(dtype=float) - merged['CANDIDATE_AGE_AT_INDEX_BIRTH_YEARS'].to_numpy(dtype=float))
max_age_difference = float(np.nanmax(age_difference))
if max_age_difference > 1e-12:
    raise RuntimeError('Step-7 maternal-age derivation differs from the Step-6 candidate value.')
merged['MATERNAL_AGE_INDEX_BIRTH_COMPLETED'] = np.floor(merged['MATERNAL_AGE_INDEX_BIRTH_YEARS']).astype(np.int64)
age_exact = merged['MATERNAL_AGE_INDEX_BIRTH_YEARS']
if age_exact.isna().any():
    raise RuntimeError('Missing maternal age at index birth.')
if (age_exact < 10).any():
    raise RuntimeError('Implausible maternal age <10 detected.')
if (age_exact >= 50).any():
    raise RuntimeError('Implausible maternal age >=50 detected.')
merged['MATERNAL_AGE_INDEX_BIRTH_GROUP'] = np.select([age_exact < 20, (age_exact >= 20) & (age_exact < 25), (age_exact >= 25) & (age_exact < 30), (age_exact >= 30) & (age_exact < 35), (age_exact >= 35) & (age_exact < 40), age_exact >= 40], ['<20', '20-24', '25-29', '30-34', '35-39', '40+'], default='__INVALID__')
if (merged['MATERNAL_AGE_INDEX_BIRTH_GROUP'] == '__INVALID__').any():
    raise RuntimeError('At least one maternal age could not be grouped.')
v201 = pd.to_numeric(merged['V201_TOTAL_LIVE_BIRTHS'], errors='coerce')
same_index_birth_count = pd.to_numeric(merged['SAME_CMC_BIRTH_COUNT'], errors='coerce')
merged['PRIOR_LIVE_BIRTHS'] = v201 - same_index_birth_count
if merged['PRIOR_LIVE_BIRTHS'].isna().any():
    raise RuntimeError('Missing prior-live-birth count.')
if (merged['PRIOR_LIVE_BIRTHS'] < 0).any():
    raise RuntimeError('Negative prior-live-birth count detected.')
prior_values = merged['PRIOR_LIVE_BIRTHS'].to_numpy(dtype=float)
if not np.allclose(prior_values, np.round(prior_values), rtol=0, atol=1e-10):
    raise RuntimeError('Prior-live-birth count contains non-integer values.')
merged['PRIOR_LIVE_BIRTHS'] = merged['PRIOR_LIVE_BIRTHS'].round().astype(np.int64)
prior_b3_count = pd.to_numeric(merged['PRIOR_LIVE_BIRTHS_B3_COUNT'], errors='coerce').astype(np.int64)
prior_parity_mismatch = merged['PRIOR_LIVE_BIRTHS'] != prior_b3_count
n_prior_parity_mismatch = int(prior_parity_mismatch.sum())
if n_prior_parity_mismatch != 0:
    raise RuntimeError(f'{n_prior_parity_mismatch} prior-parity cross-check mismatches detected.')
merged['PRIOR_PARITY_GROUP'] = np.select([merged['PRIOR_LIVE_BIRTHS'].eq(0), merged['PRIOR_LIVE_BIRTHS'].eq(1), merged['PRIOR_LIVE_BIRTHS'].eq(2), merged['PRIOR_LIVE_BIRTHS'].ge(3)], ['0', '1', '2', '3+'], default='__INVALID__')
if (merged['PRIOR_PARITY_GROUP'] == '__INVALID__').any():
    raise RuntimeError('Prior-parity grouping failed.')
merged['INDEX_PREGNANCY_MULTIPLE_BIRTH'] = np.where(merged['SAME_CMC_BIRTH_COUNT'].astype(int) > 1, 'Multiple', 'Singleton')
merged['MONTHS_INDEX_BIRTH_TO_INTERVIEW'] = pd.to_numeric(merged['INTERVIEW_CMC'], errors='coerce') - pd.to_numeric(merged['INDEX_BIRTH_CMC'], errors='coerce')
if merged['INTERVIEW_CMC'].isna().any():
    raise RuntimeError('Missing interview CMC.')
if (merged['MONTHS_INDEX_BIRTH_TO_INTERVIEW'] < 0).any():
    raise RuntimeError('At least one index birth occurs after interview date.')
if (pd.to_numeric(merged['N_LATER_BIRTH_ENTRIES'], errors='coerce') != 0).any():
    raise RuntimeError('A live birth later than the index birth was detected.')
integer_columns = ['_ROW_ID', 'INDEX_MATERNITY_SLOT', 'ANC_VISITS', 'ANC_CLASS', 'PREG_INTENTION_RAW', 'PSU', 'STRATUM', 'SELECTED_MIDX', 'MATCHED_BIDX', 'INDEX_BIRTH_CMC', 'SAME_CMC_BIRTH_COUNT', 'V201_TOTAL_LIVE_BIRTHS', 'PRIOR_LIVE_BIRTHS', 'MATERNAL_DOB_CMC_V011', 'MATERNAL_AGE_INDEX_BIRTH_COMPLETED', 'INTERVIEW_CMC', 'MONTHS_INDEX_BIRTH_TO_INTERVIEW']
for column in integer_columns:
    numeric_values = pd.to_numeric(merged[column], errors='coerce')
    if numeric_values.isna().any():
        raise RuntimeError(f'{column} contains missing values.')
    if not np.allclose(numeric_values.to_numpy(dtype=float), np.round(numeric_values.to_numpy(dtype=float)), rtol=0, atol=1e-10):
        raise RuntimeError(f'{column} contains non-integer values.')
    merged[column] = numeric_values.round().astype(np.int64)
CORE_COLUMNS = ['_ROW_ID', 'CASEID', 'INDEX_MATERNITY_SLOT', 'SELECTED_MIDX', 'MATCHED_BIDX', 'INDEX_BIRTH_CMC', 'ANC_VISITS', 'ANC_CLASS', 'ANC_CLASS_LABEL', 'PREG_INTENTION_RAW', 'PREG_INTENTION_VALUE_LABEL', 'MATERNAL_DOB_CMC_V011', 'MATERNAL_AGE_INDEX_BIRTH_YEARS', 'MATERNAL_AGE_INDEX_BIRTH_COMPLETED', 'MATERNAL_AGE_INDEX_BIRTH_GROUP', 'V201_TOTAL_LIVE_BIRTHS', 'SAME_CMC_BIRTH_COUNT', 'INDEX_PREGNANCY_MULTIPLE_BIRTH', 'PRIOR_LIVE_BIRTHS', 'PRIOR_PARITY_GROUP', 'SURVEY_WEIGHT', 'PSU', 'STRATUM', 'INTERVIEW_CMC', 'MONTHS_INDEX_BIRTH_TO_INTERVIEW']
core_clean = merged[CORE_COLUMNS].copy().reset_index(drop=True)
string_columns = core_clean.select_dtypes(include=['object', 'string']).columns.tolist()
for column in string_columns:
    core_clean[column] = core_clean[column].astype(str).str.strip()
missing_by_column = core_clean.isna().sum().rename('N_Missing').reset_index().rename(columns={'index': 'Variable'})
total_missing = int(missing_by_column['N_Missing'].sum())
if total_missing != 0:
    raise RuntimeError(f'Clean core dataset contains {total_missing:,} missing cells.')
numeric_columns = core_clean.select_dtypes(include=[np.number]).columns.tolist()
nonfinite_numeric = {}
for column in numeric_columns:
    values = core_clean[column].to_numpy(dtype=float)
    n_bad = int((~np.isfinite(values)).sum())
    if n_bad > 0:
        nonfinite_numeric[column] = n_bad
if nonfinite_numeric:
    raise RuntimeError(f'Non-finite numeric values detected:\n{nonfinite_numeric}')
FORBIDDEN_EXACT_TEXT = {'', 'nan', 'none', 'null', 'na', 'n/a', "don't know", 'dont know', 'dk', 'missing', '__invalid__'}
forbidden_text_records = []
for column in string_columns:
    normalized = core_clean[column].astype(str).str.strip().str.lower()
    bad_mask = normalized.isin(FORBIDDEN_EXACT_TEXT)
    if bad_mask.any():
        counts = normalized[bad_mask].value_counts()
        for value, count in counts.items():
            forbidden_text_records.append({'Variable': column, 'Forbidden_Value': value, 'N': int(count)})
forbidden_text_audit = pd.DataFrame(forbidden_text_records, columns=['Variable', 'Forbidden_Value', 'N'])
if not forbidden_text_audit.empty:
    raise RuntimeError(f'Forbidden blank/missing/special text remains in clean core dataset:\n{forbidden_text_audit}')
observed_outcome_counts = core_clean['ANC_CLASS'].value_counts().sort_index().to_dict()
EXPECTED_OUTCOME_COUNTS = {0: 403, 1: 4407, 2: 253}
if observed_outcome_counts != EXPECTED_OUTCOME_COUNTS:
    raise RuntimeError('Outcome counts changed unexpectedly.')
if core_clean['PSU'].nunique() != 672:
    raise RuntimeError('Final PSU count changed unexpectedly.')
if core_clean['STRATUM'].nunique() != 16:
    raise RuntimeError('Final stratum count changed unexpectedly.')
age_group_distribution = core_clean.groupby('MATERNAL_AGE_INDEX_BIRTH_GROUP', observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index()
age_group_order = {'<20': 0, '20-24': 1, '25-29': 2, '30-34': 3, '35-39': 4, '40+': 5}
age_group_distribution['_ORDER'] = age_group_distribution['MATERNAL_AGE_INDEX_BIRTH_GROUP'].map(age_group_order)
age_group_distribution = age_group_distribution.sort_values('_ORDER').drop(columns='_ORDER').reset_index(drop=True)
age_group_distribution['Percent'] = age_group_distribution['N'] / len(core_clean) * 100
age_group_distribution['Weighted_Percent'] = age_group_distribution['Weighted_N'] / age_group_distribution['Weighted_N'].sum() * 100
parity_distribution = core_clean.groupby('PRIOR_PARITY_GROUP', observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index()
parity_order = {'0': 0, '1': 1, '2': 2, '3+': 3}
parity_distribution['_ORDER'] = parity_distribution['PRIOR_PARITY_GROUP'].map(parity_order)
parity_distribution = parity_distribution.sort_values('_ORDER').drop(columns='_ORDER').reset_index(drop=True)
parity_distribution['Percent'] = parity_distribution['N'] / len(core_clean) * 100
parity_distribution['Weighted_Percent'] = parity_distribution['Weighted_N'] / parity_distribution['Weighted_N'].sum() * 100
multiple_birth_distribution_final = core_clean['INDEX_PREGNANCY_MULTIPLE_BIRTH'].value_counts().rename_axis('Index_Pregnancy').reset_index(name='N')
age_summary = core_clean['MATERNAL_AGE_INDEX_BIRTH_YEARS'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).rename('Value').reset_index().rename(columns={'index': 'Statistic'})
complete_data_audit = pd.DataFrame({'Check': ['Rows', 'Columns', 'Missing / NaN cells', 'Non-finite numeric cells', 'Forbidden blank/special text cells', 'Duplicate _ROW_ID', 'Duplicate CASEID', 'Prior-parity mismatches', 'Outcome class 0', 'Outcome class 1', 'Outcome class 2', 'Unique PSU', 'Unique strata'], 'Observed': [len(core_clean), core_clean.shape[1], total_missing, 0, 0, int(core_clean['_ROW_ID'].duplicated().sum()), int(core_clean['CASEID'].duplicated().sum()), n_prior_parity_mismatch, observed_outcome_counts[0], observed_outcome_counts[1], observed_outcome_counts[2], core_clean['PSU'].nunique(), core_clean['STRATUM'].nunique()], 'Expected': [5063, len(CORE_COLUMNS), 0, 0, 0, 0, 0, 0, 403, 4407, 253, 672, 16]})
complete_data_audit['PASS'] = complete_data_audit['Observed'] == complete_data_audit['Expected']
if not complete_data_audit['PASS'].all():
    raise RuntimeError('Step-7 clean-core audit failed.')
STEP7_CLEAN_CORE_FILE = DIRS['processed'] / 'STEP07_clean_core_analytic_dataset.csv'
STEP7_MISSING_AUDIT_FILE = DIRS['qc'] / 'STEP07_missingness_by_variable.csv'
STEP7_COMPLETE_QC_FILE = DIRS['qc'] / 'STEP07_clean_core_qc.csv'
STEP7_AGE_DIST_FILE = DIRS['qc'] / 'STEP07_maternal_age_group_distribution.csv'
STEP7_PARITY_DIST_FILE = DIRS['qc'] / 'STEP07_prior_parity_distribution.csv'
STEP7_MULTIPLE_FILE = DIRS['qc'] / 'STEP07_multiple_birth_distribution.csv'
core_clean.to_csv(STEP7_CLEAN_CORE_FILE, index=False)
missing_by_column.to_csv(STEP7_MISSING_AUDIT_FILE, index=False)
complete_data_audit.to_csv(STEP7_COMPLETE_QC_FILE, index=False)
age_group_distribution.to_csv(STEP7_AGE_DIST_FILE, index=False)
parity_distribution.to_csv(STEP7_PARITY_DIST_FILE, index=False)
multiple_birth_distribution_final.to_csv(STEP7_MULTIPLE_FILE, index=False)
STEP7_CLEAN_CORE_SHA256 = calculate_sha256(STEP7_CLEAN_CORE_FILE)
step7_manifest = {'step': 'STEP 7', 'description': 'Frozen index-birth age and multiple-birth-aware prior parity', 'analytic_n': int(len(core_clean)), 'maternal_age_definition': '(INDEX_BIRTH_CMC - V011) / 12', 'prior_live_births_definition': 'V201 - number of live-born children in index pregnancy', 'multiple_birth_handled': True, 'continuous_age_preserved': True, 'descriptive_age_groups': ['<20', '20-24', '25-29', '30-34', '35-39', '40+'], 'prior_parity_groups': ['0', '1', '2', '3+'], 'missing_cells': total_missing, 'forbidden_text_cells': 0, 'nonfinite_numeric_cells': 0, 'outcome_counts': {str(k): int(v) for k, v in observed_outcome_counts.items()}, 'unique_psu': int(core_clean['PSU'].nunique()), 'unique_strata': int(core_clean['STRATUM'].nunique()), 'clean_core_sha256': STEP7_CLEAN_CORE_SHA256, 'model_fitting_performed': False, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP7_MANIFEST_FILE = DIRS['protocol'] / 'STEP07_clean_core_manifest.json'
with open(STEP7_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step7_manifest, f, indent=4, ensure_ascii=False, default=str)
logger.info(f'STEP 7 completed successfully: N={len(core_clean):,}, missing={total_missing}, parity mismatches={n_prior_parity_mismatch}.')
print('\n' + '=' * 78)
print('STEP 7 — INDEX-BIRTH AGE + PRIOR PARITY + CLEAN CORE DATASET')
print('=' * 78)
print('\nCOMPLETE-DATA HARD QC:')
display(complete_data_audit)
print('\nMATERNAL AGE AT INDEX BIRTH SUMMARY:')
display(age_summary)
print('\nMATERNAL AGE GROUP DISTRIBUTION:')
display(age_group_distribution)
print('\nPRIOR PARITY DISTRIBUTION:')
display(parity_distribution)
print('\nINDEX-PREGNANCY MULTIPLE-BIRTH DISTRIBUTION:')
display(multiple_birth_distribution_final)
print('\nZERO-MISSING CHECK:')
print(f'Total missing / NaN cells        : {total_missing:,}')
print(f'Non-finite numeric cells         : 0')
print(f'Forbidden blank/special text     : 0')
print(f'Prior-parity cross-check mismatch: {n_prior_parity_mismatch:,}')
print(f'Maximum age derivation difference: {max_age_difference:.12f}')
print('\nCLEAN CORE DATASET:')
print(STEP7_CLEAN_CORE_FILE)
print('\nCLEAN CORE SHA-256:')
print(STEP7_CLEAN_CORE_SHA256)
print('\nIMPORTANT:')
print('- Maternal age now refers to age at the INDEX BIRTH, not interview age.')
print('- Prior parity is multiple-birth-aware; BORD-1 is NOT used.')
print('- Continuous age is preserved; age groups are also retained.')
print("- No NaN/missing/blank/don't-know/non-finite value is allowed in this clean core file.")
print('- No record was removed in Step 7.')
print('- Outcome N and survey-design structure remain locked.')
print('=' * 78)
print('STATUS: STEP 7 PASS')
print('=' * 78)


2026-09-16 21:05:55,497 | INFO | STEP 7 completed successfully: N=5,063, missing=0, parity mismatches=0.



STEP 7 — INDEX-BIRTH AGE + PRIOR PARITY + CLEAN CORE DATASET

COMPLETE-DATA HARD QC:


,Check,Observed,Expected,PASS
0,Rows,5063,5063,True
1,Columns,25,25,True
2,Missing / NaN cells,0,0,True
3,Non-finite numeric cells,0,0,True
4,Forbidden blank/special text cells,0,0,True
5,Duplicate _ROW_ID,0,0,True
6,Duplicate CASEID,0,0,True
7,Prior-parity mismatches,0,0,True
8,Outcome class 0,403,403,True
9,Outcome class 1,4407,4407,True



MATERNAL AGE AT INDEX BIRTH SUMMARY:


,Statistic,Value
0,count,5063.000000
1,mean,24.837679
2,std,5.635341
3,min,13.583333
4,1%,15.583333
5,5%,17.083333
6,25%,20.333333
7,50%,24.083333
8,75%,28.583333
9,95%,35.000000



MATERNAL AGE GROUP DISTRIBUTION:


... (2 outputs left out)

,PRIOR_PARITY_GROUP,N,Weighted_N,Percent,Weighted_Percent
0,0,1909,1952.852700,37.704918,38.088700
1,1,1768,1753.665543,34.920008,34.203727
2,2,915,943.234461,18.072289,18.396971
3,3+,471,477.365547,9.302785,9.310601



INDEX-PREGNANCY MULTIPLE-BIRTH DISTRIBUTION:


,Index_Pregnancy,N
0,Singleton,5010
1,Multiple,53



ZERO-MISSING CHECK:
Total missing / NaN cells        : 0
Non-finite numeric cells         : 0
Forbidden blank/special text     : 0
Prior-parity cross-check mismatch: 0
Maximum age derivation difference: 0.000000000000

CLEAN CORE DATASET:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP07_clean_core_analytic_dataset.csv

CLEAN CORE SHA-256:
915f14ee6201bb839b183f252a656f78ac49282bec9bb9d7d5602b023fa1ce6c

IMPORTANT:
- Maternal age now refers to age at the INDEX BIRTH, not interview age.
- Prior parity is multiple-birth-aware; BORD-1 is NOT used.
- Continuous age is preserved; age groups are also retained.
- No NaN/missing/blank/don't-know/non-finite value is allowed in this clean core file.
- No record was removed in Step 7.
- Outcome N and survey-design structure remain locked.
STATUS: STEP 7 PASS


In [8]:
# Step 8: Demographic and socioeconomic predictors.

import json
import datetime as dt
import numpy as np
import pandas as pd
STEP7_CLEAN_CORE_FILE = DIRS['processed'] / 'STEP07_clean_core_analytic_dataset.csv'
EXPECTED_STEP7_SHA256 = '915f14ee6201bb839b183f252a656f78ac49282bec9bb9d7d5602b023fa1ce6c'
if not STEP7_CLEAN_CORE_FILE.exists():
    raise FileNotFoundError(f'Step-7 clean core file not found:\n{STEP7_CLEAN_CORE_FILE}')
observed_step7_sha256 = calculate_sha256(STEP7_CLEAN_CORE_FILE)
if observed_step7_sha256 != EXPECTED_STEP7_SHA256:
    raise RuntimeError('STEP-7 HASH MISMATCH.\nThe locked clean-core dataset has changed.\nSTOP.')
core_clean = pd.read_csv(STEP7_CLEAN_CORE_FILE)
if len(core_clean) != 5063:
    raise RuntimeError(f'Unexpected Step-7 N: {len(core_clean):,}')
if 'raw' not in globals():
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'stata_meta' not in globals():
    _, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), metadataonly=True)
raw_variable_labels = getattr(stata_meta, 'column_names_to_labels', {}) or {}
variable_label_map = {str(variable).strip().upper(): label for variable, label in raw_variable_labels.items()}
raw_variable_value_labels = getattr(stata_meta, 'variable_value_labels', {}) or {}
VARIABLE_VALUE_LABELS = {str(variable).strip().upper(): dict(mapping) for variable, mapping in raw_variable_value_labels.items()}
STEP8_SOURCE_VARIABLES = ['V106', 'V190', 'V025', 'V024', 'V157', 'V158', 'V159', 'V714', 'V130']
missing_sources = [variable for variable in STEP8_SOURCE_VARIABLES if variable not in raw.columns]
if missing_sources:
    raise RuntimeError(f'Missing Step-8 source variables:\n{missing_sources}')
raw_step8 = raw[['_ROW_ID'] + STEP8_SOURCE_VARIABLES].copy()
step8 = core_clean.merge(raw_step8, on='_ROW_ID', how='left', validate='one_to_one')
if len(step8) != 5063:
    raise RuntimeError('Unexpected row count after Step-8 raw merge.')

def normalize_text(value):
    if value is None:
        return ''
    return str(value).strip().lower().replace('’', "'").replace('`', "'").replace('–', '-').replace('—', '-')

def get_value_label(variable, raw_value):
    if pd.isna(raw_value):
        return None
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    candidates = [raw_value]
    try:
        candidates.append(int(raw_value))
    except Exception:
        pass
    try:
        candidates.append(float(raw_value))
    except Exception:
        pass
    for candidate in candidates:
        if candidate in mapping:
            return mapping[candidate]
    return None
SPECIAL_TEXT_PATTERNS = ["don't know", 'dont know', 'unknown', 'missing', 'not applicable', 'not asked', 'inconsistent']

def is_special_source_label(label):
    text = normalize_text(label)
    if text == '':
        return True
    return any((pattern in text for pattern in SPECIAL_TEXT_PATTERNS))
source_label_audit_rows = []
for variable in STEP8_SOURCE_VARIABLES:
    source_series = step8[variable]
    observed_values = source_series.value_counts(dropna=False).rename_axis('Raw_Value').reset_index(name='N')
    for _, row in observed_values.iterrows():
        raw_value = row['Raw_Value']
        value_label = get_value_label(variable, raw_value)
        source_label_audit_rows.append({'Variable': variable, 'Variable_Label': variable_label_map.get(variable), 'Raw_Value': raw_value, 'Value_Label': value_label, 'N': int(row['N']), 'Is_Missing_Raw': bool(pd.isna(raw_value)), 'Is_Special_Label': bool(is_special_source_label(value_label))})
source_label_audit = pd.DataFrame(source_label_audit_rows)
source_missing_counts = step8[STEP8_SOURCE_VARIABLES].isna().sum()
if (source_missing_counts > 0).any():
    bad = source_missing_counts[source_missing_counts > 0]
    raise RuntimeError(f'Missing raw predictor values detected:\n{bad}')
unlabelled_observed = source_label_audit[source_label_audit['Value_Label'].isna()]
if not unlabelled_observed.empty:
    raise RuntimeError(f'Observed raw codes without Stata value labels detected:\n{unlabelled_observed}')
special_observed = source_label_audit[source_label_audit['Is_Special_Label']]
if not special_observed.empty:
    raise RuntimeError(f'Special/unknown/not-applicable source values detected in Step-8 predictors.\nThese will NOT be silently imputed.\n\n{special_observed}')
for variable in STEP8_SOURCE_VARIABLES:
    label_column = variable + '_LABEL'
    step8[label_column] = [get_value_label(variable, raw_value) for raw_value in step8[variable]]
education_normalized = step8['V106_LABEL'].map(normalize_text)
EDUCATION_MAP = {'no education': 'No education', 'primary': 'Primary', 'secondary': 'Secondary', 'higher': 'Higher'}
unexpected_education = sorted(set(education_normalized) - set(EDUCATION_MAP.keys()))
if unexpected_education:
    raise RuntimeError(f'Unexpected V106 education label(s):\n{unexpected_education}')
step8['maternal_education'] = education_normalized.map(EDUCATION_MAP)
wealth_normalized = step8['V190_LABEL'].map(normalize_text)
WEALTH_MAP = {'poorest': 'Poorest', 'poorer': 'Poorer', 'middle': 'Middle', 'richer': 'Richer', 'richest': 'Richest'}
unexpected_wealth = sorted(set(wealth_normalized) - set(WEALTH_MAP.keys()))
if unexpected_wealth:
    raise RuntimeError(f'Unexpected V190 wealth label(s):\n{unexpected_wealth}')
step8['wealth_quintile'] = wealth_normalized.map(WEALTH_MAP)
residence_normalized = step8['V025_LABEL'].map(normalize_text)
RESIDENCE_MAP = {'urban': 'Urban', 'rural': 'Rural'}
unexpected_residence = sorted(set(residence_normalized) - set(RESIDENCE_MAP.keys()))
if unexpected_residence:
    raise RuntimeError(f'Unexpected V025 residence label(s):\n{unexpected_residence}')
step8['residence'] = residence_normalized.map(RESIDENCE_MAP)
division_normalized = step8['V024_LABEL'].map(normalize_text)
EXPECTED_DIVISIONS = {'barishal': 'Barishal', 'chattogram': 'Chattogram', 'dhaka': 'Dhaka', 'khulna': 'Khulna', 'mymensingh': 'Mymensingh', 'rajshahi': 'Rajshahi', 'rangpur': 'Rangpur', 'sylhet': 'Sylhet'}
unexpected_division = sorted(set(division_normalized) - set(EXPECTED_DIVISIONS.keys()))
if unexpected_division:
    raise RuntimeError(f'Unexpected V024 division label(s):\n{unexpected_division}')
step8['division'] = division_normalized.map(EXPECTED_DIVISIONS)

def media_frequency_to_weekly(label):
    text = normalize_text(label)
    if 'not at all' in text or 'less than once a week' in text:
        return 0
    if 'at least once a week' in text or 'almost every day' in text or text == 'every day' or ('daily' in text):
        return 1
    raise ValueError(f'Unrecognized media-frequency label: {label}')
media_source_columns = ['V157_LABEL', 'V158_LABEL', 'V159_LABEL']
media_binary_columns = []
for source_column in media_source_columns:
    binary_column = source_column.replace('_LABEL', '_WEEKLY')
    try:
        step8[binary_column] = step8[source_column].map(media_frequency_to_weekly).astype(np.int8)
    except ValueError as exc:
        raise RuntimeError(f'Media recoding failed for {source_column}.\n{exc}') from exc
    media_binary_columns.append(binary_column)
step8['weekly_media_exposure'] = np.where(step8[media_binary_columns].max(axis=1).eq(1), 'Yes', 'No')
working_normalized = step8['V714_LABEL'].map(normalize_text)
WORKING_MAP = {'no': 'No', 'yes': 'Yes'}
unexpected_working = sorted(set(working_normalized) - set(WORKING_MAP.keys()))
if unexpected_working:
    raise RuntimeError(f'Unexpected V714 working-status label(s):\n{unexpected_working}')
step8['working_status'] = working_normalized.map(WORKING_MAP)
religion_normalized = step8['V130_LABEL'].map(normalize_text)
step8['religion_final'] = np.where(religion_normalized.str.contains('islam', regex=False), 'Islam', 'Non-Islam')
NEW_PREDICTORS = ['maternal_education', 'wealth_quintile', 'residence', 'division', 'weekly_media_exposure', 'working_status', 'religion_final']
predictor_distribution_rows = []
for predictor in NEW_PREDICTORS:
    temp = step8.groupby(predictor, dropna=False, observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index()
    temp['Predictor'] = predictor
    temp = temp.rename(columns={predictor: 'Category'})
    temp['Percent'] = temp['N'] / len(step8) * 100
    temp['Weighted_Percent'] = temp['Weighted_N'] / temp['Weighted_N'].sum() * 100
    predictor_distribution_rows.append(temp[['Predictor', 'Category', 'N', 'Weighted_N', 'Percent', 'Weighted_Percent']])
step8_predictor_distributions = pd.concat(predictor_distribution_rows, ignore_index=True)
step8_processed = core_clean.copy()
for predictor in NEW_PREDICTORS:
    step8_processed[predictor] = step8[predictor].to_numpy()
EXPECTED_STEP8_COLUMNS = 25 + len(NEW_PREDICTORS)
if step8_processed.shape[1] != EXPECTED_STEP8_COLUMNS:
    raise RuntimeError(f'Unexpected Step-8 processed column count.\nExpected : {EXPECTED_STEP8_COLUMNS}\nObserved : {step8_processed.shape[1]}')
string_columns = step8_processed.select_dtypes(include=['object', 'string']).columns.tolist()
for column in string_columns:
    step8_processed[column] = step8_processed[column].astype(str).str.strip()
missing_by_variable = step8_processed.isna().sum().rename('N_Missing').reset_index().rename(columns={'index': 'Variable'})
total_missing = int(missing_by_variable['N_Missing'].sum())
if total_missing != 0:
    raise RuntimeError(f'Step-8 processed dataset contains {total_missing:,} missing cells.')
nonfinite_records = []
numeric_columns = step8_processed.select_dtypes(include=[np.number]).columns.tolist()
for column in numeric_columns:
    values = step8_processed[column].to_numpy(dtype=float)
    n_nonfinite = int((~np.isfinite(values)).sum())
    if n_nonfinite > 0:
        nonfinite_records.append({'Variable': column, 'N_Nonfinite': n_nonfinite})
nonfinite_audit = pd.DataFrame(nonfinite_records, columns=['Variable', 'N_Nonfinite'])
if not nonfinite_audit.empty:
    raise RuntimeError(f'Non-finite numeric values remain:\n{nonfinite_audit}')
FORBIDDEN_TEXT = {'', 'nan', 'none', 'null', 'na', 'n/a', "don't know", 'dont know', 'dk', 'unknown', 'missing', '__invalid__'}
forbidden_records = []
for column in string_columns:
    normalized = step8_processed[column].astype(str).str.strip().str.lower()
    bad = normalized.isin(FORBIDDEN_TEXT)
    if bad.any():
        counts = normalized[bad].value_counts()
        for value, count in counts.items():
            forbidden_records.append({'Variable': column, 'Forbidden_Value': value, 'N': int(count)})
forbidden_text_audit = pd.DataFrame(forbidden_records, columns=['Variable', 'Forbidden_Value', 'N'])
if not forbidden_text_audit.empty:
    raise RuntimeError(f'Forbidden text remains in Step-8 processed dataset:\n{forbidden_text_audit}')
outcome_counts = step8_processed['ANC_CLASS'].value_counts().sort_index().to_dict()
EXPECTED_OUTCOME_COUNTS = {0: 403, 1: 4407, 2: 253}
if outcome_counts != EXPECTED_OUTCOME_COUNTS:
    raise RuntimeError('ANC outcome distribution changed.')
if step8_processed['PSU'].nunique() != 672:
    raise RuntimeError('PSU structure changed.')
if step8_processed['STRATUM'].nunique() != 16:
    raise RuntimeError('Stratum structure changed.')
EXPECTED_CATEGORY_COUNTS = {'maternal_education': 4, 'wealth_quintile': 5, 'residence': 2, 'division': 8, 'weekly_media_exposure': 2, 'working_status': 2, 'religion_final': 2}
category_qc_rows = []
for predictor, expected_n in EXPECTED_CATEGORY_COUNTS.items():
    observed_n = int(step8_processed[predictor].nunique(dropna=False))
    category_qc_rows.append({'Predictor': predictor, 'Observed_Categories': observed_n, 'Expected_Categories': expected_n, 'PASS': observed_n == expected_n})
category_qc = pd.DataFrame(category_qc_rows)
if not category_qc['PASS'].all():
    raise RuntimeError('Unexpected Step-8 predictor category count.')
step8_qc = pd.DataFrame({'Check': ['Rows', 'Columns', 'Missing / NaN cells', 'Non-finite numeric cells', 'Forbidden text cells', 'Duplicate _ROW_ID', 'Duplicate CASEID', 'Outcome class 0', 'Outcome class 1', 'Outcome class 2', 'Unique PSU', 'Unique strata'], 'Observed': [len(step8_processed), step8_processed.shape[1], total_missing, 0, 0, int(step8_processed['_ROW_ID'].duplicated().sum()), int(step8_processed['CASEID'].duplicated().sum()), outcome_counts[0], outcome_counts[1], outcome_counts[2], step8_processed['PSU'].nunique(), step8_processed['STRATUM'].nunique()], 'Expected': [5063, EXPECTED_STEP8_COLUMNS, 0, 0, 0, 0, 0, 403, 4407, 253, 672, 16]})
step8_qc['PASS'] = step8_qc['Observed'] == step8_qc['Expected']
if not step8_qc['PASS'].all():
    raise RuntimeError('Step-8 hard QC failed.')
STEP8_PROCESSED_FILE = DIRS['processed'] / 'STEP08_clean_demographic_socioeconomic_dataset.csv'
STEP8_SOURCE_AUDIT_FILE = DIRS['qc'] / 'STEP08_source_value_label_audit.csv'
STEP8_DISTRIBUTION_FILE = DIRS['qc'] / 'STEP08_predictor_distributions.csv'
STEP8_CATEGORY_QC_FILE = DIRS['qc'] / 'STEP08_category_qc.csv'
STEP8_MISSING_FILE = DIRS['qc'] / 'STEP08_missingness_by_variable.csv'
STEP8_QC_FILE = DIRS['qc'] / 'STEP08_complete_data_qc.csv'
step8_processed.to_csv(STEP8_PROCESSED_FILE, index=False)
source_label_audit.to_csv(STEP8_SOURCE_AUDIT_FILE, index=False)
step8_predictor_distributions.to_csv(STEP8_DISTRIBUTION_FILE, index=False)
category_qc.to_csv(STEP8_CATEGORY_QC_FILE, index=False)
missing_by_variable.to_csv(STEP8_MISSING_FILE, index=False)
step8_qc.to_csv(STEP8_QC_FILE, index=False)
STEP8_PROCESSED_SHA256 = calculate_sha256(STEP8_PROCESSED_FILE)
step8_manifest = {'step': 'STEP 8', 'description': 'Clean demographic and socioeconomic predictors', 'input_step7_sha256': observed_step7_sha256, 'analytic_n': int(len(step8_processed)), 'predictors_added': NEW_PREDICTORS, 'weekly_media_definition': 'At least weekly exposure to newspaper/magazine, radio, or television', 'religion_collapse': 'Islam vs Non-Islam', 'imputation_performed': False, 'rows_removed': 0, 'missing_cells': total_missing, 'forbidden_text_cells': 0, 'nonfinite_numeric_cells': 0, 'processed_sha256': STEP8_PROCESSED_SHA256, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP8_MANIFEST_FILE = DIRS['protocol'] / 'STEP08_demographic_socioeconomic_manifest.json'
with open(STEP8_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step8_manifest, f, indent=4, ensure_ascii=False, default=str)
logger.info(f'STEP 8 completed successfully: N={len(step8_processed):,}, columns={step8_processed.shape[1]}, missing={total_missing}.')
print('\n' + '=' * 78)
print('STEP 8 — CLEAN DEMOGRAPHIC + SOCIOECONOMIC PREDICTORS')
print('=' * 78)
print('\nSOURCE VALUE-LABEL AUDIT:')
display(source_label_audit[['Variable', 'Variable_Label', 'Raw_Value', 'Value_Label', 'N']])
print('\nCATEGORY-LEVEL QC:')
display(category_qc)
print('\nFINAL STEP-8 HARD QC:')
display(step8_qc)
print('\nPREDICTOR DISTRIBUTIONS:')
display(step8_predictor_distributions)
print('\nZERO-MISSING / CLEANLINESS CHECK:')
print(f'Rows                         : {len(step8_processed):,}')
print(f'Columns                      : {step8_processed.shape[1]}')
print(f'Missing / NaN cells          : {total_missing:,}')
print(f'Non-finite numeric cells     : 0')
print(f'Forbidden text cells         : 0')
print(f'Rows removed in Step 8       : 0')
print('\nSTEP-8 PROCESSED FILE:')
print(STEP8_PROCESSED_FILE)
print('\nSTEP-8 SHA-256:')
print(STEP8_PROCESSED_SHA256)
print('\nIMPORTANT:')
print('- All seven predictors were derived from original Stata value labels.')
print('- No missing/special value was silently imputed.')
print('- Weekly media exposure means >=weekly exposure to at least one of three media sources.')
print('- Religion is pre-specified as Islam vs Non-Islam.')
print('- Raw Vxxx fields are not retained in the clean processed modeling dataset.')
print('- Outcome counts and survey-design structure remain unchanged.')
print('=' * 78)
print('STATUS: STEP 8 PASS')
print('=' * 78)


2026-09-16 21:05:58,534 | INFO | STEP 8 completed successfully: N=5,063, columns=32, missing=0.



STEP 8 — CLEAN DEMOGRAPHIC + SOCIOECONOMIC PREDICTORS

SOURCE VALUE-LABEL AUDIT:


,Variable,Variable_Label,Raw_Value,Value_Label,N
0,V106,highest educational level,2,secondary,2666
1,V106,highest educational level,1,primary,1173
2,V106,highest educational level,3,higher,958
3,V106,highest educational level,0,no education,266
4,V190,wealth index combined,1,poorest,1053
5,V190,wealth index combined,2,poorer,1020
6,V190,wealth index combined,3,middle,1008
7,V190,wealth index combined,4,richer,997
8,V190,wealth index combined,5,richest,985
9,V025,type of place of residence,2,rural,3395



CATEGORY-LEVEL QC:


,Predictor,Observed_Categories,Expected_Categories,PASS
0,maternal_education,4,4,True
1,wealth_quintile,5,5,True
2,residence,2,2,True
3,division,8,8,True
4,weekly_media_exposure,2,2,True
5,working_status,2,2,True
6,religion_final,2,2,True



FINAL STEP-8 HARD QC:


,Check,Observed,Expected,PASS
0,Rows,5063,5063,True
1,Columns,32,32,True
2,Missing / NaN cells,0,0,True
3,Non-finite numeric cells,0,0,True
4,Forbidden text cells,0,0,True
5,Duplicate _ROW_ID,0,0,True
6,Duplicate CASEID,0,0,True
7,Outcome class 0,403,403,True
8,Outcome class 1,4407,4407,True
9,Outcome class 2,253,253,True



PREDICTOR DISTRIBUTIONS:


,Predictor,Category,N,Weighted_N,Percent,Weighted_Percent
0,maternal_education,Higher,958,906.525904,18.921588,17.681002
1,maternal_education,No education,266,274.456238,5.253802,5.353031
2,maternal_education,Primary,1173,1160.247762,23.168082,22.629628
3,maternal_education,Secondary,2666,2785.888347,52.656528,54.336339
4,wealth_quintile,Middle,1008,1056.731074,19.909145,20.610624
5,wealth_quintile,Poorer,1020,1068.354627,20.146158,20.837332
6,wealth_quintile,Poorest,1053,1047.763390,20.797946,20.435717
7,wealth_quintile,Richer,997,1017.926301,19.691882,19.853771
8,wealth_quintile,Richest,985,936.342859,19.454869,18.262556
9,residence,Rural,3395,3754.351837,67.055106,73.225380



ZERO-MISSING / CLEANLINESS CHECK:
Rows                         : 5,063
Columns                      : 32
Missing / NaN cells          : 0
Non-finite numeric cells     : 0
Forbidden text cells         : 0
Rows removed in Step 8       : 0

STEP-8 PROCESSED FILE:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP08_clean_demographic_socioeconomic_dataset.csv

STEP-8 SHA-256:
7ed71072a7ed6cc68e31c379589c5049936caf4c3da013738fe6668070aff94e

IMPORTANT:
- All seven predictors were derived from original Stata value labels.
- No missing/special value was silently imputed.
- Weekly media exposure means >=weekly exposure to at least one of three media sources.
- Religion is pre-specified as Islam vs Non-Islam.
- Raw Vxxx fields are not retained in the clean processed modeling dataset.
- Outcome counts and survey-design structure remain unchanged.
STATUS: STEP 8 PASS


In [9]:
# Step 9: Pregnancy intention, partner education, and healthcare decision.

import json
import datetime as dt
import numpy as np
import pandas as pd
STEP8_PROCESSED_FILE = DIRS['processed'] / 'STEP08_clean_demographic_socioeconomic_dataset.csv'
EXPECTED_STEP8_SHA256 = '7ed71072a7ed6cc68e31c379589c5049936caf4c3da013738fe6668070aff94e'
if not STEP8_PROCESSED_FILE.exists():
    raise FileNotFoundError(f'Step-8 processed file not found:\n{STEP8_PROCESSED_FILE}')
observed_step8_sha256 = calculate_sha256(STEP8_PROCESSED_FILE)
if observed_step8_sha256 != EXPECTED_STEP8_SHA256:
    raise RuntimeError('STEP-8 HASH MISMATCH.\nThe locked Step-8 dataset has changed.\nSTOP.')
step8_processed = pd.read_csv(STEP8_PROCESSED_FILE)
if len(step8_processed) != 5063:
    raise RuntimeError(f'Unexpected Step-8 N: {len(step8_processed):,}')
if not step8_processed['_ROW_ID'].is_unique:
    raise RuntimeError('Duplicate _ROW_ID in Step-8 dataset.')
if not step8_processed['CASEID'].is_unique:
    raise RuntimeError('Duplicate CASEID in Step-8 dataset.')
if 'raw' not in globals():
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'stata_meta' not in globals():
    _, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), metadataonly=True)
raw_variable_labels = getattr(stata_meta, 'column_names_to_labels', {}) or {}
variable_label_map = {str(variable).strip().upper(): label for variable, label in raw_variable_labels.items()}
raw_variable_value_labels = getattr(stata_meta, 'variable_value_labels', {}) or {}
VARIABLE_VALUE_LABELS = {str(variable).strip().upper(): dict(mapping) for variable, mapping in raw_variable_value_labels.items()}
STEP9_SOURCE_VARIABLES = ['V501', 'V701', 'V743A']
missing_sources = [variable for variable in STEP9_SOURCE_VARIABLES if variable not in raw.columns]
if missing_sources:
    raise RuntimeError(f'Missing Step-9 source variables:\n{missing_sources}')
raw_step9 = raw[['_ROW_ID'] + STEP9_SOURCE_VARIABLES].copy()
work = step8_processed.merge(raw_step9, on='_ROW_ID', how='left', validate='one_to_one')
if len(work) != 5063:
    raise RuntimeError('Unexpected row count after Step-9 raw merge.')
if not np.array_equal(work['_ROW_ID'].to_numpy(), step8_processed['_ROW_ID'].to_numpy()):
    raise RuntimeError('Row order changed during Step-9 merge.')

def normalize_text(value):
    if value is None:
        return ''
    return str(value).strip().lower().replace('’', "'").replace('`', "'").replace('–', '-').replace('—', '-')

def get_value_label(variable, raw_value):
    if pd.isna(raw_value):
        return None
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    candidates = [raw_value]
    try:
        candidates.append(int(raw_value))
    except Exception:
        pass
    try:
        candidates.append(float(raw_value))
    except Exception:
        pass
    for candidate in candidates:
        if candidate in mapping:
            return mapping[candidate]
    return None
SPECIAL_SOURCE_PATTERNS = ["don't know", 'dont know', 'unknown', 'missing', 'not applicable', 'not asked', 'inconsistent']

def is_special_label(value):
    text = normalize_text(value)
    if text == '':
        return False
    return any((pattern in text for pattern in SPECIAL_SOURCE_PATTERNS))
for variable in STEP9_SOURCE_VARIABLES:
    label_column = variable + '_LABEL'
    work[label_column] = [get_value_label(variable, raw_value) for raw_value in work[variable]]
if work['V501'].isna().any():
    raise RuntimeError('V501 current marital status contains missing values.')
if work['V501_LABEL'].isna().any():
    unlabelled = work.loc[work['V501_LABEL'].isna(), ['V501']].value_counts()
    raise RuntimeError(f'Observed V501 code without Stata label:\n{unlabelled}')
v501_normalized = work['V501_LABEL'].map(normalize_text)
v501_observed_labels = sorted(set(v501_normalized))
CURRENT_PARTNER_LABELS = {'married', 'living with partner', 'living together'}
NO_CURRENT_PARTNER_LABELS = {'never in union', 'never married', 'widowed', 'divorced', 'separated', 'not living together', 'no longer living together', 'no longer living together/separated'}
all_allowed_v501_labels = CURRENT_PARTNER_LABELS | NO_CURRENT_PARTNER_LABELS
unexpected_v501_labels = sorted(set(v501_observed_labels) - all_allowed_v501_labels)
if unexpected_v501_labels:
    raise RuntimeError(f'Unexpected V501 marital-status label(s):\n{unexpected_v501_labels}')
has_current_partner = v501_normalized.isin(CURRENT_PARTNER_LABELS)
classified_v501 = v501_normalized.isin(all_allowed_v501_labels)
if not classified_v501.all():
    raise RuntimeError('At least one V501 category was not classified.')
work['current_partner_status'] = np.where(has_current_partner, 'Current partner', 'No current partner')
if work['current_partner_status'].isna().any():
    raise RuntimeError('current_partner_status contains missing values.')
if work['PREG_INTENTION_VALUE_LABEL'].isna().any():
    raise RuntimeError('Slot-matched pregnancy-intention label is missing.')
preg_intention_normalized = work['PREG_INTENTION_VALUE_LABEL'].map(normalize_text)
PREGNANCY_INTENTION_MAP = {'then': 'Wanted then', 'later': 'Wanted later', 'no more': 'Wanted no more'}
unexpected_preg_intention = sorted(set(preg_intention_normalized) - set(PREGNANCY_INTENTION_MAP.keys()))
if unexpected_preg_intention:
    raise RuntimeError(f'Unexpected M10 pregnancy-intention category:\n{unexpected_preg_intention}')
work['pregnancy_intention'] = preg_intention_normalized.map(PREGNANCY_INTENTION_MAP)
PARTNER_EDUCATION_MAP = {'no education': 'No education', 'primary': 'Primary', 'secondary': 'Secondary', 'higher': 'Higher'}

def recode_partner_education(has_partner, raw_value, source_label):
    if not has_partner:
        return 'No current partner / not applicable'
    if pd.isna(raw_value):
        return 'Partner education not reported'
    if source_label is None:
        raise ValueError(f'Unlabelled V701 code: {raw_value}')
    normalized = normalize_text(source_label)
    if is_special_label(source_label):
        return 'Partner education not reported'
    if normalized in PARTNER_EDUCATION_MAP:
        return PARTNER_EDUCATION_MAP[normalized]
    raise ValueError(f'Unexpected substantive V701 label: {source_label}')
partner_education_values = []
for partner_flag, raw_value, source_label in zip(has_current_partner, work['V701'], work['V701_LABEL']):
    try:
        result = recode_partner_education(bool(partner_flag), raw_value, source_label)
    except ValueError as exc:
        raise RuntimeError(f'Partner education recoding failed: {exc}') from exc
    partner_education_values.append(result)
work['partner_education_final'] = partner_education_values

def recode_healthcare_decision(has_partner, raw_value, source_label):
    if not has_partner:
        return 'No current partner / not applicable'
    if pd.isna(raw_value):
        return 'Decision information not reported'
    if source_label is None:
        raise ValueError(f'Unlabelled V743A code: {raw_value}')
    normalized = normalize_text(source_label)
    if is_special_label(source_label):
        return 'Decision information not reported'
    if normalized == 'respondent alone':
        return 'Respondent alone'
    if 'respondent' in normalized and ('husband' in normalized or 'partner' in normalized):
        return 'Joint with partner'
    OTHER_DECISION_LABELS = {'husband/partner alone', 'husband alone', 'partner alone', 'someone else', 'other'}
    if normalized in OTHER_DECISION_LABELS:
        return 'Other decision arrangement'
    raise ValueError(f'Unexpected substantive V743A label: {source_label}')
healthcare_decision_values = []
for partner_flag, raw_value, source_label in zip(has_current_partner, work['V743A'], work['V743A_LABEL']):
    try:
        result = recode_healthcare_decision(bool(partner_flag), raw_value, source_label)
    except ValueError as exc:
        raise RuntimeError(f'Healthcare-decision recoding failed: {exc}') from exc
    healthcare_decision_values.append(result)
work['healthcare_decision_final'] = healthcare_decision_values
v501_distribution = work.groupby(['V501', 'V501_LABEL', 'current_partner_status'], dropna=False, observed=True).size().reset_index(name='N').sort_values('V501').reset_index(drop=True)
current_partner_v701_missing = int((has_current_partner & work['V701'].isna()).sum())
no_partner_v701_missing = int((~has_current_partner & work['V701'].isna()).sum())
current_partner_v743a_missing = int((has_current_partner & work['V743A'].isna()).sum())
no_partner_v743a_missing = int((~has_current_partner & work['V743A'].isna()).sum())
current_partner_v701_special = int((has_current_partner & work['V701_LABEL'].map(is_special_label)).sum())
current_partner_v743a_special = int((has_current_partner & work['V743A_LABEL'].map(is_special_label)).sum())
source_pattern_audit = pd.DataFrame({'Metric': ['Current partner', 'No current partner', 'Current partner with V701 raw missing', 'No current partner with V701 raw missing', 'Current partner with V743A raw missing', 'No current partner with V743A raw missing', 'Current partner with V701 special-label value', 'Current partner with V743A special-label value'], 'N': [int(has_current_partner.sum()), int((~has_current_partner).sum()), current_partner_v701_missing, no_partner_v701_missing, current_partner_v743a_missing, no_partner_v743a_missing, current_partner_v701_special, current_partner_v743a_special]})
source_label_audit_rows = []
for variable in STEP9_SOURCE_VARIABLES:
    raw_values = work[variable]
    labels = work[variable + '_LABEL']
    temp = pd.DataFrame({'Raw_Value': raw_values, 'Value_Label': labels, 'Raw_Missing': raw_values.isna()})
    temp['Value_Label_Display'] = temp['Value_Label'].where(temp['Value_Label'].notna(), '<RAW MISSING / STRUCTURAL>')
    grouped = temp.groupby(['Raw_Value', 'Value_Label_Display', 'Raw_Missing'], dropna=False, observed=True).size().reset_index(name='N')
    for _, row in grouped.iterrows():
        source_label_audit_rows.append({'Variable': variable, 'Variable_Label': variable_label_map.get(variable), 'Raw_Value': row['Raw_Value'], 'Value_Label': row['Value_Label_Display'], 'Raw_Missing': bool(row['Raw_Missing']), 'N': int(row['N'])})
source_label_audit = pd.DataFrame(source_label_audit_rows)
NO_PARTNER_CATEGORY = 'No current partner / not applicable'
bad_partner_structural = ~has_current_partner & (work['partner_education_final'] != NO_PARTNER_CATEGORY)
bad_decision_structural = ~has_current_partner & (work['healthcare_decision_final'] != NO_PARTNER_CATEGORY)
if bad_partner_structural.any():
    raise RuntimeError('Partner education structural-N/A logic failed.')
if bad_decision_structural.any():
    raise RuntimeError('Healthcare-decision structural-N/A logic failed.')
bad_current_partner_education = has_current_partner & (work['partner_education_final'] == NO_PARTNER_CATEGORY)
bad_current_partner_decision = has_current_partner & (work['healthcare_decision_final'] == NO_PARTNER_CATEGORY)
if bad_current_partner_education.any():
    raise RuntimeError('Currently partnered woman incorrectly assigned structural N/A for partner education.')
if bad_current_partner_decision.any():
    raise RuntimeError('Currently partnered woman incorrectly assigned structural N/A for healthcare decision.')
NEW_STEP9_FIELDS = ['pregnancy_intention', 'current_partner_status', 'partner_education_final', 'healthcare_decision_final']
distribution_parts = []
for variable in NEW_STEP9_FIELDS:
    temp = work.groupby(variable, dropna=False, observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index()
    temp = temp.rename(columns={variable: 'Category'})
    temp['Variable'] = variable
    temp['Percent'] = temp['N'] / len(work) * 100
    temp['Weighted_Percent'] = temp['Weighted_N'] / temp['Weighted_N'].sum() * 100
    distribution_parts.append(temp[['Variable', 'Category', 'N', 'Weighted_N', 'Percent', 'Weighted_Percent']])
step9_distributions = pd.concat(distribution_parts, ignore_index=True)
step9_processed = step8_processed.copy()
for variable in NEW_STEP9_FIELDS:
    step9_processed[variable] = work[variable].to_numpy()
EXPECTED_STEP9_COLUMNS = 36
if step9_processed.shape[1] != EXPECTED_STEP9_COLUMNS:
    raise RuntimeError(f'Unexpected Step-9 column count.\nExpected : {EXPECTED_STEP9_COLUMNS}\nObserved : {step9_processed.shape[1]}')
string_columns = step9_processed.select_dtypes(include=['object', 'string']).columns.tolist()
for column in string_columns:
    step9_processed[column] = step9_processed[column].astype(str).str.strip()
missing_by_variable = step9_processed.isna().sum().rename('N_Missing').reset_index().rename(columns={'index': 'Variable'})
total_missing = int(missing_by_variable['N_Missing'].sum())
if total_missing != 0:
    raise RuntimeError(f'Step-9 processed dataset contains {total_missing:,} missing / NaN cells.')
nonfinite_records = []
numeric_columns = step9_processed.select_dtypes(include=[np.number]).columns.tolist()
for column in numeric_columns:
    values = step9_processed[column].to_numpy(dtype=float)
    n_nonfinite = int((~np.isfinite(values)).sum())
    if n_nonfinite > 0:
        nonfinite_records.append({'Variable': column, 'N_Nonfinite': n_nonfinite})
if nonfinite_records:
    raise RuntimeError(f'Non-finite numeric values detected:\n{nonfinite_records}')
FORBIDDEN_EXACT_TEXT = {'', 'nan', 'none', 'null', 'na', 'n/a', "don't know", 'dont know', 'dk', 'unknown', 'missing', '__invalid__'}
forbidden_records = []
for column in string_columns:
    normalized = step9_processed[column].astype(str).str.strip().str.lower()
    bad_mask = normalized.isin(FORBIDDEN_EXACT_TEXT)
    if bad_mask.any():
        counts = normalized[bad_mask].value_counts()
        for value, count in counts.items():
            forbidden_records.append({'Variable': column, 'Value': value, 'N': int(count)})
if forbidden_records:
    raise RuntimeError(f'Forbidden raw missing/special text remains:\n{forbidden_records}')
EXPECTED_PREGNANCY_INTENTION = {'Wanted then', 'Wanted later', 'Wanted no more'}
observed_preg_intention = set(step9_processed['pregnancy_intention'].unique())
if observed_preg_intention != EXPECTED_PREGNANCY_INTENTION:
    raise RuntimeError(f'Unexpected final pregnancy-intention categories:\n{observed_preg_intention}')
EXPECTED_PARTNER_STATUS = {'Current partner', 'No current partner'}
observed_partner_status = set(step9_processed['current_partner_status'].unique())
if observed_partner_status != EXPECTED_PARTNER_STATUS:
    raise RuntimeError(f'Unexpected current-partner-status categories:\n{observed_partner_status}')
ALLOWED_PARTNER_EDUCATION = {'No education', 'Primary', 'Secondary', 'Higher', 'No current partner / not applicable', 'Partner education not reported'}
observed_partner_education = set(step9_processed['partner_education_final'].unique())
unexpected_partner_education = observed_partner_education - ALLOWED_PARTNER_EDUCATION
if unexpected_partner_education:
    raise RuntimeError(f'Unexpected partner-education categories:\n{unexpected_partner_education}')
ALLOWED_HEALTHCARE_DECISION = {'Respondent alone', 'Joint with partner', 'Other decision arrangement', 'No current partner / not applicable', 'Decision information not reported'}
observed_healthcare_decision = set(step9_processed['healthcare_decision_final'].unique())
unexpected_healthcare_decision = observed_healthcare_decision - ALLOWED_HEALTHCARE_DECISION
if unexpected_healthcare_decision:
    raise RuntimeError(f'Unexpected healthcare-decision categories:\n{unexpected_healthcare_decision}')
outcome_counts = step9_processed['ANC_CLASS'].value_counts().sort_index().to_dict()
EXPECTED_OUTCOME_COUNTS = {0: 403, 1: 4407, 2: 253}
if outcome_counts != EXPECTED_OUTCOME_COUNTS:
    raise RuntimeError('ANC outcome distribution changed unexpectedly.')
if step9_processed['PSU'].nunique() != 672:
    raise RuntimeError('PSU count changed.')
if step9_processed['STRATUM'].nunique() != 16:
    raise RuntimeError('Stratum count changed.')
step9_qc = pd.DataFrame({'Check': ['Rows', 'Columns', 'Missing / NaN cells', 'Non-finite numeric cells', 'Forbidden raw special-text cells', 'Duplicate _ROW_ID', 'Duplicate CASEID', 'Outcome class 0', 'Outcome class 1', 'Outcome class 2', 'Unique PSU', 'Unique strata'], 'Observed': [len(step9_processed), step9_processed.shape[1], total_missing, 0, 0, int(step9_processed['_ROW_ID'].duplicated().sum()), int(step9_processed['CASEID'].duplicated().sum()), outcome_counts[0], outcome_counts[1], outcome_counts[2], step9_processed['PSU'].nunique(), step9_processed['STRATUM'].nunique()], 'Expected': [5063, EXPECTED_STEP9_COLUMNS, 0, 0, 0, 0, 0, 403, 4407, 253, 672, 16]})
step9_qc['PASS'] = step9_qc['Observed'] == step9_qc['Expected']
if not step9_qc['PASS'].all():
    raise RuntimeError('Step-9 hard QC failed.')
STEP9_PROCESSED_FILE = DIRS['processed'] / 'STEP09_clean_partner_decision_dataset.csv'
STEP9_V501_FILE = DIRS['qc'] / 'STEP09_V501_partner_status_audit.csv'
STEP9_SOURCE_AUDIT_FILE = DIRS['qc'] / 'STEP09_partner_source_label_audit.csv'
STEP9_PATTERN_FILE = DIRS['qc'] / 'STEP09_structural_missingness_audit.csv'
STEP9_DISTRIBUTION_FILE = DIRS['qc'] / 'STEP09_partner_predictor_distributions.csv'
STEP9_MISSING_FILE = DIRS['qc'] / 'STEP09_missingness_by_variable.csv'
STEP9_QC_FILE = DIRS['qc'] / 'STEP09_complete_data_qc.csv'
step9_processed.to_csv(STEP9_PROCESSED_FILE, index=False)
v501_distribution.to_csv(STEP9_V501_FILE, index=False)
source_label_audit.to_csv(STEP9_SOURCE_AUDIT_FILE, index=False)
source_pattern_audit.to_csv(STEP9_PATTERN_FILE, index=False)
step9_distributions.to_csv(STEP9_DISTRIBUTION_FILE, index=False)
missing_by_variable.to_csv(STEP9_MISSING_FILE, index=False)
step9_qc.to_csv(STEP9_QC_FILE, index=False)
STEP9_PROCESSED_SHA256 = calculate_sha256(STEP9_PROCESSED_FILE)
step9_manifest = {'step': 'STEP 9', 'description': 'Pregnancy intention, partner education, and healthcare decision with explicit structural no-current-partner handling', 'input_step8_sha256': observed_step8_sha256, 'analytic_n': int(len(step9_processed)), 'predictors_added': ['pregnancy_intention', 'partner_education_final', 'healthcare_decision_final'], 'auxiliary_variable_added': 'current_partner_status', 'current_partner_v501_labels': sorted(CURRENT_PARTNER_LABELS), 'no_current_partner_v501_labels': sorted(NO_CURRENT_PARTNER_LABELS), 'structural_no_partner_category': 'No current partner / not applicable', 'imputation_performed': False, 'rows_removed': 0, 'missing_cells': total_missing, 'nonfinite_numeric_cells': 0, 'forbidden_raw_special_text_cells': 0, 'processed_sha256': STEP9_PROCESSED_SHA256, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP9_MANIFEST_FILE = DIRS['protocol'] / 'STEP09_partner_decision_manifest.json'
with open(STEP9_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step9_manifest, f, indent=4, ensure_ascii=False, default=str)
logger.info(f'STEP 9 completed successfully: N={len(step9_processed):,}, missing={total_missing}, current_partner={int(has_current_partner.sum()):,}, no_current_partner={int((~has_current_partner).sum()):,}.')
print('\n' + '=' * 78)
print('STEP 9 — PREGNANCY INTENTION + PARTNER EDUCATION + HEALTHCARE DECISION')
print('=' * 78)
print('\nV501 CURRENT-PARTNER CLASSIFICATION:')
display(v501_distribution)
print('\nRAW SOURCE LABEL AUDIT:')
display(source_label_audit)
print('\nSTRUCTURAL MISSINGNESS / PARTNERSHIP AUDIT:')
display(source_pattern_audit)
print('\nFINAL CATEGORY DISTRIBUTIONS:')
display(step9_distributions)
print('\nFINAL STEP-9 HARD QC:')
display(step9_qc)
print('\nCLEANLINESS:')
print(f'Rows                         : {len(step9_processed):,}')
print(f'Columns                      : {step9_processed.shape[1]}')
print(f'Missing / NaN cells          : {total_missing:,}')
print('Non-finite numeric cells     : 0')
print("Raw don't-know/unknown text  : 0")
print('Rows removed in Step 9       : 0')
print('\nSTRUCTURAL HANDLING:')
print(f'Current partner              : {int(has_current_partner.sum()):,}')
print(f'No current partner           : {int((~has_current_partner).sum()):,}')
print(f'Current partner V701 missing : {current_partner_v701_missing:,}')
print(f'Current partner V743A missing: {current_partner_v743a_missing:,}')
print('\nIMPORTANT:')
print("- 'no longer living together/separated' is correctly classified as No current partner.")
print('- No-current-partner women are NOT merged with Other decision arrangement.')
print('- Structural N/A is an explicit valid category, not a missing value.')
print("- Genuine current-partner missing/DK, if present, is represented as 'not reported'; never imputed.")
print('- Pregnancy intention remains exact-slot matched to the index pregnancy.')
print('- current_partner_status is auxiliary/QC only and is NOT a planned model predictor.')
print('\nSTEP-9 PROCESSED FILE:')
print(STEP9_PROCESSED_FILE)
print('\nSTEP-9 SHA-256:')
print(STEP9_PROCESSED_SHA256)
print('=' * 78)
print('STATUS: STEP 9 PASS')
print('=' * 78)


2026-09-16 21:06:01,846 | INFO | STEP 9 completed successfully: N=5,063, missing=0, current_partner=5,016, no_current_partner=47.



STEP 9 — PREGNANCY INTENTION + PARTNER EDUCATION + HEALTHCARE DECISION

V501 CURRENT-PARTNER CLASSIFICATION:


,V501,V501_LABEL,current_partner_status,N
0,1,married,Current partner,5016
1,3,widowed,No current partner,9
2,4,divorced,No current partner,17
3,5,no longer living together/separated,No current partner,21



RAW SOURCE LABEL AUDIT:


,Variable,Variable_Label,Raw_Value,Value_Label,Raw_Missing,N
0,V501,current marital status,1.0,married,False,5016
1,V501,current marital status,3.0,widowed,False,9
2,V501,current marital status,4.0,divorced,False,17
3,V501,current marital status,5.0,no longer living together/separated,False,21
4,V701,husband/partner's education level,2.0,secondary,False,1734
5,V701,husband/partner's education level,3.0,higher,False,1037
6,V701,husband/partner's education level,NaN,<RAW MISSING / STRUCTURAL>,True,47
7,V701,husband/partner's education level,0.0,no education,False,741
8,V701,husband/partner's education level,1.0,primary,False,1495
9,V701,husband/partner's education level,8.0,don't know,False,9



STRUCTURAL MISSINGNESS / PARTNERSHIP AUDIT:


... (2 outputs left out)

,Variable,Category,N,Weighted_N,Percent,Weighted_Percent
0,pregnancy_intention,Wanted later,617,608.852225,12.186451,11.875135
1,pregnancy_intention,Wanted no more,370,374.222362,7.307920,7.298883
2,pregnancy_intention,Wanted then,4076,4144.043664,80.505629,80.825982
3,current_partner_status,Current partner,5016,5075.367455,99.071697,98.990646
4,current_partner_status,No current partner,47,51.750796,0.928303,1.009354
5,partner_education_final,Higher,1037,995.546319,20.481928,19.417269
6,partner_education_final,No current partner / not applicable,47,51.750796,0.928303,1.009354
7,partner_education_final,No education,741,763.063330,14.635592,14.882889
8,partner_education_final,Partner education not reported,9,11.439631,0.177760,0.223120
9,partner_education_final,Primary,1495,1509.697260,29.527948,29.445337



FINAL STEP-9 HARD QC:


,Check,Observed,Expected,PASS
0,Rows,5063,5063,True
1,Columns,36,36,True
2,Missing / NaN cells,0,0,True
3,Non-finite numeric cells,0,0,True
4,Forbidden raw special-text cells,0,0,True
5,Duplicate _ROW_ID,0,0,True
6,Duplicate CASEID,0,0,True
7,Outcome class 0,403,403,True
8,Outcome class 1,4407,4407,True
9,Outcome class 2,253,253,True



CLEANLINESS:
Rows                         : 5,063
Columns                      : 36
Missing / NaN cells          : 0
Non-finite numeric cells     : 0
Raw don't-know/unknown text  : 0
Rows removed in Step 9       : 0

STRUCTURAL HANDLING:
Current partner              : 5,016
No current partner           : 47
Current partner V701 missing : 0
Current partner V743A missing: 0

IMPORTANT:
- 'no longer living together/separated' is correctly classified as No current partner.
- No-current-partner women are NOT merged with Other decision arrangement.
- Structural N/A is an explicit valid category, not a missing value.
- Genuine current-partner missing/DK, if present, is represented as 'not reported'; never imputed.
- Pregnancy intention remains exact-slot matched to the index pregnancy.
- current_partner_status is auxiliary/QC only and is NOT a planned model predictor.

STEP-9 PROCESSED FILE:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP09_clean_partner_decision_dataset.csv

STEP-9

In [10]:
# Step 10: Derive household/access variables and create the interim master used by Step 11. The interim predictor specification is superseded later.

import json
import datetime as dt
import numpy as np
import pandas as pd
STEP9_PROCESSED_FILE = DIRS['processed'] / 'STEP09_clean_partner_decision_dataset.csv'
EXPECTED_STEP9_SHA256 = 'e51a436a8d7f210232e558a0f5277bd25f7c3cca64199186198677b8bd367be4'
if not STEP9_PROCESSED_FILE.exists():
    raise FileNotFoundError(f'Step-9 processed file not found:\n{STEP9_PROCESSED_FILE}')
observed_step9_sha256 = calculate_sha256(STEP9_PROCESSED_FILE)
if observed_step9_sha256 != EXPECTED_STEP9_SHA256:
    raise RuntimeError('STEP-9 HASH MISMATCH.\nThe locked Step-9 dataset has changed.\nSTOP.')
step9_processed = pd.read_csv(STEP9_PROCESSED_FILE)
if len(step9_processed) != 5063:
    raise RuntimeError(f'Unexpected Step-9 N: {len(step9_processed):,}')
if step9_processed.shape[1] != 36:
    raise RuntimeError(f'Unexpected Step-9 column count.\nObserved: {step9_processed.shape[1]}')
if not step9_processed['_ROW_ID'].is_unique:
    raise RuntimeError('Duplicate _ROW_ID in Step-9 input.')
if not step9_processed['CASEID'].is_unique:
    raise RuntimeError('Duplicate CASEID in Step-9 input.')
if 'raw' not in globals():
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(col).strip().upper() for col in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'stata_meta' not in globals():
    _, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), metadataonly=True)
raw_variable_labels = getattr(stata_meta, 'column_names_to_labels', {}) or {}
variable_label_map = {str(variable).strip().upper(): label for variable, label in raw_variable_labels.items()}
raw_variable_value_labels = getattr(stata_meta, 'variable_value_labels', {}) or {}
VARIABLE_VALUE_LABELS = {str(variable).strip().upper(): dict(mapping) for variable, mapping in raw_variable_value_labels.items()}
STEP10_SOURCE_VARIABLES = ['V136', 'V151', 'V467D']
missing_sources = [variable for variable in STEP10_SOURCE_VARIABLES if variable not in raw.columns]
if missing_sources:
    raise RuntimeError(f'Missing Step-10 source variables:\n{missing_sources}')
raw_step10 = raw[['_ROW_ID'] + STEP10_SOURCE_VARIABLES].copy()
work = step9_processed.merge(raw_step10, on='_ROW_ID', how='left', validate='one_to_one')
if len(work) != 5063:
    raise RuntimeError('Unexpected row count after Step-10 merge.')
if not np.array_equal(work['_ROW_ID'].to_numpy(), step9_processed['_ROW_ID'].to_numpy()):
    raise RuntimeError('Row order changed during Step-10 merge.')

def normalize_text(value):
    if value is None:
        return ''
    return str(value).strip().lower().replace('’', "'").replace('`', "'").replace('–', '-').replace('—', '-')

def get_value_label(variable, raw_value):
    if pd.isna(raw_value):
        return None
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    candidates = [raw_value]
    try:
        candidates.append(int(raw_value))
    except Exception:
        pass
    try:
        candidates.append(float(raw_value))
    except Exception:
        pass
    for candidate in candidates:
        if candidate in mapping:
            return mapping[candidate]
    return None
SPECIAL_SOURCE_PATTERNS = ["don't know", 'dont know', 'unknown', 'missing', 'not applicable', 'not asked', 'inconsistent']

def is_special_label(value):
    text = normalize_text(value)
    if text == '':
        return False
    return any((pattern in text for pattern in SPECIAL_SOURCE_PATTERNS))
v136_numeric = pd.to_numeric(work['V136'], errors='coerce')
n_v136_missing = int(v136_numeric.isna().sum())
if n_v136_missing != 0:
    raise RuntimeError(f'V136 contains {n_v136_missing} missing values.')
v136_values = v136_numeric.to_numpy(dtype=float)
if not np.isfinite(v136_values).all():
    raise RuntimeError('V136 contains non-finite values.')
if not np.allclose(v136_values, np.round(v136_values), rtol=0, atol=1e-10):
    raise RuntimeError('V136 contains non-integer household-size values.')
if (v136_numeric < 1).any():
    bad_values = sorted(v136_numeric[v136_numeric < 1].unique().tolist())
    raise RuntimeError(f'Invalid household-size value(s) detected:\n{bad_values}')
work['household_size_members'] = v136_numeric.round().astype(np.int64)
work['household_size_group'] = np.where(work['household_size_members'] <= 4, '1-4 members', '5+ members')
household_size_summary = work['household_size_members'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).rename('Value').reset_index().rename(columns={'index': 'Statistic'})
if work['V151'].isna().any():
    raise RuntimeError('V151 household-head sex contains missing values.')
work['V151_LABEL'] = [get_value_label('V151', raw_value) for raw_value in work['V151']]
if work['V151_LABEL'].isna().any():
    bad_codes = work.loc[work['V151_LABEL'].isna(), 'V151'].value_counts().to_dict()
    raise RuntimeError(f'Observed V151 code(s) without Stata label:\n{bad_codes}')
if work['V151_LABEL'].map(is_special_label).any():
    bad = work.loc[work['V151_LABEL'].map(is_special_label), ['V151', 'V151_LABEL']].value_counts()
    raise RuntimeError(f'Special V151 category detected:\n{bad}')
v151_normalized = work['V151_LABEL'].map(normalize_text)
HOUSEHOLD_HEAD_SEX_MAP = {'male': 'Male', 'female': 'Female'}
unexpected_v151 = sorted(set(v151_normalized) - set(HOUSEHOLD_HEAD_SEX_MAP.keys()))
if unexpected_v151:
    raise RuntimeError(f'Unexpected V151 label(s):\n{unexpected_v151}')
work['household_head_sex'] = v151_normalized.map(HOUSEHOLD_HEAD_SEX_MAP)
if work['V467D'].isna().any():
    raise RuntimeError('V467D distance-barrier variable contains missing values.')
work['V467D_LABEL'] = [get_value_label('V467D', raw_value) for raw_value in work['V467D']]
if work['V467D_LABEL'].isna().any():
    bad_codes = work.loc[work['V467D_LABEL'].isna(), 'V467D'].value_counts().to_dict()
    raise RuntimeError(f'Observed V467D code(s) without Stata label:\n{bad_codes}')
if work['V467D_LABEL'].map(is_special_label).any():
    bad = work.loc[work['V467D_LABEL'].map(is_special_label), ['V467D', 'V467D_LABEL']].value_counts()
    raise RuntimeError(f'Special V467D category detected:\n{bad}')

def recode_distance_barrier(label):
    text = normalize_text(label)
    if 'not a big problem' in text or text == 'no problem':
        return 'Not a big problem'
    if text == 'big problem':
        return 'Big problem'
    raise ValueError(f'Unexpected V467D label: {label}')
try:
    work['distance_barrier'] = work['V467D_LABEL'].map(recode_distance_barrier)
except ValueError as exc:
    raise RuntimeError(f'Distance-barrier recoding failed: {exc}') from exc
source_audit_parts = []
v136_distribution = work['V136'].value_counts(dropna=False).sort_index().rename_axis('Raw_Value').reset_index(name='N')
v136_distribution['Variable'] = 'V136'
v136_distribution['Variable_Label'] = variable_label_map.get('V136')
v136_distribution['Value_Label'] = '<numeric count>'
source_audit_parts.append(v136_distribution[['Variable', 'Variable_Label', 'Raw_Value', 'Value_Label', 'N']])
for variable in ['V151', 'V467D']:
    temp = work.groupby([variable, variable + '_LABEL'], dropna=False, observed=True).size().reset_index(name='N').rename(columns={variable: 'Raw_Value', variable + '_LABEL': 'Value_Label'})
    temp['Variable'] = variable
    temp['Variable_Label'] = variable_label_map.get(variable)
    source_audit_parts.append(temp[['Variable', 'Variable_Label', 'Raw_Value', 'Value_Label', 'N']])
step10_source_audit = pd.concat(source_audit_parts, ignore_index=True)
step10_processed = step9_processed.copy()
STEP10_NEW_FIELDS = ['household_size_members', 'household_size_group', 'household_head_sex', 'distance_barrier']
for variable in STEP10_NEW_FIELDS:
    step10_processed[variable] = work[variable].to_numpy()
EXPECTED_STEP10_FULL_COLUMNS = 40
if step10_processed.shape[1] != EXPECTED_STEP10_FULL_COLUMNS:
    raise RuntimeError(f'Unexpected Step-10 full processed column count.\nExpected : {EXPECTED_STEP10_FULL_COLUMNS}\nObserved : {step10_processed.shape[1]}')
step10_processed['maternal_age_group'] = step10_processed['MATERNAL_AGE_INDEX_BIRTH_GROUP']
step10_processed['prior_parity_group'] = step10_processed['PRIOR_PARITY_GROUP']
PRIMARY_PREDICTORS = ['maternal_age_group', 'prior_parity_group', 'maternal_education', 'partner_education_final', 'wealth_quintile', 'residence', 'division', 'weekly_media_exposure', 'working_status', 'pregnancy_intention', 'religion_final', 'healthcare_decision_final', 'household_size_group', 'household_head_sex', 'distance_barrier']
if len(PRIMARY_PREDICTORS) != 15:
    raise RuntimeError('Primary predictor list does not contain exactly 15 variables.')
if len(set(PRIMARY_PREDICTORS)) != 15:
    raise RuntimeError('Duplicate variable detected in primary predictor list.')
missing_primary_predictors = [variable for variable in PRIMARY_PREDICTORS if variable not in step10_processed.columns]
if missing_primary_predictors:
    raise RuntimeError(f'Missing final primary predictor(s):\n{missing_primary_predictors}')
FORBIDDEN_MODEL_FEATURES = {'_ROW_ID', 'CASEID', 'ANC_VISITS', 'ANC_CLASS', 'ANC_CLASS_LABEL', 'ANC_VISITS_RAW', 'ANC_VISITS_VALUE_LABEL', 'SURVEY_WEIGHT', 'V005', 'PSU', 'V021', 'STRATUM', 'V023', 'INDEX_MATERNITY_SLOT', 'M80_SELECTED_VALUE', 'M80_SOURCE_VARIABLE', 'M14_SOURCE_VARIABLE', 'SELECTED_MIDX', 'MATCHED_BIDX', 'INDEX_BIRTH_CMC'}
leakage_overlap = sorted(set(PRIMARY_PREDICTORS) & FORBIDDEN_MODEL_FEATURES)
if leakage_overlap:
    raise RuntimeError(f'FORBIDDEN leakage/design variable in predictor list:\n{leakage_overlap}')
MODEL_MASTER_ADMIN_COLUMNS = ['_ROW_ID', 'CASEID', 'ANC_VISITS', 'ANC_CLASS', 'ANC_CLASS_LABEL', 'SURVEY_WEIGHT', 'PSU', 'STRATUM']
FINAL_MODEL_MASTER_COLUMNS = MODEL_MASTER_ADMIN_COLUMNS + PRIMARY_PREDICTORS
final_model_master = step10_processed[FINAL_MODEL_MASTER_COLUMNS].copy().reset_index(drop=True)
EXPECTED_MODEL_MASTER_COLUMNS = 8 + 15
if final_model_master.shape[1] != EXPECTED_MODEL_MASTER_COLUMNS:
    raise RuntimeError(f'Unexpected final model-master column count.\nExpected : {EXPECTED_MODEL_MASTER_COLUMNS}\nObserved : {final_model_master.shape[1]}')
string_columns = final_model_master.select_dtypes(include=['object', 'string']).columns.tolist()
for column in string_columns:
    final_model_master[column] = final_model_master[column].astype(str).str.strip()
missing_by_variable = final_model_master.isna().sum().rename('N_Missing').reset_index().rename(columns={'index': 'Variable'})
total_missing = int(missing_by_variable['N_Missing'].sum())
if total_missing != 0:
    raise RuntimeError(f"FINAL MODEL MASTER contains missing/NaN cells:\n{missing_by_variable.loc[missing_by_variable['N_Missing'] > 0]}")
nonfinite_records = []
numeric_columns = final_model_master.select_dtypes(include=[np.number]).columns.tolist()
for column in numeric_columns:
    values = final_model_master[column].to_numpy(dtype=float)
    n_bad = int((~np.isfinite(values)).sum())
    if n_bad > 0:
        nonfinite_records.append({'Variable': column, 'N_Nonfinite': n_bad})
if nonfinite_records:
    raise RuntimeError(f'FINAL MODEL MASTER contains non-finite numeric values:\n{nonfinite_records}')
FORBIDDEN_EXACT_TEXT = {'', 'nan', 'none', 'null', 'na', 'n/a', "don't know", 'dont know', 'dk', 'unknown', 'missing', '__invalid__'}
forbidden_text_records = []
for column in string_columns:
    normalized = final_model_master[column].astype(str).str.strip().str.lower()
    bad_mask = normalized.isin(FORBIDDEN_EXACT_TEXT)
    if bad_mask.any():
        counts = normalized[bad_mask].value_counts()
        for value, count in counts.items():
            forbidden_text_records.append({'Variable': column, 'Forbidden_Value': value, 'N': int(count)})
if forbidden_text_records:
    raise RuntimeError(f'FINAL MODEL MASTER contains forbidden raw text:\n{forbidden_text_records}')
if len(final_model_master) != 5063:
    raise RuntimeError('Final model-master N changed.')
if final_model_master['_ROW_ID'].duplicated().any():
    raise RuntimeError('Duplicate _ROW_ID in final model master.')
if final_model_master['CASEID'].duplicated().any():
    raise RuntimeError('Duplicate CASEID in final model master.')
EXPECTED_OUTCOME_COUNTS = {0: 403, 1: 4407, 2: 253}
observed_outcome_counts = final_model_master['ANC_CLASS'].value_counts().sort_index().to_dict()
if observed_outcome_counts != EXPECTED_OUTCOME_COUNTS:
    raise RuntimeError(f'Final model-master outcome distribution changed.\n{observed_outcome_counts}')
if final_model_master['PSU'].nunique() != 672:
    raise RuntimeError('Final model-master PSU count changed.')
if final_model_master['STRATUM'].nunique() != 16:
    raise RuntimeError('Final model-master stratum count changed.')
if final_model_master['SURVEY_WEIGHT'].isna().any():
    raise RuntimeError('Survey weight contains missing values.')
if (final_model_master['SURVEY_WEIGHT'] <= 0).any():
    raise RuntimeError('Non-positive survey weight detected.')
ORDINAL_PREDICTORS = ['maternal_age_group', 'prior_parity_group', 'maternal_education', 'wealth_quintile']
NOMINAL_PREDICTORS = ['partner_education_final', 'division', 'pregnancy_intention', 'healthcare_decision_final']
BINARY_PREDICTORS = ['residence', 'weekly_media_exposure', 'working_status', 'religion_final', 'household_size_group', 'household_head_sex', 'distance_barrier']
architecture_union = ORDINAL_PREDICTORS + NOMINAL_PREDICTORS + BINARY_PREDICTORS
if set(architecture_union) != set(PRIMARY_PREDICTORS):
    raise RuntimeError('Predictor architecture does not exactly match the 15 primary predictors.')
if len(architecture_union) != len(set(architecture_union)):
    raise RuntimeError('A predictor appears in more than one architecture group.')
ORDERED_LEVELS = {'maternal_age_group': ['<20', '20-24', '25-29', '30-34', '35-39', '40+'], 'prior_parity_group': ['0', '1', '2', '3+'], 'maternal_education': ['No education', 'Primary', 'Secondary', 'Higher'], 'wealth_quintile': ['Poorest', 'Poorer', 'Middle', 'Richer', 'Richest']}
for variable, expected_levels in ORDERED_LEVELS.items():
    observed_levels = set(final_model_master[variable].unique())
    unexpected_levels = observed_levels - set(expected_levels)
    missing_levels = set(expected_levels) - observed_levels
    if unexpected_levels:
        raise RuntimeError(f'{variable}: unexpected level(s): {unexpected_levels}')
    if missing_levels:
        raise RuntimeError(f'{variable}: expected level(s) absent: {missing_levels}')
predictor_distribution_parts = []
for predictor in PRIMARY_PREDICTORS:
    temp = final_model_master.groupby(predictor, dropna=False, observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index().rename(columns={predictor: 'Category'})
    temp['Predictor'] = predictor
    temp['Percent'] = temp['N'] / len(final_model_master) * 100
    temp['Weighted_Percent'] = temp['Weighted_N'] / temp['Weighted_N'].sum() * 100
    predictor_distribution_parts.append(temp[['Predictor', 'Category', 'N', 'Weighted_N', 'Percent', 'Weighted_Percent']])
final_predictor_distributions = pd.concat(predictor_distribution_parts, ignore_index=True)
category_level_qc = final_predictor_distributions[['Predictor', 'Category', 'N', 'Weighted_N', 'Percent', 'Weighted_Percent']].copy()
category_level_qc['N_LT_10'] = category_level_qc['N'] < 10
category_level_qc['N_LT_20'] = category_level_qc['N'] < 20
feature_protocol_rows = []
for predictor in PRIMARY_PREDICTORS:
    if predictor in ORDINAL_PREDICTORS:
        predictor_type = 'Ordinal categorical'
    elif predictor in NOMINAL_PREDICTORS:
        predictor_type = 'Nominal categorical'
    elif predictor in BINARY_PREDICTORS:
        predictor_type = 'Binary categorical'
    else:
        raise RuntimeError(f'Predictor type not assigned: {predictor}')
    feature_protocol_rows.append({'Predictor': predictor, 'Predictor_Type': predictor_type, 'Primary_Model_Predictor': True, 'Encoding_At_Master_Stage': 'None', 'Encoding_Inside_CV': True})
feature_protocol = pd.DataFrame(feature_protocol_rows)
final_master_qc = pd.DataFrame({'Check': ['Rows', 'Columns', 'Primary predictors', 'Missing / NaN cells', 'Non-finite numeric cells', 'Forbidden raw special-text cells', 'Duplicate _ROW_ID', 'Duplicate CASEID', 'Leakage variables in predictor list', 'Outcome class 0', 'Outcome class 1', 'Outcome class 2', 'Unique PSU', 'Unique strata', 'Non-positive survey weights'], 'Observed': [len(final_model_master), final_model_master.shape[1], len(PRIMARY_PREDICTORS), total_missing, 0, 0, int(final_model_master['_ROW_ID'].duplicated().sum()), int(final_model_master['CASEID'].duplicated().sum()), len(leakage_overlap), observed_outcome_counts[0], observed_outcome_counts[1], observed_outcome_counts[2], final_model_master['PSU'].nunique(), final_model_master['STRATUM'].nunique(), int((final_model_master['SURVEY_WEIGHT'] <= 0).sum())], 'Expected': [5063, 23, 15, 0, 0, 0, 0, 0, 0, 403, 4407, 253, 672, 16, 0]})
final_master_qc['PASS'] = final_master_qc['Observed'] == final_master_qc['Expected']
if not final_master_qc['PASS'].all():
    raise RuntimeError('FINAL MODEL MASTER hard QC failed.')
STEP10_FULL_PROCESSED_FILE = DIRS['processed'] / 'STEP10_full_clean_processed_dataset.csv'
step10_processed.to_csv(STEP10_FULL_PROCESSED_FILE, index=False)
FINAL_MODEL_MASTER_FILE = DIRS['processed'] / 'FINAL_model_master_15_predictors.csv'
final_model_master.to_csv(FINAL_MODEL_MASTER_FILE, index=False)
STEP10_SOURCE_AUDIT_FILE = DIRS['qc'] / 'STEP10_household_access_source_audit.csv'
STEP10_HHSIZE_SUMMARY_FILE = DIRS['qc'] / 'STEP10_household_size_summary.csv'
STEP10_PREDICTOR_DIST_FILE = DIRS['qc'] / 'STEP10_final_predictor_distributions.csv'
STEP10_CATEGORY_QC_FILE = DIRS['qc'] / 'STEP10_final_category_level_qc.csv'
STEP10_MISSING_FILE = DIRS['qc'] / 'STEP10_final_missingness_by_variable.csv'
STEP10_MASTER_QC_FILE = DIRS['qc'] / 'STEP10_final_model_master_qc.csv'
FEATURE_PROTOCOL_FILE = DIRS['protocol'] / 'FINAL_15_predictor_protocol.csv'
step10_source_audit.to_csv(STEP10_SOURCE_AUDIT_FILE, index=False)
household_size_summary.to_csv(STEP10_HHSIZE_SUMMARY_FILE, index=False)
final_predictor_distributions.to_csv(STEP10_PREDICTOR_DIST_FILE, index=False)
category_level_qc.to_csv(STEP10_CATEGORY_QC_FILE, index=False)
missing_by_variable.to_csv(STEP10_MISSING_FILE, index=False)
final_master_qc.to_csv(STEP10_MASTER_QC_FILE, index=False)
feature_protocol.to_csv(FEATURE_PROTOCOL_FILE, index=False)
STEP10_FULL_SHA256 = calculate_sha256(STEP10_FULL_PROCESSED_FILE)
FINAL_MODEL_MASTER_SHA256 = calculate_sha256(FINAL_MODEL_MASTER_FILE)
step10_manifest = {'step': 'STEP 10', 'description': 'Household/access predictors and frozen 15-predictor clean model master', 'input_step9_sha256': observed_step9_sha256, 'analytic_n': int(len(final_model_master)), 'primary_predictors': PRIMARY_PREDICTORS, 'ordinal_predictors': ORDINAL_PREDICTORS, 'nominal_predictors': NOMINAL_PREDICTORS, 'binary_predictors': BINARY_PREDICTORS, 'ordered_levels': ORDERED_LEVELS, 'household_size_continuous_retained': True, 'household_size_primary_group': '1-4 members vs 5+ members', 'imputation_performed': False, 'rows_removed': 0, 'master_missing_cells': total_missing, 'master_nonfinite_cells': 0, 'master_forbidden_raw_text_cells': 0, 'survey_design_used_as_predictor': False, 'outcome_used_as_predictor': False, 'encoding_performed_at_master_stage': False, 'encoding_required_inside_cv': True, 'step10_full_sha256': STEP10_FULL_SHA256, 'final_model_master_sha256': FINAL_MODEL_MASTER_SHA256, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP10_MANIFEST_FILE = DIRS['protocol'] / 'STEP10_final_model_master_manifest.json'
with open(STEP10_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step10_manifest, f, indent=4, ensure_ascii=False, default=str)
logger.info(f'STEP 10 completed successfully: N={len(final_model_master):,}, predictors={len(PRIMARY_PREDICTORS)}, missing={total_missing}.')
print('\n' + '=' * 78)
print('STEP 10 — FINAL HOUSEHOLD / ACCESS PREDICTORS + MODEL MASTER')
print('=' * 78)
print('\nRAW SOURCE AUDIT:')
display(step10_source_audit)
print('\nHOUSEHOLD SIZE SUMMARY:')
display(household_size_summary)
print('\nFINAL 15-PREDICTOR PROTOCOL:')
display(feature_protocol)
print('\nFINAL 15-PREDICTOR DISTRIBUTIONS:')
display(final_predictor_distributions)
print('\nRARE-LEVEL AUDIT:')
display(category_level_qc.loc[category_level_qc['N_LT_20']])
print('\nFINAL MODEL MASTER HARD QC:')
display(final_master_qc)
print('\nFINAL MASTER SHAPE:')
print(f'{final_model_master.shape[0]:,} rows x {final_model_master.shape[1]} columns')
print('\nPRIMARY PREDICTORS:')
for i, predictor in enumerate(PRIMARY_PREDICTORS, start=1):
    print(f'{i:02d}. {predictor}')
print('\nCLEANLINESS:')
print(f'Missing / NaN cells          : {total_missing:,}')
print('Non-finite numeric cells     : 0')
print("Raw don't-know/unknown text  : 0")
print('Rows removed in Step 10      : 0')
print(f'Leakage variables in X       : {len(leakage_overlap)}')
print('\nFINAL MODEL MASTER:')
print(FINAL_MODEL_MASTER_FILE)
print('\nFINAL MODEL MASTER SHA-256:')
print(FINAL_MODEL_MASTER_SHA256)
print('\nFULL CLEAN PROCESSED DATASET:')
print(STEP10_FULL_PROCESSED_FILE)
print('\nIMPORTANT:')
print('- Final model master contains exactly 15 planned predictors.')
print('- No raw Vxxx source variable is present in the model predictor set.')
print('- Survey weight, PSU, stratum, IDs and outcomes are administrative fields only.')
print('- No one-hot or ordinal encoding has been performed yet.')
print('- All encoding will occur INSIDE cross-validation folds.')
print('- Exact household size is retained in the full processed file for sensitivity analysis.')
print('- The primary household-size predictor remains the pre-specified 1-4 vs 5+ grouping.')
print('=' * 78)
print('STATUS: STEP 10 PASS')
print('=' * 78)


2026-09-16 21:06:06,948 | INFO | STEP 10 completed successfully: N=5,063, predictors=15, missing=0.



STEP 10 — FINAL HOUSEHOLD / ACCESS PREDICTORS + MODEL MASTER

RAW SOURCE AUDIT:


,Variable,Variable_Label,Raw_Value,Value_Label,N
0,V136,number of household members (listed),2,<numeric count>,32
1,V136,number of household members (listed),3,<numeric count>,532
2,V136,number of household members (listed),4,<numeric count>,1060
3,V136,number of household members (listed),5,<numeric count>,1149
4,V136,number of household members (listed),6,<numeric count>,814
5,V136,number of household members (listed),7,<numeric count>,539
6,V136,number of household members (listed),8,<numeric count>,354
7,V136,number of household members (listed),9,<numeric count>,207
8,V136,number of household members (listed),10,<numeric count>,141
9,V136,number of household members (listed),11,<numeric count>,87



HOUSEHOLD SIZE SUMMARY:


,Statistic,Value
0,count,5063.000000
1,mean,5.802291
2,std,2.355550
3,min,2.000000
4,1%,3.000000
5,5%,3.000000
6,25%,4.000000
7,50%,5.000000
8,75%,7.000000
9,95%,10.000000



FINAL 15-PREDICTOR PROTOCOL:


... (4 outputs left out)

,Predictor,Category,N,Weighted_N,Percent,Weighted_Percent,N_LT_10,N_LT_20
17,partner_education_final,Partner education not reported,9,11.439631,0.17776,0.22312,True,True



FINAL MODEL MASTER HARD QC:


,Check,Observed,Expected,PASS
0,Rows,5063,5063,True
1,Columns,23,23,True
2,Primary predictors,15,15,True
3,Missing / NaN cells,0,0,True
4,Non-finite numeric cells,0,0,True
5,Forbidden raw special-text cells,0,0,True
6,Duplicate _ROW_ID,0,0,True
7,Duplicate CASEID,0,0,True
8,Leakage variables in predictor list,0,0,True
9,Outcome class 0,403,403,True



FINAL MASTER SHAPE:
5,063 rows x 23 columns

PRIMARY PREDICTORS:
01. maternal_age_group
02. prior_parity_group
03. maternal_education
04. partner_education_final
05. wealth_quintile
06. residence
07. division
08. weekly_media_exposure
09. working_status
10. pregnancy_intention
11. religion_final
12. healthcare_decision_final
13. household_size_group
14. household_head_sex
15. distance_barrier

CLEANLINESS:
Missing / NaN cells          : 0
Non-finite numeric cells     : 0
Raw don't-know/unknown text  : 0
Rows removed in Step 10      : 0
Leakage variables in X       : 0

FINAL MODEL MASTER:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\FINAL_model_master_15_predictors.csv

FINAL MODEL MASTER SHA-256:
dc006e7972db1a4cd27fddbbf961950b36d761e82e5e5b0381e7889209e11a76

FULL CLEAN PROCESSED DATASET:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\STEP10_full_clean_processed_dataset.csv

IMPORTANT:
- Final model master contains exactly 15 planned predictors.
- No raw Vxxx source var

In [11]:
# Step 11: Reference-lock the preprocessing base and 15-predictor reference master. Final partner/decision coding is completed in Step 12B.

import json
import hashlib
import datetime as dt
import numpy as np
import pandas as pd
import pyreadstat
from IPython.display import display

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
STEP10_MANIFEST_FILE = DIRS['protocol'] / 'STEP10_final_model_master_manifest.json'
STEP10_FULL_FILE = DIRS['processed'] / 'STEP10_full_clean_processed_dataset.csv'
STEP10_OLD_MASTER_FILE = DIRS['processed'] / 'FINAL_model_master_15_predictors.csv'
if not STEP10_MANIFEST_FILE.exists():
    raise FileNotFoundError(f'Missing Step-10 manifest:\n{STEP10_MANIFEST_FILE}')
if not STEP10_FULL_FILE.exists():
    raise FileNotFoundError(f'Missing Step-10 full dataset:\n{STEP10_FULL_FILE}')
if not STEP10_OLD_MASTER_FILE.exists():
    raise FileNotFoundError(f'Missing Step-10 model master:\n{STEP10_OLD_MASTER_FILE}')
with open(STEP10_MANIFEST_FILE, 'r', encoding='utf-8') as f:
    step10_manifest = json.load(f)
EXPECTED_PRINTED_STEP10_MASTER_SHA256 = 'dc006e7972db1a4cd27fddbbf961950b36d761e82e5e5b0381e7889209e11a76'
manifest_step10_master_sha = step10_manifest['final_model_master_sha256']
if manifest_step10_master_sha != EXPECTED_PRINTED_STEP10_MASTER_SHA256:
    raise RuntimeError('Step-10 manifest does not match the previously verified model-master hash.')
if sha256_file(STEP10_OLD_MASTER_FILE) != EXPECTED_PRINTED_STEP10_MASTER_SHA256:
    raise RuntimeError('Step-10 model-master file hash changed.')
expected_step10_full_sha = step10_manifest['step10_full_sha256']
if sha256_file(STEP10_FULL_FILE) != expected_step10_full_sha:
    raise RuntimeError('Step-10 full processed dataset hash changed.')
full = pd.read_csv(STEP10_FULL_FILE)
if len(full) != 5063:
    raise RuntimeError(f'Unexpected Step-10 N: {len(full):,}')
if not full['_ROW_ID'].is_unique:
    raise RuntimeError('Duplicate _ROW_ID in Step-10 full dataset.')
if not full['CASEID'].is_unique:
    raise RuntimeError('Duplicate CASEID in Step-10 full dataset.')
if 'raw' not in globals():
    raw_reload, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), apply_value_formats=False, formats_as_category=False, disable_datetime_conversion=True)
    raw_reload.columns = [str(column).strip().upper() for column in raw_reload.columns]
    raw = raw_reload.reset_index(drop=True).copy()
    raw.insert(0, '_ROW_ID', np.arange(len(raw), dtype=np.int64))
if 'stata_meta' not in globals():
    _, stata_meta = pyreadstat.read_dta(str(RAW_DATA_FILE), metadataonly=True)
raw_variable_labels = getattr(stata_meta, 'column_names_to_labels', {}) or {}
VARIABLE_LABELS = {str(variable).strip().upper(): label for variable, label in raw_variable_labels.items()}
raw_value_labels = getattr(stata_meta, 'variable_value_labels', {}) or {}
VARIABLE_VALUE_LABELS = {str(variable).strip().upper(): dict(mapping) for variable, mapping in raw_value_labels.items()}

def normalize_text(value):
    if value is None:
        return ''
    return str(value).strip().lower().replace('’', "'").replace('`', "'").replace('–', '-').replace('—', '-')

def get_value_label(variable, raw_value):
    if pd.isna(raw_value):
        return None
    mapping = VARIABLE_VALUE_LABELS.get(variable, {})
    candidates = [raw_value]
    try:
        candidates.append(int(raw_value))
    except Exception:
        pass
    try:
        candidates.append(float(raw_value))
    except Exception:
        pass
    for candidate in candidates:
        if candidate in mapping:
            return mapping[candidate]
    return None
REFERENCE_CHECK_VARIABLES = ['V502', 'V701', 'V743A', 'V130', 'V157', 'V158', 'V159']
missing_raw_variables = [variable for variable in REFERENCE_CHECK_VARIABLES if variable not in raw.columns]
if missing_raw_variables:
    raise RuntimeError(f'Required raw variable(s) absent:\n{missing_raw_variables}')
raw_reference = raw[['_ROW_ID'] + REFERENCE_CHECK_VARIABLES].copy()
audit = full.merge(raw_reference, on='_ROW_ID', how='left', validate='one_to_one')
if len(audit) != 5063:
    raise RuntimeError('Raw reference merge changed analytic N.')
if not np.array_equal(audit['_ROW_ID'].to_numpy(), full['_ROW_ID'].to_numpy()):
    raise RuntimeError('Row order changed during Step-11 raw merge.')
for variable in REFERENCE_CHECK_VARIABLES:
    audit[variable + '_LABEL'] = [get_value_label(variable, value) for value in audit[variable]]
v502 = pd.to_numeric(audit['V502'], errors='coerce')
if v502.isna().any():
    raise RuntimeError('V502 contains missing values in analytic cohort.')
allowed_v502_codes = {0, 1, 2}
observed_v502_codes = set(v502.astype(int).unique().tolist())
unexpected_v502 = observed_v502_codes - allowed_v502_codes
if unexpected_v502:
    raise RuntimeError(f'Unexpected V502 code(s):\n{unexpected_v502}')
CURRENT_UNION = v502.astype(int).eq(1)
current_union_status = np.where(CURRENT_UNION, 'Current partner', 'No current partner')
if 'current_partner_status' not in audit.columns:
    raise RuntimeError('current_partner_status missing from Step-10 dataset.')
partner_status_mismatch = audit['current_partner_status'].astype(str).str.strip().to_numpy() != current_union_status
n_partner_status_mismatch = int(partner_status_mismatch.sum())
if n_partner_status_mismatch != 0:
    raise RuntimeError(f'V501-derived current-partner status disagrees with authoritative V502 classification:\n{n_partner_status_mismatch} mismatches.')
EXPECTED_V701_LABELS = {0: 'no education', 1: 'primary', 2: 'secondary', 3: 'higher', 8: "don't know"}
for code, expected_label in EXPECTED_V701_LABELS.items():
    observed_label = get_value_label('V701', code)
    if observed_label is not None:
        if normalize_text(observed_label) != expected_label:
            raise RuntimeError(f'Unexpected V701 metadata label for code {code}: {observed_label}')
PARTNER_EDUCATION_CODE_MAP = {0: 'No education', 1: 'Primary', 2: 'Secondary', 3: 'Higher'}
partner_education_final = []
for is_current_union, raw_value in zip(CURRENT_UNION, audit['V701']):
    if not bool(is_current_union):
        partner_education_final.append('No current partner / not applicable')
        continue
    if pd.isna(raw_value):
        partner_education_final.append('Partner education not reported')
        continue
    code = int(raw_value)
    if code in PARTNER_EDUCATION_CODE_MAP:
        partner_education_final.append(PARTNER_EDUCATION_CODE_MAP[code])
    elif code in {8, 9}:
        partner_education_final.append('Partner education not reported')
    else:
        raise RuntimeError(f'Unexpected V701 code for a currently partnered respondent: {code}')
audit['partner_education_final_reference_locked'] = partner_education_final
EXPECTED_V743A_LABELS = {1: 'respondent alone', 2: 'respondent and husband/partner', 3: 'respondent and other person', 4: 'husband/partner alone', 5: 'someone else', 6: 'other'}
for code, expected_label in EXPECTED_V743A_LABELS.items():
    observed_label = get_value_label('V743A', code)
    if observed_label is not None:
        if normalize_text(observed_label) != expected_label:
            raise RuntimeError(f'Unexpected V743A metadata label for code {code}: {observed_label}')
HEALTHCARE_DECISION_CODE_MAP = {1: 'Respondent alone', 2: 'Respondent and husband/partner', 3: 'Respondent and other person', 4: 'Husband/partner alone', 5: 'Someone else', 6: 'Other'}
healthcare_decision_final = []
for is_current_union, raw_value in zip(CURRENT_UNION, audit['V743A']):
    if not bool(is_current_union):
        healthcare_decision_final.append('No current partner / not applicable')
        continue
    if pd.isna(raw_value):
        healthcare_decision_final.append('Decision information not reported')
        continue
    code = int(raw_value)
    if code in HEALTHCARE_DECISION_CODE_MAP:
        healthcare_decision_final.append(HEALTHCARE_DECISION_CODE_MAP[code])
    elif code == 9:
        healthcare_decision_final.append('Decision information not reported')
    else:
        raise RuntimeError(f'Unexpected V743A code among currently partnered women: {code}')
audit['healthcare_decision_final_reference_locked'] = healthcare_decision_final
MEDIA_VARIABLES = ['V157', 'V158', 'V159']
media_numeric = audit[MEDIA_VARIABLES].apply(pd.to_numeric, errors='coerce')
if media_numeric.isna().any().any():
    raise RuntimeError('At least one media source contains missing values.')
allowed_media_codes = {0, 1, 2, 3}
for variable in MEDIA_VARIABLES:
    observed_codes = set(media_numeric[variable].astype(int).unique().tolist())
    unexpected_codes = observed_codes - allowed_media_codes
    if unexpected_codes:
        raise RuntimeError(f'{variable}: unexpected media code(s): {unexpected_codes}')
weekly_any = media_numeric.isin([2, 3]).any(axis=1)
audit['weekly_media_exposure_reference_locked'] = np.where(weekly_any, 'Yes', 'No')
media_mismatch = audit['weekly_media_exposure'].astype(str).str.strip() != audit['weekly_media_exposure_reference_locked']
n_media_mismatch = int(media_mismatch.sum())
if n_media_mismatch != 0:
    raise RuntimeError(f'Existing weekly_media_exposure disagrees with DHS code-based reconstruction:\n{n_media_mismatch} mismatches.')
religion_labels = [get_value_label('V130', value) for value in audit['V130']]
if any((value is None for value in religion_labels)):
    raise RuntimeError('At least one V130 religion value lacks a label.')
religion_normalized = pd.Series(religion_labels).map(normalize_text)
RELIGION_FOUR_CATEGORY_MAP = {'islam': 'Islam', 'hinduism': 'Hinduism', 'buddhist': 'Buddhist', 'christianity': 'Christianity'}
unexpected_religion = set(religion_normalized.unique()) - set(RELIGION_FOUR_CATEGORY_MAP.keys())
if unexpected_religion:
    raise RuntimeError(f'Unexpected religion category detected:\n{unexpected_religion}')
audit['religion_4cat_sensitivity'] = religion_normalized.map(RELIGION_FOUR_CATEGORY_MAP).to_numpy()
audit['religion_final_reference_locked'] = np.where(religion_normalized.eq('islam'), 'Islam', 'Non-Islam')
religion_mismatch = audit['religion_final'].astype(str).str.strip() != audit['religion_final_reference_locked']
n_religion_mismatch = int(religion_mismatch.sum())
if n_religion_mismatch != 0:
    raise RuntimeError('Step-8 religion collapse disagrees with reference-locked reconstruction.')
required_exact_fields = ['MATERNAL_AGE_INDEX_BIRTH_YEARS', 'PRIOR_LIVE_BIRTHS', 'household_size_members']
missing_exact_fields = [variable for variable in required_exact_fields if variable not in full.columns]
if missing_exact_fields:
    raise RuntimeError(f'Required exact predictor field(s) missing:\n{missing_exact_fields}')
maternal_age_exact = pd.to_numeric(full['MATERNAL_AGE_INDEX_BIRTH_YEARS'], errors='coerce')
prior_births_exact = pd.to_numeric(full['PRIOR_LIVE_BIRTHS'], errors='coerce')
household_size_exact = pd.to_numeric(full['household_size_members'], errors='coerce')
for variable_name, series in {'maternal_age_at_index_birth_years': maternal_age_exact, 'prior_live_births': prior_births_exact, 'household_size_members': household_size_exact}.items():
    if series.isna().any():
        raise RuntimeError(f'{variable_name} contains missing values.')
    if not np.isfinite(series.to_numpy(dtype=float)).all():
        raise RuntimeError(f'{variable_name} contains non-finite values.')
if (maternal_age_exact < 10).any():
    raise RuntimeError('Maternal age <10 detected.')
if (maternal_age_exact >= 50).any():
    raise RuntimeError('Maternal age >=50 detected.')
if not np.allclose(prior_births_exact.to_numpy(dtype=float), np.round(prior_births_exact.to_numpy(dtype=float)), rtol=0, atol=1e-10):
    raise RuntimeError('prior_live_births contains non-integer values.')
if (prior_births_exact < 0).any():
    raise RuntimeError('Negative prior-live-birth count detected.')
if not np.allclose(household_size_exact.to_numpy(dtype=float), np.round(household_size_exact.to_numpy(dtype=float)), rtol=0, atol=1e-10):
    raise RuntimeError('household_size_members contains non-integer values.')
if (household_size_exact < 1).any():
    raise RuntimeError('Household size <1 detected.')
final_processed = full.copy()
final_processed['maternal_age_at_index_birth_years'] = maternal_age_exact.astype(float)
final_processed['prior_live_births'] = prior_births_exact.round().astype(np.int64)
final_processed['household_size_members'] = household_size_exact.round().astype(np.int64)
final_processed['partner_education_final'] = audit['partner_education_final_reference_locked'].to_numpy()
final_processed['healthcare_decision_final'] = audit['healthcare_decision_final_reference_locked'].to_numpy()
final_processed['weekly_media_exposure'] = audit['weekly_media_exposure_reference_locked'].to_numpy()
final_processed['religion_final'] = audit['religion_final_reference_locked'].to_numpy()
final_processed['religion_4cat_sensitivity'] = audit['religion_4cat_sensitivity'].to_numpy()
full_string_columns = final_processed.select_dtypes(include=['object', 'string']).columns.tolist()
for column in full_string_columns:
    final_processed[column] = final_processed[column].astype(str).str.strip()
full_total_missing = int(final_processed.isna().sum().sum())
if full_total_missing != 0:
    bad = final_processed.isna().sum()
    raise RuntimeError(f'Final full processed dataset contains missing cells:\n{bad[bad > 0]}')
full_nonfinite = []
for column in final_processed.select_dtypes(include=[np.number]).columns:
    values = final_processed[column].to_numpy(dtype=float)
    n_bad = int((~np.isfinite(values)).sum())
    if n_bad:
        full_nonfinite.append({'Variable': column, 'N': n_bad})
if full_nonfinite:
    raise RuntimeError(f'Non-finite values in final full dataset:\n{full_nonfinite}')
FORBIDDEN_EXACT_TEXT = {'', 'nan', 'none', 'null', 'na', 'n/a', "don't know", 'dont know', 'dk', 'unknown', 'missing', '__invalid__'}
full_forbidden_records = []
for column in full_string_columns:
    normalized = final_processed[column].astype(str).str.strip().str.lower()
    bad = normalized.isin(FORBIDDEN_EXACT_TEXT)
    if bad.any():
        counts = normalized[bad].value_counts()
        for value, count in counts.items():
            full_forbidden_records.append({'Variable': column, 'Value': value, 'N': int(count)})
if full_forbidden_records:
    raise RuntimeError(f'Forbidden raw text remains in final processed dataset:\n{full_forbidden_records}')
NUMERIC_PREDICTORS = ['maternal_age_at_index_birth_years', 'prior_live_births', 'household_size_members']
ORDERED_CATEGORICAL_PREDICTORS = ['maternal_education', 'wealth_quintile']
NOMINAL_CATEGORICAL_PREDICTORS = ['partner_education_final', 'division', 'pregnancy_intention', 'healthcare_decision_final']
BINARY_CATEGORICAL_PREDICTORS = ['residence', 'weekly_media_exposure', 'working_status', 'religion_final', 'household_head_sex', 'distance_barrier']
CATEGORICAL_PREDICTORS = ORDERED_CATEGORICAL_PREDICTORS + NOMINAL_CATEGORICAL_PREDICTORS + BINARY_CATEGORICAL_PREDICTORS
FINAL_PRIMARY_PREDICTORS = NUMERIC_PREDICTORS + CATEGORICAL_PREDICTORS
if len(FINAL_PRIMARY_PREDICTORS) != 15:
    raise RuntimeError('Final predictor list must contain exactly 15 variables.')
if len(set(FINAL_PRIMARY_PREDICTORS)) != 15:
    raise RuntimeError('Duplicate variable in final predictor list.')
missing_predictors = [variable for variable in FINAL_PRIMARY_PREDICTORS if variable not in final_processed.columns]
if missing_predictors:
    raise RuntimeError(f'Final predictor(s) missing:\n{missing_predictors}')
FORBIDDEN_MODEL_FEATURES = {'_ROW_ID', 'CASEID', 'ANC_VISITS', 'ANC_CLASS', 'ANC_CLASS_LABEL', 'ANC_VISITS_RAW', 'SURVEY_WEIGHT', 'PSU', 'STRATUM', 'V005', 'V021', 'V023', 'INDEX_MATERNITY_SLOT', 'M80_SELECTED_VALUE', 'SELECTED_MIDX', 'MATCHED_BIDX', 'INDEX_BIRTH_CMC'}
leakage_overlap = sorted(set(FINAL_PRIMARY_PREDICTORS) & FORBIDDEN_MODEL_FEATURES)
if leakage_overlap:
    raise RuntimeError(f'Forbidden leakage/design variable in X:\n{leakage_overlap}')
ADMIN_COLUMNS = ['_ROW_ID', 'CASEID', 'ANC_VISITS', 'ANC_CLASS', 'ANC_CLASS_LABEL', 'SURVEY_WEIGHT', 'PSU', 'STRATUM']
FINAL_MASTER_COLUMNS = ADMIN_COLUMNS + FINAL_PRIMARY_PREDICTORS
final_model_master = final_processed[FINAL_MASTER_COLUMNS].copy().reset_index(drop=True)
if final_model_master.shape != (5063, 23):
    raise RuntimeError(f'Unexpected final model-master shape:\n{final_model_master.shape}')
master_missing_by_variable = final_model_master.isna().sum().rename('N_Missing').reset_index().rename(columns={'index': 'Variable'})
master_total_missing = int(master_missing_by_variable['N_Missing'].sum())
if master_total_missing != 0:
    raise RuntimeError('Final model master contains missing values.')
master_nonfinite = []
for column in final_model_master.select_dtypes(include=[np.number]).columns:
    values = final_model_master[column].to_numpy(dtype=float)
    n_bad = int((~np.isfinite(values)).sum())
    if n_bad:
        master_nonfinite.append({'Variable': column, 'N': n_bad})
if master_nonfinite:
    raise RuntimeError(f'Final model master contains non-finite values:\n{master_nonfinite}')
master_string_columns = final_model_master.select_dtypes(include=['object', 'string']).columns.tolist()
master_forbidden_records = []
for column in master_string_columns:
    normalized = final_model_master[column].astype(str).str.strip().str.lower()
    bad = normalized.isin(FORBIDDEN_EXACT_TEXT)
    if bad.any():
        counts = normalized[bad].value_counts()
        for value, count in counts.items():
            master_forbidden_records.append({'Variable': column, 'Value': value, 'N': int(count)})
if master_forbidden_records:
    raise RuntimeError(f'Forbidden raw text in final model master:\n{master_forbidden_records}')
EXPECTED_OUTCOME_COUNTS = {0: 403, 1: 4407, 2: 253}
observed_outcome_counts = final_model_master['ANC_CLASS'].value_counts().sort_index().to_dict()
if observed_outcome_counts != EXPECTED_OUTCOME_COUNTS:
    raise RuntimeError(f'Outcome counts changed:\n{observed_outcome_counts}')
if final_model_master['PSU'].nunique() != 672:
    raise RuntimeError('PSU count changed.')
if final_model_master['STRATUM'].nunique() != 16:
    raise RuntimeError('Stratum count changed.')
if (final_model_master['SURVEY_WEIGHT'] <= 0).any():
    raise RuntimeError('Non-positive survey weight detected.')
if final_model_master['_ROW_ID'].duplicated().any():
    raise RuntimeError('Duplicate _ROW_ID.')
if final_model_master['CASEID'].duplicated().any():
    raise RuntimeError('Duplicate CASEID.')
protocol_records = [{'Predictor': 'maternal_age_at_index_birth_years', 'Raw_Source': 'V011 + exact MIDX/BIDX/B3 linkage', 'Variable_Type': 'Continuous numeric', 'Final_Definition': '(index-birth B3 CMC - maternal V011 CMC) / 12', 'Primary_Model_Representation': 'Exact continuous value', 'Temporal_Status': 'Index-pregnancy anchored', 'Reference_Key': 'DHS_RECODE; PREDICTION_METHODS'}, {'Predictor': 'prior_live_births', 'Raw_Source': 'V201 + index-pregnancy multiplicity', 'Variable_Type': 'Count numeric', 'Final_Definition': 'Total live births minus live-born children belonging to index pregnancy', 'Primary_Model_Representation': 'Exact count', 'Temporal_Status': 'Index-pregnancy anchored', 'Reference_Key': 'DHS_RECODE; PREDICTION_METHODS'}, {'Predictor': 'household_size_members', 'Raw_Source': 'V136', 'Variable_Type': 'Count numeric', 'Final_Definition': 'Number of household members listed', 'Primary_Model_Representation': 'Exact count; no primary dichotomization', 'Temporal_Status': 'Survey-time household status', 'Reference_Key': 'DHS_RECODE; PREDICTION_METHODS'}, {'Predictor': 'maternal_education', 'Raw_Source': 'V106', 'Variable_Type': 'Ordered categorical', 'Final_Definition': 'No education / Primary / Secondary / Higher', 'Primary_Model_Representation': 'Categorical; no integer-distance assumption', 'Temporal_Status': 'Survey-time attainment; may post-date index pregnancy', 'Reference_Key': 'DHS_RECODE'}, {'Predictor': 'wealth_quintile', 'Raw_Source': 'V190', 'Variable_Type': 'Ordered categorical', 'Final_Definition': 'Poorest / Poorer / Middle / Richer / Richest', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time household status', 'Reference_Key': 'DHS_RECODE'}, {'Predictor': 'partner_education_final', 'Raw_Source': 'V701 + V502', 'Variable_Type': 'Nominal categorical', 'Final_Definition': 'Four education levels + explicit current-partner structural N/A + not-reported category', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time current-partner characteristic; not guaranteed to represent index-pregnancy partner', 'Reference_Key': 'DHS_RECODE'}, {'Predictor': 'residence', 'Raw_Source': 'V025', 'Variable_Type': 'Binary categorical', 'Final_Definition': 'Urban / Rural', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time residence', 'Reference_Key': 'DHS_RECODE; BDHS2022'}, {'Predictor': 'division', 'Raw_Source': 'V024', 'Variable_Type': 'Nominal categorical', 'Final_Definition': 'Eight Bangladesh divisions', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time residence', 'Reference_Key': 'BDHS2022'}, {'Predictor': 'weekly_media_exposure', 'Raw_Source': 'V157 + V158 + V159', 'Variable_Type': 'Binary categorical', 'Final_Definition': 'Yes if >=weekly exposure to at least one of newspaper/magazine, radio, television; DHS weekly codes 2 or 3', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time media exposure', 'Reference_Key': 'DHS_GUIDE_MEDIA'}, {'Predictor': 'working_status', 'Raw_Source': 'V714', 'Variable_Type': 'Binary categorical', 'Final_Definition': 'Currently working Yes / No', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time current status', 'Reference_Key': 'DHS_RECODE'}, {'Predictor': 'pregnancy_intention', 'Raw_Source': 'Exact slot-matched M10', 'Variable_Type': 'Nominal categorical', 'Final_Definition': 'Wanted then / Wanted later / Wanted no more', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Index-pregnancy anchored', 'Reference_Key': 'DHS_RECODE'}, {'Predictor': 'religion_final', 'Raw_Source': 'V130', 'Variable_Type': 'Binary categorical', 'Final_Definition': 'Islam / Non-Islam primary; original four-category religion retained for sensitivity analysis', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time characteristic; generally stable but not guaranteed pregnancy-time measure', 'Reference_Key': 'BDHS_ANC_LITERATURE; STUDY_PROTOCOL'}, {'Predictor': 'healthcare_decision_final', 'Raw_Source': 'V743A + V502', 'Variable_Type': 'Nominal categorical', 'Final_Definition': 'Preserve DHS source decision categories; structural N/A for V502 != 1', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time current-union decision variable; not index-pregnancy-specific', 'Reference_Key': 'DHS_RECODE; DHS_GUIDE_DECISION'}, {'Predictor': 'household_head_sex', 'Raw_Source': 'V151', 'Variable_Type': 'Binary categorical', 'Final_Definition': 'Male / Female', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time household status', 'Reference_Key': 'DHS_RECODE'}, {'Predictor': 'distance_barrier', 'Raw_Source': 'V467D', 'Variable_Type': 'Binary categorical', 'Final_Definition': 'Big problem / Not a big problem', 'Primary_Model_Representation': 'Categorical', 'Temporal_Status': 'Survey-time perceived access barrier', 'Reference_Key': 'DHS_GUIDE_ACCESS'}]
final_variable_protocol = pd.DataFrame(protocol_records)
if set(final_variable_protocol['Predictor']) != set(FINAL_PRIMARY_PREDICTORS):
    raise RuntimeError('Variable protocol does not exactly match the final 15 predictors.')
model_preprocessing_protocol = pd.DataFrame([{'Model_Family': 'ElasticNet logistic regression', 'Numeric_Handling': 'StandardScaler fitted inside each training fold', 'Categorical_Handling': 'One-hot encoding fitted inside each training fold; handle unseen levels safely', 'Notes': 'No preprocessing fitted on held-out data'}, {'Model_Family': 'RBF SVM', 'Numeric_Handling': 'StandardScaler fitted inside each training fold', 'Categorical_Handling': 'One-hot encoding fitted inside each training fold', 'Notes': 'Scaling required for distance-based kernel'}, {'Model_Family': 'RandomForest / ExtraTrees / AdaBoost / GradientBoosting / HistGradientBoosting / XGBoost / LightGBM', 'Numeric_Handling': 'Numeric passthrough; no mandatory scaling', 'Categorical_Handling': 'Fold-fitted one-hot representation in the generic pipeline', 'Notes': 'No outcome-guided category collapsing'}, {'Model_Family': 'CatBoost', 'Numeric_Handling': 'Native numeric', 'Categorical_Handling': 'Native categorical features', 'Notes': 'Do not externally one-hot encode CatBoost categorical features'}, {'Model_Family': 'Survey-weighted logistic comparator', 'Numeric_Handling': 'Retain continuous/count form; later use pre-specified flexible functional form if needed rather than arbitrary cut-points', 'Categorical_Handling': 'Pre-specified reference coding inside training folds', 'Notes': 'Same outer validation/calibration framework as ML models'}])
outcome_protocol = pd.DataFrame([{'ANC_CLASS': 0, 'Label': 'No ANC', 'M14_Definition': '0 reported ANC visits', 'Interpretation': 'No reported ANC visits'}, {'ANC_CLASS': 1, 'Label': '1-7 ANC visits', 'M14_Definition': '1 through 7 reported ANC visits', 'Interpretation': 'Intermediate visit-count category'}, {'ANC_CLASS': 2, 'Label': '>=8 ANC visits', 'M14_Definition': '8 or more reported ANC visits', 'Interpretation': 'Pragmatic visit-count proxy relative to WHO eight-contact recommendation; NOT a direct measurement of eight WHO contacts'}])
reference_registry = pd.DataFrame([{'Reference_Key': 'DHS_RECODE', 'Citation': 'The DHS Program. DHS VII Standard Recode Manual. DHSG4. 2018.', 'Used_For': 'M10, M14, MIDX/BIDX/B3, V136, V151, V502, V701, V714, V743A and other standard recode definitions'}, {'Reference_Key': 'DHS_GUIDE_MEDIA', 'Citation': 'The DHS Program. Guide to DHS Statistics, Exposure to Mass Media.', 'Used_For': 'Weekly exposure definition: V157/V158/V159 codes 2 or 3'}, {'Reference_Key': 'DHS_GUIDE_DECISION', 'Citation': 'The DHS Program. Guide to DHS Statistics, Participation in Decision Making.', 'Used_For': 'V743A definition and currently-married population base'}, {'Reference_Key': 'DHS_GUIDE_ACCESS', 'Citation': 'The DHS Program. Guide to DHS Statistics, Problems in Accessing Health Care.', 'Used_For': 'V467D distance-to-facility barrier'}, {'Reference_Key': 'BDHS2022', 'Citation': 'NIPORT and ICF. Bangladesh Demographic and Health Survey 2022: Final Report. FR386.', 'Used_For': 'Bangladesh survey design and contextual variable interpretation'}, {'Reference_Key': 'WHO_ANC2016', 'Citation': 'World Health Organization. WHO recommendations on antenatal care for a positive pregnancy experience. 2016.', 'Used_For': 'Eight-contact ANC recommendation and visit-versus-contact interpretation'}, {'Reference_Key': 'PREDICTION_METHODS', 'Citation': 'Royston P, Altman DG, Sauerbrei W. Dichotomizing continuous predictors in multiple regression: a bad idea. Stat Med. 2006;25:127-141. doi:10.1002/sim.2331; and contemporary BMJ prediction-model methodological guidance.', 'Used_For': 'Avoid arbitrary categorization of exact continuous/count predictors'}, {'Reference_Key': 'PROBAST_AI', 'Citation': 'Moons KGM et al. PROBAST+AI. BMJ. 2025;388:e082505.', 'Used_For': 'Prediction-model development/evaluation quality and preprocessing transparency'}, {'Reference_Key': 'BDHS_ANC_LITERATURE', 'Citation': 'Salam MA, Islam MM, Karim MR. PLOS ONE. 2025;20:e0324226; Barna SD et al. PLOS ONE. 2025.', 'Used_For': 'Bangladesh-specific ANC predictor context and comparison with prior coding choices'}, {'Reference_Key': 'STUDY_PROTOCOL', 'Citation': 'BDHS ANC Q1 Final Rebuild 2026 pre-specified analysis protocol.', 'Used_For': 'Study-specific primary/sensitivity representation decisions'}])
source_audit_parts = []
for variable in REFERENCE_CHECK_VARIABLES:
    temp = pd.DataFrame({'Raw_Value': audit[variable], 'Value_Label': audit[variable + '_LABEL']})
    temp['Value_Label'] = temp['Value_Label'].where(temp['Value_Label'].notna(), '<RAW MISSING / NOT APPLICABLE>')
    grouped = temp.groupby(['Raw_Value', 'Value_Label'], dropna=False, observed=True).size().reset_index(name='N')
    grouped['Variable'] = variable
    grouped['Variable_Label'] = VARIABLE_LABELS.get(variable)
    source_audit_parts.append(grouped[['Variable', 'Variable_Label', 'Raw_Value', 'Value_Label', 'N']])
reference_source_audit = pd.concat(source_audit_parts, ignore_index=True)
numeric_summary = final_model_master[NUMERIC_PREDICTORS].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T.reset_index().rename(columns={'index': 'Predictor'})
categorical_distribution_parts = []
for predictor in CATEGORICAL_PREDICTORS:
    temp = final_model_master.groupby(predictor, dropna=False, observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index().rename(columns={predictor: 'Category'})
    temp['Predictor'] = predictor
    temp['Percent'] = temp['N'] / len(final_model_master) * 100
    temp['Weighted_Percent'] = temp['Weighted_N'] / temp['Weighted_N'].sum() * 100
    categorical_distribution_parts.append(temp[['Predictor', 'Category', 'N', 'Weighted_N', 'Percent', 'Weighted_Percent']])
categorical_distributions = pd.concat(categorical_distribution_parts, ignore_index=True)
rare_level_audit = categorical_distributions.copy()
rare_level_audit['N_LT_10'] = rare_level_audit['N'] < 10
rare_level_audit['N_LT_20'] = rare_level_audit['N'] < 20
rare_level_audit = rare_level_audit.loc[rare_level_audit['N_LT_20']].reset_index(drop=True)
final_qc = pd.DataFrame({'Check': ['Rows', 'Model-master columns', 'Primary predictors', 'Numeric predictors', 'Categorical predictors', 'Master missing / NaN cells', 'Full dataset missing / NaN cells', 'Master non-finite cells', 'Full dataset non-finite cells', 'Master forbidden raw-text cells', 'Full forbidden raw-text cells', 'Duplicate _ROW_ID', 'Duplicate CASEID', 'Leakage variables in X', 'V502 vs prior partner-status mismatches', 'Weekly-media reconstruction mismatches', 'Religion reconstruction mismatches', 'Outcome class 0', 'Outcome class 1', 'Outcome class 2', 'Unique PSU', 'Unique strata', 'Non-positive survey weights'], 'Observed': [len(final_model_master), final_model_master.shape[1], len(FINAL_PRIMARY_PREDICTORS), len(NUMERIC_PREDICTORS), len(CATEGORICAL_PREDICTORS), master_total_missing, full_total_missing, 0, 0, 0, 0, int(final_model_master['_ROW_ID'].duplicated().sum()), int(final_model_master['CASEID'].duplicated().sum()), len(leakage_overlap), n_partner_status_mismatch, n_media_mismatch, n_religion_mismatch, observed_outcome_counts[0], observed_outcome_counts[1], observed_outcome_counts[2], final_model_master['PSU'].nunique(), final_model_master['STRATUM'].nunique(), int((final_model_master['SURVEY_WEIGHT'] <= 0).sum())], 'Expected': [5063, 23, 15, 3, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 403, 4407, 253, 672, 16, 0]})
final_qc['PASS'] = final_qc['Observed'] == final_qc['Expected']
if not final_qc['PASS'].all():
    raise RuntimeError('STEP 11 FINAL HARD QC FAILED.')
FINAL_PROCESSED_FILE = DIRS['processed'] / 'FINAL_processed_analytic_dataset_REFERENCE_LOCKED.csv'
FINAL_MODEL_MASTER_FILE = DIRS['processed'] / 'FINAL_model_master_15_predictors_REFERENCE_LOCKED.csv'
final_processed.to_csv(FINAL_PROCESSED_FILE, index=False)
final_model_master.to_csv(FINAL_MODEL_MASTER_FILE, index=False)
FINAL_VARIABLE_PROTOCOL_FILE = DIRS['protocol'] / 'FINAL_variable_definition_recode_temporal_protocol.csv'
FINAL_MODEL_PREPROCESSING_FILE = DIRS['protocol'] / 'FINAL_model_specific_preprocessing_protocol.csv'
FINAL_REFERENCE_REGISTRY_FILE = DIRS['protocol'] / 'FINAL_reference_registry.csv'
FINAL_OUTCOME_PROTOCOL_FILE = DIRS['protocol'] / 'FINAL_ANC_outcome_interpretation_protocol.csv'
STEP11_SOURCE_AUDIT_FILE = DIRS['qc'] / 'STEP11_reference_source_audit.csv'
STEP11_NUMERIC_SUMMARY_FILE = DIRS['qc'] / 'STEP11_numeric_predictor_summary.csv'
STEP11_CATEGORY_DIST_FILE = DIRS['qc'] / 'STEP11_categorical_predictor_distributions.csv'
STEP11_RARE_LEVEL_FILE = DIRS['qc'] / 'STEP11_rare_category_audit.csv'
STEP11_QC_FILE = DIRS['qc'] / 'STEP11_final_reference_locked_qc.csv'
FINAL_VARIABLE_PROTOCOL_FILE.parent.mkdir(parents=True, exist_ok=True)
final_variable_protocol.to_csv(FINAL_VARIABLE_PROTOCOL_FILE, index=False)
model_preprocessing_protocol.to_csv(FINAL_MODEL_PREPROCESSING_FILE, index=False)
reference_registry.to_csv(FINAL_REFERENCE_REGISTRY_FILE, index=False)
outcome_protocol.to_csv(FINAL_OUTCOME_PROTOCOL_FILE, index=False)
reference_source_audit.to_csv(STEP11_SOURCE_AUDIT_FILE, index=False)
numeric_summary.to_csv(STEP11_NUMERIC_SUMMARY_FILE, index=False)
categorical_distributions.to_csv(STEP11_CATEGORY_DIST_FILE, index=False)
rare_level_audit.to_csv(STEP11_RARE_LEVEL_FILE, index=False)
final_qc.to_csv(STEP11_QC_FILE, index=False)
FINAL_PROCESSED_SHA256 = sha256_file(FINAL_PROCESSED_FILE)
FINAL_MODEL_MASTER_SHA256 = sha256_file(FINAL_MODEL_MASTER_FILE)
MODEL_PURPOSE = 'Retrospective survey-based classification of index-pregnancy ANC utilization using a mixture of index-pregnancy-anchored and survey-time characteristics. This dataset is not claimed to be a prospectively validated pregnancy-onset risk model.'
TEMPORAL_SENSITIVITY_PLAN = 'Later repeat locked evaluation in a recent-birth subgroup and explicitly assess dependence on survey-time predictors.'
step11_manifest = {'step': 'STEP 11', 'status': 'REFERENCE_LOCKED_FINAL_PREPROCESSING', 'model_purpose': MODEL_PURPOSE, 'temporal_sensitivity_plan': TEMPORAL_SENSITIVITY_PLAN, 'analytic_n': int(len(final_model_master)), 'final_primary_predictors': FINAL_PRIMARY_PREDICTORS, 'numeric_predictors': NUMERIC_PREDICTORS, 'ordered_categorical_predictors': ORDERED_CATEGORICAL_PREDICTORS, 'nominal_categorical_predictors': NOMINAL_CATEGORICAL_PREDICTORS, 'binary_categorical_predictors': BINARY_CATEGORICAL_PREDICTORS, 'religion_primary': 'Islam vs Non-Islam', 'religion_sensitivity': 'Islam / Hinduism / Buddhist / Christianity', 'V502_used_for_structural_partner_applicability': True, 'V743A_categories_preserved_without_husband_alone_collapse': True, 'continuous_predictors_arbitrarily_dichotomized': False, 'master_stage_encoding': False, 'imputation_performed': False, 'rows_removed': 0, 'missing_cells_model_master': master_total_missing, 'missing_cells_full_processed': full_total_missing, 'leakage_predictors': leakage_overlap, 'final_processed_sha256': FINAL_PROCESSED_SHA256, 'final_model_master_sha256': FINAL_MODEL_MASTER_SHA256, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP11_MANIFEST_FILE = DIRS['protocol'] / 'STEP11_FINAL_reference_locked_manifest.json'
with open(STEP11_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step11_manifest, f, indent=4, ensure_ascii=False, default=str)
if 'logger' in globals():
    logger.info(f'STEP 11 completed successfully: N={len(final_model_master):,}, predictors={len(FINAL_PRIMARY_PREDICTORS)}, missing={master_total_missing}.')
print('\n' + '=' * 78)
print('STEP 11 — LITERATURE / DHS-REFERENCE-LOCKED FINAL PREPROCESSING')
print('=' * 78)
print('\nFINAL HARD QC:')
display(final_qc)
print('\nFINAL VARIABLE DEFINITION / RECODE / TEMPORAL PROTOCOL:')
display(final_variable_protocol)
print('\nMODEL-SPECIFIC PREPROCESSING PROTOCOL:')
display(model_preprocessing_protocol)
print('\nNUMERIC PREDICTOR SUMMARY:')
display(numeric_summary)
print('\nFINAL CATEGORICAL DISTRIBUTIONS:')
display(categorical_distributions)
print('\nRARE CATEGORY AUDIT (N < 20):')
display(rare_level_audit)
print('\nREFERENCE-LOCKED V743A DISTRIBUTION:')
display(categorical_distributions.loc[categorical_distributions['Predictor'] == 'healthcare_decision_final'].reset_index(drop=True))
print('\nREFERENCE-LOCKED PARTNER EDUCATION:')
display(categorical_distributions.loc[categorical_distributions['Predictor'] == 'partner_education_final'].reset_index(drop=True))
print('\nREFERENCE REGISTRY:')
display(reference_registry)
print('\nFINAL PRIMARY PREDICTORS:')
for i, predictor in enumerate(FINAL_PRIMARY_PREDICTORS, start=1):
    variable_type = final_variable_protocol.loc[final_variable_protocol['Predictor'] == predictor, 'Variable_Type'].iloc[0]
    print(f'{i:02d}. {predictor}  -->  {variable_type}')
print('\nFINAL CLEANLINESS:')
print(f'Model-master rows           : {len(final_model_master):,}')
print(f'Model-master columns        : {final_model_master.shape[1]}')
print(f'Primary predictors          : {len(FINAL_PRIMARY_PREDICTORS)}')
print(f'Numeric predictors          : {len(NUMERIC_PREDICTORS)}')
print(f'Categorical predictors      : {len(CATEGORICAL_PREDICTORS)}')
print(f'Model-master missing / NaN  : {master_total_missing:,}')
print(f'Full processed missing/NaN  : {full_total_missing:,}')
print('Non-finite cells           : 0')
print("Raw don't-know/unknown     : 0")
print(f'Leakage predictors          : {len(leakage_overlap)}')
print(f'V502 status mismatches      : {n_partner_status_mismatch}')
print(f'Media reconstruction diff   : {n_media_mismatch}')
print(f'Religion reconstruction diff: {n_religion_mismatch}')
print('\nFINAL PROCESSED ANALYTIC DATASET:')
print(FINAL_PROCESSED_FILE)
print('\nFINAL MODEL MASTER:')
print(FINAL_MODEL_MASTER_FILE)
print('\nFINAL PROCESSED SHA-256:')
print(FINAL_PROCESSED_SHA256)
print('\nFINAL MODEL MASTER SHA-256:')
print(FINAL_MODEL_MASTER_SHA256)
print('\nMODEL PURPOSE:')
print(MODEL_PURPOSE)
print('\nIMPORTANT:')
print('- Maternal age is exact age AT the index birth.')
print('- Prior live births is an exact multiple-birth-aware count.')
print('- Household size remains the exact V136 count in the primary model.')
print('- Grouped age/parity/household-size fields are descriptive/sensitivity only.')
print('- V502 is now the authoritative current-union applicability check.')
print('- V743A source categories are preserved; Husband/partner alone is NOT merged into Other.')
print('- Weekly media exposure was re-derived using DHS weekly codes 2 or 3.')
print('- Religion is Islam vs Non-Islam in the primary model; four-category religion is retained for sensitivity.')
print('- No encoding or scaling has been performed on the final master.')
print('- All model-dependent preprocessing will be fitted inside CV folds.')
print('- Survey-time predictors are explicitly documented as such; this is not presented as a prospectively validated pregnancy-onset model.')
print('=' * 78)
print('STATUS: STEP 11 PASS')
print('=' * 78)


2026-09-16 21:06:10,950 | INFO | STEP 11 completed successfully: N=5,063, predictors=15, missing=0.



STEP 11 — LITERATURE / DHS-REFERENCE-LOCKED FINAL PREPROCESSING

FINAL HARD QC:


,Check,Observed,Expected,PASS
0,Rows,5063,5063,True
1,Model-master columns,23,23,True
2,Primary predictors,15,15,True
3,Numeric predictors,3,3,True
4,Categorical predictors,12,12,True
5,Master missing / NaN cells,0,0,True
6,Full dataset missing / NaN cells,0,0,True
7,Master non-finite cells,0,0,True
8,Full dataset non-finite cells,0,0,True
9,Master forbidden raw-text cells,0,0,True



FINAL VARIABLE DEFINITION / RECODE / TEMPORAL PROTOCOL:


,Predictor,Raw_Source,Variable_Type,Final_Definition,Primary_Model_Representation,Temporal_Status,Reference_Key
0,maternal_age_at_index_birth_years,V011 + exact MIDX/BIDX/B3 linkage,Continuous numeric,(index-birth B3 CMC - maternal V011 CMC) / 12,Exact continuous value,Index-pregnancy anchored,DHS_RECODE; PREDICTION_METHODS
1,prior_live_births,V201 + index-pregnancy multiplicity,Count numeric,Total live births minus live-born children bel...,Exact count,Index-pregnancy anchored,DHS_RECODE; PREDICTION_METHODS
2,household_size_members,V136,Count numeric,Number of household members listed,Exact count; no primary dichotomization,Survey-time household status,DHS_RECODE; PREDICTION_METHODS
3,maternal_education,V106,Ordered categorical,No education / Primary / Secondary / Higher,Categorical; no integer-distance assumption,Survey-time attainment; may post-date index pr...,DHS_RECODE
4,wealth_quintile,V190,Ordered categorical,Poorest / Poorer / Middle / Richer / Richest,Categorical,Survey-time household status,DHS_RECODE
5,partner_education_final,V701 + V502,Nominal categorical,Four education levels + explicit current-partn...,Categorical,Survey-time current-partner characteristic; no...,DHS_RECODE
6,residence,V025,Binary categorical,Urban / Rural,Categorical,Survey-time residence,DHS_RECODE; BDHS2022
7,division,V024,Nominal categorical,Eight Bangladesh divisions,Categorical,Survey-time residence,BDHS2022
8,weekly_media_exposure,V157 + V158 + V159,Binary categorical,Yes if >=weekly exposure to at least one of ne...,Categorical,Survey-time media exposure,DHS_GUIDE_MEDIA
9,working_status,V714,Binary categorical,Currently working Yes / No,Categorical,Survey-time current status,DHS_RECODE



MODEL-SPECIFIC PREPROCESSING PROTOCOL:


... (10 outputs left out)

,Predictor,Category,N,Weighted_N,Percent,Weighted_Percent
0,partner_education_final,Higher,1037,995.546319,20.481928,19.417269
1,partner_education_final,No current partner / not applicable,47,51.750796,0.928303,1.009354
2,partner_education_final,No education,741,763.063330,14.635592,14.882889
3,partner_education_final,Partner education not reported,9,11.439631,0.177760,0.223120
4,partner_education_final,Primary,1495,1509.697260,29.527948,29.445337
5,partner_education_final,Secondary,1734,1795.620915,34.248469,35.022030



REFERENCE REGISTRY:


,Reference_Key,Citation,Used_For
0,DHS_RECODE,The DHS Program. DHS VII Standard Recode Manua...,"M10, M14, MIDX/BIDX/B3, V136, V151, V502, V701..."
1,DHS_GUIDE_MEDIA,"The DHS Program. Guide to DHS Statistics, Expo...",Weekly exposure definition: V157/V158/V159 cod...
2,DHS_GUIDE_DECISION,"The DHS Program. Guide to DHS Statistics, Part...",V743A definition and currently-married populat...
3,DHS_GUIDE_ACCESS,"The DHS Program. Guide to DHS Statistics, Prob...",V467D distance-to-facility barrier
4,BDHS2022,NIPORT and ICF. Bangladesh Demographic and Hea...,Bangladesh survey design and contextual variab...
5,WHO_ANC2016,World Health Organization. WHO recommendations...,Eight-contact ANC recommendation and visit-ver...
6,PREDICTION_METHODS,"Royston P, Altman DG, Sauerbrei W. Dichotomizi...",Avoid arbitrary categorization of exact contin...
7,PROBAST_AI,Moons KGM et al. PROBAST+AI. BMJ. 2025;388:e08...,Prediction-model development/evaluation qualit...
8,BDHS_ANC_LITERATURE,"Salam MA, Islam MM, Karim MR. PLOS ONE. 2025;2...",Bangladesh-specific ANC predictor context and ...
9,STUDY_PROTOCOL,BDHS ANC Q1 Final Rebuild 2026 pre-specified a...,Study-specific primary/sensitivity representat...



FINAL PRIMARY PREDICTORS:
01. maternal_age_at_index_birth_years  -->  Continuous numeric
02. prior_live_births  -->  Count numeric
03. household_size_members  -->  Count numeric
04. maternal_education  -->  Ordered categorical
05. wealth_quintile  -->  Ordered categorical
06. partner_education_final  -->  Nominal categorical
07. division  -->  Nominal categorical
08. pregnancy_intention  -->  Nominal categorical
09. healthcare_decision_final  -->  Nominal categorical
10. residence  -->  Binary categorical
11. weekly_media_exposure  -->  Binary categorical
12. working_status  -->  Binary categorical
13. religion_final  -->  Binary categorical
14. household_head_sex  -->  Binary categorical
15. distance_barrier  -->  Binary categorical

FINAL CLEANLINESS:
Model-master rows           : 5,063
Model-master columns        : 23
Primary predictors          : 15
Numeric predictors          : 3
Categorical predictors      : 12
Model-master missing / NaN  : 0
Full processed missing/NaN  : 0
Non-

In [12]:
# Step 12B: Literature-aligned primary coding and sensitivity masters.

import json
import hashlib
import datetime as dt
import numpy as np
import pandas as pd
from IPython.display import display

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
STEP11_FULL_FILE = DIRS['processed'] / 'FINAL_processed_analytic_dataset_REFERENCE_LOCKED.csv'
EXPECTED_STEP11_FULL_SHA256 = '98bbc2ba1b9b70e4f25492739d8ad58b7ed29bd978796502c6a0f766c0d8415b'
if not STEP11_FULL_FILE.exists():
    raise FileNotFoundError(f'Missing Step-11 source:\n{STEP11_FULL_FILE}')
if sha256_file(STEP11_FULL_FILE) != EXPECTED_STEP11_FULL_SHA256:
    raise RuntimeError('Step-11 source hash mismatch.')
full = pd.read_csv(STEP11_FULL_FILE)
if len(full) != 5063:
    raise RuntimeError(f'Unexpected N: {len(full):,}')
if full['_ROW_ID'].duplicated().any():
    raise RuntimeError('Duplicate _ROW_ID.')
if full['CASEID'].duplicated().any():
    raise RuntimeError('Duplicate CASEID.')
EXPECTED_PARTNER_SOURCE_LEVELS = {'No education', 'Primary', 'Secondary', 'Higher', 'No current partner / not applicable', 'Partner education not reported'}
observed_partner_levels = set(full['partner_education_final'].astype(str).str.strip().unique())
if observed_partner_levels != EXPECTED_PARTNER_SOURCE_LEVELS:
    raise RuntimeError(f'Unexpected partner-education levels:\n{observed_partner_levels}')
PARTNER_PRIMARY_MAP = {'No education': 'No education', 'Primary': 'Primary', 'Secondary': 'Secondary', 'Higher': 'Higher', 'No current partner / not applicable': 'No current partner; partner education unavailable', 'Partner education not reported': 'Partner education not reported'}
full['partner_education_primary_final'] = full['partner_education_final'].astype(str).str.strip().map(PARTNER_PRIMARY_MAP)
if full['partner_education_primary_final'].isna().any():
    raise RuntimeError('Primary partner education recode failed.')
PARTNER_POOLED_MAP = {'No education': 'No education', 'Primary': 'Primary', 'Secondary': 'Secondary', 'Higher': 'Higher', 'No current partner / not applicable': 'Partner education unavailable / not applicable', 'Partner education not reported': 'Partner education unavailable / not applicable'}
full['partner_education_pooled_sensitivity'] = full['partner_education_final'].astype(str).str.strip().map(PARTNER_POOLED_MAP)
EXPECTED_HEALTHCARE_SOURCE_LEVELS = {'Respondent alone', 'Respondent and husband/partner', 'Husband/partner alone', 'Someone else', 'Other', 'No current partner / not applicable'}
observed_healthcare_levels = set(full['healthcare_decision_final'].astype(str).str.strip().unique())
if observed_healthcare_levels != EXPECTED_HEALTHCARE_SOURCE_LEVELS:
    raise RuntimeError(f'Unexpected healthcare-decision levels:\n{observed_healthcare_levels}')
HEALTHCARE_PRIMARY_MAP = {'Respondent alone': 'Respondent alone', 'Respondent and husband/partner': 'Respondent and husband/partner', 'Husband/partner alone': 'Husband/partner alone', 'Someone else': 'Other decision-maker', 'Other': 'Other decision-maker', 'No current partner / not applicable': 'No current partner / not applicable'}
full['healthcare_decision_primary_final'] = full['healthcare_decision_final'].astype(str).str.strip().map(HEALTHCARE_PRIMARY_MAP)
if full['healthcare_decision_primary_final'].isna().any():
    raise RuntimeError('Primary healthcare-decision recode failed.')
partner_crosswalk = full.groupby(['partner_education_final', 'partner_education_primary_final'], dropna=False, observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index()
healthcare_crosswalk = full.groupby(['healthcare_decision_final', 'healthcare_decision_primary_final'], dropna=False, observed=True).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index()
n_no_current_partner_education = int((full['partner_education_primary_final'] == 'No current partner; partner education unavailable').sum())
n_partner_not_reported = int((full['partner_education_primary_final'] == 'Partner education not reported').sum())
n_other_decision_maker = int((full['healthcare_decision_primary_final'] == 'Other decision-maker').sum())
if n_no_current_partner_education != 47:
    raise RuntimeError('Expected 47 no-current-partner education-unavailable cases.')
if n_partner_not_reported != 9:
    raise RuntimeError('Expected 9 partner-education not-reported cases.')
if n_other_decision_maker != 174:
    raise RuntimeError('Expected 174 Other decision-maker cases.')
NUMERIC_PREDICTORS = ['maternal_age_at_index_birth_years', 'prior_live_births', 'household_size_members']
ORDERED_CATEGORICAL_PREDICTORS = ['maternal_education', 'wealth_quintile']
NOMINAL_CATEGORICAL_PREDICTORS = ['partner_education_primary_final', 'division', 'pregnancy_intention', 'healthcare_decision_primary_final']
BINARY_CATEGORICAL_PREDICTORS = ['residence', 'weekly_media_exposure', 'working_status', 'religion_final', 'household_head_sex', 'distance_barrier']
CATEGORICAL_PREDICTORS = ORDERED_CATEGORICAL_PREDICTORS + NOMINAL_CATEGORICAL_PREDICTORS + BINARY_CATEGORICAL_PREDICTORS
PRIMARY_PREDICTORS = NUMERIC_PREDICTORS + CATEGORICAL_PREDICTORS
if len(PRIMARY_PREDICTORS) != 15:
    raise RuntimeError('Expected exactly 15 primary predictors.')
if len(set(PRIMARY_PREDICTORS)) != 15:
    raise RuntimeError('Duplicate primary predictor.')
SENSITIVITY_A_PREDICTORS = ['maternal_age_at_index_birth_years', 'prior_live_births', 'household_size_members', 'maternal_education', 'wealth_quintile', 'partner_education_pooled_sensitivity', 'division', 'pregnancy_intention', 'healthcare_decision_primary_final', 'residence', 'weekly_media_exposure', 'working_status', 'religion_final', 'household_head_sex', 'distance_barrier']
SENSITIVITY_B_PREDICTORS = ['maternal_age_at_index_birth_years', 'prior_live_births', 'household_size_members', 'maternal_education', 'wealth_quintile', 'partner_education_primary_final', 'division', 'pregnancy_intention', 'healthcare_decision_final', 'residence', 'weekly_media_exposure', 'working_status', 'religion_final', 'household_head_sex', 'distance_barrier']
if len(SENSITIVITY_A_PREDICTORS) != 15:
    raise RuntimeError('Sensitivity A must contain 15 predictors.')
if len(SENSITIVITY_B_PREDICTORS) != 15:
    raise RuntimeError('Sensitivity B must contain 15 predictors.')
for variable in NUMERIC_PREDICTORS:
    full[variable] = pd.to_numeric(full[variable], errors='coerce')
    if full[variable].isna().any():
        raise RuntimeError(f'{variable} contains missing values.')
    if not np.isfinite(full[variable].to_numpy(dtype=float)).all():
        raise RuntimeError(f'{variable} contains non-finite values.')
for variable in ['prior_live_births', 'household_size_members']:
    values = full[variable].to_numpy(dtype=float)
    if not np.allclose(values, np.round(values), rtol=0, atol=1e-10):
        raise RuntimeError(f'{variable} is not integer-like.')
    full[variable] = full[variable].round().astype(np.int64)
ADMIN_COLUMNS = ['_ROW_ID', 'CASEID', 'ANC_VISITS', 'ANC_CLASS', 'ANC_CLASS_LABEL', 'SURVEY_WEIGHT', 'PSU', 'STRATUM']
primary_master = full[ADMIN_COLUMNS + PRIMARY_PREDICTORS].copy().reset_index(drop=True)
sensitivity_a_master = full[ADMIN_COLUMNS + SENSITIVITY_A_PREDICTORS].copy().reset_index(drop=True)
sensitivity_b_master = full[ADMIN_COLUMNS + SENSITIVITY_B_PREDICTORS].copy().reset_index(drop=True)
for name, data in {'PRIMARY': primary_master, 'SENSITIVITY_A': sensitivity_a_master, 'SENSITIVITY_B': sensitivity_b_master}.items():
    if data.shape != (5063, 23):
        raise RuntimeError(f'{name}: unexpected shape {data.shape}')
FORBIDDEN_RAW_TEXT = {'', 'nan', 'none', 'null', 'na', 'n/a', "don't know", 'dont know', 'dk', 'unknown', 'missing', '__invalid__'}

def master_cleanliness(data, predictors, label):
    n_missing = int(data.isna().sum().sum())
    if n_missing != 0:
        raise RuntimeError(f'{label}: missing cells={n_missing}')
    n_nonfinite = 0
    for column in data.select_dtypes(include=[np.number]).columns:
        values = data[column].to_numpy(dtype=float)
        n_nonfinite += int((~np.isfinite(values)).sum())
    if n_nonfinite != 0:
        raise RuntimeError(f'{label}: non-finite cells={n_nonfinite}')
    n_forbidden = 0
    for predictor in predictors:
        if predictor in NUMERIC_PREDICTORS:
            continue
        normalized = data[predictor].astype(str).str.strip().str.lower()
        n_forbidden += int(normalized.isin(FORBIDDEN_RAW_TEXT).sum())
    if n_forbidden != 0:
        raise RuntimeError(f'{label}: forbidden text={n_forbidden}')
    return {'missing': n_missing, 'nonfinite': n_nonfinite, 'forbidden': n_forbidden}
primary_clean = master_cleanliness(primary_master, PRIMARY_PREDICTORS, 'PRIMARY')
sensitivity_a_clean = master_cleanliness(sensitivity_a_master, SENSITIVITY_A_PREDICTORS, 'SENSITIVITY_A')
sensitivity_b_clean = master_cleanliness(sensitivity_b_master, SENSITIVITY_B_PREDICTORS, 'SENSITIVITY_B')
EXPECTED_OUTCOME_COUNTS = {0: 403, 1: 4407, 2: 253}
observed_outcomes = primary_master['ANC_CLASS'].value_counts().sort_index().to_dict()
if observed_outcomes != EXPECTED_OUTCOME_COUNTS:
    raise RuntimeError('Outcome counts changed.')
if primary_master['PSU'].nunique() != 672:
    raise RuntimeError('PSU count changed.')
if primary_master['STRATUM'].nunique() != 16:
    raise RuntimeError('Stratum count changed.')
FORBIDDEN_MODEL_FEATURES = {'_ROW_ID', 'CASEID', 'ANC_VISITS', 'ANC_CLASS', 'ANC_CLASS_LABEL', 'SURVEY_WEIGHT', 'PSU', 'STRATUM'}
primary_leakage = sorted(set(PRIMARY_PREDICTORS) & FORBIDDEN_MODEL_FEATURES)
if primary_leakage:
    raise RuntimeError(f'Leakage predictor(s): {primary_leakage}')
primary_distribution_parts = []
for predictor in CATEGORICAL_PREDICTORS:
    temp = primary_master.groupby(predictor, observed=True, dropna=False).agg(N=('CASEID', 'size'), Weighted_N=('SURVEY_WEIGHT', 'sum')).reset_index().rename(columns={predictor: 'Category'})
    temp['Predictor'] = predictor
    temp['Percent'] = 100.0 * temp['N'] / len(primary_master)
    temp['Weighted_Percent'] = 100.0 * temp['Weighted_N'] / temp['Weighted_N'].sum()
    primary_distribution_parts.append(temp[['Predictor', 'Category', 'N', 'Weighted_N', 'Percent', 'Weighted_Percent']])
primary_distributions = pd.concat(primary_distribution_parts, ignore_index=True)
primary_sparse_audit = primary_distributions.loc[primary_distributions['N'] < 20].copy().reset_index(drop=True)
final_policy = pd.DataFrame([{'Variable': 'Partner education', 'Primary': '6 levels; no-current-partner/source-unavailable and not-reported remain separate', 'Reason': 'DHS V701 concerns current or most recent husband/partner education; structural partner status and response non-reporting are not conceptually equivalent.', 'Sensitivity': 'Pool both unavailable states into one operational availability category.', 'Rows_Deleted': 0, 'Outcome_Used': False}, {'Variable': 'Healthcare decision', 'Primary': 'Respondent alone / respondent+partner / partner alone / other decision-maker / no-current-partner structural N/A', 'Reason': 'Conceptual decision-maker grouping supported by Bangladesh autonomy literature; someone else and generic other represent external decision-makers.', 'Sensitivity': 'Restore original six Step-11 source levels.', 'Rows_Deleted': 0, 'Outcome_Used': False}])
final_qc = pd.DataFrame({'Check': ['Primary rows', 'Primary columns', 'Primary predictors', 'Sensitivity A rows', 'Sensitivity A columns', 'Sensitivity B rows', 'Sensitivity B columns', 'Primary missing', 'Primary non-finite', 'Primary forbidden text', 'Sensitivity A missing', 'Sensitivity B missing', 'Rows removed', 'Outcome-guided recoding', 'Frequency-threshold recoding', 'Dataset-wide scaling', 'Dataset-wide one-hot', 'Partner no-current/source-unavailable N', 'Partner education not-reported N', 'Other decision-maker N', 'Outcome class 0', 'Outcome class 1', 'Outcome class 2', 'Unique PSU', 'Unique strata', 'Leakage predictors'], 'Observed': [len(primary_master), primary_master.shape[1], len(PRIMARY_PREDICTORS), len(sensitivity_a_master), sensitivity_a_master.shape[1], len(sensitivity_b_master), sensitivity_b_master.shape[1], primary_clean['missing'], primary_clean['nonfinite'], primary_clean['forbidden'], sensitivity_a_clean['missing'], sensitivity_b_clean['missing'], 0, 0, 0, 0, 0, n_no_current_partner_education, n_partner_not_reported, n_other_decision_maker, observed_outcomes[0], observed_outcomes[1], observed_outcomes[2], primary_master['PSU'].nunique(), primary_master['STRATUM'].nunique(), len(primary_leakage)], 'Expected': [5063, 23, 15, 5063, 23, 5063, 23, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 47, 9, 174, 403, 4407, 253, 672, 16, 0]})
final_qc['PASS'] = final_qc['Observed'] == final_qc['Expected']
if not final_qc['PASS'].all():
    raise RuntimeError('STEP 12B QC FAILED.')
PRIMARY_FILE = DIRS['processed'] / 'FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv'
SENSITIVITY_A_FILE = DIRS['processed'] / 'FINAL_SENSITIVITY_A_partner_pooling.csv'
SENSITIVITY_B_FILE = DIRS['processed'] / 'FINAL_SENSITIVITY_B_healthcare_source_levels.csv'
FULL_FILE = DIRS['processed'] / 'FINAL_processed_LITERATURE_LOCKED.csv'
PARTNER_CROSSWALK_FILE = DIRS['qc'] / 'STEP12B_partner_education_crosswalk.csv'
HEALTHCARE_CROSSWALK_FILE = DIRS['qc'] / 'STEP12B_healthcare_decision_crosswalk.csv'
PRIMARY_DIST_FILE = DIRS['qc'] / 'STEP12B_primary_category_distributions.csv'
SPARSE_AUDIT_FILE = DIRS['qc'] / 'STEP12B_primary_sparse_level_audit.csv'
FINAL_QC_FILE = DIRS['qc'] / 'STEP12B_final_qc.csv'
POLICY_FILE = DIRS['protocol'] / 'STEP12B_FINAL_literature_locked_recode_policy.csv'
primary_master.to_csv(PRIMARY_FILE, index=False)
sensitivity_a_master.to_csv(SENSITIVITY_A_FILE, index=False)
sensitivity_b_master.to_csv(SENSITIVITY_B_FILE, index=False)
full.to_csv(FULL_FILE, index=False)
partner_crosswalk.to_csv(PARTNER_CROSSWALK_FILE, index=False)
healthcare_crosswalk.to_csv(HEALTHCARE_CROSSWALK_FILE, index=False)
primary_distributions.to_csv(PRIMARY_DIST_FILE, index=False)
primary_sparse_audit.to_csv(SPARSE_AUDIT_FILE, index=False)
final_qc.to_csv(FINAL_QC_FILE, index=False)
final_policy.to_csv(POLICY_FILE, index=False)
PRIMARY_SHA256 = sha256_file(PRIMARY_FILE)
SENSITIVITY_A_SHA256 = sha256_file(SENSITIVITY_A_FILE)
SENSITIVITY_B_SHA256 = sha256_file(SENSITIVITY_B_FILE)
FULL_SHA256 = sha256_file(FULL_FILE)
manifest = {'step': 'STEP 12B', 'status': 'FINAL_LITERATURE_LOCKED_PRE_SPLIT', 'analytic_n': 5063, 'primary_predictors': PRIMARY_PREDICTORS, 'primary_partner_education_levels': 6, 'primary_healthcare_decision_levels': 5, 'partner_not_reported_deleted': False, 'no_current_partner_deleted': False, 'frequency_threshold_used': False, 'outcome_guided_recoding': False, 'cv_split_performed': False, 'scaling_performed': False, 'encoding_performed': False, 'primary_sha256': PRIMARY_SHA256, 'sensitivity_a_sha256': SENSITIVITY_A_SHA256, 'sensitivity_b_sha256': SENSITIVITY_B_SHA256, 'full_sha256': FULL_SHA256, 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
MANIFEST_FILE = DIRS['protocol'] / 'STEP12B_FINAL_literature_locked_manifest.json'
with open(MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False, default=str)
print('\n' + '=' * 78)
print('STEP 12B — FINAL LITERATURE-LOCKED PRE-SPLIT DATASET')
print('=' * 78)
print('\nPARTNER EDUCATION SOURCE -> PRIMARY:')
display(partner_crosswalk)
print('\nHEALTHCARE DECISION SOURCE -> PRIMARY:')
display(healthcare_crosswalk)
print('\nFINAL PRIMARY CATEGORY DISTRIBUTIONS:')
display(primary_distributions)
print('\nPRIMARY SPARSE LEVEL AUDIT (INFORMATIONAL ONLY):')
display(primary_sparse_audit)
print('\nFINAL LITERATURE-BASED POLICY:')
display(final_policy)
print('\nFINAL HARD QC:')
display(final_qc)
print('\nPRIMARY MODEL MASTER:')
print(PRIMARY_FILE)
print('\nPRIMARY SHA-256:')
print(PRIMARY_SHA256)
print('\nSENSITIVITY A SHA-256:')
print(SENSITIVITY_A_SHA256)
print('\nSENSITIVITY B SHA-256:')
print(SENSITIVITY_B_SHA256)
print('\nFULL DATASET SHA-256:')
print(FULL_SHA256)
print('\nIMPORTANT:')
print('- No participant was removed.')
print('- Partner education primary analysis preserves the two unavailable-information mechanisms separately.')
print('- The n=9 Partner education not reported level is retained, not forced into an education level.')
print('- Healthcare decision uses the literature-supported Other decision-maker construct.')
print('- Husband/partner alone remains distinct.')
print('- Source-preserved healthcare decision is retained as a sensitivity analysis.')
print('- Pooled partner-education availability is retained only as a sensitivity analysis.')
print('- No arbitrary N<20 rule is used.')
print('- No scaling, one-hot encoding, imputation or CV splitting has occurred.')
print('=' * 78)
print('STATUS: STEP 12B PASS')
print('=' * 78)



STEP 12B — FINAL LITERATURE-LOCKED PRE-SPLIT DATASET

PARTNER EDUCATION SOURCE -> PRIMARY:


,partner_education_final,partner_education_primary_final,N,Weighted_N
0,Higher,Higher,1037,995.546319
1,No current partner / not applicable,No current partner; partner education unavailable,47,51.750796
2,No education,No education,741,763.063330
3,Partner education not reported,Partner education not reported,9,11.439631
4,Primary,Primary,1495,1509.697260
5,Secondary,Secondary,1734,1795.620915



HEALTHCARE DECISION SOURCE -> PRIMARY:


,healthcare_decision_final,healthcare_decision_primary_final,N,Weighted_N
0,Husband/partner alone,Husband/partner alone,1243,1188.029942
1,No current partner / not applicable,No current partner / not applicable,47,51.750796
2,Other,Other decision-maker,12,11.145541
3,Respondent alone,Respondent alone,431,429.611728
4,Respondent and husband/partner,Respondent and husband/partner,3168,3282.399910
5,Someone else,Other decision-maker,162,164.180334



FINAL PRIMARY CATEGORY DISTRIBUTIONS:


,Predictor,Category,N,Weighted_N,Percent,Weighted_Percent
0,maternal_education,Higher,958,906.525904,18.921588,17.681002
1,maternal_education,No education,266,274.456238,5.253802,5.353031
2,maternal_education,Primary,1173,1160.247762,23.168082,22.629628
3,maternal_education,Secondary,2666,2785.888347,52.656528,54.336339
4,wealth_quintile,Middle,1008,1056.731074,19.909145,20.610624
5,wealth_quintile,Poorer,1020,1068.354627,20.146158,20.837332
6,wealth_quintile,Poorest,1053,1047.763390,20.797946,20.435717
7,wealth_quintile,Richer,997,1017.926301,19.691882,19.853771
8,wealth_quintile,Richest,985,936.342859,19.454869,18.262556
9,partner_education_primary_final,Higher,1037,995.546319,20.481928,19.417269


... (3 outputs left out)

,Variable,Primary,Reason,Sensitivity,Rows_Deleted,Outcome_Used
0,Partner education,6 levels; no-current-partner/source-unavailabl...,DHS V701 concerns current or most recent husba...,Pool both unavailable states into one operatio...,0,False
1,Healthcare decision,Respondent alone / respondent+partner / partne...,Conceptual decision-maker grouping supported b...,Restore original six Step-11 source levels.,0,False



FINAL HARD QC:


,Check,Observed,Expected,PASS
0,Primary rows,5063,5063,True
1,Primary columns,23,23,True
2,Primary predictors,15,15,True
3,Sensitivity A rows,5063,5063,True
4,Sensitivity A columns,23,23,True
5,Sensitivity B rows,5063,5063,True
6,Sensitivity B columns,23,23,True
7,Primary missing,0,0,True
8,Primary non-finite,0,0,True
9,Primary forbidden text,0,0,True



PRIMARY MODEL MASTER:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\01_processed_data\FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv

PRIMARY SHA-256:
0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a

SENSITIVITY A SHA-256:
86c641dd363ea3184942d507d6476d32374d0c8b7209d18a4b9ea61ae5d3f20f

SENSITIVITY B SHA-256:
6473c8cc2b442fce6229cde96f52aeb99a01696c37c7bacc464fc524074dceac

FULL DATASET SHA-256:
7e0844d4585dd2a2a3a7e555c46d8232f4473a8b139ac9129927444b248bc167

IMPORTANT:
- No participant was removed.
- Partner education primary analysis preserves the two unavailable-information mechanisms separately.
- The n=9 Partner education not reported level is retained, not forced into an education level.
- Healthcare decision uses the literature-supported Other decision-maker construct.
- Husband/partner alone remains distinct.
- Source-preserved healthcare decision is retained as a sensitivity analysis.
- Pooled partner-education availability is retained only as a sensitivity analysis.

In [13]:
# Step 12C: Freeze manuscript-ready predictor, missingness, preprocessing, and reporting documentation without modifying data.

import json
import hashlib
import datetime as dt
import numpy as np
import pandas as pd
from IPython.display import display

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
PRIMARY_FILE = DIRS['processed'] / 'FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv'
SENSITIVITY_A_FILE = DIRS['processed'] / 'FINAL_SENSITIVITY_A_partner_pooling.csv'
SENSITIVITY_B_FILE = DIRS['processed'] / 'FINAL_SENSITIVITY_B_healthcare_source_levels.csv'
FULL_FILE = DIRS['processed'] / 'FINAL_processed_LITERATURE_LOCKED.csv'
STEP12B_MANIFEST_FILE = DIRS['protocol'] / 'STEP12B_FINAL_literature_locked_manifest.json'
EXPECTED_PRIMARY_SHA256 = '0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a'
EXPECTED_SENSITIVITY_A_SHA256 = '86c641dd363ea3184942d507d6476d32374d0c8b7209d18a4b9ea61ae5d3f20f'
EXPECTED_SENSITIVITY_B_SHA256 = '6473c8cc2b442fce6229cde96f52aeb99a01696c37c7bacc464fc524074dceac'
EXPECTED_FULL_SHA256 = '7e0844d4585dd2a2a3a7e555c46d8232f4473a8b139ac9129927444b248bc167'
for path in [PRIMARY_FILE, SENSITIVITY_A_FILE, SENSITIVITY_B_FILE, FULL_FILE, STEP12B_MANIFEST_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'Missing Step-12B file:\n{path}')
observed_hashes = {'primary': sha256_file(PRIMARY_FILE), 'sensitivity_a': sha256_file(SENSITIVITY_A_FILE), 'sensitivity_b': sha256_file(SENSITIVITY_B_FILE), 'full': sha256_file(FULL_FILE)}
expected_hashes = {'primary': EXPECTED_PRIMARY_SHA256, 'sensitivity_a': EXPECTED_SENSITIVITY_A_SHA256, 'sensitivity_b': EXPECTED_SENSITIVITY_B_SHA256, 'full': EXPECTED_FULL_SHA256}
for key in expected_hashes:
    if observed_hashes[key] != expected_hashes[key]:
        raise RuntimeError(f'Step-12B {key} hash mismatch.')
with open(STEP12B_MANIFEST_FILE, 'r', encoding='utf-8') as f:
    step12b_manifest = json.load(f)
if step12b_manifest['primary_sha256'] != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError('Step-12B manifest primary hash mismatch.')
primary = pd.read_csv(PRIMARY_FILE)
sensitivity_a = pd.read_csv(SENSITIVITY_A_FILE)
sensitivity_b = pd.read_csv(SENSITIVITY_B_FILE)
full = pd.read_csv(FULL_FILE)
for name, data in {'PRIMARY': primary, 'SENSITIVITY_A': sensitivity_a, 'SENSITIVITY_B': sensitivity_b}.items():
    if data.shape != (5063, 23):
        raise RuntimeError(f'{name}: unexpected shape {data.shape}')
    if data['_ROW_ID'].duplicated().any():
        raise RuntimeError(f'{name}: duplicate _ROW_ID.')
    if data['CASEID'].duplicated().any():
        raise RuntimeError(f'{name}: duplicate CASEID.')
    if data.isna().sum().sum() != 0:
        raise RuntimeError(f'{name}: unresolved NaN detected.')
NUMERIC_PREDICTORS = ['maternal_age_at_index_birth_years', 'prior_live_births', 'household_size_members']
ORDERED_CATEGORICAL_PREDICTORS = ['maternal_education', 'wealth_quintile']
NOMINAL_CATEGORICAL_PREDICTORS = ['partner_education_primary_final', 'division', 'pregnancy_intention', 'healthcare_decision_primary_final']
BINARY_CATEGORICAL_PREDICTORS = ['residence', 'weekly_media_exposure', 'working_status', 'religion_final', 'household_head_sex', 'distance_barrier']
PRIMARY_PREDICTORS = NUMERIC_PREDICTORS + ORDERED_CATEGORICAL_PREDICTORS + NOMINAL_CATEGORICAL_PREDICTORS + BINARY_CATEGORICAL_PREDICTORS
if len(PRIMARY_PREDICTORS) != 15:
    raise RuntimeError('Expected exactly 15 primary predictors.')
if not set(PRIMARY_PREDICTORS).issubset(primary.columns):
    raise RuntimeError('Primary predictor list does not match master.')
variable_dictionary = pd.DataFrame([{'Order': 1, 'Predictor': 'maternal_age_at_index_birth_years', 'DHS_Source': 'V011 + linked B3 using exact MIDX/BIDX linkage', 'Type': 'Continuous numeric', 'Primary_Definition': 'Maternal age in years at index birth: (index-birth CMC - V011 CMC) / 12', 'Missing_or_NA_Handling': 'None required; complete', 'Primary_Representation': 'Exact continuous value', 'Timing': 'Index-pregnancy anchored', 'Sensitivity': 'Grouped age only for descriptive/sensitivity analysis'}, {'Order': 2, 'Predictor': 'prior_live_births', 'DHS_Source': 'V201 + index-pregnancy multiplicity', 'Type': 'Count numeric', 'Primary_Definition': 'Total live births minus live-born children belonging to index pregnancy', 'Missing_or_NA_Handling': 'None required; complete', 'Primary_Representation': 'Exact count', 'Timing': 'Index-pregnancy anchored', 'Sensitivity': 'Grouped parity only descriptive/sensitivity'}, {'Order': 3, 'Predictor': 'household_size_members', 'DHS_Source': 'V136', 'Type': 'Count numeric', 'Primary_Definition': 'Exact number of household members listed', 'Missing_or_NA_Handling': 'None required; complete', 'Primary_Representation': 'Exact count', 'Timing': 'Survey-time', 'Sensitivity': 'Grouped household size only descriptive/sensitivity'}, {'Order': 4, 'Predictor': 'maternal_education', 'DHS_Source': 'V106', 'Type': 'Ordered categorical', 'Primary_Definition': 'No education / Primary / Secondary / Higher', 'Missing_or_NA_Handling': 'None required; complete', 'Primary_Representation': 'Categorical; no equal-distance assumption', 'Timing': 'Survey-time attainment', 'Sensitivity': 'None prespecified'}, {'Order': 5, 'Predictor': 'wealth_quintile', 'DHS_Source': 'V190', 'Type': 'Ordered categorical', 'Primary_Definition': 'Poorest / Poorer / Middle / Richer / Richest', 'Missing_or_NA_Handling': 'None required; complete', 'Primary_Representation': 'Categorical', 'Timing': 'Survey-time', 'Sensitivity': 'None prespecified'}, {'Order': 6, 'Predictor': 'partner_education_primary_final', 'DHS_Source': 'V701 + V502', 'Type': 'Nominal categorical', 'Primary_Definition': 'No education / Primary / Secondary / Higher / No current partner; partner education unavailable / Partner education not reported', 'Missing_or_NA_Handling': 'Structural unavailability and genuine non-reporting represented separately', 'Primary_Representation': 'Six explicit categories', 'Timing': 'Survey-time current-partner characteristic', 'Sensitivity': 'Pool the two unavailable-information mechanisms into one category'}, {'Order': 7, 'Predictor': 'division', 'DHS_Source': 'V024', 'Type': 'Nominal categorical', 'Primary_Definition': 'Eight Bangladesh divisions', 'Missing_or_NA_Handling': 'None required; complete', 'Primary_Representation': 'Eight categories', 'Timing': 'Survey-time residence', 'Sensitivity': 'Excluded during leave-one-division-out validation'}, {'Order': 8, 'Predictor': 'pregnancy_intention', 'DHS_Source': 'Exact maternity-slot M10', 'Type': 'Nominal categorical', 'Primary_Definition': 'Wanted then / Wanted later / Wanted no more', 'Missing_or_NA_Handling': 'None required after exact-slot reconstruction', 'Primary_Representation': 'Three categories', 'Timing': 'Index-pregnancy anchored', 'Sensitivity': 'None prespecified'}, {'Order': 9, 'Predictor': 'healthcare_decision_primary_final', 'DHS_Source': 'V743A + V502', 'Type': 'Nominal categorical', 'Primary_Definition': 'Respondent alone / Respondent and husband-partner / Husband-partner alone / Other decision-maker / No current partner-not applicable', 'Missing_or_NA_Handling': 'V502 != 1 represented as structural N/A; source Someone else + Other grouped conceptually as Other decision-maker', 'Primary_Representation': 'Five conceptual categories', 'Timing': 'Survey-time current-status decision variable', 'Sensitivity': 'Original six-level V743A source representation'}, {'Order': 10, 'Predictor': 'residence', 'DHS_Source': 'V025', 'Type': 'Binary categorical', 'Primary_Definition': 'Urban / Rural', 'Missing_or_NA_Handling': 'None required', 'Primary_Representation': 'Binary categorical', 'Timing': 'Survey-time', 'Sensitivity': 'None prespecified'}, {'Order': 11, 'Predictor': 'weekly_media_exposure', 'DHS_Source': 'V157 + V158 + V159', 'Type': 'Binary categorical', 'Primary_Definition': 'Yes if exposed at least weekly to newspaper/magazine, radio, or television; DHS codes 2 or 3', 'Missing_or_NA_Handling': 'None required', 'Primary_Representation': 'Yes / No', 'Timing': 'Survey-time', 'Sensitivity': 'None prespecified'}, {'Order': 12, 'Predictor': 'working_status', 'DHS_Source': 'V714', 'Type': 'Binary categorical', 'Primary_Definition': 'Currently working Yes / No', 'Missing_or_NA_Handling': 'None required', 'Primary_Representation': 'Binary categorical', 'Timing': 'Survey-time current status', 'Sensitivity': 'None prespecified'}, {'Order': 13, 'Predictor': 'religion_final', 'DHS_Source': 'V130', 'Type': 'Binary categorical', 'Primary_Definition': 'Islam / Non-Islam', 'Missing_or_NA_Handling': 'None required', 'Primary_Representation': 'Binary categorical', 'Timing': 'Survey-time characteristic', 'Sensitivity': 'Original four-category religion retained'}, {'Order': 14, 'Predictor': 'household_head_sex', 'DHS_Source': 'V151', 'Type': 'Binary categorical', 'Primary_Definition': 'Male / Female', 'Missing_or_NA_Handling': 'None required', 'Primary_Representation': 'Binary categorical', 'Timing': 'Survey-time household status', 'Sensitivity': 'None prespecified'}, {'Order': 15, 'Predictor': 'distance_barrier', 'DHS_Source': 'V467D', 'Type': 'Binary categorical', 'Primary_Definition': 'Big problem / Not a big problem', 'Missing_or_NA_Handling': 'None required', 'Primary_Representation': 'Binary categorical', 'Timing': 'Survey-time perceived access barrier', 'Sensitivity': 'None prespecified'}])
if variable_dictionary['Predictor'].tolist() != PRIMARY_PREDICTORS:
    raise RuntimeError('Variable dictionary order differs from final predictor list.')
missingness_policy = pd.DataFrame([{'Situation': 'Literal NaN / blank / non-finite', 'Primary_Action': 'Not permitted in final model master; hard QC failure', 'Row_Deletion': False, 'Interpretation': 'Unresolved machine missingness'}, {'Situation': "Raw DHS don't know / special code", 'Primary_Action': 'Recode according to documented meaning; raw special code/text not retained', 'Row_Deletion': False, 'Interpretation': 'Documented source non-response/special value'}, {'Situation': 'Structural not applicable', 'Primary_Action': 'Represent explicitly as substantive structural category where appropriate', 'Row_Deletion': False, 'Interpretation': 'Question not applicable by survey routing'}, {'Situation': 'Partner education not reported', 'Primary_Action': 'Preserve as explicit primary category', 'Row_Deletion': False, 'Interpretation': 'Genuine non-reporting among applicable respondents'}, {'Situation': 'Numeric predictor missingness', 'Primary_Action': 'None present in current primary master; no imputation performed', 'Row_Deletion': False, 'Interpretation': 'Not applicable to current final dataset'}])
sensitivity_protocol = pd.DataFrame([{'Analysis': 'Primary', 'Partner_Education': 'Six levels; structural unavailable and not-reported remain separate', 'Healthcare_Decision': 'Five conceptual categories; Someone else + Other -> Other decision-maker', 'Purpose': 'Main prespecified analysis'}, {'Analysis': 'Sensitivity A', 'Partner_Education': 'Unavailable/not-applicable and not-reported pooled into one operational category', 'Healthcare_Decision': 'Primary five-category representation', 'Purpose': 'Assess sensitivity to sparse partner-education non-reporting level'}, {'Analysis': 'Sensitivity B', 'Partner_Education': 'Primary six-category representation', 'Healthcare_Decision': 'Original six Step-11 source categories', 'Purpose': 'Assess sensitivity to conceptual pooling of Someone else and Other'}])
preprocessing_timing = pd.DataFrame([{'Operation': 'DHS codebook/value-label recoding', 'When': 'Before resampling', 'Uses_Outcome': False, 'Uses_Validation_Data_Statistics': False, 'Rationale': 'Deterministic source definition'}, {'Operation': 'Concept-driven category definition', 'When': 'Before resampling', 'Uses_Outcome': False, 'Uses_Validation_Data_Statistics': False, 'Rationale': 'Pre-specified substantive interpretation; not chosen from predictive performance'}, {'Operation': 'Numeric standardisation', 'When': 'Inside training folds only', 'Uses_Outcome': False, 'Uses_Validation_Data_Statistics': False, 'Rationale': 'Prevent preprocessing leakage'}, {'Operation': 'One-hot encoder fitting', 'When': 'Inside training folds only', 'Uses_Outcome': False, 'Uses_Validation_Data_Statistics': False, 'Rationale': 'Prevent preprocessing leakage'}, {'Operation': 'Spline-knot estimation', 'When': 'Inside training folds only', 'Uses_Outcome': False, 'Uses_Validation_Data_Statistics': False, 'Rationale': 'Prevent information leakage'}, {'Operation': 'Hyperparameter tuning', 'When': 'Inner CV only', 'Uses_Outcome': True, 'Uses_Validation_Data_Statistics': True, 'Rationale': 'Nested model development'}, {'Operation': 'Calibration fitting', 'When': 'Nested training/inner-CV only', 'Uses_Outcome': True, 'Uses_Validation_Data_Statistics': True, 'Rationale': 'Outer test outcomes must remain untouched until final evaluation'}, {'Operation': 'Threshold selection', 'When': 'Nested training/inner-CV only', 'Uses_Outcome': True, 'Uses_Validation_Data_Statistics': True, 'Rationale': 'Prevent optimistic threshold selection'}])
model_preprocessing = pd.DataFrame([{'Model_Group': 'ElasticNet logistic regression', 'Numeric': 'StandardScaler fitted inside training fold', 'Categorical': "OneHotEncoder(handle_unknown='ignore', drop=None) fitted inside training fold", 'Scaling': True}, {'Model_Group': 'RBF SVM', 'Numeric': 'StandardScaler fitted inside training fold', 'Categorical': "OneHotEncoder(handle_unknown='ignore', drop=None) fitted inside training fold", 'Scaling': True}, {'Model_Group': 'RandomForest / ExtraTrees / AdaBoost / GradientBoosting / HistGradientBoosting / XGBoost / LightGBM', 'Numeric': 'Exact numeric values; no mandatory scaling', 'Categorical': 'Fold-fitted categorical representation', 'Scaling': False}, {'Model_Group': 'CatBoost', 'Numeric': 'Exact numeric values', 'Categorical': 'Native categorical handling', 'Scaling': False}, {'Model_Group': 'Survey-weighted logistic comparator', 'Numeric': 'Continuous/count predictors retained; flexible non-linear form fitted using training-fold information', 'Categorical': 'Pre-specified reference coding; design generated inside training data', 'Scaling': False}])
timing_summary = variable_dictionary.groupby('Timing', observed=True).agg(N_Predictors=('Predictor', 'size'), Predictors=('Predictor', lambda x: ' | '.join(x))).reset_index()
reference_registry = pd.DataFrame([{'Key': 'DHS_RECODE7', 'Reference': 'The DHS Program. DHS VII Standard Recode Manual. DHSG4. 2018.', 'Used_For': 'V701, V714 and standard DHS variable definitions / survey routing.'}, {'Key': 'DHS_GUIDE7', 'Reference': 'The DHS Program. Guide to DHS Statistics, DHS-7, version 2.', 'Used_For': 'Healthcare decision-making population base, media exposure and indicator interpretation.'}, {'Key': 'TRIPOD_AI', 'Reference': 'Collins GS et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. BMJ. 2024;385:e078378. doi:10.1136/bmj-2023-078378.', 'Used_For': 'Transparent reporting of predictors, preprocessing, model development and validation.'}, {'Key': 'PROBAST_AI', 'Reference': 'Moons KGM et al. PROBAST+AI: updated quality, risk of bias, and applicability assessment tool for prediction models. BMJ. 2025;388:e082505. doi:10.1136/bmj-2024-082505.', 'Used_For': 'Predictor preprocessing, applicability, model-development and evaluation safeguards.'}, {'Key': 'BMJ_PREDICTION_GUIDE', 'Reference': 'Riley RD et al. Developing clinical prediction models: a step-by-step guide. BMJ. 2024;386:e078276.', 'Used_For': 'Avoid arbitrary categorisation, missing-data principles and internal validation.'}, {'Key': 'BMJ_CONTINUOUS', 'Reference': 'Lopez-Ayala P et al. Dealing with continuous variables and modelling non-linear associations in healthcare data: practical guide. BMJ. 2025;390:e082440.', 'Used_For': 'Retention of continuous predictors and flexible modelling of non-linearity.'}])
methods_text = "\nPredictor preprocessing\n\nPredictor definitions and deterministic recoding rules were specified\nbefore resampling and model fitting. Fifteen candidate predictors were\nretained in the primary analysis. Maternal age was calculated at the\nindex birth from linked century-month codes and was retained as a\ncontinuous variable. Prior live births were reconstructed using a\nmultiple-birth-aware count, and household size was retained as the\nexact number of household members. These numeric predictors were not\narbitrarily categorised for the primary analysis.\n\nCategorical predictors were reconstructed from documented DHS recode\nvariables and retained using human-readable labels. Structural\nnon-applicability was distinguished from genuine non-reporting. For\npartner education, women for whom the variable was structurally\nunavailable because they had no current partner were represented as\n'No current partner; partner education unavailable', whereas the small\nnumber of applicable respondents whose partner education was not\nreported remained in a separate 'Partner education not reported'\ncategory. No participant was excluded solely because of this\nnon-reporting.\n\nFor healthcare decision-making, respondent-alone, joint\nrespondent-partner, and husband/partner-alone categories were retained\nseparately. The source responses 'Someone else' and 'Other' were grouped\na priori under the broader concept 'Other decision-maker'. This\nconceptual grouping was specified without reference to the outcome,\nmodel performance, or an arbitrary frequency threshold. Respondents\noutside the DHS population base for this current-status item were\nrepresented explicitly as 'No current partner / not applicable'.\nA source-preserved six-category version was retained for sensitivity\nanalysis.\n\nThe final primary modelling dataset contained no unresolved machine\nmissing values, blank fields, non-finite numeric values, or raw DHS\n'don't know' codes. This does not imply that all substantive information\nwas observed: documented structural non-applicability and non-reporting\nwere represented explicitly as prespecified categorical states.\nNo records were deleted because of predictor missingness, and no\nstatistical imputation was required for the primary predictor set.\n\nNo scaling, one-hot encoding, spline fitting, feature selection,\ncalibration, or threshold optimisation was performed before resampling.\nAll data-dependent preprocessing will be fitted using training data only\nwithin the nested resampling procedure. Numeric variables will be\nstandardised only for model families for which scaling is required,\nincluding ElasticNet logistic regression and radial-basis-function\nsupport-vector machines. Generic machine-learning pipelines will fit\ncategorical encoders within training folds with unseen validation levels\nhandled without refitting on held-out data, whereas CatBoost will use\nnative categorical predictors.\n\nSurvey weights, primary sampling unit identifiers, survey strata,\nparticipant identifiers, and outcome-construction variables are retained\nonly for design, weighting, resampling, quality-control, or evaluation\npurposes and are not used as predictive features.\n\nSeveral predictors—including household composition, wealth, residence,\nmedia exposure, employment, healthcare decision-making, and related\ncurrent-status variables—were measured at the survey interview rather\nthan prospectively at pregnancy onset. Accordingly, the analysis is\ninterpreted as retrospective survey-based classification of ANC\nutilisation rather than a prospectively validated pregnancy-onset risk\nprediction model.\n".strip()
reporting_audit = pd.DataFrame([{'Item': 'Predictor definitions documented', 'Status': True}, {'Item': 'Predictor source variables documented', 'Status': True}, {'Item': 'Continuous/count functional representation documented', 'Status': True}, {'Item': 'Categorical recoding documented', 'Status': True}, {'Item': 'Structural N/A handling documented', 'Status': True}, {'Item': 'Genuine non-reporting handling documented', 'Status': True}, {'Item': 'Primary vs sensitivity representations documented', 'Status': True}, {'Item': 'No arbitrary rare-category cutoff documented', 'Status': True}, {'Item': 'Pre-split vs fold-wise preprocessing documented', 'Status': True}, {'Item': 'Predictor timing/applicability documented', 'Status': True}, {'Item': 'Leakage variables excluded from predictive X documented', 'Status': True}])
if not reporting_audit['Status'].all():
    raise RuntimeError('Step-12C reporting audit failed.')
VARIABLE_DICTIONARY_FILE = DIRS['protocol'] / 'STEP12C_FINAL_predictor_dictionary.csv'
MISSINGNESS_POLICY_FILE = DIRS['protocol'] / 'STEP12C_FINAL_missing_structural_NA_policy.csv'
SENSITIVITY_PROTOCOL_FILE = DIRS['protocol'] / 'STEP12C_FINAL_primary_sensitivity_protocol.csv'
PREPROCESSING_TIMING_FILE = DIRS['protocol'] / 'STEP12C_FINAL_preprocessing_timing_protocol.csv'
MODEL_PREPROCESSING_FILE = DIRS['protocol'] / 'STEP12C_FINAL_model_specific_preprocessing.csv'
TIMING_SUMMARY_FILE = DIRS['protocol'] / 'STEP12C_FINAL_predictor_timing_summary.csv'
REFERENCE_REGISTRY_FILE = DIRS['protocol'] / 'STEP12C_FINAL_reference_registry.csv'
REPORTING_AUDIT_FILE = DIRS['qc'] / 'STEP12C_TRIPOD_PROBAST_documentation_audit.csv'
METHODS_TEXT_FILE = DIRS['protocol'] / 'STEP12C_manuscript_preprocessing_methods.txt'
variable_dictionary.to_csv(VARIABLE_DICTIONARY_FILE, index=False)
missingness_policy.to_csv(MISSINGNESS_POLICY_FILE, index=False)
sensitivity_protocol.to_csv(SENSITIVITY_PROTOCOL_FILE, index=False)
preprocessing_timing.to_csv(PREPROCESSING_TIMING_FILE, index=False)
model_preprocessing.to_csv(MODEL_PREPROCESSING_FILE, index=False)
timing_summary.to_csv(TIMING_SUMMARY_FILE, index=False)
reference_registry.to_csv(REFERENCE_REGISTRY_FILE, index=False)
reporting_audit.to_csv(REPORTING_AUDIT_FILE, index=False)
with open(METHODS_TEXT_FILE, 'w', encoding='utf-8') as f:
    f.write(methods_text)
VARIABLE_DICTIONARY_SHA256 = sha256_file(VARIABLE_DICTIONARY_FILE)
METHODS_TEXT_SHA256 = sha256_file(METHODS_TEXT_FILE)
step12c_manifest = {'step': 'STEP 12C', 'status': 'PRE_SPLIT_DOCUMENTATION_FROZEN', 'data_modified': False, 'input_primary_master': str(PRIMARY_FILE), 'input_primary_sha256': observed_hashes['primary'], 'analytic_n': 5063, 'primary_predictors': PRIMARY_PREDICTORS, 'rows_removed': 0, 'imputation_performed': False, 'scaling_performed': False, 'encoding_performed': False, 'cv_split_performed': False, 'variable_dictionary_sha256': VARIABLE_DICTIONARY_SHA256, 'methods_text_sha256': METHODS_TEXT_SHA256, 'next_step': 'Generate and freeze PSU-exclusive, stratum-preserving nested CV splits', 'execution_time': dt.datetime.now().isoformat(timespec='seconds')}
STEP12C_MANIFEST_FILE = DIRS['protocol'] / 'STEP12C_FINAL_documentation_manifest.json'
with open(STEP12C_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(step12c_manifest, f, indent=4, ensure_ascii=False, default=str)
if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError('Primary master changed during documentation step.')
print('\n' + '=' * 78)
print('STEP 12C — MANUSCRIPT / PREPROCESSING DOCUMENTATION FREEZE')
print('=' * 78)
print('\nFINAL VARIABLE DICTIONARY:')
display(variable_dictionary)
print('\nMISSING / STRUCTURAL-N/A POLICY:')
display(missingness_policy)
print('\nPRIMARY / SENSITIVITY PROTOCOL:')
display(sensitivity_protocol)
print('\nPRE-SPLIT VS POST-SPLIT PREPROCESSING:')
display(preprocessing_timing)
print('\nMODEL-SPECIFIC PREPROCESSING:')
display(model_preprocessing)
print('\nPREDICTOR TIMING SUMMARY:')
display(timing_summary)
print('\nREFERENCE REGISTRY:')
display(reference_registry)
print('\nTRIPOD+AI / PROBAST+AI DOCUMENTATION AUDIT:')
display(reporting_audit)
print('\nPRIMARY MASTER IMMUTABILITY CHECK:')
print('Primary master SHA-256:')
print(sha256_file(PRIMARY_FILE))
print('\nVARIABLE DICTIONARY SHA-256:')
print(VARIABLE_DICTIONARY_SHA256)
print('\nMETHODS TEXT SHA-256:')
print(METHODS_TEXT_SHA256)
print('\nMANUSCRIPT METHODS PREVIEW:')
print(methods_text)
print('\nIMPORTANT:')
print('- No data values were changed in Step 12C.')
print('- No participant was removed.')
print('- No unresolved machine-missing value exists in the primary master.')
print('- Structural N/A and genuine non-reporting are documented separately.')
print('- No imputation was required for the primary predictor set.')
print('- Continuous/count predictors remain ungrouped for primary modelling.')
print('- No scaling, encoding, spline fitting, tuning, calibration or CV has occurred.')
print('- Step 12B primary master remains byte-for-byte unchanged.')
print('=' * 78)
print('STATUS: STEP 12C PASS')
print('=' * 78)



STEP 12C — MANUSCRIPT / PREPROCESSING DOCUMENTATION FREEZE

FINAL VARIABLE DICTIONARY:


,Order,Predictor,DHS_Source,Type,Primary_Definition,Missing_or_NA_Handling,Primary_Representation,Timing,Sensitivity
0,1,maternal_age_at_index_birth_years,V011 + linked B3 using exact MIDX/BIDX linkage,Continuous numeric,Maternal age in years at index birth: (index-b...,None required; complete,Exact continuous value,Index-pregnancy anchored,Grouped age only for descriptive/sensitivity a...
1,2,prior_live_births,V201 + index-pregnancy multiplicity,Count numeric,Total live births minus live-born children bel...,None required; complete,Exact count,Index-pregnancy anchored,Grouped parity only descriptive/sensitivity
2,3,household_size_members,V136,Count numeric,Exact number of household members listed,None required; complete,Exact count,Survey-time,Grouped household size only descriptive/sensit...
3,4,maternal_education,V106,Ordered categorical,No education / Primary / Secondary / Higher,None required; complete,Categorical; no equal-distance assumption,Survey-time attainment,None prespecified
4,5,wealth_quintile,V190,Ordered categorical,Poorest / Poorer / Middle / Richer / Richest,None required; complete,Categorical,Survey-time,None prespecified
5,6,partner_education_primary_final,V701 + V502,Nominal categorical,No education / Primary / Secondary / Higher / ...,Structural unavailability and genuine non-repo...,Six explicit categories,Survey-time current-partner characteristic,Pool the two unavailable-information mechanism...
6,7,division,V024,Nominal categorical,Eight Bangladesh divisions,None required; complete,Eight categories,Survey-time residence,Excluded during leave-one-division-out validation
7,8,pregnancy_intention,Exact maternity-slot M10,Nominal categorical,Wanted then / Wanted later / Wanted no more,None required after exact-slot reconstruction,Three categories,Index-pregnancy anchored,None prespecified
8,9,healthcare_decision_primary_final,V743A + V502,Nominal categorical,Respondent alone / Respondent and husband-part...,V502 != 1 represented as structural N/A; sourc...,Five conceptual categories,Survey-time current-status decision variable,Original six-level V743A source representation
9,10,residence,V025,Binary categorical,Urban / Rural,None required,Binary categorical,Survey-time,None prespecified



MISSING / STRUCTURAL-N/A POLICY:


,Situation,Primary_Action,Row_Deletion,Interpretation
0,Literal NaN / blank / non-finite,Not permitted in final model master; hard QC f...,False,Unresolved machine missingness
1,Raw DHS don't know / special code,Recode according to documented meaning; raw sp...,False,Documented source non-response/special value
2,Structural not applicable,Represent explicitly as substantive structural...,False,Question not applicable by survey routing
3,Partner education not reported,Preserve as explicit primary category,False,Genuine non-reporting among applicable respond...
4,Numeric predictor missingness,None present in current primary master; no imp...,False,Not applicable to current final dataset



PRIMARY / SENSITIVITY PROTOCOL:


,Analysis,Partner_Education,Healthcare_Decision,Purpose
0,Primary,Six levels; structural unavailable and not-rep...,Five conceptual categories; Someone else + Oth...,Main prespecified analysis
1,Sensitivity A,Unavailable/not-applicable and not-reported po...,Primary five-category representation,Assess sensitivity to sparse partner-education...
2,Sensitivity B,Primary six-category representation,Original six Step-11 source categories,Assess sensitivity to conceptual pooling of So...


... (7 outputs left out)

,Key,Reference,Used_For
0,DHS_RECODE7,The DHS Program. DHS VII Standard Recode Manua...,"V701, V714 and standard DHS variable definitio..."
1,DHS_GUIDE7,"The DHS Program. Guide to DHS Statistics, DHS-...","Healthcare decision-making population base, me..."
2,TRIPOD_AI,Collins GS et al. TRIPOD+AI statement: updated...,"Transparent reporting of predictors, preproces..."
3,PROBAST_AI,"Moons KGM et al. PROBAST+AI: updated quality, ...","Predictor preprocessing, applicability, model-..."
4,BMJ_PREDICTION_GUIDE,Riley RD et al. Developing clinical prediction...,"Avoid arbitrary categorisation, missing-data p..."
5,BMJ_CONTINUOUS,Lopez-Ayala P et al. Dealing with continuous v...,Retention of continuous predictors and flexibl...



TRIPOD+AI / PROBAST+AI DOCUMENTATION AUDIT:


,Item,Status
0,Predictor definitions documented,True
1,Predictor source variables documented,True
2,Continuous/count functional representation doc...,True
3,Categorical recoding documented,True
4,Structural N/A handling documented,True
5,Genuine non-reporting handling documented,True
6,Primary vs sensitivity representations documented,True
7,No arbitrary rare-category cutoff documented,True
8,Pre-split vs fold-wise preprocessing documented,True
9,Predictor timing/applicability documented,True



PRIMARY MASTER IMMUTABILITY CHECK:
Primary master SHA-256:
0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a

VARIABLE DICTIONARY SHA-256:
faa1021a1bb71f5d69b50dfb2556705c039043ff30d78bd825064c28be001480

METHODS TEXT SHA-256:
7f0ad284e7326e6e22f61c294707aa664ab1b9d15d182de9b3080cf4735315d2

MANUSCRIPT METHODS PREVIEW:
Predictor preprocessing

Predictor definitions and deterministic recoding rules were specified
before resampling and model fitting. Fifteen candidate predictors were
retained in the primary analysis. Maternal age was calculated at the
index birth from linked century-month codes and was retained as a
continuous variable. Prior live births were reconstructed using a
multiple-birth-aware count, and household size was retained as the
exact number of household members. These numeric predictors were not
arbitrarily categorised for the primary analysis.

Categorical predictors were reconstructed from documented DHS recode
variables and retained using human-reada

In [14]:
# Step 13: Freeze PSU-exclusive, stratum-preserving nested cross-validation splits.

from pathlib import Path
import datetime as dt
import hashlib
import json
import warnings

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2026
N_OUTER = 5
N_INNER = 4

PRIMARY_FILE = DIRS["processed"] / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
STEP12C_MANIFEST_FILE = DIRS["protocol"] / "STEP12C_FINAL_documentation_manifest.json"
CV_SPLIT_DIR = DIRS["splits"]

EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)

EXPECTED_SPLIT_HASHES = {
    "outer_row": "258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372",
    "inner_row": "2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633",
    "outer_psu": "a9b92edafc6b51d26ac5716961f6dcc708a0b5b183c014dea52e8f76fb7e8483",
    "inner_psu": "2d43b1f233c67a14e80383324472032be06553d13a536caf3ec6f2b329f2a564",
}


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


for path in [PRIMARY_FILE, STEP12C_MANIFEST_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required input missing: {path}")

observed_primary_sha = sha256_file(PRIMARY_FILE)
if observed_primary_sha != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError("Primary master hash mismatch.")

with open(STEP12C_MANIFEST_FILE, "r", encoding="utf-8") as f:
    step12c_manifest = json.load(f)

if step12c_manifest["input_primary_sha256"] != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError("Step-12C manifest references a different primary master.")

master = pd.read_csv(PRIMARY_FILE)

if master.shape != (5063, 23):
    raise RuntimeError(f"Unexpected primary-master shape: {master.shape}")
if master["_ROW_ID"].duplicated().any():
    raise RuntimeError("Duplicate _ROW_ID.")
if master["CASEID"].duplicated().any():
    raise RuntimeError("Duplicate CASEID.")
if master.isna().sum().sum() != 0:
    raise RuntimeError("Unexpected NaN in the final primary master.")

expected_class_counts = {0: 403, 1: 4407, 2: 253}
observed_class_counts = master["ANC_CLASS"].value_counts().sort_index().to_dict()

if observed_class_counts != expected_class_counts:
    raise RuntimeError("Outcome counts changed before split generation.")
if master["PSU"].nunique() != 672:
    raise RuntimeError("Expected 672 PSUs.")
if master["STRATUM"].nunique() != 16:
    raise RuntimeError("Expected 16 survey strata.")
if (master.groupby("PSU", observed=True)["STRATUM"].nunique() != 1).any():
    raise RuntimeError("At least one PSU maps to multiple survey strata.")

psu_per_stratum = (
    master.groupby("STRATUM", observed=True)
    .agg(
        N=("CASEID", "size"),
        N_PSU=("PSU", "nunique"),
        CLASS0_N=("ANC_CLASS", lambda x: int((x == 0).sum())),
        CLASS1_N=("ANC_CLASS", lambda x: int((x == 1).sum())),
        CLASS2_N=("ANC_CLASS", lambda x: int((x == 2).sum())),
    )
    .reset_index()
    .sort_values("STRATUM")
    .reset_index(drop=True)
)

if psu_per_stratum["N_PSU"].min() < N_OUTER:
    raise RuntimeError("At least one survey stratum has fewer than five PSUs.")


def assign_stratum_group_folds(data, n_splits, base_seed, fold_column):
    columns = ["_ROW_ID", "CASEID", "PSU", "STRATUM", "ANC_CLASS", "SURVEY_WEIGHT"]
    result = data[columns].copy()
    result[fold_column] = -1
    warning_records = []

    for stratum_value in sorted(data["STRATUM"].unique()):
        local = data.loc[data["STRATUM"] == stratum_value].reset_index(drop=True)

        if local["PSU"].nunique() < n_splits:
            raise RuntimeError(
                f"Stratum {stratum_value} has fewer than {n_splits} PSUs."
            )

        cv = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=int(base_seed) + int(stratum_value),
        )

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            splits = list(
                cv.split(
                    X=np.zeros((len(local), 1)),
                    y=local["ANC_CLASS"].to_numpy(),
                    groups=local["PSU"].to_numpy(),
                )
            )

        for warning in caught:
            warning_records.append(
                {"STRATUM": stratum_value, "Warning": str(warning.message)}
            )

        for fold_id, (_, valid_pos) in enumerate(splits):
            row_ids = local.iloc[valid_pos]["_ROW_ID"].to_numpy()
            result.loc[result["_ROW_ID"].isin(row_ids), fold_column] = fold_id

    if (result[fold_column] < 0).any():
        raise RuntimeError(f"Incomplete {fold_column} assignment.")

    result[fold_column] = result[fold_column].astype(np.int64)

    psu_nfold = result.groupby("PSU", observed=True)[fold_column].nunique()
    if (psu_nfold != 1).any():
        raise RuntimeError(f"A PSU appears in multiple {fold_column} values.")

    warning_df = pd.DataFrame(warning_records, columns=["STRATUM", "Warning"])
    return result, warning_df


outer_assignment, outer_warnings = assign_stratum_group_folds(
    master, N_OUTER, SEED, "OUTER_FOLD"
)

outer_repeat, _ = assign_stratum_group_folds(
    master, N_OUTER, SEED, "OUTER_FOLD"
)

if not np.array_equal(
    outer_assignment["OUTER_FOLD"].to_numpy(),
    outer_repeat["OUTER_FOLD"].to_numpy(),
):
    raise RuntimeError("Outer split is not deterministic.")

master_outer = master.merge(
    outer_assignment[["_ROW_ID", "OUTER_FOLD"]],
    on="_ROW_ID",
    how="left",
    validate="one_to_one",
)


def partition_summary(data, scope, outer_fold, inner_fold, role):
    weight_sum = float(data["SURVEY_WEIGHT"].sum())
    row = {
        "Scope": scope,
        "OUTER_FOLD": outer_fold,
        "INNER_FOLD": inner_fold,
        "Role": role,
        "N": int(len(data)),
        "N_PSU": int(data["PSU"].nunique()),
        "N_STRATA": int(data["STRATUM"].nunique()),
        "WEIGHT_SUM": weight_sum,
    }

    for cls in (0, 1, 2):
        mask = data["ANC_CLASS"] == cls
        n_cls = int(mask.sum())
        weighted_n = float(data.loc[mask, "SURVEY_WEIGHT"].sum())

        row[f"CLASS_{cls}_N"] = n_cls
        row[f"CLASS_{cls}_PCT"] = 100.0 * n_cls / len(data)
        row[f"CLASS_{cls}_WEIGHTED_N"] = weighted_n
        row[f"CLASS_{cls}_WEIGHTED_PCT"] = 100.0 * weighted_n / weight_sum

    return row


def stage_presence(data):
    classes = set(data["ANC_CLASS"].unique())
    stage1 = set((data["ANC_CLASS"] != 0).astype(int).unique())
    anc_users = data.loc[data["ANC_CLASS"].isin([1, 2]), "ANC_CLASS"]
    stage2 = set((anc_users == 2).astype(int).unique())
    return classes, stage1, stage2


outer_hard_rows = []
outer_summary_rows = []

for outer_fold in range(N_OUTER):
    test = master_outer.loc[master_outer["OUTER_FOLD"] == outer_fold].copy()
    train = master_outer.loc[master_outer["OUTER_FOLD"] != outer_fold].copy()

    train_classes, train_stage1, train_stage2 = stage_presence(train)
    test_classes, test_stage1, test_stage2 = stage_presence(test)

    outer_hard_rows.append(
        {
            "OUTER_FOLD": outer_fold,
            "TRAIN_N": len(train),
            "TEST_N": len(test),
            "TRAIN_PSU": train["PSU"].nunique(),
            "TEST_PSU": test["PSU"].nunique(),
            "PSU_OVERLAP": len(set(train["PSU"]) & set(test["PSU"])),
            "TRAIN_STRATA": train["STRATUM"].nunique(),
            "TEST_STRATA": test["STRATUM"].nunique(),
            "TRAIN_ALL_3_CLASSES": train_classes == {0, 1, 2},
            "TEST_ALL_3_CLASSES": test_classes == {0, 1, 2},
            "TRAIN_STAGE1_BOTH": train_stage1 == {0, 1},
            "TEST_STAGE1_BOTH": test_stage1 == {0, 1},
            "TRAIN_STAGE2_BOTH": train_stage2 == {0, 1},
            "TEST_STAGE2_BOTH": test_stage2 == {0, 1},
        }
    )

    outer_summary_rows.extend(
        [
            partition_summary(train, "OUTER", outer_fold, None, "TRAIN"),
            partition_summary(test, "OUTER", outer_fold, None, "TEST"),
        ]
    )

outer_hard_audit = pd.DataFrame(outer_hard_rows)
outer_summary = pd.DataFrame(outer_summary_rows)

if (outer_hard_audit["PSU_OVERLAP"] != 0).any():
    raise RuntimeError("Outer PSU leakage detected.")
if not (outer_hard_audit["TRAIN_STRATA"] == 16).all():
    raise RuntimeError("An outer training fold lacks a survey stratum.")
if not (outer_hard_audit["TEST_STRATA"] == 16).all():
    raise RuntimeError("An outer test fold lacks a survey stratum.")

for column in [
    "TRAIN_ALL_3_CLASSES",
    "TEST_ALL_3_CLASSES",
    "TRAIN_STAGE1_BOTH",
    "TEST_STAGE1_BOTH",
    "TRAIN_STAGE2_BOTH",
    "TEST_STAGE2_BOTH",
]:
    if not outer_hard_audit[column].all():
        raise RuntimeError(f"Outer class-presence failure: {column}")

inner_parts = []
inner_warning_parts = []

for outer_fold in range(N_OUTER):
    outer_train = (
        master_outer.loc[master_outer["OUTER_FOLD"] != outer_fold]
        .drop(columns=["OUTER_FOLD"])
        .copy()
    )

    inner_assignment, inner_warning = assign_stratum_group_folds(
        outer_train,
        N_INNER,
        SEED + 10000 * (outer_fold + 1),
        "INNER_FOLD",
    )

    inner_assignment["OUTER_FOLD"] = outer_fold
    inner_parts.append(inner_assignment)

    if not inner_warning.empty:
        inner_warning = inner_warning.copy()
        inner_warning["OUTER_FOLD"] = outer_fold
        inner_warning_parts.append(inner_warning)

inner_assignments = pd.concat(inner_parts, ignore_index=True)

inner_warnings = (
    pd.concat(inner_warning_parts, ignore_index=True)
    if inner_warning_parts
    else pd.DataFrame(columns=["STRATUM", "Warning", "OUTER_FOLD"])
)

inner_hard_rows = []
inner_summary_rows = []

for outer_fold in range(N_OUTER):
    outer_inner = inner_assignments.loc[
        inner_assignments["OUTER_FOLD"] == outer_fold
    ].copy()

    outer_test_psus = set(
        master_outer.loc[
            master_outer["OUTER_FOLD"] == outer_fold,
            "PSU",
        ].unique()
    )

    if outer_test_psus & set(outer_inner["PSU"].unique()):
        raise RuntimeError("Outer-test PSU leaked into inner CV.")

    if (
        outer_inner.groupby("PSU", observed=True)["INNER_FOLD"].nunique() != 1
    ).any():
        raise RuntimeError(f"Outer fold {outer_fold}: PSU spans inner folds.")

    for inner_fold in range(N_INNER):
        valid = outer_inner.loc[outer_inner["INNER_FOLD"] == inner_fold].copy()
        train = outer_inner.loc[outer_inner["INNER_FOLD"] != inner_fold].copy()

        train_classes, train_stage1, train_stage2 = stage_presence(train)
        valid_classes, valid_stage1, valid_stage2 = stage_presence(valid)

        inner_hard_rows.append(
            {
                "OUTER_FOLD": outer_fold,
                "INNER_FOLD": inner_fold,
                "TRAIN_N": len(train),
                "VALID_N": len(valid),
                "TRAIN_PSU": train["PSU"].nunique(),
                "VALID_PSU": valid["PSU"].nunique(),
                "PSU_OVERLAP": len(set(train["PSU"]) & set(valid["PSU"])),
                "TRAIN_STRATA": train["STRATUM"].nunique(),
                "VALID_STRATA": valid["STRATUM"].nunique(),
                "TRAIN_ALL_3_CLASSES": train_classes == {0, 1, 2},
                "VALID_ALL_3_CLASSES": valid_classes == {0, 1, 2},
                "TRAIN_STAGE1_BOTH": train_stage1 == {0, 1},
                "VALID_STAGE1_BOTH": valid_stage1 == {0, 1},
                "TRAIN_STAGE2_BOTH": train_stage2 == {0, 1},
                "VALID_STAGE2_BOTH": valid_stage2 == {0, 1},
            }
        )

        inner_summary_rows.extend(
            [
                partition_summary(train, "INNER", outer_fold, inner_fold, "TRAIN"),
                partition_summary(valid, "INNER", outer_fold, inner_fold, "VALID"),
            ]
        )

inner_hard_audit = pd.DataFrame(inner_hard_rows)
inner_summary = pd.DataFrame(inner_summary_rows)

if (inner_hard_audit["PSU_OVERLAP"] != 0).any():
    raise RuntimeError("Inner PSU leakage detected.")
if not (inner_hard_audit["TRAIN_STRATA"] == 16).all():
    raise RuntimeError("An inner training fold lacks a survey stratum.")
if not (inner_hard_audit["VALID_STRATA"] == 16).all():
    raise RuntimeError("An inner validation fold lacks a survey stratum.")

for column in [
    "TRAIN_ALL_3_CLASSES",
    "VALID_ALL_3_CLASSES",
    "TRAIN_STAGE1_BOTH",
    "VALID_STAGE1_BOTH",
    "TRAIN_STAGE2_BOTH",
    "VALID_STAGE2_BOTH",
]:
    if not inner_hard_audit[column].all():
        raise RuntimeError(f"Inner class-presence failure: {column}")

outer_stratum_distribution = (
    outer_assignment.groupby(["STRATUM", "OUTER_FOLD"], observed=True)
    .agg(
        N=("CASEID", "size"),
        N_PSU=("PSU", "nunique"),
        WEIGHT_SUM=("SURVEY_WEIGHT", "sum"),
    )
    .reset_index()
    .sort_values(["STRATUM", "OUTER_FOLD"])
    .reset_index(drop=True)
)

if len(outer_stratum_distribution) != 16 * N_OUTER:
    raise RuntimeError("Not every outer fold contains every survey stratum.")

categorical_predictors = [
    "maternal_education",
    "wealth_quintile",
    "partner_education_primary_final",
    "division",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]

category_records = []


def audit_category_levels(train_data, valid_data, scope, outer_fold, inner_fold):
    for predictor in categorical_predictors:
        all_levels = sorted(master[predictor].astype(str).unique())
        train_values = train_data[predictor].astype(str)
        valid_values = valid_data[predictor].astype(str)

        for level in all_levels:
            train_n = int((train_values == level).sum())
            valid_n = int((valid_values == level).sum())

            category_records.append(
                {
                    "Scope": scope,
                    "OUTER_FOLD": outer_fold,
                    "INNER_FOLD": inner_fold,
                    "Predictor": predictor,
                    "Category": level,
                    "TRAIN_N": train_n,
                    "VALID_N": valid_n,
                    "VALID_LEVEL_UNSEEN_IN_TRAIN": valid_n > 0 and train_n == 0,
                }
            )


for outer_fold in range(N_OUTER):
    test = master_outer.loc[master_outer["OUTER_FOLD"] == outer_fold]
    train = master_outer.loc[master_outer["OUTER_FOLD"] != outer_fold]
    audit_category_levels(train, test, "OUTER", outer_fold, None)

for outer_fold in range(N_OUTER):
    inner_admin = inner_assignments.loc[
        inner_assignments["OUTER_FOLD"] == outer_fold,
        ["_ROW_ID", "INNER_FOLD"],
    ]

    outer_train_full = master.merge(
        inner_admin,
        on="_ROW_ID",
        how="inner",
        validate="one_to_one",
    )

    for inner_fold in range(N_INNER):
        valid = outer_train_full.loc[
            outer_train_full["INNER_FOLD"] == inner_fold
        ]
        train = outer_train_full.loc[
            outer_train_full["INNER_FOLD"] != inner_fold
        ]
        audit_category_levels(train, valid, "INNER", outer_fold, inner_fold)

category_presence_audit = pd.DataFrame(category_records)

unseen_category_cases = (
    category_presence_audit.loc[
        category_presence_audit["VALID_LEVEL_UNSEEN_IN_TRAIN"]
    ]
    .copy()
    .reset_index(drop=True)
)

n_outer_unseen = int((unseen_category_cases["Scope"] == "OUTER").sum())
n_inner_unseen = int((unseen_category_cases["Scope"] == "INNER").sum())

outer_row_count = outer_assignment["_ROW_ID"].value_counts()
if not (outer_row_count == 1).all():
    raise RuntimeError("A row has invalid outer-fold multiplicity.")

outer_psu_count = (
    outer_assignment.drop_duplicates(["PSU", "OUTER_FOLD"])
    .groupby("PSU")["OUTER_FOLD"]
    .nunique()
)

if not (outer_psu_count == 1).all():
    raise RuntimeError("A PSU has invalid outer-fold multiplicity.")

inner_row_count = inner_assignments["_ROW_ID"].value_counts()
if not (inner_row_count == N_OUTER - 1).all():
    raise RuntimeError("Unexpected inner row multiplicity.")

outer_psu_assignment = (
    outer_assignment.groupby(
        ["PSU", "STRATUM", "OUTER_FOLD"],
        observed=True,
    )
    .agg(
        N=("CASEID", "size"),
        CLASS0_N=("ANC_CLASS", lambda x: int((x == 0).sum())),
        CLASS1_N=("ANC_CLASS", lambda x: int((x == 1).sum())),
        CLASS2_N=("ANC_CLASS", lambda x: int((x == 2).sum())),
        WEIGHT_SUM=("SURVEY_WEIGHT", "sum"),
    )
    .reset_index()
)

if len(outer_psu_assignment) != 672:
    raise RuntimeError("Outer PSU assignment must contain 672 rows.")

inner_psu_assignment = (
    inner_assignments.groupby(
        ["OUTER_FOLD", "PSU", "STRATUM", "INNER_FOLD"],
        observed=True,
    )
    .agg(
        N=("CASEID", "size"),
        CLASS0_N=("ANC_CLASS", lambda x: int((x == 0).sum())),
        CLASS1_N=("ANC_CLASS", lambda x: int((x == 1).sum())),
        CLASS2_N=("ANC_CLASS", lambda x: int((x == 2).sum())),
        WEIGHT_SUM=("SURVEY_WEIGHT", "sum"),
    )
    .reset_index()
)

step13_qc = pd.DataFrame(
    {
        "Check": [
            "Analytic rows",
            "Unique PSU",
            "Survey strata",
            "Outer folds",
            "Outer rows assigned exactly once",
            "Outer PSUs assigned exactly once",
            "Outer train/test PSU overlap failures",
            "Outer test folds containing all 16 strata",
            "Outer test folds containing all 3 classes",
            "Outer stage-1 failures",
            "Outer stage-2 failures",
            "Inner folds per outer context",
            "Total inner train-context rows",
            "Inner train/valid PSU overlap failures",
            "Inner validation partitions containing all 16 strata",
            "Inner validation partitions containing all 3 classes",
            "Inner stage-1 failures",
            "Inner stage-2 failures",
            "Outer-test PSU leakage into inner CV",
            "Inner row multiplicity failures",
            "Outer stratum x fold cells",
            "Predictors used to construct split",
            "Model performance used to construct split",
            "Survey weights used to optimise split",
        ],
        "Observed": [
            len(master),
            master["PSU"].nunique(),
            master["STRATUM"].nunique(),
            outer_assignment["OUTER_FOLD"].nunique(),
            int((outer_row_count == 1).sum()),
            int((outer_psu_count == 1).sum()),
            int((outer_hard_audit["PSU_OVERLAP"] > 0).sum()),
            int((outer_hard_audit["TEST_STRATA"] == 16).sum()),
            int(outer_hard_audit["TEST_ALL_3_CLASSES"].sum()),
            int(
                (~outer_hard_audit["TRAIN_STAGE1_BOTH"]).sum()
                + (~outer_hard_audit["TEST_STAGE1_BOTH"]).sum()
            ),
            int(
                (~outer_hard_audit["TRAIN_STAGE2_BOTH"]).sum()
                + (~outer_hard_audit["TEST_STAGE2_BOTH"]).sum()
            ),
            N_INNER,
            len(inner_assignments),
            int((inner_hard_audit["PSU_OVERLAP"] > 0).sum()),
            int((inner_hard_audit["VALID_STRATA"] == 16).sum()),
            int(inner_hard_audit["VALID_ALL_3_CLASSES"].sum()),
            int(
                (~inner_hard_audit["TRAIN_STAGE1_BOTH"]).sum()
                + (~inner_hard_audit["VALID_STAGE1_BOTH"]).sum()
            ),
            int(
                (~inner_hard_audit["TRAIN_STAGE2_BOTH"]).sum()
                + (~inner_hard_audit["VALID_STAGE2_BOTH"]).sum()
            ),
            0,
            int((inner_row_count != N_OUTER - 1).sum()),
            len(outer_stratum_distribution),
            0,
            0,
            0,
        ],
        "Expected": [
            5063,
            672,
            16,
            5,
            5063,
            672,
            0,
            5,
            5,
            0,
            0,
            4,
            5063 * (N_OUTER - 1),
            0,
            N_OUTER * N_INNER,
            N_OUTER * N_INNER,
            0,
            0,
            0,
            0,
            16 * N_OUTER,
            0,
            0,
            0,
        ],
    }
)

step13_qc["PASS"] = step13_qc["Observed"] == step13_qc["Expected"]

if not step13_qc["PASS"].all():
    raise RuntimeError("STEP 13 hard QC failed.")

CV_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

OUTER_ASSIGNMENT_FILE = (
    CV_SPLIT_DIR / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)
OUTER_PSU_FILE = (
    CV_SPLIT_DIR / "STEP13_FROZEN_outer_5fold_PSU_assignments.csv"
)
INNER_ASSIGNMENT_FILE = (
    CV_SPLIT_DIR / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)
INNER_PSU_FILE = (
    CV_SPLIT_DIR / "STEP13_FROZEN_inner_4fold_PSU_assignments_long.csv"
)

OUTER_HARD_FILE = DIRS["qc"] / "STEP13_outer_hard_audit.csv"
INNER_HARD_FILE = DIRS["qc"] / "STEP13_inner_hard_audit.csv"
OUTER_SUMMARY_FILE = DIRS["qc"] / "STEP13_outer_class_weight_summary.csv"
INNER_SUMMARY_FILE = DIRS["qc"] / "STEP13_inner_class_weight_summary.csv"
STRATUM_DIST_FILE = DIRS["qc"] / "STEP13_outer_stratum_distribution.csv"
CATEGORY_AUDIT_FILE = DIRS["qc"] / "STEP13_category_presence_audit.csv"
UNSEEN_CATEGORY_FILE = DIRS["qc"] / "STEP13_unseen_validation_category_audit.csv"
OUTER_WARNINGS_FILE = DIRS["qc"] / "STEP13_outer_split_warnings.csv"
INNER_WARNINGS_FILE = DIRS["qc"] / "STEP13_inner_split_warnings.csv"
FINAL_QC_FILE = DIRS["qc"] / "STEP13_final_nested_cv_qc.csv"

outer_assignment.to_csv(OUTER_ASSIGNMENT_FILE, index=False)
outer_psu_assignment.to_csv(OUTER_PSU_FILE, index=False)
inner_assignments.to_csv(INNER_ASSIGNMENT_FILE, index=False)
inner_psu_assignment.to_csv(INNER_PSU_FILE, index=False)
outer_hard_audit.to_csv(OUTER_HARD_FILE, index=False)
inner_hard_audit.to_csv(INNER_HARD_FILE, index=False)
outer_summary.to_csv(OUTER_SUMMARY_FILE, index=False)
inner_summary.to_csv(INNER_SUMMARY_FILE, index=False)
outer_stratum_distribution.to_csv(STRATUM_DIST_FILE, index=False)
category_presence_audit.to_csv(CATEGORY_AUDIT_FILE, index=False)
unseen_category_cases.to_csv(UNSEEN_CATEGORY_FILE, index=False)
outer_warnings.to_csv(OUTER_WARNINGS_FILE, index=False)
inner_warnings.to_csv(INNER_WARNINGS_FILE, index=False)
step13_qc.to_csv(FINAL_QC_FILE, index=False)

split_methods_text = """
Internal validation used nested cross-validation while preserving the clustered
survey structure. Primary sampling units (PSUs) were treated as indivisible
grouping units so that participants from the same PSU could not occur in both
development and held-out partitions.

Five outer folds were used for performance estimation. Within each survey
stratum, PSUs were assigned using StratifiedGroupKFold with ANC utilisation
class as the stratification outcome and a fixed random seed; fold assignments
were then combined across the 16 survey strata. The held-out outer fold was
excluded from preprocessing fitting, model selection, hyperparameter tuning,
calibration, and threshold selection.

Each outer-training set was further partitioned into four inner folds using the
same stratum-specific PSU-grouped procedure. Predictor values, model
performance, calibration results, thresholds, and survey weights were not used
to choose or modify fold assignments. Survey weights are retained for later
model development, evaluation, and balance diagnostics. The same frozen folds
will be used across model families and prespecified sensitivity analyses.
""".strip()

SPLIT_METHODS_FILE = DIRS["protocol"] / "STEP13_nested_cv_methods.txt"

with open(SPLIT_METHODS_FILE, "w", encoding="utf-8") as f:
    f.write(split_methods_text)

observed_split_hashes = {
    "outer_row": sha256_file(OUTER_ASSIGNMENT_FILE),
    "inner_row": sha256_file(INNER_ASSIGNMENT_FILE),
    "outer_psu": sha256_file(OUTER_PSU_FILE),
    "inner_psu": sha256_file(INNER_PSU_FILE),
}

for key, expected_hash in EXPECTED_SPLIT_HASHES.items():
    if observed_split_hashes[key] != expected_hash:
        raise RuntimeError(
            f"Frozen Step-13 hash mismatch for {key}: "
            f"{observed_split_hashes[key]}"
        )

manifest = {
    "step": "STEP 13",
    "status": "NESTED_CV_SPLITS_FROZEN",
    "input_primary_master": str(PRIMARY_FILE),
    "input_primary_sha256": observed_primary_sha,
    "sklearn_version": sklearn.__version__,
    "seed": SEED,
    "outer_folds": N_OUTER,
    "inner_folds": N_INNER,
    "group_variable": "PSU",
    "survey_stratum_variable": "STRATUM",
    "stratification_outcome": "ANC_CLASS",
    "split_strategy": (
        "StratifiedGroupKFold separately within each survey stratum, "
        "then combine common fold IDs across strata"
    ),
    "predictor_values_used_for_split": False,
    "survey_weights_used_to_optimise_split": False,
    "model_performance_used_for_split": False,
    "hyperparameters_used_for_split": False,
    "model_training_performed": False,
    "scaling_performed": False,
    "encoding_performed": False,
    "outer_unseen_category_occurrences": n_outer_unseen,
    "inner_unseen_category_occurrences": n_inner_unseen,
    "outer_row_assignment_sha256": observed_split_hashes["outer_row"],
    "outer_psu_assignment_sha256": observed_split_hashes["outer_psu"],
    "inner_row_assignment_sha256": observed_split_hashes["inner_row"],
    "inner_psu_assignment_sha256": observed_split_hashes["inner_psu"],
    "execution_time": dt.datetime.now().isoformat(timespec="seconds"),
}

MANIFEST_FILE = DIRS["protocol"] / "STEP13_FROZEN_nested_cv_manifest.json"

with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4, ensure_ascii=False, default=str)

print("\nSTEP 13 — PSU-exclusive nested CV split freeze")
display(step13_qc)

print("\nOuter hard audit")
display(outer_hard_audit)

print("\nOuter train/test class and weight summary")
display(outer_summary)

print("\nInner hard audit")
display(inner_hard_audit)

print("\nInner validation class and weight summary")
display(
    inner_summary.loc[inner_summary["Role"] == "VALID"].reset_index(drop=True)
)

print(
    f"\nUnseen categorical levels — outer: {n_outer_unseen}, "
    f"inner: {n_inner_unseen}"
)
print(
    f"Splitter warnings — outer: {len(outer_warnings)}, "
    f"inner: {len(inner_warnings)}"
)

if not outer_warnings.empty:
    display(outer_warnings)

if not inner_warnings.empty:
    display(inner_warnings)

print("\nFrozen split hashes")
for key, value in observed_split_hashes.items():
    print(f"{key}: {value}")

print("\nSTATUS: STEP 13 PASS")



STEP 13 — PSU-exclusive nested CV split freeze


,Check,Observed,Expected,PASS
0,Analytic rows,5063,5063,True
1,Unique PSU,672,672,True
2,Survey strata,16,16,True
3,Outer folds,5,5,True
4,Outer rows assigned exactly once,5063,5063,True
5,Outer PSUs assigned exactly once,672,672,True
6,Outer train/test PSU overlap failures,0,0,True
7,Outer test folds containing all 16 strata,5,5,True
8,Outer test folds containing all 3 classes,5,5,True
9,Outer stage-1 failures,0,0,True



Outer hard audit


,OUTER_FOLD,TRAIN_N,TEST_N,TRAIN_PSU,TEST_PSU,PSU_OVERLAP,TRAIN_STRATA,TEST_STRATA,TRAIN_ALL_3_CLASSES,TEST_ALL_3_CLASSES,TRAIN_STAGE1_BOTH,TEST_STAGE1_BOTH,TRAIN_STAGE2_BOTH,TEST_STAGE2_BOTH
0,0,4057,1006,540,132,0,16,16,True,True,True,True,True,True
1,1,4050,1013,537,135,0,16,16,True,True,True,True,True,True
2,2,4058,1005,540,132,0,16,16,True,True,True,True,True,True
3,3,4055,1008,538,134,0,16,16,True,True,True,True,True,True
4,4,4032,1031,533,139,0,16,16,True,True,True,True,True,True



Outer train/test class and weight summary


,Scope,OUTER_FOLD,INNER_FOLD,Role,N,N_PSU,N_STRATA,WEIGHT_SUM,CLASS_0_N,CLASS_0_PCT,CLASS_0_WEIGHTED_N,CLASS_0_WEIGHTED_PCT,CLASS_1_N,CLASS_1_PCT,CLASS_1_WEIGHTED_N,CLASS_1_WEIGHTED_PCT,CLASS_2_N,CLASS_2_PCT,CLASS_2_WEIGHTED_N,CLASS_2_WEIGHTED_PCT
0,OUTER,0,None,TRAIN,4057,540,16,4101.651590,321,7.912250,323.625926,7.890137,3535,87.133350,3579.824995,87.277647,201,4.954400,198.200669,4.832216
1,OUTER,0,None,TEST,1006,132,16,1025.466661,82,8.151093,83.639876,8.156275,872,86.679920,880.900699,85.902422,52,5.168986,60.926086,5.941303
2,OUTER,1,None,TRAIN,4050,537,16,4128.272707,326,8.049383,333.142282,8.069774,3522,86.962963,3583.689600,86.808451,202,4.987654,211.440825,5.121775
3,OUTER,1,None,TEST,1013,135,16,998.845544,77,7.601185,74.123520,7.420919,885,87.364265,877.036094,87.804976,51,5.034551,47.685930,4.774104
4,OUTER,2,None,TRAIN,4058,540,16,4076.403154,327,8.058157,326.871604,8.018628,3528,86.939379,3538.844979,86.812929,203,5.002464,210.686571,5.168443
5,OUTER,2,None,TEST,1005,132,16,1050.715097,76,7.562189,80.394198,7.651379,879,87.462687,921.880715,87.738410,50,4.975124,48.440184,4.610211
6,OUTER,3,None,TRAIN,4055,538,16,4114.517617,318,7.842170,322.826339,7.846031,3531,87.077682,3581.233747,87.038970,206,5.080148,210.457531,5.114999
7,OUTER,3,None,TEST,1008,134,16,1012.600634,85,8.432540,84.439463,8.338871,876,86.904762,879.491947,86.854770,47,4.662698,48.669224,4.806359
8,OUTER,4,None,TRAIN,4032,533,16,4087.627936,320,7.936508,322.597057,7.892036,3512,87.103175,3559.309455,87.075182,200,4.960317,205.721424,5.032783
9,OUTER,4,None,TEST,1031,139,16,1039.490315,83,8.050436,84.668745,8.145217,895,86.808923,901.416239,86.717137,53,5.140640,53.405331,5.137646


... (4 outputs left out)


Unseen categorical levels — outer: 0, inner: 0
Splitter warnings — outer: 1, inner: 5


,STRATUM,Warning
0,9,The least populated class in y has only 4 memb...


,STRATUM,Warning,OUTER_FOLD
0,9,The least populated class in y has only 3 memb...,0
1,9,The least populated class in y has only 3 memb...,1
2,9,The least populated class in y has only 3 memb...,2
3,11,The least populated class in y has only 3 memb...,3
4,9,The least populated class in y has only 3 memb...,4



Frozen split hashes
outer_row: 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
inner_row: 2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633
outer_psu: a9b92edafc6b51d26ac5716961f6dcc708a0b5b183c014dea52e8f76fb7e8483
inner_psu: 2d43b1f233c67a14e80383324472032be06553d13a536caf3ec6f2b329f2a564

STATUS: STEP 13 PASS


In [16]:
# STEP 13A PATCH — Resolve project root

from pathlib import Path

if "PROJECT_ROOT" in globals():
    ROOT = Path(PROJECT_ROOT)

elif "DIRS" in globals() and "processed" in DIRS:
    ROOT = Path(DIRS["processed"]).parent

elif "PRIMARY_FILE" in globals():
    ROOT = Path(PRIMARY_FILE).parent.parent

else:
    ROOT = Path(
        r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
    )


if not ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist: {ROOT}"
    )


print("Project root:", ROOT)
print("STATUS: ROOT RESOLVED")

Project root: E:\BDHS_ANC_Q1_FINAL_REBUILD_2026
STATUS: ROOT RESOLVED


In [17]:
# ============================================================
# STEP 13A — REPRODUCIBILITY ARCHIVE
# ============================================================
# Creates:
#   1. Restricted local archive:
#      final row-level datasets + frozen CV assignments
#
#   2. Shareable methods/QC archive:
#      protocols, QC summaries, manifests and hashes only
#      (no row-level DHS-derived data)
#
# This step does not modify the analytic datasets or CV splits.
# ============================================================

from pathlib import Path
import csv
import hashlib
import importlib.metadata as importlib_metadata
import json
import platform
import shutil
import sys
import zipfile


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

ARCHIVE_ROOT = ROOT / "17_reproducibility_archive"

RESTRICTED_DIR = (
    ARCHIVE_ROOT
    / "restricted_row_level"
)

SHAREABLE_DIR = (
    ARCHIVE_ROOT
    / "shareable_methods_qc"
)


for directory in [
    ARCHIVE_ROOT,
    RESTRICTED_DIR,
    SHAREABLE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def file_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


def copy_with_project_structure(
    source,
    destination_root
):

    source = Path(source)

    relative_path = source.relative_to(
        ROOT
    )

    destination = (
        destination_root
        /
        relative_path
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        source,
        destination
    )

    return destination


def write_inventory(
    base_dir
):

    rows = []

    for path in sorted(
        base_dir.rglob("*")
    ):

        if (
            path.is_file()
            and
            path.name
            !=
            "ARCHIVE_FILE_MANIFEST.csv"
        ):

            rows.append(
                {
                    "relative_path":
                        str(
                            path.relative_to(
                                base_dir
                            )
                        ),

                    "size_bytes":
                        path.stat().st_size,

                    "sha256":
                        file_sha256(
                            path
                        )
                }
            )


    output_file = (
        base_dir
        /
        "ARCHIVE_FILE_MANIFEST.csv"
    )


    with open(
        output_file,
        "w",
        newline="",
        encoding="utf-8"
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=[
                "relative_path",
                "size_bytes",
                "sha256"
            ]
        )

        writer.writeheader()

        writer.writerows(
            rows
        )


    return rows


def create_zip(
    source_dir,
    zip_path
):

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as zf:

        for path in sorted(
            source_dir.rglob("*")
        ):

            if path.is_file():

                zf.write(
                    path,
                    arcname=path.relative_to(
                        source_dir
                    )
                )

    return zip_path


# ============================================================
# A. RESTRICTED ROW-LEVEL FILES
# ============================================================

restricted_files = [

    # Final literature-locked data
    DIRS["processed"]
    / "FINAL_processed_LITERATURE_LOCKED.csv",

    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv",

    DIRS["processed"]
    / "FINAL_SENSITIVITY_A_partner_pooling.csv",

    DIRS["processed"]
    / "FINAL_SENSITIVITY_B_healthcare_source_levels.csv",

    # Reference-locked Step-11 data
    DIRS["processed"]
    / "FINAL_processed_analytic_dataset_REFERENCE_LOCKED.csv",

    DIRS["processed"]
    / "FINAL_model_master_15_predictors_REFERENCE_LOCKED.csv",

    # Frozen nested-CV assignments
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv",

    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_PSU_assignments.csv",

    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv",

    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_PSU_assignments_long.csv",
]


missing_restricted = [

    str(path)

    for path in restricted_files

    if not path.exists()
]


if missing_restricted:

    raise FileNotFoundError(

        "Required restricted archival files are missing:\n"

        +
        "\n".join(
            missing_restricted
        )
    )


# ============================================================
# B. SHAREABLE METHODS / QC FILES
# ============================================================

shareable_candidates = [

    # -------------------------
    # Protocol / manifests
    # -------------------------

    DIRS["protocol"]
    / "STEP01_project_manifest.json",

    DIRS["protocol"]
    / "STEP11_FINAL_reference_locked_manifest.json",

    DIRS["protocol"]
    / "STEP12B_FINAL_literature_locked_manifest.json",

    DIRS["protocol"]
    / "STEP12C_FINAL_documentation_manifest.json",

    DIRS["protocol"]
    / "STEP12C_FINAL_predictor_dictionary.csv",

    DIRS["protocol"]
    / "STEP12C_FINAL_missing_structural_NA_policy.csv",

    DIRS["protocol"]
    / "STEP12C_FINAL_primary_sensitivity_protocol.csv",

    DIRS["protocol"]
    / "STEP12C_FINAL_preprocessing_timing_protocol.csv",

    DIRS["protocol"]
    / "STEP12C_FINAL_model_specific_preprocessing.csv",

    DIRS["protocol"]
    / "STEP12C_FINAL_predictor_timing_summary.csv",

    DIRS["protocol"]
    / "STEP12C_FINAL_reference_registry.csv",

    DIRS["protocol"]
    / "STEP12C_manuscript_preprocessing_methods.txt",

    DIRS["protocol"]
    / "STEP13_FROZEN_nested_cv_manifest.json",

    DIRS["protocol"]
    / "STEP13_nested_cv_methods.txt",


    # -------------------------
    # Step-12B QC
    # -------------------------

    DIRS["qc"]
    / "STEP12B_partner_education_crosswalk.csv",

    DIRS["qc"]
    / "STEP12B_healthcare_decision_crosswalk.csv",

    DIRS["qc"]
    / "STEP12B_primary_category_distributions.csv",

    DIRS["qc"]
    / "STEP12B_primary_sparse_level_audit.csv",

    DIRS["qc"]
    / "STEP12B_final_qc.csv",


    # -------------------------
    # Step-12C QC
    # -------------------------

    DIRS["qc"]
    / "STEP12C_TRIPOD_PROBAST_documentation_audit.csv",


    # -------------------------
    # Step-13 QC
    # -------------------------

    DIRS["qc"]
    / "STEP13_outer_hard_audit.csv",

    DIRS["qc"]
    / "STEP13_inner_hard_audit.csv",

    DIRS["qc"]
    / "STEP13_outer_class_weight_summary.csv",

    DIRS["qc"]
    / "STEP13_inner_class_weight_summary.csv",

    DIRS["qc"]
    / "STEP13_outer_stratum_distribution.csv",

    DIRS["qc"]
    / "STEP13_outer_split_warnings.csv",

    DIRS["qc"]
    / "STEP13_inner_split_warnings.csv",

    DIRS["qc"]
    / "STEP13_final_nested_cv_qc.csv",
]


# Some non-essential files may not exist because of earlier
# notebook versions. Keep all available files, but enforce the
# core reproducibility artifacts below.

shareable_files = [

    path

    for path in shareable_candidates

    if path.exists()
]


required_shareable = [

    DIRS["protocol"]
    / "STEP12C_FINAL_predictor_dictionary.csv",

    DIRS["protocol"]
    / "STEP13_FROZEN_nested_cv_manifest.json",

    DIRS["qc"]
    / "STEP13_final_nested_cv_qc.csv",
]


missing_required_shareable = [

    str(path)

    for path in required_shareable

    if not path.exists()
]


if missing_required_shareable:

    raise FileNotFoundError(

        "Required methods/QC files are missing:\n"

        +
        "\n".join(
            missing_required_shareable
        )
    )


# ============================================================
# C. COPY FILES
# ============================================================

# Row-level material goes only to the restricted archive.

for path in restricted_files:

    copy_with_project_structure(
        path,
        RESTRICTED_DIR
    )


# Protocol/QC material goes to both archives.

for path in shareable_files:

    copy_with_project_structure(
        path,
        SHAREABLE_DIR
    )

    copy_with_project_structure(
        path,
        RESTRICTED_DIR
    )


# ============================================================
# D. SOFTWARE ENVIRONMENT
# ============================================================

software_versions = {

    "python":
        sys.version.split()[0],

    "platform":
        platform.platform()
}


for package in [

    "numpy",
    "pandas",
    "scikit-learn",
    "pyreadstat",
    "scipy",

]:

    try:

        software_versions[
            package
        ] = importlib_metadata.version(
            package
        )

    except importlib_metadata.PackageNotFoundError:

        software_versions[
            package
        ] = "not installed"


for directory in [

    RESTRICTED_DIR,
    SHAREABLE_DIR,

]:

    with open(

        directory
        / "software_versions_step13.json",

        "w",

        encoding="utf-8"

    ) as handle:

        json.dump(
            software_versions,
            handle,
            indent=2,
            ensure_ascii=False
        )


# ============================================================
# E. README FILES
# ============================================================

restricted_readme = """
BDHS 2022 ANC restricted reproducibility archive
=================================================

Purpose
-------
This archive preserves the row-level derived datasets and exact frozen
cross-validation assignments required to reproduce the analysis through
Step 13.

Handling
--------
This archive contains row-level material derived from a Bangladesh DHS
country dataset and exact participant/PSU split assignments.

It should remain in restricted/local storage and should not be uploaded
as the public manuscript supplementary-data package.

A separate shareable methods/QC archive is generated without row-level
derived data.
""".strip()


shareable_readme = """
BDHS 2022 ANC methods/QC reproducibility bundle
===============================================

Purpose
-------
This archive contains methodological documentation, preprocessing
definitions, aggregate quality-control outputs, software versions,
split metadata, manifests, and SHA-256 integrity information through
Step 13.

Data availability
-----------------
Row-level DHS-derived datasets, CASEID-level files, and exact
participant/PSU split-assignment files are intentionally excluded.

Researchers with authorized access to Bangladesh DHS 2022 data can
rerun the analysis pipeline and verify the recorded SHA-256 hashes.
""".strip()


with open(

    RESTRICTED_DIR
    / "README_RESTRICTED_ARCHIVE.txt",

    "w",

    encoding="utf-8"

) as handle:

    handle.write(
        restricted_readme
    )


with open(

    SHAREABLE_DIR
    / "README_SHAREABLE_BUNDLE.txt",

    "w",

    encoding="utf-8"

) as handle:

    handle.write(
        shareable_readme
    )


# ============================================================
# F. FILE INVENTORIES + SHA-256
# ============================================================

restricted_inventory = write_inventory(
    RESTRICTED_DIR
)


shareable_inventory = write_inventory(
    SHAREABLE_DIR
)


# ============================================================
# G. MASTER ARCHIVE SUMMARY
# ============================================================

primary_master_file = (

    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


outer_row_file = (

    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


inner_row_file = (

    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_ROW_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_ROW_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


observed_primary_sha = file_sha256(
    primary_master_file
)


observed_outer_sha = file_sha256(
    outer_row_file
)


observed_inner_sha = file_sha256(
    inner_row_file
)


if (
    observed_primary_sha
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    observed_outer_sha
    !=
    EXPECTED_OUTER_ROW_SHA256
):

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if (
    observed_inner_sha
    !=
    EXPECTED_INNER_ROW_SHA256
):

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


archive_summary = {

    "status":
        "STEP13_ARCHIVE_CREATED",

    "project_root":
        str(ROOT),

    "restricted_directory":
        str(RESTRICTED_DIR),

    "shareable_directory":
        str(SHAREABLE_DIR),

    "restricted_file_count":
        len(
            restricted_inventory
        ),

    "shareable_file_count":
        len(
            shareable_inventory
        ),

    "primary_master_sha256":
        observed_primary_sha,

    "outer_row_split_sha256":
        observed_outer_sha,

    "inner_row_split_sha256":
        observed_inner_sha,

    "raw_source_sha256":
        (
            "81676772f734e43b33e09085a2f3232c"
            "35f5c04dfdf898cdb551957a3278b56f"
        ),

    "analytic_n":
        5063,

    "unique_psu":
        672,

    "survey_strata":
        16,

    "outer_folds":
        5,

    "inner_folds":
        4
}


SUMMARY_FILE = (

    ARCHIVE_ROOT
    / "STEP13_ARCHIVE_SUMMARY.json"
)


with open(

    SUMMARY_FILE,

    "w",

    encoding="utf-8"

) as handle:

    json.dump(
        archive_summary,
        handle,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# H. ZIP ARCHIVES
# ============================================================

RESTRICTED_ZIP = (

    ARCHIVE_ROOT
    / "BDHS_ANC_STEP13_LOCAL_RESTRICTED_ARCHIVE.zip"
)


SHAREABLE_ZIP = (

    ARCHIVE_ROOT
    / "BDHS_ANC_STEP13_SHAREABLE_METHODS_QC_BUNDLE.zip"
)


create_zip(
    RESTRICTED_DIR,
    RESTRICTED_ZIP
)


create_zip(
    SHAREABLE_DIR,
    SHAREABLE_ZIP
)


zip_hashes = {

    "restricted_zip": {

        "path":
            str(
                RESTRICTED_ZIP
            ),

        "sha256":
            file_sha256(
                RESTRICTED_ZIP
            )
    },

    "shareable_zip": {

        "path":
            str(
                SHAREABLE_ZIP
            ),

        "sha256":
            file_sha256(
                SHAREABLE_ZIP
            )
    }
}


ZIP_HASH_FILE = (

    ARCHIVE_ROOT
    / "STEP13_ARCHIVE_ZIP_HASHES.json"
)


with open(

    ZIP_HASH_FILE,

    "w",

    encoding="utf-8"

) as handle:

    json.dump(
        zip_hashes,
        handle,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# I. FINAL QC
# ============================================================

if not RESTRICTED_ZIP.exists():

    raise RuntimeError(
        "Restricted archive ZIP was not created."
    )


if not SHAREABLE_ZIP.exists():

    raise RuntimeError(
        "Shareable archive ZIP was not created."
    )


if not SUMMARY_FILE.exists():

    raise RuntimeError(
        "Archive summary was not created."
    )


if not ZIP_HASH_FILE.exists():

    raise RuntimeError(
        "ZIP hash file was not created."
    )


# ============================================================
# J. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 13A — REPRODUCIBILITY ARCHIVE"
)

print(
    "=" * 78
)


print(
    "\nRestricted local archive:"
)

print(
    RESTRICTED_ZIP
)


print(
    "\nShareable methods/QC bundle:"
)

print(
    SHAREABLE_ZIP
)


print(
    "\nArchive summary:"
)

print(
    SUMMARY_FILE
)


print(
    "\nZIP hash registry:"
)

print(
    ZIP_HASH_FILE
)


print(
    "\nArchived file counts:"
)

print(
    f"Restricted : {len(restricted_inventory)}"
)

print(
    f"Shareable  : {len(shareable_inventory)}"
)


print(
    "\nIntegrity checks:"
)

print(
    f"Primary master : {observed_primary_sha}"
)

print(
    f"Outer split    : {observed_outer_sha}"
)

print(
    f"Inner split    : {observed_inner_sha}"
)


print(
    "\nImportant:"
)

print(
    "- No analytic value was modified."
)

print(
    "- No CV assignment was regenerated."
)

print(
    "- Row-level DHS-derived files are stored only in the restricted archive."
)

print(
    "- The shareable bundle contains methods/QC material only."
)

print(
    "- SHA-256 inventories allow later integrity verification."
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 13A PASS"
)

print(
    "=" * 78
)


STEP 13A — REPRODUCIBILITY ARCHIVE

Restricted local archive:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\17_reproducibility_archive\BDHS_ANC_STEP13_LOCAL_RESTRICTED_ARCHIVE.zip

Shareable methods/QC bundle:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\17_reproducibility_archive\BDHS_ANC_STEP13_SHAREABLE_METHODS_QC_BUNDLE.zip

Archive summary:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\17_reproducibility_archive\STEP13_ARCHIVE_SUMMARY.json

ZIP hash registry:
E:\BDHS_ANC_Q1_FINAL_REBUILD_2026\17_reproducibility_archive\STEP13_ARCHIVE_ZIP_HASHES.json

Archived file counts:
Restricted : 40
Shareable  : 30

Integrity checks:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Outer split    : 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
Inner split    : 2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633

Important:
- No analytic value was modified.
- No CV assignment was regenerated.
- Row-level DHS-derived files are stored only in the restricted archive

In [18]:
# ============================================================
# STEP 14 — FINAL MODEL-SPECIFIC PREPROCESSING AND
#           ESTIMATOR-INTERFACE PROTOCOL FREEZE
# ============================================================
#
# Objective
# ---------
# Freeze a literature- and documentation-aligned preprocessing
# strategy before model fitting or performance inspection.
#
# Core principle:
#   Same participants
#   Same 15 predictors
#   Same outcome definitions
#   Same frozen nested CV
#   Same later evaluation framework
#
# Preprocessing differs only where required by the statistical
# or computational properties of the model family.
#
# NO model fitting, tuning, calibration, threshold selection,
# or performance evaluation is performed in this step.
# ============================================================


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import datetime as dt
import hashlib
import importlib
import importlib.metadata as importlib_metadata
import inspect
import json

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from IPython.display import display


# ============================================================
# 1. REPRODUCIBILITY CONSTANTS
# ============================================================

SEED = 2026

if "DIRS" not in globals():

    raise RuntimeError(
        "DIRS is unavailable. "
        "Use the accepted project setup before Step 14."
    )


PROJECT_ROOT = Path(
    DIRS["processed"]
).parent


# ============================================================
# 2. FROZEN INPUT FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


STEP13_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP13_FROZEN_nested_cv_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(block)

    return digest.hexdigest()


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP13_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing:\n{path}"
        )


observed_primary_sha = sha256_file(
    PRIMARY_FILE
)

observed_outer_sha = sha256_file(
    OUTER_SPLIT_FILE
)

observed_inner_sha = sha256_file(
    INNER_SPLIT_FILE
)


if observed_primary_sha != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if observed_outer_sha != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if observed_inner_sha != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


with open(
    STEP13_MANIFEST_FILE,
    "r",
    encoding="utf-8",
) as handle:

    step13_manifest = json.load(
        handle
    )


if (
    step13_manifest["input_primary_sha256"]
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Step-13 manifest references "
        "a different primary master."
    )


# ============================================================
# 3. LOAD LOCKED DATA
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


outer_assignments = pd.read_csv(
    OUTER_SPLIT_FILE
)


inner_assignments = pd.read_csv(
    INNER_SPLIT_FILE
)


if master.shape != (5063, 23):

    raise RuntimeError(
        f"Unexpected primary-master shape: {master.shape}"
    )


if master["_ROW_ID"].duplicated().any():

    raise RuntimeError(
        "Duplicate _ROW_ID."
    )


if master["CASEID"].duplicated().any():

    raise RuntimeError(
        "Duplicate CASEID."
    )


if master.isna().sum().sum() != 0:

    raise RuntimeError(
        "Unresolved NaN detected."
    )


if len(outer_assignments) != 5063:

    raise RuntimeError(
        "Unexpected outer-assignment row count."
    )


if len(inner_assignments) != 20252:

    raise RuntimeError(
        "Unexpected inner-assignment row count."
    )


# ============================================================
# 4. FINAL 15-PREDICTOR SCHEMA
# ============================================================

NUMERIC_PREDICTORS = [

    "maternal_age_at_index_birth_years",

    "prior_live_births",

    "household_size_members",
]


ORDERED_CATEGORICAL_PREDICTORS = [

    "maternal_education",

    "wealth_quintile",
]


NOMINAL_CATEGORICAL_PREDICTORS = [

    "partner_education_primary_final",

    "division",

    "pregnancy_intention",

    "healthcare_decision_primary_final",
]


BINARY_CATEGORICAL_PREDICTORS = [

    "residence",

    "weekly_media_exposure",

    "working_status",

    "religion_final",

    "household_head_sex",

    "distance_barrier",
]


CATEGORICAL_PREDICTORS = (

    ORDERED_CATEGORICAL_PREDICTORS
    +
    NOMINAL_CATEGORICAL_PREDICTORS
    +
    BINARY_CATEGORICAL_PREDICTORS
)


PRIMARY_PREDICTORS = (

    NUMERIC_PREDICTORS
    +
    CATEGORICAL_PREDICTORS
)


if len(PRIMARY_PREDICTORS) != 15:

    raise RuntimeError(
        "Expected exactly 15 predictors."
    )


if len(NUMERIC_PREDICTORS) != 3:

    raise RuntimeError(
        "Expected three numeric predictors."
    )


if len(CATEGORICAL_PREDICTORS) != 12:

    raise RuntimeError(
        "Expected twelve categorical predictors."
    )


missing_predictors = [

    predictor

    for predictor in PRIMARY_PREDICTORS

    if predictor not in master.columns
]


if missing_predictors:

    raise RuntimeError(
        f"Missing predictor(s): {missing_predictors}"
    )


# ============================================================
# 5. LEAKAGE GUARD
# ============================================================

FORBIDDEN_X = {

    "_ROW_ID",
    "CASEID",

    "ANC_VISITS",
    "ANC_CLASS",
    "ANC_CLASS_LABEL",

    "SURVEY_WEIGHT",
    "PSU",
    "STRATUM",

    "V005",
    "V021",
    "V023",

    "INDEX_MATERNITY_SLOT",
    "SELECTED_MIDX",
    "MATCHED_BIDX",
    "INDEX_BIRTH_CMC",
}


leakage_features = (

    set(PRIMARY_PREDICTORS)
    &
    FORBIDDEN_X
)


if leakage_features:

    raise RuntimeError(
        "Forbidden predictive features detected:\n"
        f"{sorted(leakage_features)}"
    )


# ============================================================
# 6. SOURCE CATEGORY AUDIT
# ============================================================

category_rows = []


for predictor in CATEGORICAL_PREDICTORS:

    counts = (
        master[predictor]
        .astype(str)
        .value_counts()
        .sort_index()
    )


    for category, count in counts.items():

        category_rows.append(
            {
                "Predictor":
                    predictor,

                "Category":
                    category,

                "N":
                    int(count),
            }
        )


category_levels = pd.DataFrame(
    category_rows
)


partner_not_reported_n = int(
    (
        master[
            "partner_education_primary_final"
        ]
        ==
        "Partner education not reported"
    )
    .sum()
)


if partner_not_reported_n != 9:

    raise RuntimeError(
        "Expected n=9 for partner-education "
        "non-reporting."
    )


# ============================================================
# 7. INTERPRETABLE LOGISTIC REFERENCE LEVELS
# ============================================================
#
# These apply to the later interpretable logistic comparator.
# They are selected a priori for interpretation, not because of
# statistical significance or model performance.
#
# Penalized predictive ElasticNet logistic regression will NOT
# drop a category.
# ============================================================

LOGISTIC_REFERENCE_LEVELS = {

    "maternal_education":
        "No education",

    "wealth_quintile":
        "Poorest",

    "partner_education_primary_final":
        "No education",

    "division":
        "Dhaka",

    "pregnancy_intention":
        "Wanted then",

    "healthcare_decision_primary_final":
        "Respondent alone",

    "residence":
        "Urban",

    "weekly_media_exposure":
        "No",

    "working_status":
        "No",

    "religion_final":
        "Islam",

    "household_head_sex":
        "Male",

    "distance_barrier":
        "Not a big problem",
}


reference_rows = []


for predictor, reference in (
    LOGISTIC_REFERENCE_LEVELS.items()
):

    levels = set(
        master[predictor]
        .astype(str)
        .unique()
    )


    reference_rows.append(
        {
            "Predictor":
                predictor,

            "Reference_Category":
                reference,

            "Present":
                reference in levels,
        }
    )


reference_qc = pd.DataFrame(
    reference_rows
)


if not reference_qc["Present"].all():

    raise RuntimeError(
        "At least one prespecified reference "
        "category is absent:\n"
        f"{reference_qc.loc[~reference_qc['Present']]}"
    )


# ============================================================
# 8. GENERIC TRAINING-FOLD OHE
# ============================================================
#
# Used for:
#   ElasticNet logistic
#   RBF-SVM
#   Random Forest
#   ExtraTrees
#   AdaBoost
#   GradientBoosting
#   XGBoost
#
# Principles:
#   - vocabulary learned from training only
#   - drop=None
#   - no min_frequency grouping
#   - no max_categories grouping
#   - unknown held-out levels ignored without refitting
# ============================================================

def make_ohe():

    signature = inspect.signature(
        OneHotEncoder
    )


    common = dict(

        categories="auto",

        drop=None,

        handle_unknown="ignore",

        dtype=np.float64,
    )


    if (
        "sparse_output"
        in
        signature.parameters
    ):

        return OneHotEncoder(
            sparse_output=False,
            **common
        )


    return OneHotEncoder(
        sparse=False,
        **common
    )


# ============================================================
# 9. SCALED OHE PREPROCESSOR
# ============================================================
#
# ElasticNet logistic + RBF-SVM
# ============================================================

def make_scaled_ohe_preprocessor():

    return ColumnTransformer(

        transformers=[

            (
                "numeric",

                StandardScaler(),

                NUMERIC_PREDICTORS,
            ),

            (
                "categorical",

                make_ohe(),

                CATEGORICAL_PREDICTORS,
            ),
        ],

        remainder="drop",

        sparse_threshold=0.0,

        verbose_feature_names_out=True,
    )


# ============================================================
# 10. UNSCALED OHE PREPROCESSOR
# ============================================================
#
# Conventional tree ensembles + XGBoost
# ============================================================

def make_unscaled_ohe_preprocessor():

    return ColumnTransformer(

        transformers=[

            (
                "numeric",

                "passthrough",

                NUMERIC_PREDICTORS,
            ),

            (
                "categorical",

                make_ohe(),

                CATEGORICAL_PREDICTORS,
            ),
        ],

        remainder="drop",

        sparse_threshold=0.0,

        verbose_feature_names_out=True,
    )


# ============================================================
# 11. NATIVE-CATEGORICAL FRAME BUILDER
# ============================================================
#
# Used for:
#   HistGradientBoosting
#   LightGBM
#
# Category vocabularies are learned from the training partition.
# Validation categories are aligned to training categories.
#
# Unknown validation categories become pandas categorical NaN,
# which these native implementations can handle as unknown /
# missing categorical states.
# ============================================================

def make_native_categorical_frames(
    train_df,
    valid_df,
):

    train = (
        train_df[
            PRIMARY_PREDICTORS
        ]
        .copy()
    )


    valid = (
        valid_df[
            PRIMARY_PREDICTORS
        ]
        .copy()
    )


    unseen_records = []


    for predictor in NUMERIC_PREDICTORS:

        train[predictor] = pd.to_numeric(
            train[predictor],
            errors="raise"
        ).astype(float)


        valid[predictor] = pd.to_numeric(
            valid[predictor],
            errors="raise"
        ).astype(float)


    for predictor in CATEGORICAL_PREDICTORS:

        train_raw = (
            train[predictor]
            .astype(str)
        )


        valid_raw = (
            valid[predictor]
            .astype(str)
        )


        train_categories = sorted(
            train_raw.unique()
        )


        unseen_levels = sorted(
            set(valid_raw.unique())
            -
            set(train_categories)
        )


        if unseen_levels:

            for level in unseen_levels:

                unseen_records.append(
                    {
                        "Predictor":
                            predictor,

                        "Unseen_Level":
                            level,
                    }
                )


        train[predictor] = pd.Categorical(
            train_raw,
            categories=train_categories,
            ordered=False,
        )


        valid[predictor] = pd.Categorical(
            valid_raw,
            categories=train_categories,
            ordered=False,
        )


    return (
        train,
        valid,
        unseen_records,
    )


# ============================================================
# 12. CATBOOST FEATURE FRAME
# ============================================================
#
# No external OHE.
# Numeric predictors remain numeric.
# Categorical predictors are passed as strings.
# ============================================================

def make_catboost_frames(
    train_df,
    valid_df,
):

    train = (
        train_df[
            PRIMARY_PREDICTORS
        ]
        .copy()
    )


    valid = (
        valid_df[
            PRIMARY_PREDICTORS
        ]
        .copy()
    )


    for predictor in NUMERIC_PREDICTORS:

        train[predictor] = pd.to_numeric(
            train[predictor],
            errors="raise"
        ).astype(float)


        valid[predictor] = pd.to_numeric(
            valid[predictor],
            errors="raise"
        ).astype(float)


    for predictor in CATEGORICAL_PREDICTORS:

        train[predictor] = (
            train[predictor]
            .astype(str)
        )


        valid[predictor] = (
            valid[predictor]
            .astype(str)
        )


    return train, valid


# ============================================================
# 13. FINAL MODEL-SPECIFIC POLICY
# ============================================================

model_preprocessing_policy = pd.DataFrame(
    [
        {
            "Model":
                "ElasticNet logistic regression",

            "Numeric_Preprocessing":
                "StandardScaler fitted within training partition",

            "Categorical_Preprocessing":
                (
                    "Training-fitted OHE; drop=None; "
                    "handle_unknown=ignore"
                ),

            "Representation":
                "Scaled dense numeric matrix",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                (
                    "Scale-sensitive regularized linear model; "
                    "retain symmetric OHE representation"
                ),
        },

        {
            "Model":
                "RBF SVM",

            "Numeric_Preprocessing":
                "StandardScaler fitted within training partition",

            "Categorical_Preprocessing":
                (
                    "Training-fitted OHE; drop=None; "
                    "handle_unknown=ignore"
                ),

            "Representation":
                "Scaled dense numeric matrix",

            "Probability_Interface":
                (
                    "decision_function at raw stage; "
                    "probabilities obtained later through nested calibration"
                ),

            "Primary_Rationale":
                (
                    "RBF distance depends on feature scale; "
                    "avoid SVC(probability=True)"
                ),
        },

        {
            "Model":
                "Random Forest",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                "Training-fitted OHE",

            "Representation":
                "Unscaled dense numeric matrix",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                "Tree splits do not require standardization",
        },

        {
            "Model":
                "ExtraTrees",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                "Training-fitted OHE",

            "Representation":
                "Unscaled dense numeric matrix",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                "Tree splits do not require standardization",
        },

        {
            "Model":
                "AdaBoost",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                "Training-fitted OHE",

            "Representation":
                "Unscaled dense numeric matrix",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                "Tree-based weak learners do not require scaling",
        },

        {
            "Model":
                "GradientBoosting",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                "Training-fitted OHE",

            "Representation":
                "Unscaled dense numeric matrix",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                "Tree splits do not require standardization",
        },

        {
            "Model":
                "HistGradientBoosting",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                (
                    "Native pandas categorical dtype; "
                    "categorical_features='from_dtype'"
                ),

            "Representation":
                "Mixed-type pandas DataFrame",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                "Use sklearn native categorical splitting",
        },

        {
            "Model":
                "XGBoost",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                "Training-fitted OHE",

            "Representation":
                "Unscaled dense numeric matrix",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                (
                    "Stable primary interface; native categorical "
                    "support remains version-sensitive/experimental"
                ),
        },

        {
            "Model":
                "LightGBM",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                "Native pandas categorical dtype",

            "Representation":
                "Mixed-type pandas DataFrame",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                "Use LightGBM categorical split mechanism",
        },

        {
            "Model":
                "CatBoost",

            "Numeric_Preprocessing":
                "No scaling",

            "Categorical_Preprocessing":
                "Native CatBoost categorical features",

            "Representation":
                "Mixed numeric/string DataFrame",

            "Probability_Interface":
                "predict_proba",

            "Primary_Rationale":
                (
                    "CatBoost-specific categorical processing; "
                    "no external OHE"
                ),
        },

        {
            "Model":
                "Survey-weighted logistic comparator",

            "Numeric_Preprocessing":
                "Continuous/count predictors retained",

            "Categorical_Preprocessing":
                "Prespecified reference-category dummy coding",

            "Representation":
                "Interpretable regression design matrix",

            "Probability_Interface":
                "Logistic probability",

            "Primary_Rationale":
                "Interpretability and epidemiologic comparator",
        },
    ]
)


# ============================================================
# 14. DEPENDENCY / VERSION AUDIT
# ============================================================

dependency_specs = {

    "xgboost":
        "XGBClassifier",

    "lightgbm":
        "LGBMClassifier",

    "catboost":
        "CatBoostClassifier",
}


external_classes = {}

dependency_rows = []


for package_name, class_name in (
    dependency_specs.items()
):

    try:

        module = importlib.import_module(
            package_name
        )


        cls = getattr(
            module,
            class_name
        )


        version = importlib_metadata.version(
            package_name
        )


        external_classes[
            package_name
        ] = cls


        dependency_rows.append(
            {
                "Package":
                    package_name,

                "Version":
                    version,

                "Installed":
                    True,

                "Estimator":
                    class_name,
            }
        )


    except Exception as exc:

        dependency_rows.append(
            {
                "Package":
                    package_name,

                "Version":
                    None,

                "Installed":
                    False,

                "Estimator":
                    class_name,
            }
        )


dependency_audit = pd.DataFrame(
    dependency_rows
)


missing_packages = (
    dependency_audit
    .loc[
        ~dependency_audit["Installed"],
        "Package"
    ]
    .tolist()
)


# ============================================================
# 15. SKLEARN NATIVE-CATEGORICAL SUPPORT CHECK
# ============================================================

hgb_signature = inspect.signature(
    HistGradientBoostingClassifier
)


if (
    "categorical_features"
    not in
    hgb_signature.parameters
):

    raise RuntimeError(
        "Installed scikit-learn does not expose "
        "HistGradientBoosting categorical_features."
    )


# Instantiation only; no model fitting.
try:

    _hgb_interface_check = (
        HistGradientBoostingClassifier(
            categorical_features="from_dtype",
            random_state=SEED,
        )
    )

except Exception as exc:

    raise RuntimeError(
        "Installed scikit-learn does not support "
        "categorical_features='from_dtype'."
    ) from exc


# ============================================================
# 16. ESTIMATOR INTERFACE AUDIT
# ============================================================

sklearn_estimators = {

    "ElasticNet logistic regression":
        LogisticRegression,

    "RBF SVM":
        SVC,

    "Random Forest":
        RandomForestClassifier,

    "ExtraTrees":
        ExtraTreesClassifier,

    "AdaBoost":
        AdaBoostClassifier,

    "GradientBoosting":
        GradientBoostingClassifier,

    "HistGradientBoosting":
        HistGradientBoostingClassifier,
}


interface_rows = []


for model_name, estimator_class in (
    sklearn_estimators.items()
):

    fit_signature = inspect.signature(
        estimator_class.fit
    )


    interface_rows.append(
        {
            "Model":
                model_name,

            "Package":
                "scikit-learn",

            "Version":
                sklearn.__version__,

            "Fit_Has_sample_weight":
                (
                    "sample_weight"
                    in
                    fit_signature.parameters
                ),

            "Has_predict_proba":
                hasattr(
                    estimator_class,
                    "predict_proba"
                ),

            "Has_decision_function":
                hasattr(
                    estimator_class,
                    "decision_function"
                ),
        }
    )


external_model_names = {

    "xgboost":
        "XGBoost",

    "lightgbm":
        "LightGBM",

    "catboost":
        "CatBoost",
}


for package_name, estimator_class in (
    external_classes.items()
):

    fit_signature = inspect.signature(
        estimator_class.fit
    )


    interface_rows.append(
        {
            "Model":
                external_model_names[
                    package_name
                ],

            "Package":
                package_name,

            "Version":
                importlib_metadata.version(
                    package_name
                ),

            "Fit_Has_sample_weight":
                (
                    "sample_weight"
                    in
                    fit_signature.parameters
                ),

            "Has_predict_proba":
                hasattr(
                    estimator_class,
                    "predict_proba"
                ),

            "Has_decision_function":
                hasattr(
                    estimator_class,
                    "decision_function"
                ),
        }
    )


estimator_interface_registry = pd.DataFrame(
    interface_rows
)


if not estimator_interface_registry[
    "Fit_Has_sample_weight"
].all():

    bad = (
        estimator_interface_registry
        .loc[
            ~estimator_interface_registry[
                "Fit_Has_sample_weight"
            ]
        ]
    )

    raise RuntimeError(
        "At least one installed planned estimator "
        "does not expose sample_weight:\n"
        f"{bad}"
    )


# ============================================================
# 17. GENERIC MATRIX QC
# ============================================================

def assert_finite_matrix(
    matrix,
    label,
):

    array = np.asarray(
        matrix
    )


    if array.ndim != 2:

        raise RuntimeError(
            f"{label} is not 2-dimensional."
        )


    if not np.isfinite(
        array
    ).all():

        raise RuntimeError(
            f"{label} contains NaN/Inf."
        )


    return array


# ============================================================
# 18. PARTITION-LEVEL PREPROCESSING AUDIT
# ============================================================

def audit_partition(
    train_df,
    valid_df,
    scope,
    outer_fold,
    inner_fold,
):

    X_train = (
        train_df[
            PRIMARY_PREDICTORS
        ]
        .copy()
    )


    X_valid = (
        valid_df[
            PRIMARY_PREDICTORS
        ]
        .copy()
    )


    if X_train.isna().sum().sum() != 0:

        raise RuntimeError(
            "Training predictors contain NaN."
        )


    if X_valid.isna().sum().sum() != 0:

        raise RuntimeError(
            "Validation predictors contain NaN."
        )


    # --------------------------------------------------------
    # Scaled OHE
    # --------------------------------------------------------

    scaled_preprocessor = (
        make_scaled_ohe_preprocessor()
    )


    scaled_train = (
        scaled_preprocessor
        .fit_transform(
            X_train
        )
    )


    scaled_valid = (
        scaled_preprocessor
        .transform(
            X_valid
        )
    )


    scaled_train = assert_finite_matrix(
        scaled_train,
        "Scaled training matrix"
    )


    scaled_valid = assert_finite_matrix(
        scaled_valid,
        "Scaled validation matrix"
    )


    if (
        scaled_train.shape[1]
        !=
        scaled_valid.shape[1]
    ):

        raise RuntimeError(
            "Scaled train/validation feature counts differ."
        )


    fitted_scaler = (
        scaled_preprocessor
        .named_transformers_[
            "numeric"
        ]
    )


    expected_train_means = (
        X_train[
            NUMERIC_PREDICTORS
        ]
        .astype(float)
        .mean()
        .to_numpy()
    )


    if not np.allclose(
        fitted_scaler.mean_,
        expected_train_means,
        rtol=0,
        atol=1e-12,
    ):

        raise RuntimeError(
            "StandardScaler was not fitted "
            "from training-partition means."
        )


    fitted_ohe = (
        scaled_preprocessor
        .named_transformers_[
            "categorical"
        ]
    )


    unseen_ohe_levels = 0


    for predictor, categories in zip(
        CATEGORICAL_PREDICTORS,
        fitted_ohe.categories_,
    ):

        training_levels = set(
            X_train[
                predictor
            ]
            .astype(str)
            .unique()
        )


        validation_levels = set(
            X_valid[
                predictor
            ]
            .astype(str)
            .unique()
        )


        fitted_levels = set(
            map(
                str,
                categories
            )
        )


        if fitted_levels != training_levels:

            raise RuntimeError(
                f"OHE vocabulary mismatch for {predictor}."
            )


        unseen_ohe_levels += len(
            validation_levels
            -
            training_levels
        )


    # --------------------------------------------------------
    # Unscaled OHE
    # --------------------------------------------------------

    unscaled_preprocessor = (
        make_unscaled_ohe_preprocessor()
    )


    unscaled_train = (
        unscaled_preprocessor
        .fit_transform(
            X_train
        )
    )


    unscaled_valid = (
        unscaled_preprocessor
        .transform(
            X_valid
        )
    )


    unscaled_train = assert_finite_matrix(
        unscaled_train,
        "Unscaled training matrix"
    )


    unscaled_valid = assert_finite_matrix(
        unscaled_valid,
        "Unscaled validation matrix"
    )


    if (
        unscaled_train.shape[1]
        !=
        unscaled_valid.shape[1]
    ):

        raise RuntimeError(
            "Unscaled train/validation "
            "feature counts differ."
        )


    # --------------------------------------------------------
    # HistGradientBoosting / LightGBM native categorical
    # --------------------------------------------------------

    (
        native_train,
        native_valid,
        native_unseen_records,
    ) = make_native_categorical_frames(
        train_df,
        valid_df,
    )


    if not np.isfinite(
        native_train[
            NUMERIC_PREDICTORS
        ]
        .to_numpy(
            dtype=float
        )
    ).all():

        raise RuntimeError(
            "Native training numeric matrix "
            "contains NaN/Inf."
        )


    if not np.isfinite(
        native_valid[
            NUMERIC_PREDICTORS
        ]
        .to_numpy(
            dtype=float
        )
    ).all():

        raise RuntimeError(
            "Native validation numeric matrix "
            "contains NaN/Inf."
        )


    max_native_categories = max(

        native_train[
            predictor
        ]
        .cat.categories
        .size

        for predictor in CATEGORICAL_PREDICTORS
    )


    if max_native_categories > 255:

        raise RuntimeError(
            "A native categorical predictor exceeds "
            "HistGradientBoosting max_bins support."
        )


    # --------------------------------------------------------
    # CatBoost native frame
    # --------------------------------------------------------

    (
        cat_train,
        cat_valid,
    ) = make_catboost_frames(
        train_df,
        valid_df,
    )


    if cat_train.isna().sum().sum() != 0:

        raise RuntimeError(
            "CatBoost training frame contains NaN."
        )


    if cat_valid.isna().sum().sum() != 0:

        raise RuntimeError(
            "CatBoost validation frame contains NaN."
        )


    # --------------------------------------------------------
    # Return QC row
    # --------------------------------------------------------

    return {

        "Scope":
            scope,

        "OUTER_FOLD":
            outer_fold,

        "INNER_FOLD":
            inner_fold,

        "TRAIN_N":
            int(len(train_df)),

        "VALID_N":
            int(len(valid_df)),

        "SCALED_OHE_FEATURES":
            int(
                scaled_train.shape[1]
            ),

        "UNSCALED_OHE_FEATURES":
            int(
                unscaled_train.shape[1]
            ),

        "OHE_UNSEEN_VALIDATION_LEVELS":
            int(
                unseen_ohe_levels
            ),

        "NATIVE_UNSEEN_VALIDATION_LEVELS":
            int(
                len(
                    native_unseen_records
                )
            ),

        "MAX_NATIVE_CATEGORY_COUNT":
            int(
                max_native_categories
            ),

        "SCALED_TRAIN_FINITE":
            bool(
                np.isfinite(
                    scaled_train
                ).all()
            ),

        "SCALED_VALID_FINITE":
            bool(
                np.isfinite(
                    scaled_valid
                ).all()
            ),

        "UNSCALED_TRAIN_FINITE":
            bool(
                np.isfinite(
                    unscaled_train
                ).all()
            ),

        "UNSCALED_VALID_FINITE":
            bool(
                np.isfinite(
                    unscaled_valid
                ).all()
            ),

        "SCALER_TRAIN_ONLY":
            True,

        "OHE_TRAIN_ONLY":
            True,

        "NATIVE_CATEGORY_VOCAB_TRAIN_ONLY":
            True,

        "CATBOOST_EXTERNAL_ENCODING":
            False,
    }


# ============================================================
# 19. OUTER-FOLD PREPROCESSING QC
# ============================================================

master_outer = master.merge(

    outer_assignments[
        [
            "_ROW_ID",
            "OUTER_FOLD"
        ]
    ],

    on="_ROW_ID",

    how="left",

    validate="one_to_one",
)


fold_qc_rows = []


for outer_fold in range(5):

    train_df = (
        master_outer
        .loc[
            master_outer[
                "OUTER_FOLD"
            ]
            !=
            outer_fold
        ]
        .copy()
    )


    test_df = (
        master_outer
        .loc[
            master_outer[
                "OUTER_FOLD"
            ]
            ==
            outer_fold
        ]
        .copy()
    )


    fold_qc_rows.append(
        audit_partition(
            train_df=train_df,
            valid_df=test_df,
            scope="OUTER",
            outer_fold=outer_fold,
            inner_fold=None,
        )
    )


# ============================================================
# 20. INNER-FOLD PREPROCESSING QC
# ============================================================

for outer_fold in range(5):

    inner_admin = (
        inner_assignments
        .loc[
            inner_assignments[
                "OUTER_FOLD"
            ]
            ==
            outer_fold,
            [
                "_ROW_ID",
                "INNER_FOLD",
            ]
        ]
        .copy()
    )


    outer_training_data = master.merge(

        inner_admin,

        on="_ROW_ID",

        how="inner",

        validate="one_to_one",
    )


    for inner_fold in range(4):

        train_df = (
            outer_training_data
            .loc[
                outer_training_data[
                    "INNER_FOLD"
                ]
                !=
                inner_fold
            ]
            .copy()
        )


        valid_df = (
            outer_training_data
            .loc[
                outer_training_data[
                    "INNER_FOLD"
                ]
                ==
                inner_fold
            ]
            .copy()
        )


        fold_qc_rows.append(
            audit_partition(
                train_df=train_df,
                valid_df=valid_df,
                scope="INNER",
                outer_fold=outer_fold,
                inner_fold=inner_fold,
            )
        )


fold_preprocessing_qc = pd.DataFrame(
    fold_qc_rows
)


if len(
    fold_preprocessing_qc
) != 25:

    raise RuntimeError(
        "Expected 25 preprocessing contexts "
        "(5 outer + 20 inner)."
    )


# ============================================================
# 21. DETERMINISM QC
# ============================================================
#
# Preprocessing itself contains no random component.
# Repeat one inner context and require exact equality.
# ============================================================

det_outer = 0
det_inner = 0


det_admin = (
    inner_assignments
    .loc[
        inner_assignments[
            "OUTER_FOLD"
        ]
        ==
        det_outer,
        [
            "_ROW_ID",
            "INNER_FOLD",
        ]
    ]
)


det_data = master.merge(

    det_admin,

    on="_ROW_ID",

    how="inner",

    validate="one_to_one",
)


det_train = (
    det_data
    .loc[
        det_data[
            "INNER_FOLD"
        ]
        !=
        det_inner
    ]
)


det_valid = (
    det_data
    .loc[
        det_data[
            "INNER_FOLD"
        ]
        ==
        det_inner
    ]
)


prep_1 = make_scaled_ohe_preprocessor()

prep_2 = make_scaled_ohe_preprocessor()


X1_train = prep_1.fit_transform(
    det_train[
        PRIMARY_PREDICTORS
    ]
)


X1_valid = prep_1.transform(
    det_valid[
        PRIMARY_PREDICTORS
    ]
)


X2_train = prep_2.fit_transform(
    det_train[
        PRIMARY_PREDICTORS
    ]
)


X2_valid = prep_2.transform(
    det_valid[
        PRIMARY_PREDICTORS
    ]
)


if not np.array_equal(
    X1_train,
    X2_train
):

    raise RuntimeError(
        "Training preprocessing is not deterministic."
    )


if not np.array_equal(
    X1_valid,
    X2_valid
):

    raise RuntimeError(
        "Validation preprocessing is not deterministic."
    )


# ============================================================
# 22. REPRODUCIBILITY POLICY
# ============================================================

reproducibility_policy = {

    "global_seed":
        SEED,

    "frozen_outer_folds":
        True,

    "frozen_inner_folds":
        True,

    "preprocessing_randomness":
        "none",

    "model_random_state_rule":
        (
            "Use random_state/random_seed=2026 for every "
            "stochastic estimator where supported."
        ),

    "performance_guided_preprocessing_changes":
        False,

    "automatic_frequency_grouping":
        False,

    "full_dataset_scaler_fit":
        False,

    "full_dataset_encoder_fit":
        False,

    "ordered_categories_as_equal_spaced_numeric":
        False,
}


# ============================================================
# 23. MANUSCRIPT REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "TRIPOD_AI",

            "Reference":
                (
                    "Collins GS, Moons KGM, Dhiman P, et al. "
                    "TRIPOD+AI statement. BMJ. "
                    "2024;385:e078378."
                ),

            "DOI":
                "10.1136/bmj-2023-078378",

            "Use":
                "Transparent reporting of model development and preprocessing",
        },

        {
            "ID":
                "PROBAST_AI",

            "Reference":
                (
                    "Moons KGM, Damen JAA, Kaul T, et al. "
                    "PROBAST+AI. BMJ. 2025;388:e082505."
                ),

            "DOI":
                "10.1136/bmj-2024-082505",

            "Use":
                "Risk-of-bias safeguards in model development/evaluation",
        },

        {
            "ID":
                "SKLEARN",

            "Reference":
                (
                    "Pedregosa F, Varoquaux G, Gramfort A, et al. "
                    "Scikit-learn: Machine Learning in Python. "
                    "JMLR. 2011;12:2825-2830."
                ),

            "DOI":
                "",

            "Use":
                "Software framework",
        },

        {
            "ID":
                "LIGHTGBM",

            "Reference":
                (
                    "Ke G, Meng Q, Finley T, et al. "
                    "LightGBM: A Highly Efficient Gradient "
                    "Boosting Decision Tree. NeurIPS. 2017;30."
                ),

            "DOI":
                "",

            "Use":
                "LightGBM algorithm and native categorical handling",
        },

        {
            "ID":
                "CATBOOST",

            "Reference":
                (
                    "Prokhorenkova L, Gusev G, Vorobev A, "
                    "Dorogush AV, Gulin A. CatBoost: unbiased "
                    "boosting with categorical features. "
                    "NeurIPS. 2018;31."
                ),

            "DOI":
                "",

            "Use":
                "Native categorical boosting",
        },

        {
            "ID":
                "XGBOOST",

            "Reference":
                (
                    "Chen T, Guestrin C. XGBoost: A Scalable "
                    "Tree Boosting System. KDD. 2016."
                ),

            "DOI":
                "10.1145/2939672.2939785",

            "Use":
                "XGBoost algorithm",
        },
    ]
)


# ============================================================
# 24. MANUSCRIPT-READY METHODS TEXT
# ============================================================

methods_text = """
Model-specific preprocessing

All candidate models used the same prespecified set of 15 predictors
and the same frozen nested cross-validation partitions. Preprocessing
was tailored only to the statistical and computational requirements of
each model family and was specified before model performance was
examined. All data-dependent preprocessing parameters were estimated
exclusively from the corresponding training partition and subsequently
applied unchanged to validation or test observations.

For scale-sensitive algorithms, including elastic-net logistic
regression and radial-basis-function support-vector machines, numeric
predictors were standardized using training-partition means and
standard deviations. Categorical predictors were represented using
training-fitted one-hot encoding. All encoded categories were retained
(drop=None), no automatic frequency-based category grouping was used,
and previously unseen held-out categories were handled without
refitting the encoder.

Random forests, extremely randomized trees, AdaBoost, conventional
gradient boosting, and XGBoost retained numeric predictors on their
original scales and used training-fitted one-hot encoding for
categorical predictors. XGBoost used this explicit encoded
representation in the primary analysis to provide a stable,
version-independent interface rather than relying on its evolving
native categorical implementation.

Algorithms with established native categorical-feature mechanisms used
those mechanisms directly. HistGradientBoosting received categorical
predictors as pandas categorical variables and used native categorical
splitting. LightGBM likewise received training-aligned pandas
categorical variables and used its native categorical split mechanism.
CatBoost received categorical predictors directly without external
one-hot encoding.

Maternal education and household wealth quintile were retained as
categorical predictors for machine-learning models rather than being
assigned equally spaced numerical scores. The documented
partner-education non-response category was retained in the primary
analysis and was not automatically pooled because of its small
frequency.

No preprocessing rule, category pooling decision, or feature
representation was changed after inspecting model performance.
""".strip()


# ============================================================
# 25. FINAL HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Exactly 15 primary predictors",

            "PASS":
                len(PRIMARY_PREDICTORS)
                ==
                15,
        },

        {
            "Check":
                "No forbidden variable in X",

            "PASS":
                len(leakage_features)
                ==
                0,
        },

        {
            "Check":
                "All logistic reference categories present",

            "PASS":
                bool(
                    reference_qc[
                        "Present"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Partner-education non-response n=9 retained",

            "PASS":
                partner_not_reported_n
                ==
                9,
        },

        {
            "Check":
                "All 25 preprocessing contexts audited",

            "PASS":
                len(
                    fold_preprocessing_qc
                )
                ==
                25,
        },

        {
            "Check":
                "Scaled training matrices finite",

            "PASS":
                bool(
                    fold_preprocessing_qc[
                        "SCALED_TRAIN_FINITE"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Scaled validation matrices finite",

            "PASS":
                bool(
                    fold_preprocessing_qc[
                        "SCALED_VALID_FINITE"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Unscaled training matrices finite",

            "PASS":
                bool(
                    fold_preprocessing_qc[
                        "UNSCALED_TRAIN_FINITE"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Unscaled validation matrices finite",

            "PASS":
                bool(
                    fold_preprocessing_qc[
                        "UNSCALED_VALID_FINITE"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Scaler statistics training-only",

            "PASS":
                bool(
                    fold_preprocessing_qc[
                        "SCALER_TRAIN_ONLY"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "OHE vocabulary training-only",

            "PASS":
                bool(
                    fold_preprocessing_qc[
                        "OHE_TRAIN_ONLY"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Native categorical vocabulary training-only",

            "PASS":
                bool(
                    fold_preprocessing_qc[
                        "NATIVE_CATEGORY_VOCAB_TRAIN_ONLY"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "HistGradientBoosting categorical cardinality supported",

            "PASS":
                bool(
                    (
                        fold_preprocessing_qc[
                            "MAX_NATIVE_CATEGORY_COUNT"
                        ]
                        <=
                        255
                    )
                    .all()
                ),
        },

        {
            "Check":
                "CatBoost receives no external encoding",

            "PASS":
                bool(
                    (
                        ~fold_preprocessing_qc[
                            "CATBOOST_EXTERNAL_ENCODING"
                        ]
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Preprocessing deterministic",

            "PASS":
                True,
        },

        {
            "Check":
                "Installed estimator fit interfaces accept sample_weight",

            "PASS":
                bool(
                    estimator_interface_registry[
                        "Fit_Has_sample_weight"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "No prediction model fitted",

            "PASS":
                True,
        },

        {
            "Check":
                "No performance metric calculated",

            "PASS":
                True,
        },

        {
            "Check":
                "No calibration fitted",

            "PASS":
                True,
        },

        {
            "Check":
                "No threshold selected",

            "PASS":
                True,
        },
    ]
)


if not hard_qc["PASS"].all():

    failed = hard_qc.loc[
        ~hard_qc["PASS"]
    ]

    raise RuntimeError(
        "STEP 14 HARD QC FAILED:\n"
        f"{failed}"
    )


# ============================================================
# 26. SAVE PROTOCOL / QC OUTPUTS
# ============================================================

PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_model_specific_preprocessing_protocol.csv"
)


SCHEMA_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_feature_schema.json"
)


REFERENCE_LEVEL_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_logistic_reference_categories.csv"
)


INTERFACE_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_estimator_interface_registry.csv"
)


DEPENDENCY_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_dependency_registry.csv"
)


REFERENCE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_reference_registry.csv"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_manuscript_preprocessing_methods.txt"
)


REPRODUCIBILITY_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_reproducibility_policy.json"
)


CATEGORY_FILE = (
    DIRS["qc"]
    / "STEP14_FINAL_locked_category_levels.csv"
)


FOLD_QC_FILE = (
    DIRS["qc"]
    / "STEP14_FINAL_fold_preprocessing_qc.csv"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP14_FINAL_hard_qc.csv"
)


model_preprocessing_policy.to_csv(
    PROTOCOL_FILE,
    index=False
)


reference_qc.to_csv(
    REFERENCE_LEVEL_FILE,
    index=False
)


estimator_interface_registry.to_csv(
    INTERFACE_FILE,
    index=False
)


dependency_audit.to_csv(
    DEPENDENCY_FILE,
    index=False
)


reference_registry.to_csv(
    REFERENCE_REGISTRY_FILE,
    index=False
)


category_levels.to_csv(
    CATEGORY_FILE,
    index=False
)


fold_preprocessing_qc.to_csv(
    FOLD_QC_FILE,
    index=False
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False
)


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


with open(
    REPRODUCIBILITY_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        reproducibility_policy,
        handle,
        indent=2,
        ensure_ascii=False,
    )


feature_schema = {

    "numeric_predictors":
        NUMERIC_PREDICTORS,

    "ordered_categorical_predictors":
        ORDERED_CATEGORICAL_PREDICTORS,

    "nominal_categorical_predictors":
        NOMINAL_CATEGORICAL_PREDICTORS,

    "binary_categorical_predictors":
        BINARY_CATEGORICAL_PREDICTORS,

    "categorical_predictors":
        CATEGORICAL_PREDICTORS,

    "primary_predictors":
        PRIMARY_PREDICTORS,

    "generic_ohe": {

        "drop":
            None,

        "handle_unknown":
            "ignore",

        "automatic_frequency_grouping":
            False,

        "fit_scope":
            "training partition only",
    },

    "scaled_models": [

        "ElasticNet logistic regression",

        "RBF SVM",
    ],

    "unscaled_ohe_models": [

        "Random Forest",

        "ExtraTrees",

        "AdaBoost",

        "GradientBoosting",

        "XGBoost",
    ],

    "native_categorical_models": [

        "HistGradientBoosting",

        "LightGBM",

        "CatBoost",
    ],

    "svm_raw_output":
        "decision_function",

    "svm_probability_true":
        False,

    "catboost_external_ohe":
        False,

    "random_seed":
        SEED,
}


with open(
    SCHEMA_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        feature_schema,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 27. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 14",

    "status":
        (
            "PREPROCESSING_PROTOCOL_FROZEN"
            if not missing_packages
            else
            "PREPROCESSING_FROZEN_DEPENDENCY_ACTION_REQUIRED"
        ),

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "random_seed":
        SEED,

    "primary_master_sha256":
        observed_primary_sha,

    "outer_split_sha256":
        observed_outer_sha,

    "inner_split_sha256":
        observed_inner_sha,

    "scikit_learn_version":
        sklearn.__version__,

    "external_dependencies":
        dependency_audit.to_dict(
            orient="records"
        ),

    "missing_external_packages":
        missing_packages,

    "predictor_count":
        15,

    "numeric_predictor_count":
        3,

    "categorical_predictor_count":
        12,

    "outer_preprocessing_contexts":
        5,

    "inner_preprocessing_contexts":
        20,

    "preprocessing_performance_guided":
        False,

    "model_fitting_performed":
        False,

    "performance_evaluated":
        False,

    "calibration_fitted":
        False,

    "threshold_selected":
        False,

    "protocol_sha256":
        sha256_file(
            PROTOCOL_FILE
        ),

    "schema_sha256":
        sha256_file(
            SCHEMA_FILE
        ),

    "fold_qc_sha256":
        sha256_file(
            FOLD_QC_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),

    "methods_sha256":
        sha256_file(
            METHODS_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_preprocessing_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 28. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 14."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 14."
    )


if sha256_file(
    INNER_SPLIT_FILE
) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 14."
    )


# ============================================================
# 29. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 14 — FINAL MODEL-SPECIFIC PREPROCESSING FREEZE"
)

print(
    "=" * 78
)


print(
    "\nDEPENDENCY AUDIT:"
)

display(
    dependency_audit
)


print(
    "\nMODEL-SPECIFIC PREPROCESSING POLICY:"
)

display(
    model_preprocessing_policy
)


print(
    "\nLOGISTIC REFERENCE CATEGORIES:"
)

display(
    reference_qc
)


print(
    "\nESTIMATOR INTERFACE REGISTRY:"
)

display(
    estimator_interface_registry
)


print(
    "\nFOLD PREPROCESSING QC:"
)

display(
    fold_preprocessing_qc
)


print(
    "\nFEATURE-DIMENSION / UNKNOWN-LEVEL SUMMARY:"
)

display(
    fold_preprocessing_qc[
        [
            "Scope",
            "SCALED_OHE_FEATURES",
            "UNSCALED_OHE_FEATURES",
            "OHE_UNSEEN_VALIDATION_LEVELS",
            "NATIVE_UNSEEN_VALIDATION_LEVELS",
            "MAX_NATIVE_CATEGORY_COUNT",
        ]
    ]
    .groupby(
        "Scope"
    )
    .agg(
        [
            "min",
            "max",
        ]
    )
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)

print(
    methods_text
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {observed_primary_sha}"
)

print(
    f"Outer split    : {observed_outer_sha}"
)

print(
    f"Inner split    : {observed_inner_sha}"
)


print(
    "\nIMPORTANT:"
)

print(
    "- Same 15 original predictors are used for every candidate model."
)

print(
    "- Same frozen outer/inner CV assignments are used for every model."
)

print(
    "- Preprocessing differences are algorithm-specific and prespecified."
)

print(
    "- All learned preprocessing is fitted using training partitions only."
)

print(
    "- ElasticNet and RBF-SVM receive standardized numeric predictors."
)

print(
    "- Conventional tree models and XGBoost do not receive numeric scaling."
)

print(
    "- HistGradientBoosting and LightGBM use native categorical predictors."
)

print(
    "- CatBoost uses native categorical handling without external OHE."
)

print(
    "- Maternal education and wealth are not treated as equally spaced numeric variables."
)

print(
    "- The n=9 partner-education non-response level is retained.")
    
print(
    "- No preprocessing choice was based on model performance."
)

print(
    "- No model was fitted in Step 14."
)


if missing_packages:

    print(
        "\nDEPENDENCY ACTION REQUIRED:"
    )

    print(
        missing_packages
    )

    print(
        "\nInstall the missing planned package(s) before model development."
    )

    print(
        "=" * 78
    )

    print(
        "STATUS: STEP 14 DEPENDENCY ACTION REQUIRED"
    )

    print(
        "=" * 78
    )


else:

    print(
        "=" * 78
    )

    print(
        "STATUS: STEP 14 PASS"
    )

    print(
        "=" * 78
    )


STEP 14 — FINAL MODEL-SPECIFIC PREPROCESSING FREEZE

DEPENDENCY AUDIT:


,Package,Version,Installed,Estimator
0,xgboost,3.4.1,True,XGBClassifier
1,lightgbm,4.6.0,True,LGBMClassifier
2,catboost,1.2.10,True,CatBoostClassifier



MODEL-SPECIFIC PREPROCESSING POLICY:


,Model,Numeric_Preprocessing,Categorical_Preprocessing,Representation,Probability_Interface,Primary_Rationale
0,ElasticNet logistic regression,StandardScaler fitted within training partition,Training-fitted OHE; drop=None; handle_unknown...,Scaled dense numeric matrix,predict_proba,Scale-sensitive regularized linear model; reta...
1,RBF SVM,StandardScaler fitted within training partition,Training-fitted OHE; drop=None; handle_unknown...,Scaled dense numeric matrix,decision_function at raw stage; probabilities ...,RBF distance depends on feature scale; avoid S...
2,Random Forest,No scaling,Training-fitted OHE,Unscaled dense numeric matrix,predict_proba,Tree splits do not require standardization
3,ExtraTrees,No scaling,Training-fitted OHE,Unscaled dense numeric matrix,predict_proba,Tree splits do not require standardization
4,AdaBoost,No scaling,Training-fitted OHE,Unscaled dense numeric matrix,predict_proba,Tree-based weak learners do not require scaling
5,GradientBoosting,No scaling,Training-fitted OHE,Unscaled dense numeric matrix,predict_proba,Tree splits do not require standardization
6,HistGradientBoosting,No scaling,Native pandas categorical dtype; categorical_f...,Mixed-type pandas DataFrame,predict_proba,Use sklearn native categorical splitting
7,XGBoost,No scaling,Training-fitted OHE,Unscaled dense numeric matrix,predict_proba,Stable primary interface; native categorical s...
8,LightGBM,No scaling,Native pandas categorical dtype,Mixed-type pandas DataFrame,predict_proba,Use LightGBM categorical split mechanism
9,CatBoost,No scaling,Native CatBoost categorical features,Mixed numeric/string DataFrame,predict_proba,CatBoost-specific categorical processing; no e...



LOGISTIC REFERENCE CATEGORIES:


,Predictor,Reference_Category,Present
0,maternal_education,No education,True
1,wealth_quintile,Poorest,True
2,partner_education_primary_final,No education,True
3,division,Dhaka,True
4,pregnancy_intention,Wanted then,True
5,healthcare_decision_primary_final,Respondent alone,True
6,residence,Urban,True
7,weekly_media_exposure,No,True
8,working_status,No,True
9,religion_final,Islam,True


... (5 outputs left out)

SCALED_OHE_FEATURES     UNSCALED_OHE_FEATURES      \
                      min max                   min max   
Scope                                                     
INNER                  46  46                    46  46   
OUTER                  46  46                    46  46   

      OHE_UNSEEN_VALIDATION_LEVELS     NATIVE_UNSEEN_VALIDATION_LEVELS      \
                               min max                             min max   
Scope                                                                        
INNER                            0   0                               0   0   
OUTER                            0   0                               0   0   

      MAX_NATIVE_CATEGORY_COUNT      
                            min max  
Scope                                
INNER                         8   8  
OUTER                         8   8


FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Exactly 15 primary predictors,True
4,No forbidden variable in X,True
5,All logistic reference categories present,True
6,Partner-education non-response n=9 retained,True
7,All 25 preprocessing contexts audited,True
8,Scaled training matrices finite,True
9,Scaled validation matrices finite,True



MANUSCRIPT METHODS PREVIEW:
Model-specific preprocessing

All candidate models used the same prespecified set of 15 predictors
and the same frozen nested cross-validation partitions. Preprocessing
was tailored only to the statistical and computational requirements of
each model family and was specified before model performance was
examined. All data-dependent preprocessing parameters were estimated
exclusively from the corresponding training partition and subsequently
applied unchanged to validation or test observations.

For scale-sensitive algorithms, including elastic-net logistic
regression and radial-basis-function support-vector machines, numeric
predictors were standardized using training-partition means and
standard deviations. Categorical predictors were represented using
training-fitted one-hot encoding. All encoded categories were retained
(drop=None), no automatic frequency-based category grouping was used,
and previously unseen held-out categories were handled without
ref

In [21]:
# ============================================================
# STEP 15 — FINAL HIERARCHICAL TARGETS AND
#           SURVEY-WEIGHT TRAINING PROTOCOL FREEZE
# ============================================================
#
# Primary modelling structure
# ---------------------------
# Stage 1:
#   1 = No ANC
#   0 = Any ANC
#
# Stage 2 (ANC users only):
#   1 = >=8 reported ANC visits
#   0 = 1-7 reported ANC visits
#
# Final hierarchical probabilities:
#   p0 = q0
#   p2 = (1 - q0) * q2
#   p1 = (1 - q0) * (1 - q2)
#
# Weighting policy
# ----------------
# Primary model fitting:
#   DHS survey sampling weight only.
#
# Training-fold normalization:
#   w_fit = SURVEY_WEIGHT / mean(SURVEY_WEIGHT)
#
# Weight normalization is performed separately within each
# stage-specific training population.
#
# No:
#   - SMOTE
#   - random oversampling
#   - random undersampling
#   - inverse-frequency class weighting
#   - class_weight="balanced"
#   - scale_pos_weight tuning
#   - outcome-prevalence reweighting
#
# Held-out evaluation retains the original survey weights.
#
# This step does NOT fit prediction models, inspect performance,
# fit calibration models, or select decision thresholds.
# ============================================================


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. PROJECT / FROZEN INPUTS
# ============================================================

SEED = 2026


if "DIRS" not in globals():

    raise RuntimeError(
        "DIRS is unavailable. "
        "Run the accepted project setup first."
    )


PROJECT_ROOT = Path(
    DIRS["processed"]
).parent


PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


STEP14_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_preprocessing_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(block)

    return digest.hexdigest()


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP14_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing:\n{path}"
        )


observed_primary_sha = sha256_file(
    PRIMARY_FILE
)

observed_outer_sha = sha256_file(
    OUTER_SPLIT_FILE
)

observed_inner_sha = sha256_file(
    INNER_SPLIT_FILE
)


if observed_primary_sha != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if observed_outer_sha != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if observed_inner_sha != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


with open(
    STEP14_MANIFEST_FILE,
    "r",
    encoding="utf-8",
) as handle:

    step14_manifest = json.load(
        handle
    )


if (
    step14_manifest.get(
        "primary_master_sha256"
    )
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Step-14 manifest references "
        "a different primary master."
    )


if (
    step14_manifest.get(
        "outer_split_sha256"
    )
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Step-14 outer-split hash mismatch."
    )


if (
    step14_manifest.get(
        "inner_split_sha256"
    )
    !=
    EXPECTED_INNER_SHA256
):

    raise RuntimeError(
        "Step-14 inner-split hash mismatch."
    )


# ============================================================
# 2. LOAD DATA
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


outer_assignments = pd.read_csv(
    OUTER_SPLIT_FILE
)


inner_assignments = pd.read_csv(
    INNER_SPLIT_FILE
)


required_columns = {

    "_ROW_ID",
    "ANC_CLASS",
    "SURVEY_WEIGHT",
    "PSU",
    "STRATUM",
}


missing_columns = (
    required_columns
    -
    set(master.columns)
)


if missing_columns:

    raise RuntimeError(
        "Required columns missing:\n"
        f"{sorted(missing_columns)}"
    )


if master.shape != (5063, 23):

    raise RuntimeError(
        f"Unexpected master shape: {master.shape}"
    )


if master["_ROW_ID"].duplicated().any():

    raise RuntimeError(
        "Duplicate _ROW_ID detected."
    )


# ============================================================
# 3. OUTCOME SOURCE QC
# ============================================================

master["ANC_CLASS"] = pd.to_numeric(
    master["ANC_CLASS"],
    errors="raise",
).astype(int)


observed_classes = set(
    master["ANC_CLASS"].unique()
)


if observed_classes != {0, 1, 2}:

    raise RuntimeError(
        "Unexpected ANC_CLASS values:\n"
        f"{sorted(observed_classes)}"
    )


class_counts = (
    master["ANC_CLASS"]
    .value_counts()
    .sort_index()
)


expected_class_counts = {

    0: 403,
    1: 4407,
    2: 253,
}


for class_id, expected_n in (
    expected_class_counts.items()
):

    observed_n = int(
        class_counts.loc[
            class_id
        ]
    )


    if observed_n != expected_n:

        raise RuntimeError(
            f"ANC_CLASS={class_id}: "
            f"expected {expected_n}, "
            f"observed {observed_n}."
        )


# ============================================================
# 4. SURVEY-WEIGHT QC
# ============================================================

master["SURVEY_WEIGHT"] = pd.to_numeric(
    master["SURVEY_WEIGHT"],
    errors="raise",
).astype(float)


survey_weight = (
    master["SURVEY_WEIGHT"]
    .to_numpy(
        dtype=float
    )
)


if not np.isfinite(
    survey_weight
).all():

    raise RuntimeError(
        "SURVEY_WEIGHT contains NaN/Inf."
    )


if (
    survey_weight
    <=
    0
).any():

    raise RuntimeError(
        "SURVEY_WEIGHT contains "
        "non-positive values."
    )


# ============================================================
# 5. FINAL HIERARCHICAL TARGETS
# ============================================================

# Stage 1:
# 1 = No ANC
# 0 = Any ANC

master[
    "STAGE1_NO_ANC_TARGET"
] = (
    master[
        "ANC_CLASS"
    ]
    .eq(0)
    .astype(np.int8)
)


# Stage 2 eligibility:
# ANC users only (ANC_CLASS 1 or 2).

master[
    "STAGE2_ELIGIBLE"
] = (
    master[
        "ANC_CLASS"
    ]
    .isin(
        [1, 2]
    )
)


# Stage 2 contains only ANC users.
# Structural non-applicability is therefore not encoded as
# a missing binary outcome.

stage2_master = (
    master.loc[
        master[
            "STAGE2_ELIGIBLE"
        ]
    ]
    .copy()
)


stage2_master[
    "STAGE2_GE8_TARGET"
] = (
    stage2_master[
        "ANC_CLASS"
    ]
    .eq(2)
    .astype(np.int8)
)


# ============================================================
# 6. TARGET COUNT QC
# ============================================================

stage1_positive_n = int(
    master[
        "STAGE1_NO_ANC_TARGET"
    ]
    .sum()
)


stage1_negative_n = int(
    len(master)
    -
    stage1_positive_n
)


stage2_positive_n = int(
    stage2_master[
        "STAGE2_GE8_TARGET"
    ]
    .sum()
)


stage2_negative_n = int(
    len(stage2_master)
    -
    stage2_positive_n
)


if stage1_positive_n != 403:

    raise RuntimeError(
        "Stage-1 positive count must be 403."
    )


if stage1_negative_n != 4660:

    raise RuntimeError(
        "Stage-1 negative count must be 4660."
    )


if len(stage2_master) != 4660:

    raise RuntimeError(
        "Stage-2 eligible N must be 4660."
    )


if stage2_positive_n != 253:

    raise RuntimeError(
        "Stage-2 positive count must be 253."
    )


if stage2_negative_n != 4407:

    raise RuntimeError(
        "Stage-2 negative count must be 4407."
    )


# Stage 2 must contain ANC users only.
if (
    stage2_master[
        "ANC_CLASS"
    ]
    .eq(0)
    .any()
):

    raise RuntimeError(
        "Stage 2 incorrectly contains "
        "one or more No-ANC observations."
    )


# ============================================================
# 7. WEIGHT UTILITIES
# ============================================================

def weighted_mean(
    values,
    weights,
):

    values = np.asarray(
        values,
        dtype=float,
    )


    weights = np.asarray(
        weights,
        dtype=float,
    )


    if len(values) != len(weights):

        raise RuntimeError(
            "Values/weights length mismatch."
        )


    if not np.isfinite(
        values
    ).all():

        raise RuntimeError(
            "Non-finite values."
        )


    if not np.isfinite(
        weights
    ).all():

        raise RuntimeError(
            "Non-finite weights."
        )


    if (
        weights
        <=
        0
    ).any():

        raise RuntimeError(
            "Weights must be positive."
        )


    return float(
        np.average(
            values,
            weights=weights,
        )
    )


def kish_effective_sample_size(
    weights,
):

    weights = np.asarray(
        weights,
        dtype=float,
    )


    numerator = (
        weights.sum()
        ** 2
    )


    denominator = np.square(
        weights
    ).sum()


    return float(
        numerator
        /
        denominator
    )


def normalize_training_weights(
    weights,
):

    weights = np.asarray(
        weights,
        dtype=float,
    )


    if len(weights) == 0:

        raise RuntimeError(
            "Cannot normalize empty weight vector."
        )


    if not np.isfinite(
        weights
    ).all():

        raise RuntimeError(
            "Training weights contain NaN/Inf."
        )


    if (
        weights
        <=
        0
    ).any():

        raise RuntimeError(
            "Training weights must be positive."
        )


    normalized = (
        weights
        /
        weights.mean()
    )


    if not np.isclose(
        normalized.mean(),
        1.0,
        rtol=0,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Normalized training weights "
            "do not have mean 1."
        )


    return normalized


# ============================================================
# 8. FULL-COHORT TARGET SUMMARY
# ============================================================

stage1_raw_weights = (
    master[
        "SURVEY_WEIGHT"
    ]
    .to_numpy(
        dtype=float
    )
)


stage2_raw_weights = (
    stage2_master[
        "SURVEY_WEIGHT"
    ]
    .to_numpy(
        dtype=float
    )
)


target_summary = pd.DataFrame(
    [
        {
            "Stage":
                "Stage 1",

            "Target":
                "No ANC vs Any ANC",

            "Eligible_N":
                int(
                    len(master)
                ),

            "Positive_Label":
                "No ANC",

            "Positive_N":
                stage1_positive_n,

            "Negative_Label":
                "Any ANC",

            "Negative_N":
                stage1_negative_n,

            "Unweighted_Positive_Prevalence":
                float(
                    master[
                        "STAGE1_NO_ANC_TARGET"
                    ]
                    .mean()
                ),

            "Survey_Weighted_Positive_Prevalence":
                weighted_mean(
                    master[
                        "STAGE1_NO_ANC_TARGET"
                    ],
                    stage1_raw_weights,
                ),

            "Survey_Weight_Kish_ESS":
                kish_effective_sample_size(
                    stage1_raw_weights
                ),
        },

        {
            "Stage":
                "Stage 2",

            "Target":
                ">=8 vs 1-7 ANC visits among ANC users",

            "Eligible_N":
                int(
                    len(
                        stage2_master
                    )
                ),

            "Positive_Label":
                ">=8 reported ANC visits",

            "Positive_N":
                stage2_positive_n,

            "Negative_Label":
                "1-7 ANC visits",

            "Negative_N":
                stage2_negative_n,

            "Unweighted_Positive_Prevalence":
                float(
                    stage2_master[
                        "STAGE2_GE8_TARGET"
                    ]
                    .mean()
                ),

            "Survey_Weighted_Positive_Prevalence":
                weighted_mean(
                    stage2_master[
                        "STAGE2_GE8_TARGET"
                    ],
                    stage2_raw_weights,
                ),

            "Survey_Weight_Kish_ESS":
                kish_effective_sample_size(
                    stage2_raw_weights
                ),
        },
    ]
)


# ============================================================
# 9. STAGE-SPECIFIC TRAINING DATA
# ============================================================

def get_stage_training_data(
    dataframe,
    stage,
):

    if stage == 1:

        out = dataframe.copy()

        y = (
            out[
                "ANC_CLASS"
            ]
            .eq(0)
            .astype(np.int8)
            .to_numpy()
        )


    elif stage == 2:

        out = (
            dataframe.loc[
                dataframe[
                    "ANC_CLASS"
                ]
                .isin(
                    [1, 2]
                )
            ]
            .copy()
        )


        y = (
            out[
                "ANC_CLASS"
            ]
            .eq(2)
            .astype(np.int8)
            .to_numpy()
        )


    else:

        raise ValueError(
            "stage must be 1 or 2."
        )


    raw_weights = (
        out[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    fit_weights = (
        normalize_training_weights(
            raw_weights
        )
    )


    return (
        out,
        y,
        raw_weights,
        fit_weights,
    )


# ============================================================
# 10. TRAINING-WEIGHT AUDIT
# ============================================================

def audit_training_partition(
    dataframe,
    scope,
    outer_fold,
    inner_fold,
    stage,
):

    (
        stage_data,
        y,
        raw_weights,
        fit_weights,
    ) = get_stage_training_data(
        dataframe,
        stage,
    )


    if len(
        np.unique(
            y
        )
    ) != 2:

        raise RuntimeError(
            f"{scope}, outer={outer_fold}, "
            f"inner={inner_fold}, stage={stage}: "
            "both target classes are required."
        )


    return {

        "Scope":
            scope,

        "OUTER_FOLD":
            outer_fold,

        "INNER_FOLD":
            inner_fold,

        "Stage":
            stage,

        "Eligible_Training_N":
            int(
                len(
                    stage_data
                )
            ),

        "Positive_N":
            int(
                y.sum()
            ),

        "Negative_N":
            int(
                len(y)
                -
                y.sum()
            ),

        "Unweighted_Positive_Prevalence":
            float(
                y.mean()
            ),

        "Survey_Weighted_Positive_Prevalence":
            weighted_mean(
                y,
                raw_weights,
            ),

        "Raw_Weight_Min":
            float(
                raw_weights.min()
            ),

        "Raw_Weight_Mean":
            float(
                raw_weights.mean()
            ),

        "Raw_Weight_Max":
            float(
                raw_weights.max()
            ),

        "Normalized_Weight_Min":
            float(
                fit_weights.min()
            ),

        "Normalized_Weight_Mean":
            float(
                fit_weights.mean()
            ),

        "Normalized_Weight_Max":
            float(
                fit_weights.max()
            ),

        "Kish_ESS":
            kish_effective_sample_size(
                raw_weights
            ),

        "Class_Balancing_Applied":
            False,

        "Survey_Weight_Only":
            True,

        "Weight_Normalization":
            "mean=1 within stage-specific training partition",
    }


# ============================================================
# 11. HELD-OUT COMPOSITION AUDIT
# ============================================================

def audit_validation_partition(
    dataframe,
    scope,
    outer_fold,
    inner_fold,
    stage,
):

    if stage == 1:

        stage_data = dataframe.copy()

        y = (
            stage_data[
                "ANC_CLASS"
            ]
            .eq(0)
            .astype(np.int8)
            .to_numpy()
        )


    elif stage == 2:

        stage_data = (
            dataframe.loc[
                dataframe[
                    "ANC_CLASS"
                ]
                .isin(
                    [1, 2]
                )
            ]
            .copy()
        )


        y = (
            stage_data[
                "ANC_CLASS"
            ]
            .eq(2)
            .astype(np.int8)
            .to_numpy()
        )


    else:

        raise ValueError(
            "stage must be 1 or 2."
        )


    raw_weights = (
        stage_data[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    if len(
        np.unique(
            y
        )
    ) != 2:

        raise RuntimeError(
            f"{scope}, outer={outer_fold}, "
            f"inner={inner_fold}, stage={stage}: "
            "held-out partition lacks a target class."
        )


    return {

        "Scope":
            scope,

        "OUTER_FOLD":
            outer_fold,

        "INNER_FOLD":
            inner_fold,

        "Stage":
            stage,

        "Eligible_Validation_N":
            int(
                len(
                    stage_data
                )
            ),

        "Positive_N":
            int(
                y.sum()
            ),

        "Negative_N":
            int(
                len(y)
                -
                y.sum()
            ),

        "Unweighted_Positive_Prevalence":
            float(
                y.mean()
            ),

        "Survey_Weighted_Positive_Prevalence":
            weighted_mean(
                y,
                raw_weights,
            ),

        "Evaluation_Weight":
            "original SURVEY_WEIGHT",

        "Outcome_Based_Reweighting":
            False,
    }


# ============================================================
# 12. OUTER-FOLD AUDIT
# ============================================================

master_outer = master.merge(

    outer_assignments[
        [
            "_ROW_ID",
            "OUTER_FOLD",
        ]
    ],

    on="_ROW_ID",

    how="left",

    validate="one_to_one",
)


if master_outer[
    "OUTER_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing outer-fold assignment."
    )


training_audit_rows = []

validation_audit_rows = []


for outer_fold in range(5):

    outer_train = (
        master_outer.loc[
            master_outer[
                "OUTER_FOLD"
            ]
            !=
            outer_fold
        ]
        .copy()
    )


    outer_test = (
        master_outer.loc[
            master_outer[
                "OUTER_FOLD"
            ]
            ==
            outer_fold
        ]
        .copy()
    )


    for stage in [1, 2]:

        training_audit_rows.append(
            audit_training_partition(
                dataframe=outer_train,
                scope="OUTER_TRAIN",
                outer_fold=outer_fold,
                inner_fold=None,
                stage=stage,
            )
        )


        validation_audit_rows.append(
            audit_validation_partition(
                dataframe=outer_test,
                scope="OUTER_TEST",
                outer_fold=outer_fold,
                inner_fold=None,
                stage=stage,
            )
        )


# ============================================================
# 13. INNER-FOLD AUDIT
# ============================================================

for outer_fold in range(5):

    inner_admin = (
        inner_assignments.loc[
            inner_assignments[
                "OUTER_FOLD"
            ]
            ==
            outer_fold,
            [
                "_ROW_ID",
                "INNER_FOLD",
            ]
        ]
        .copy()
    )


    outer_training_data = master.merge(

        inner_admin,

        on="_ROW_ID",

        how="inner",

        validate="one_to_one",
    )


    expected_outer_training_n = int(
        (
            master_outer[
                "OUTER_FOLD"
            ]
            !=
            outer_fold
        )
        .sum()
    )


    if (
        len(
            outer_training_data
        )
        !=
        expected_outer_training_n
    ):

        raise RuntimeError(
            f"Outer fold {outer_fold}: "
            "inner-CV population does not equal "
            "the frozen outer-training population."
        )


    for inner_fold in range(4):

        inner_train = (
            outer_training_data.loc[
                outer_training_data[
                    "INNER_FOLD"
                ]
                !=
                inner_fold
            ]
            .copy()
        )


        inner_valid = (
            outer_training_data.loc[
                outer_training_data[
                    "INNER_FOLD"
                ]
                ==
                inner_fold
            ]
            .copy()
        )


        for stage in [1, 2]:

            training_audit_rows.append(
                audit_training_partition(
                    dataframe=inner_train,
                    scope="INNER_TRAIN",
                    outer_fold=outer_fold,
                    inner_fold=inner_fold,
                    stage=stage,
                )
            )


            validation_audit_rows.append(
                audit_validation_partition(
                    dataframe=inner_valid,
                    scope="INNER_VALID",
                    outer_fold=outer_fold,
                    inner_fold=inner_fold,
                    stage=stage,
                )
            )


training_weight_audit = pd.DataFrame(
    training_audit_rows
)


validation_composition_audit = pd.DataFrame(
    validation_audit_rows
)


# 25 resampling contexts x 2 stages.

if len(
    training_weight_audit
) != 50:

    raise RuntimeError(
        "Expected 50 training-weight audit rows."
    )


if len(
    validation_composition_audit
) != 50:

    raise RuntimeError(
        "Expected 50 validation audit rows."
    )


# ============================================================
# 14. TRAINING-WEIGHT HARD CHECKS
# ============================================================

if not np.allclose(
    training_weight_audit[
        "Normalized_Weight_Mean"
    ],
    1.0,
    rtol=0,
    atol=1e-12,
):

    raise RuntimeError(
        "At least one training weight vector "
        "does not have mean 1."
    )


if training_weight_audit[
    "Class_Balancing_Applied"
].any():

    raise RuntimeError(
        "Unexpected class balancing detected."
    )


if not training_weight_audit[
    "Survey_Weight_Only"
].all():

    raise RuntimeError(
        "Non-survey training weight detected."
    )


if (
    training_weight_audit[
        "Positive_N"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "Training partition without positive cases."
    )


if (
    training_weight_audit[
        "Negative_N"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "Training partition without negative cases."
    )


if (
    validation_composition_audit[
        "Positive_N"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "Held-out partition without positive cases."
    )


if (
    validation_composition_audit[
        "Negative_N"
    ]
    <=
    0
).any():

    raise RuntimeError(
        "Held-out partition without negative cases."
    )


# ============================================================
# 15. MODEL-SPECIFIC IMBALANCE CONTROLS
# ============================================================
#
# All automatic imbalance controls are frozen at neutral values.
# They will not enter hyperparameter optimization.
# ============================================================

imbalance_control_policy = pd.DataFrame(
    [
        {
            "Model":
                "ElasticNet logistic regression",

            "Control":
                "class_weight",

            "Primary_Value":
                "None",
        },

        {
            "Model":
                "RBF SVM",

            "Control":
                "class_weight",

            "Primary_Value":
                "None",
        },

        {
            "Model":
                "Random Forest",

            "Control":
                "class_weight",

            "Primary_Value":
                "None",
        },

        {
            "Model":
                "ExtraTrees",

            "Control":
                "class_weight",

            "Primary_Value":
                "None",
        },

        {
            "Model":
                "AdaBoost",

            "Control":
                "resampling / class multiplier",

            "Primary_Value":
                "None",
        },

        {
            "Model":
                "GradientBoosting",

            "Control":
                "resampling / class multiplier",

            "Primary_Value":
                "None",
        },

        {
            "Model":
                "HistGradientBoosting",

            "Control":
                "class weighting beyond sample_weight",

            "Primary_Value":
                "None",
        },

        {
            "Model":
                "XGBoost",

            "Control":
                "scale_pos_weight",

            "Primary_Value":
                "1.0",
        },

        {
            "Model":
                "LightGBM",

            "Control":
                "class_weight / is_unbalance / scale_pos_weight",

            "Primary_Value":
                "None / False / 1.0",
        },

        {
            "Model":
                "CatBoost",

            "Control":
                "class_weights / auto_class_weights",

            "Primary_Value":
                "None / None",
        },
    ]
)


# ============================================================
# 16. FINAL WEIGHTING PROTOCOL
# ============================================================

weighting_protocol = {

    "primary_training_weight":
        "DHS survey sampling weight only",

    "training_weight_formula":
        (
            "SURVEY_WEIGHT divided by its mean within "
            "the current stage-specific training partition"
        ),

    "training_weight_mean":
        1.0,

    "stage1_weight_normalization_population":
        "all observations in the current training partition",

    "stage2_weight_normalization_population":
        (
            "ANC users only (ANC_CLASS in {1,2}) "
            "within the current training partition"
        ),

    "validation_and_test_weight":
        "original SURVEY_WEIGHT",

    "survey_weight_used_as_predictor":
        False,

    "outcome_used_to_construct_survey_weight":
        False,

    "class_weighting_primary":
        False,

    "inverse_frequency_weighting":
        False,

    "smote":
        False,

    "random_oversampling":
        False,

    "random_undersampling":
        False,

    "synthetic_sampling":
        False,

    "scale_pos_weight_tuning":
        False,

    "class_weight_balanced":
        False,

    "rationale":
        (
            "Preserve survey-weighted target-population representation "
            "without altering the outcome prevalence learned by "
            "probabilistic models."
        ),

    "class_imbalance_handling":
        (
            "No training-distribution manipulation. "
            "Later decision thresholds, where needed, will be selected "
            "only within the frozen nested validation framework."
        ),
}


# ============================================================
# 17. HIERARCHICAL PROBABILITY POLICY
# ============================================================

hierarchical_protocol = {

    "stage1": {

        "training_population":
            "all analytic observations",

        "positive_class":
            "No ANC",

        "positive_code":
            1,

        "negative_class":
            "Any ANC",

        "negative_code":
            0,
    },

    "stage2": {

        "training_population":
            "ANC users only",

        "eligibility":
            "ANC_CLASS in {1,2}",

        "positive_class":
            ">=8 reported ANC visits",

        "positive_code":
            1,

        "negative_class":
            "1-7 ANC visits",

        "negative_code":
            0,

        "standalone_validation_population":
            "true ANC users only",

        "fusion_prediction_population":
            (
                "all held-out observations; "
                "q2 is interpreted conditionally on Any ANC"
            ),
    },

    "fusion": {

        "q0":
            "P(No ANC | X)",

        "q2":
            "P(>=8 ANC visits | Any ANC, X)",

        "p0":
            "q0",

        "p1":
            "(1 - q0) * (1 - q2)",

        "p2":
            "(1 - q0) * q2",

        "probability_sum":
            "p0 + p1 + p2 = 1",
    },

    "final_classes": {

        "0":
            "No ANC",

        "1":
            "1-7 ANC visits",

        "2":
            ">=8 reported ANC visits",
    },
}


# ============================================================
# 18. RESTRICTED ROW-LEVEL TARGET FILES
# ============================================================
#
# Local/restricted reproducibility artifacts.
# Do NOT upload these CSVs to a public GitHub repository.
# ============================================================

stage1_restricted = master[
    [
        "_ROW_ID",
        "ANC_CLASS",
        "STAGE1_NO_ANC_TARGET",
        "SURVEY_WEIGHT",
        "PSU",
        "STRATUM",
    ]
].copy()


stage2_restricted = stage2_master[
    [
        "_ROW_ID",
        "ANC_CLASS",
        "STAGE2_GE8_TARGET",
        "SURVEY_WEIGHT",
        "PSU",
        "STRATUM",
    ]
].copy()


STAGE1_RESTRICTED_FILE = (
    DIRS["processed"]
    / "STEP15_STAGE1_targets_RESTRICTED.csv"
)


STAGE2_RESTRICTED_FILE = (
    DIRS["processed"]
    / "STEP15_STAGE2_targets_RESTRICTED.csv"
)


stage1_restricted.to_csv(
    STAGE1_RESTRICTED_FILE,
    index=False,
)


stage2_restricted.to_csv(
    STAGE2_RESTRICTED_FILE,
    index=False,
)


# ============================================================
# 19. REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "TRIPOD_AI",

            "Reference":
                (
                    "Collins GS, Moons KGM, Dhiman P, et al. "
                    "TRIPOD+AI statement. "
                    "BMJ. 2024;385:e078378."
                ),

            "DOI":
                "10.1136/bmj-2023-078378",

            "Use":
                (
                    "Transparent reporting of model "
                    "development and validation"
                ),
        },

        {
            "ID":
                "PROBAST_AI",

            "Reference":
                (
                    "Moons KGM, Damen JAA, Kaul T, et al. "
                    "PROBAST+AI. BMJ. 2025;388:e082505."
                ),

            "DOI":
                "10.1136/bmj-2024-082505",

            "Use":
                (
                    "Risk-of-bias safeguards and "
                    "class-imbalance handling"
                ),
        },

        {
            "ID":
                "MACNELL_2023",

            "Reference":
                (
                    "MacNell N, Feinstein L, Wilkerson J, "
                    "et al. Implementing machine learning "
                    "methods with complex survey data: "
                    "Lessons learned on the impacts of "
                    "accounting sampling weights in "
                    "gradient boosting. "
                    "PLoS One. 2023;18(1):e0280387."
                ),

            "DOI":
                "10.1371/journal.pone.0280387",

            "Use":
                (
                    "Sampling weights in machine learning "
                    "with complex survey data"
                ),
        },

        {
            "ID":
                "VAN_DEN_GOORBERGH_2022",

            "Reference":
                (
                    "van den Goorbergh R, van Smeden M, "
                    "Timmerman D, Van Calster B. "
                    "The harm of class imbalance corrections "
                    "for risk prediction models: illustration "
                    "and simulation using logistic regression. "
                    "J Am Med Inform Assoc. "
                    "2022;29(9):1525-1534."
                ),

            "DOI":
                "10.1093/jamia/ocac093",

            "Use":
                (
                    "Avoid routine class-imbalance correction "
                    "when calibrated probabilities are required"
                ),
        },

        {
            "ID":
                "PICCININNI_2024",

            "Reference":
                (
                    "Piccininni M, Wechsung M, Van Calster B, "
                    "et al. Understanding random resampling "
                    "techniques for class imbalance correction "
                    "and their consequences on calibration and "
                    "discrimination of clinical risk prediction "
                    "models. J Biomed Inform. "
                    "2024;155:104666."
                ),

            "DOI":
                "10.1016/j.jbi.2024.104666",

            "Use":
                (
                    "Calibration consequences of "
                    "class-resampling methods"
                ),
        },
    ]
)


# ============================================================
# 20. MANUSCRIPT-READY METHODS TEXT
# ============================================================

methods_text = """
Hierarchical outcome modelling and survey weighting

The three-category antenatal-care outcome was modelled using a
prespecified two-stage hierarchical classification framework. Stage 1
distinguished women reporting no antenatal-care visits from those
reporting any antenatal care. Stage 2 was developed exclusively among
women reporting at least one antenatal-care visit and distinguished
women reporting eight or more visits from those reporting one to seven
visits. For an individual with Stage-1 probability q0=P(no ANC|X) and
Stage-2 conditional probability q2=P(>=8 visits|any ANC,X), the final
three-category probabilities were defined as p0=q0,
p1=(1-q0)(1-q2), and p2=(1-q0)q2.

DHS sampling weights were incorporated during model fitting to retain
the relative sampling-weight structure of the survey. Within each
stage-specific training partition, sampling weights were rescaled by a
single positive constant to have mean one. Thus, relative sampling
weights were preserved while the overall scale of the weighted loss was
kept comparable across resampling partitions. Original survey weights
were retained for validation and test-set performance estimation.

No random over-sampling, random under-sampling, SMOTE, synthetic
sampling, inverse-frequency class weighting, or algorithm-specific
automatic class balancing was used in the primary analysis. This
decision was prespecified before model fitting because altering the
training class distribution can distort probability calibration and is
not required solely because an outcome is infrequent. Class-specific
decision thresholds, where required, will instead be selected entirely
within the nested validation procedure without modifying the training
outcome prevalence.

Survey weights were not included as predictive features. PSU and
stratum identifiers were likewise excluded from the predictor set;
their design information was addressed through the prespecified
cluster-aware resampling framework and subsequent survey-aware
uncertainty procedures.
""".strip()


# ============================================================
# 21. SAVE SHAREABLE PROTOCOL / QC OUTPUTS
# ============================================================

TARGET_SUMMARY_FILE = (
    DIRS["qc"]
    / "STEP15_FINAL_target_prevalence_summary.csv"
)


TRAIN_WEIGHT_AUDIT_FILE = (
    DIRS["qc"]
    / "STEP15_FINAL_training_weight_audit.csv"
)


VALIDATION_AUDIT_FILE = (
    DIRS["qc"]
    / "STEP15_FINAL_validation_composition_audit.csv"
)


IMBALANCE_POLICY_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_model_imbalance_control_policy.csv"
)


WEIGHT_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_survey_weight_protocol.json"
)


HIERARCHICAL_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_hierarchical_target_protocol.json"
)


REFERENCE_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_reference_registry.csv"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_manuscript_methods.txt"
)


target_summary.to_csv(
    TARGET_SUMMARY_FILE,
    index=False,
)


training_weight_audit.to_csv(
    TRAIN_WEIGHT_AUDIT_FILE,
    index=False,
)


validation_composition_audit.to_csv(
    VALIDATION_AUDIT_FILE,
    index=False,
)


imbalance_control_policy.to_csv(
    IMBALANCE_POLICY_FILE,
    index=False,
)


reference_registry.to_csv(
    REFERENCE_FILE,
    index=False,
)


with open(
    WEIGHT_PROTOCOL_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        weighting_protocol,
        handle,
        indent=2,
        ensure_ascii=False,
    )


with open(
    HIERARCHICAL_PROTOCOL_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        hierarchical_protocol,
        handle,
        indent=2,
        ensure_ascii=False,
    )


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


# ============================================================
# 22. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Stage 1 N=5063",

            "PASS":
                len(master)
                ==
                5063,
        },

        {
            "Check":
                "Stage 1 positives=403",

            "PASS":
                stage1_positive_n
                ==
                403,
        },

        {
            "Check":
                "Stage 1 negatives=4660",

            "PASS":
                stage1_negative_n
                ==
                4660,
        },

        {
            "Check":
                "Stage 2 eligible N=4660",

            "PASS":
                len(stage2_master)
                ==
                4660,
        },

        {
            "Check":
                "Stage 2 positives=253",

            "PASS":
                stage2_positive_n
                ==
                253,
        },

        {
            "Check":
                "Stage 2 negatives=4407",

            "PASS":
                stage2_negative_n
                ==
                4407,
        },

        {
            "Check":
                "Stage 2 contains no No-ANC observations",

            "PASS":
                not stage2_master[
                    "ANC_CLASS"
                ]
                .eq(0)
                .any(),
        },

        {
            "Check":
                "All survey weights positive",

            "PASS":
                bool(
                    (
                        master[
                            "SURVEY_WEIGHT"
                        ]
                        >
                        0
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All survey weights finite",

            "PASS":
                bool(
                    np.isfinite(
                        master[
                            "SURVEY_WEIGHT"
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "50 training-stage contexts audited",

            "PASS":
                len(
                    training_weight_audit
                )
                ==
                50,
        },

        {
            "Check":
                "50 held-out-stage contexts audited",

            "PASS":
                len(
                    validation_composition_audit
                )
                ==
                50,
        },

        {
            "Check":
                "Training weight mean=1 in every context",

            "PASS":
                bool(
                    np.allclose(
                        training_weight_audit[
                            "Normalized_Weight_Mean"
                        ],
                        1.0,
                        rtol=0,
                        atol=1e-12,
                    )
                ),
        },

        {
            "Check":
                "Survey weight only for primary model fitting",

            "PASS":
                bool(
                    training_weight_audit[
                        "Survey_Weight_Only"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "No class balancing applied",

            "PASS":
                bool(
                    (
                        ~training_weight_audit[
                            "Class_Balancing_Applied"
                        ]
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Both classes present in every training context",

            "PASS":
                bool(
                    (
                        (
                            training_weight_audit[
                                "Positive_N"
                            ]
                            >
                            0
                        )
                        &
                        (
                            training_weight_audit[
                                "Negative_N"
                            ]
                            >
                            0
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Both classes present in every held-out context",

            "PASS":
                bool(
                    (
                        (
                            validation_composition_audit[
                                "Positive_N"
                            ]
                            >
                            0
                        )
                        &
                        (
                            validation_composition_audit[
                                "Negative_N"
                            ]
                            >
                            0
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "No oversampling/undersampling/SMOTE",

            "PASS":
                (
                    not weighting_protocol[
                        "smote"
                    ]
                    and
                    not weighting_protocol[
                        "random_oversampling"
                    ]
                    and
                    not weighting_protocol[
                        "random_undersampling"
                    ]
                ),
        },

        {
            "Check":
                "No inverse-frequency class weighting",

            "PASS":
                not weighting_protocol[
                    "inverse_frequency_weighting"
                ],
        },

        {
            "Check":
                "No model fitted",

            "PASS":
                True,
        },

        {
            "Check":
                "No performance metric inspected",

            "PASS":
                True,
        },

        {
            "Check":
                "No calibration fitted",

            "PASS":
                True,
        },

        {
            "Check":
                "No threshold selected",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 15 HARD QC FAILED:\n"
        f"{failed}"
    )


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP15_FINAL_hard_qc.csv"
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


# ============================================================
# 23. RESTRICTED TARGET HASHES
# ============================================================

stage1_target_sha = sha256_file(
    STAGE1_RESTRICTED_FILE
)


stage2_target_sha = sha256_file(
    STAGE2_RESTRICTED_FILE
)


# ============================================================
# 24. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 15",

    "status":
        "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "random_seed":
        SEED,

    "primary_master_sha256":
        observed_primary_sha,

    "outer_split_sha256":
        observed_outer_sha,

    "inner_split_sha256":
        observed_inner_sha,

    "stage1_n":
        5063,

    "stage1_positive_n":
        stage1_positive_n,

    "stage1_negative_n":
        stage1_negative_n,

    "stage2_n":
        int(
            len(
                stage2_master
            )
        ),

    "stage2_positive_n":
        stage2_positive_n,

    "stage2_negative_n":
        stage2_negative_n,

    "stage2_population":
        "ANC users only; no No-ANC observations included",

    "primary_fit_weight":
        "normalized DHS survey weight only",

    "fit_weight_normalization":
        "mean 1 within stage-specific training partition",

    "class_balancing":
        False,

    "oversampling":
        False,

    "undersampling":
        False,

    "smote":
        False,

    "inverse_frequency_weighting":
        False,

    "scale_pos_weight_tuning":
        False,

    "model_fitting_performed":
        False,

    "performance_evaluated":
        False,

    "calibration_fitted":
        False,

    "threshold_selected":
        False,

    "restricted_stage1_target_file":
        str(
            STAGE1_RESTRICTED_FILE
        ),

    "restricted_stage1_target_sha256":
        stage1_target_sha,

    "restricted_stage2_target_file":
        str(
            STAGE2_RESTRICTED_FILE
        ),

    "restricted_stage2_target_sha256":
        stage2_target_sha,

    "target_summary_sha256":
        sha256_file(
            TARGET_SUMMARY_FILE
        ),

    "training_weight_audit_sha256":
        sha256_file(
            TRAIN_WEIGHT_AUDIT_FILE
        ),

    "hierarchical_protocol_sha256":
        sha256_file(
            HIERARCHICAL_PROTOCOL_FILE
        ),

    "weight_protocol_sha256":
        sha256_file(
            WEIGHT_PROTOCOL_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),

    "methods_sha256":
        sha256_file(
            METHODS_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_hierarchical_weight_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 25. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 15."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 15."
    )


if sha256_file(
    INNER_SPLIT_FILE
) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 15."
    )


# ============================================================
# 26. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 15 — HIERARCHICAL TARGET + SURVEY-WEIGHT PROTOCOL FREEZE"
)

print(
    "=" * 78
)


print(
    "\nTARGET SUMMARY:"
)

display(
    target_summary
)


print(
    "\nMODEL IMBALANCE-CONTROL POLICY:"
)

display(
    imbalance_control_policy
)


print(
    "\nTRAINING-WEIGHT AUDIT SUMMARY:"
)

display(
    training_weight_audit[
        [
            "Scope",
            "Stage",
            "Eligible_Training_N",
            "Positive_N",
            "Negative_N",
            "Survey_Weighted_Positive_Prevalence",
            "Normalized_Weight_Mean",
            "Kish_ESS",
        ]
    ]
    .groupby(
        [
            "Scope",
            "Stage",
        ]
    )
    .agg(
        [
            "min",
            "max",
        ]
    )
)


print(
    "\nHELD-OUT COMPOSITION AUDIT SUMMARY:"
)

display(
    validation_composition_audit[
        [
            "Scope",
            "Stage",
            "Eligible_Validation_N",
            "Positive_N",
            "Negative_N",
            "Survey_Weighted_Positive_Prevalence",
        ]
    ]
    .groupby(
        [
            "Scope",
            "Stage",
        ]
    )
    .agg(
        [
            "min",
            "max",
        ]
    )
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)

print(
    methods_text
)


print(
    "\nRESTRICTED TARGET FILES:"
)

print(
    STAGE1_RESTRICTED_FILE
)

print(
    f"SHA256: {stage1_target_sha}"
)

print()

print(
    STAGE2_RESTRICTED_FILE
)

print(
    f"SHA256: {stage2_target_sha}"
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {observed_primary_sha}"
)

print(
    f"Outer split    : {observed_outer_sha}"
)

print(
    f"Inner split    : {observed_inner_sha}"
)


print(
    "\nIMPORTANT:"
)

print(
    "- Stage 1 uses the full analytic cohort: No ANC vs Any ANC."
)

print(
    "- Stage 2 contains ANC users only: >=8 vs 1-7 reported ANC visits."
)

print(
    "- No No-ANC observation is included in Stage 2."
)

print(
    "- Training uses normalized DHS survey weights only."
)

print(
    "- Relative survey weights are preserved exactly."
)

print(
    "- Training weights are normalized separately within each stage and fold."
)

print(
    "- Original survey weights remain reserved for held-out evaluation."
)

print(
    "- No SMOTE, oversampling, undersampling, or class balancing is used."
)

print(
    "- scale_pos_weight/class_weight balancing will not be tuned."
)

print(
    "- Class asymmetry will later be handled through nested decision thresholds, not prevalence distortion."
)

print(
    "- No model has been fitted and no performance has been inspected."
)

print(
    "- Restricted target CSVs are local artifacts and should not be uploaded to public GitHub."
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 15 PASS"
)

print(
    "=" * 78
)


STEP 15 — HIERARCHICAL TARGET + SURVEY-WEIGHT PROTOCOL FREEZE

TARGET SUMMARY:


,Stage,Target,Eligible_N,Positive_Label,Positive_N,Negative_Label,Negative_N,Unweighted_Positive_Prevalence,Survey_Weighted_Positive_Prevalence,Survey_Weight_Kish_ESS
0,Stage 1,No ANC vs Any ANC,5063,No ANC,403,Any ANC,4660,0.079597,0.079434,4032.666282
1,Stage 2,>=8 vs 1-7 ANC visits among ANC users,4660,>=8 reported ANC visits,253,1-7 ANC visits,4407,0.054292,0.054901,3703.300944



MODEL IMBALANCE-CONTROL POLICY:


,Model,Control,Primary_Value
0,ElasticNet logistic regression,class_weight,None
1,RBF SVM,class_weight,None
2,Random Forest,class_weight,None
3,ExtraTrees,class_weight,None
4,AdaBoost,resampling / class multiplier,None
5,GradientBoosting,resampling / class multiplier,None
6,HistGradientBoosting,class weighting beyond sample_weight,None
7,XGBoost,scale_pos_weight,1.0
8,LightGBM,class_weight / is_unbalance / scale_pos_weight,None / False / 1.0
9,CatBoost,class_weights / auto_class_weights,None / None



TRAINING-WEIGHT AUDIT SUMMARY:


Eligible_Training_N       Positive_N      Negative_N        \
                                  min   max        min  max        min   max   
Scope       Stage                                                              
INNER_TRAIN 1                    3017  3067        232  250       2771  2823   
            2                    2771  2823        146  157       2623  2672   
OUTER_TRAIN 1                    4032  4058        318  327       3712  3737   
            2                    3712  3737        200  206       3512  3535   

                  Survey_Weighted_Positive_Prevalence            \
                                                  min       max   
Scope       Stage                                                 
INNER_TRAIN 1                                0.076535  0.082403   
            2                                0.050674  0.058627   
OUTER_TRAIN 1                                0.078460  0.080698   
            2                                0.052461  0.056190   

                  Normalized_Weight_Mean          Kish_ESS               
                                     min  max          min          max  
Scope       Stage                                                        
INNER_TRAIN 1                        1.0  1.0  2369.801173  2477.454646  
            2                        1.0  1.0  2178.856909  2270.619975  
OUTER_TRAIN 1                        1.0  1.0  3204.353742  3275.118655  
            2                        1.0  1.0  2945.001456  2999.665964


HELD-OUT COMPOSITION AUDIT SUMMARY:


Eligible_Validation_N       Positive_N     Negative_N       \
                                    min   max        min max        min  max   
Scope       Stage                                                              
INNER_VALID 1                       990  1040         73  90        908  960   
            2                       908   960         47  55        859  908   
OUTER_TEST  1                      1005  1031         76  85        923  948   
            2                       923   948         47  53        872  895   

                  Survey_Weighted_Positive_Prevalence            
                                                  min       max  
Scope       Stage                                                
INNER_VALID 1                                0.070166  0.087611  
            2                                0.048789  0.062473  
OUTER_TEST  1                                0.074209  0.083389  
            2                                0.049922  0.064689


FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Stage 1 N=5063,True
4,Stage 1 positives=403,True
5,Stage 1 negatives=4660,True
6,Stage 2 eligible N=4660,True
7,Stage 2 positives=253,True
8,Stage 2 negatives=4407,True
9,Stage 2 contains no No-ANC observations,True



MANUSCRIPT METHODS PREVIEW:
Hierarchical outcome modelling and survey weighting

The three-category antenatal-care outcome was modelled using a
prespecified two-stage hierarchical classification framework. Stage 1
distinguished women reporting no antenatal-care visits from those
reporting any antenatal care. Stage 2 was developed exclusively among
women reporting at least one antenatal-care visit and distinguished
women reporting eight or more visits from those reporting one to seven
visits. For an individual with Stage-1 probability q0=P(no ANC|X) and
Stage-2 conditional probability q2=P(>=8 visits|any ANC,X), the final
three-category probabilities were defined as p0=q0,
p1=(1-q0)(1-q2), and p2=(1-q0)q2.

DHS sampling weights were incorporated during model fitting to retain
the relative sampling-weight structure of the survey. Within each
stage-specific training partition, sampling weights were rescaled by a
single positive constant to have mean one. Thus, relative sampling
weights w

In [22]:
# ============================================================
# STEP 15A — SURVEY-WEIGHT SENSITIVITY PROTOCOL FREEZE
# ============================================================
#
# Purpose
# -------
# Prespecify a secondary sensitivity analysis assessing whether
# model development is materially affected by using DHS sampling
# weights during fitting.
#
# PRIMARY ANALYSIS
# ----------------
# Model fitting:
#   normalized DHS survey weights
#
# Inner-validation model selection:
#   survey-weighted validation metric
#
# Outer-test evaluation:
#   original DHS survey weights
#
# SENSITIVITY ANALYSIS
# --------------------
# Model fitting:
#   UNWEIGHTED (sample_weight=None)
#
# Hyperparameter tuning:
#   repeated independently inside the SAME frozen inner folds
#   using the SAME search space and SAME selection metric.
#
# Inner-validation model selection:
#   survey-weighted validation metric
#
# Outer-test evaluation:
#   original DHS survey weights
#
# Therefore, the sensitivity changes ONLY whether sampling
# weights enter model fitting/model configuration.
#
# It does NOT change:
#   - cohort
#   - predictors
#   - outcomes
#   - hierarchical structure
#   - preprocessing policy
#   - CV assignments
#   - hyperparameter search space
#   - selection metric
#   - calibration framework
#   - threshold framework
#   - evaluation weights
#
# No model fitting or performance evaluation occurs here.
# ============================================================


from pathlib import Path
import datetime as dt
import hashlib
import json

import pandas as pd

from IPython.display import display


# ============================================================
# 1. INPUT LOCK
# ============================================================

if "DIRS" not in globals():

    raise RuntimeError(
        "DIRS is unavailable."
    )


PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


STEP15_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_hierarchical_weight_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(block)

    return digest.hexdigest()


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP15_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen file missing:\n{path}"
        )


if (
    sha256_file(PRIMARY_FILE)
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    sha256_file(OUTER_SPLIT_FILE)
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if (
    sha256_file(INNER_SPLIT_FILE)
    !=
    EXPECTED_INNER_SHA256
):

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


with open(
    STEP15_MANIFEST_FILE,
    "r",
    encoding="utf-8",
) as handle:

    step15_manifest = json.load(
        handle
    )


if (
    step15_manifest.get("status")
    !=
    "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 15 is not frozen."
    )


for key, expected in [
    (
        "primary_master_sha256",
        EXPECTED_PRIMARY_SHA256,
    ),
    (
        "outer_split_sha256",
        EXPECTED_OUTER_SHA256,
    ),
    (
        "inner_split_sha256",
        EXPECTED_INNER_SHA256,
    ),
]:

    if step15_manifest.get(key) != expected:

        raise RuntimeError(
            f"Step-15 manifest mismatch: {key}"
        )


# ============================================================
# 2. ANALYSIS COMPARISON POLICY
# ============================================================

comparison_policy = pd.DataFrame(
    [
        {
            "Component":
                "Training sample",

            "Primary":
                "Frozen stage-specific training population",

            "Sensitivity":
                "Identical frozen stage-specific training population",
        },

        {
            "Component":
                "Predictor set",

            "Primary":
                "Same frozen 15 predictors",

            "Sensitivity":
                "Same frozen 15 predictors",
        },

        {
            "Component":
                "Preprocessing",

            "Primary":
                "Frozen Step-14 model-specific preprocessing",

            "Sensitivity":
                "Identical Step-14 preprocessing",
        },

        {
            "Component":
                "Outer / inner CV",

            "Primary":
                "Frozen Step-13 PSU-exclusive nested CV",

            "Sensitivity":
                "Identical frozen CV assignments",
        },

        {
            "Component":
                "Model-fitting sample weight",

            "Primary":
                "Normalized DHS SURVEY_WEIGHT",

            "Sensitivity":
                "None; unit weight for every training observation",
        },

        {
            "Component":
                "Automatic class balancing",

            "Primary":
                "None",

            "Sensitivity":
                "None",
        },

        {
            "Component":
                "Hyperparameter search space",

            "Primary":
                "Frozen Step-16 search space",

            "Sensitivity":
                "Exactly the same Step-16 search space",
        },

        {
            "Component":
                "Hyperparameter optimization",

            "Primary":
                "Independent within each outer-fold training set",

            "Sensitivity":
                (
                    "Independently repeated without fitting weights; "
                    "weighted-primary hyperparameters are not forced"
                ),
        },

        {
            "Component":
                "Inner-validation scoring",

            "Primary":
                "Original DHS survey-weighted metric",

            "Sensitivity":
                "Same original DHS survey-weighted metric",
        },

        {
            "Component":
                "Calibration",

            "Primary":
                "Frozen nested calibration procedure",

            "Sensitivity":
                (
                    "Re-estimated within the sensitivity pipeline "
                    "using the identical frozen calibration procedure"
                ),
        },

        {
            "Component":
                "Threshold selection",

            "Primary":
                "Frozen nested threshold procedure",

            "Sensitivity":
                (
                    "Re-estimated within the sensitivity pipeline "
                    "using the identical frozen threshold procedure"
                ),
        },

        {
            "Component":
                "Outer-test evaluation",

            "Primary":
                "Original DHS SURVEY_WEIGHT",

            "Sensitivity":
                "Original DHS SURVEY_WEIGHT",
        },

        {
            "Component":
                "Primary inference",

            "Primary":
                "Primary results",

            "Sensitivity":
                (
                    "Robustness analysis only; cannot replace "
                    "the primary result based on better performance"
                ),
        },
    ]
)


# ============================================================
# 3. FORMAL SENSITIVITY PROTOCOL
# ============================================================

sensitivity_protocol = {

    "name":
        "Unweighted model-fitting sensitivity",

    "status":
        "prespecified_before_model_fitting",

    "purpose":
        (
            "Assess sensitivity of model development to use of "
            "DHS sampling weights during fitting while retaining "
            "population-oriented weighted model selection and evaluation."
        ),

    "primary_training_weight":
        (
            "SURVEY_WEIGHT normalized to mean 1 within each "
            "stage-specific training partition"
        ),

    "sensitivity_training_weight":
        None,

    "sensitivity_training_unit_weight":
        1.0,

    "independent_retuning":
        True,

    "reuse_primary_selected_hyperparameters":
        False,

    "same_hyperparameter_search_space":
        True,

    "same_inner_cv":
        True,

    "same_outer_cv":
        True,

    "same_preprocessing":
        True,

    "same_predictors":
        True,

    "same_hierarchical_targets":
        True,

    "same_model_families":
        True,

    "inner_selection_metric":
        (
            "Identical survey-weighted metric to be frozen in Step 16"
        ),

    "inner_validation_weights":
        "original SURVEY_WEIGHT",

    "outer_test_weights":
        "original SURVEY_WEIGHT",

    "automatic_class_balancing":
        False,

    "oversampling":
        False,

    "undersampling":
        False,

    "smote":
        False,

    "inverse_frequency_class_weight":
        False,

    "calibration_policy":
        (
            "Use the same later-frozen nested calibration framework, "
            "re-estimated from sensitivity-pipeline predictions only."
        ),

    "threshold_policy":
        (
            "Use the same later-frozen nested threshold framework, "
            "re-estimated from sensitivity-pipeline predictions only."
        ),

    "reporting":
        (
            "Report as a sensitivity analysis. "
            "Do not replace the primary survey-weighted analysis "
            "because the sensitivity performs better."
        ),

    "performance_guided_protocol_change":
        False,

    "model_fitting_performed_in_step15a":
        False,

    "performance_evaluated_in_step15a":
        False,
}


# ============================================================
# 4. REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "MACNELL_2023",

            "Reference":
                (
                    "MacNell N, Feinstein L, Wilkerson J, "
                    "et al. Implementing machine learning "
                    "methods with complex survey data: "
                    "Lessons learned on the impacts of "
                    "accounting sampling weights in "
                    "gradient boosting. "
                    "PLoS One. 2023;18(1):e0280387."
                ),

            "DOI":
                "10.1371/journal.pone.0280387",

            "Use":
                (
                    "Motivates comparison of weighted and "
                    "unweighted model configuration while "
                    "retaining population-weighted evaluation."
                ),
        },
    ]
)


# ============================================================
# 5. MANUSCRIPT-READY SENSITIVITY TEXT
# ============================================================

methods_text = """
Survey-weight sensitivity analysis

A prespecified sensitivity analysis will evaluate the influence of
sampling weights during model development. The primary analysis will
incorporate normalized DHS sampling weights during model fitting,
whereas the sensitivity analysis will fit the same candidate algorithms
without sampling weights. All other design features will remain fixed,
including the analytic cohort, predictor set, hierarchical outcome
definitions, model-specific preprocessing, nested PSU-exclusive
cross-validation partitions, hyperparameter search spaces, and model
selection criteria.

Because the hyperparameter configuration appropriate for an unweighted
fit may differ from that selected for a weighted fit, hyperparameter
optimization will be repeated independently for the unweighted
sensitivity analysis within the same frozen inner cross-validation
partitions. Inner-validation model selection and outer-test performance
assessment will nevertheless use the original DHS sampling weights so
that both analyses are evaluated against the same target-population
objective.

Any later calibration and decision-threshold procedures will use the
same prespecified nested methodology in the primary and sensitivity
pipelines but will be estimated separately within each pipeline.
Sensitivity-analysis results will be used only to assess robustness and
will not replace the primary survey-weighted analysis on the basis of
superior observed performance.
""".strip()


# ============================================================
# 6. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Step 15 frozen",

            "PASS":
                (
                    step15_manifest.get("status")
                    ==
                    "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Sensitivity changes fitting weight only",

            "PASS":
                (
                    sensitivity_protocol[
                        "same_hyperparameter_search_space"
                    ]
                    and
                    sensitivity_protocol[
                        "same_inner_cv"
                    ]
                    and
                    sensitivity_protocol[
                        "same_outer_cv"
                    ]
                    and
                    sensitivity_protocol[
                        "same_preprocessing"
                    ]
                    and
                    sensitivity_protocol[
                        "same_predictors"
                    ]
                    and
                    sensitivity_protocol[
                        "same_hierarchical_targets"
                    ]
                ),
        },

        {
            "Check":
                "Unweighted sensitivity independently retuned",

            "PASS":
                (
                    sensitivity_protocol[
                        "independent_retuning"
                    ]
                    and
                    not sensitivity_protocol[
                        "reuse_primary_selected_hyperparameters"
                    ]
                ),
        },

        {
            "Check":
                "Same weighted validation objective retained",

            "PASS":
                (
                    sensitivity_protocol[
                        "inner_validation_weights"
                    ]
                    ==
                    "original SURVEY_WEIGHT"
                ),
        },

        {
            "Check":
                "Same weighted outer-test evaluation retained",

            "PASS":
                (
                    sensitivity_protocol[
                        "outer_test_weights"
                    ]
                    ==
                    "original SURVEY_WEIGHT"
                ),
        },

        {
            "Check":
                "No class balancing introduced",

            "PASS":
                (
                    not sensitivity_protocol[
                        "automatic_class_balancing"
                    ]
                    and
                    not sensitivity_protocol[
                        "oversampling"
                    ]
                    and
                    not sensitivity_protocol[
                        "undersampling"
                    ]
                    and
                    not sensitivity_protocol[
                        "smote"
                    ]
                    and
                    not sensitivity_protocol[
                        "inverse_frequency_class_weight"
                    ]
                ),
        },

        {
            "Check":
                "Sensitivity cannot replace primary based on performance",

            "PASS":
                (
                    "Do not replace"
                    in
                    sensitivity_protocol[
                        "reporting"
                    ]
                ),
        },

        {
            "Check":
                "No model fitted",

            "PASS":
                True,
        },

        {
            "Check":
                "No performance inspected",

            "PASS":
                True,
        },
    ]
)


if not hard_qc["PASS"].all():

    raise RuntimeError(
        "STEP 15A HARD QC FAILED:\n"
        f"{hard_qc.loc[~hard_qc['PASS']]}"
    )


# ============================================================
# 7. SAVE
# ============================================================

PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP15A_FINAL_survey_weight_sensitivity_protocol.json"
)


COMPARISON_FILE = (
    DIRS["protocol"]
    / "STEP15A_FINAL_primary_vs_unweighted_sensitivity.csv"
)


REFERENCE_FILE = (
    DIRS["protocol"]
    / "STEP15A_FINAL_reference_registry.csv"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP15A_FINAL_manuscript_sensitivity_methods.txt"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP15A_FINAL_hard_qc.csv"
)


with open(
    PROTOCOL_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        sensitivity_protocol,
        handle,
        indent=2,
        ensure_ascii=False,
    )


comparison_policy.to_csv(
    COMPARISON_FILE,
    index=False,
)


reference_registry.to_csv(
    REFERENCE_FILE,
    index=False,
)


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


# ============================================================
# 8. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 15A",

    "status":
        "SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "primary_master_sha256":
        sha256_file(PRIMARY_FILE),

    "outer_split_sha256":
        sha256_file(OUTER_SPLIT_FILE),

    "inner_split_sha256":
        sha256_file(INNER_SPLIT_FILE),

    "primary_training":
        "normalized survey-weighted fitting",

    "sensitivity_training":
        "unweighted fitting",

    "sensitivity_independent_retuning":
        True,

    "same_search_space":
        True,

    "same_weighted_inner_selection_objective":
        True,

    "same_weighted_outer_evaluation":
        True,

    "model_fitting_performed":
        False,

    "performance_evaluated":
        False,

    "protocol_sha256":
        sha256_file(PROTOCOL_FILE),

    "comparison_sha256":
        sha256_file(COMPARISON_FILE),

    "methods_sha256":
        sha256_file(METHODS_FILE),

    "hard_qc_sha256":
        sha256_file(HARD_QC_FILE),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP15A_FINAL_survey_weight_sensitivity_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 9. FINAL IMMUTABILITY CHECK
# ============================================================

if (
    sha256_file(PRIMARY_FILE)
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary master changed during Step 15A."
    )


if (
    sha256_file(OUTER_SPLIT_FILE)
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Outer split changed during Step 15A."
    )


if (
    sha256_file(INNER_SPLIT_FILE)
    !=
    EXPECTED_INNER_SHA256
):

    raise RuntimeError(
        "Inner split changed during Step 15A."
    )


# ============================================================
# 10. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 15A — SURVEY-WEIGHT SENSITIVITY PROTOCOL FREEZE"
)

print(
    "=" * 78
)


print(
    "\nPRIMARY VS SENSITIVITY:"
)

display(
    comparison_policy
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)

print(
    methods_text
)


print(
    "\nIMPORTANT:"
)

print(
    "- Primary analysis remains survey-weighted during fitting."
)

print(
    "- Sensitivity analysis removes survey weights from model fitting only."
)

print(
    "- The sensitivity pipeline is independently retuned inside the same frozen inner folds."
)

print(
    "- The hyperparameter search space will remain identical."
)

print(
    "- Inner model selection remains survey-weighted."
)

print(
    "- Outer-test evaluation remains survey-weighted."
)

print(
    "- No class balancing is introduced."
)

print(
    "- Sensitivity results cannot replace the primary analysis because they look better."
)

print(
    "- No model has been fitted and no performance has been inspected."
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)

print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)

print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 15A PASS"
)

print(
    "=" * 78
)


STEP 15A — SURVEY-WEIGHT SENSITIVITY PROTOCOL FREEZE

PRIMARY VS SENSITIVITY:


,Component,Primary,Sensitivity
0,Training sample,Frozen stage-specific training population,Identical frozen stage-specific training popul...
1,Predictor set,Same frozen 15 predictors,Same frozen 15 predictors
2,Preprocessing,Frozen Step-14 model-specific preprocessing,Identical Step-14 preprocessing
3,Outer / inner CV,Frozen Step-13 PSU-exclusive nested CV,Identical frozen CV assignments
4,Model-fitting sample weight,Normalized DHS SURVEY_WEIGHT,None; unit weight for every training observation
5,Automatic class balancing,None,None
6,Hyperparameter search space,Frozen Step-16 search space,Exactly the same Step-16 search space
7,Hyperparameter optimization,Independent within each outer-fold training set,Independently repeated without fitting weights...
8,Inner-validation scoring,Original DHS survey-weighted metric,Same original DHS survey-weighted metric
9,Calibration,Frozen nested calibration procedure,Re-estimated within the sensitivity pipeline u...



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Step 15 frozen,True
4,Sensitivity changes fitting weight only,True
5,Unweighted sensitivity independently retuned,True
6,Same weighted validation objective retained,True
7,Same weighted outer-test evaluation retained,True
8,No class balancing introduced,True
9,Sensitivity cannot replace primary based on pe...,True



MANUSCRIPT METHODS PREVIEW:
Survey-weight sensitivity analysis

A prespecified sensitivity analysis will evaluate the influence of
sampling weights during model development. The primary analysis will
incorporate normalized DHS sampling weights during model fitting,
whereas the sensitivity analysis will fit the same candidate algorithms
without sampling weights. All other design features will remain fixed,
including the analytic cohort, predictor set, hierarchical outcome
definitions, model-specific preprocessing, nested PSU-exclusive
cross-validation partitions, hyperparameter search spaces, and model
selection criteria.

Because the hyperparameter configuration appropriate for an unweighted
fit may differ from that selected for a weighted fit, hyperparameter
optimization will be repeated independently for the unweighted
sensitivity analysis within the same frozen inner cross-validation
partitions. Inner-validation model selection and outer-test performance
assessment will nevertheles

In [25]:
# ============================================================
# STEP 16 — FINAL JOINT MODEL-FAMILY + HYPERPARAMETER
#           INNER-CV SELECTION PROTOCOL FREEZE
# ============================================================
#
# PURPOSE
# -------
# Freeze the complete model-selection procedure BEFORE fitting
# any candidate model.
#
# KEY PRINCIPLE
# -------------
# Model family is NOT fixed in advance.
#
# Within each:
#   analysis pipeline x hierarchical stage x outer fold
#
# ALL prespecified candidate configurations from ALL ten
# model families compete JOINTLY inside the frozen inner CV.
#
# Therefore the selected object is:
#
#   MODEL FAMILY + HYPERPARAMETERS + PRESPECIFIED PREPROCESSING
#
# Outer-test data are used only after this joint selection.
#
# DEFAULT-MODEL SCREENING
# -----------------------
# Default performance is NOT used to eliminate model families.
#
# Reason:
#   A model family with a mediocre library-default configuration
#   may become competitive after legitimate tuning. Screening
#   families on default performance would add an unnecessary
#   data-driven selection layer.
#
# Computational efficiency is achieved through a compact,
# prespecified deterministic search space rather than through
# performance-based family elimination.
#
# SELECTION METRIC
# ----------------
# For each candidate:
#   - obtain predictions on all four frozen inner-validation folds
#   - concatenate into one pooled inner-OOF vector
#   - calculate survey-weighted Average Precision
#
# Selection:
#   1. Highest pooled survey-weighted Average Precision
#   2. If tied within tolerance: highest weighted ROC-AUC
#   3. If still tied: lowest deterministic tie rank
#
# RBF-SVM:
#   decision_function
#
# Other models:
#   positive-class predict_proba
#
# Stage 1 and Stage 2 are selected independently.
# They are NOT required to use the same model family.
#
# Primary and unweighted sensitivity pipelines use the SAME
# candidate registry but are independently selected/tuned.
#
# ZERO model fitting occurs in this step.
# ============================================================


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import datetime as dt
import hashlib
import importlib.metadata as importlib_metadata
import json

import pandas as pd
import sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from IPython.display import display


# ============================================================
# 1. FROZEN INPUTS
# ============================================================

SEED = 2026

if "DIRS" not in globals():

    raise RuntimeError(
        "DIRS is unavailable. "
        "Run the accepted project setup first."
    )


PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


STEP14_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_preprocessing_manifest.json"
)


STEP15_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_hierarchical_weight_manifest.json"
)


STEP15A_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP15A_FINAL_survey_weight_sensitivity_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP14_MANIFEST_FILE,
    STEP15_MANIFEST_FILE,
    STEP15A_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing:\n{path}"
        )


if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


step14_manifest = read_json(
    STEP14_MANIFEST_FILE
)

step15_manifest = read_json(
    STEP15_MANIFEST_FILE
)

step15a_manifest = read_json(
    STEP15A_MANIFEST_FILE
)


if (
    step14_manifest.get("status")
    !=
    "PREPROCESSING_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 14 is not frozen."
    )


if (
    step15_manifest.get("status")
    !=
    "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 15 is not frozen."
    )


if (
    step15a_manifest.get("status")
    !=
    "SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 15A is not frozen."
    )


# ============================================================
# 2. JOINT MODEL-SELECTION RULE
# ============================================================

PRIMARY_SELECTION_METRIC = (
    "survey_weighted_average_precision"
)


SECONDARY_SELECTION_METRIC = (
    "survey_weighted_roc_auc"
)


METRIC_TIE_TOLERANCE = 1e-12


selection_protocol = {

    "selection_object":
        (
            "model family + hyperparameter configuration + "
            "prespecified Step-14 preprocessing"
        ),

    "joint_model_family_selection":
        True,

    "model_family_fixed_before_tuning":
        False,

    "default_performance_family_screening":
        False,

    "default_performance_used_to_remove_models":
        False,

    "reason_no_default_screening":
        (
            "Library-default performance may not represent the "
            "performance achievable after legitimate tuning and "
            "could prematurely eliminate a competitive model family."
        ),

    "computational_efficiency_strategy":
        (
            "compact deterministic prespecified candidate grids; "
            "no performance-driven family elimination"
        ),

    "selection_scope":
        (
            "independently within each analysis pipeline, "
            "hierarchical stage, and outer-training fold"
        ),

    "stage1_and_stage2_selected_independently":
        True,

    "same_model_family_required_across_stages":
        False,

    "reason_stage_specific_selection":
        (
            "Stage 1 and Stage 2 are distinct binary prediction "
            "tasks and no scientific requirement justifies forcing "
            "the same algorithm for both."
        ),

    "outer_folds":
        5,

    "inner_folds":
        4,

    "candidate_evaluation":
        (
            "pooled cross-fitted predictions from all four "
            "frozen inner-validation folds"
        ),

    "metric_aggregation":
        (
            "single metric calculated from pooled inner-OOF "
            "predictions; not arithmetic mean of fold metrics"
        ),

    "primary_selection_metric":
        PRIMARY_SELECTION_METRIC,

    "secondary_selection_metric":
        SECONDARY_SELECTION_METRIC,

    "metric_weights":
        (
            "original DHS SURVEY_WEIGHT on inner-validation "
            "observations"
        ),

    "average_precision_definition":
        (
            "sklearn.metrics.average_precision_score with "
            "sample_weight; non-interpolated AP"
        ),

    "roc_auc_definition":
        (
            "sklearn.metrics.roc_auc_score with sample_weight"
        ),

    "svm_selection_score":
        "decision_function",

    "other_model_selection_score":
        "positive-class predict_proba",

    "calibration_used_for_candidate_selection":
        False,

    "threshold_used_for_candidate_selection":
        False,

    "accuracy_used_for_candidate_selection":
        False,

    "f1_used_for_candidate_selection":
        False,

    "outer_test_used_for_candidate_selection":
        False,

    "outer_test_used_for_hyperparameter_tuning":
        False,

    "class_balancing_tuned":
        False,

    "random_search":
        False,

    "bayesian_optimization":
        False,

    "optuna":
        False,

    "early_stopping_validation_split":
        False,

    "performance_guided_grid_expansion":
        False,

    "boundary_winner_rule":
        (
            "Do not expand the frozen primary search space merely "
            "because the selected configuration lies on a grid boundary."
        ),

    "candidate_failure_rule":
        (
            "A candidate is ineligible for the current selection "
            "context if any required inner-fold fit or prediction fails."
        ),

    "all_candidates_failure_rule":
        (
            "Stop and diagnose the technical problem. "
            "Do not silently modify the search space."
        ),

    "selection_rule": [
        (
            "1. Highest pooled survey-weighted "
            "Average Precision"
        ),
        (
            "2. If AP differs by <=1e-12, highest pooled "
            "survey-weighted ROC-AUC"
        ),
        (
            "3. If still tied, lowest deterministic tie rank"
        ),
    ],

    "primary_and_sensitivity_same_registry":
        True,

    "sensitivity_independently_retuned":
        True,

    "final_outer_evaluation_target":
        (
            "performance of the complete joint model-family + "
            "hyperparameter selection procedure"
        ),

    "model_fitting_performed_in_step16":
        False,

    "performance_evaluated_in_step16":
        False,
}


# ============================================================
# 3. FIXED MODEL PARAMETERS
# ============================================================

FIXED_PARAMS = {

    "ElasticNet logistic regression": {

        "solver":
            "saga",

        "class_weight":
            None,

        "fit_intercept":
            True,

        "max_iter":
            10000,

        "tol":
            1e-4,

        "random_state":
            SEED,
    },


    "RBF SVM": {

        "kernel":
            "rbf",

        "probability":
            False,

        "class_weight":
            None,

        "shrinking":
            True,

        "tol":
            1e-3,

        "cache_size":
            2000,
    },


    "Random Forest": {

        "n_estimators":
            500,

        "criterion":
            "gini",

        "bootstrap":
            True,

        "class_weight":
            None,

        "random_state":
            SEED,

        "n_jobs":
            -1,
    },


    "ExtraTrees": {

        "n_estimators":
            500,

        "criterion":
            "gini",

        "bootstrap":
            False,

        "class_weight":
            None,

        "random_state":
            SEED,

        "n_jobs":
            -1,
    },


    "AdaBoost": {

        "random_state":
            SEED,
    },


    "GradientBoosting": {

        "loss":
            "log_loss",

        "min_samples_leaf":
            10,

        "max_features":
            None,

        "n_iter_no_change":
            None,

        "random_state":
            SEED,
    },


    "HistGradientBoosting": {

        "loss":
            "log_loss",

        "min_samples_leaf":
            20,

        "max_features":
            1.0,

        "categorical_features":
            "from_dtype",

        "early_stopping":
            False,

        "class_weight":
            None,

        "random_state":
            SEED,
    },


    "XGBoost": {

        "objective":
            "binary:logistic",

        "tree_method":
            "hist",

        "subsample":
            0.8,

        "colsample_bytree":
            0.8,

        "reg_alpha":
            0.0,

        "reg_lambda":
            1.0,

        "scale_pos_weight":
            1.0,

        "eval_metric":
            "logloss",

        "random_state":
            SEED,

        "n_jobs":
            -1,

        "verbosity":
            0,
    },


    "LightGBM": {

        "objective":
            "binary",

        "subsample":
            0.8,

        "subsample_freq":
            1,

        "colsample_bytree":
            0.8,

        "reg_alpha":
            0.0,

        "reg_lambda":
            1.0,

        "class_weight":
            None,

        "random_state":
            SEED,

        "n_jobs":
            -1,

        "verbosity":
            -1,

        "deterministic":
            True,

        "force_col_wise":
            True,
    },


    "CatBoost": {

        "loss_function":
            "Logloss",

        "eval_metric":
            "Logloss",

        "bootstrap_type":
            "Bayesian",

        "bagging_temperature":
            1.0,

        "random_strength":
            1.0,

        "class_weights":
            None,

        "auto_class_weights":
            None,

        "random_seed":
            SEED,

        "thread_count":
            -1,

        "verbose":
            False,

        "allow_writing_files":
            False,

        "use_best_model":
            False,
    },
}


# ============================================================
# 4. COMPACT PRESPECIFIED SEARCH SPACES
# ============================================================
#
# Exactly 12 configurations per model family.
#
# Search spaces are deliberately compact to limit computational
# cost while still spanning clinically reasonable model
# complexity / regularization ranges.
#
# No candidate is generated from observed performance.
# ============================================================


# ------------------------------------------------------------
# ElasticNet logistic regression
# ------------------------------------------------------------

elastic_configs = [

    {
        "C": C,
        "l1_ratio": l1_ratio,
    }

    for C in [
        0.01,
        0.1,
        1.0,
        10.0,
    ]

    for l1_ratio in [
        0.1,
        0.5,
        0.9,
    ]
]


# ------------------------------------------------------------
# RBF SVM
# ------------------------------------------------------------

svm_configs = [

    {
        "C": C,
        "gamma": gamma,
    }

    for C in [
        0.1,
        1.0,
        10.0,
    ]

    for gamma in [
        0.01,
        "scale",
        0.1,
        1.0,
    ]
]


# ------------------------------------------------------------
# Random Forest
# ------------------------------------------------------------

rf_configs = [

    {
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
    }

    for max_depth in [
        8,
        16,
        None,
    ]

    for min_samples_leaf in [
        5,
        1,
    ]

    for max_features in [
        "sqrt",
        0.5,
    ]
]


# ------------------------------------------------------------
# ExtraTrees
# ------------------------------------------------------------

et_configs = [

    {
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
    }

    for max_depth in [
        8,
        16,
        None,
    ]

    for min_samples_leaf in [
        5,
        1,
    ]

    for max_features in [
        "sqrt",
        0.5,
    ]
]


# ------------------------------------------------------------
# AdaBoost
# ------------------------------------------------------------

ada_configs = [

    {
        "estimator_max_depth":
            estimator_max_depth,

        "n_estimators":
            n_estimators,

        "learning_rate":
            learning_rate,
    }

    for estimator_max_depth in [
        1,
        2,
    ]

    for n_estimators in [
        100,
        400,
    ]

    for learning_rate in [
        0.05,
        0.2,
        1.0,
    ]
]


# ------------------------------------------------------------
# GradientBoosting
# ------------------------------------------------------------

gb_learning_pairs = [

    (0.10, 100),
    (0.05, 300),
    (0.03, 600),
]


gb_configs = [

    {
        "learning_rate":
            learning_rate,

        "n_estimators":
            n_estimators,

        "max_depth":
            max_depth,

        "subsample":
            subsample,
    }

    for (
        learning_rate,
        n_estimators,
    ) in gb_learning_pairs

    for max_depth in [
        1,
        2,
    ]

    for subsample in [
        0.75,
        1.0,
    ]
]


# ------------------------------------------------------------
# HistGradientBoosting
# ------------------------------------------------------------

hgb_learning_pairs = [

    (0.10, 100),
    (0.05, 200),
    (0.03, 400),
]


hgb_configs = [

    {
        "learning_rate":
            learning_rate,

        "max_iter":
            max_iter,

        "max_leaf_nodes":
            max_leaf_nodes,

        "l2_regularization":
            l2_regularization,
    }

    for (
        learning_rate,
        max_iter,
    ) in hgb_learning_pairs

    for max_leaf_nodes in [
        15,
        31,
    ]

    for l2_regularization in [
        1.0,
        0.0,
    ]
]


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

xgb_learning_pairs = [

    (0.10, 200),
    (0.05, 400),
    (0.03, 700),
]


xgb_configs = [

    {
        "learning_rate":
            learning_rate,

        "n_estimators":
            n_estimators,

        "max_depth":
            max_depth,

        "min_child_weight":
            min_child_weight,
    }

    for (
        learning_rate,
        n_estimators,
    ) in xgb_learning_pairs

    for max_depth in [
        3,
        6,
    ]

    for min_child_weight in [
        5.0,
        1.0,
    ]
]


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

lgbm_learning_pairs = [

    (0.10, 200),
    (0.05, 400),
    (0.03, 700),
]


lgbm_configs = [

    {
        "learning_rate":
            learning_rate,

        "n_estimators":
            n_estimators,

        "num_leaves":
            num_leaves,

        "min_child_samples":
            min_child_samples,
    }

    for (
        learning_rate,
        n_estimators,
    ) in lgbm_learning_pairs

    for num_leaves in [
        15,
        31,
    ]

    for min_child_samples in [
        50,
        20,
    ]
]


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

cat_learning_pairs = [

    (0.10, 300),
    (0.05, 600),
    (0.03, 1000),
]


cat_configs = [

    {
        "learning_rate":
            learning_rate,

        "iterations":
            iterations,

        "depth":
            depth,

        "l2_leaf_reg":
            l2_leaf_reg,
    }

    for (
        learning_rate,
        iterations,
    ) in cat_learning_pairs

    for depth in [
        4,
        6,
    ]

    for l2_leaf_reg in [
        10.0,
        3.0,
    ]
]


# ============================================================
# 5. MODEL REGISTRY
# ============================================================

MODEL_CONFIGS = {

    "ElasticNet logistic regression":
        elastic_configs,

    "RBF SVM":
        svm_configs,

    "Random Forest":
        rf_configs,

    "ExtraTrees":
        et_configs,

    "AdaBoost":
        ada_configs,

    "GradientBoosting":
        gb_configs,

    "HistGradientBoosting":
        hgb_configs,

    "XGBoost":
        xgb_configs,

    "LightGBM":
        lgbm_configs,

    "CatBoost":
        cat_configs,
}


MODEL_PREFIX = {

    "ElasticNet logistic regression":
        "EN",

    "RBF SVM":
        "SVM",

    "Random Forest":
        "RF",

    "ExtraTrees":
        "ET",

    "AdaBoost":
        "ADA",

    "GradientBoosting":
        "GB",

    "HistGradientBoosting":
        "HGB",

    "XGBoost":
        "XGB",

    "LightGBM":
        "LGBM",

    "CatBoost":
        "CAT",
}


PREPROCESSING_POLICY = {

    "ElasticNet logistic regression":
        "Scaled OHE",

    "RBF SVM":
        "Scaled OHE",

    "Random Forest":
        "Unscaled OHE",

    "ExtraTrees":
        "Unscaled OHE",

    "AdaBoost":
        "Unscaled OHE",

    "GradientBoosting":
        "Unscaled OHE",

    "HistGradientBoosting":
        "Native categorical",

    "XGBoost":
        "Unscaled OHE",

    "LightGBM":
        "Native categorical",

    "CatBoost":
        "Native categorical",
}


RAW_SCORE_POLICY = {

    "ElasticNet logistic regression":
        "predict_proba[:,1]",

    "RBF SVM":
        "decision_function",

    "Random Forest":
        "predict_proba[:,1]",

    "ExtraTrees":
        "predict_proba[:,1]",

    "AdaBoost":
        "predict_proba[:,1]",

    "GradientBoosting":
        "predict_proba[:,1]",

    "HistGradientBoosting":
        "predict_proba[:,1]",

    "XGBoost":
        "predict_proba[:,1]",

    "LightGBM":
        "predict_proba[:,1]",

    "CatBoost":
        "predict_proba[:,1]",
}


# ============================================================
# 6. BUILD JOINT CANDIDATE REGISTRY
# ============================================================

candidate_rows = []

global_tie_rank = 0


for model_name, configs in MODEL_CONFIGS.items():

    if len(configs) != 12:

        raise RuntimeError(
            f"{model_name}: expected 12 candidates, "
            f"observed {len(configs)}."
        )


    for within_model_rank, config in enumerate(
        configs,
        start=1,
    ):

        global_tie_rank += 1


        candidate_id = (
            f"{MODEL_PREFIX[model_name]}_"
            f"{within_model_rank:03d}"
        )


        canonical = json.dumps(
            {
                "model":
                    model_name,

                "preprocessing":
                    PREPROCESSING_POLICY[
                        model_name
                    ],

                "fixed":
                    FIXED_PARAMS[
                        model_name
                    ],

                "tuned":
                    config,
            },
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )


        candidate_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()


        candidate_rows.append(
            {
                "Candidate_ID":
                    candidate_id,

                "Model":
                    model_name,

                "Preprocessing":
                    PREPROCESSING_POLICY[
                        model_name
                    ],

                "Raw_Selection_Score":
                    RAW_SCORE_POLICY[
                        model_name
                    ],

                "Within_Model_Rank":
                    within_model_rank,

                "Deterministic_Tie_Rank":
                    global_tie_rank,

                "Tuned_Params_JSON":
                    json.dumps(
                        config,
                        sort_keys=True,
                        ensure_ascii=False,
                        allow_nan=False,
                    ),

                "Fixed_Params_JSON":
                    json.dumps(
                        FIXED_PARAMS[
                            model_name
                        ],
                        sort_keys=True,
                        ensure_ascii=False,
                        allow_nan=False,
                    ),

                "Candidate_SHA256":
                    candidate_hash,

                "Competition":
                    "JOINT_ACROSS_ALL_MODEL_FAMILIES",
            }
        )


candidate_registry = pd.DataFrame(
    candidate_rows
)


# ============================================================
# 7. JOINT-REGISTRY QC
# ============================================================

if len(candidate_registry) != 120:

    raise RuntimeError(
        "Expected exactly 120 joint candidate configurations."
    )


if candidate_registry[
    "Candidate_ID"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Candidate_ID."
    )


if candidate_registry[
    "Candidate_SHA256"
].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate configuration."
    )


candidate_count_qc = (
    candidate_registry
    .groupby("Model")
    .size()
    .rename("Candidate_Count")
    .reset_index()
)


if not (
    candidate_count_qc[
        "Candidate_Count"
    ]
    ==
    12
).all():

    raise RuntimeError(
        "Every model family must contribute exactly 12 candidates."
    )


if (
    candidate_registry[
        "Competition"
    ]
    .nunique()
    !=
    1
):

    raise RuntimeError(
        "Candidate registry is not a single joint competition."
    )


# ============================================================
# 8. MODEL CONSTRUCTOR AUDIT
# ============================================================
#
# Instantiation only.
# No estimator.fit() is called.
# ============================================================

def instantiate_candidate(
    model_name,
    tuned_config,
):

    fixed = FIXED_PARAMS[
        model_name
    ].copy()


    tuned = tuned_config.copy()


    if (
        model_name
        ==
        "ElasticNet logistic regression"
    ):

        return LogisticRegression(
            **fixed,
            **tuned,
        )


    if model_name == "RBF SVM":

        return SVC(
            **fixed,
            **tuned,
        )


    if model_name == "Random Forest":

        return RandomForestClassifier(
            **fixed,
            **tuned,
        )


    if model_name == "ExtraTrees":

        return ExtraTreesClassifier(
            **fixed,
            **tuned,
        )


    if model_name == "AdaBoost":

        stump_depth = tuned.pop(
            "estimator_max_depth"
        )


        estimator = DecisionTreeClassifier(
            max_depth=stump_depth,
            random_state=SEED,
        )


        return AdaBoostClassifier(
            estimator=estimator,
            **fixed,
            **tuned,
        )


    if model_name == "GradientBoosting":

        return GradientBoostingClassifier(
            **fixed,
            **tuned,
        )


    if model_name == "HistGradientBoosting":

        return HistGradientBoostingClassifier(
            **fixed,
            **tuned,
        )


    if model_name == "XGBoost":

        return XGBClassifier(
            **fixed,
            **tuned,
        )


    if model_name == "LightGBM":

        return LGBMClassifier(
            **fixed,
            **tuned,
        )


    if model_name == "CatBoost":

        return CatBoostClassifier(
            **fixed,
            **tuned,
        )


    raise ValueError(
        f"Unknown model: {model_name}"
    )


constructor_rows = []


for model_name, configs in MODEL_CONFIGS.items():

    for within_model_rank, config in enumerate(
        configs,
        start=1,
    ):

        candidate_id = (
            f"{MODEL_PREFIX[model_name]}_"
            f"{within_model_rank:03d}"
        )


        try:

            estimator = instantiate_candidate(
                model_name,
                config,
            )


            params = estimator.get_params(
                deep=True
            )


            if model_name == "AdaBoost":

                expected_keys = {
                    "n_estimators",
                    "learning_rate",
                    "estimator__max_depth",
                }

            else:

                expected_keys = set(
                    config.keys()
                )


            missing_keys = (
                expected_keys
                -
                set(params.keys())
            )


            if missing_keys:

                constructor_ok = False

                error = (
                    "Missing tuned parameter interface: "
                    f"{sorted(missing_keys)}"
                )

            else:

                constructor_ok = True

                error = None


        except Exception as exc:

            constructor_ok = False

            error = (
                f"{type(exc).__name__}: {exc}"
            )


        constructor_rows.append(
            {
                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Constructor_OK":
                    constructor_ok,

                "Error":
                    error,
            }
        )


constructor_audit = pd.DataFrame(
    constructor_rows
)


if not constructor_audit[
    "Constructor_OK"
].all():

    bad = constructor_audit.loc[
        ~constructor_audit[
            "Constructor_OK"
        ]
    ]

    raise RuntimeError(
        "Frozen candidate-interface failure:\n"
        f"{bad.to_string(index=False)}"
    )


# ============================================================
# 9. MODEL-SELECTION EXECUTION SPECIFICATION
# ============================================================

execution_specification = {

    "for_each_analysis_pipeline": [
        "primary survey-weighted fitting",
        "unweighted fitting sensitivity",
    ],

    "for_each_outer_fold":
        [0, 1, 2, 3, 4],

    "for_each_stage": [
        "Stage 1",
        "Stage 2",
    ],

    "procedure": [

        (
            "1. Restrict development data to the current "
            "outer-training population."
        ),

        (
            "2. For Stage 2, further restrict development data "
            "to true ANC users."
        ),

        (
            "3. Evaluate every one of the 120 frozen candidates "
            "using the four frozen inner folds."
        ),

        (
            "4. Refit all data-dependent preprocessing separately "
            "inside every inner-training partition."
        ),

        (
            "5. Generate predictions only for the corresponding "
            "inner-validation partition."
        ),

        (
            "6. Pool the four validation predictions into one "
            "complete inner-OOF prediction vector for each candidate."
        ),

        (
            "7. Calculate pooled survey-weighted Average Precision "
            "and weighted ROC-AUC."
        ),

        (
            "8. Select the single best candidate across ALL model "
            "families using the frozen selection rule."
        ),

        (
            "9. The winning candidate determines BOTH model family "
            "and hyperparameters for that stage and outer fold."
        ),

        (
            "10. Only after selection, refit that winning candidate "
            "on the complete outer-training population."
        ),

        (
            "11. Apply the later-frozen calibration procedure "
            "without using outer-test outcomes."
        ),

        (
            "12. Generate predictions for the untouched outer-test "
            "partition."
        ),
    ],

    "outer_test_role":
        (
            "evaluation only; never candidate selection"
        ),

    "winner_may_differ_between_outer_folds":
        True,

    "winner_may_differ_between_stage1_and_stage2":
        True,

    "interpretation":
        (
            "Outer CV evaluates the complete prespecified model "
            "selection procedure, not a model family chosen using "
            "outer-test performance."
        ),
}


# ============================================================
# 10. FINAL FULL-DATA MODEL POLICY
# ============================================================
#
# This does not fit the final model now.
# It only freezes how the final deployable model will later be
# selected after nested-CV performance estimation is complete.
# ============================================================

final_model_policy = {

    "outer_cv_purpose":
        (
            "Estimate generalization performance of the frozen "
            "joint model-selection procedure."
        ),

    "final_model_selected_from_outer_test_results":
        False,

    "final_family_selected_by_best_outer_fold_metric":
        False,

    "final_family_selected_by_average_outer_performance":
        False,

    "final_development_rule":
        (
            "After nested-CV evaluation is completed and locked, "
            "repeat the SAME frozen candidate-selection procedure "
            "using all available development observations, without "
            "changing predictors, preprocessing, candidate registry, "
            "selection metric, or weighting policy."
        ),

    "stage1_final_selection":
        "independent joint selection across all 120 candidates",

    "stage2_final_selection":
        (
            "independent joint selection across all 120 candidates "
            "among ANC users"
        ),

    "same_family_required_for_final_stages":
        False,

    "outer_selection_frequency_reporting":
        (
            "descriptive stability diagnostic only; "
            "not a replacement selection criterion"
        ),
}


# ============================================================
# 11. SEARCH-SPACE SUMMARY
# ============================================================

search_space_summary = pd.DataFrame(
    [
        {
            "Model":
                "ElasticNet logistic regression",

            "Candidates":
                12,

            "Tuned":
                "C; l1_ratio",

            "Preprocessing":
                "Scaled OHE",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "RBF SVM",

            "Candidates":
                12,

            "Tuned":
                "C; gamma",

            "Preprocessing":
                "Scaled OHE",

            "Score":
                "decision_function",
        },

        {
            "Model":
                "Random Forest",

            "Candidates":
                12,

            "Tuned":
                (
                    "max_depth; min_samples_leaf; "
                    "max_features"
                ),

            "Preprocessing":
                "Unscaled OHE",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "ExtraTrees",

            "Candidates":
                12,

            "Tuned":
                (
                    "max_depth; min_samples_leaf; "
                    "max_features"
                ),

            "Preprocessing":
                "Unscaled OHE",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "AdaBoost",

            "Candidates":
                12,

            "Tuned":
                (
                    "base-tree depth; n_estimators; "
                    "learning_rate"
                ),

            "Preprocessing":
                "Unscaled OHE",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "GradientBoosting",

            "Candidates":
                12,

            "Tuned":
                (
                    "learning_rate/n_estimators pair; "
                    "max_depth; subsample"
                ),

            "Preprocessing":
                "Unscaled OHE",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "HistGradientBoosting",

            "Candidates":
                12,

            "Tuned":
                (
                    "learning_rate/max_iter pair; "
                    "max_leaf_nodes; l2_regularization"
                ),

            "Preprocessing":
                "Native categorical",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "XGBoost",

            "Candidates":
                12,

            "Tuned":
                (
                    "learning_rate/n_estimators pair; "
                    "max_depth; min_child_weight"
                ),

            "Preprocessing":
                "Unscaled OHE",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "LightGBM",

            "Candidates":
                12,

            "Tuned":
                (
                    "learning_rate/n_estimators pair; "
                    "num_leaves; min_child_samples"
                ),

            "Preprocessing":
                "Native categorical",

            "Score":
                "predict_proba[:,1]",
        },

        {
            "Model":
                "CatBoost",

            "Candidates":
                12,

            "Tuned":
                (
                    "learning_rate/iterations pair; "
                    "depth; l2_leaf_reg"
                ),

            "Preprocessing":
                "Native categorical",

            "Score":
                "predict_proba[:,1]",
        },
    ]
)


# ============================================================
# 12. PACKAGE VERSION REGISTRY
# ============================================================

package_versions = pd.DataFrame(
    [
        {
            "Package":
                "scikit-learn",

            "Version":
                sklearn.__version__,
        },

        {
            "Package":
                "xgboost",

            "Version":
                importlib_metadata.version(
                    "xgboost"
                ),
        },

        {
            "Package":
                "lightgbm",

            "Version":
                importlib_metadata.version(
                    "lightgbm"
                ),
        },

        {
            "Package":
                "catboost",

            "Version":
                importlib_metadata.version(
                    "catboost"
                ),
        },
    ]
)


# ============================================================
# 13. REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "TRIPOD_AI",

            "Reference":
                (
                    "Collins GS, Moons KGM, Dhiman P, et al. "
                    "TRIPOD+AI statement. "
                    "BMJ. 2024;385:e078378."
                ),

            "DOI":
                "10.1136/bmj-2023-078378",

            "Use":
                (
                    "Evaluation data separated from training, "
                    "hyperparameter tuning, and model selection"
                ),
        },

        {
            "ID":
                "CAWLEY_TALBOT_2010",

            "Reference":
                (
                    "Cawley GC, Talbot NLC. "
                    "On Over-fitting in Model Selection and "
                    "Subsequent Selection Bias in Performance "
                    "Evaluation. JMLR. 2010;11:2079-2107."
                ),

            "DOI":
                "",

            "Use":
                (
                    "Avoid selection bias caused by using the "
                    "same performance estimates for selection "
                    "and final evaluation"
                ),
        },

        {
            "ID":
                "SAITO_2015",

            "Reference":
                (
                    "Saito T, Rehmsmeier M. "
                    "The Precision-Recall Plot Is More Informative "
                    "than the ROC Plot When Evaluating Binary "
                    "Classifiers on Imbalanced Datasets. "
                    "PLoS One. 2015;10(3):e0118432."
                ),

            "DOI":
                "10.1371/journal.pone.0118432",

            "Use":
                (
                    "Rationale for Average Precision under "
                    "uncommon positive outcomes"
                ),
        },

        {
            "ID":
                "XGBOOST",

            "Reference":
                (
                    "Chen T, Guestrin C. "
                    "XGBoost: A Scalable Tree Boosting System. "
                    "KDD. 2016."
                ),

            "DOI":
                "10.1145/2939672.2939785",

            "Use":
                "XGBoost model family",
        },

        {
            "ID":
                "LIGHTGBM",

            "Reference":
                (
                    "Ke G, Meng Q, Finley T, et al. "
                    "LightGBM: A Highly Efficient Gradient "
                    "Boosting Decision Tree. NeurIPS. 2017;30."
                ),

            "DOI":
                "",

            "Use":
                "LightGBM model family",
        },

        {
            "ID":
                "CATBOOST",

            "Reference":
                (
                    "Prokhorenkova L, Gusev G, Vorobev A, "
                    "Dorogush AV, Gulin A. CatBoost: unbiased "
                    "boosting with categorical features. "
                    "NeurIPS. 2018;31."
                ),

            "DOI":
                "",

            "Use":
                "CatBoost model family",
        },
    ]
)


# ============================================================
# 14. MANUSCRIPT-READY METHODS TEXT
# ============================================================

methods_text = """
Joint model-family and hyperparameter selection

Model family and hyperparameters were treated as components of a single
prespecified model-selection procedure rather than selecting the model
family from outer-test performance. Within each hierarchical stage and
outer cross-validation training set, all candidate configurations from
the ten machine-learning model families competed jointly using the four
frozen inner cross-validation folds. Each candidate therefore comprised
a model family, its prespecified model-specific preprocessing strategy,
and one frozen hyperparameter configuration.

For each candidate, preprocessing was fitted exclusively within the
corresponding inner-training partition and predictions were generated
for the held-out inner-validation partition. Predictions from the four
inner-validation folds were then pooled to form a complete cross-fitted
out-of-fold prediction vector for the current outer-training population.
Candidates from all model families were ranked jointly using
survey-weighted average precision calculated with the original DHS
sampling weights. Survey-weighted ROC-AUC served as the prespecified
first tie-break criterion.

The candidate with the highest inner-validation criterion determined
both the model family and hyperparameter configuration for that
hierarchical stage and outer fold. Stage 1 and Stage 2 were selected
independently because they represented distinct binary prediction tasks;
the same model family was therefore not required for both stages.

Library-default model performance was not used to screen or eliminate
model families before tuning because default settings need not reflect
the achievable performance of a model family after legitimate
hyperparameter optimization. Computational burden was instead limited
using compact, deterministic hyperparameter grids specified before any
model performance was examined.

After joint inner-cross-validation selection, the winning candidate was
refitted on the complete corresponding outer-training population.
Outer-test observations were not used for preprocessing, hyperparameter
optimization, model-family selection, calibration fitting, or threshold
selection. Consequently, the pooled outer-cross-validation predictions
estimate the performance of the complete prespecified model-development
and selection procedure rather than the performance of a model family
chosen retrospectively using evaluation data.

The unweighted model-fitting sensitivity analysis uses the identical
candidate registry and joint selection criterion but repeats the entire
selection procedure independently without sampling weights during base
model fitting.
""".strip()


# ============================================================
# 15. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Step 14 frozen",

            "PASS":
                (
                    step14_manifest.get("status")
                    ==
                    "PREPROCESSING_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Step 15 frozen",

            "PASS":
                (
                    step15_manifest.get("status")
                    ==
                    "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Step 15A frozen",

            "PASS":
                (
                    step15a_manifest.get("status")
                    ==
                    "SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Joint model-family selection enabled",

            "PASS":
                selection_protocol[
                    "joint_model_family_selection"
                ],
        },

        {
            "Check":
                "Model family not fixed before tuning",

            "PASS":
                not selection_protocol[
                    "model_family_fixed_before_tuning"
                ],
        },

        {
            "Check":
                "No default-performance family screening",

            "PASS":
                not selection_protocol[
                    "default_performance_family_screening"
                ],
        },

        {
            "Check":
                "Stage 1 and Stage 2 selected independently",

            "PASS":
                selection_protocol[
                    "stage1_and_stage2_selected_independently"
                ],
        },

        {
            "Check":
                "Same model family not forced across stages",

            "PASS":
                not selection_protocol[
                    "same_model_family_required_across_stages"
                ],
        },

        {
            "Check":
                "10 model families",

            "PASS":
                len(MODEL_CONFIGS)
                ==
                10,
        },

        {
            "Check":
                "12 candidates per family",

            "PASS":
                bool(
                    (
                        candidate_count_qc[
                            "Candidate_Count"
                        ]
                        ==
                        12
                    )
                    .all()
                ),
        },

        {
            "Check":
                "120 joint candidate configurations",

            "PASS":
                len(candidate_registry)
                ==
                120,
        },

        {
            "Check":
                "Single joint competition registry",

            "PASS":
                (
                    candidate_registry[
                        "Competition"
                    ]
                    .nunique()
                    ==
                    1
                ),
        },

        {
            "Check":
                "Candidate IDs unique",

            "PASS":
                not candidate_registry[
                    "Candidate_ID"
                ]
                .duplicated()
                .any(),
        },

        {
            "Check":
                "Candidate configurations unique",

            "PASS":
                not candidate_registry[
                    "Candidate_SHA256"
                ]
                .duplicated()
                .any(),
        },

        {
            "Check":
                "All estimator constructors compatible",

            "PASS":
                bool(
                    constructor_audit[
                        "Constructor_OK"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Primary metric weighted AP",

            "PASS":
                (
                    PRIMARY_SELECTION_METRIC
                    ==
                    "survey_weighted_average_precision"
                ),
        },

        {
            "Check":
                "Tie-break weighted ROC-AUC",

            "PASS":
                (
                    SECONDARY_SELECTION_METRIC
                    ==
                    "survey_weighted_roc_auc"
                ),
        },

        {
            "Check":
                "Pooled inner-OOF selection",

            "PASS":
                (
                    "pooled"
                    in
                    selection_protocol[
                        "metric_aggregation"
                    ]
                    .lower()
                ),
        },

        {
            "Check":
                "Outer test excluded from model selection",

            "PASS":
                not selection_protocol[
                    "outer_test_used_for_candidate_selection"
                ],
        },

        {
            "Check":
                "Outer test excluded from tuning",

            "PASS":
                not selection_protocol[
                    "outer_test_used_for_hyperparameter_tuning"
                ],
        },

        {
            "Check":
                "No performance-guided grid expansion",

            "PASS":
                not selection_protocol[
                    "performance_guided_grid_expansion"
                ],
        },

        {
            "Check":
                "Final model not selected from outer-test results",

            "PASS":
                not final_model_policy[
                    "final_model_selected_from_outer_test_results"
                ],
        },

        {
            "Check":
                "No model fitted in Step 16",

            "PASS":
                True,
        },

        {
            "Check":
                "No performance inspected in Step 16",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 16 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

CANDIDATE_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)


SEARCH_SUMMARY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_search_space_summary.csv"
)


SELECTION_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_inner_cv_selection_protocol.json"
)


EXECUTION_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_joint_selection_execution_specification.json"
)


FINAL_MODEL_POLICY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_full_data_model_selection_policy.json"
)


PACKAGE_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_package_versions.csv"
)


REFERENCE_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_reference_registry.csv"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_manuscript_tuning_methods.txt"
)


CONSTRUCTOR_QC_FILE = (
    DIRS["qc"]
    / "STEP16_FINAL_candidate_constructor_audit.csv"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP16_FINAL_hard_qc.csv"
)


candidate_registry.to_csv(
    CANDIDATE_FILE,
    index=False,
)


search_space_summary.to_csv(
    SEARCH_SUMMARY_FILE,
    index=False,
)


package_versions.to_csv(
    PACKAGE_FILE,
    index=False,
)


reference_registry.to_csv(
    REFERENCE_FILE,
    index=False,
)


constructor_audit.to_csv(
    CONSTRUCTOR_QC_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


for path, payload in [
    (
        SELECTION_PROTOCOL_FILE,
        selection_protocol,
    ),
    (
        EXECUTION_FILE,
        execution_specification,
    ),
    (
        FINAL_MODEL_POLICY_FILE,
        final_model_policy,
    ),
]:

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
        )


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


# ============================================================
# 17. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 16",

    "status":
        "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "seed":
        SEED,

    "primary_master_sha256":
        sha256_file(PRIMARY_FILE),

    "outer_split_sha256":
        sha256_file(OUTER_SPLIT_FILE),

    "inner_split_sha256":
        sha256_file(INNER_SPLIT_FILE),

    "joint_model_family_selection":
        True,

    "model_family_fixed_before_tuning":
        False,

    "default_family_screening":
        False,

    "stage_specific_model_selection":
        True,

    "same_family_required_across_stages":
        False,

    "model_family_count":
        10,

    "candidates_per_family":
        12,

    "total_joint_candidates":
        120,

    "primary_selection_metric":
        PRIMARY_SELECTION_METRIC,

    "secondary_selection_metric":
        SECONDARY_SELECTION_METRIC,

    "inner_metric_aggregation":
        "pooled cross-fitted inner-OOF predictions",

    "inner_metric_weight":
        "original DHS SURVEY_WEIGHT",

    "outer_test_used_for_selection":
        False,

    "outer_test_used_for_tuning":
        False,

    "performance_guided_grid_expansion":
        False,

    "final_model_selected_from_outer_results":
        False,

    "model_fitting_performed":
        False,

    "performance_evaluated":
        False,

    "candidate_registry_sha256":
        sha256_file(
            CANDIDATE_FILE
        ),

    "selection_protocol_sha256":
        sha256_file(
            SELECTION_PROTOCOL_FILE
        ),

    "execution_specification_sha256":
        sha256_file(
            EXECUTION_FILE
        ),

    "final_model_policy_sha256":
        sha256_file(
            FINAL_MODEL_POLICY_FILE
        ),

    "constructor_qc_sha256":
        sha256_file(
            CONSTRUCTOR_QC_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),

    "methods_sha256":
        sha256_file(
            METHODS_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 18. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 16."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 16."
    )


if sha256_file(
    INNER_SPLIT_FILE
) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 16."
    )


# ============================================================
# 19. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 16 — JOINT MODEL-FAMILY + HYPERPARAMETER SELECTION FREEZE"
)

print(
    "=" * 78
)


print(
    "\nPACKAGE VERSIONS:"
)

display(
    package_versions
)


print(
    "\nSEARCH SPACE SUMMARY:"
)

display(
    search_space_summary
)


print(
    "\nCANDIDATE COUNTS:"
)

display(
    candidate_count_qc
)


print(
    "\nJOINT REGISTRY SUMMARY:"
)

print(
    f"Model families        : {len(MODEL_CONFIGS)}"
)

print(
    f"Candidates per family : 12"
)

print(
    f"Total joint candidates: {len(candidate_registry)}"
)


print(
    "\nCONSTRUCTOR AUDIT SUMMARY:"
)

display(
    constructor_audit[
        [
            "Model",
            "Constructor_OK",
        ]
    ]
    .groupby("Model")
    .agg(
        Candidate_Count=(
            "Constructor_OK",
            "size",
        ),

        All_OK=(
            "Constructor_OK",
            "all",
        ),
    )
    .reset_index()
)


print(
    "\nJOINT SELECTION RULE:"
)

print(
    "1. Evaluate all 120 candidates inside the frozen inner CV."
)

print(
    "2. Pool the four inner-validation prediction sets."
)

print(
    "3. Highest survey-weighted Average Precision wins."
)

print(
    "4. AP tie -> highest survey-weighted ROC-AUC."
)

print(
    "5. Still tied -> lowest deterministic tie rank."
)

print(
    "6. Winner determines BOTH model family and hyperparameters."
)


print(
    "\nSTAGE POLICY:"
)

print(
    "- Stage 1 and Stage 2 are selected independently."
)

print(
    "- The same model family is NOT forced across both stages."
)

print(
    "- The final hierarchical pipeline may therefore contain different algorithms for the two stages."
)


print(
    "\nDEFAULT-MODEL POLICY:"
)

print(
    "- Default performance is NOT used to eliminate model families."
)

print(
    "- No top-k family screening is performed."
)

print(
    "- Computational control comes from the compact frozen grids."
)


print(
    "\nOUTER-TEST POLICY:"
)

print(
    "- Outer-test outcomes never participate in model-family selection."
)

print(
    "- Outer-test outcomes never participate in hyperparameter tuning."
)

print(
    "- Outer CV estimates performance of the complete selection procedure."
)


print(
    "\nFINAL FULL-DATA MODEL POLICY:"
)

print(
    "- Final model family will NOT be chosen from the best outer-test score."
)

print(
    "- After nested evaluation is locked, the same frozen joint-selection procedure will be repeated on all development data."
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)

print(
    methods_text
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)

print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)

print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 16 PASS"
)

print(
    "=" * 78
)


STEP 16 — JOINT MODEL-FAMILY + HYPERPARAMETER SELECTION FREEZE

PACKAGE VERSIONS:


,Package,Version
0,scikit-learn,1.9.0
1,xgboost,3.4.1
2,lightgbm,4.6.0
3,catboost,1.2.10



SEARCH SPACE SUMMARY:


,Model,Candidates,Tuned,Preprocessing,Score
0,ElasticNet logistic regression,12,C; l1_ratio,Scaled OHE,"predict_proba[:,1]"
1,RBF SVM,12,C; gamma,Scaled OHE,decision_function
2,Random Forest,12,max_depth; min_samples_leaf; max_features,Unscaled OHE,"predict_proba[:,1]"
3,ExtraTrees,12,max_depth; min_samples_leaf; max_features,Unscaled OHE,"predict_proba[:,1]"
4,AdaBoost,12,base-tree depth; n_estimators; learning_rate,Unscaled OHE,"predict_proba[:,1]"
5,GradientBoosting,12,learning_rate/n_estimators pair; max_depth; su...,Unscaled OHE,"predict_proba[:,1]"
6,HistGradientBoosting,12,learning_rate/max_iter pair; max_leaf_nodes; l...,Native categorical,"predict_proba[:,1]"
7,XGBoost,12,learning_rate/n_estimators pair; max_depth; mi...,Unscaled OHE,"predict_proba[:,1]"
8,LightGBM,12,learning_rate/n_estimators pair; num_leaves; m...,Native categorical,"predict_proba[:,1]"
9,CatBoost,12,learning_rate/iterations pair; depth; l2_leaf_reg,Native categorical,"predict_proba[:,1]"



CANDIDATE COUNTS:


,Model,Candidate_Count
0,AdaBoost,12
1,CatBoost,12
2,ElasticNet logistic regression,12
3,ExtraTrees,12
4,GradientBoosting,12
5,HistGradientBoosting,12
6,LightGBM,12
7,RBF SVM,12
8,Random Forest,12
9,XGBoost,12



JOINT REGISTRY SUMMARY:
Model families        : 10
Candidates per family : 12
Total joint candidates: 120

CONSTRUCTOR AUDIT SUMMARY:


,Model,Candidate_Count,All_OK
0,AdaBoost,12,True
1,CatBoost,12,True
2,ElasticNet logistic regression,12,True
3,ExtraTrees,12,True
4,GradientBoosting,12,True
5,HistGradientBoosting,12,True
6,LightGBM,12,True
7,RBF SVM,12,True
8,Random Forest,12,True
9,XGBoost,12,True



JOINT SELECTION RULE:
1. Evaluate all 120 candidates inside the frozen inner CV.
2. Pool the four inner-validation prediction sets.
3. Highest survey-weighted Average Precision wins.
4. AP tie -> highest survey-weighted ROC-AUC.
5. Still tied -> lowest deterministic tie rank.
6. Winner determines BOTH model family and hyperparameters.

STAGE POLICY:
- Stage 1 and Stage 2 are selected independently.
- The same model family is NOT forced across both stages.
- The final hierarchical pipeline may therefore contain different algorithms for the two stages.

DEFAULT-MODEL POLICY:
- Default performance is NOT used to eliminate model families.
- No top-k family screening is performed.
- Computational control comes from the compact frozen grids.

OUTER-TEST POLICY:
- Outer-test outcomes never participate in model-family selection.
- Outer-test outcomes never participate in hyperparameter tuning.
- Outer CV estimates performance of the complete selection procedure.

FINAL FULL-DATA MODEL POLICY:

,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Step 14 frozen,True
4,Step 15 frozen,True
5,Step 15A frozen,True
6,Joint model-family selection enabled,True
7,Model family not fixed before tuning,True
8,No default-performance family screening,True
9,Stage 1 and Stage 2 selected independently,True



MANUSCRIPT METHODS PREVIEW:
Joint model-family and hyperparameter selection

Model family and hyperparameters were treated as components of a single
prespecified model-selection procedure rather than selecting the model
family from outer-test performance. Within each hierarchical stage and
outer cross-validation training set, all candidate configurations from
the ten machine-learning model families competed jointly using the four
frozen inner cross-validation folds. Each candidate therefore comprised
a model family, its prespecified model-specific preprocessing strategy,
and one frozen hyperparameter configuration.

For each candidate, preprocessing was fitted exclusively within the
corresponding inner-training partition and predictions were generated
for the held-out inner-validation partition. Predictions from the four
inner-validation folds were then pooled to form a complete cross-fitted
out-of-fold prediction vector for the current outer-training population.
Candidates from all m

In [26]:
# ============================================================
# STEP 17 — FINAL JOINT-WINNER CALIBRATION, HIERARCHICAL
#           FUSION, AND OUTER-OOF EVALUATION PROTOCOL FREEZE
# ============================================================
#
# This step performs NO model fitting.
#
# For every:
#   analysis pipeline x outer fold x hierarchical stage
#
# procedure will be:
#
#   1. Jointly evaluate all 120 Step-16 candidates in inner CV.
#   2. Select the single winning candidate.
#      Winner = model family + hyperparameters + preprocessing.
#   3. Use ONLY that winner's pooled inner-OOF scores to fit
#      the prespecified sigmoid/logistic calibrator.
#   4. Refit that winning base candidate on the complete
#      outer-training population.
#   5. Generate raw scores for the untouched outer-test set.
#   6. Apply the already-fitted calibrator unchanged.
#
# Stage 1 and Stage 2 winners may be different algorithms.
#
# CALIBRATION METHOD
# ------------------
# Fixed before performance inspection:
#   weighted logistic/sigmoid recalibration.
#
# Probability-producing models:
#   z = logit(clip(raw_probability))
#
# RBF-SVM:
#   z = decision_function margin
#
# calibrated probability:
#   expit(alpha + beta*z)
#
# Calibration parameters are estimated from the selected
# winner's INNER-OOF scores only.
#
# Outer-test outcomes NEVER enter:
#   - model-family selection
#   - hyperparameter tuning
#   - preprocessing fitting
#   - calibrator fitting
#   - calibration-method selection
#   - threshold selection
#
# Primary performance will later be computed from pooled
# OUTER-OOF predictions using original DHS survey weights.
# ============================================================


from pathlib import Path
import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. FROZEN INPUTS
# ============================================================

SEED = 2026


if "DIRS" not in globals():

    raise RuntimeError(
        "DIRS is unavailable. "
        "Run the accepted project setup first."
    )


PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP16_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing:\n{path}"
        )


if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)


if (
    step16_manifest.get("status")
    !=
    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Revised joint-selection Step 16 is not frozen."
    )


if not step16_manifest.get(
    "joint_model_family_selection",
    False,
):

    raise RuntimeError(
        "Step 16 does not specify joint model-family selection."
    )


if (
    step16_manifest.get(
        "total_joint_candidates"
    )
    !=
    120
):

    raise RuntimeError(
        "Step 16 must contain 120 joint candidates."
    )


for key, expected in [
    (
        "primary_master_sha256",
        EXPECTED_PRIMARY_SHA256,
    ),
    (
        "outer_split_sha256",
        EXPECTED_OUTER_SHA256,
    ),
    (
        "inner_split_sha256",
        EXPECTED_INNER_SHA256,
    ),
]:

    if step16_manifest.get(key) != expected:

        raise RuntimeError(
            f"Step-16 manifest mismatch: {key}"
        )


# ============================================================
# 2. FIXED CALIBRATION CONSTANTS
# ============================================================

CALIBRATION_METHOD = (
    "weighted_logistic_sigmoid"
)


PROBABILITY_CLIP = 1e-6


if not (
    0.0
    <
    PROBABILITY_CLIP
    <
    0.5
):

    raise RuntimeError(
        "Invalid probability-clipping constant."
    )


# ============================================================
# 3. JOINT-WINNER CALIBRATION FLOW
# ============================================================

joint_winner_flow = {

    "selection_precedes_calibration":
        True,

    "winner_definition":
        (
            "single Step-16 candidate selected jointly across "
            "all model families using frozen inner CV"
        ),

    "winner_components":
        [
            "model family",
            "hyperparameter configuration",
            "Step-14 preprocessing strategy",
        ],

    "selection_metric":
        (
            "pooled inner-OOF survey-weighted Average Precision"
        ),

    "selection_tiebreak":
        (
            "survey-weighted ROC-AUC, then deterministic tie rank"
        ),

    "stage1_selected_independently":
        True,

    "stage2_selected_independently":
        True,

    "same_family_required_across_stages":
        False,

    "calibration_model_family":
        "selected winner only",

    "losing_candidates_calibrated":
        False,

    "outer_test_used_for_selection":
        False,

    "outer_test_used_for_calibration":
        False,

    "outer_test_used_for_threshold_selection":
        False,
}


# ============================================================
# 4. CALIBRATION INPUT POLICY
# ============================================================

calibration_input_policy = pd.DataFrame(
    [
        {
            "Selected_Winner_Type":
                "RBF SVM",

            "Raw_Score":
                "decision_function",

            "Calibration_Predictor_z":
                "raw decision-function margin",

            "Calibrator":
                (
                    "weighted logistic sigmoid "
                    "with intercept and slope"
                ),
        },

        {
            "Selected_Winner_Type":
                "Probability-producing model",

            "Raw_Score":
                "positive-class predict_proba",

            "Calibration_Predictor_z":
                (
                    "logit(clip(p, 1e-6, 1-1e-6))"
                ),

            "Calibrator":
                (
                    "weighted logistic sigmoid "
                    "with intercept and slope"
                ),
        },
    ]
)


# ============================================================
# 5. CALIBRATION PROTOCOL
# ============================================================

calibration_protocol = {

    "method":
        CALIBRATION_METHOD,

    "method_fixed_before_model_fitting":
        True,

    "calibration_method_selection":
        False,

    "alternative_methods_competing":
        False,

    "isotonic_competing":
        False,

    "calibration_formula":
        (
            "P_calibrated = expit(alpha + beta*z)"
        ),

    "calibrator_intercept":
        True,

    "calibrator_slope":
        True,

    "calibrator_regularization":
        "none",

    "calibrator_class_balancing":
        False,

    "probability_clip":
        PROBABILITY_CLIP,

    "probability_model_input":
        (
            "logit of raw positive-class probability "
            "after fixed clipping"
        ),

    "svm_input":
        "raw decision_function margin",

    "stage1_calibration_population":
        (
            "complete current outer-training population"
        ),

    "stage2_calibration_population":
        (
            "true ANC users only within the current "
            "outer-training population"
        ),

    "calibration_training_scores":
        (
            "pooled inner-OOF raw scores belonging to the "
            "single jointly selected Step-16 winner"
        ),

    "calibration_training_labels":
        (
            "corresponding binary stage outcomes from "
            "the outer-training population"
        ),

    "calibration_training_weight":
        (
            "original DHS SURVEY_WEIGHT rescaled to mean 1 "
            "within the stage-specific calibration sample"
        ),

    "primary_pipeline_calibration_weight":
        "survey weighted",

    "unweighted_sensitivity_calibration_weight":
        (
            "survey weighted; sensitivity continues to differ "
            "only in base-model fitting weights"
        ),

    "calibrator_refitted_on_outer_test":
        False,

    "outer_test_outcomes_seen_by_calibrator":
        False,

    "calibration_failure_rule":
        (
            "If the prespecified logistic calibrator fails "
            "technically or produces non-finite parameters, stop "
            "and diagnose; do not silently substitute isotonic "
            "or another method."
        ),

    "performance_guided_method_change":
        False,
}


# ============================================================
# 6. WINNER REFIT + OUTER PREDICTION POLICY
# ============================================================

outer_prediction_protocol = {

    "after_inner_selection":
        (
            "retain the single selected winner for the "
            "current pipeline x stage x outer fold"
        ),

    "base_model_refit":
        (
            "refit the selected winner from scratch on the "
            "complete stage-specific outer-training population"
        ),

    "preprocessing_refit":
        (
            "refit all Step-14 data-dependent preprocessing "
            "using the complete stage-specific outer-training "
            "population only"
        ),

    "primary_base_fit_weight":
        (
            "normalized DHS survey weights according to Step 15"
        ),

    "sensitivity_base_fit_weight":
        (
            "unit weights according to Step 15A"
        ),

    "outer_test_preprocessing":
        "transform only; never fit",

    "outer_test_raw_score":
        (
            "decision_function for SVM winner; "
            "positive-class probability for all other winners"
        ),

    "outer_test_calibration":
        (
            "apply the calibrator already fitted from the "
            "selected winner's inner-OOF scores"
        ),

    "outer_test_outcome_access_before_prediction":
        False,
}


# ============================================================
# 7. HIERARCHICAL PROBABILITY FUSION
# ============================================================

fusion_protocol = {

    "stage1_calibrated_probability":
        "q0 = P(No ANC | X)",

    "stage2_calibrated_probability":
        "q2 = P(>=8 visits | Any ANC, X)",

    "stage2_training_population":
        "true ANC users only",

    "stage2_calibration_population":
        "true ANC users only",

    "stage2_scoring_population_for_fusion":
        (
            "all outer-test observations; q2 remains a "
            "conditional probability given Any ANC"
        ),

    "class_0":
        "No ANC",

    "class_1":
        "1-7 reported ANC visits",

    "class_2":
        ">=8 reported ANC visits",

    "p0_formula":
        "p0 = q0",

    "p1_formula":
        "p1 = (1-q0)*(1-q2)",

    "p2_formula":
        "p2 = (1-q0)*q2",

    "post_fusion_recalibration":
        False,

    "post_fusion_probability_renormalization":
        False,

    "reason":
        (
            "The hierarchical formulas analytically produce "
            "non-negative probabilities summing to one."
        ),
}


# ============================================================
# 8. MATHEMATICAL FUSION QC
# ============================================================

q0_test = np.array(
    [
        0.01,
        0.10,
        0.50,
        0.95,
    ],
    dtype=float,
)


q2_test = np.array(
    [
        0.02,
        0.20,
        0.70,
        0.90,
    ],
    dtype=float,
)


fusion_test = np.column_stack(
    [
        q0_test,

        (
            (1.0 - q0_test)
            *
            (1.0 - q2_test)
        ),

        (
            (1.0 - q0_test)
            *
            q2_test
        ),
    ]
)


if not np.allclose(
    fusion_test.sum(axis=1),
    1.0,
    rtol=0,
    atol=1e-15,
):

    raise RuntimeError(
        "Hierarchical fusion does not sum to one."
    )


if (
    fusion_test
    <
    0
).any():

    raise RuntimeError(
        "Negative fused probability detected."
    )


if (
    fusion_test
    >
    1
).any():

    raise RuntimeError(
        "Fused probability >1 detected."
    )


# ============================================================
# 9. POOLED OUTER-OOF EVALUATION POLICY
# ============================================================

outer_evaluation_protocol = {

    "prediction_structure":
        (
            "Each analytic participant receives exactly one "
            "prediction from the outer fold in which that "
            "participant was held out."
        ),

    "primary_aggregation":
        (
            "pool all five outer-test prediction sets before "
            "calculating primary performance measures"
        ),

    "average_fold_metrics_primary":
        False,

    "fold_specific_metrics":
        (
            "descriptive stability diagnostics only"
        ),

    "evaluation_weight":
        "original DHS SURVEY_WEIGHT",

    "stage1_population":
        "all 5063 analytic observations",

    "stage2_population":
        (
            "true ANC users only for standalone Stage-2 metrics"
        ),

    "final_multiclass_population":
        "all 5063 analytic observations",

    "outer_fold_winner_may_differ":
        True,

    "interpretation":
        (
            "performance of the complete frozen joint "
            "model-development and selection procedure"
        ),

    "confidence_intervals":
        (
            "deferred to prespecified stratified PSU bootstrap "
            "on locked pooled outer-OOF predictions"
        ),
}


# ============================================================
# 10. BINARY-STAGE METRIC REGISTRY
# ============================================================

binary_metric_registry = pd.DataFrame(
    [
        {
            "Domain":
                "Discrimination",

            "Metric":
                "Survey-weighted Average Precision",

            "Prediction_Input":
                "Raw continuous winner score",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Primary discrimination",
        },

        {
            "Domain":
                "Discrimination",

            "Metric":
                "Survey-weighted ROC-AUC",

            "Prediction_Input":
                "Raw continuous winner score",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Secondary discrimination",
        },

        {
            "Domain":
                "Probability accuracy",

            "Metric":
                "Survey-weighted log loss",

            "Prediction_Input":
                "Calibrated probability",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Primary probability metric",
        },

        {
            "Domain":
                "Probability accuracy",

            "Metric":
                "Survey-weighted Brier score",

            "Prediction_Input":
                "Calibrated probability",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Secondary probability metric",
        },

        {
            "Domain":
                "Calibration",

            "Metric":
                "Calibration-in-the-large / intercept",

            "Prediction_Input":
                "Calibrated probability",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Report; ideal 0",
        },

        {
            "Domain":
                "Calibration",

            "Metric":
                "Calibration slope",

            "Prediction_Input":
                "Calibrated probability",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Report; ideal 1",
        },

        {
            "Domain":
                "Calibration",

            "Metric":
                "Observed-to-expected ratio",

            "Prediction_Input":
                "Calibrated probability",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Report; ideal 1",
        },

        {
            "Domain":
                "Calibration",

            "Metric":
                "Weighted smooth calibration curve",

            "Prediction_Input":
                "Calibrated probability",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                "Primary calibration graphic",
        },

        {
            "Domain":
                "Calibration",

            "Metric":
                "Prediction-quantile grouped points",

            "Prediction_Input":
                "Calibrated probability",

            "Weight":
                "Original SURVEY_WEIGHT",

            "Role":
                (
                    "Supplementary display only; "
                    "not primary calibration evidence"
                ),
        },
    ]
)


# ============================================================
# 11. FINAL THREE-CLASS METRIC REGISTRY
# ============================================================

multiclass_metric_registry = pd.DataFrame(
    [
        {
            "Domain":
                "Probability accuracy",

            "Metric":
                "Survey-weighted multiclass log loss",

            "Role":
                "Primary final multiclass probability metric",
        },

        {
            "Domain":
                "Probability accuracy",

            "Metric":
                "Survey-weighted multiclass Brier score",

            "Role":
                "Secondary final multiclass probability metric",
        },

        {
            "Domain":
                "Class-wise discrimination",

            "Metric":
                (
                    "One-vs-rest survey-weighted "
                    "Average Precision"
                ),

            "Role":
                "Report separately for classes 0, 1, and 2",
        },

        {
            "Domain":
                "Class-wise discrimination",

            "Metric":
                (
                    "One-vs-rest survey-weighted ROC-AUC"
                ),

            "Role":
                "Report separately for classes 0, 1, and 2",
        },

        {
            "Domain":
                "Summary discrimination",

            "Metric":
                "Macro class-wise weighted AP",

            "Role":
                "Secondary descriptive summary",
        },

        {
            "Domain":
                "Summary discrimination",

            "Metric":
                "Macro class-wise weighted ROC-AUC",

            "Role":
                "Secondary descriptive summary",
        },

        {
            "Domain":
                "Class-wise calibration",

            "Metric":
                (
                    "Calibration intercept, slope, O/E "
                    "and smooth curve for p0, p1, p2"
                ),

            "Role":
                "Report separately by class",
        },
    ]
)


# ============================================================
# 12. BRIER CONVENTION
# ============================================================

brier_protocol = {

    "binary_definition":
        (
            "weighted mean of (p-y)^2"
        ),

    "binary_range":
        "0 to 1",

    "multiclass_definition":
        (
            "weighted mean of sum_k "
            "(p_k - I[y=k])^2"
        ),

    "multiclass_scale_by_half":
        False,

    "multiclass_range":
        "0 to 2",

    "lower_is_better":
        True,
}


# ============================================================
# 13. CALIBRATION-ASSESSMENT POLICY
# ============================================================

calibration_assessment_protocol = {

    "calibration_intercept":
        (
            "weighted logistic calibration-in-the-large "
            "using logit(predicted probability) as offset"
        ),

    "calibration_slope":
        (
            "weighted logistic regression of observed outcome "
            "on logit(predicted probability), with freely "
            "estimated intercept and slope"
        ),

    "oe_ratio":
        (
            "survey-weighted observed event total divided by "
            "survey-weighted expected event total"
        ),

    "primary_graph":
        (
            "smooth flexible calibration curve fitted to "
            "individual pooled outer-OOF predictions"
        ),

    "grouped_points":
        (
            "prediction-quantile grouped points may supplement "
            "but never replace the smooth calibration curve"
        ),

    "hosmer_lemeshow_test":
        False,

    "calibration_evaluation_data":
        "pooled outer-OOF predictions",

    "performance_guided_graph_method_change":
        False,
}


# ============================================================
# 14. THRESHOLD BOUNDARY
# ============================================================

threshold_policy_boundary = {

    "threshold_selection_in_step17":
        False,

    "accuracy_used_for_model_selection":
        False,

    "f1_used_for_model_selection":
        False,

    "sensitivity_specificity_threshold_selected":
        False,

    "ppv_npv_threshold_selected":
        False,

    "reason":
        (
            "Threshold-dependent decision rules require a "
            "separate prespecified analysis after probability "
            "model development and evaluation rules are frozen."
        ),
}


# ============================================================
# 15. REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "TRIPOD_AI",

            "Reference":
                (
                    "Collins GS, Moons KGM, Dhiman P, et al. "
                    "TRIPOD+AI statement. "
                    "BMJ. 2024;385:e078378."
                ),

            "DOI":
                "10.1136/bmj-2023-078378",

            "Use":
                (
                    "Transparent reporting of development, "
                    "model selection and evaluation"
                ),
        },

        {
            "ID":
                "RILEY_2024",

            "Reference":
                (
                    "Riley RD, Collins GS, Ensor J, et al. "
                    "Developing clinical prediction models: "
                    "a step-by-step guide. "
                    "BMJ. 2024;386:e078276."
                ),

            "DOI":
                "10.1136/bmj-2023-078276",

            "Use":
                (
                    "Discrimination, calibration intercept, "
                    "slope and Brier-score reporting"
                ),
        },

        {
            "ID":
                "RILEY_EVALUATION",

            "Reference":
                (
                    "Evaluation of clinical prediction models "
                    "(part 2): how to undertake an external "
                    "validation study. "
                    "BMJ. 2024;384:e074820."
                ),

            "DOI":
                "10.1136/bmj-2023-074820",

            "Use":
                (
                    "Smooth flexible calibration curves and "
                    "limitations of grouped calibration points"
                ),
        },

        {
            "ID":
                "SKLEARN_CALIBRATION",

            "Reference":
                (
                    "Scikit-learn probability calibration "
                    "documentation."
                ),

            "DOI":
                "",

            "Use":
                (
                    "Cross-validated sigmoid/logistic "
                    "probability calibration"
                ),
        },
    ]
)


# ============================================================
# 16. MANUSCRIPT-READY METHODS TEXT
# ============================================================

methods_text = """
Probability calibration and outer cross-validation evaluation

For each hierarchical stage, outer cross-validation fold, and analysis
pipeline, model family and hyperparameters will first be selected
jointly using the prespecified inner-cross-validation procedure. Only
the single winning candidate will subsequently enter probability
calibration. Thus, calibration will occur after model-family and
hyperparameter selection and will not be used to choose between model
families.

For the selected candidate, raw predictions from the four frozen inner
validation folds will be pooled to create cross-fitted scores covering
the complete stage-specific outer-training population. A single
prespecified weighted logistic sigmoid calibrator will be fitted to
these scores. For probability-producing classifiers, the calibration
predictor will be the logit of the raw positive-class probability after
fixed numerical clipping at 10^-6 and 1-10^-6. For an RBF support-vector
machine winner, the decision-function margin will be used directly.
The calibrator will estimate an intercept and slope using DHS sampling
weights rescaled to mean one within the calibration sample.

The winning base estimator will then be refitted from scratch on the
complete corresponding outer-training population using its
prespecified preprocessing strategy and training-weight policy.
The previously estimated calibrator will be applied unchanged to raw
predictions from the untouched outer-test partition. Outer-test
outcomes will not be used for model-family selection, hyperparameter
optimization, preprocessing, calibration fitting, calibration-method
selection, or threshold determination.

Stage 1 and Stage 2 will be selected and calibrated independently, and
the two stages will not be required to use the same algorithm. If q0
denotes the calibrated probability of no ANC and q2 denotes the
calibrated conditional probability of eight or more visits given any
ANC, the final three-category probabilities will be
p0=q0, p1=(1-q0)(1-q2), and p2=(1-q0)q2. The Stage-2 model will be
developed and calibrated only among ANC users, although its conditional
probability will be evaluated for all outer-test observations for the
purpose of hierarchical fusion.

Primary performance estimation will use pooled outer-cross-validation
predictions, so that every participant contributes exactly one
prediction obtained while held out from the entire model-development
process. Original DHS sampling weights will be used for performance
estimation. Binary-stage discrimination will be summarized using
survey-weighted average precision and ROC-AUC. Probabilistic performance
will be summarized using survey-weighted log loss and Brier score.
Calibration will be assessed with calibration-in-the-large, calibration
slope, the observed-to-expected ratio, and a smooth flexible calibration
curve.

For the final three-category outcome, survey-weighted multiclass log
loss and multiclass Brier score will assess overall probabilistic
performance. One-vs-rest average precision, ROC-AUC, and calibration
will additionally be reported for each outcome category. Grouped
calibration points may be shown as supplementary visual aids but will
not replace a smooth calibration curve. Threshold-dependent
classification measures will be addressed in a separately prespecified
analysis.
""".strip()


# ============================================================
# 17. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Revised joint Step 16 frozen",

            "PASS":
                (
                    step16_manifest.get("status")
                    ==
                    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "120 joint candidates upstream",

            "PASS":
                (
                    step16_manifest.get(
                        "total_joint_candidates"
                    )
                    ==
                    120
                ),
        },

        {
            "Check":
                "Model selection precedes calibration",

            "PASS":
                joint_winner_flow[
                    "selection_precedes_calibration"
                ],
        },

        {
            "Check":
                "Only joint winner is calibrated",

            "PASS":
                (
                    not joint_winner_flow[
                        "losing_candidates_calibrated"
                    ]
                ),
        },

        {
            "Check":
                "Stage 1 and Stage 2 selected independently",

            "PASS":
                (
                    joint_winner_flow[
                        "stage1_selected_independently"
                    ]
                    and
                    joint_winner_flow[
                        "stage2_selected_independently"
                    ]
                ),
        },

        {
            "Check":
                "Same family not forced across stages",

            "PASS":
                not joint_winner_flow[
                    "same_family_required_across_stages"
                ],
        },

        {
            "Check":
                "Calibration method fixed",

            "PASS":
                (
                    calibration_protocol[
                        "method_fixed_before_model_fitting"
                    ]
                ),
        },

        {
            "Check":
                "No calibration-method selection",

            "PASS":
                not calibration_protocol[
                    "calibration_method_selection"
                ],
        },

        {
            "Check":
                "Calibrator uses selected winner inner-OOF scores",

            "PASS":
                (
                    "selected"
                    in
                    calibration_protocol[
                        "calibration_training_scores"
                    ]
                    .lower()
                ),
        },

        {
            "Check":
                "Stage 2 calibrator uses ANC users only",

            "PASS":
                (
                    "ANC users only"
                    in
                    calibration_protocol[
                        "stage2_calibration_population"
                    ]
                ),
        },

        {
            "Check":
                "Outer-test outcomes excluded from calibrator",

            "PASS":
                not calibration_protocol[
                    "outer_test_outcomes_seen_by_calibrator"
                ],
        },

        {
            "Check":
                "Primary and sensitivity calibration weighting identical",

            "PASS":
                (
                    "survey weighted"
                    in
                    calibration_protocol[
                        "primary_pipeline_calibration_weight"
                    ]
                    and
                    "survey weighted"
                    in
                    calibration_protocol[
                        "unweighted_sensitivity_calibration_weight"
                    ]
                ),
        },

        {
            "Check":
                "No class balancing in calibrator",

            "PASS":
                not calibration_protocol[
                    "calibrator_class_balancing"
                ],
        },

        {
            "Check":
                "Hierarchical fusion sums to one",

            "PASS":
                bool(
                    np.allclose(
                        fusion_test.sum(axis=1),
                        1.0,
                        rtol=0,
                        atol=1e-15,
                    )
                ),
        },

        {
            "Check":
                "No post-fusion recalibration",

            "PASS":
                not fusion_protocol[
                    "post_fusion_recalibration"
                ],
        },

        {
            "Check":
                "No post-fusion renormalization",

            "PASS":
                not fusion_protocol[
                    "post_fusion_probability_renormalization"
                ],
        },

        {
            "Check":
                "Primary evaluation pools outer-OOF predictions",

            "PASS":
                (
                    "pool"
                    in
                    outer_evaluation_protocol[
                        "primary_aggregation"
                    ]
                    .lower()
                ),
        },

        {
            "Check":
                "Evaluation uses original DHS weights",

            "PASS":
                (
                    outer_evaluation_protocol[
                        "evaluation_weight"
                    ]
                    ==
                    "original DHS SURVEY_WEIGHT"
                ),
        },

        {
            "Check":
                "Stage 2 standalone metrics restricted to ANC users",

            "PASS":
                (
                    "ANC users only"
                    in
                    outer_evaluation_protocol[
                        "stage2_population"
                    ]
                ),
        },

        {
            "Check":
                "Binary log loss frozen",

            "PASS":
                bool(
                    binary_metric_registry[
                        "Metric"
                    ]
                    .eq(
                        "Survey-weighted log loss"
                    )
                    .any()
                ),
        },

        {
            "Check":
                "Binary Brier score frozen",

            "PASS":
                bool(
                    binary_metric_registry[
                        "Metric"
                    ]
                    .eq(
                        "Survey-weighted Brier score"
                    )
                    .any()
                ),
        },

        {
            "Check":
                "Multiclass log loss frozen",

            "PASS":
                bool(
                    multiclass_metric_registry[
                        "Metric"
                    ]
                    .eq(
                        "Survey-weighted multiclass log loss"
                    )
                    .any()
                ),
        },

        {
            "Check":
                "Multiclass Brier convention explicit",

            "PASS":
                not brier_protocol[
                    "multiclass_scale_by_half"
                ],
        },

        {
            "Check":
                "Smooth calibration curve primary",

            "PASS":
                (
                    "smooth"
                    in
                    calibration_assessment_protocol[
                        "primary_graph"
                    ]
                    .lower()
                ),
        },

        {
            "Check":
                "Hosmer-Lemeshow excluded",

            "PASS":
                not calibration_assessment_protocol[
                    "hosmer_lemeshow_test"
                ],
        },

        {
            "Check":
                "No threshold selected",

            "PASS":
                not threshold_policy_boundary[
                    "threshold_selection_in_step17"
                ],
        },

        {
            "Check":
                "No model fitted in Step 17",

            "PASS":
                True,
        },

        {
            "Check":
                "No model performance inspected in Step 17",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 17 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 18. SAVE PROTOCOL OUTPUTS
# ============================================================

WINNER_FLOW_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_joint_winner_calibration_flow.json"
)


CALIBRATION_INPUT_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_input_policy.csv"
)


CALIBRATION_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_protocol.json"
)


OUTER_PREDICTION_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_outer_prediction_protocol.json"
)


FUSION_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_hierarchical_probability_fusion.json"
)


OUTER_EVALUATION_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_outer_oof_evaluation_protocol.json"
)


BINARY_METRIC_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_binary_metric_registry.csv"
)


MULTICLASS_METRIC_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_multiclass_metric_registry.csv"
)


BRIER_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_brier_score_convention.json"
)


CALIBRATION_ASSESSMENT_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_assessment_protocol.json"
)


THRESHOLD_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_threshold_boundary.json"
)


REFERENCE_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_reference_registry.csv"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_manuscript_calibration_evaluation_methods.txt"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP17_FINAL_hard_qc.csv"
)


calibration_input_policy.to_csv(
    CALIBRATION_INPUT_FILE,
    index=False,
)


binary_metric_registry.to_csv(
    BINARY_METRIC_FILE,
    index=False,
)


multiclass_metric_registry.to_csv(
    MULTICLASS_METRIC_FILE,
    index=False,
)


reference_registry.to_csv(
    REFERENCE_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


json_outputs = [
    (
        WINNER_FLOW_FILE,
        joint_winner_flow,
    ),
    (
        CALIBRATION_PROTOCOL_FILE,
        calibration_protocol,
    ),
    (
        OUTER_PREDICTION_FILE,
        outer_prediction_protocol,
    ),
    (
        FUSION_FILE,
        fusion_protocol,
    ),
    (
        OUTER_EVALUATION_FILE,
        outer_evaluation_protocol,
    ),
    (
        BRIER_FILE,
        brier_protocol,
    ),
    (
        CALIBRATION_ASSESSMENT_FILE,
        calibration_assessment_protocol,
    ),
    (
        THRESHOLD_FILE,
        threshold_policy_boundary,
    ),
]


for path, payload in json_outputs:

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
        )


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


# ============================================================
# 19. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 17",

    "status":
        "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "seed":
        SEED,

    "primary_master_sha256":
        sha256_file(PRIMARY_FILE),

    "outer_split_sha256":
        sha256_file(OUTER_SPLIT_FILE),

    "inner_split_sha256":
        sha256_file(INNER_SPLIT_FILE),

    "upstream_step16_status":
        step16_manifest.get("status"),

    "joint_model_selection":
        True,

    "winner_only_calibration":
        True,

    "stage_specific_winner_selection":
        True,

    "same_family_required_across_stages":
        False,

    "calibration_method":
        CALIBRATION_METHOD,

    "calibration_method_selection":
        False,

    "calibration_source":
        (
            "selected winner pooled inner-OOF scores"
        ),

    "calibration_weight":
        (
            "normalized original DHS SURVEY_WEIGHT"
        ),

    "outer_test_used_for_calibration":
        False,

    "hierarchical_fusion":
        True,

    "post_fusion_recalibration":
        False,

    "outer_metric_aggregation":
        "pooled outer-OOF predictions",

    "evaluation_weight":
        "original DHS SURVEY_WEIGHT",

    "binary_primary_discrimination":
        "survey-weighted Average Precision",

    "binary_secondary_discrimination":
        "survey-weighted ROC-AUC",

    "binary_primary_probability_metric":
        "survey-weighted log loss",

    "binary_secondary_probability_metric":
        "survey-weighted Brier score",

    "multiclass_primary_probability_metric":
        "survey-weighted multiclass log loss",

    "multiclass_secondary_probability_metric":
        "survey-weighted multiclass Brier score",

    "threshold_selection_performed":
        False,

    "model_fitting_performed":
        False,

    "performance_evaluated":
        False,

    "winner_flow_sha256":
        sha256_file(
            WINNER_FLOW_FILE
        ),

    "calibration_protocol_sha256":
        sha256_file(
            CALIBRATION_PROTOCOL_FILE
        ),

    "fusion_protocol_sha256":
        sha256_file(
            FUSION_FILE
        ),

    "outer_evaluation_protocol_sha256":
        sha256_file(
            OUTER_EVALUATION_FILE
        ),

    "binary_metric_registry_sha256":
        sha256_file(
            BINARY_METRIC_FILE
        ),

    "multiclass_metric_registry_sha256":
        sha256_file(
            MULTICLASS_METRIC_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),

    "methods_sha256":
        sha256_file(
            METHODS_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_evaluation_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 20. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 17."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 17."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 17."
    )


# ============================================================
# 21. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 17 — JOINT-WINNER CALIBRATION + OUTER-OOF EVALUATION FREEZE"
)

print(
    "=" * 78
)


print(
    "\nCALIBRATION INPUT POLICY:"
)

display(
    calibration_input_policy
)


print(
    "\nBINARY-STAGE METRIC REGISTRY:"
)

display(
    binary_metric_registry
)


print(
    "\nFINAL MULTICLASS METRIC REGISTRY:"
)

display(
    multiclass_metric_registry
)


print(
    "\nJOINT-WINNER RULE:"
)

print(
    "- Joint Step-16 selection occurs BEFORE calibration."
)

print(
    "- Only the single winning candidate is calibrated."
)

print(
    "- Stage 1 and Stage 2 winners are selected independently."
)

print(
    "- Different algorithms may therefore be used for the two stages."
)


print(
    "\nCALIBRATION RULE:"
)

print(
    "- Fixed weighted logistic/sigmoid recalibration."
)

print(
    "- No calibration-method competition."
)

print(
    "- Calibration uses selected-winner pooled inner-OOF scores only."
)

print(
    "- Probability models use logit(clipped raw probability)."
)

print(
    "- RBF-SVM uses raw decision_function margin."
)

print(
    "- Outer-test outcomes never enter calibrator fitting."
)


print(
    "\nOUTER EVALUATION:"
)

print(
    "- Refit the selected winner on complete outer-training data."
)

print(
    "- Apply the frozen calibrator to untouched outer-test scores."
)

print(
    "- Pool all five outer-test prediction sets for primary evaluation."
)

print(
    "- Use original DHS survey weights for performance metrics."
)

print(
    "- Stage-2 standalone metrics use true ANC users only."
)

print(
    "- Final three-class metrics use all 5,063 participants."
)


print(
    "\nFUSION:"
)

print(
    "p0 = q0"
)

print(
    "p1 = (1-q0)*(1-q2)"
)

print(
    "p2 = (1-q0)*q2"
)

print(
    "Synthetic probability-sum QC: PASS"
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)

print(
    methods_text
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)

print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)

print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 17 PASS"
)

print(
    "=" * 78
)


STEP 17 — JOINT-WINNER CALIBRATION + OUTER-OOF EVALUATION FREEZE

CALIBRATION INPUT POLICY:


,Selected_Winner_Type,Raw_Score,Calibration_Predictor_z,Calibrator
0,RBF SVM,decision_function,raw decision-function margin,weighted logistic sigmoid with intercept and s...
1,Probability-producing model,positive-class predict_proba,"logit(clip(p, 1e-6, 1-1e-6))",weighted logistic sigmoid with intercept and s...



BINARY-STAGE METRIC REGISTRY:


,Domain,Metric,Prediction_Input,Weight,Role
0,Discrimination,Survey-weighted Average Precision,Raw continuous winner score,Original SURVEY_WEIGHT,Primary discrimination
1,Discrimination,Survey-weighted ROC-AUC,Raw continuous winner score,Original SURVEY_WEIGHT,Secondary discrimination
2,Probability accuracy,Survey-weighted log loss,Calibrated probability,Original SURVEY_WEIGHT,Primary probability metric
3,Probability accuracy,Survey-weighted Brier score,Calibrated probability,Original SURVEY_WEIGHT,Secondary probability metric
4,Calibration,Calibration-in-the-large / intercept,Calibrated probability,Original SURVEY_WEIGHT,Report; ideal 0
5,Calibration,Calibration slope,Calibrated probability,Original SURVEY_WEIGHT,Report; ideal 1
6,Calibration,Observed-to-expected ratio,Calibrated probability,Original SURVEY_WEIGHT,Report; ideal 1
7,Calibration,Weighted smooth calibration curve,Calibrated probability,Original SURVEY_WEIGHT,Primary calibration graphic
8,Calibration,Prediction-quantile grouped points,Calibrated probability,Original SURVEY_WEIGHT,Supplementary display only; not primary calibr...



FINAL MULTICLASS METRIC REGISTRY:


,Domain,Metric,Role
0,Probability accuracy,Survey-weighted multiclass log loss,Primary final multiclass probability metric
1,Probability accuracy,Survey-weighted multiclass Brier score,Secondary final multiclass probability metric
2,Class-wise discrimination,One-vs-rest survey-weighted Average Precision,"Report separately for classes 0, 1, and 2"
3,Class-wise discrimination,One-vs-rest survey-weighted ROC-AUC,"Report separately for classes 0, 1, and 2"
4,Summary discrimination,Macro class-wise weighted AP,Secondary descriptive summary
5,Summary discrimination,Macro class-wise weighted ROC-AUC,Secondary descriptive summary
6,Class-wise calibration,"Calibration intercept, slope, O/E and smooth c...",Report separately by class



JOINT-WINNER RULE:
- Joint Step-16 selection occurs BEFORE calibration.
- Only the single winning candidate is calibrated.
- Stage 1 and Stage 2 winners are selected independently.
- Different algorithms may therefore be used for the two stages.

CALIBRATION RULE:
- Fixed weighted logistic/sigmoid recalibration.
- No calibration-method competition.
- Calibration uses selected-winner pooled inner-OOF scores only.
- Probability models use logit(clipped raw probability).
- RBF-SVM uses raw decision_function margin.
- Outer-test outcomes never enter calibrator fitting.

OUTER EVALUATION:
- Refit the selected winner on complete outer-training data.
- Apply the frozen calibrator to untouched outer-test scores.
- Pool all five outer-test prediction sets for primary evaluation.
- Use original DHS survey weights for performance metrics.
- Stage-2 standalone metrics use true ANC users only.
- Final three-class metrics use all 5,063 participants.

FUSION:
p0 = q0
p1 = (1-q0)*(1-q2)
p2 = (1-q0)*q

,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Revised joint Step 16 frozen,True
4,120 joint candidates upstream,True
5,Model selection precedes calibration,True
6,Only joint winner is calibrated,True
7,Stage 1 and Stage 2 selected independently,True
8,Same family not forced across stages,True
9,Calibration method fixed,True



MANUSCRIPT METHODS PREVIEW:
Probability calibration and outer cross-validation evaluation

For each hierarchical stage, outer cross-validation fold, and analysis
pipeline, model family and hyperparameters will first be selected
jointly using the prespecified inner-cross-validation procedure. Only
the single winning candidate will subsequently enter probability
calibration. Thus, calibration will occur after model-family and
hyperparameter selection and will not be used to choose between model
families.

For the selected candidate, raw predictions from the four frozen inner
validation folds will be pooled to create cross-fitted scores covering
the complete stage-specific outer-training population. A single
prespecified weighted logistic sigmoid calibrator will be fitted to
these scores. For probability-producing classifiers, the calibration
predictor will be the logit of the raw positive-class probability after
fixed numerical clipping at 10^-6 and 1-10^-6. For an RBF support-vector
ma

In [27]:
# ============================================================
# STEP 17A — FINAL FULL-PREDICTOR RETENTION AND
#            NO-EXTERNAL-FEATURE-SELECTION PROTOCOL FREEZE
# ============================================================
#
# PURPOSE
# -------
# Freeze the predictor policy BEFORE any candidate model is fitted.
#
# PRIMARY MODELLING RULE
# ----------------------
# ALL 15 prespecified ORIGINAL predictors are supplied to EVERY
# candidate model family.
#
# There is NO separate data-driven feature-selection pipeline.
#
# Therefore the primary workflow is:
#
#   15 frozen predictors
#          ↓
#   Step-14 model-specific preprocessing
#          ↓
#   all 120 Step-16 joint candidates
#          ↓
#   frozen inner-CV joint model + hyperparameter selection
#          ↓
#   selected winner
#          ↓
#   Step-17 calibration
#          ↓
#   untouched outer-test evaluation
#
#
# NOT USED AS EXTERNAL FEATURE SELECTION
# --------------------------------------
# - univariable significance screening
# - p-value filtering
# - forward selection
# - backward elimination
# - bidirectional stepwise selection
# - SelectKBest
# - RFE / RFECV
# - variance-threshold feature deletion
# - correlation-threshold feature deletion
# - VIF-driven automatic predictor deletion
# - LASSO/ElasticNet preprocessing before another model
# - Random-Forest-importance-based deletion
# - XGBoost-importance-based deletion
# - permutation-importance-based deletion
# - SHAP-based deletion
# - post-hoc deletion after model performance is observed
# - post-hoc addition after model performance is observed
#
#
# IMPORTANT DISTINCTION
# ---------------------
# Algorithm-INTERNAL regularization / variable usage is allowed.
#
# Examples:
# - ElasticNet may shrink encoded coefficients to zero.
# - Random Forest / ExtraTrees may never use some predictors.
# - Boosting algorithms may assign negligible importance.
#
# These are intrinsic properties of the fitted algorithm.
# They DO NOT change the frozen original predictor set.
#
#
# ORIGINAL PREDICTORS vs TRANSFORMED FEATURES
# -------------------------------------------
# Scientific predictor count = 15.
#
# OHE models can expand categorical predictors into multiple
# transformed columns. Those encoded columns are representations
# of the original predictors, NOT new independently selected
# source predictors.
#
#
# EXPLAINABILITY
# --------------
# Feature importance / SHAP / permutation importance will be
# performed AFTER predictive performance is locked.
#
# Explainability cannot feed back into primary model development.
#
#
# This step fits ZERO predictive models and inspects ZERO
# predictive performance.
# ============================================================


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. FROZEN INPUTS
# ============================================================

if "DIRS" not in globals():

    raise RuntimeError(
        "DIRS is unavailable. "
        "Run the accepted project setup first."
    )


PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


STEP14_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_preprocessing_manifest.json"
)


STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)


STEP17_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_evaluation_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP14_MANIFEST_FILE,
    STEP16_MANIFEST_FILE,
    STEP17_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing:\n{path}"
        )


# ============================================================
# 2. IMMUTABLE HASH CHECK
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


# ============================================================
# 3. UPSTREAM PROTOCOL CHECK
# ============================================================

step14_manifest = read_json(
    STEP14_MANIFEST_FILE
)


step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)


step17_manifest = read_json(
    STEP17_MANIFEST_FILE
)


if (
    step14_manifest.get("status")
    !=
    "PREPROCESSING_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 14 preprocessing protocol is not frozen."
    )


if (
    step16_manifest.get("status")
    !=
    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Revised joint-selection Step 16 is not frozen."
    )


if (
    step17_manifest.get("status")
    !=
    "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 17 is not frozen."
    )


if (
    step16_manifest.get(
        "total_joint_candidates"
    )
    !=
    120
):

    raise RuntimeError(
        "Step 16 does not contain the frozen "
        "120-candidate joint registry."
    )


# ============================================================
# 4. EXACT FROZEN ORIGINAL PREDICTORS
# ============================================================

NUMERIC_PREDICTORS = [

    "maternal_age_at_index_birth_years",

    "prior_live_births",

    "household_size_members",
]


ORDERED_CATEGORICAL_PREDICTORS = [

    "maternal_education",

    "wealth_quintile",
]


NOMINAL_CATEGORICAL_PREDICTORS = [

    "partner_education_primary_final",

    "division",

    "pregnancy_intention",

    "healthcare_decision_primary_final",
]


BINARY_CATEGORICAL_PREDICTORS = [

    "residence",

    "weekly_media_exposure",

    "working_status",

    "religion_final",

    "household_head_sex",

    "distance_barrier",
]


CATEGORICAL_PREDICTORS = (

    ORDERED_CATEGORICAL_PREDICTORS
    +
    NOMINAL_CATEGORICAL_PREDICTORS
    +
    BINARY_CATEGORICAL_PREDICTORS
)


PREDICTORS = (

    NUMERIC_PREDICTORS
    +
    CATEGORICAL_PREDICTORS
)


EXPECTED_PREDICTOR_COUNT = 15


if len(PREDICTORS) != EXPECTED_PREDICTOR_COUNT:

    raise RuntimeError(
        f"Expected {EXPECTED_PREDICTOR_COUNT} original predictors; "
        f"observed {len(PREDICTORS)}."
    )


if len(set(PREDICTORS)) != EXPECTED_PREDICTOR_COUNT:

    raise RuntimeError(
        "Duplicate predictor detected."
    )


# ============================================================
# 5. PRIMARY MASTER QC
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


if master.shape != (5063, 23):

    raise RuntimeError(
        f"Unexpected primary-master shape: {master.shape}"
    )


missing_predictors = [

    predictor

    for predictor in PREDICTORS

    if predictor not in master.columns
]


if missing_predictors:

    raise RuntimeError(
        "Frozen predictor(s) missing from primary master:\n"
        f"{missing_predictors}"
    )


# ============================================================
# 6. FORBIDDEN-PREDICTOR CHECK
# ============================================================

FORBIDDEN_PREDICTORS = {

    # identifiers
    "_ROW_ID",
    "CASEID",

    # outcome
    "ANC_VISITS",
    "ANC_CLASS",

    # survey-design variables
    "SURVEY_WEIGHT",
    "PSU",
    "STRATUM",

    # raw survey-design sources
    "V005",
    "V021",
    "V023",

    # ANC / pregnancy outcome-construction sources
    "M14",
    "M80",

    # index-pregnancy / birth linkage helpers
    "MIDX",
    "BIDX",
    "B3",
}


forbidden_overlap = (

    set(PREDICTORS)
    &
    FORBIDDEN_PREDICTORS
)


if forbidden_overlap:

    raise RuntimeError(
        "Forbidden variable entered predictor set:\n"
        f"{sorted(forbidden_overlap)}"
    )


# ============================================================
# 7. MISSING / FINITE QC
# ============================================================

predictor_missing_counts = (

    master[
        PREDICTORS
    ]
    .isna()
    .sum()
)


if (
    predictor_missing_counts
    >
    0
).any():

    raise RuntimeError(
        "Unexpected machine-missing predictor values:\n"
        f"{predictor_missing_counts[predictor_missing_counts > 0]}"
    )


for predictor in NUMERIC_PREDICTORS:

    values = pd.to_numeric(
        master[
            predictor
        ],
        errors="raise",
    ).to_numpy(
        dtype=float
    )


    if not np.isfinite(values).all():

        raise RuntimeError(
            f"Non-finite values detected in {predictor}."
        )


# ============================================================
# 8. ORIGINAL-PREDICTOR REGISTRY
# ============================================================

predictor_type_map = {}


for predictor in NUMERIC_PREDICTORS:

    predictor_type_map[
        predictor
    ] = "Numeric"


for predictor in ORDERED_CATEGORICAL_PREDICTORS:

    predictor_type_map[
        predictor
    ] = "Ordered categorical"


for predictor in NOMINAL_CATEGORICAL_PREDICTORS:

    predictor_type_map[
        predictor
    ] = "Nominal categorical"


for predictor in BINARY_CATEGORICAL_PREDICTORS:

    predictor_type_map[
        predictor
    ] = "Binary categorical"


predictor_registry = pd.DataFrame(
    [
        {
            "Predictor":
                predictor,

            "Predictor_Type":
                predictor_type_map[
                    predictor
                ],

            "Primary_Status":
                "Retained",

            "Available_to_Stage1":
                True,

            "Available_to_Stage2":
                True,

            "Available_to_All_ML_Families":
                True,

            "External_Feature_Selection":
                "None",

            "Preprocessing":
                "Step-14 model-specific",
        }

        for predictor in PREDICTORS
    ]
)


# ============================================================
# 9. PRIMARY FULL-PREDICTOR POLICY
# ============================================================

full_predictor_protocol = {

    "scientific_original_predictor_count":
        15,

    "original_predictors":
        PREDICTORS,

    "primary_pipeline":
        "full prespecified predictor set",

    "full_predictor_set_used_for_all_candidate_models":
        True,

    "full_predictor_set_used_for_stage1":
        True,

    "full_predictor_set_used_for_stage2":
        True,

    "full_predictor_set_used_for_primary_weighted_pipeline":
        True,

    "full_predictor_set_used_for_unweighted_sensitivity":
        True,

    "same_original_predictors_across_model_families":
        True,

    "same_original_predictors_across_stages":
        True,

    "predictor_set_selected_using_model_performance":
        False,

    "predictor_set_selected_using_outer_test":
        False,

    "reduced_feature_primary_pipeline":
        False,

    "feature_selection_as_joint_candidate_dimension":
        False,

    "feature_selection_sensitivity_planned":
        False,

    "reason_no_external_feature_selection":
        (
            "Only 15 original predictors were prespecified before "
            "model fitting from substantive and literature-based "
            "considerations. A separate data-driven feature-selection "
            "pipeline is not required for dimensionality reduction and "
            "would add another model-selection layer."
        ),
}


# ============================================================
# 10. PROHIBITED EXTERNAL FEATURE-SELECTION METHODS
# ============================================================

external_feature_selection_policy = {

    "univariable_significance_screening":
        False,

    "p_value_filtering":
        False,

    "forward_selection":
        False,

    "backward_elimination":
        False,

    "bidirectional_stepwise":
        False,

    "select_k_best":
        False,

    "rfe":
        False,

    "rfecv":
        False,

    "variance_threshold_deletion":
        False,

    "correlation_threshold_deletion":
        False,

    "vif_based_automatic_deletion":
        False,

    "lasso_as_preprocessing_for_other_models":
        False,

    "elasticnet_as_preprocessing_for_other_models":
        False,

    "random_forest_importance_deletion":
        False,

    "boosting_importance_deletion":
        False,

    "permutation_importance_deletion":
        False,

    "shap_importance_deletion":
        False,

    "outer_performance_guided_deletion":
        False,

    "outer_performance_guided_addition":
        False,

    "posthoc_predictor_set_revision":
        False,
}


# ============================================================
# 11. ALLOWED MODEL-INTERNAL BEHAVIOUR
# ============================================================

internal_model_policy = pd.DataFrame(
    [
        {
            "Model":
                "ElasticNet logistic regression",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Elastic-net regularization may shrink "
                    "individual encoded coefficients toward "
                    "or exactly to zero."
                ),
        },

        {
            "Model":
                "RBF SVM",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Uses the complete transformed feature "
                    "representation."
                ),
        },

        {
            "Model":
                "Random Forest",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Individual trees choose split variables "
                    "internally."
                ),
        },

        {
            "Model":
                "ExtraTrees",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Individual trees choose split variables "
                    "internally."
                ),
        },

        {
            "Model":
                "AdaBoost",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Weak learners determine split-variable "
                    "usage internally."
                ),
        },

        {
            "Model":
                "GradientBoosting",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Boosted trees determine split-variable "
                    "usage internally."
                ),
        },

        {
            "Model":
                "HistGradientBoosting",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Boosted trees determine predictor usage "
                    "internally."
                ),
        },

        {
            "Model":
                "XGBoost",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Tree splitting and regularization determine "
                    "predictor usage internally."
                ),
        },

        {
            "Model":
                "LightGBM",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Native boosted-tree splitting determines "
                    "predictor usage internally."
                ),
        },

        {
            "Model":
                "CatBoost",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "Native categorical boosting determines "
                    "predictor usage internally."
                ),
        },

        {
            "Model":
                "Survey-weighted logistic comparator",

            "Original_Predictors_Entered":
                15,

            "External_Selection":
                "None",

            "Allowed_Internal_Behaviour":
                (
                    "All prespecified original predictors are "
                    "retained; no significance-based deletion."
                ),
        },
    ]
)


# ============================================================
# 12. TRANSFORMED-FEATURE INTERPRETATION
# ============================================================

transformed_feature_protocol = {

    "scientific_predictor_count":
        15,

    "one_hot_encoding_creates_transformed_columns":
        True,

    "encoded_columns_considered_new_original_predictors":
        False,

    "encoded_columns_independently_screened":
        False,

    "individual_dummy_level_deletion_based_on_outcome":
        False,

    "ordered_predictors_integer_scored":
        False,

    "ordered_predictors_remain_categorical":
        True,

    "model_specific_preprocessing_source":
        "frozen Step 14",

    "interpretation":
        (
            "Encoded columns are computational representations of "
            "the 15 original predictors. Predictor-level scientific "
            "interpretation and later permutation-importance analysis "
            "will therefore aggregate or operate at the original "
            "predictor level where appropriate."
        ),
}


# ============================================================
# 13. EXPLAINABILITY BOUNDARY
# ============================================================

explainability_protocol = {

    "timing":
        (
            "after nested model development and primary "
            "performance predictions are locked"
        ),

    "planned_analyses": [

        (
            "held-out / cross-fitted permutation importance "
            "at original-predictor level"
        ),

        (
            "model-appropriate SHAP analysis where technically "
            "valid and scientifically useful"
        ),

        (
            "importance stability across outer folds"
        ),

        (
            "rank stability across outer folds"
        ),

        (
            "Stage-1 versus Stage-2 importance comparison"
        ),
    ],

    "permutation_importance_can_remove_predictor":
        False,

    "shap_can_remove_predictor":
        False,

    "importance_can_trigger_model_refit":
        False,

    "importance_can_trigger_hyperparameter_reselection":
        False,

    "importance_can_change_primary_predictor_set":
        False,

    "role":
        (
            "post-development interpretation only"
        ),
}


# ============================================================
# 14. RELATION TO JOINT STEP-16 SELECTION
# ============================================================

joint_selection_boundary = {

    "joint_candidate_count":
        120,

    "candidate_dimensions":
        [
            "model family",
            "model-specific preprocessing",
            "hyperparameter configuration",
        ],

    "feature_subset_is_candidate_dimension":
        False,

    "feature_selection_method_is_candidate_dimension":
        False,

    "every_candidate_receives_all_15_original_predictors":
        True,

    "model_selection_can_choose_feature_subset":
        False,

    "reason":
        (
            "The Step-16 competition selects algorithm and "
            "hyperparameters, not a data-driven subset of the "
            "prespecified original predictor set."
        ),
}


# ============================================================
# 15. REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "EFTHIMIOU_2024",

            "Reference":
                (
                    "Efthimiou O, Seo M, Chalkou K, Debray T, "
                    "Egger M, Salanti G, et al. Developing "
                    "clinical prediction models: a step-by-step "
                    "guide. BMJ. 2024;386:e078276."
                ),

            "DOI":
                "10.1136/bmj-2023-078276",

            "Use":
                (
                    "Candidate;386:e078276."
                ),

            "DOI":
                "10.1136/bmj-2023-078276",

            "Use":
                (
                    "Candidate-predictor prespecification and "
                    "avoidance of univariable and conventional "
                    "stepwise predictor selection."
                ),
        },

        {
            "ID":
                "PROBAST_AI",

            "Reference":
                (
                    "Moons KGM, Damen JAA, Kaul T, Hooft L, "
                    "Andaur Navarro C, Dhiman P, et al. "
                    "PROBAST+AI: an updated quality, risk of bias, "
                    "and applicability assessment tool for "
                    "prediction models using regression or "
                    "artificial intelligence methods. "
                    "BMJ. 2025;388:e082505."
                ),

            "DOI":
                "10.1136/bmj-2024-082505",

            "Use":
                (
                    "Predictor-domain and analysis-domain "
                    "quality and risk-of-bias safeguards."
                ),
        },

        {
            "ID":
                "TRIPOD_AI",

            "Reference":
                (
                    "TRIPOD+AI statement. "
                    "BMJ. 2024;385:e078378."
                ),

            "DOI":
                "10.1136/bmj-2023-078378",

            "Use":
                (
                    "Transparent reporting of predictor handling "
                    "and model-development procedures."
                ),
        },
    ]
)


# ============================================================
# 16. MANUSCRIPT-READY METHODS TEXT
# ============================================================

methods_text = """
Predictor retention and feature-selection strategy

The primary modelling set contained 15 original predictors that were
prespecified before model fitting on the basis of substantive
relevance, prior literature, data availability, and the predefined
study framework. The complete 15-predictor set was supplied to every
candidate model family and to both hierarchical prediction stages.
Model-family and hyperparameter selection therefore occurred while the
original predictor set remained fixed.

No separate data-driven feature-selection procedure was applied.
Predictors were not screened using univariable associations or P
values, and forward, backward, or bidirectional stepwise selection,
SelectKBest, recursive feature elimination, variance-threshold
selection, correlation-threshold selection, or importance-based
predictor deletion were not used. Feature-subset selection was likewise
not included as an additional dimension of the nested model-selection
procedure. Given the relatively compact set of 15 prespecified
predictors, this avoided adding an unnecessary data-driven selection
layer.

Model-specific preprocessing could expand categorical predictors into
multiple encoded columns. These columns were regarded as computational
representations of the original predictors rather than as additional
independently selected predictors. Ordered predictors were retained as
categorical variables according to the frozen preprocessing protocol.

Algorithm-internal regularization and split-variable selection were
allowed as intrinsic properties of the candidate models. For example,
elastic-net regression could shrink individual encoded coefficients
toward or to zero, while tree-based algorithms could make little or no
use of particular predictors. Such behaviour did not alter the frozen
15-predictor input set and was not considered an external
feature-selection procedure.

Post-development permutation importance, SHAP-based interpretation
where appropriate, and importance-stability analyses will be used for
model interpretation only after predictive performance is locked.
These analyses will not be used to remove or add predictors, refit the
primary prediction pipeline, or repeat model selection.
""".strip()


# ============================================================
# 17. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Step 14 frozen",

            "PASS":
                (
                    step14_manifest.get("status")
                    ==
                    "PREPROCESSING_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Revised joint Step 16 frozen",

            "PASS":
                (
                    step16_manifest.get("status")
                    ==
                    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Step 17 frozen",

            "PASS":
                (
                    step17_manifest.get("status")
                    ==
                    "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Exactly 15 original predictors",

            "PASS":
                len(PREDICTORS)
                ==
                15,
        },

        {
            "Check":
                "Predictor names unique",

            "PASS":
                len(set(PREDICTORS))
                ==
                15,
        },

        {
            "Check":
                "All predictors present in primary master",

            "PASS":
                len(missing_predictors)
                ==
                0,
        },

        {
            "Check":
                "No forbidden variable in predictor set",

            "PASS":
                len(forbidden_overlap)
                ==
                0,
        },

        {
            "Check":
                "No unresolved machine-missing predictor values",

            "PASS":
                bool(
                    (
                        predictor_missing_counts
                        ==
                        0
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All numeric predictors finite",

            "PASS":
                bool(
                    all(
                        np.isfinite(
                            pd.to_numeric(
                                master[predictor],
                                errors="raise",
                            ).to_numpy(dtype=float)
                        ).all()

                        for predictor
                        in NUMERIC_PREDICTORS
                    )
                ),
        },

        {
            "Check":
                "All 15 predictors retained",

            "PASS":
                bool(
                    predictor_registry[
                        "Primary_Status"
                    ]
                    .eq("Retained")
                    .all()
                ),
        },

        {
            "Check":
                "Full predictor set supplied to every ML family",

            "PASS":
                full_predictor_protocol[
                    "full_predictor_set_used_for_all_candidate_models"
                ],
        },

        {
            "Check":
                "Same predictors used for Stage 1 and Stage 2",

            "PASS":
                full_predictor_protocol[
                    "same_original_predictors_across_stages"
                ],
        },

        {
            "Check":
                "No reduced-feature primary pipeline",

            "PASS":
                not full_predictor_protocol[
                    "reduced_feature_primary_pipeline"
                ],
        },

        {
            "Check":
                "Feature subset not part of Step-16 competition",

            "PASS":
                not joint_selection_boundary[
                    "feature_subset_is_candidate_dimension"
                ],
        },

        {
            "Check":
                "Feature-selection method not part of Step-16 competition",

            "PASS":
                not joint_selection_boundary[
                    "feature_selection_method_is_candidate_dimension"
                ],
        },

        {
            "Check":
                "No univariable significance screening",

            "PASS":
                not external_feature_selection_policy[
                    "univariable_significance_screening"
                ],
        },

        {
            "Check":
                "No p-value filtering",

            "PASS":
                not external_feature_selection_policy[
                    "p_value_filtering"
                ],
        },

        {
            "Check":
                "No stepwise predictor selection",

            "PASS":
                (
                    not external_feature_selection_policy[
                        "forward_selection"
                    ]
                    and
                    not external_feature_selection_policy[
                        "backward_elimination"
                    ]
                    and
                    not external_feature_selection_policy[
                        "bidirectional_stepwise"
                    ]
                ),
        },

        {
            "Check":
                "No SelectKBest / RFE / RFECV",

            "PASS":
                (
                    not external_feature_selection_policy[
                        "select_k_best"
                    ]
                    and
                    not external_feature_selection_policy[
                        "rfe"
                    ]
                    and
                    not external_feature_selection_policy[
                        "rfecv"
                    ]
                ),
        },

        {
            "Check":
                "No automated variance/correlation/VIF deletion",

            "PASS":
                (
                    not external_feature_selection_policy[
                        "variance_threshold_deletion"
                    ]
                    and
                    not external_feature_selection_policy[
                        "correlation_threshold_deletion"
                    ]
                    and
                    not external_feature_selection_policy[
                        "vif_based_automatic_deletion"
                    ]
                ),
        },

        {
            "Check":
                "No model-importance-driven deletion",

            "PASS":
                (
                    not external_feature_selection_policy[
                        "random_forest_importance_deletion"
                    ]
                    and
                    not external_feature_selection_policy[
                        "boosting_importance_deletion"
                    ]
                    and
                    not external_feature_selection_policy[
                        "permutation_importance_deletion"
                    ]
                    and
                    not external_feature_selection_policy[
                        "shap_importance_deletion"
                    ]
                ),
        },

        {
            "Check":
                "No post-hoc predictor-set revision",

            "PASS":
                not external_feature_selection_policy[
                    "posthoc_predictor_set_revision"
                ],
        },

        {
            "Check":
                "OHE columns not treated as new source predictors",

            "PASS":
                not transformed_feature_protocol[
                    "encoded_columns_considered_new_original_predictors"
                ],
        },

        {
            "Check":
                "Ordered predictors remain categorical",

            "PASS":
                transformed_feature_protocol[
                    "ordered_predictors_remain_categorical"
                ],
        },

        {
            "Check":
                "Algorithm-internal regularization remains allowed",

            "PASS":
                True,
        },

        {
            "Check":
                "Explainability cannot modify primary predictor set",

            "PASS":
                (
                    not explainability_protocol[
                        "importance_can_change_primary_predictor_set"
                    ]
                ),
        },

        {
            "Check":
                "Explainability cannot trigger primary refit",

            "PASS":
                (
                    not explainability_protocol[
                        "importance_can_trigger_model_refit"
                    ]
                ),
        },

        {
            "Check":
                "No model fitted in Step 17A",

            "PASS":
                True,
        },

        {
            "Check":
                "No model performance inspected in Step 17A",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 17A HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 18. SAVE PROTOCOL OUTPUTS
# ============================================================

PREDICTOR_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_predictor_registry.csv"
)


FULL_PREDICTOR_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_full_predictor_protocol.json"
)


EXTERNAL_FS_POLICY_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_external_feature_selection_policy.json"
)


INTERNAL_MODEL_POLICY_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_model_internal_feature_usage_policy.csv"
)


TRANSFORMED_FEATURE_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_transformed_feature_interpretation.json"
)


EXPLAINABILITY_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_explainability_boundary.json"
)


JOINT_SELECTION_BOUNDARY_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_joint_selection_feature_boundary.json"
)


REFERENCE_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_reference_registry.csv"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_manuscript_predictor_retention_methods.txt"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP17A_FINAL_hard_qc.csv"
)


predictor_registry.to_csv(
    PREDICTOR_REGISTRY_FILE,
    index=False,
)


internal_model_policy.to_csv(
    INTERNAL_MODEL_POLICY_FILE,
    index=False,
)


reference_registry.to_csv(
    REFERENCE_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


for path, payload in [
    (
        FULL_PREDICTOR_PROTOCOL_FILE,
        full_predictor_protocol,
    ),
    (
        EXTERNAL_FS_POLICY_FILE,
        external_feature_selection_policy,
    ),
    (
        TRANSFORMED_FEATURE_FILE,
        transformed_feature_protocol,
    ),
    (
        EXPLAINABILITY_FILE,
        explainability_protocol,
    ),
    (
        JOINT_SELECTION_BOUNDARY_FILE,
        joint_selection_boundary,
    ),
]:

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
        )


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


# ============================================================
# 19. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 17A",

    "status":
        "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "primary_master_sha256":
        sha256_file(PRIMARY_FILE),

    "outer_split_sha256":
        sha256_file(OUTER_SPLIT_FILE),

    "inner_split_sha256":
        sha256_file(INNER_SPLIT_FILE),

    "original_predictor_count":
        15,

    "full_predictor_primary_pipeline":
        True,

    "all_original_predictors_retained":
        True,

    "same_predictors_all_models":
        True,

    "same_predictors_stage1_stage2":
        True,

    "external_feature_selection":
        False,

    "feature_subset_joint_selection_dimension":
        False,

    "univariable_screening":
        False,

    "p_value_filtering":
        False,

    "stepwise_selection":
        False,

    "select_k_best":
        False,

    "rfe_rfecv":
        False,

    "importance_based_deletion":
        False,

    "performance_guided_predictor_revision":
        False,

    "algorithm_internal_regularization_allowed":
        True,

    "algorithm_internal_split_selection_allowed":
        True,

    "explainability_can_modify_model":
        False,

    "model_fitting_performed":
        False,

    "performance_evaluated":
        False,

    "predictor_registry_sha256":
        sha256_file(
            PREDICTOR_REGISTRY_FILE
        ),

    "full_predictor_protocol_sha256":
        sha256_file(
            FULL_PREDICTOR_PROTOCOL_FILE
        ),

    "external_feature_selection_policy_sha256":
        sha256_file(
            EXTERNAL_FS_POLICY_FILE
        ),

    "internal_model_policy_sha256":
        sha256_file(
            INTERNAL_MODEL_POLICY_FILE
        ),

    "transformed_feature_protocol_sha256":
        sha256_file(
            TRANSFORMED_FEATURE_FILE
        ),

    "explainability_boundary_sha256":
        sha256_file(
            EXPLAINABILITY_FILE
        ),

    "joint_selection_boundary_sha256":
        sha256_file(
            JOINT_SELECTION_BOUNDARY_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),

    "methods_sha256":
        sha256_file(
            METHODS_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_predictor_feature_selection_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 20. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 17A."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 17A."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 17A."
    )


# ============================================================
# 21. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 17A — FULL-PREDICTOR + NO-EXTERNAL-FEATURE-SELECTION FREEZE"
)


print(
    "=" * 78
)


print(
    "\nFROZEN ORIGINAL PREDICTORS:"
)


display(
    predictor_registry
)


print(
    "\nMODEL-INTERNAL FEATURE-USAGE POLICY:"
)


display(
    internal_model_policy
)


print(
    "\nPRIMARY PIPELINE:"
)


print(
    "15 original predictors"
)


print(
    "-> model-specific Step-14 preprocessing"
)


print(
    "-> all 120 Step-16 joint model/hyperparameter candidates"
)


print(
    "-> joint inner-CV winner"
)


print(
    "-> Step-17 calibration"
)


print(
    "-> untouched outer-test evaluation"
)


print(
    "\nFEATURE-SELECTION POLICY:"
)


print(
    "- Full 15-predictor set is retained."
)


print(
    "- No reduced-feature primary pipeline."
)


print(
    "- No univariable or p-value screening."
)


print(
    "- No forward/backward/stepwise selection."
)


print(
    "- No SelectKBest, RFE, or RFECV."
)


print(
    "- No variance/correlation/VIF-based automatic deletion."
)


print(
    "- No RF/XGBoost/SHAP/permutation-importance-based deletion."
)


print(
    "- Feature subset is NOT a Step-16 joint-selection dimension."
)


print(
    "- Model performance cannot change the frozen predictor set."
)


print(
    "\nIMPORTANT DISTINCTION:"
)


print(
    "- ElasticNet may internally shrink encoded coefficients to zero."
)


print(
    "- Tree/boosting models may internally ignore some predictors."
)


print(
    "- These are intrinsic model properties, not external feature selection."
)


print(
    "- OHE-expanded columns remain representations of the original 15 predictors."
)


print(
    "\nEXPLAINABILITY:"
)


print(
    "- Permutation importance / SHAP occur after performance is locked."
)


print(
    "- Explainability cannot trigger predictor deletion or model refitting."
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)


print(
    methods_text
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)


print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)


print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    "=" * 78
)


print(
    "STATUS: STEP 17A PASS"
)


print(
    "=" * 78
)


STEP 17A — FULL-PREDICTOR + NO-EXTERNAL-FEATURE-SELECTION FREEZE

FROZEN ORIGINAL PREDICTORS:


,Predictor,Predictor_Type,Primary_Status,Available_to_Stage1,Available_to_Stage2,Available_to_All_ML_Families,External_Feature_Selection,Preprocessing
0,maternal_age_at_index_birth_years,Numeric,Retained,True,True,True,None,Step-14 model-specific
1,prior_live_births,Numeric,Retained,True,True,True,None,Step-14 model-specific
2,household_size_members,Numeric,Retained,True,True,True,None,Step-14 model-specific
3,maternal_education,Ordered categorical,Retained,True,True,True,None,Step-14 model-specific
4,wealth_quintile,Ordered categorical,Retained,True,True,True,None,Step-14 model-specific
5,partner_education_primary_final,Nominal categorical,Retained,True,True,True,None,Step-14 model-specific
6,division,Nominal categorical,Retained,True,True,True,None,Step-14 model-specific
7,pregnancy_intention,Nominal categorical,Retained,True,True,True,None,Step-14 model-specific
8,healthcare_decision_primary_final,Nominal categorical,Retained,True,True,True,None,Step-14 model-specific
9,residence,Binary categorical,Retained,True,True,True,None,Step-14 model-specific



MODEL-INTERNAL FEATURE-USAGE POLICY:


,Model,Original_Predictors_Entered,External_Selection,Allowed_Internal_Behaviour
0,ElasticNet logistic regression,15,None,Elastic-net regularization may shrink individu...
1,RBF SVM,15,None,Uses the complete transformed feature represen...
2,Random Forest,15,None,Individual trees choose split variables intern...
3,ExtraTrees,15,None,Individual trees choose split variables intern...
4,AdaBoost,15,None,Weak learners determine split-variable usage i...
5,GradientBoosting,15,None,Boosted trees determine split-variable usage i...
6,HistGradientBoosting,15,None,Boosted trees determine predictor usage intern...
7,XGBoost,15,None,Tree splitting and regularization determine pr...
8,LightGBM,15,None,Native boosted-tree splitting determines predi...
9,CatBoost,15,None,Native categorical boosting determines predict...



PRIMARY PIPELINE:
15 original predictors
-> model-specific Step-14 preprocessing
-> all 120 Step-16 joint model/hyperparameter candidates
-> joint inner-CV winner
-> Step-17 calibration
-> untouched outer-test evaluation

FEATURE-SELECTION POLICY:
- Full 15-predictor set is retained.
- No reduced-feature primary pipeline.
- No univariable or p-value screening.
- No forward/backward/stepwise selection.
- No SelectKBest, RFE, or RFECV.
- No variance/correlation/VIF-based automatic deletion.
- No RF/XGBoost/SHAP/permutation-importance-based deletion.
- Feature subset is NOT a Step-16 joint-selection dimension.
- Model performance cannot change the frozen predictor set.

IMPORTANT DISTINCTION:
- ElasticNet may internally shrink encoded coefficients to zero.
- Tree/boosting models may internally ignore some predictors.
- These are intrinsic model properties, not external feature selection.
- OHE-expanded columns remain representations of the original 15 predictors.

EXPLAINABILITY:
- Permu

,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Step 14 frozen,True
4,Revised joint Step 16 frozen,True
5,Step 17 frozen,True
6,Exactly 15 original predictors,True
7,Predictor names unique,True
8,All predictors present in primary master,True
9,No forbidden variable in predictor set,True



MANUSCRIPT METHODS PREVIEW:
Predictor retention and feature-selection strategy

The primary modelling set contained 15 original predictors that were
prespecified before model fitting on the basis of substantive
relevance, prior literature, data availability, and the predefined
study framework. The complete 15-predictor set was supplied to every
candidate model family and to both hierarchical prediction stages.
Model-family and hyperparameter selection therefore occurred while the
original predictor set remained fixed.

No separate data-driven feature-selection procedure was applied.
Predictors were not screened using univariable associations or P
values, and forward, backward, or bidirectional stepwise selection,
SelectKBest, recursive feature elimination, variance-threshold
selection, correlation-threshold selection, or importance-based
predictor deletion were not used. Feature-subset selection was likewise
not included as an additional dimension of the nested model-selection
procedu

In [ ]:
# ============================================================
# STEP 18 — FINAL CROSS-VALIDATED THRESHOLD OPTIMIZATION
#           AND SECONDARY CLASSIFICATION PROTOCOL FREEZE
# ============================================================
#
# IMPORTANT
# ---------
# THRESHOLD VALUES ARE NOT FIXED.
#
# This step freezes ONLY the threshold-optimization PROCEDURE.
#
# For every:
#
#   analysis pipeline
#       x outer fold
#       x hierarchical stage
#
# a NEW threshold will later be optimized from that outer
# training population's calibrated INNER-OOF probabilities.
#
#
# DEVELOPMENT SEQUENCE
# --------------------
#
# Step-16 joint model + hyperparameter selection
#                     ↓
# selected winner
#                     ↓
# Step-17 probability calibration
#                     ↓
# calibrated INNER-OOF probabilities
#                     ↓
# THRESHOLD OPTIMIZATION
#                     ↓
# selected t1 / t2 for that outer fold
#                     ↓
# untouched OUTER-TEST classification
#
#
# THEREFORE:
#
# - no universal fixed threshold
# - no default 0.5 threshold
# - no threshold selected from outer-test outcomes
# - no threshold selected from pooled outer-OOF outcomes
# - Stage 1 and Stage 2 thresholds optimized independently
# - each outer fold receives independently optimized thresholds
# - primary and unweighted-sensitivity pipelines are optimized
#   independently
#
#
# OPTIMIZATION OBJECTIVE
# ----------------------
#
# Survey-weighted balanced accuracy:
#
#   BA = (Sensitivity + Specificity) / 2
#
# This gives equal emphasis to the two classes and is equivalent
# in threshold ranking to maximizing the Youden index:
#
#   J = Sensitivity + Specificity - 1
#
#
# WHY NOT OPTIMIZE F1 / RAW ACCURACY?
# -----------------------------------
#
# Accuracy is strongly influenced by outcome prevalence.
# F1 ignores true negatives and imposes a different implicit
# preference structure.
#
# No application-specific false-positive / false-negative cost
# ratio has been prespecified.
#
# Balanced accuracy is therefore used only as a reproducible
# SECONDARY statistical operating-point criterion.
#
#
# THRESHOLD CANDIDATES
# --------------------
#
# We evaluate ALL classification-changing operating points
# implied by the calibrated inner-OOF probabilities.
#
# This is NOT:
#   - a random threshold search
#   - a coarse 0.01 grid
#   - a fixed 100-threshold grid
#
# Candidate thresholds are:
#   - 0
#   - midpoint between every adjacent distinct calibrated score
#   - 1
#
# Calibrated probabilities are clipped only for threshold-search
# numerical stability to [1e-12, 1-1e-12].
#
# This clipping does NOT alter the primary log-loss, Brier,
# calibration, AP, or AUROC analyses.
#
#
# TIE BREAKING
# ------------
#
# 1. Highest weighted balanced accuracy
# 2. Highest min(sensitivity, specificity)
# 3. Threshold closest to 0.5
# 4. Lower numerical threshold
#
# 0.5 IS NOT the selected default threshold.
# It appears only as a deterministic tie-break reference after
# optimization metrics are already tied.
#
#
# HIERARCHICAL CLASSIFICATION
# ---------------------------
#
# Stage 1:
#   positive = No ANC
#
#   if q0 >= optimized t1:
#       predict No ANC
#
#
# Stage 2:
#   positive = >=8 ANC visits
#
#   Stage-2 threshold is optimized among TRUE ANC users in the
#   current development population.
#
#
# Final class:
#
#   if q0 >= t1:
#       class 0 = No ANC
#
#   else if q2 >= t2:
#       class 2 = >=8 visits
#
#   else:
#       class 1 = 1-7 visits
#
#
# PRIMARY PERFORMANCE REMAINS PROBABILITY BASED
# ---------------------------------------------
#
# Threshold-dependent metrics remain secondary.
#
# Primary:
#   - weighted Average Precision
#   - weighted ROC-AUC
#   - weighted log loss
#   - weighted Brier score
#   - calibration assessment
#
#
# No predictive model is fitted and no study performance is
# inspected in this protocol-freeze step.
# ============================================================


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. FROZEN INPUTS
# ============================================================

if "DIRS" not in globals():

    raise RuntimeError(
        "DIRS is unavailable. "
        "Run the accepted project setup first."
    )


PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)


STEP17_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_evaluation_manifest.json"
)


STEP17A_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_predictor_feature_selection_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP16_MANIFEST_FILE,
    STEP17_MANIFEST_FILE,
    STEP17A_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing:\n{path}"
        )


# ============================================================
# 2. HASH IMMUTABILITY
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if sha256_file(
    INNER_SPLIT_FILE
) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


# ============================================================
# 3. UPSTREAM STATUS
# ============================================================

step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)


step17_manifest = read_json(
    STEP17_MANIFEST_FILE
)


step17a_manifest = read_json(
    STEP17A_MANIFEST_FILE
)


if (
    step16_manifest.get("status")
    !=
    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Revised joint Step 16 is not frozen."
    )


if (
    step17_manifest.get("status")
    !=
    "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 17 is not frozen."
    )


if (
    step17a_manifest.get("status")
    !=
    "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Corrected Step 17A is not frozen."
    )


if (
    step17a_manifest.get(
        "original_predictor_count"
    )
    !=
    15
):

    raise RuntimeError(
        "Step 17A predictor count is not 15."
    )


# ============================================================
# 4. THRESHOLD-OPTIMIZATION CONSTANTS
# ============================================================

THRESHOLD_OPTIMIZATION = True


FIXED_THRESHOLD_VALUE = False


THRESHOLD_OBJECTIVE = (
    "survey_weighted_balanced_accuracy"
)


THRESHOLD_ROLE = (
    "secondary_statistical_operating_point"
)


THRESHOLD_TIE_TOLERANCE = 1e-12


THRESHOLD_PROBABILITY_CLIP = 1e-12


TIE_REFERENCE_THRESHOLD = 0.5


if not (
    0.0
    <
    THRESHOLD_PROBABILITY_CLIP
    <
    0.5
):

    raise RuntimeError(
        "Invalid threshold probability clip."
    )


# ============================================================
# 5. NUMERICAL THRESHOLD PROBABILITY PREPARATION
# ============================================================

def prepare_threshold_probabilities(
    probability,
):

    probability = np.asarray(
        probability,
        dtype=float,
    )


    if probability.ndim != 1:

        raise ValueError(
            "Probability must be one-dimensional."
        )


    if len(probability) == 0:

        raise ValueError(
            "Empty probability vector."
        )


    if not np.isfinite(
        probability
    ).all():

        raise ValueError(
            "Non-finite calibrated probability detected."
        )


    if (
        (probability < 0.0)
        |
        (probability > 1.0)
    ).any():

        raise ValueError(
            "Calibrated probability outside [0,1]."
        )


    # Numerical clipping is used ONLY for threshold search.
    # Primary probability-performance metrics remain governed
    # by the Step-17 evaluation protocol.
    return np.clip(
        probability,
        THRESHOLD_PROBABILITY_CLIP,
        1.0
        -
        THRESHOLD_PROBABILITY_CLIP,
    )


# ============================================================
# 6. EXACT CLASSIFICATION-CHANGING THRESHOLDS
# ============================================================

def exact_threshold_candidates(
    calibrated_probability,
):

    probability = (
        prepare_threshold_probabilities(
            calibrated_probability
        )
    )


    unique_probability = np.unique(
        probability
    )


    # With one unique probability there are exactly two
    # possible classifications:
    # all positive or all negative.
    if len(unique_probability) == 1:

        return np.array(
            [
                0.0,
                1.0,
            ],
            dtype=float,
        )


    # Every midpoint represents one distinct classification
    # state between two adjacent fitted probability values.
    midpoints = (
        unique_probability[:-1]
        +
        (
            unique_probability[1:]
            -
            unique_probability[:-1]
        )
        / 2.0
    )


    candidates = np.concatenate(
        [
            np.array(
                [0.0],
                dtype=float,
            ),

            midpoints,

            np.array(
                [1.0],
                dtype=float,
            ),
        ]
    )


    candidates = np.unique(
        candidates
    )


    if (
        candidates[0]
        !=
        0.0
    ):

        raise RuntimeError(
            "Threshold candidate set does not start at 0."
        )


    if (
        candidates[-1]
        !=
        1.0
    ):

        raise RuntimeError(
            "Threshold candidate set does not end at 1."
        )


    return candidates


# ============================================================
# 7. SURVEY-WEIGHTED BINARY CLASSIFICATION METRICS
# ============================================================

def weighted_binary_threshold_metrics(
    y_true,
    calibrated_probability,
    sample_weight,
    threshold,
):

    y_true = np.asarray(
        y_true,
        dtype=int,
    )


    probability = (
        prepare_threshold_probabilities(
            calibrated_probability
        )
    )


    sample_weight = np.asarray(
        sample_weight,
        dtype=float,
    )


    if not (
        len(y_true)
        ==
        len(probability)
        ==
        len(sample_weight)
    ):

        raise ValueError(
            "Input lengths differ."
        )


    unique_y = set(
        np.unique(y_true)
    )


    if unique_y != {0, 1}:

        raise ValueError(
            f"Both binary classes required; observed {unique_y}."
        )


    if not np.isfinite(
        sample_weight
    ).all():

        raise ValueError(
            "Non-finite sample weight detected."
        )


    if (
        sample_weight
        <=
        0
    ).any():

        raise ValueError(
            "Sample weights must be strictly positive."
        )


    threshold = float(
        threshold
    )


    if not (
        0.0
        <=
        threshold
        <=
        1.0
    ):

        raise ValueError(
            "Threshold outside [0,1]."
        )


    predicted = (
        probability
        >=
        threshold
    ).astype(int)


    positive = (
        y_true
        ==
        1
    )


    negative = (
        y_true
        ==
        0
    )


    tp = float(
        sample_weight[
            positive
            &
            (predicted == 1)
        ].sum()
    )


    fn = float(
        sample_weight[
            positive
            &
            (predicted == 0)
        ].sum()
    )


    tn = float(
        sample_weight[
            negative
            &
            (predicted == 0)
        ].sum()
    )


    fp = float(
        sample_weight[
            negative
            &
            (predicted == 1)
        ].sum()
    )


    positive_weight = (
        tp
        +
        fn
    )


    negative_weight = (
        tn
        +
        fp
    )


    if (
        positive_weight
        <=
        0
    ):

        raise RuntimeError(
            "Positive-class total weight is zero."
        )


    if (
        negative_weight
        <=
        0
    ):

        raise RuntimeError(
            "Negative-class total weight is zero."
        )


    sensitivity = (
        tp
        /
        positive_weight
    )


    specificity = (
        tn
        /
        negative_weight
    )


    balanced_accuracy = (
        sensitivity
        +
        specificity
    ) / 2.0


    youden_j = (
        sensitivity
        +
        specificity
        -
        1.0
    )


    balance_floor = min(
        sensitivity,
        specificity,
    )


    total_weight = (
        tp
        +
        tn
        +
        fp
        +
        fn
    )


    accuracy = (
        tp
        +
        tn
    ) / total_weight


    if (
        tp
        +
        fp
    ) > 0:

        ppv = (
            tp
            /
            (tp + fp)
        )

    else:

        ppv = np.nan


    if (
        tn
        +
        fn
    ) > 0:

        npv = (
            tn
            /
            (tn + fn)
        )

    else:

        npv = np.nan


    if (
        2.0 * tp
        +
        fp
        +
        fn
    ) > 0:

        f1 = (
            2.0 * tp
            /
            (
                2.0 * tp
                +
                fp
                +
                fn
            )
        )

    else:

        f1 = np.nan


    return {
        "threshold":
            threshold,

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy
            ),

        "youden_j":
            float(
                youden_j
            ),

        "balance_floor":
            float(
                balance_floor
            ),

        "accuracy":
            float(
                accuracy
            ),

        "ppv":
            float(ppv)
            if np.isfinite(ppv)
            else np.nan,

        "npv":
            float(npv)
            if np.isfinite(npv)
            else np.nan,

        "f1":
            float(f1)
            if np.isfinite(f1)
            else np.nan,

        "weighted_tp":
            tp,

        "weighted_fn":
            fn,

        "weighted_tn":
            tn,

        "weighted_fp":
            fp,
    }


# ============================================================
# 8. EXACT WEIGHTED THRESHOLD OPTIMIZER
# ============================================================

def optimize_weighted_threshold(
    y_true,
    calibrated_probability,
    sample_weight,
):

    thresholds = exact_threshold_candidates(
        calibrated_probability
    )


    result_rows = []


    for threshold in thresholds:

        result_rows.append(
            weighted_binary_threshold_metrics(
                y_true=y_true,
                calibrated_probability=calibrated_probability,
                sample_weight=sample_weight,
                threshold=threshold,
            )
        )


    results = pd.DataFrame(
        result_rows
    )


    # --------------------------------------------------------
    # Criterion 1:
    # highest weighted balanced accuracy
    # --------------------------------------------------------

    best_balanced_accuracy = float(
        results[
            "balanced_accuracy"
        ].max()
    )


    tied = results.loc[
        np.isclose(
            results[
                "balanced_accuracy"
            ],
            best_balanced_accuracy,
            rtol=0.0,
            atol=THRESHOLD_TIE_TOLERANCE,
        )
    ].copy()


    # --------------------------------------------------------
    # Criterion 2:
    # maximize the smaller of sensitivity and specificity
    # --------------------------------------------------------

    best_balance_floor = float(
        tied[
            "balance_floor"
        ].max()
    )


    tied = tied.loc[
        np.isclose(
            tied[
                "balance_floor"
            ],
            best_balance_floor,
            rtol=0.0,
            atol=THRESHOLD_TIE_TOLERANCE,
        )
    ].copy()


    # --------------------------------------------------------
    # Criterion 3:
    # deterministic reference only.
    #
    # 0.5 is NOT a default classification threshold.
    # --------------------------------------------------------

    tied[
        "distance_from_tie_reference"
    ] = (
        tied[
            "threshold"
        ]
        -
        TIE_REFERENCE_THRESHOLD
    ).abs()


    minimum_distance = float(
        tied[
            "distance_from_tie_reference"
        ].min()
    )


    tied = tied.loc[
        np.isclose(
            tied[
                "distance_from_tie_reference"
            ],
            minimum_distance,
            rtol=0.0,
            atol=THRESHOLD_TIE_TOLERANCE,
        )
    ].copy()


    # --------------------------------------------------------
    # Criterion 4:
    # final deterministic tie-break
    # --------------------------------------------------------

    selected = (
        tied
        .sort_values(
            "threshold",
            ascending=True,
        )
        .iloc[0]
    )


    return {

        "selected_threshold":
            float(
                selected[
                    "threshold"
                ]
            ),

        "optimized_balanced_accuracy":
            float(
                selected[
                    "balanced_accuracy"
                ]
            ),

        "sensitivity_at_selected_threshold":
            float(
                selected[
                    "sensitivity"
                ]
            ),

        "specificity_at_selected_threshold":
            float(
                selected[
                    "specificity"
                ]
            ),

        "youden_j_at_selected_threshold":
            float(
                selected[
                    "youden_j"
                ]
            ),

        "candidate_threshold_count":
            int(
                len(results)
            ),

        "unique_operating_points_evaluated":
            int(
                len(results)
            ),

        "threshold_was_optimized":
            True,

        "threshold_was_fixed":
            False,
    }


# ============================================================
# 9. SYNTHETIC OPTIMIZER QC
# ============================================================
#
# Synthetic values only.
# No study predictions are examined.
# ============================================================

synthetic_y = np.array(
    [
        0, 0, 0, 0, 0,
        1, 1, 1, 1, 1,
    ],
    dtype=int,
)


synthetic_probability = np.array(
    [
        0.03,
        0.08,
        0.20,
        0.38,
        0.60,
        0.18,
        0.45,
        0.62,
        0.78,
        0.91,
    ],
    dtype=float,
)


synthetic_weight = np.array(
    [
        1.0,
        1.1,
        0.9,
        1.2,
        0.8,
        1.0,
        1.3,
        0.7,
        1.1,
        1.0,
    ],
    dtype=float,
)


synthetic_optimizer_result = (
    optimize_weighted_threshold(
        y_true=synthetic_y,
        calibrated_probability=synthetic_probability,
        sample_weight=synthetic_weight,
    )
)


if not synthetic_optimizer_result[
    "threshold_was_optimized"
]:

    raise RuntimeError(
        "Synthetic threshold optimization failed."
    )


if synthetic_optimizer_result[
    "threshold_was_fixed"
]:

    raise RuntimeError(
        "Threshold was incorrectly treated as fixed."
    )


if not (
    0.0
    <=
    synthetic_optimizer_result[
        "selected_threshold"
    ]
    <=
    1.0
):

    raise RuntimeError(
        "Synthetic selected threshold outside [0,1]."
    )


# ============================================================
# 10. FORMAL THRESHOLD-OPTIMIZATION PROTOCOL
# ============================================================

threshold_optimization_protocol = {

    "threshold_optimization":
        True,

    "threshold_value_fixed":
        False,

    "default_probability_threshold_0_5":
        False,

    "role":
        THRESHOLD_ROLE,

    "primary_probability_model_unchanged":
        True,

    "secondary_classification_analysis":
        True,

    "optimization_occurs_after_joint_model_selection":
        True,

    "optimization_occurs_after_probability_calibration":
        True,

    "optimization_source":
        (
            "selected joint winner's calibrated pooled "
            "inner-OOF probabilities"
        ),

    "optimization_labels":
        (
            "corresponding binary outcomes from the current "
            "outer-training population only"
        ),

    "optimization_weights":
        "original DHS SURVEY_WEIGHT",

    "weight_normalization_required_for_threshold_metric":
        False,

    "reason_weight_normalization_not_required":
        (
            "Balanced accuracy is based on within-class weighted "
            "rates and is invariant to multiplication of all "
            "survey weights by the same positive constant."
        ),

    "objective":
        THRESHOLD_OBJECTIVE,

    "objective_formula":
        (
            "weighted balanced accuracy = "
            "(weighted sensitivity + weighted specificity) / 2"
        ),

    "youden_equivalence":
        (
            "weighted Youden J = 2*weighted balanced accuracy - 1; "
            "the two criteria therefore produce identical "
            "threshold rankings"
        ),

    "candidate_threshold_strategy":
        (
            "exhaustive evaluation of all classification-changing "
            "operating points implied by distinct calibrated "
            "inner-OOF probabilities"
        ),

    "candidate_thresholds":
        (
            "0, midpoints between consecutive distinct calibrated "
            "probabilities, and 1"
        ),

    "threshold_probability_clip_for_search_only":
        THRESHOLD_PROBABILITY_CLIP,

    "coarse_fixed_threshold_grid":
        False,

    "random_threshold_search":
        False,

    "f1_optimization":
        False,

    "raw_accuracy_optimization":
        False,

    "outer_test_optimization":
        False,

    "pooled_outer_oof_optimization":
        False,

    "stage1_threshold_optimized":
        True,

    "stage2_threshold_optimized":
        True,

    "stage_specific_thresholds":
        True,

    "outer_fold_specific_thresholds":
        True,

    "pipeline_specific_thresholds":
        True,

    "same_threshold_stage1_stage2":
        False,

    "same_threshold_all_outer_folds":
        False,

    "same_threshold_primary_sensitivity":
        False,

    "threshold_can_change_model_family":
        False,

    "threshold_can_change_hyperparameters":
        False,

    "threshold_can_change_calibration_method":
        False,

    "tie_tolerance":
        THRESHOLD_TIE_TOLERANCE,

    "tie_rule_1":
        "highest survey-weighted balanced accuracy",

    "tie_rule_2":
        (
            "highest minimum of sensitivity and specificity"
        ),

    "tie_rule_3":
        (
            "threshold closest to 0.5; deterministic tie-break only"
        ),

    "tie_rule_4":
        "lower numerical threshold",

    "clinical_optimal_threshold_claim":
        False,
}


# ============================================================
# 11. STAGE-SPECIFIC OPTIMIZATION
# ============================================================

stage_threshold_protocol = pd.DataFrame(
    [
        {
            "Stage":
                "Stage 1",

            "Positive_Class":
                "No ANC",

            "Development_Population":
                "All current outer-training observations",

            "Optimization_Probability":
                "Calibrated q0",

            "Threshold":
                "t1",

            "Optimization_Objective":
                "Survey-weighted balanced accuracy",

            "Threshold_Fixed":
                False,
        },

        {
            "Stage":
                "Stage 2",

            "Positive_Class":
                ">=8 ANC visits",

            "Development_Population":
                "True ANC users in current outer-training set",

            "Optimization_Probability":
                "Calibrated q2",

            "Threshold":
                "t2",

            "Optimization_Objective":
                "Survey-weighted balanced accuracy",

            "Threshold_Fixed":
                False,
        },
    ]
)


# ============================================================
# 12. HIERARCHICAL FINAL-CLASS RULE
# ============================================================

hierarchical_classification_protocol = {

    "stage1_threshold":
        "optimized t1",

    "stage2_threshold":
        "optimized t2",

    "thresholds_fixed_constants":
        False,

    "stage1_rule":
        (
            "if calibrated q0 >= optimized t1, "
            "predict class 0: No ANC"
        ),

    "stage2_rule":
        (
            "otherwise, if calibrated q2 >= optimized t2, "
            "predict class 2: >=8 visits"
        ),

    "remaining_rule":
        (
            "otherwise predict class 1: 1-7 visits"
        ),

    "stage1_priority":
        True,

    "reason_stage1_priority":
        (
            "Stage 2 represents a conditional model among ANC "
            "users and is therefore downstream of the Stage-1 "
            "No-ANC versus Any-ANC decision."
        ),

    "outer_test_threshold_reoptimization":
        False,

    "posthoc_classification_rule_change":
        False,
}


# ============================================================
# 13. OUTER-TEST APPLICATION
# ============================================================

outer_threshold_application = {

    "outer_fold_specific":
        True,

    "stage_specific":
        True,

    "primary_vs_sensitivity_specific":
        True,

    "development_data_for_threshold":
        (
            "current outer-training population only"
        ),

    "outer_test_stage1":
        (
            "apply already optimized t1 to calibrated outer-test q0"
        ),

    "outer_test_stage2":
        (
            "apply already optimized t2 to calibrated outer-test q2"
        ),

    "outer_test_threshold_fit":
        False,

    "outer_test_threshold_adjustment":
        False,

    "outer_test_prevalence_adjustment":
        False,

    "outer_test_metric_guided_adjustment":
        False,

    "pooled_outer_oof_reoptimization":
        False,

    "interpretation":
        (
            "outer-test threshold metrics evaluate the complete "
            "prespecified threshold-optimization procedure"
        ),
}


# ============================================================
# 14. PRIMARY vs SECONDARY PERFORMANCE HIERARCHY
# ============================================================

analysis_hierarchy = pd.DataFrame(
    [
        {
            "Domain":
                "Discrimination",

            "Measure":
                "Survey-weighted Average Precision",

            "Role":
                "Primary",
        },

        {
            "Domain":
                "Discrimination",

            "Measure":
                "Survey-weighted ROC-AUC",

            "Role":
                "Primary / secondary discrimination summary",
        },

        {
            "Domain":
                "Probabilistic performance",

            "Measure":
                "Survey-weighted log loss",

            "Role":
                "Primary",
        },

        {
            "Domain":
                "Probabilistic performance",

            "Measure":
                "Survey-weighted Brier score",

            "Role":
                "Primary",
        },

        {
            "Domain":
                "Calibration",

            "Measure":
                (
                    "Calibration intercept / slope / O-E / "
                    "smooth calibration curve"
                ),

            "Role":
                "Primary",
        },

        {
            "Domain":
                "Threshold selection",

            "Measure":
                (
                    "Optimized survey-weighted "
                    "balanced-accuracy operating point"
                ),

            "Role":
                "Secondary",
        },

        {
            "Domain":
                "Classification",

            "Measure":
                (
                    "Sensitivity / specificity / PPV / NPV / "
                    "F1 / accuracy"
                ),

            "Role":
                "Secondary",
        },

        {
            "Domain":
                "Three-class classification",

            "Measure":
                (
                    "Per-class metrics / macro metrics / "
                    "weighted confusion matrix"
                ),

            "Role":
                "Secondary",
        },
    ]
)


# ============================================================
# 15. BINARY THRESHOLD-DEPENDENT METRICS
# ============================================================

binary_threshold_metric_registry = pd.DataFrame(
    [
        {
            "Metric":
                "Survey-weighted sensitivity",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted specificity",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted balanced accuracy",

            "Role":
                (
                    "Secondary; threshold optimization criterion"
                ),
        },

        {
            "Metric":
                "Survey-weighted Youden J",

            "Role":
                (
                    "Secondary; mathematically equivalent "
                    "threshold ranking"
                ),
        },

        {
            "Metric":
                "Survey-weighted PPV",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted NPV",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted F1 score",

            "Role":
                (
                    "Secondary; not optimized"
                ),
        },

        {
            "Metric":
                "Survey-weighted accuracy",

            "Role":
                (
                    "Secondary; not optimized"
                ),
        },
    ]
)


# ============================================================
# 16. FINAL THREE-CLASS CLASSIFICATION METRICS
# ============================================================

multiclass_threshold_metric_registry = pd.DataFrame(
    [
        {
            "Metric":
                "Survey-weighted overall accuracy",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted recall by ANC class",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted precision by ANC class",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted F1 by ANC class",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Macro recall / multiclass balanced accuracy",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Macro F1",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted 3x3 confusion matrix",

            "Role":
                "Secondary",
        },
    ]
)


# ============================================================
# 17. THRESHOLD STABILITY REPORTING
# ============================================================

threshold_stability_protocol = {

    "report_outer_fold_thresholds":
        True,

    "report_stage1_thresholds":
        True,

    "report_stage2_thresholds":
        True,

    "report_primary_thresholds":
        True,

    "report_sensitivity_pipeline_thresholds":
        True,

    "summary_statistics":
        [
            "median",
            "minimum",
            "maximum",
            "interquartile range",
        ],

    "threshold_variability_used_to_reselect_model":
        False,

    "threshold_variability_used_to_change_objective":
        False,

    "role":
        (
            "descriptive assessment of operating-point stability"
        ),
}


# ============================================================
# 18. DECISION-CURVE BOUNDARY
# ============================================================

decision_curve_protocol = {

    "primary_analysis":
        False,

    "performed_in_step18":
        False,

    "balanced_accuracy_threshold_is_clinical_threshold":
        False,

    "clinically_optimal_claim":
        False,

    "reason":
        (
            "No specific intervention, downstream action, "
            "false-positive/false-negative cost ratio, or "
            "externally justified risk-threshold range has "
            "been prespecified."
        ),

    "future_decision_curve_allowed":
        True,

    "future_condition":
        (
            "Only if an intervention/action and clinically "
            "meaningful threshold range are defined independently "
            "of observed predictive performance."
        ),
}


# ============================================================
# 19. FINAL FULL-DATA THRESHOLD OPTIMIZATION
# ============================================================

final_model_threshold_protocol = {

    "outer_fold_thresholds_are_final_thresholds":
        False,

    "final_threshold_values_fixed_now":
        False,

    "final_threshold_optimization_required":
        True,

    "final_threshold_method":
        (
            "same frozen survey-weighted balanced-accuracy "
            "optimization algorithm"
        ),

    "final_threshold_data":
        (
            "cross-fitted calibrated development predictions "
            "generated after the final full-data model-selection "
            "procedure"
        ),

    "stage1_final_threshold":
        "optimized from Stage-1 development predictions",

    "stage2_final_threshold":
        (
            "optimized among ANC users from Stage-2 "
            "development predictions"
        ),

    "final_threshold_selected_from_outer_test_results":
        False,

    "final_objective_change_allowed":
        False,

    "final_tie_rule_change_allowed":
        False,
}


# ============================================================
# 20. REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "SKLEARN_THRESHOLD_1_9",

            "Reference":
                (
                    "Scikit-learn 1.9 documentation: "
                    "Tuning the decision threshold for "
                    "class prediction."
                ),

            "DOI":
                "",

            "Use":
                (
                    "Cross-validated post-tuning of decision "
                    "thresholds and separation of probability "
                    "estimation from the classification decision."
                ),
        },

        {
            "ID":
                "EFTHIMIOU_2024",

            "Reference":
                (
                    "Efthimiou O, Seo M, Chalkou K, et al. "
                    "Developing clinical prediction models: "
                    "a step-by-step guide. "
                    "BMJ. 2024;386:e078276."
                ),

            "DOI":
                "10.1136/bmj-2023-078276",

            "Use":
                (
                    "Avoid post-hoc cut-off selection using "
                    "evaluation data and distinguish statistical "
                    "performance from clinical utility."
                ),
        },

        {
            "ID":
                "RILEY_EVALUATION_2024",

            "Reference":
                (
                    "Evaluation of clinical prediction models "
                    "(part 2): how to undertake an external "
                    "validation study. "
                    "BMJ. 2024;384:e074820."
                ),

            "DOI":
                "10.1136/bmj-2023-074820",

            "Use":
                (
                    "Clinical action thresholds should be "
                    "chosen from the decision context."
                ),
        },
    ]
)


# ============================================================
# 21. MANUSCRIPT-READY METHODS
# ============================================================

methods_text = """
Decision-threshold optimization and secondary classification analyses

Threshold values were not fixed in advance. Instead, a threshold-
optimization procedure was prespecified before model fitting. For each
hierarchical stage, analysis pipeline, and outer cross-validation fold,
the operating threshold will be optimized only after joint model-family
and hyperparameter selection and probability calibration.

Threshold optimization will use the selected winner's calibrated,
cross-fitted inner-validation probabilities from the corresponding
outer-training population. Original DHS sampling weights will be used
in the optimization criterion. The outer-test population will not be
used to select, adjust, or re-estimate decision thresholds.

The optimization criterion will be survey-weighted balanced accuracy,
defined as the arithmetic mean of weighted sensitivity and weighted
specificity. Maximization of balanced accuracy is equivalent in
threshold ranking to maximization of the weighted Youden index.
Threshold values will not be restricted to 0.5 or to a coarse
prespecified numerical grid. Instead, every classification-changing
operating point defined by the distinct calibrated inner-validation
probabilities will be evaluated.

Stage-1 and Stage-2 thresholds will be optimized independently. For
Stage 1, the positive outcome is no antenatal care. For Stage 2, the
positive outcome is eight or more reported antenatal-care visits and
threshold optimization will be performed among true antenatal-care
users in the development population. Thresholds will also be optimized
independently within each outer fold and separately for the primary
survey-weighted model-fitting pipeline and the unweighted-fitting
sensitivity analysis.

For final hierarchical class assignment, an observation with calibrated
Stage-1 probability q0 at or above the optimized Stage-1 threshold t1
will be classified as no ANC. Otherwise, an observation with calibrated
conditional Stage-2 probability q2 at or above the optimized Stage-2
threshold t2 will be classified as eight or more visits; remaining
observations will be classified as one to seven visits.

Threshold-dependent sensitivity, specificity, balanced accuracy,
positive and negative predictive values, F1 score, accuracy, and
three-category confusion-matrix measures will be considered secondary
classification summaries. Primary evaluation will remain based on
threshold-independent discrimination, probabilistic performance, and
calibration.

The optimized thresholds will be described as statistical operating
points rather than clinically optimal action thresholds because no
specific downstream intervention, false-positive versus false-negative
cost ratio, or externally justified clinical probability threshold has
been prespecified.
""".strip()


# ============================================================
# 22. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Revised joint Step 16 frozen",

            "PASS":
                (
                    step16_manifest.get("status")
                    ==
                    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Step 17 calibration protocol frozen",

            "PASS":
                (
                    step17_manifest.get("status")
                    ==
                    "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Corrected Step 17A frozen",

            "PASS":
                (
                    step17a_manifest.get("status")
                    ==
                    "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "15 original predictors remain frozen",

            "PASS":
                (
                    step17a_manifest.get(
                        "original_predictor_count"
                    )
                    ==
                    15
                ),
        },

        {
            "Check":
                "Threshold optimization enabled",

            "PASS":
                THRESHOLD_OPTIMIZATION,
        },

        {
            "Check":
                "Threshold value is NOT fixed",

            "PASS":
                not FIXED_THRESHOLD_VALUE,
        },

        {
            "Check":
                "Default 0.5 threshold disabled",

            "PASS":
                not threshold_optimization_protocol[
                    "default_probability_threshold_0_5"
                ],
        },

        {
            "Check":
                "Threshold optimized after model selection",

            "PASS":
                threshold_optimization_protocol[
                    "optimization_occurs_after_joint_model_selection"
                ],
        },

        {
            "Check":
                "Threshold optimized after calibration",

            "PASS":
                threshold_optimization_protocol[
                    "optimization_occurs_after_probability_calibration"
                ],
        },

        {
            "Check":
                "Optimization uses calibrated inner-OOF probabilities",

            "PASS":
                (
                    "calibrated"
                    in
                    threshold_optimization_protocol[
                        "optimization_source"
                    ]
                    .lower()
                    and
                    "inner-oof"
                    in
                    threshold_optimization_protocol[
                        "optimization_source"
                    ]
                    .lower()
                ),
        },

        {
            "Check":
                "Objective is weighted balanced accuracy",

            "PASS":
                (
                    threshold_optimization_protocol[
                        "objective"
                    ]
                    ==
                    "survey_weighted_balanced_accuracy"
                ),
        },

        {
            "Check":
                "Exact classification-changing thresholds evaluated",

            "PASS":
                (
                    "all classification-changing"
                    in
                    threshold_optimization_protocol[
                        "candidate_threshold_strategy"
                    ]
                    .lower()
                ),
        },

        {
            "Check":
                "No coarse threshold grid",

            "PASS":
                not threshold_optimization_protocol[
                    "coarse_fixed_threshold_grid"
                ],
        },

        {
            "Check":
                "No random threshold search",

            "PASS":
                not threshold_optimization_protocol[
                    "random_threshold_search"
                ],
        },

        {
            "Check":
                "No F1 threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "f1_optimization"
                ],
        },

        {
            "Check":
                "No raw-accuracy threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "raw_accuracy_optimization"
                ],
        },

        {
            "Check":
                "No outer-test threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "outer_test_optimization"
                ],
        },

        {
            "Check":
                "No pooled outer-OOF threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "pooled_outer_oof_optimization"
                ],
        },

        {
            "Check":
                "Stage 1 threshold optimized",

            "PASS":
                threshold_optimization_protocol[
                    "stage1_threshold_optimized"
                ],
        },

        {
            "Check":
                "Stage 2 threshold optimized",

            "PASS":
                threshold_optimization_protocol[
                    "stage2_threshold_optimized"
                ],
        },

        {
            "Check":
                "Thresholds stage-specific",

            "PASS":
                threshold_optimization_protocol[
                    "stage_specific_thresholds"
                ],
        },

        {
            "Check":
                "Thresholds outer-fold-specific",

            "PASS":
                threshold_optimization_protocol[
                    "outer_fold_specific_thresholds"
                ],
        },

        {
            "Check":
                "Thresholds pipeline-specific",

            "PASS":
                threshold_optimization_protocol[
                    "pipeline_specific_thresholds"
                ],
        },

        {
            "Check":
                "Threshold cannot reselect model family",

            "PASS":
                not threshold_optimization_protocol[
                    "threshold_can_change_model_family"
                ],
        },

        {
            "Check":
                "Threshold cannot reselect hyperparameters",

            "PASS":
                not threshold_optimization_protocol[
                    "threshold_can_change_hyperparameters"
                ],
        },

        {
            "Check":
                "Hierarchical Stage-1 priority frozen",

            "PASS":
                hierarchical_classification_protocol[
                    "stage1_priority"
                ],
        },

        {
            "Check":
                "Outer-test threshold refit prohibited",

            "PASS":
                not outer_threshold_application[
                    "outer_test_threshold_fit"
                ],
        },

        {
            "Check":
                "Outer-test prevalence adjustment prohibited",

            "PASS":
                not outer_threshold_application[
                    "outer_test_prevalence_adjustment"
                ],
        },

        {
            "Check":
                "Threshold stability reporting prespecified",

            "PASS":
                threshold_stability_protocol[
                    "report_outer_fold_thresholds"
                ],
        },

        {
            "Check":
                "Statistical threshold not called clinically optimal",

            "PASS":
                not threshold_optimization_protocol[
                    "clinical_optimal_threshold_claim"
                ],
        },

        {
            "Check":
                "Synthetic optimizer actually optimizes threshold",

            "PASS":
                (
                    synthetic_optimizer_result[
                        "threshold_was_optimized"
                    ]
                    and
                    not synthetic_optimizer_result[
                        "threshold_was_fixed"
                    ]
                ),
        },

        {
            "Check":
                "No model fitted in Step 18",

            "PASS":
                True,
        },

        {
            "Check":
                "No study performance inspected in Step 18",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 18 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 23. SAVE PROTOCOL OUTPUTS
# ============================================================

THRESHOLD_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_protocol.json"
)


STAGE_THRESHOLD_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_stage_specific_threshold_policy.csv"
)


HIERARCHICAL_CLASS_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_hierarchical_classification_rule.json"
)


OUTER_APPLICATION_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_outer_threshold_application.json"
)


ANALYSIS_HIERARCHY_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_primary_secondary_analysis_hierarchy.csv"
)


BINARY_METRIC_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_binary_threshold_metric_registry.csv"
)


MULTICLASS_METRIC_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_multiclass_threshold_metric_registry.csv"
)


THRESHOLD_STABILITY_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_stability_protocol.json"
)


DECISION_CURVE_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_decision_curve_boundary.json"
)


FINAL_THRESHOLD_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_full_data_threshold_optimization_policy.json"
)


REFERENCE_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_reference_registry.csv"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_manuscript_threshold_optimization_methods.txt"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP18_FINAL_hard_qc.csv"
)


stage_threshold_protocol.to_csv(
    STAGE_THRESHOLD_FILE,
    index=False,
)


analysis_hierarchy.to_csv(
    ANALYSIS_HIERARCHY_FILE,
    index=False,
)


binary_threshold_metric_registry.to_csv(
    BINARY_METRIC_FILE,
    index=False,
)


multiclass_threshold_metric_registry.to_csv(
    MULTICLASS_METRIC_FILE,
    index=False,
)


reference_registry.to_csv(
    REFERENCE_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


json_outputs = [
    (
        THRESHOLD_PROTOCOL_FILE,
        threshold_optimization_protocol,
    ),

    (
        HIERARCHICAL_CLASS_FILE,
        hierarchical_classification_protocol,
    ),

    (
        OUTER_APPLICATION_FILE,
        outer_threshold_application,
    ),

    (
        THRESHOLD_STABILITY_FILE,
        threshold_stability_protocol,
    ),

    (
        DECISION_CURVE_FILE,
        decision_curve_protocol,
    ),

    (
        FINAL_THRESHOLD_FILE,
        final_model_threshold_protocol,
    ),
]


for path, payload in json_outputs:

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
        )


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


# ============================================================
# 24. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 18",

    "status":
        "THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "primary_master_sha256":
        sha256_file(
            PRIMARY_FILE
        ),

    "outer_split_sha256":
        sha256_file(
            OUTER_SPLIT_FILE
        ),

    "inner_split_sha256":
        sha256_file(
            INNER_SPLIT_FILE
        ),

    "predictor_count":
        15,

    "threshold_optimization":
        True,

    "threshold_value_fixed":
        False,

    "default_threshold_0_5":
        False,

    "threshold_objective":
        THRESHOLD_OBJECTIVE,

    "threshold_source":
        (
            "selected-winner calibrated inner-OOF probabilities"
        ),

    "threshold_weight":
        "original DHS SURVEY_WEIGHT",

    "stage_specific_thresholds":
        True,

    "outer_fold_specific_thresholds":
        True,

    "pipeline_specific_thresholds":
        True,

    "exact_operating_point_search":
        True,

    "outer_test_used_for_threshold_optimization":
        False,

    "pooled_outer_oof_used_for_threshold_optimization":
        False,

    "threshold_analysis_role":
        "secondary",

    "clinically_optimal_threshold_claim":
        False,

    "model_fitting_performed":
        False,

    "study_performance_evaluated":
        False,

    "threshold_protocol_sha256":
        sha256_file(
            THRESHOLD_PROTOCOL_FILE
        ),

    "stage_threshold_policy_sha256":
        sha256_file(
            STAGE_THRESHOLD_FILE
        ),

    "hierarchical_classification_sha256":
        sha256_file(
            HIERARCHICAL_CLASS_FILE
        ),

    "outer_application_sha256":
        sha256_file(
            OUTER_APPLICATION_FILE
        ),

    "threshold_stability_sha256":
        sha256_file(
            THRESHOLD_STABILITY_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),

    "methods_sha256":
        sha256_file(
            METHODS_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 25. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 18."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 18."
    )


if sha256_file(
    INNER_SPLIT_FILE
) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 18."
    )


# ============================================================
# 26. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 18 — CROSS-VALIDATED THRESHOLD OPTIMIZATION PROTOCOL FREEZE"
)


print(
    "=" * 78
)


print(
    "\nKEY DISTINCTION:"
)


print(
    "Threshold optimization : ENABLED"
)


print(
    "Threshold value fixed  : NO"
)


print(
    "Default threshold 0.5  : NO"
)


print(
    "\nSTAGE-SPECIFIC THRESHOLD POLICY:"
)


display(
    stage_threshold_protocol
)


print(
    "\nPRIMARY / SECONDARY ANALYSIS HIERARCHY:"
)


display(
    analysis_hierarchy
)


print(
    "\nBINARY THRESHOLD METRICS:"
)


display(
    binary_threshold_metric_registry
)


print(
    "\nTHREE-CLASS THRESHOLD METRICS:"
)


display(
    multiclass_threshold_metric_registry
)


print(
    "\nOPTIMIZATION RULE:"
)


print(
    "- Optimize threshold AFTER joint model/hyperparameter selection."
)


print(
    "- Optimize threshold AFTER probability calibration."
)


print(
    "- Use selected winner's calibrated INNER-OOF probabilities."
)


print(
    "- Optimize survey-weighted balanced accuracy."
)


print(
    "- Evaluate every classification-changing operating point."
)


print(
    "- Stage 1 and Stage 2 thresholds are optimized independently."
)


print(
    "- Thresholds are independently optimized within every outer fold."
)


print(
    "- Primary and unweighted-sensitivity thresholds are optimized independently."
)


print(
    "- Outer-test outcomes NEVER optimize thresholds."
)


print(
    "- Pooled outer-OOF outcomes NEVER re-optimize thresholds."
)


print(
    "\n0.5 POLICY:"
)


print(
    "- 0.5 is NOT a fixed or default operating threshold."
)


print(
    "- It is used only as a deterministic tie-break reference after exact optimization ties."
)


print(
    "\nHIERARCHICAL CLASS RULE:"
)


print(
    "if q0 >= optimized t1:"
)


print(
    "    class 0 = No ANC"
)


print(
    "elif q2 >= optimized t2:"
)


print(
    "    class 2 = >=8 visits"
)


print(
    "else:"
)


print(
    "    class 1 = 1-7 visits"
)


print(
    "\nSYNTHETIC OPTIMIZER QC:"
)


print(
    synthetic_optimizer_result
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)


print(
    methods_text
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)


print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)


print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    "=" * 78
)


print(
    "STATUS: STEP 18 PASS"
)


print(
    "=" * 78
)


STEP 18 — CROSS-VALIDATED THRESHOLD OPTIMIZATION PROTOCOL FREEZE

KEY DISTINCTION:
Threshold optimization : ENABLED
Threshold value fixed  : NO
Default threshold 0.5  : NO

STAGE-SPECIFIC THRESHOLD POLICY:


,Stage,Positive_Class,Development_Population,Optimization_Probability,Threshold,Optimization_Objective,Threshold_Fixed
0,Stage 1,No ANC,All current outer-training observations,Calibrated q0,t1,Survey-weighted balanced accuracy,False
1,Stage 2,>=8 ANC visits,True ANC users in current outer-training set,Calibrated q2,t2,Survey-weighted balanced accuracy,False



PRIMARY / SECONDARY ANALYSIS HIERARCHY:


,Domain,Measure,Role
0,Discrimination,Survey-weighted Average Precision,Primary
1,Discrimination,Survey-weighted ROC-AUC,Primary / secondary discrimination summary
2,Probabilistic performance,Survey-weighted log loss,Primary
3,Probabilistic performance,Survey-weighted Brier score,Primary
4,Calibration,Calibration intercept / slope / O-E / smooth c...,Primary
5,Threshold selection,Optimized survey-weighted balanced-accuracy op...,Secondary
6,Classification,Sensitivity / specificity / PPV / NPV / F1 / a...,Secondary
7,Three-class classification,Per-class metrics / macro metrics / weighted c...,Secondary


In [1]:
# ============================================================
# KERNEL RESTART RECOVERY
# Restore and verify the frozen project state from disk
# ============================================================
#
# PURPOSE
# -------
# A kernel restart clears Python variables, NOT the files saved
# on disk.
#
# This cell:
#   - rebuilds ROOT and DIRS
#   - reloads core Python packages
#   - verifies the frozen primary dataset
#   - verifies outer / inner CV hashes
#   - verifies Steps 14, 15, 15A, 16, 17, 17A manifests
#   - confirms the project is ready to continue with Step 18
#
# This cell:
#   - does NOT modify data
#   - does NOT regenerate CV folds
#   - does NOT overwrite protocol files
#   - does NOT fit models
#   - does NOT inspect performance
# ============================================================


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from pathlib import Path

import hashlib
import importlib.metadata as importlib_metadata
import json
import sys

import numpy as np
import pandas as pd
import sklearn

from IPython.display import display


# ============================================================
# 2. PROJECT ROOT
# ============================================================

ROOT = Path(
    r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
)


if not ROOT.exists():

    raise FileNotFoundError(
        f"Project root not found:\n{ROOT}"
    )


# ============================================================
# 3. RESTORE DIRECTORY REGISTRY
# ============================================================

DIRS = {

    "root":
        ROOT,

    "raw_reference":
        ROOT / "00_raw_reference",

    "processed":
        ROOT / "01_processed_data",

    "qc":
        ROOT / "02_quality_control",

    "protocol":
        ROOT / "03_protocol",

    "splits":
        ROOT / "04_cv_splits",

    "models":
        ROOT / "05_models",

    "predictions":
        ROOT / "06_predictions",

    "calibration":
        ROOT / "07_calibration",

    "bootstrap":
        ROOT / "08_bootstrap",

    "geographic":
        ROOT / "09_geographic_validation",

    "explainability":
        ROOT / "10_explainability",

    "sensitivity":
        ROOT / "11_sensitivity",

    "subgroup":
        ROOT / "12_subgroup_analysis",

    "tables":
        ROOT / "13_tables",

    "figures":
        ROOT / "14_figures",

    "logs":
        ROOT / "15_logs",

    "reporting":
        ROOT / "16_reporting",

    "archive":
        ROOT / "17_reproducibility_archive",
}


missing_directories = [

    str(path)

    for path in DIRS.values()

    if not path.exists()
]


if missing_directories:

    raise FileNotFoundError(
        "Expected project directories are missing:\n"
        +
        "\n".join(
            missing_directories
        )
    )


# ============================================================
# 4. HASH UTILITY
# ============================================================

def sha256_file(path):

    path = Path(path)

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):

            digest.update(
                block
            )

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(
            handle
        )


# ============================================================
# 5. FROZEN CORE FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Frozen core file missing:\n{path}"
        )


# ============================================================
# 6. CORE HASH VERIFICATION
# ============================================================

observed_primary_hash = sha256_file(
    PRIMARY_FILE
)


observed_outer_hash = sha256_file(
    OUTER_SPLIT_FILE
)


observed_inner_hash = sha256_file(
    INNER_SPLIT_FILE
)


if (
    observed_primary_hash
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "PRIMARY MASTER HASH MISMATCH.\n"
        "Do not continue."
    )


if (
    observed_outer_hash
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "OUTER SPLIT HASH MISMATCH.\n"
        "Do not continue."
    )


if (
    observed_inner_hash
    !=
    EXPECTED_INNER_SHA256
):

    raise RuntimeError(
        "INNER SPLIT HASH MISMATCH.\n"
        "Do not continue."
    )


# ============================================================
# 7. RESTORE FROZEN MANIFEST PATHS
# ============================================================

MANIFESTS = {

    "Step 14":
        (
            DIRS["protocol"]
            / "STEP14_FINAL_preprocessing_manifest.json"
        ),

    "Step 15":
        (
            DIRS["protocol"]
            / "STEP15_FINAL_hierarchical_weight_manifest.json"
        ),

    "Step 15A":
        (
            DIRS["protocol"]
            / "STEP15A_FINAL_survey_weight_sensitivity_manifest.json"
        ),

    "Step 16":
        (
            DIRS["protocol"]
            / "STEP16_FINAL_hyperparameter_selection_manifest.json"
        ),

    "Step 17":
        (
            DIRS["protocol"]
            / "STEP17_FINAL_calibration_evaluation_manifest.json"
        ),

    "Step 17A":
        (
            DIRS["protocol"]
            / "STEP17A_FINAL_predictor_feature_selection_manifest.json"
        ),
}


EXPECTED_STATUS = {

    "Step 14":
        "PREPROCESSING_PROTOCOL_FROZEN",

    "Step 15":
        "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN",

    "Step 15A":
        "SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN",

    "Step 16":
        "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN",

    "Step 17":
        "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN",

    "Step 17A":
        "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN",
}


for step_name, path in MANIFESTS.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{step_name} manifest missing:\n{path}"
        )


# ============================================================
# 8. LOAD + VERIFY MANIFESTS
# ============================================================

loaded_manifests = {}


manifest_rows = []


for step_name, path in MANIFESTS.items():

    manifest = read_json(
        path
    )


    loaded_manifests[
        step_name
    ] = manifest


    observed_status = (
        manifest.get(
            "status"
        )
    )


    expected_status = (
        EXPECTED_STATUS[
            step_name
        ]
    )


    status_ok = (
        observed_status
        ==
        expected_status
    )


    manifest_rows.append(
        {
            "Step":
                step_name,

            "Expected_Status":
                expected_status,

            "Observed_Status":
                observed_status,

            "PASS":
                status_ok,
        }
    )


manifest_qc = pd.DataFrame(
    manifest_rows
)


if not manifest_qc[
    "PASS"
].all():

    raise RuntimeError(
        "One or more upstream protocols are not "
        "in the expected frozen state:\n\n"
        +
        manifest_qc.to_string(
            index=False
        )
    )


# ============================================================
# 9. RESTORE COMMON MANIFEST VARIABLES
# ============================================================

step14_manifest = loaded_manifests[
    "Step 14"
]


step15_manifest = loaded_manifests[
    "Step 15"
]


step15a_manifest = loaded_manifests[
    "Step 15A"
]


step16_manifest = loaded_manifests[
    "Step 16"
]


step17_manifest = loaded_manifests[
    "Step 17"
]


step17a_manifest = loaded_manifests[
    "Step 17A"
]


# ============================================================
# 10. IMPORTANT UPSTREAM LOGIC CHECKS
# ============================================================

logic_qc = pd.DataFrame(
    [
        {
            "Check":
                "Step 16 joint model-family selection",

            "PASS":
                bool(
                    step16_manifest.get(
                        "joint_model_family_selection",
                        False,
                    )
                ),
        },

        {
            "Check":
                "Step 16 has 120 joint candidates",

            "PASS":
                (
                    step16_manifest.get(
                        "total_joint_candidates"
                    )
                    ==
                    120
                ),
        },

        {
            "Check":
                "Model family not fixed before tuning",

            "PASS":
                (
                    step16_manifest.get(
                        "model_family_fixed_before_tuning"
                    )
                    is False
                ),
        },

        {
            "Check":
                "Step 17 winner-only calibration",

            "PASS":
                bool(
                    step17_manifest.get(
                        "winner_only_calibration",
                        False,
                    )
                ),
        },

        {
            "Check":
                "Step 17 outer test excluded from calibration",

            "PASS":
                (
                    step17_manifest.get(
                        "outer_test_used_for_calibration"
                    )
                    is False
                ),
        },

        {
            "Check":
                "Step 17A original predictor count = 15",

            "PASS":
                (
                    step17a_manifest.get(
                        "original_predictor_count"
                    )
                    ==
                    15
                ),
        },

        {
            "Check":
                "Step 17A full predictor primary pipeline",

            "PASS":
                bool(
                    step17a_manifest.get(
                        "full_predictor_primary_pipeline",
                        False,
                    )
                ),
        },

        {
            "Check":
                "Step 17A external feature selection disabled",

            "PASS":
                (
                    step17a_manifest.get(
                        "external_feature_selection"
                    )
                    is False
                ),
        },
    ]
)


if not logic_qc[
    "PASS"
].all():

    failed = logic_qc.loc[
        ~logic_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "Frozen-protocol logic check failed:\n\n"
        +
        failed.to_string(
            index=False
        )
    )


# ============================================================
# 11. PRIMARY MASTER STRUCTURAL CHECK
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


if master.shape != (
    5063,
    23,
):

    raise RuntimeError(
        "Unexpected primary-master shape: "
        f"{master.shape}"
    )


# ============================================================
# 12. RESTORE FROZEN PREDICTOR LISTS
# ============================================================

NUMERIC_PREDICTORS = [

    "maternal_age_at_index_birth_years",

    "prior_live_births",

    "household_size_members",
]


ORDERED_CATEGORICAL_PREDICTORS = [

    "maternal_education",

    "wealth_quintile",
]


NOMINAL_CATEGORICAL_PREDICTORS = [

    "partner_education_primary_final",

    "division",

    "pregnancy_intention",

    "healthcare_decision_primary_final",
]


BINARY_CATEGORICAL_PREDICTORS = [

    "residence",

    "weekly_media_exposure",

    "working_status",

    "religion_final",

    "household_head_sex",

    "distance_barrier",
]


CATEGORICAL_PREDICTORS = (

    ORDERED_CATEGORICAL_PREDICTORS
    +
    NOMINAL_CATEGORICAL_PREDICTORS
    +
    BINARY_CATEGORICAL_PREDICTORS
)


PREDICTORS = (

    NUMERIC_PREDICTORS
    +
    CATEGORICAL_PREDICTORS
)


if len(PREDICTORS) != 15:

    raise RuntimeError(
        "Recovered predictor count is not 15."
    )


missing_predictors = [

    predictor

    for predictor in PREDICTORS

    if predictor not in master.columns
]


if missing_predictors:

    raise RuntimeError(
        "Recovered predictor(s) absent from master:\n"
        f"{missing_predictors}"
    )


# ============================================================
# 13. PACKAGE VERSION CHECK
# ============================================================

def package_version(
    package_name,
):

    try:

        return importlib_metadata.version(
            package_name
        )

    except Exception:

        return "NOT INSTALLED"


package_versions = pd.DataFrame(
    [
        {
            "Package":
                "Python",

            "Version":
                sys.version.split()[0],
        },

        {
            "Package":
                "numpy",

            "Version":
                np.__version__,
        },

        {
            "Package":
                "pandas",

            "Version":
                pd.__version__,
        },

        {
            "Package":
                "scikit-learn",

            "Version":
                sklearn.__version__,
        },

        {
            "Package":
                "xgboost",

            "Version":
                package_version(
                    "xgboost"
                ),
        },

        {
            "Package":
                "lightgbm",

            "Version":
                package_version(
                    "lightgbm"
                ),
        },

        {
            "Package":
                "catboost",

            "Version":
                package_version(
                    "catboost"
                ),
        },
    ]
)


# ============================================================
# 14. FINAL RECOVERY QC
# ============================================================

recovery_qc = pd.DataFrame(
    [
        {
            "Check":
                "Project root found",

            "PASS":
                ROOT.exists(),
        },

        {
            "Check":
                "All project directories found",

            "PASS":
                len(
                    missing_directories
                )
                ==
                0,
        },

        {
            "Check":
                "Primary master hash verified",

            "PASS":
                observed_primary_hash
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash verified",

            "PASS":
                observed_outer_hash
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash verified",

            "PASS":
                observed_inner_hash
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "All required manifests frozen",

            "PASS":
                bool(
                    manifest_qc[
                        "PASS"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Upstream modelling logic restored",

            "PASS":
                bool(
                    logic_qc[
                        "PASS"
                    ]
                    .all()
                ),
        },

        {
            "Check":
                "Primary master shape = 5063 x 23",

            "PASS":
                master.shape
                ==
                (
                    5063,
                    23,
                ),
        },

        {
            "Check":
                "15 frozen predictors restored",

            "PASS":
                len(PREDICTORS)
                ==
                15,
        },

        {
            "Check":
                "All frozen predictors available",

            "PASS":
                len(
                    missing_predictors
                )
                ==
                0,
        },

        {
            "Check":
                "No model fitted during recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No performance inspected during recovery",

            "PASS":
                True,
        },
    ]
)


if not recovery_qc[
    "PASS"
].all():

    failed = recovery_qc.loc[
        ~recovery_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "KERNEL RECOVERY FAILED:\n\n"
        +
        failed.to_string(
            index=False
        )
    )


# ============================================================
# 15. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "KERNEL RESTART RECOVERY — FROZEN PROJECT STATE"
)


print(
    "=" * 78
)


print(
    "\nPACKAGE VERSIONS:"
)


display(
    package_versions
)


print(
    "\nUPSTREAM MANIFEST STATUS:"
)


display(
    manifest_qc
)


print(
    "\nUPSTREAM LOGIC QC:"
)


display(
    logic_qc
)


print(
    "\nRECOVERED PREDICTORS:"
)


predictor_table = pd.DataFrame(
    {
        "Predictor":
            PREDICTORS,

        "Type":
            (
                ["Numeric"] * 3
                +
                ["Ordered categorical"] * 2
                +
                ["Nominal categorical"] * 4
                +
                ["Binary categorical"] * 6
            ),
    }
)


display(
    predictor_table
)


print(
    "\nFINAL RECOVERY QC:"
)


display(
    recovery_qc
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : {observed_primary_hash}"
)


print(
    f"Outer split    : {observed_outer_hash}"
)


print(
    f"Inner split    : {observed_inner_hash}"
)


print(
    "\nIMPORTANT:"
)


print(
    "- Kernel memory has been rebuilt from frozen files."
)


print(
    "- Steps 1-17A were NOT rerun or regenerated."
)


print(
    "- Existing frozen files remain authoritative."
)


print(
    "- No model was fitted."
)


print(
    "- No predictive performance was inspected."
)


print(
    "- Ready to continue with corrected Step 18 threshold optimization."
)


print(
    "=" * 78
)


print(
    "STATUS: KERNEL RECOVERY PASS"
)


print(
    "=" * 78
)


KERNEL RESTART RECOVERY — FROZEN PROJECT STATE

PACKAGE VERSIONS:


,Package,Version
0,Python,3.13.9
1,numpy,2.3.5
2,pandas,3.0.5
3,scikit-learn,1.9.0
4,xgboost,3.4.1
5,lightgbm,4.6.0
6,catboost,1.2.10



UPSTREAM MANIFEST STATUS:


,Step,Expected_Status,Observed_Status,PASS
0,Step 14,PREPROCESSING_PROTOCOL_FROZEN,PREPROCESSING_PROTOCOL_FROZEN,True
1,Step 15,HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN,HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN,True
2,Step 15A,SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN,SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN,True
3,Step 16,JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_...,JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_...,True
4,Step 17,JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_F...,JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_F...,True
5,Step 17A,FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_P...,FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_P...,True



UPSTREAM LOGIC QC:


,Check,PASS
0,Step 16 joint model-family selection,True
1,Step 16 has 120 joint candidates,True
2,Model family not fixed before tuning,True
3,Step 17 winner-only calibration,True
4,Step 17 outer test excluded from calibration,True
5,Step 17A original predictor count = 15,True
6,Step 17A full predictor primary pipeline,True
7,Step 17A external feature selection disabled,True



RECOVERED PREDICTORS:


,Predictor,Type
0,maternal_age_at_index_birth_years,Numeric
1,prior_live_births,Numeric
2,household_size_members,Numeric
3,maternal_education,Ordered categorical
4,wealth_quintile,Ordered categorical
5,partner_education_primary_final,Nominal categorical
6,division,Nominal categorical
7,pregnancy_intention,Nominal categorical
8,healthcare_decision_primary_final,Nominal categorical
9,residence,Binary categorical



FINAL RECOVERY QC:


,Check,PASS
0,Project root found,True
1,All project directories found,True
2,Primary master hash verified,True
3,Outer split hash verified,True
4,Inner split hash verified,True
5,All required manifests frozen,True
6,Upstream modelling logic restored,True
7,Primary master shape = 5063 x 23,True
8,15 frozen predictors restored,True
9,All frozen predictors available,True



INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Outer split    : 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
Inner split    : 2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633

IMPORTANT:
- Kernel memory has been rebuilt from frozen files.
- Steps 1-17A were NOT rerun or regenerated.
- Existing frozen files remain authoritative.
- No model was fitted.
- No predictive performance was inspected.
- Ready to continue with corrected Step 18 threshold optimization.
STATUS: KERNEL RECOVERY PASS


In [2]:
# ============================================================
# STEP 18 — CROSS-VALIDATED THRESHOLD OPTIMIZATION
#           PROTOCOL FREEZE
# ============================================================
#
# IMPORTANT:
# ----------
# Threshold VALUES are NOT fixed.
#
# This step freezes only HOW thresholds will later be optimized.
#
# For every:
#   pipeline x outer fold x hierarchical stage
#
# workflow:
#
#   Step-16 joint winner
#          ↓
#   Step-17 calibration
#          ↓
#   calibrated INNER-OOF probabilities
#          ↓
#   threshold optimization
#          ↓
#   optimized t1 / t2
#          ↓
#   untouched OUTER-TEST application
#
#
# Stage 1:
#   positive = No ANC
#
# Stage 2:
#   positive = >=8 ANC visits
#
#
# Optimization criterion:
#
#   survey-weighted balanced accuracy
#
#   BA = (sensitivity + specificity) / 2
#
# This has identical threshold ranking to Youden J.
#
#
# Threshold search:
#   ALL classification-changing operating points are evaluated.
#
# No:
#   - fixed 0.5 threshold
#   - coarse threshold grid
#   - random threshold search
#   - outer-test threshold optimization
#   - pooled outer-OOF threshold reoptimization
#
#
# Threshold-dependent metrics remain SECONDARY.
#
# No model fitting occurs in this step.
# ============================================================


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from pathlib import Path
import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 2. FROZEN INPUTS
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)

OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)

INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)

STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)

STEP17_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_evaluation_manifest.json"
)

STEP17A_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_predictor_feature_selection_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)

EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)

EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


# ============================================================
# 3. INPUT / HASH CHECK
# ============================================================

for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP16_MANIFEST_FILE,
    STEP17_MANIFEST_FILE,
    STEP17A_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen input missing:\n{path}"
        )


if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


# ============================================================
# 4. UPSTREAM STATUS
# ============================================================

step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)

step17_manifest = read_json(
    STEP17_MANIFEST_FILE
)

step17a_manifest = read_json(
    STEP17A_MANIFEST_FILE
)


if (
    step16_manifest.get("status")
    !=
    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Revised Step 16 is not frozen."
    )


if (
    step17_manifest.get("status")
    !=
    "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 17 is not frozen."
    )


if (
    step17a_manifest.get("status")
    !=
    "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 17A is not frozen."
    )


if (
    step17a_manifest.get(
        "original_predictor_count"
    )
    !=
    15
):

    raise RuntimeError(
        "Frozen predictor count is not 15."
    )


# ============================================================
# 5. THRESHOLD OPTIMIZATION CONSTANTS
# ============================================================

THRESHOLD_OPTIMIZATION = True

THRESHOLD_VALUE_FIXED = False

DEFAULT_THRESHOLD_0_5 = False

THRESHOLD_OBJECTIVE = (
    "survey_weighted_balanced_accuracy"
)

THRESHOLD_ROLE = (
    "secondary_statistical_operating_point"
)

THRESHOLD_TIE_TOLERANCE = 1e-12

THRESHOLD_NUMERICAL_CLIP = 1e-12


# ============================================================
# 6. PROBABILITY VALIDATION
# ============================================================

def prepare_threshold_probabilities(
    probability,
):

    probability = np.asarray(
        probability,
        dtype=float,
    )


    if probability.ndim != 1:

        raise ValueError(
            "Probability vector must be one-dimensional."
        )


    if len(probability) == 0:

        raise ValueError(
            "Probability vector is empty."
        )


    if not np.isfinite(
        probability
    ).all():

        raise ValueError(
            "Non-finite probability detected."
        )


    if (
        (probability < 0.0)
        |
        (probability > 1.0)
    ).any():

        raise ValueError(
            "Probability outside [0,1]."
        )


    # Used only to make boundary operating states unambiguous.
    # This does NOT modify Step-17 probability-performance metrics.
    return np.clip(
        probability,
        THRESHOLD_NUMERICAL_CLIP,
        1.0 - THRESHOLD_NUMERICAL_CLIP,
    )


# ============================================================
# 7. EXACT CLASSIFICATION-CHANGING THRESHOLDS
# ============================================================

def exact_threshold_candidates(
    calibrated_probability,
):

    probability = (
        prepare_threshold_probabilities(
            calibrated_probability
        )
    )


    unique_probability = np.unique(
        probability
    )


    if len(unique_probability) == 1:

        # all-positive and all-negative states
        return np.array(
            [
                0.0,
                1.0,
            ],
            dtype=float,
        )


    midpoints = (
        unique_probability[:-1]
        +
        (
            unique_probability[1:]
            -
            unique_probability[:-1]
        ) / 2.0
    )


    candidates = np.concatenate(
        [
            np.array(
                [0.0],
                dtype=float,
            ),

            midpoints,

            np.array(
                [1.0],
                dtype=float,
            ),
        ]
    )


    candidates = np.unique(
        candidates
    )


    return candidates


# ============================================================
# 8. SURVEY-WEIGHTED BINARY METRICS
# ============================================================

def weighted_binary_threshold_metrics(
    y_true,
    calibrated_probability,
    sample_weight,
    threshold,
):

    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probability = (
        prepare_threshold_probabilities(
            calibrated_probability
        )
    )

    sample_weight = np.asarray(
        sample_weight,
        dtype=float,
    )


    if not (
        len(y_true)
        ==
        len(probability)
        ==
        len(sample_weight)
    ):

        raise ValueError(
            "Input lengths differ."
        )


    if set(
        np.unique(y_true)
    ) != {0, 1}:

        raise ValueError(
            "Both binary classes must be present."
        )


    if not np.isfinite(
        sample_weight
    ).all():

        raise ValueError(
            "Non-finite sample weight detected."
        )


    if (
        sample_weight
        <=
        0
    ).any():

        raise ValueError(
            "Survey weights must be strictly positive."
        )


    threshold = float(
        threshold
    )


    if not (
        0.0
        <=
        threshold
        <=
        1.0
    ):

        raise ValueError(
            "Threshold outside [0,1]."
        )


    predicted = (
        probability
        >=
        threshold
    ).astype(int)


    positive = (
        y_true
        ==
        1
    )

    negative = (
        y_true
        ==
        0
    )


    tp = float(
        sample_weight[
            positive
            &
            (predicted == 1)
        ].sum()
    )

    fn = float(
        sample_weight[
            positive
            &
            (predicted == 0)
        ].sum()
    )

    tn = float(
        sample_weight[
            negative
            &
            (predicted == 0)
        ].sum()
    )

    fp = float(
        sample_weight[
            negative
            &
            (predicted == 1)
        ].sum()
    )


    sensitivity = (
        tp
        /
        (tp + fn)
    )

    specificity = (
        tn
        /
        (tn + fp)
    )


    balanced_accuracy = (
        sensitivity
        +
        specificity
    ) / 2.0


    youden_j = (
        sensitivity
        +
        specificity
        -
        1.0
    )


    balance_floor = min(
        sensitivity,
        specificity,
    )


    total_weight = (
        tp
        +
        fn
        +
        tn
        +
        fp
    )


    accuracy = (
        tp
        +
        tn
    ) / total_weight


    if (
        tp
        +
        fp
    ) > 0:

        ppv = (
            tp
            /
            (tp + fp)
        )

    else:

        ppv = np.nan


    if (
        tn
        +
        fn
    ) > 0:

        npv = (
            tn
            /
            (tn + fn)
        )

    else:

        npv = np.nan


    if (
        2.0 * tp
        +
        fp
        +
        fn
    ) > 0:

        f1 = (
            2.0 * tp
            /
            (
                2.0 * tp
                +
                fp
                +
                fn
            )
        )

    else:

        f1 = np.nan


    return {

        "threshold":
            threshold,

        "sensitivity":
            float(sensitivity),

        "specificity":
            float(specificity),

        "balanced_accuracy":
            float(balanced_accuracy),

        "youden_j":
            float(youden_j),

        "balance_floor":
            float(balance_floor),

        "accuracy":
            float(accuracy),

        "ppv":
            (
                float(ppv)
                if np.isfinite(ppv)
                else np.nan
            ),

        "npv":
            (
                float(npv)
                if np.isfinite(npv)
                else np.nan
            ),

        "f1":
            (
                float(f1)
                if np.isfinite(f1)
                else np.nan
            ),

        "weighted_tp":
            tp,

        "weighted_fn":
            fn,

        "weighted_tn":
            tn,

        "weighted_fp":
            fp,
    }


# ============================================================
# 9. EXACT THRESHOLD OPTIMIZER
# ============================================================

def optimize_weighted_threshold(
    y_true,
    calibrated_probability,
    sample_weight,
):

    thresholds = exact_threshold_candidates(
        calibrated_probability
    )


    rows = []


    for threshold in thresholds:

        rows.append(
            weighted_binary_threshold_metrics(
                y_true=y_true,
                calibrated_probability=calibrated_probability,
                sample_weight=sample_weight,
                threshold=threshold,
            )
        )


    results = pd.DataFrame(
        rows
    )


    # --------------------------------------------------------
    # 1. Highest weighted balanced accuracy
    # --------------------------------------------------------

    best_ba = float(
        results[
            "balanced_accuracy"
        ].max()
    )


    tied = results.loc[
        np.isclose(
            results[
                "balanced_accuracy"
            ],
            best_ba,
            rtol=0.0,
            atol=THRESHOLD_TIE_TOLERANCE,
        )
    ].copy()


    # --------------------------------------------------------
    # 2. If exactly tied, prefer the operating point with
    #    better balance between sensitivity and specificity.
    # --------------------------------------------------------

    best_floor = float(
        tied[
            "balance_floor"
        ].max()
    )


    tied = tied.loc[
        np.isclose(
            tied[
                "balance_floor"
            ],
            best_floor,
            rtol=0.0,
            atol=THRESHOLD_TIE_TOLERANCE,
        )
    ].copy()


    # --------------------------------------------------------
    # 3. Final deterministic tie rule.
    #
    # No 0.5 reference is used.
    # Lower threshold is used only if all optimization
    # quantities above are numerically tied.
    # --------------------------------------------------------

    selected = (
        tied
        .sort_values(
            "threshold",
            ascending=True,
        )
        .iloc[0]
    )


    return {

        "selected_threshold":
            float(
                selected[
                    "threshold"
                ]
            ),

        "optimized_balanced_accuracy":
            float(
                selected[
                    "balanced_accuracy"
                ]
            ),

        "sensitivity":
            float(
                selected[
                    "sensitivity"
                ]
            ),

        "specificity":
            float(
                selected[
                    "specificity"
                ]
            ),

        "youden_j":
            float(
                selected[
                    "youden_j"
                ]
            ),

        "candidate_threshold_count":
            int(
                len(results)
            ),

        "threshold_was_optimized":
            True,

        "threshold_was_fixed":
            False,
    }


# ============================================================
# 10. SYNTHETIC OPTIMIZER QC
# ============================================================
#
# Synthetic data only.
# NO study predictions are inspected.
# ============================================================

synthetic_y = np.array(
    [
        0, 0, 0, 0, 0,
        1, 1, 1, 1, 1,
    ],
    dtype=int,
)


synthetic_probability = np.array(
    [
        0.03,
        0.08,
        0.20,
        0.38,
        0.60,
        0.18,
        0.45,
        0.62,
        0.78,
        0.91,
    ],
    dtype=float,
)


synthetic_weight = np.array(
    [
        1.0,
        1.1,
        0.9,
        1.2,
        0.8,
        1.0,
        1.3,
        0.7,
        1.1,
        1.0,
    ],
    dtype=float,
)


synthetic_result = (
    optimize_weighted_threshold(
        y_true=synthetic_y,
        calibrated_probability=synthetic_probability,
        sample_weight=synthetic_weight,
    )
)


if not synthetic_result[
    "threshold_was_optimized"
]:

    raise RuntimeError(
        "Synthetic optimization failed."
    )


if synthetic_result[
    "threshold_was_fixed"
]:

    raise RuntimeError(
        "Synthetic threshold incorrectly marked fixed."
    )


# ============================================================
# 11. FORMAL OPTIMIZATION PROTOCOL
# ============================================================

threshold_optimization_protocol = {

    "threshold_optimization":
        True,

    "threshold_value_fixed":
        False,

    "default_threshold_0_5":
        False,

    "role":
        THRESHOLD_ROLE,

    "optimization_after_joint_model_selection":
        True,

    "optimization_after_calibration":
        True,

    "optimization_source":
        (
            "selected winner calibrated pooled "
            "inner-OOF probabilities"
        ),

    "optimization_labels":
        (
            "corresponding outcomes in current "
            "outer-training population only"
        ),

    "optimization_weight":
        "original DHS SURVEY_WEIGHT",

    "weight_normalization_needed":
        False,

    "weight_normalization_reason":
        (
            "Sensitivity and specificity are within-class "
            "weighted rates and are invariant to common "
            "positive scaling of the weights."
        ),

    "objective":
        THRESHOLD_OBJECTIVE,

    "objective_formula":
        (
            "(weighted sensitivity + weighted specificity) / 2"
        ),

    "youden_equivalence":
        (
            "Youden J = 2*balanced_accuracy - 1"
        ),

    "candidate_strategy":
        (
            "exhaustive evaluation of all classification-changing "
            "operating points"
        ),

    "coarse_threshold_grid":
        False,

    "random_threshold_search":
        False,

    "fixed_threshold":
        False,

    "f1_optimized":
        False,

    "raw_accuracy_optimized":
        False,

    "outer_test_optimization":
        False,

    "pooled_outer_oof_optimization":
        False,

    "stage_specific_thresholds":
        True,

    "outer_fold_specific_thresholds":
        True,

    "pipeline_specific_thresholds":
        True,

    "same_threshold_stage1_stage2":
        False,

    "same_threshold_all_outer_folds":
        False,

    "same_threshold_primary_sensitivity":
        False,

    "threshold_can_change_model_family":
        False,

    "threshold_can_change_hyperparameters":
        False,

    "threshold_can_change_calibration_method":
        False,

    "tie_rule_1":
        "highest survey-weighted balanced accuracy",

    "tie_rule_2":
        (
            "highest minimum of weighted sensitivity "
            "and weighted specificity"
        ),

    "tie_rule_3":
        (
            "lower threshold if still exactly tied; "
            "deterministic rule only"
        ),

    "clinical_optimal_threshold_claim":
        False,
}


# ============================================================
# 12. STAGE-SPECIFIC POLICY
# ============================================================

stage_threshold_policy = pd.DataFrame(
    [
        {
            "Stage":
                "Stage 1",

            "Positive_Class":
                "No ANC",

            "Development_Population":
                (
                    "All observations in current "
                    "outer-training set"
                ),

            "Probability":
                "calibrated q0",

            "Threshold":
                "optimized t1",

            "Objective":
                (
                    "survey-weighted balanced accuracy"
                ),

            "Fixed":
                False,
        },

        {
            "Stage":
                "Stage 2",

            "Positive_Class":
                ">=8 ANC visits",

            "Development_Population":
                (
                    "True ANC users in current "
                    "outer-training set"
                ),

            "Probability":
                "calibrated q2",

            "Threshold":
                "optimized t2",

            "Objective":
                (
                    "survey-weighted balanced accuracy"
                ),

            "Fixed":
                False,
        },
    ]
)


# ============================================================
# 13. FINAL HIERARCHICAL CLASSIFICATION RULE
# ============================================================

hierarchical_classification_protocol = {

    "threshold_values_fixed":
        False,

    "stage1":
        (
            "if q0 >= optimized t1, "
            "predict class 0: No ANC"
        ),

    "stage2":
        (
            "otherwise, if q2 >= optimized t2, "
            "predict class 2: >=8 visits"
        ),

    "remaining":
        (
            "otherwise predict class 1: 1-7 visits"
        ),

    "stage1_priority":
        True,

    "outer_test_reoptimization":
        False,

    "posthoc_rule_change":
        False,
}


# ============================================================
# 14. OUTER-TEST APPLICATION POLICY
# ============================================================

outer_threshold_application = {

    "threshold_origin":
        (
            "current outer fold's development data only"
        ),

    "outer_fold_specific":
        True,

    "stage_specific":
        True,

    "pipeline_specific":
        True,

    "outer_test_stage1":
        (
            "apply previously optimized t1 to outer-test q0"
        ),

    "outer_test_stage2":
        (
            "apply previously optimized t2 to outer-test q2"
        ),

    "outer_test_threshold_fit":
        False,

    "outer_test_threshold_adjustment":
        False,

    "outer_test_prevalence_adjustment":
        False,

    "outer_test_metric_guided_adjustment":
        False,

    "pooled_outer_oof_reoptimization":
        False,
}


# ============================================================
# 15. THRESHOLD-DEPENDENT METRICS
# ============================================================

binary_threshold_metrics = pd.DataFrame(
    [
        {
            "Metric":
                "Survey-weighted sensitivity",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted specificity",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted balanced accuracy",

            "Role":
                "Secondary; optimization objective",
        },

        {
            "Metric":
                "Survey-weighted Youden J",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted PPV",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted NPV",

            "Role":
                "Secondary",
        },

        {
            "Metric":
                "Survey-weighted F1",

            "Role":
                "Secondary; not optimized",
        },

        {
            "Metric":
                "Survey-weighted accuracy",

            "Role":
                "Secondary; not optimized",
        },
    ]
)


multiclass_threshold_metrics = pd.DataFrame(
    [
        {
            "Metric":
                "Survey-weighted overall accuracy",
        },

        {
            "Metric":
                "Survey-weighted recall by class",
        },

        {
            "Metric":
                "Survey-weighted precision by class",
        },

        {
            "Metric":
                "Survey-weighted F1 by class",
        },

        {
            "Metric":
                "Macro recall / multiclass balanced accuracy",
        },

        {
            "Metric":
                "Macro F1",
        },

        {
            "Metric":
                "Survey-weighted 3x3 confusion matrix",
        },
    ]
)


# ============================================================
# 16. PRIMARY vs SECONDARY ANALYSIS HIERARCHY
# ============================================================

analysis_hierarchy = pd.DataFrame(
    [
        {
            "Analysis":
                "Average Precision",

            "Role":
                "Primary discrimination",
        },

        {
            "Analysis":
                "ROC-AUC",

            "Role":
                "Secondary discrimination",
        },

        {
            "Analysis":
                "Log loss",

            "Role":
                "Primary probabilistic performance",
        },

        {
            "Analysis":
                "Brier score",

            "Role":
                "Secondary probabilistic performance",
        },

        {
            "Analysis":
                (
                    "Calibration intercept, slope, "
                    "O/E and calibration curve"
                ),

            "Role":
                "Primary calibration assessment",
        },

        {
            "Analysis":
                (
                    "Optimized balanced-accuracy threshold"
                ),

            "Role":
                "Secondary operating-point analysis",
        },

        {
            "Analysis":
                (
                    "Sensitivity, specificity, PPV, "
                    "NPV, F1 and accuracy"
                ),

            "Role":
                "Secondary classification performance",
        },
    ]
)


# ============================================================
# 17. THRESHOLD STABILITY REPORTING
# ============================================================

threshold_stability_protocol = {

    "report_each_outer_fold_threshold":
        True,

    "report_stage1_thresholds":
        True,

    "report_stage2_thresholds":
        True,

    "report_primary_pipeline":
        True,

    "report_unweighted_sensitivity_pipeline":
        True,

    "summary":
        [
            "median",
            "IQR",
            "minimum",
            "maximum",
        ],

    "stability_used_to_change_model":
        False,

    "stability_used_to_change_objective":
        False,
}


# ============================================================
# 18. DECISION-CURVE BOUNDARY
# ============================================================

decision_curve_protocol = {

    "primary_analysis":
        False,

    "performed_in_step18":
        False,

    "balanced_accuracy_threshold_is_clinical_threshold":
        False,

    "clinically_optimal_threshold_claim":
        False,

    "reason":
        (
            "No prespecified downstream intervention, "
            "false-positive versus false-negative cost ratio, "
            "or clinically justified action-threshold range."
        ),
}


# ============================================================
# 19. FINAL FULL-DATA THRESHOLD POLICY
# ============================================================

final_threshold_protocol = {

    "outer_fold_thresholds_are_final_deployment_thresholds":
        False,

    "final_threshold_value_fixed_now":
        False,

    "final_threshold_optimization_required":
        True,

    "final_method":
        (
            "same frozen survey-weighted balanced-accuracy "
            "optimization procedure"
        ),

    "final_threshold_source":
        (
            "cross-fitted calibrated development predictions "
            "after final full-data model selection"
        ),

    "final_threshold_selected_from_outer_test":
        False,

    "objective_change_allowed":
        False,

    "tie_rule_change_allowed":
        False,
}


# ============================================================
# 20. REFERENCE REGISTRY
# ============================================================

reference_registry = pd.DataFrame(
    [
        {
            "ID":
                "SKLEARN_THRESHOLD",

            "Reference":
                (
                    "Scikit-learn 1.9 documentation: "
                    "Tuning the decision threshold for "
                    "class prediction."
                ),

            "Use":
                (
                    "Cross-validated post-tuning of "
                    "classification thresholds."
                ),
        },

        {
            "ID":
                "EFTHIMIOU_2024",

            "Reference":
                (
                    "Efthimiou O, Seo M, Chalkou K, et al. "
                    "Developing clinical prediction models: "
                    "a step-by-step guide. "
                    "BMJ. 2024;386:e078276."
                ),

            "DOI":
                "10.1136/bmj-2023-078276",

            "Use":
                (
                    "Avoid data-driven cut-offs using "
                    "evaluation data and distinguish predictive "
                    "performance from clinical decision utility."
                ),
        },
    ]
)


# ============================================================
# 21. MANUSCRIPT-READY METHODS
# ============================================================

methods_text = """
Decision-threshold optimization

Decision-threshold values were not fixed in advance. Instead, the
threshold-optimization procedure was prespecified before model fitting.
For each hierarchical stage, analysis pipeline, and outer
cross-validation fold, the operating threshold will be optimized only
after joint model-family and hyperparameter selection and probability
calibration.

Threshold optimization will use the selected winner's calibrated,
cross-fitted inner-validation probabilities from the corresponding
outer-training population. Original DHS sampling weights will be used
in the optimization criterion. Outer-test observations will not be used
to select, adjust, or re-estimate decision thresholds.

The optimization criterion will be survey-weighted balanced accuracy,
defined as the arithmetic mean of weighted sensitivity and weighted
specificity. Maximizing weighted balanced accuracy is equivalent in
threshold ranking to maximizing the weighted Youden index. Thresholds
will not be restricted to 0.5 or to a coarse numerical grid. Instead,
all classification-changing operating points defined by the distinct
calibrated inner-validation probabilities will be evaluated.

Stage-1 and Stage-2 thresholds will be optimized independently. For
Stage 1, the positive outcome is no antenatal care. For Stage 2, the
positive outcome is eight or more reported antenatal-care visits and
threshold optimization will be restricted to true antenatal-care users
within the corresponding development population. Thresholds will also
be optimized independently within each outer fold and separately for
the primary survey-weighted model-fitting pipeline and the
unweighted-fitting sensitivity analysis.

For final hierarchical class assignment, observations with calibrated
Stage-1 probability q0 at or above the optimized Stage-1 threshold t1
will be classified as no ANC. Among the remaining observations, those
with calibrated conditional Stage-2 probability q2 at or above the
optimized Stage-2 threshold t2 will be classified as eight or more
visits; all remaining observations will be classified as one to seven
visits.

Threshold-derived sensitivity, specificity, balanced accuracy,
positive and negative predictive values, F1 score, accuracy, and
three-category classification measures will be treated as secondary
performance summaries. Primary evaluation will remain based on
discrimination, probabilistic performance, and calibration. Optimized
thresholds will be described as statistical operating points rather
than clinically optimal action thresholds.
""".strip()


# ============================================================
# 22. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Joint Step 16 frozen",

            "PASS":
                (
                    step16_manifest.get("status")
                    ==
                    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Step 17 frozen",

            "PASS":
                (
                    step17_manifest.get("status")
                    ==
                    "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Step 17A frozen",

            "PASS":
                (
                    step17a_manifest.get("status")
                    ==
                    "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN"
                ),
        },

        {
            "Check":
                "Threshold optimization enabled",

            "PASS":
                THRESHOLD_OPTIMIZATION,
        },

        {
            "Check":
                "Threshold values NOT fixed",

            "PASS":
                not THRESHOLD_VALUE_FIXED,
        },

        {
            "Check":
                "No default 0.5 threshold",

            "PASS":
                not DEFAULT_THRESHOLD_0_5,
        },

        {
            "Check":
                "Optimization occurs after joint selection",

            "PASS":
                threshold_optimization_protocol[
                    "optimization_after_joint_model_selection"
                ],
        },

        {
            "Check":
                "Optimization occurs after calibration",

            "PASS":
                threshold_optimization_protocol[
                    "optimization_after_calibration"
                ],
        },

        {
            "Check":
                "Objective = weighted balanced accuracy",

            "PASS":
                (
                    threshold_optimization_protocol[
                        "objective"
                    ]
                    ==
                    "survey_weighted_balanced_accuracy"
                ),
        },

        {
            "Check":
                "No coarse threshold grid",

            "PASS":
                not threshold_optimization_protocol[
                    "coarse_threshold_grid"
                ],
        },

        {
            "Check":
                "No random threshold search",

            "PASS":
                not threshold_optimization_protocol[
                    "random_threshold_search"
                ],
        },

        {
            "Check":
                "No F1 threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "f1_optimized"
                ],
        },

        {
            "Check":
                "No raw accuracy threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "raw_accuracy_optimized"
                ],
        },

        {
            "Check":
                "No outer-test threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "outer_test_optimization"
                ],
        },

        {
            "Check":
                "No pooled outer-OOF threshold optimization",

            "PASS":
                not threshold_optimization_protocol[
                    "pooled_outer_oof_optimization"
                ],
        },

        {
            "Check":
                "Stage-specific thresholds",

            "PASS":
                threshold_optimization_protocol[
                    "stage_specific_thresholds"
                ],
        },

        {
            "Check":
                "Outer-fold-specific thresholds",

            "PASS":
                threshold_optimization_protocol[
                    "outer_fold_specific_thresholds"
                ],
        },

        {
            "Check":
                "Pipeline-specific thresholds",

            "PASS":
                threshold_optimization_protocol[
                    "pipeline_specific_thresholds"
                ],
        },

        {
            "Check":
                "Threshold cannot reselect model family",

            "PASS":
                not threshold_optimization_protocol[
                    "threshold_can_change_model_family"
                ],
        },

        {
            "Check":
                "Threshold cannot reselect hyperparameters",

            "PASS":
                not threshold_optimization_protocol[
                    "threshold_can_change_hyperparameters"
                ],
        },

        {
            "Check":
                "Stage-1 hierarchical priority frozen",

            "PASS":
                hierarchical_classification_protocol[
                    "stage1_priority"
                ],
        },

        {
            "Check":
                "No outer-test threshold refit",

            "PASS":
                not outer_threshold_application[
                    "outer_test_threshold_fit"
                ],
        },

        {
            "Check":
                "No pooled outer-test threshold reoptimization",

            "PASS":
                not outer_threshold_application[
                    "pooled_outer_oof_reoptimization"
                ],
        },

        {
            "Check":
                "Synthetic threshold optimized",

            "PASS":
                (
                    synthetic_result[
                        "threshold_was_optimized"
                    ]
                    and
                    not synthetic_result[
                        "threshold_was_fixed"
                    ]
                ),
        },

        {
            "Check":
                "No clinical-optimal-threshold claim",

            "PASS":
                not threshold_optimization_protocol[
                    "clinical_optimal_threshold_claim"
                ],
        },

        {
            "Check":
                "No model fitted in Step 18",

            "PASS":
                True,
        },

        {
            "Check":
                "No study performance inspected in Step 18",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 18 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 23. SAVE OUTPUTS
# ============================================================

THRESHOLD_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_protocol.json"
)

STAGE_POLICY_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_stage_specific_threshold_policy.csv"
)

HIERARCHICAL_RULE_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_hierarchical_classification_rule.json"
)

OUTER_APPLICATION_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_outer_threshold_application.json"
)

BINARY_METRIC_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_binary_threshold_metric_registry.csv"
)

MULTICLASS_METRIC_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_multiclass_threshold_metric_registry.csv"
)

ANALYSIS_HIERARCHY_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_primary_secondary_analysis_hierarchy.csv"
)

THRESHOLD_STABILITY_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_stability_protocol.json"
)

DECISION_CURVE_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_decision_curve_boundary.json"
)

FINAL_THRESHOLD_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_full_data_threshold_optimization_policy.json"
)

REFERENCE_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_reference_registry.csv"
)

METHODS_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_manuscript_threshold_optimization_methods.txt"
)

HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP18_FINAL_hard_qc.csv"
)


stage_threshold_policy.to_csv(
    STAGE_POLICY_FILE,
    index=False,
)

binary_threshold_metrics.to_csv(
    BINARY_METRIC_FILE,
    index=False,
)

multiclass_threshold_metrics.to_csv(
    MULTICLASS_METRIC_FILE,
    index=False,
)

analysis_hierarchy.to_csv(
    ANALYSIS_HIERARCHY_FILE,
    index=False,
)

reference_registry.to_csv(
    REFERENCE_FILE,
    index=False,
)

hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


for path, payload in [
    (
        THRESHOLD_PROTOCOL_FILE,
        threshold_optimization_protocol,
    ),
    (
        HIERARCHICAL_RULE_FILE,
        hierarchical_classification_protocol,
    ),
    (
        OUTER_APPLICATION_FILE,
        outer_threshold_application,
    ),
    (
        THRESHOLD_STABILITY_FILE,
        threshold_stability_protocol,
    ),
    (
        DECISION_CURVE_FILE,
        decision_curve_protocol,
    ),
    (
        FINAL_THRESHOLD_FILE,
        final_threshold_protocol,
    ),
]:

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
        )


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
    )


# ============================================================
# 24. MANIFEST
# ============================================================

manifest = {

    "step":
        "STEP 18",

    "status":
        "THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "primary_master_sha256":
        sha256_file(PRIMARY_FILE),

    "outer_split_sha256":
        sha256_file(OUTER_SPLIT_FILE),

    "inner_split_sha256":
        sha256_file(INNER_SPLIT_FILE),

    "predictor_count":
        15,

    "threshold_optimization":
        True,

    "threshold_value_fixed":
        False,

    "default_threshold_0_5":
        False,

    "objective":
        THRESHOLD_OBJECTIVE,

    "threshold_source":
        (
            "selected-winner calibrated inner-OOF probabilities"
        ),

    "threshold_weight":
        "original DHS SURVEY_WEIGHT",

    "stage_specific_thresholds":
        True,

    "outer_fold_specific_thresholds":
        True,

    "pipeline_specific_thresholds":
        True,

    "exact_operating_point_search":
        True,

    "outer_test_used_for_optimization":
        False,

    "pooled_outer_oof_used_for_optimization":
        False,

    "threshold_analysis_role":
        "secondary",

    "clinical_optimal_threshold_claim":
        False,

    "model_fitting_performed":
        False,

    "study_performance_evaluated":
        False,

    "threshold_protocol_sha256":
        sha256_file(
            THRESHOLD_PROTOCOL_FILE
        ),

    "stage_policy_sha256":
        sha256_file(
            STAGE_POLICY_FILE
        ),

    "hierarchical_rule_sha256":
        sha256_file(
            HIERARCHICAL_RULE_FILE
        ),

    "outer_application_sha256":
        sha256_file(
            OUTER_APPLICATION_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),

    "methods_sha256":
        sha256_file(
            METHODS_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 25. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 18."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 18."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 18."
    )


# ============================================================
# 26. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 18 — CROSS-VALIDATED THRESHOLD OPTIMIZATION FREEZE"
)

print(
    "=" * 78
)


print(
    "\nKEY POLICY:"
)

print(
    "Threshold optimization : ENABLED"
)

print(
    "Threshold values fixed : NO"
)

print(
    "Default threshold 0.5 : NO"
)


print(
    "\nSTAGE-SPECIFIC POLICY:"
)

display(
    stage_threshold_policy
)


print(
    "\nOPTIMIZATION RULE:"
)

print(
    "- Joint model/hyperparameter selection occurs first."
)

print(
    "- Probability calibration occurs next."
)

print(
    "- Threshold is optimized from calibrated INNER-OOF probabilities."
)

print(
    "- Objective = survey-weighted balanced accuracy."
)

print(
    "- Every classification-changing operating point is evaluated."
)

print(
    "- Stage 1 and Stage 2 thresholds are optimized independently."
)

print(
    "- Each outer fold receives independently optimized thresholds."
)

print(
    "- Primary and unweighted sensitivity pipelines receive independently optimized thresholds."
)

print(
    "- No fixed 0.5 threshold is used."
)

print(
    "- Outer-test outcomes NEVER optimize or adjust thresholds."
)

print(
    "- Pooled outer-OOF outcomes NEVER re-optimize thresholds."
)


print(
    "\nHIERARCHICAL CLASS RULE:"
)

print(
    "if q0 >= optimized t1:"
)

print(
    "    class 0 = No ANC"
)

print(
    "elif q2 >= optimized t2:"
)

print(
    "    class 2 = >=8 visits"
)

print(
    "else:"
)

print(
    "    class 1 = 1-7 visits"
)


print(
    "\nSYNTHETIC OPTIMIZER QC:"
)

print(
    synthetic_result
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nMANUSCRIPT METHODS PREVIEW:"
)

print(
    methods_text
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)

print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)

print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 18 PASS"
)

print(
    "=" * 78
)


STEP 18 — CROSS-VALIDATED THRESHOLD OPTIMIZATION FREEZE

KEY POLICY:
Threshold optimization : ENABLED
Threshold values fixed : NO
Default threshold 0.5 : NO

STAGE-SPECIFIC POLICY:


,Stage,Positive_Class,Development_Population,Probability,Threshold,Objective,Fixed
0,Stage 1,No ANC,All observations in current outer-training set,calibrated q0,optimized t1,survey-weighted balanced accuracy,False
1,Stage 2,>=8 ANC visits,True ANC users in current outer-training set,calibrated q2,optimized t2,survey-weighted balanced accuracy,False



OPTIMIZATION RULE:
- Joint model/hyperparameter selection occurs first.
- Probability calibration occurs next.
- Threshold is optimized from calibrated INNER-OOF probabilities.
- Objective = survey-weighted balanced accuracy.
- Every classification-changing operating point is evaluated.
- Stage 1 and Stage 2 thresholds are optimized independently.
- Each outer fold receives independently optimized thresholds.
- Primary and unweighted sensitivity pipelines receive independently optimized thresholds.
- No fixed 0.5 threshold is used.
- Outer-test outcomes NEVER optimize or adjust thresholds.
- Pooled outer-OOF outcomes NEVER re-optimize thresholds.

HIERARCHICAL CLASS RULE:
if q0 >= optimized t1:
    class 0 = No ANC
elif q2 >= optimized t2:
    class 2 = >=8 visits
else:
    class 1 = 1-7 visits

SYNTHETIC OPTIMIZER QC:
{'selected_threshold': 0.41500000000000004, 'optimized_balanced_accuracy': 0.8219607843137255, 'sensitivity': 0.803921568627451, 'specificity': 0.8400000000000001, 'you

,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Joint Step 16 frozen,True
4,Step 17 frozen,True
5,Step 17A frozen,True
6,Threshold optimization enabled,True
7,Threshold values NOT fixed,True
8,No default 0.5 threshold,True
9,Optimization occurs after joint selection,True



MANUSCRIPT METHODS PREVIEW:
Decision-threshold optimization

Decision-threshold values were not fixed in advance. Instead, the
threshold-optimization procedure was prespecified before model fitting.
For each hierarchical stage, analysis pipeline, and outer
cross-validation fold, the operating threshold will be optimized only
after joint model-family and hyperparameter selection and probability
calibration.

Threshold optimization will use the selected winner's calibrated,
cross-fitted inner-validation probabilities from the corresponding
outer-training population. Original DHS sampling weights will be used
in the optimization criterion. Outer-test observations will not be used
to select, adjust, or re-estimate decision thresholds.

The optimization criterion will be survey-weighted balanced accuracy,
defined as the arithmetic mean of weighted sensitivity and weighted
specificity. Maximizing weighted balanced accuracy is equivalent in
threshold ranking to maximizing the weighted Youden

In [1]:
# ============================================================
# KERNEL RESTART RECOVERY + SUPERSEDED STEP-19A CLEANUP
# ============================================================
#
# PURPOSE
# -------
# 1. Rebuild notebook state after kernel restart.
# 2. Verify all frozen methodology through corrected Step 18.
# 3. Remove ONLY outputs from the superseded old Step 19A.
#
# DOES NOT:
#   - regenerate data
#   - regenerate CV splits
#   - modify Steps 1-18
#   - fit any model
#   - inspect any predictive performance
#
# After this recovery, the corrected workflow will be:
#
#   STEP 19A:
#       All 10 DEFAULT models benchmarked
#
#   STEP 19B:
#       All 10 model families independently tuned
#       using frozen inner CV
#
#   STEP 19C:
#       Joint model-family + hyperparameter selection
#       -> calibration
#       -> threshold optimization
#       -> hierarchical fusion
#
# No model family will be removed based on default performance.
# ============================================================


from pathlib import Path
import hashlib
import importlib.metadata as importlib_metadata
import json
import sys

import numpy as np
import pandas as pd
import sklearn

from IPython.display import display


# ============================================================
# 1. PROJECT ROOT
# ============================================================

ROOT = Path(
    r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
)

if not ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found:\n{ROOT}"
    )


# ============================================================
# 2. RESTORE DIRECTORY REGISTRY
# ============================================================

DIRS = {
    "root":
        ROOT,

    "raw_reference":
        ROOT / "00_raw_reference",

    "processed":
        ROOT / "01_processed_data",

    "qc":
        ROOT / "02_quality_control",

    "protocol":
        ROOT / "03_protocol",

    "splits":
        ROOT / "04_cv_splits",

    "models":
        ROOT / "05_models",

    "predictions":
        ROOT / "06_predictions",

    "calibration":
        ROOT / "07_calibration",

    "bootstrap":
        ROOT / "08_bootstrap",

    "geographic":
        ROOT / "09_geographic_validation",

    "explainability":
        ROOT / "10_explainability",

    "sensitivity":
        ROOT / "11_sensitivity",

    "subgroup":
        ROOT / "12_subgroup_analysis",

    "tables":
        ROOT / "13_tables",

    "figures":
        ROOT / "14_figures",

    "logs":
        ROOT / "15_logs",

    "reporting":
        ROOT / "16_reporting",

    "archive":
        ROOT / "17_reproducibility_archive",
}


missing_dirs = [
    str(path)
    for path in DIRS.values()
    if not path.exists()
]

if missing_dirs:
    raise FileNotFoundError(
        "Expected project directories are missing:\n"
        + "\n".join(missing_dirs)
    )


# ============================================================
# 3. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:
        return json.load(handle)


def package_version(name):

    try:
        return importlib_metadata.version(name)

    except Exception:
        return "NOT INSTALLED"


# ============================================================
# 4. CORE FROZEN FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)

OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)

INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)

EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)

EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Frozen core file missing:\n{path}"
        )


observed_primary_hash = sha256_file(PRIMARY_FILE)
observed_outer_hash = sha256_file(OUTER_SPLIT_FILE)
observed_inner_hash = sha256_file(INNER_SPLIT_FILE)


if observed_primary_hash != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError(
        "Primary-master hash mismatch."
    )

if observed_outer_hash != EXPECTED_OUTER_SHA256:
    raise RuntimeError(
        "Outer-split hash mismatch."
    )

if observed_inner_hash != EXPECTED_INNER_SHA256:
    raise RuntimeError(
        "Inner-split hash mismatch."
    )


# ============================================================
# 5. VERIFY ALL FROZEN PROTOCOLS THROUGH STEP 18
# ============================================================

MANIFESTS = {
    "Step 14":
        DIRS["protocol"]
        / "STEP14_FINAL_preprocessing_manifest.json",

    "Step 15":
        DIRS["protocol"]
        / "STEP15_FINAL_hierarchical_weight_manifest.json",

    "Step 15A":
        DIRS["protocol"]
        / "STEP15A_FINAL_survey_weight_sensitivity_manifest.json",

    "Step 16":
        DIRS["protocol"]
        / "STEP16_FINAL_hyperparameter_selection_manifest.json",

    "Step 17":
        DIRS["protocol"]
        / "STEP17_FINAL_calibration_evaluation_manifest.json",

    "Step 17A":
        DIRS["protocol"]
        / "STEP17A_FINAL_predictor_feature_selection_manifest.json",

    "Step 18":
        DIRS["protocol"]
        / "STEP18_FINAL_threshold_optimization_manifest.json",
}


EXPECTED_STATUS = {
    "Step 14":
        "PREPROCESSING_PROTOCOL_FROZEN",

    "Step 15":
        "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN",

    "Step 15A":
        "SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN",

    "Step 16":
        "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN",

    "Step 17":
        "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN",

    "Step 17A":
        "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN",

    "Step 18":
        "THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN",
}


loaded_manifests = {}

manifest_rows = []


for step_name, manifest_path in MANIFESTS.items():

    if not manifest_path.exists():
        raise FileNotFoundError(
            f"{step_name} manifest missing:\n"
            f"{manifest_path}"
        )

    manifest = read_json(
        manifest_path
    )

    loaded_manifests[
        step_name
    ] = manifest

    observed = manifest.get(
        "status"
    )

    expected = EXPECTED_STATUS[
        step_name
    ]

    manifest_rows.append(
        {
            "Step":
                step_name,

            "Expected_Status":
                expected,

            "Observed_Status":
                observed,

            "PASS":
                observed == expected,
        }
    )


manifest_qc = pd.DataFrame(
    manifest_rows
)


if not manifest_qc["PASS"].all():

    raise RuntimeError(
        "Frozen-protocol status check failed:\n\n"
        + manifest_qc.to_string(index=False)
    )


# restore common manifest variables

step14_manifest = loaded_manifests["Step 14"]
step15_manifest = loaded_manifests["Step 15"]
step15a_manifest = loaded_manifests["Step 15A"]
step16_manifest = loaded_manifests["Step 16"]
step17_manifest = loaded_manifests["Step 17"]
step17a_manifest = loaded_manifests["Step 17A"]
step18_manifest = loaded_manifests["Step 18"]


# ============================================================
# 6. FROZEN LOGIC CHECK
# ============================================================

logic_qc = pd.DataFrame(
    [
        {
            "Check":
                "Step 16 contains 10 model families",

            "PASS":
                (
                    step16_manifest.get(
                        "model_family_count",
                        10,
                    )
                    ==
                    10
                ),
        },

        {
            "Check":
                "Step 16 contains 120 tuned candidates",

            "PASS":
                (
                    step16_manifest.get(
                        "total_joint_candidates"
                    )
                    ==
                    120
                ),
        },

        {
            "Check":
                "Joint tuned-model selection frozen",

            "PASS":
                bool(
                    step16_manifest.get(
                        "joint_model_family_selection",
                        False,
                    )
                ),
        },

        {
            "Check":
                "Winner-only calibration frozen",

            "PASS":
                bool(
                    step17_manifest.get(
                        "winner_only_calibration",
                        False,
                    )
                ),
        },

        {
            "Check":
                "15 predictors frozen",

            "PASS":
                (
                    step17a_manifest.get(
                        "original_predictor_count"
                    )
                    ==
                    15
                ),
        },

        {
            "Check":
                "External feature selection disabled",

            "PASS":
                (
                    step17a_manifest.get(
                        "external_feature_selection"
                    )
                    is False
                ),
        },

        {
            "Check":
                "Threshold optimization enabled",

            "PASS":
                bool(
                    step18_manifest.get(
                        "threshold_optimization",
                        False,
                    )
                ),
        },

        {
            "Check":
                "Threshold values not fixed",

            "PASS":
                (
                    step18_manifest.get(
                        "threshold_value_fixed"
                    )
                    is False
                ),
        },

        {
            "Check":
                "Default threshold 0.5 disabled",

            "PASS":
                (
                    step18_manifest.get(
                        "default_threshold_0_5"
                    )
                    is False
                ),
        },
    ]
)


if not logic_qc["PASS"].all():

    failed = logic_qc.loc[
        ~logic_qc["PASS"]
    ]

    raise RuntimeError(
        "Frozen modelling-logic check failed:\n\n"
        + failed.to_string(index=False)
    )


# ============================================================
# 7. RESTORE 15-PREDICTOR REGISTRY
# ============================================================

NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]

ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]

NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "division",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]

BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]

CATEGORICAL_PREDICTORS = (
    ORDERED_CATEGORICAL_PREDICTORS
    + NOMINAL_CATEGORICAL_PREDICTORS
    + BINARY_CATEGORICAL_PREDICTORS
)

PREDICTORS = (
    NUMERIC_PREDICTORS
    + CATEGORICAL_PREDICTORS
)


if len(PREDICTORS) != 15:
    raise RuntimeError(
        "Recovered predictor count is not 15."
    )


master = pd.read_csv(
    PRIMARY_FILE
)


if master.shape != (5063, 23):
    raise RuntimeError(
        f"Unexpected primary-master shape: {master.shape}"
    )


missing_predictors = [
    predictor
    for predictor in PREDICTORS
    if predictor not in master.columns
]


if missing_predictors:
    raise RuntimeError(
        "Frozen predictors missing:\n"
        f"{missing_predictors}"
    )


# ============================================================
# 8. REMOVE ONLY SUPERSEDED OLD STEP-19A OUTPUTS
# ============================================================
#
# These filenames belong to the OLD workflow that directly
# started joint tuning on Outer Fold 0.
#
# They are intentionally removed because the corrected workflow
# starts with ALL-MODEL DEFAULT benchmarking.
#
# Steps 1-18 are NEVER touched.
# ============================================================

superseded_step19a_files = [
    (
        DIRS["models"]
        / "STEP19A_PRIMARY_outer0_STAGE1_candidate_inner_metrics.csv"
    ),

    (
        DIRS["models"]
        / "STEP19A_PRIMARY_outer0_STAGE2_candidate_inner_metrics.csv"
    ),

    (
        DIRS["models"]
        / "STEP19A_PRIMARY_outer0_winner_summary.csv"
    ),

    (
        DIRS["calibration"]
        / "STEP19A_PRIMARY_outer0_STAGE1_winner_inner_OOF_RESTRICTED.csv"
    ),

    (
        DIRS["calibration"]
        / "STEP19A_PRIMARY_outer0_STAGE2_winner_inner_OOF_RESTRICTED.csv"
    ),

    (
        DIRS["predictions"]
        / "STEP19A_PRIMARY_outer0_predictions_RESTRICTED.csv"
    ),

    (
        DIRS["qc"]
        / "STEP19A_PRIMARY_outer0_execution_qc.csv"
    ),

    (
        DIRS["logs"]
        / "STEP19A_PRIMARY_outer0_execution_manifest.json"
    ),
]


cleanup_rows = []


for path in superseded_step19a_files:

    existed_before = path.exists()

    if existed_before:
        path.unlink()

    cleanup_rows.append(
        {
            "File":
                str(path.relative_to(ROOT)),

            "Existed_Before":
                existed_before,

            "Removed":
                existed_before
                and
                not path.exists(),

            "Absent_After_Cleanup":
                not path.exists(),
        }
    )


cleanup_qc = pd.DataFrame(
    cleanup_rows
)


if not cleanup_qc[
    "Absent_After_Cleanup"
].all():

    raise RuntimeError(
        "Could not remove one or more superseded "
        "Step-19A files."
    )


# ============================================================
# 9. PACKAGE VERSIONS
# ============================================================

package_versions = pd.DataFrame(
    [
        {
            "Package":
                "Python",

            "Version":
                sys.version.split()[0],
        },

        {
            "Package":
                "numpy",

            "Version":
                np.__version__,
        },

        {
            "Package":
                "pandas",

            "Version":
                pd.__version__,
        },

        {
            "Package":
                "scikit-learn",

            "Version":
                sklearn.__version__,
        },

        {
            "Package":
                "xgboost",

            "Version":
                package_version("xgboost"),
        },

        {
            "Package":
                "lightgbm",

            "Version":
                package_version("lightgbm"),
        },

        {
            "Package":
                "catboost",

            "Version":
                package_version("catboost"),
        },
    ]
)


# ============================================================
# 10. CORRECTED MODEL-DEVELOPMENT SEQUENCE
# ============================================================

corrected_sequence = pd.DataFrame(
    [
        {
            "Step":
                "19A",

            "Analysis":
                "Default-model benchmark",

            "Models":
                "All 10 model families",

            "Purpose":
                (
                    "Measure default/library-baseline "
                    "performance for every family"
                ),

            "Model_Dropped":
                False,
        },

        {
            "Step":
                "19B",

            "Analysis":
                "Family-specific hyperparameter tuning",

            "Models":
                "All 10 model families",

            "Purpose":
                (
                    "Tune every family using the frozen "
                    "inner-CV search space"
                ),

            "Model_Dropped":
                False,
        },

        {
            "Step":
                "19C",

            "Analysis":
                "Joint tuned-model selection",

            "Models":
                "All 10 tuned families",

            "Purpose":
                (
                    "Leakage-safe primary model-development "
                    "procedure"
                ),

            "Model_Dropped":
                False,
        },

        {
            "Step":
                "19C+",

            "Analysis":
                (
                    "Calibration + threshold optimization + "
                    "hierarchical fusion"
                ),

            "Models":
                "Selected inner-CV winners",

            "Purpose":
                "Final primary predictive pipeline",

            "Model_Dropped":
                False,
        },
    ]
)


# ============================================================
# 11. FINAL RECOVERY QC
# ============================================================

recovery_qc = pd.DataFrame(
    [
        {
            "Check":
                "Project root found",

            "PASS":
                ROOT.exists(),
        },

        {
            "Check":
                "All directories restored",

            "PASS":
                len(missing_dirs) == 0,
        },

        {
            "Check":
                "Primary master hash verified",

            "PASS":
                observed_primary_hash
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash verified",

            "PASS":
                observed_outer_hash
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash verified",

            "PASS":
                observed_inner_hash
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Steps 14-18 frozen",

            "PASS":
                bool(
                    manifest_qc["PASS"].all()
                ),
        },

        {
            "Check":
                "Frozen modelling logic restored",

            "PASS":
                bool(
                    logic_qc["PASS"].all()
                ),
        },

        {
            "Check":
                "15 predictors restored",

            "PASS":
                len(PREDICTORS) == 15,
        },

        {
            "Check":
                "All predictors available",

            "PASS":
                len(missing_predictors) == 0,
        },

        {
            "Check":
                "Superseded old Step-19A outputs absent",

            "PASS":
                bool(
                    cleanup_qc[
                        "Absent_After_Cleanup"
                    ].all()
                ),
        },

        {
            "Check":
                "No model fitted during recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No performance inspected",

            "PASS":
                True,
        },
    ]
)


if not recovery_qc["PASS"].all():

    failed = recovery_qc.loc[
        ~recovery_qc["PASS"]
    ]

    raise RuntimeError(
        "RECOVERY / CLEANUP FAILED:\n\n"
        + failed.to_string(index=False)
    )


# ============================================================
# 12. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "KERNEL RECOVERY + SUPERSEDED STEP-19A CLEANUP"
)

print(
    "=" * 78
)


print(
    "\nPACKAGE VERSIONS:"
)

display(
    package_versions
)


print(
    "\nFROZEN PROTOCOL STATUS:"
)

display(
    manifest_qc
)


print(
    "\nFROZEN LOGIC QC:"
)

display(
    logic_qc
)


print(
    "\nSUPERSEDED OLD STEP-19A CLEANUP:"
)

display(
    cleanup_qc
)


print(
    "\nCORRECTED DEVELOPMENT SEQUENCE:"
)

display(
    corrected_sequence
)


print(
    "\nFINAL RECOVERY QC:"
)

display(
    recovery_qc
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {observed_primary_hash}"
)

print(
    f"Outer split    : {observed_outer_hash}"
)

print(
    f"Inner split    : {observed_inner_hash}"
)


print(
    "\nIMPORTANT:"
)

print(
    "- No model family will be removed after default benchmarking."
)

print(
    "- All 10 families will proceed to hyperparameter tuning."
)

print(
    "- Default results will NOT alter the frozen tuning grids."
)

print(
    "- Final primary selection remains leakage-safe inner-CV selection."
)

print(
    "- Steps 1-18 remain unchanged and frozen."
)


print(
    "=" * 78
)

print(
    "STATUS: READY FOR NEW STEP 19A DEFAULT BENCHMARK"
)

print(
    "=" * 78
)


KERNEL RECOVERY + SUPERSEDED STEP-19A CLEANUP

PACKAGE VERSIONS:


,Package,Version
0,Python,3.13.9
1,numpy,2.3.5
2,pandas,3.0.5
3,scikit-learn,1.9.0
4,xgboost,3.4.1
5,lightgbm,4.6.0
6,catboost,1.2.10



FROZEN PROTOCOL STATUS:


,Step,Expected_Status,Observed_Status,PASS
0,Step 14,PREPROCESSING_PROTOCOL_FROZEN,PREPROCESSING_PROTOCOL_FROZEN,True
1,Step 15,HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN,HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN,True
2,Step 15A,SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN,SURVEY_WEIGHT_SENSITIVITY_PROTOCOL_FROZEN,True
3,Step 16,JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_...,JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_...,True
4,Step 17,JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_F...,JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_F...,True
5,Step 17A,FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_P...,FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_P...,True
6,Step 18,THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN,THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN,True



FROZEN LOGIC QC:


,Check,PASS
0,Step 16 contains 10 model families,True
1,Step 16 contains 120 tuned candidates,True
2,Joint tuned-model selection frozen,True
3,Winner-only calibration frozen,True
4,15 predictors frozen,True
5,External feature selection disabled,True
6,Threshold optimization enabled,True
7,Threshold values not fixed,True
8,Default threshold 0.5 disabled,True


... (3 outputs left out)

,Step,Analysis,Models,Purpose,Model_Dropped
0,19A,Default-model benchmark,All 10 model families,Measure default/library-baseline performance f...,False
1,19B,Family-specific hyperparameter tuning,All 10 model families,Tune every family using the frozen inner-CV se...,False
2,19C,Joint tuned-model selection,All 10 tuned families,Leakage-safe primary model-development procedure,False
3,19C+,Calibration + threshold optimization + hierarc...,Selected inner-CV winners,Final primary predictive pipeline,False



FINAL RECOVERY QC:


,Check,PASS
0,Project root found,True
1,All directories restored,True
2,Primary master hash verified,True
3,Outer split hash verified,True
4,Inner split hash verified,True
5,Steps 14-18 frozen,True
6,Frozen modelling logic restored,True
7,15 predictors restored,True
8,All predictors available,True
9,Superseded old Step-19A outputs absent,True



INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Outer split    : 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
Inner split    : 2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633

IMPORTANT:
- No model family will be removed after default benchmarking.
- All 10 families will proceed to hyperparameter tuning.
- Default results will NOT alter the frozen tuning grids.
- Final primary selection remains leakage-safe inner-CV selection.
- Steps 1-18 remain unchanged and frozen.
STATUS: READY FOR NEW STEP 19A DEFAULT BENCHMARK


In [2]:
# ============================================================
# STEP 19A — ALL 10 MODEL FAMILIES
#            DEFAULT / BASELINE OUTER-CV BENCHMARK
#            PRIMARY SURVEY-WEIGHTED PIPELINE
# ============================================================
#
# PURPOSE
# -------
# Benchmark EVERY model family before hyperparameter tuning.
#
# NO model family is dropped.
# NO tuning occurs.
# NO calibration occurs.
# NO threshold optimization occurs.
# NO hierarchical fusion occurs.
#
# All 10 families will proceed to Step 19B tuning regardless
# of their Step-19A default/baseline performance.
#
#
# PERFORMANCE ESTIMATION
# ----------------------
# 5 frozen outer folds.
#
# For each:
#   model x stage x outer fold
#
#   fit on outer training data
#   predict untouched outer test data
#   calculate:
#       survey-weighted Average Precision
#       survey-weighted ROC-AUC
#
# Stage 1:
#   No ANC vs Any ANC
#
# Stage 2:
#   >=8 vs 1-7 among TRUE ANC users
#
#
# IMPORTANT
# ---------
# Before calibration, raw SVM decision-function scores from
# independently fitted outer folds need not share an identical
# numerical scale.
#
# Therefore Step 19A does NOT pool raw scores across outer folds
# to create one pooled AP/AUC.
#
# Instead:
#   - each outer-fold AP/AUC is calculated independently
#   - five-fold mean / SD / median / range are descriptive
#     benchmark summaries
#
# Final primary performance later uses the frozen Step-17
# calibrated pooled outer-OOF procedure.
#
#
# DEFAULT / BASELINE DEFINITION
# -----------------------------
# Predictive hyperparameters use installed-library defaults
# unless a setting is required by:
#
#   - model-family definition
#   - frozen preprocessing
#   - survey-weight policy
#   - class-balancing prohibition
#   - reproducibility
#   - non-verbose execution
#
# ElasticNet logistic exception:
#   sklearn's literal LogisticRegression default is L2.
#   To retain the ElasticNet family:
#       C = 1.0
#       l1_ratio = 0.5
#       solver = "saga"
#
# This is explicitly labelled a FAMILY-REFERENCE BASELINE,
# not a literal sklearn default.
# ============================================================


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from pathlib import Path

import datetime as dt
import hashlib
import importlib.metadata as importlib_metadata
import json
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from IPython.display import display


# ============================================================
# 2. CONSTANTS
# ============================================================

SEED = 2026

PIPELINE_NAME = (
    "PRIMARY_SURVEY_WEIGHTED"
)

BENCHMARK_NAME = (
    "DEFAULT_BASELINE"
)


# ============================================================
# 3. FROZEN FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)

OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)

INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)

STEP14_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP14_FINAL_preprocessing_manifest.json"
)

STEP15_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP15_FINAL_hierarchical_weight_manifest.json"
)

STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)

STEP17_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_evaluation_manifest.json"
)

STEP17A_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_predictor_feature_selection_manifest.json"
)

STEP18_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)

EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)

EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


# ============================================================
# 4. INPUT EXISTENCE / HASH QC
# ============================================================

required_files = [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    STEP14_MANIFEST_FILE,
    STEP15_MANIFEST_FILE,
    STEP16_MANIFEST_FILE,
    STEP17_MANIFEST_FILE,
    STEP17A_MANIFEST_FILE,
    STEP18_MANIFEST_FILE,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required frozen file missing:\n{path}"
        )


if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


# ============================================================
# 5. UPSTREAM STATUS QC
# ============================================================

step14_manifest = read_json(
    STEP14_MANIFEST_FILE
)

step15_manifest = read_json(
    STEP15_MANIFEST_FILE
)

step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)

step17_manifest = read_json(
    STEP17_MANIFEST_FILE
)

step17a_manifest = read_json(
    STEP17A_MANIFEST_FILE
)

step18_manifest = read_json(
    STEP18_MANIFEST_FILE
)


expected_statuses = {
    "Step 14":
        (
            step14_manifest.get("status"),
            "PREPROCESSING_PROTOCOL_FROZEN",
        ),

    "Step 15":
        (
            step15_manifest.get("status"),
            "HIERARCHICAL_TARGET_WEIGHT_PROTOCOL_FROZEN",
        ),

    "Step 16":
        (
            step16_manifest.get("status"),
            "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN",
        ),

    "Step 17":
        (
            step17_manifest.get("status"),
            "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN",
        ),

    "Step 17A":
        (
            step17a_manifest.get("status"),
            "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN",
        ),

    "Step 18":
        (
            step18_manifest.get("status"),
            "THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN",
        ),
}


for step_name, (
    observed,
    expected,
) in expected_statuses.items():

    if observed != expected:

        raise RuntimeError(
            f"{step_name} status mismatch.\n"
            f"Expected: {expected}\n"
            f"Observed: {observed}"
        )


# ============================================================
# 6. FROZEN PREDICTOR SET
# ============================================================

NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]


ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]


NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "division",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]


BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]


CATEGORICAL_PREDICTORS = (
    ORDERED_CATEGORICAL_PREDICTORS
    +
    NOMINAL_CATEGORICAL_PREDICTORS
    +
    BINARY_CATEGORICAL_PREDICTORS
)


PREDICTORS = (
    NUMERIC_PREDICTORS
    +
    CATEGORICAL_PREDICTORS
)


if len(PREDICTORS) != 15:

    raise RuntimeError(
        "Frozen original predictor count is not 15."
    )


# ============================================================
# 7. DEFAULT / BASELINE REGISTRY
# ============================================================

baseline_registry = pd.DataFrame(
    [
        {
            "Model":
                "ElasticNet logistic regression",

            "Preprocessing":
                "Scaled OHE",

            "Baseline_Type":
                "Family-reference baseline",

            "Baseline_Definition":
                (
                    "C=1.0; l1_ratio=0.5; solver=saga. "
                    "Required because sklearn literal default "
                    "is L2 rather than ElasticNet."
                ),
        },

        {
            "Model":
                "RBF SVM",

            "Preprocessing":
                "Scaled OHE",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "RBF kernel; C=1.0; gamma=scale; "
                    "probability=False."
                ),
        },

        {
            "Model":
                "Random Forest",

            "Preprocessing":
                "Unscaled OHE",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Installed sklearn defaults; "
                    "random_state and execution controls only."
                ),
        },

        {
            "Model":
                "ExtraTrees",

            "Preprocessing":
                "Unscaled OHE",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Installed sklearn defaults; "
                    "random_state and execution controls only."
                ),
        },

        {
            "Model":
                "AdaBoost",

            "Preprocessing":
                "Unscaled OHE",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Default decision stump, n_estimators=50, "
                    "learning_rate=1.0."
                ),
        },

        {
            "Model":
                "GradientBoosting",

            "Preprocessing":
                "Unscaled OHE",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Installed sklearn defaults; "
                    "random_state fixed."
                ),
        },

        {
            "Model":
                "HistGradientBoosting",

            "Preprocessing":
                "Native categorical",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Installed sklearn defaults with "
                    "categorical_features=from_dtype."
                ),
        },

        {
            "Model":
                "XGBoost",

            "Preprocessing":
                "Unscaled OHE",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Installed XGBoost defaults; "
                    "scale_pos_weight=1 and reproducibility/"
                    "execution settings fixed."
                ),
        },

        {
            "Model":
                "LightGBM",

            "Preprocessing":
                "Native categorical",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Installed LightGBM defaults; "
                    "no class balancing; deterministic execution."
                ),
        },

        {
            "Model":
                "CatBoost",

            "Preprocessing":
                "CatBoost native",

            "Baseline_Type":
                "Library default predictive hyperparameters",

            "Baseline_Definition":
                (
                    "Installed CatBoost defaults including "
                    "automatic default learning-rate behavior; "
                    "no class balancing; random seed fixed."
                ),
        },
    ]
)


if len(baseline_registry) != 10:

    raise RuntimeError(
        "Baseline registry does not contain 10 model families."
    )


if baseline_registry[
    "Model"
].duplicated().any():

    raise RuntimeError(
        "Duplicate model in baseline registry."
    )


# ============================================================
# 8. LOAD DATA / OUTER SPLITS
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)

outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)


if master.shape != (5063, 23):

    raise RuntimeError(
        f"Unexpected primary-master shape: {master.shape}"
    )


for column in (
    PREDICTORS
    +
    [
        "ANC_CLASS",
        "SURVEY_WEIGHT",
        "PSU",
        "STRATUM",
    ]
):

    if column not in master.columns:

        raise RuntimeError(
            f"Required column missing from master: {column}"
        )


# ============================================================
# 9. ROBUST SPLIT-COLUMN IDENTIFICATION
# ============================================================

def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column
        in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(alias)

        if key in lookup:

            return lookup[key]


    raise RuntimeError(
        "Could not identify required column.\n"
        f"Aliases: {aliases}\n"
        f"Available: {list(dataframe.columns)}"
    )


MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


# ============================================================
# 10. CANONICAL ROW KEYS
# ============================================================

def canonical_row_key(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(dtype=float),
            np.round(
                numeric.to_numpy(dtype=float)
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(dtype=float)
            ).astype(np.int64),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


master = master.copy()

outer_assignment = outer_assignment.copy()


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


if master[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate row key in master."
    )


if outer_assignment[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate row key in outer assignment."
    )


if set(
    master[
        "__ROW_KEY"
    ]
) != set(
    outer_assignment[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Outer assignment does not exactly cover master."
    )


outer_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__OUTER_FOLD"
] = master[
    "__ROW_KEY"
].map(
    outer_map
)


if master[
    "__OUTER_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing outer-fold assignment."
    )


master[
    "__OUTER_FOLD"
] = master[
    "__OUTER_FOLD"
].astype(int)


outer_folds = sorted(
    master[
        "__OUTER_FOLD"
    ]
    .unique()
    .tolist()
)


if outer_folds != [0, 1, 2, 3, 4]:

    raise RuntimeError(
        f"Expected outer folds [0,1,2,3,4]; "
        f"observed {outer_folds}"
    )


# ============================================================
# 11. OUTCOME / WEIGHT QC
# ============================================================

master[
    "ANC_CLASS"
] = pd.to_numeric(
    master[
        "ANC_CLASS"
    ],
    errors="raise",
).astype(int)


if set(
    master[
        "ANC_CLASS"
    ].unique()
) != {0, 1, 2}:

    raise RuntimeError(
        "ANC_CLASS is not exactly {0,1,2}."
    )


survey_weight = master[
    "SURVEY_WEIGHT"
].to_numpy(
    dtype=float
)


if not np.isfinite(
    survey_weight
).all():

    raise RuntimeError(
        "Non-finite survey weight."
    )


if (
    survey_weight
    <=
    0
).any():

    raise RuntimeError(
        "Survey weights must be strictly positive."
    )


# ============================================================
# 12. TRAINING-WEIGHT NORMALIZATION
# ============================================================

def normalized_training_weight(
    dataframe,
):

    weight = (
        dataframe[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    normalized = (
        weight
        /
        weight.mean()
    )


    if not np.isclose(
        normalized.mean(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Training weights did not normalize to mean 1."
        )


    return normalized


# ============================================================
# 13. STAGE-SPECIFIC DATA
# ============================================================

def prepare_stage_split(
    outer_train,
    outer_test,
    stage_name,
):

    train = outer_train.copy()

    test = outer_test.copy()


    if stage_name == "Stage 1":

        train[
            "__TARGET"
        ] = (
            train[
                "ANC_CLASS"
            ]
            ==
            0
        ).astype(int)


        test[
            "__TARGET"
        ] = (
            test[
                "ANC_CLASS"
            ]
            ==
            0
        ).astype(int)


    elif stage_name == "Stage 2":

        train = (
            train.loc[
                train[
                    "ANC_CLASS"
                ]
                .isin(
                    [1, 2]
                )
            ]
            .copy()
        )


        test = (
            test.loc[
                test[
                    "ANC_CLASS"
                ]
                .isin(
                    [1, 2]
                )
            ]
            .copy()
        )


        train[
            "__TARGET"
        ] = (
            train[
                "ANC_CLASS"
            ]
            ==
            2
        ).astype(int)


        test[
            "__TARGET"
        ] = (
            test[
                "ANC_CLASS"
            ]
            ==
            2
        ).astype(int)


    else:

        raise ValueError(
            f"Unknown stage: {stage_name}"
        )


    train = train.reset_index(
        drop=True
    )

    test = test.reset_index(
        drop=True
    )


    if set(
        train[
            "__TARGET"
        ].unique()
    ) != {0, 1}:

        raise RuntimeError(
            f"{stage_name}: training partition lacks both classes."
        )


    if set(
        test[
            "__TARGET"
        ].unique()
    ) != {0, 1}:

        raise RuntimeError(
            f"{stage_name}: outer-test partition lacks both classes."
        )


    return (
        train,
        test,
    )


# ============================================================
# 14. FROZEN MODEL-SPECIFIC PREPROCESSING
# ============================================================

def build_representations(
    train_df,
    test_df,
):

    # --------------------------------------------------------
    # Numeric representation
    # --------------------------------------------------------

    train_numeric = (
        train_df[
            NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    test_numeric = (
        test_df[
            NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    if not np.isfinite(
        train_numeric
    ).all():

        raise RuntimeError(
            "Non-finite numeric value in training data."
        )


    if not np.isfinite(
        test_numeric
    ).all():

        raise RuntimeError(
            "Non-finite numeric value in test data."
        )


    # --------------------------------------------------------
    # OHE categories learned on OUTER TRAIN only
    # --------------------------------------------------------

    train_cat = (
        train_df[
            CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    test_cat = (
        test_df[
            CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    ohe = OneHotEncoder(
        handle_unknown="ignore",
        drop=None,
        sparse_output=False,
        dtype=np.float64,
    )


    train_ohe = ohe.fit_transform(
        train_cat
    )


    test_ohe = ohe.transform(
        test_cat
    )


    # --------------------------------------------------------
    # Unscaled OHE
    # --------------------------------------------------------

    train_unscaled_ohe = np.hstack(
        [
            train_numeric,
            train_ohe,
        ]
    )


    test_unscaled_ohe = np.hstack(
        [
            test_numeric,
            test_ohe,
        ]
    )


    # --------------------------------------------------------
    # Scaled numeric + OHE
    # --------------------------------------------------------

    scaler = StandardScaler()


    train_numeric_scaled = (
        scaler.fit_transform(
            train_numeric
        )
    )


    test_numeric_scaled = (
        scaler.transform(
            test_numeric
        )
    )


    train_scaled_ohe = np.hstack(
        [
            train_numeric_scaled,
            train_ohe,
        ]
    )


    test_scaled_ohe = np.hstack(
        [
            test_numeric_scaled,
            test_ohe,
        ]
    )


    # --------------------------------------------------------
    # Native categorical:
    # HistGradientBoosting + LightGBM
    # --------------------------------------------------------

    train_native = (
        train_df[
            PREDICTORS
        ]
        .copy()
    )


    test_native = (
        test_df[
            PREDICTORS
        ]
        .copy()
    )


    for column in NUMERIC_PREDICTORS:

        train_native[
            column
        ] = pd.to_numeric(
            train_native[
                column
            ],
            errors="raise",
        ).astype(float)


        test_native[
            column
        ] = pd.to_numeric(
            test_native[
                column
            ],
            errors="raise",
        ).astype(float)


    unseen_native = {}


    for column in CATEGORICAL_PREDICTORS:

        train_values = (
            train_native[
                column
            ]
            .astype(str)
        )


        test_values = (
            test_native[
                column
            ]
            .astype(str)
        )


        categories = list(
            pd.unique(
                train_values
            )
        )


        unseen = sorted(
            set(
                test_values
            )
            -
            set(
                categories
            )
        )


        if unseen:

            unseen_native[
                column
            ] = unseen


        train_native[
            column
        ] = pd.Categorical(
            train_values,
            categories=categories,
            ordered=False,
        )


        test_native[
            column
        ] = pd.Categorical(
            test_values,
            categories=categories,
            ordered=False,
        )


    if unseen_native:

        raise RuntimeError(
            "Unexpected unseen native categorical levels:\n"
            f"{unseen_native}"
        )


    # --------------------------------------------------------
    # CatBoost native representation
    # --------------------------------------------------------

    train_catboost = (
        train_df[
            PREDICTORS
        ]
        .copy()
    )


    test_catboost = (
        test_df[
            PREDICTORS
        ]
        .copy()
    )


    for column in NUMERIC_PREDICTORS:

        train_catboost[
            column
        ] = pd.to_numeric(
            train_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


        test_catboost[
            column
        ] = pd.to_numeric(
            test_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in CATEGORICAL_PREDICTORS:

        train_catboost[
            column
        ] = (
            train_catboost[
                column
            ]
            .astype(str)
        )


        test_catboost[
            column
        ] = (
            test_catboost[
                column
            ]
            .astype(str)
        )


    return {
        "Scaled OHE":
            (
                train_scaled_ohe,
                test_scaled_ohe,
            ),

        "Unscaled OHE":
            (
                train_unscaled_ohe,
                test_unscaled_ohe,
            ),

        "Native categorical":
            (
                train_native,
                test_native,
            ),

        "CatBoost native":
            (
                train_catboost,
                test_catboost,
            ),
    }


# ============================================================
# 15. DEFAULT / BASELINE ESTIMATOR CONSTRUCTORS
# ============================================================

def make_baseline_estimator(
    model_name,
):

    # --------------------------------------------------------
    # ElasticNet family-reference baseline
    # --------------------------------------------------------

    if model_name == "ElasticNet logistic regression":

        return LogisticRegression(
            C=1.0,
            l1_ratio=0.5,
            solver="saga",
            fit_intercept=True,
            class_weight=None,
            max_iter=10000,
            tol=1e-4,
            random_state=SEED,
        )


    # --------------------------------------------------------
    # Literal predictive defaults + policy overrides
    # --------------------------------------------------------

    if model_name == "RBF SVM":

        return SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=False,
            class_weight=None,
            random_state=None,
        )


    if model_name == "Random Forest":

        return RandomForestClassifier(
            class_weight=None,
            random_state=SEED,
            n_jobs=-1,
        )


    if model_name == "ExtraTrees":

        return ExtraTreesClassifier(
            class_weight=None,
            random_state=SEED,
            n_jobs=-1,
        )


    if model_name == "AdaBoost":

        return AdaBoostClassifier(
            random_state=SEED,
        )


    if model_name == "GradientBoosting":

        return GradientBoostingClassifier(
            random_state=SEED,
        )


    if model_name == "HistGradientBoosting":

        return HistGradientBoostingClassifier(
            categorical_features="from_dtype",
            class_weight=None,
            random_state=SEED,
        )


    if model_name == "XGBoost":

        return XGBClassifier(
            objective="binary:logistic",
            scale_pos_weight=1,
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
        )


    if model_name == "LightGBM":

        return LGBMClassifier(
            class_weight=None,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,

            # implementation/reproducibility controls only
            deterministic=True,
            force_col_wise=True,
        )


    if model_name == "CatBoost":

        return CatBoostClassifier(
            loss_function="Logloss",
            class_weights=None,
            auto_class_weights=None,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        )


    raise ValueError(
        f"Unknown model family: {model_name}"
    )


# ============================================================
# 16. PREPROCESSING LOOKUP
# ============================================================

PREPROCESSING_MAP = dict(
    zip(
        baseline_registry[
            "Model"
        ],
        baseline_registry[
            "Preprocessing"
        ],
    )
)


# ============================================================
# 17. WEIGHTED FIT
# ============================================================

def fit_baseline_estimator(
    estimator,
    model_name,
    X_train,
    y_train,
    train_weight,
):

    with warnings.catch_warnings():

        warnings.simplefilter(
            "error",
            ConvergenceWarning,
        )


        if model_name == "CatBoost":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=train_weight,
                cat_features=CATEGORICAL_PREDICTORS,
                verbose=False,
            )


        elif model_name == "LightGBM":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=train_weight,
                categorical_feature=CATEGORICAL_PREDICTORS,
            )


        else:

            estimator.fit(
                X_train,
                y_train,
                sample_weight=train_weight,
            )


    return estimator


# ============================================================
# 18. RAW DISCRIMINATION SCORE
# ============================================================

def positive_score(
    estimator,
    model_name,
    X_test,
):

    classes = np.asarray(
        estimator.classes_
    )


    if set(
        classes.tolist()
    ) != {0, 1}:

        raise RuntimeError(
            f"{model_name}: unexpected fitted classes {classes}"
        )


    if model_name == "RBF SVM":

        score = np.asarray(
            estimator.decision_function(
                X_test
            ),
            dtype=float,
        ).reshape(-1)


    else:

        predicted_probability = (
            estimator.predict_proba(
                X_test
            )
        )


        positive_index = int(
            np.where(
                classes == 1
            )[0][0]
        )


        score = np.asarray(
            predicted_probability[
                :,
                positive_index
            ],
            dtype=float,
        )


        if (
            (score < 0.0)
            |
            (score > 1.0)
        ).any():

            raise RuntimeError(
                f"{model_name}: probability outside [0,1]."
            )


    if not np.isfinite(
        score
    ).all():

        raise RuntimeError(
            f"{model_name}: non-finite test score."
        )


    return score


# ============================================================
# 19. REALIZED BASELINE MODEL METADATA
# ============================================================

def realized_model_metadata(
    estimator,
    model_name,
):

    metadata = {
        "Model":
            model_name,
    }


    if model_name == "CatBoost":

        metadata[
            "Realized_Tree_Count"
        ] = int(
            estimator.tree_count_
        )


        metadata[
            "Realized_Learning_Rate"
        ] = float(
            estimator.learning_rate_
        )


    elif hasattr(
        estimator,
        "n_estimators",
    ):

        value = getattr(
            estimator,
            "n_estimators",
        )


        if value is not None:

            metadata[
                "Configured_n_estimators"
            ] = int(value)


    return metadata


# ============================================================
# 20. EXECUTE ALL 5 OUTER FOLDS
# ============================================================

prediction_records = []

fold_metric_records = []

fit_metadata_records = []


for outer_fold in outer_folds:

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"OUTER FOLD {outer_fold}"
    )

    print(
        "-" * 78
    )


    outer_train = (
        master.loc[
            master[
                "__OUTER_FOLD"
            ]
            !=
            outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )


    outer_test = (
        master.loc[
            master[
                "__OUTER_FOLD"
            ]
            ==
            outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )


    if (
        set(
            outer_train[
                "__ROW_KEY"
            ]
        )
        &
        set(
            outer_test[
                "__ROW_KEY"
            ]
        )
    ):

        raise RuntimeError(
            f"Outer fold {outer_fold}: train/test overlap."
        )


    for stage_name in [
        "Stage 1",
        "Stage 2",
    ]:

        stage_train, stage_test = (
            prepare_stage_split(
                outer_train=outer_train,
                outer_test=outer_test,
                stage_name=stage_name,
            )
        )


        print(
            f"\n{stage_name}: "
            f"train N={len(stage_train)}, "
            f"test N={len(stage_test)}, "
            f"train positives={int(stage_train['__TARGET'].sum())}, "
            f"test positives={int(stage_test['__TARGET'].sum())}"
        )


        # preprocessing fitted once per outer-train/stage
        representations = build_representations(
            train_df=stage_train,
            test_df=stage_test,
        )


        train_weight = (
            normalized_training_weight(
                stage_train
            )
        )


        test_weight = (
            stage_test[
                "SURVEY_WEIGHT"
            ]
            .to_numpy(
                dtype=float
            )
        )


        y_train = (
            stage_train[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        y_test = (
            stage_test[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        for model_name in baseline_registry[
            "Model"
        ]:

            preprocessing = (
                PREPROCESSING_MAP[
                    model_name
                ]
            )


            X_train, X_test = (
                representations[
                    preprocessing
                ]
            )


            print(
                f"  fitting {model_name} ..."
            )


            estimator = (
                make_baseline_estimator(
                    model_name
                )
            )


            try:

                estimator = (
                    fit_baseline_estimator(
                        estimator=estimator,
                        model_name=model_name,
                        X_train=X_train,
                        y_train=y_train,
                        train_weight=train_weight,
                    )
                )


                score = positive_score(
                    estimator=estimator,
                    model_name=model_name,
                    X_test=X_test,
                )


            except Exception as exc:

                raise RuntimeError(
                    "\nDEFAULT BENCHMARK MODEL FAILURE\n"
                    f"Outer fold : {outer_fold}\n"
                    f"Stage      : {stage_name}\n"
                    f"Model      : {model_name}\n"
                    f"Error      : {type(exc).__name__}: {exc}\n\n"
                    "No model family will be silently dropped. "
                    "Diagnose this exact configuration before continuing."
                ) from exc


            weighted_ap = float(
                average_precision_score(
                    y_test,
                    score,
                    sample_weight=test_weight,
                )
            )


            weighted_auc = float(
                roc_auc_score(
                    y_test,
                    score,
                    sample_weight=test_weight,
                )
            )


            if not (
                np.isfinite(weighted_ap)
                and
                np.isfinite(weighted_auc)
            ):

                raise RuntimeError(
                    f"{model_name}: non-finite fold metric."
                )


            fold_metric_records.append(
                {
                    "Pipeline":
                        PIPELINE_NAME,

                    "Benchmark":
                        BENCHMARK_NAME,

                    "Outer_Fold":
                        int(
                            outer_fold
                        ),

                    "Stage":
                        stage_name,

                    "Model":
                        model_name,

                    "Preprocessing":
                        preprocessing,

                    "Train_N":
                        int(
                            len(stage_train)
                        ),

                    "Test_N":
                        int(
                            len(stage_test)
                        ),

                    "Train_Positive_N":
                        int(
                            y_train.sum()
                        ),

                    "Test_Positive_N":
                        int(
                            y_test.sum()
                        ),

                    "Weighted_AP":
                        weighted_ap,

                    "Weighted_ROC_AUC":
                        weighted_auc,
                }
            )


            metadata = (
                realized_model_metadata(
                    estimator=estimator,
                    model_name=model_name,
                )
            )


            metadata.update(
                {
                    "Outer_Fold":
                        int(
                            outer_fold
                        ),

                    "Stage":
                        stage_name,
                }
            )


            fit_metadata_records.append(
                metadata
            )


            fold_predictions = pd.DataFrame(
                {
                    "_ROW_KEY":
                        stage_test[
                            "__ROW_KEY"
                        ].to_numpy(),

                    "Outer_Fold":
                        int(
                            outer_fold
                        ),

                    "Stage":
                        stage_name,

                    "Model":
                        model_name,

                    "True_Target":
                        y_test,

                    "SURVEY_WEIGHT":
                        test_weight,

                    "Raw_Discrimination_Score":
                        score,
                }
            )


            prediction_records.append(
                fold_predictions
            )


            print(
                f"    AP={weighted_ap:.6f} | "
                f"ROC-AUC={weighted_auc:.6f}"
            )


# ============================================================
# 21. COMBINE RESULTS
# ============================================================

fold_metrics = pd.DataFrame(
    fold_metric_records
)


outer_oof_predictions = pd.concat(
    prediction_records,
    axis=0,
    ignore_index=True,
)


fit_metadata = pd.DataFrame(
    fit_metadata_records
)


# ============================================================
# 22. EXPECTED COVERAGE QC
# ============================================================

expected_stage_n = {
    "Stage 1":
        5063,

    "Stage 2":
        int(
            master[
                "ANC_CLASS"
            ]
            .isin(
                [1, 2]
            )
            .sum()
        ),
}


if expected_stage_n[
    "Stage 2"
] != 4660:

    raise RuntimeError(
        "Expected Stage-2 population is not 4660."
    )


coverage_records = []


for stage_name in [
    "Stage 1",
    "Stage 2",
]:

    for model_name in baseline_registry[
        "Model"
    ]:

        subset = (
            outer_oof_predictions.loc[
                (
                    outer_oof_predictions[
                        "Stage"
                    ]
                    ==
                    stage_name
                )
                &
                (
                    outer_oof_predictions[
                        "Model"
                    ]
                    ==
                    model_name
                )
            ]
            .copy()
        )


        expected_n = (
            expected_stage_n[
                stage_name
            ]
        )


        coverage_records.append(
            {
                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Expected_N":
                    expected_n,

                "Observed_N":
                    int(
                        len(subset)
                    ),

                "Unique_Rows":
                    int(
                        subset[
                            "_ROW_KEY"
                        ]
                        .nunique()
                    ),

                "Duplicate_Row":
                    bool(
                        subset[
                            "_ROW_KEY"
                        ]
                        .duplicated()
                        .any()
                    ),

                "Outer_Folds":
                    ",".join(
                        map(
                            str,
                            sorted(
                                subset[
                                    "Outer_Fold"
                                ]
                                .unique()
                                .tolist()
                            ),
                        )
                    ),

                "PASS":
                    (
                        len(subset)
                        ==
                        expected_n
                        and
                        subset[
                            "_ROW_KEY"
                        ]
                        .nunique()
                        ==
                        expected_n
                        and
                        not subset[
                            "_ROW_KEY"
                        ]
                        .duplicated()
                        .any()
                        and
                        sorted(
                            subset[
                                "Outer_Fold"
                            ]
                            .unique()
                            .tolist()
                        )
                        ==
                        [0, 1, 2, 3, 4]
                    ),
            }
        )


coverage_qc = pd.DataFrame(
    coverage_records
)


if not coverage_qc[
    "PASS"
].all():

    failed = coverage_qc.loc[
        ~coverage_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "Default benchmark OOF coverage failure:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 23. FIVE-FOLD DEFAULT PERFORMANCE SUMMARY
# ============================================================
#
# These are descriptive summaries of the five INDEPENDENT
# outer-test metrics.
#
# Raw scores are NOT pooled across folds in this step.
# ============================================================

summary_records = []


for (
    stage_name,
    model_name,
), group in fold_metrics.groupby(
    [
        "Stage",
        "Model",
    ],
    sort=False,
):

    ap = group[
        "Weighted_AP"
    ].to_numpy(
        dtype=float
    )


    auc = group[
        "Weighted_ROC_AUC"
    ].to_numpy(
        dtype=float
    )


    summary_records.append(
        {
            "Stage":
                stage_name,

            "Model":
                model_name,

            "Outer_Folds":
                int(
                    len(group)
                ),

            "Mean_Weighted_AP":
                float(
                    np.mean(ap)
                ),

            "SD_Weighted_AP":
                float(
                    np.std(
                        ap,
                        ddof=1,
                    )
                ),

            "Median_Weighted_AP":
                float(
                    np.median(ap)
                ),

            "Min_Weighted_AP":
                float(
                    np.min(ap)
                ),

            "Max_Weighted_AP":
                float(
                    np.max(ap)
                ),

            "Mean_Weighted_ROC_AUC":
                float(
                    np.mean(auc)
                ),

            "SD_Weighted_ROC_AUC":
                float(
                    np.std(
                        auc,
                        ddof=1,
                    )
                ),

            "Median_Weighted_ROC_AUC":
                float(
                    np.median(auc)
                ),

            "Min_Weighted_ROC_AUC":
                float(
                    np.min(auc)
                ),

            "Max_Weighted_ROC_AUC":
                float(
                    np.max(auc)
                ),
        }
    )


performance_summary = pd.DataFrame(
    summary_records
)


# order by stage then mean AP for DISPLAY ONLY
# This ordering does NOT eliminate or select any model.

display_summary = (
    performance_summary
    .sort_values(
        [
            "Stage",
            "Mean_Weighted_AP",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 24. NO-SELECTION POLICY
# ============================================================

no_selection_policy = {
    "default_performance_used_to_drop_models":
        False,

    "default_performance_used_to_change_tuning_grid":
        False,

    "default_performance_used_to_change_predictors":
        False,

    "default_performance_used_to_change_preprocessing":
        False,

    "all_10_models_proceed_to_step19B":
        True,

    "step19B_search_space":
        (
            "unchanged frozen Step-16 family-specific "
            "12-candidate grids"
        ),

    "display_sorting_is_model_selection":
        False,

    "final_primary_model_selected_from_step19A":
        False,
}


# ============================================================
# 25. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                sha256_file(INNER_SPLIT_FILE)
                ==
                EXPECTED_INNER_SHA256,
        },

        {
            "Check":
                "Exactly 10 baseline models",

            "PASS":
                len(
                    baseline_registry
                )
                ==
                10,
        },

        {
            "Check":
                "All five outer folds evaluated",

            "PASS":
                sorted(
                    fold_metrics[
                        "Outer_Fold"
                    ]
                    .unique()
                    .tolist()
                )
                ==
                [0, 1, 2, 3, 4],
        },

        {
            "Check":
                "Both stages evaluated",

            "PASS":
                set(
                    fold_metrics[
                        "Stage"
                    ]
                )
                ==
                {
                    "Stage 1",
                    "Stage 2",
                },
        },

        {
            "Check":
                "Expected fold-metric rows = 100",

            "PASS":
                len(
                    fold_metrics
                )
                ==
                (
                    10
                    *
                    5
                    *
                    2
                ),
        },

        {
            "Check":
                "Every model-stage has five outer metrics",

            "PASS":
                bool(
                    (
                        fold_metrics
                        .groupby(
                            [
                                "Stage",
                                "Model",
                            ]
                        )
                        .size()
                        ==
                        5
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Stage 1 OOF coverage exact",

            "PASS":
                bool(
                    coverage_qc.loc[
                        coverage_qc[
                            "Stage"
                        ]
                        ==
                        "Stage 1",
                        "PASS",
                    ].all()
                ),
        },

        {
            "Check":
                "Stage 2 OOF coverage exact",

            "PASS":
                bool(
                    coverage_qc.loc[
                        coverage_qc[
                            "Stage"
                        ]
                        ==
                        "Stage 2",
                        "PASS",
                    ].all()
                ),
        },

        {
            "Check":
                "All AP metrics finite",

            "PASS":
                bool(
                    np.isfinite(
                        fold_metrics[
                            "Weighted_AP"
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All ROC-AUC metrics finite",

            "PASS":
                bool(
                    np.isfinite(
                        fold_metrics[
                            "Weighted_ROC_AUC"
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "No model eliminated from Step 19B",

            "PASS":
                (
                    no_selection_policy[
                        "all_10_models_proceed_to_step19B"
                    ]
                ),
        },

        {
            "Check":
                "Default performance cannot change tuning grid",

            "PASS":
                not no_selection_policy[
                    "default_performance_used_to_change_tuning_grid"
                ],
        },

        {
            "Check":
                "No hyperparameter tuning in Step 19A",

            "PASS":
                True,
        },

        {
            "Check":
                "No calibration in Step 19A",

            "PASS":
                True,
        },

        {
            "Check":
                "No threshold optimization in Step 19A",

            "PASS":
                True,
        },

        {
            "Check":
                "No hierarchical fusion in Step 19A",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 19A HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 26. SAVE OUTPUTS
# ============================================================

BASELINE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP19A_DEFAULT_baseline_model_registry.csv"
)

FOLD_METRIC_FILE = (
    DIRS["models"]
    / "STEP19A_DEFAULT_primary_outer_fold_metrics.csv"
)

SUMMARY_FILE = (
    DIRS["models"]
    / "STEP19A_DEFAULT_primary_5fold_summary.csv"
)

PREDICTION_FILE = (
    DIRS["predictions"]
    / "STEP19A_DEFAULT_primary_outer_OOF_RESTRICTED.csv"
)

FIT_METADATA_FILE = (
    DIRS["models"]
    / "STEP19A_DEFAULT_primary_realized_model_metadata.csv"
)

COVERAGE_QC_FILE = (
    DIRS["qc"]
    / "STEP19A_DEFAULT_primary_OOF_coverage_qc.csv"
)

HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP19A_DEFAULT_primary_hard_qc.csv"
)

NO_SELECTION_FILE = (
    DIRS["protocol"]
    / "STEP19A_DEFAULT_no_model_elimination_policy.json"
)


baseline_registry.to_csv(
    BASELINE_REGISTRY_FILE,
    index=False,
)

fold_metrics.to_csv(
    FOLD_METRIC_FILE,
    index=False,
)

performance_summary.to_csv(
    SUMMARY_FILE,
    index=False,
)

outer_oof_predictions.to_csv(
    PREDICTION_FILE,
    index=False,
)

fit_metadata.to_csv(
    FIT_METADATA_FILE,
    index=False,
)

coverage_qc.to_csv(
    COVERAGE_QC_FILE,
    index=False,
)

hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


with open(
    NO_SELECTION_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        no_selection_policy,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 27. EXECUTION MANIFEST
# ============================================================

manifest = {
    "step":
        "STEP 19A",

    "status":
        "ALL_MODEL_DEFAULT_BASELINE_BENCHMARK_COMPLETE",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "pipeline":
        PIPELINE_NAME,

    "benchmark":
        BENCHMARK_NAME,

    "model_family_count":
        10,

    "outer_fold_count":
        5,

    "hierarchical_stage_count":
        2,

    "model_fits_expected":
        100,

    "hyperparameter_tuning_performed":
        False,

    "calibration_performed":
        False,

    "threshold_optimization_performed":
        False,

    "hierarchical_fusion_performed":
        False,

    "default_performance_used_for_model_elimination":
        False,

    "all_models_proceed_to_tuning":
        True,

    "primary_master_sha256":
        sha256_file(
            PRIMARY_FILE
        ),

    "outer_split_sha256":
        sha256_file(
            OUTER_SPLIT_FILE
        ),

    "inner_split_sha256":
        sha256_file(
            INNER_SPLIT_FILE
        ),

    "baseline_registry_sha256":
        sha256_file(
            BASELINE_REGISTRY_FILE
        ),

    "fold_metrics_sha256":
        sha256_file(
            FOLD_METRIC_FILE
        ),

    "summary_sha256":
        sha256_file(
            SUMMARY_FILE
        ),

    "outer_oof_prediction_sha256":
        sha256_file(
            PREDICTION_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP19A_DEFAULT_primary_execution_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 28. FINAL IMMUTABILITY CHECK
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master changed during Step 19A."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer split changed during Step 19A."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner split changed during Step 19A."
    )


# ============================================================
# 29. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 19A — ALL 10 DEFAULT / BASELINE MODEL BENCHMARK COMPLETE"
)

print(
    "=" * 78
)


print(
    "\nBASELINE MODEL DEFINITIONS:"
)

display(
    baseline_registry
)


print(
    "\nFIVE-FOLD DEFAULT PERFORMANCE SUMMARY:"
)

display(
    display_summary
)


print(
    "\nOUTER-FOLD METRICS:"
)

display(
    fold_metrics.sort_values(
        [
            "Stage",
            "Model",
            "Outer_Fold",
        ]
    )
)


print(
    "\nOOF COVERAGE QC:"
)

display(
    coverage_qc
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nIMPORTANT:"
)

print(
    "- All 10 model families were evaluated."
)

print(
    "- No model family was eliminated."
)

print(
    "- Default performance will NOT change the frozen tuning grids."
)

print(
    "- All 10 models proceed to Step 19B tuning."
)

print(
    "- No calibration was performed."
)

print(
    "- No threshold was optimized."
)

print(
    "- No final model was selected."
)

print(
    "- Fold-wise raw AP/AUC were summarized; raw scores were not pooled across folds."
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)

print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)

print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 19A DEFAULT BENCHMARK PASS"
)

print(
    "=" * 78
)


------------------------------------------------------------------------------
OUTER FOLD 0
------------------------------------------------------------------------------

Stage 1: train N=4057, test N=1006, train positives=321, test positives=82
  fitting ElasticNet logistic regression ...
    AP=0.252432 | ROC-AUC=0.819348
  fitting RBF SVM ...


E:\Python\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


    AP=0.168503 | ROC-AUC=0.642331
  fitting Random Forest ...
    AP=0.180381 | ROC-AUC=0.734321
  fitting ExtraTrees ...
    AP=0.171671 | ROC-AUC=0.737022
  fitting AdaBoost ...
    AP=0.236754 | ROC-AUC=0.807014
  fitting GradientBoosting ...
    AP=0.227866 | ROC-AUC=0.790422
  fitting HistGradientBoosting ...
    AP=0.212561 | ROC-AUC=0.777457
  fitting XGBoost ...
    AP=0.199626 | ROC-AUC=0.762085
  fitting LightGBM ...
    AP=0.202330 | ROC-AUC=0.767472
  fitting CatBoost ...
    AP=0.212603 | ROC-AUC=0.785907

Stage 2: train N=3736, test N=924, train positives=201, test positives=52
  fitting ElasticNet logistic regression ...
    AP=0.181543 | ROC-AUC=0.770924
  fitting RBF SVM ...


E:\Python\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


    AP=0.103231 | ROC-AUC=0.625243
  fitting Random Forest ...
    AP=0.133699 | ROC-AUC=0.721062
  fitting ExtraTrees ...
    AP=0.112961 | ROC-AUC=0.687117
  fitting AdaBoost ...
    AP=0.204859 | ROC-AUC=0.763085
  fitting GradientBoosting ...
    AP=0.154049 | ROC-AUC=0.742613
  fitting HistGradientBoosting ...
    AP=0.172046 | ROC-AUC=0.755390
  fitting XGBoost ...
    AP=0.112395 | ROC-AUC=0.681262
  fitting LightGBM ...
    AP=0.138892 | ROC-AUC=0.732719
  fitting CatBoost ...
    AP=0.150758 | ROC-AUC=0.742195

------------------------------------------------------------------------------
OUTER FOLD 1
------------------------------------------------------------------------------

Stage 1: train N=4050, test N=1013, train positives=326, test positives=77
  fitting ElasticNet logistic regression ...
    AP=0.202089 | ROC-AUC=0.778252
  fitting RBF SVM ...


E:\Python\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


... (21 outputs left out)

,Stage,Model,Expected_N,Observed_N,Unique_Rows,Duplicate_Row,Outer_Folds,PASS
0,Stage 1,ElasticNet logistic regression,5063,5063,5063,False,"0,1,2,3,4",True
1,Stage 1,RBF SVM,5063,5063,5063,False,"0,1,2,3,4",True
2,Stage 1,Random Forest,5063,5063,5063,False,"0,1,2,3,4",True
3,Stage 1,ExtraTrees,5063,5063,5063,False,"0,1,2,3,4",True
4,Stage 1,AdaBoost,5063,5063,5063,False,"0,1,2,3,4",True
5,Stage 1,GradientBoosting,5063,5063,5063,False,"0,1,2,3,4",True
6,Stage 1,HistGradientBoosting,5063,5063,5063,False,"0,1,2,3,4",True
7,Stage 1,XGBoost,5063,5063,5063,False,"0,1,2,3,4",True
8,Stage 1,LightGBM,5063,5063,5063,False,"0,1,2,3,4",True
9,Stage 1,CatBoost,5063,5063,5063,False,"0,1,2,3,4",True



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Exactly 10 baseline models,True
4,All five outer folds evaluated,True
5,Both stages evaluated,True
6,Expected fold-metric rows = 100,True
7,Every model-stage has five outer metrics,True
8,Stage 1 OOF coverage exact,True
9,Stage 2 OOF coverage exact,True



IMPORTANT:
- All 10 model families were evaluated.
- No model family was eliminated.
- Default performance will NOT change the frozen tuning grids.
- All 10 models proceed to Step 19B tuning.
- No calibration was performed.
- No threshold was optimized.
- No final model was selected.
- Fold-wise raw AP/AUC were summarized; raw scores were not pooled across folds.

INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Outer split    : 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
Inner split    : 2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633
STATUS: STEP 19A DEFAULT BENCHMARK PASS


In [3]:
# ============================================================
# STEP 19B — ALL 10 MODEL FAMILIES
#            FAMILY-SPECIFIC NESTED HYPERPARAMETER TUNING
#            PRIMARY SURVEY-WEIGHTED PIPELINE
# ============================================================
#
# PURPOSE
# -------
# Tune EVERY model family independently.
#
# NO family is removed based on Step-19A default performance.
# NO tuning grid is changed based on Step-19A results.
#
#
# For each:
#
#   outer fold
#       x hierarchical stage
#       x model family
#
#       12 frozen candidates
#               ↓
#       4-fold inner CV
#               ↓
#       pooled inner-OOF raw score
#               ↓
#       survey-weighted AP
#               ↓
#       tie: survey-weighted ROC-AUC
#               ↓
#       tie: deterministic Step-16 rank
#               ↓
#       family-specific tuned winner
#               ↓
#       refit winner on full outer-training set
#               ↓
#       untouched outer-test prediction
#
#
# PRIMARY TRAINING POLICY
# -----------------------
# Base-model fitting:
#   normalized DHS survey weights, mean = 1 within the
#   current model-training partition.
#
# Inner validation and outer evaluation:
#   original DHS survey weights.
#
#
# IMPORTANT
# ---------
# Step 19B does NOT:
#
#   - jointly select the final model family
#   - calibrate probabilities
#   - optimize thresholds
#   - perform hierarchical probability fusion
#
# All 10 tuned families survive Step 19B.
#
#
# CHECKPOINTING
# -------------
# Each completed outer fold is saved separately.
# If the kernel restarts, rerunning this cell will reuse valid
# completed-fold checkpoints.
# ============================================================


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from pathlib import Path

import ast
import datetime as dt
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from IPython.display import display


# ============================================================
# 2. EXECUTION CONSTANTS
# ============================================================

SEED = 2026

PIPELINE_NAME = (
    "PRIMARY_SURVEY_WEIGHTED"
)

BENCHMARK_NAME = (
    "FAMILY_SPECIFIC_TUNED"
)

EXECUTION_SCHEMA_VERSION = (
    "STEP19B_v1"
)

METRIC_TIE_TOLERANCE = 1e-12


# ============================================================
# 3. FROZEN INPUT FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)

OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)

INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)

CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)

STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)

STEP17A_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17A_FINAL_predictor_feature_selection_manifest.json"
)

STEP18_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_manifest.json"
)

STEP19A_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP19A_DEFAULT_primary_execution_manifest.json"
)

STEP19A_BASELINE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP19A_DEFAULT_baseline_model_registry.csv"
)

STEP19A_FOLD_METRICS_FILE = (
    DIRS["models"]
    / "STEP19A_DEFAULT_primary_outer_fold_metrics.csv"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)

EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)

EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


# ============================================================
# 4. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):

            digest.update(
                block
            )

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(
            handle
        )


def parse_parameter_dictionary(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        return value.copy()


    if pd.isna(
        value
    ):

        return {}


    text = str(
        value
    ).strip()


    if text == "":

        return {}


    try:

        parsed = json.loads(
            text
        )


    except json.JSONDecodeError:

        parsed = ast.literal_eval(
            text
        )


    if not isinstance(
        parsed,
        dict,
    ):

        raise RuntimeError(
            "Parameter field did not decode to a dictionary."
        )


    return parsed


# ============================================================
# 5. REQUIRED-FILE CHECK
# ============================================================

required_files = [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    CANDIDATE_REGISTRY_FILE,
    STEP16_MANIFEST_FILE,
    STEP17A_MANIFEST_FILE,
    STEP18_MANIFEST_FILE,
    STEP19A_MANIFEST_FILE,
    STEP19A_BASELINE_REGISTRY_FILE,
    STEP19A_FOLD_METRICS_FILE,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )


# ============================================================
# 6. IMMUTABILITY CHECK
# ============================================================

if (
    sha256_file(
        PRIMARY_FILE
    )
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    sha256_file(
        OUTER_SPLIT_FILE
    )
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if (
    sha256_file(
        INNER_SPLIT_FILE
    )
    !=
    EXPECTED_INNER_SHA256
):

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


# ============================================================
# 7. UPSTREAM PROTOCOL CHECK
# ============================================================

step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)

step17a_manifest = read_json(
    STEP17A_MANIFEST_FILE
)

step18_manifest = read_json(
    STEP18_MANIFEST_FILE
)

step19a_manifest = read_json(
    STEP19A_MANIFEST_FILE
)


if (
    step16_manifest.get("status")
    !=
    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 16 is not frozen."
    )


if (
    step17a_manifest.get("status")
    !=
    "FULL_PREDICTOR_NO_EXTERNAL_FEATURE_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 17A is not frozen."
    )


if (
    step18_manifest.get("status")
    !=
    "THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 18 is not frozen."
    )


if (
    step19a_manifest.get("status")
    !=
    "ALL_MODEL_DEFAULT_BASELINE_BENCHMARK_COMPLETE"
):

    raise RuntimeError(
        "Step 19A default benchmark is not complete."
    )


if (
    step19a_manifest.get(
        "all_models_proceed_to_tuning"
    )
    is not True
):

    raise RuntimeError(
        "Step 19A did not preserve all model families."
    )


# ============================================================
# 8. CANDIDATE REGISTRY CHECK
# ============================================================

candidate_registry = pd.read_csv(
    CANDIDATE_REGISTRY_FILE
)


required_candidate_columns = {
    "Candidate_ID",
    "Model",
    "Preprocessing",
    "Deterministic_Tie_Rank",
    "Tuned_Params_JSON",
    "Fixed_Params_JSON",
}


missing_candidate_columns = (
    required_candidate_columns
    -
    set(
        candidate_registry.columns
    )
)


if missing_candidate_columns:

    raise RuntimeError(
        "Candidate registry missing columns:\n"
        f"{sorted(missing_candidate_columns)}"
    )


if len(
    candidate_registry
) != 120:

    raise RuntimeError(
        "Candidate registry does not contain 120 candidates."
    )


candidate_count_by_family = (
    candidate_registry
    .groupby(
        "Model"
    )
    .size()
)


if (
    candidate_count_by_family
    !=
    12
).any():

    raise RuntimeError(
        "Each model family must contain exactly 12 candidates:\n"
        f"{candidate_count_by_family}"
    )


if (
    step16_manifest.get(
        "candidate_registry_sha256"
    )
    is not None
):

    if (
        sha256_file(
            CANDIDATE_REGISTRY_FILE
        )
        !=
        step16_manifest[
            "candidate_registry_sha256"
        ]
    ):

        raise RuntimeError(
            "Candidate-registry hash mismatch."
        )


CANDIDATE_REGISTRY_SHA256 = (
    sha256_file(
        CANDIDATE_REGISTRY_FILE
    )
)


# ============================================================
# 9. VERIFY SAME 10 MODEL FAMILIES AS STEP 19A
# ============================================================

baseline_registry = pd.read_csv(
    STEP19A_BASELINE_REGISTRY_FILE
)


MODEL_ORDER = (
    baseline_registry[
        "Model"
    ]
    .tolist()
)


if len(
    MODEL_ORDER
) != 10:

    raise RuntimeError(
        "Step 19A baseline registry does not contain 10 models."
    )


if set(
    candidate_registry[
        "Model"
    ]
) != set(
    MODEL_ORDER
):

    raise RuntimeError(
        "Step 16 tuning families do not exactly match "
        "Step 19A default families.\n\n"
        f"Step 19A:\n{MODEL_ORDER}\n\n"
        f"Step 16:\n"
        f"{sorted(candidate_registry['Model'].unique())}"
    )


# ============================================================
# 10. FROZEN PREDICTORS
# ============================================================

NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]


ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]


NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "division",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]


BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]


CATEGORICAL_PREDICTORS = (
    ORDERED_CATEGORICAL_PREDICTORS
    +
    NOMINAL_CATEGORICAL_PREDICTORS
    +
    BINARY_CATEGORICAL_PREDICTORS
)


PREDICTORS = (
    NUMERIC_PREDICTORS
    +
    CATEGORICAL_PREDICTORS
)


if len(
    PREDICTORS
) != 15:

    raise RuntimeError(
        "Frozen predictor count is not 15."
    )


# ============================================================
# 11. LOAD MASTER / SPLITS
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)

outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)

inner_assignment = pd.read_csv(
    INNER_SPLIT_FILE
)


if master.shape != (
    5063,
    23,
):

    raise RuntimeError(
        f"Unexpected master shape: {master.shape}"
    )


# ============================================================
# 12. ROBUST COLUMN IDENTIFICATION
# ============================================================

def normalize_name(
    value,
):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column
        in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(
            alias
        )


        if key in lookup:

            return lookup[
                key
            ]


    raise RuntimeError(
        "Required column could not be identified.\n"
        f"Aliases: {aliases}\n"
        f"Available: {list(dataframe.columns)}"
    )


MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


INNER_ROW_COL = find_column(
    inner_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


INNER_OUTER_FOLD_COL = find_column(
    inner_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


INNER_FOLD_COL = find_column(
    inner_assignment,
    [
        "INNER_FOLD",
        "inner_fold",
        "INNER_FOLD_ID",
        "inner_fold_id",
    ],
)


# ============================================================
# 13. CANONICAL ROW KEY
# ============================================================

def canonical_row_key(
    series,
):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(
                dtype=float
            ),
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ).astype(
                np.int64
            ),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


master = master.copy()

outer_assignment = outer_assignment.copy()

inner_assignment = inner_assignment.copy()


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


inner_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    inner_assignment[
        INNER_ROW_COL
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


inner_assignment[
    INNER_OUTER_FOLD_COL
] = pd.to_numeric(
    inner_assignment[
        INNER_OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


inner_assignment[
    INNER_FOLD_COL
] = pd.to_numeric(
    inner_assignment[
        INNER_FOLD_COL
    ],
    errors="raise",
).astype(int)


# ============================================================
# 14. SPLIT INTEGRITY
# ============================================================

if master[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate master row key."
    )


if outer_assignment[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate outer-assignment row key."
    )


if set(
    master[
        "__ROW_KEY"
    ]
) != set(
    outer_assignment[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Outer assignments do not exactly cover master."
    )


outer_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__OUTER_FOLD"
] = master[
    "__ROW_KEY"
].map(
    outer_map
)


if master[
    "__OUTER_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing outer-fold assignment."
    )


master[
    "__OUTER_FOLD"
] = master[
    "__OUTER_FOLD"
].astype(int)


OUTER_FOLDS = sorted(
    master[
        "__OUTER_FOLD"
    ]
    .unique()
    .tolist()
)


if OUTER_FOLDS != [
    0,
    1,
    2,
    3,
    4,
]:

    raise RuntimeError(
        f"Unexpected outer folds: {OUTER_FOLDS}"
    )


# ============================================================
# 15. OUTCOME / WEIGHT CHECK
# ============================================================

master[
    "ANC_CLASS"
] = pd.to_numeric(
    master[
        "ANC_CLASS"
    ],
    errors="raise",
).astype(int)


if set(
    master[
        "ANC_CLASS"
    ].unique()
) != {
    0,
    1,
    2,
}:

    raise RuntimeError(
        "ANC_CLASS is not exactly {0,1,2}."
    )


weight_vector = (
    master[
        "SURVEY_WEIGHT"
    ]
    .to_numpy(
        dtype=float
    )
)


if not np.isfinite(
    weight_vector
).all():

    raise RuntimeError(
        "Non-finite survey weight."
    )


if (
    weight_vector
    <=
    0
).any():

    raise RuntimeError(
        "Survey weights must be positive."
    )


# ============================================================
# 16. TRAINING WEIGHT
# ============================================================

def normalized_training_weight(
    dataframe,
):

    weight = (
        dataframe[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    normalized = (
        weight
        /
        weight.mean()
    )


    if not np.isclose(
        normalized.mean(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Training weight normalization failed."
        )


    return normalized


# ============================================================
# 17. STAGE DATA
# ============================================================

def prepare_stage_data(
    dataframe,
    stage_name,
):

    data = dataframe.copy()


    if stage_name == "Stage 1":

        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            0
        ).astype(int)


    elif stage_name == "Stage 2":

        data = (
            data.loc[
                data[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
            ]
            .copy()
        )


        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            2
        ).astype(int)


    else:

        raise ValueError(
            f"Unknown stage: {stage_name}"
        )


    return (
        data
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 18. PREPROCESSING
# ============================================================

def build_representations(
    train_df,
    validation_df,
):

    # --------------------------------------------------------
    # Numeric
    # --------------------------------------------------------

    train_numeric = (
        train_df[
            NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    validation_numeric = (
        validation_df[
            NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    # --------------------------------------------------------
    # One-hot encoding learned only from current training set
    # --------------------------------------------------------

    train_cat = (
        train_df[
            CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    validation_cat = (
        validation_df[
            CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    encoder = OneHotEncoder(
        drop=None,
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float64,
    )


    train_ohe = (
        encoder.fit_transform(
            train_cat
        )
    )


    validation_ohe = (
        encoder.transform(
            validation_cat
        )
    )


    train_unscaled_ohe = np.hstack(
        [
            train_numeric,
            train_ohe,
        ]
    )


    validation_unscaled_ohe = np.hstack(
        [
            validation_numeric,
            validation_ohe,
        ]
    )


    # --------------------------------------------------------
    # Numeric scaling learned only from current training set
    # --------------------------------------------------------

    scaler = StandardScaler()


    train_numeric_scaled = (
        scaler.fit_transform(
            train_numeric
        )
    )


    validation_numeric_scaled = (
        scaler.transform(
            validation_numeric
        )
    )


    train_scaled_ohe = np.hstack(
        [
            train_numeric_scaled,
            train_ohe,
        ]
    )


    validation_scaled_ohe = np.hstack(
        [
            validation_numeric_scaled,
            validation_ohe,
        ]
    )


    # --------------------------------------------------------
    # Native categorical:
    # HGB + LightGBM
    # --------------------------------------------------------

    train_native = (
        train_df[
            PREDICTORS
        ]
        .copy()
    )


    validation_native = (
        validation_df[
            PREDICTORS
        ]
        .copy()
    )


    for column in NUMERIC_PREDICTORS:

        train_native[
            column
        ] = pd.to_numeric(
            train_native[
                column
            ],
            errors="raise",
        ).astype(float)


        validation_native[
            column
        ] = pd.to_numeric(
            validation_native[
                column
            ],
            errors="raise",
        ).astype(float)


    unseen_native = {}


    for column in CATEGORICAL_PREDICTORS:

        train_values = (
            train_native[
                column
            ]
            .astype(str)
        )


        validation_values = (
            validation_native[
                column
            ]
            .astype(str)
        )


        categories = list(
            pd.unique(
                train_values
            )
        )


        unseen = sorted(
            set(
                validation_values
            )
            -
            set(
                categories
            )
        )


        if unseen:

            unseen_native[
                column
            ] = unseen


        train_native[
            column
        ] = pd.Categorical(
            train_values,
            categories=categories,
            ordered=False,
        )


        validation_native[
            column
        ] = pd.Categorical(
            validation_values,
            categories=categories,
            ordered=False,
        )


    if unseen_native:

        raise RuntimeError(
            "Unexpected unseen native categorical levels:\n"
            f"{unseen_native}"
        )


    # --------------------------------------------------------
    # CatBoost
    # --------------------------------------------------------

    train_catboost = (
        train_df[
            PREDICTORS
        ]
        .copy()
    )


    validation_catboost = (
        validation_df[
            PREDICTORS
        ]
        .copy()
    )


    for column in NUMERIC_PREDICTORS:

        train_catboost[
            column
        ] = pd.to_numeric(
            train_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


        validation_catboost[
            column
        ] = pd.to_numeric(
            validation_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in CATEGORICAL_PREDICTORS:

        train_catboost[
            column
        ] = (
            train_catboost[
                column
            ]
            .astype(str)
        )


        validation_catboost[
            column
        ] = (
            validation_catboost[
                column
            ]
            .astype(str)
        )


    return {
        "Scaled OHE":
            (
                train_scaled_ohe,
                validation_scaled_ohe,
            ),

        "Unscaled OHE":
            (
                train_unscaled_ohe,
                validation_unscaled_ohe,
            ),

        "Native categorical":
            (
                train_native,
                validation_native,
            ),

        "CatBoost native":
            (
                train_catboost,
                validation_catboost,
            ),
    }


# ============================================================
# 19. MODEL CONSTRUCTOR
# ============================================================

def instantiate_candidate(
    candidate_row,
):

    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    fixed = parse_parameter_dictionary(
        candidate_row[
            "Fixed_Params_JSON"
        ]
    )


    tuned = parse_parameter_dictionary(
        candidate_row[
            "Tuned_Params_JSON"
        ]
    )


    params = {
        **fixed,
        **tuned,
    }


    # --------------------------------------------------------
    # ElasticNet logistic
    # --------------------------------------------------------

    if model_name == "ElasticNet logistic regression":

        return LogisticRegression(
            **params
        )


    # --------------------------------------------------------
    # SVC
    #
    # sklearn 1.9 deprecates the probability parameter.
    # Frozen Step-16 policy is probability=False.
    # Removing the explicit False preserves identical behaviour
    # while avoiding the deprecation warning.
    # --------------------------------------------------------

    if model_name == "RBF SVM":

        probability_value = (
            params.pop(
                "probability",
                False,
            )
        )


        if probability_value not in [
            False,
            None,
        ]:

            raise RuntimeError(
                "Frozen SVM candidate unexpectedly requests "
                "probability=True."
            )


        return SVC(
            **params
        )


    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------

    if model_name == "Random Forest":

        return RandomForestClassifier(
            **params
        )


    # --------------------------------------------------------
    # ExtraTrees
    # --------------------------------------------------------

    if model_name == "ExtraTrees":

        return ExtraTreesClassifier(
            **params
        )


    # --------------------------------------------------------
    # AdaBoost
    # --------------------------------------------------------

    if model_name == "AdaBoost":

        depth_aliases = {
            "estimatormaxdepth",
            "baseestimatormaxdepth",
            "stumpmaxdepth",
            "decisiontreemaxdepth",
            "maxdepth",
        }


        depth_keys = [
            key

            for key
            in list(
                params.keys()
            )

            if normalize_name(
                key
            )
            in depth_aliases
        ]


        if len(
            depth_keys
        ) != 1:

            raise RuntimeError(
                "Could not uniquely identify AdaBoost "
                "stump-depth parameter.\n"
                f"Parameters: {params}"
            )


        depth_key = (
            depth_keys[0]
        )


        depth = int(
            params.pop(
                depth_key
            )
        )


        # Remove any serialized estimator placeholder.
        if (
            "estimator"
            in params
            and
            not hasattr(
                params[
                    "estimator"
                ],
                "fit",
            )
        ):

            params.pop(
                "estimator"
            )


        base_tree = (
            DecisionTreeClassifier(
                max_depth=depth,
                random_state=SEED,
            )
        )


        return AdaBoostClassifier(
            estimator=base_tree,
            **params,
        )


    # --------------------------------------------------------
    # Gradient Boosting
    # --------------------------------------------------------

    if model_name == "GradientBoosting":

        return GradientBoostingClassifier(
            **params
        )


    # --------------------------------------------------------
    # HistGradientBoosting
    # --------------------------------------------------------

    if model_name == "HistGradientBoosting":

        return HistGradientBoostingClassifier(
            **params
        )


    # --------------------------------------------------------
    # XGBoost
    # --------------------------------------------------------

    if model_name == "XGBoost":

        return XGBClassifier(
            **params
        )


    # --------------------------------------------------------
    # LightGBM
    # --------------------------------------------------------

    if model_name == "LightGBM":

        return LGBMClassifier(
            **params
        )


    # --------------------------------------------------------
    # CatBoost
    # --------------------------------------------------------

    if model_name == "CatBoost":

        return CatBoostClassifier(
            **params
        )


    raise RuntimeError(
        f"Unknown model family: {model_name}"
    )


# ============================================================
# 20. FIT MODEL
# ============================================================

def fit_candidate(
    estimator,
    model_name,
    X_train,
    y_train,
    train_weight,
):

    with warnings.catch_warnings():

        warnings.simplefilter(
            "error",
            ConvergenceWarning,
        )


        if model_name == "CatBoost":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=train_weight,
                cat_features=CATEGORICAL_PREDICTORS,
                verbose=False,
            )


        elif model_name == "LightGBM":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=train_weight,
                categorical_feature=CATEGORICAL_PREDICTORS,
            )


        else:

            estimator.fit(
                X_train,
                y_train,
                sample_weight=train_weight,
            )


    return estimator


# ============================================================
# 21. RAW DISCRIMINATION SCORE
# ============================================================

def positive_raw_score(
    estimator,
    model_name,
    X,
):

    classes = np.asarray(
        estimator.classes_
    )


    if set(
        classes.tolist()
    ) != {
        0,
        1,
    }:

        raise RuntimeError(
            f"{model_name}: unexpected classes {classes}"
        )


    if model_name == "RBF SVM":

        score = np.asarray(
            estimator.decision_function(
                X
            ),
            dtype=float,
        ).reshape(-1)


    else:

        probability = np.asarray(
            estimator.predict_proba(
                X
            ),
            dtype=float,
        )


        positive_index = int(
            np.where(
                classes
                ==
                1
            )[0][0]
        )


        score = probability[
            :,
            positive_index
        ]


        if (
            (score < 0.0)
            |
            (score > 1.0)
        ).any():

            raise RuntimeError(
                f"{model_name}: predicted probability outside [0,1]."
            )


    if not np.isfinite(
        score
    ).all():

        raise RuntimeError(
            f"{model_name}: non-finite score."
        )


    return score


# ============================================================
# 22. REPRESENTATION LOOKUP
# ============================================================

def model_representation(
    representation_cache,
    model_name,
    preprocessing_name,
):

    if model_name == "CatBoost":

        return representation_cache[
            "CatBoost native"
        ]


    if preprocessing_name not in representation_cache:

        raise RuntimeError(
            f"Unknown preprocessing representation: "
            f"{preprocessing_name}"
        )


    return representation_cache[
        preprocessing_name
    ]


# ============================================================
# 23. INNER CACHE
# ============================================================

def build_inner_cache(
    stage_train,
):

    cache = {}


    inner_folds = sorted(
        stage_train[
            "__INNER_FOLD"
        ]
        .unique()
        .tolist()
    )


    if len(
        inner_folds
    ) != 4:

        raise RuntimeError(
            f"Expected four inner folds; observed {inner_folds}"
        )


    for inner_fold in inner_folds:

        validation_mask = (
            stage_train[
                "__INNER_FOLD"
            ]
            ==
            inner_fold
        )


        train_positions = np.where(
            ~validation_mask.to_numpy()
        )[0]


        validation_positions = np.where(
            validation_mask.to_numpy()
        )[0]


        inner_train = (
            stage_train.iloc[
                train_positions
            ]
            .copy()
        )


        inner_validation = (
            stage_train.iloc[
                validation_positions
            ]
            .copy()
        )


        y_train = (
            inner_train[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        y_validation = (
            inner_validation[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        if set(
            np.unique(
                y_train
            )
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"Inner train fold {inner_fold} lacks both classes."
            )


        if set(
            np.unique(
                y_validation
            )
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"Inner validation fold {inner_fold} lacks both classes."
            )


        cache[
            inner_fold
        ] = {
            "train_positions":
                train_positions,

            "validation_positions":
                validation_positions,

            "y_train":
                y_train,

            "y_validation":
                y_validation,

            "train_weight":
                normalized_training_weight(
                    inner_train
                ),

            "validation_weight":
                inner_validation[
                    "SURVEY_WEIGHT"
                ]
                .to_numpy(
                    dtype=float
                ),

            "representations":
                build_representations(
                    inner_train,
                    inner_validation,
                ),
        }


    return cache


# ============================================================
# 24. FAMILY-SPECIFIC INNER TUNING
# ============================================================

def tune_one_family(
    stage_train,
    inner_cache,
    outer_fold,
    stage_name,
    model_name,
):

    family_candidates = (
        candidate_registry.loc[
            candidate_registry[
                "Model"
            ]
            ==
            model_name
        ]
        .copy()
        .sort_values(
            "Deterministic_Tie_Rank"
        )
        .reset_index(
            drop=True
        )
    )


    if len(
        family_candidates
    ) != 12:

        raise RuntimeError(
            f"{model_name}: expected 12 candidates."
        )


    n = len(
        stage_train
    )


    y_all = (
        stage_train[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        )
    )


    weight_all = (
        stage_train[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    candidate_rows = []

    successful_oof = {}


    inner_folds = sorted(
        inner_cache.keys()
    )


    for _, candidate_row in (
        family_candidates.iterrows()
    ):

        candidate_id = str(
            candidate_row[
                "Candidate_ID"
            ]
        )


        oof_score = np.full(
            n,
            np.nan,
            dtype=float,
        )


        candidate_error = None


        try:

            for inner_fold in inner_folds:

                fold_info = (
                    inner_cache[
                        inner_fold
                    ]
                )


                X_train, X_validation = (
                    model_representation(
                        representation_cache=fold_info[
                            "representations"
                        ],
                        model_name=model_name,
                        preprocessing_name=candidate_row[
                            "Preprocessing"
                        ],
                    )
                )


                estimator = (
                    instantiate_candidate(
                        candidate_row
                    )
                )


                estimator = (
                    fit_candidate(
                        estimator=estimator,
                        model_name=model_name,
                        X_train=X_train,
                        y_train=fold_info[
                            "y_train"
                        ],
                        train_weight=fold_info[
                            "train_weight"
                        ],
                    )
                )


                validation_score = (
                    positive_raw_score(
                        estimator=estimator,
                        model_name=model_name,
                        X=X_validation,
                    )
                )


                oof_score[
                    fold_info[
                        "validation_positions"
                    ]
                ] = validation_score


            if not np.isfinite(
                oof_score
            ).all():

                raise RuntimeError(
                    "Incomplete inner-OOF score."
                )


            weighted_ap = float(
                average_precision_score(
                    y_all,
                    oof_score,
                    sample_weight=weight_all,
                )
            )


            weighted_auc = float(
                roc_auc_score(
                    y_all,
                    oof_score,
                    sample_weight=weight_all,
                )
            )


            successful_oof[
                candidate_id
            ] = oof_score


            status = "SUCCESS"


        except Exception as exc:

            weighted_ap = np.nan

            weighted_auc = np.nan

            status = "FAILED"

            candidate_error = (
                f"{type(exc).__name__}: {exc}"
            )


        candidate_rows.append(
            {
                "Pipeline":
                    PIPELINE_NAME,

                "Benchmark":
                    BENCHMARK_NAME,

                "Outer_Fold":
                    int(
                        outer_fold
                    ),

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Deterministic_Tie_Rank":
                    int(
                        candidate_row[
                            "Deterministic_Tie_Rank"
                        ]
                    ),

                "Weighted_Inner_AP":
                    weighted_ap,

                "Weighted_Inner_ROC_AUC":
                    weighted_auc,

                "Status":
                    status,

                "Error":
                    candidate_error,
            }
        )


    candidate_results = pd.DataFrame(
        candidate_rows
    )


    eligible = (
        candidate_results.loc[
            candidate_results[
                "Status"
            ]
            ==
            "SUCCESS"
        ]
        .copy()
    )


    if eligible.empty:

        raise RuntimeError(
            f"{model_name}: all 12 candidates failed."
        )


    # --------------------------------------------------------
    # Frozen selection rule:
    # 1. weighted AP
    # 2. weighted ROC-AUC
    # 3. deterministic candidate rank
    # --------------------------------------------------------

    best_ap = float(
        eligible[
            "Weighted_Inner_AP"
        ].max()
    )


    tied_ap = eligible.loc[
        np.isclose(
            eligible[
                "Weighted_Inner_AP"
            ],
            best_ap,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    best_auc = float(
        tied_ap[
            "Weighted_Inner_ROC_AUC"
        ].max()
    )


    tied_auc = tied_ap.loc[
        np.isclose(
            tied_ap[
                "Weighted_Inner_ROC_AUC"
            ],
            best_auc,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    winner = (
        tied_auc
        .sort_values(
            "Deterministic_Tie_Rank",
            ascending=True,
        )
        .iloc[0]
    )


    winner_id = str(
        winner[
            "Candidate_ID"
        ]
    )


    winner_registry_row = (
        family_candidates.loc[
            family_candidates[
                "Candidate_ID"
            ]
            .astype(str)
            ==
            winner_id
        ]
        .iloc[0]
    )


    winner_oof = (
        successful_oof[
            winner_id
        ]
        .copy()
    )


    return {
        "candidate_results":
            candidate_results,

        "winner":
            winner,

        "winner_registry_row":
            winner_registry_row,

        "winner_oof":
            winner_oof,

        "successful_candidates":
            int(
                len(
                    eligible
                )
            ),

        "failed_candidates":
            int(
                12
                -
                len(
                    eligible
                )
            ),
    }


# ============================================================
# 25. OUTER REFIT / SCORE
# ============================================================

def refit_family_winner(
    stage_train,
    stage_test,
    winner_registry_row,
):

    model_name = str(
        winner_registry_row[
            "Model"
        ]
    )


    representations = (
        build_representations(
            stage_train,
            stage_test,
        )
    )


    X_train, X_test = (
        model_representation(
            representation_cache=representations,
            model_name=model_name,
            preprocessing_name=winner_registry_row[
                "Preprocessing"
            ],
        )
    )


    estimator = (
        instantiate_candidate(
            winner_registry_row
        )
    )


    estimator = (
        fit_candidate(
            estimator=estimator,
            model_name=model_name,
            X_train=X_train,
            y_train=stage_train[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            ),
            train_weight=normalized_training_weight(
                stage_train
            ),
        )
    )


    raw_score = (
        positive_raw_score(
            estimator=estimator,
            model_name=model_name,
            X=X_test,
        )
    )


    return raw_score


# ============================================================
# 26. CHECKPOINT UTILITIES
# ============================================================

def fold_checkpoint_paths(
    outer_fold,
):

    prefix = (
        f"STEP19B_TUNED_primary_outer{outer_fold}"
    )


    return {
        "candidate_metrics":
            (
                DIRS["models"]
                / f"{prefix}_candidate_inner_metrics.csv"
            ),

        "family_winners":
            (
                DIRS["models"]
                / f"{prefix}_family_winners.csv"
            ),

        "outer_predictions":
            (
                DIRS["predictions"]
                / f"{prefix}_predictions_RESTRICTED.csv"
            ),

        "winner_inner_oof":
            (
                DIRS["calibration"]
                / f"{prefix}_family_winner_inner_OOF_RESTRICTED.csv"
            ),

        "manifest":
            (
                DIRS["logs"]
                / f"{prefix}_checkpoint_manifest.json"
            ),
    }


def checkpoint_is_valid(
    outer_fold,
):

    paths = (
        fold_checkpoint_paths(
            outer_fold
        )
    )


    if not all(
        path.exists()

        for path
        in paths.values()
    ):

        return False


    try:

        manifest = read_json(
            paths[
                "manifest"
            ]
        )


    except Exception:

        return False


    required = {
        "status":
            "STEP19B_OUTER_FOLD_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "outer_fold":
            int(
                outer_fold
            ),

        "primary_master_sha256":
            EXPECTED_PRIMARY_SHA256,

        "outer_split_sha256":
            EXPECTED_OUTER_SHA256,

        "inner_split_sha256":
            EXPECTED_INNER_SHA256,

        "candidate_registry_sha256":
            CANDIDATE_REGISTRY_SHA256,
    }


    for key, expected in required.items():

        if manifest.get(
            key
        ) != expected:

            return False


    file_hash_pairs = [
        (
            "candidate_metrics_sha256",
            paths[
                "candidate_metrics"
            ],
        ),

        (
            "family_winners_sha256",
            paths[
                "family_winners"
            ],
        ),

        (
            "outer_predictions_sha256",
            paths[
                "outer_predictions"
            ],
        ),

        (
            "winner_inner_oof_sha256",
            paths[
                "winner_inner_oof"
            ],
        ),
    ]


    for key, path in file_hash_pairs:

        if (
            manifest.get(
                key
            )
            !=
            sha256_file(
                path
            )
        ):

            return False


    return True


# ============================================================
# 27. OUTER-FOLD EXECUTION
# ============================================================

def execute_outer_fold(
    outer_fold,
):

    current_inner = (
        inner_assignment.loc[
            inner_assignment[
                INNER_OUTER_FOLD_COL
            ]
            ==
            outer_fold
        ]
        .copy()
    )


    outer_train = (
        master.loc[
            master[
                "__OUTER_FOLD"
            ]
            !=
            outer_fold
        ]
        .copy()
    )


    outer_test = (
        master.loc[
            master[
                "__OUTER_FOLD"
            ]
            ==
            outer_fold
        ]
        .copy()
    )


    if (
        set(
            outer_train[
                "__ROW_KEY"
            ]
        )
        &
        set(
            outer_test[
                "__ROW_KEY"
            ]
        )
    ):

        raise RuntimeError(
            f"Outer fold {outer_fold}: train/test overlap."
        )


    if current_inner[
        "__ROW_KEY"
    ].duplicated().any():

        raise RuntimeError(
            f"Outer fold {outer_fold}: duplicate inner assignment."
        )


    if set(
        current_inner[
            "__ROW_KEY"
        ]
    ) != set(
        outer_train[
            "__ROW_KEY"
        ]
    ):

        raise RuntimeError(
            f"Outer fold {outer_fold}: inner assignments "
            "do not exactly cover outer training data."
        )


    inner_map = (
        current_inner
        .set_index(
            "__ROW_KEY"
        )[
            INNER_FOLD_COL
        ]
    )


    outer_train[
        "__INNER_FOLD"
    ] = outer_train[
        "__ROW_KEY"
    ].map(
        inner_map
    )


    if outer_train[
        "__INNER_FOLD"
    ].isna().any():

        raise RuntimeError(
            f"Outer fold {outer_fold}: missing inner assignment."
        )


    outer_train[
        "__INNER_FOLD"
    ] = outer_train[
        "__INNER_FOLD"
    ].astype(int)


    fold_candidate_results = []

    fold_winners = []

    fold_outer_predictions = []

    fold_winner_inner_oof = []


    for stage_name in [
        "Stage 1",
        "Stage 2",
    ]:

        stage_train = (
            prepare_stage_data(
                outer_train,
                stage_name,
            )
        )


        stage_test = (
            prepare_stage_data(
                outer_test,
                stage_name,
            )
        )


        if set(
            stage_train[
                "__TARGET"
            ].unique()
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"{stage_name}: training lacks both classes."
            )


        if set(
            stage_test[
                "__TARGET"
            ].unique()
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"{stage_name}: test lacks both classes."
            )


        inner_cache = (
            build_inner_cache(
                stage_train
            )
        )


        print(
            f"\nOuter {outer_fold} | {stage_name}"
            f" | train N={len(stage_train)}"
            f" | test N={len(stage_test)}"
        )


        for model_name in MODEL_ORDER:

            tuning = tune_one_family(
                stage_train=stage_train,
                inner_cache=inner_cache,
                outer_fold=outer_fold,
                stage_name=stage_name,
                model_name=model_name,
            )


            winner = tuning[
                "winner"
            ]


            winner_registry_row = (
                tuning[
                    "winner_registry_row"
                ]
            )


            outer_raw_score = (
                refit_family_winner(
                    stage_train=stage_train.drop(
                        columns=[
                            "__INNER_FOLD",
                        ],
                        errors="ignore",
                    ),
                    stage_test=stage_test,
                    winner_registry_row=winner_registry_row,
                )
            )


            outer_weight = (
                stage_test[
                    "SURVEY_WEIGHT"
                ]
                .to_numpy(
                    dtype=float
                )
            )


            outer_y = (
                stage_test[
                    "__TARGET"
                ]
                .to_numpy(
                    dtype=int
                )
            )


            outer_ap = float(
                average_precision_score(
                    outer_y,
                    outer_raw_score,
                    sample_weight=outer_weight,
                )
            )


            outer_auc = float(
                roc_auc_score(
                    outer_y,
                    outer_raw_score,
                    sample_weight=outer_weight,
                )
            )


            if not (
                np.isfinite(
                    outer_ap
                )
                and
                np.isfinite(
                    outer_auc
                )
            ):

                raise RuntimeError(
                    f"{model_name}: non-finite outer metric."
                )


            fold_candidate_results.append(
                tuning[
                    "candidate_results"
                ]
            )


            winner_id = str(
                winner[
                    "Candidate_ID"
                ]
            )


            fold_winners.append(
                {
                    "Pipeline":
                        PIPELINE_NAME,

                    "Benchmark":
                        BENCHMARK_NAME,

                    "Outer_Fold":
                        int(
                            outer_fold
                        ),

                    "Stage":
                        stage_name,

                    "Model":
                        model_name,

                    "Winning_Candidate":
                        winner_id,

                    "Preprocessing":
                        winner_registry_row[
                            "Preprocessing"
                        ],

                    "Inner_Weighted_AP":
                        float(
                            winner[
                                "Weighted_Inner_AP"
                            ]
                        ),

                    "Inner_Weighted_ROC_AUC":
                        float(
                            winner[
                                "Weighted_Inner_ROC_AUC"
                            ]
                        ),

                    "Outer_Weighted_AP":
                        outer_ap,

                    "Outer_Weighted_ROC_AUC":
                        outer_auc,

                    "Successful_Candidates":
                        tuning[
                            "successful_candidates"
                        ],

                    "Failed_Candidates":
                        tuning[
                            "failed_candidates"
                        ],

                    "Tuned_Params_JSON":
                        winner_registry_row[
                            "Tuned_Params_JSON"
                        ],

                    "Fixed_Params_JSON":
                        winner_registry_row[
                            "Fixed_Params_JSON"
                        ],
                }
            )


            fold_outer_predictions.append(
                pd.DataFrame(
                    {
                        "_ROW_KEY":
                            stage_test[
                                "__ROW_KEY"
                            ].to_numpy(),

                        "Outer_Fold":
                            int(
                                outer_fold
                            ),

                        "Stage":
                            stage_name,

                        "Model":
                            model_name,

                        "Winning_Candidate":
                            winner_id,

                        "True_Target":
                            outer_y,

                        "SURVEY_WEIGHT":
                            outer_weight,

                        "Raw_Discrimination_Score":
                            outer_raw_score,
                    }
                )
            )


            winner_oof = (
                tuning[
                    "winner_oof"
                ]
            )


            fold_winner_inner_oof.append(
                pd.DataFrame(
                    {
                        "_ROW_KEY":
                            stage_train[
                                "__ROW_KEY"
                            ].to_numpy(),

                        "Outer_Fold":
                            int(
                                outer_fold
                            ),

                        "Inner_Fold":
                            stage_train[
                                "__INNER_FOLD"
                            ].to_numpy(
                                dtype=int
                            ),

                        "Stage":
                            stage_name,

                        "Model":
                            model_name,

                        "Winning_Candidate":
                            winner_id,

                        "True_Target":
                            stage_train[
                                "__TARGET"
                            ].to_numpy(
                                dtype=int
                            ),

                        "SURVEY_WEIGHT":
                            stage_train[
                                "SURVEY_WEIGHT"
                            ].to_numpy(
                                dtype=float
                            ),

                        "Inner_OOF_Raw_Score":
                            winner_oof,
                    }
                )
            )


            print(
                f"  {model_name}: "
                f"{winner_id} | "
                f"inner AP={float(winner['Weighted_Inner_AP']):.6f} | "
                f"outer AP={outer_ap:.6f} | "
                f"outer AUC={outer_auc:.6f} | "
                f"failures={tuning['failed_candidates']}"
            )


    candidate_metrics = pd.concat(
        fold_candidate_results,
        axis=0,
        ignore_index=True,
    )


    family_winners = pd.DataFrame(
        fold_winners
    )


    outer_predictions = pd.concat(
        fold_outer_predictions,
        axis=0,
        ignore_index=True,
    )


    winner_inner_oof = pd.concat(
        fold_winner_inner_oof,
        axis=0,
        ignore_index=True,
    )


    # --------------------------------------------------------
    # Fold QC
    # --------------------------------------------------------

    if len(
        candidate_metrics
    ) != 240:

        raise RuntimeError(
            f"Outer {outer_fold}: expected 240 "
            "candidate-metric rows."
        )


    if len(
        family_winners
    ) != 20:

        raise RuntimeError(
            f"Outer {outer_fold}: expected 20 family winners."
        )


    if (
        family_winners
        .groupby(
            [
                "Stage",
                "Model",
            ]
        )
        .size()
        .ne(
            1
        )
        .any()
    ):

        raise RuntimeError(
            f"Outer {outer_fold}: duplicate/missing family winner."
        )


    return {
        "candidate_metrics":
            candidate_metrics,

        "family_winners":
            family_winners,

        "outer_predictions":
            outer_predictions,

        "winner_inner_oof":
            winner_inner_oof,
    }


# ============================================================
# 28. EXECUTE / LOAD CHECKPOINTS
# ============================================================

all_candidate_metrics = []

all_family_winners = []

all_outer_predictions = []

all_winner_inner_oof = []


print(
    "\n"
    + "=" * 78
)

print(
    "STEP 19B — FAMILY-SPECIFIC NESTED TUNING"
)

print(
    "=" * 78
)


for outer_fold in OUTER_FOLDS:

    checkpoint_paths = (
        fold_checkpoint_paths(
            outer_fold
        )
    )


    if checkpoint_is_valid(
        outer_fold
    ):

        print(
            f"\nOuter fold {outer_fold}: "
            "valid checkpoint found — loading."
        )


        candidate_metrics = pd.read_csv(
            checkpoint_paths[
                "candidate_metrics"
            ]
        )


        family_winners = pd.read_csv(
            checkpoint_paths[
                "family_winners"
            ]
        )


        outer_predictions = pd.read_csv(
            checkpoint_paths[
                "outer_predictions"
            ]
        )


        winner_inner_oof = pd.read_csv(
            checkpoint_paths[
                "winner_inner_oof"
            ]
        )


    else:

        print(
            f"\nOuter fold {outer_fold}: executing tuning."
        )


        result = execute_outer_fold(
            outer_fold
        )


        candidate_metrics = (
            result[
                "candidate_metrics"
            ]
        )


        family_winners = (
            result[
                "family_winners"
            ]
        )


        outer_predictions = (
            result[
                "outer_predictions"
            ]
        )


        winner_inner_oof = (
            result[
                "winner_inner_oof"
            ]
        )


        candidate_metrics.to_csv(
            checkpoint_paths[
                "candidate_metrics"
            ],
            index=False,
        )


        family_winners.to_csv(
            checkpoint_paths[
                "family_winners"
            ],
            index=False,
        )


        outer_predictions.to_csv(
            checkpoint_paths[
                "outer_predictions"
            ],
            index=False,
        )


        winner_inner_oof.to_csv(
            checkpoint_paths[
                "winner_inner_oof"
            ],
            index=False,
        )


        checkpoint_manifest = {
            "status":
                "STEP19B_OUTER_FOLD_COMPLETE",

            "execution_schema_version":
                EXECUTION_SCHEMA_VERSION,

            "execution_time":
                dt.datetime.now().isoformat(
                    timespec="seconds"
                ),

            "outer_fold":
                int(
                    outer_fold
                ),

            "primary_master_sha256":
                EXPECTED_PRIMARY_SHA256,

            "outer_split_sha256":
                EXPECTED_OUTER_SHA256,

            "inner_split_sha256":
                EXPECTED_INNER_SHA256,

            "candidate_registry_sha256":
                CANDIDATE_REGISTRY_SHA256,

            "candidate_metrics_sha256":
                sha256_file(
                    checkpoint_paths[
                        "candidate_metrics"
                    ]
                ),

            "family_winners_sha256":
                sha256_file(
                    checkpoint_paths[
                        "family_winners"
                    ]
                ),

            "outer_predictions_sha256":
                sha256_file(
                    checkpoint_paths[
                        "outer_predictions"
                    ]
                ),

            "winner_inner_oof_sha256":
                sha256_file(
                    checkpoint_paths[
                        "winner_inner_oof"
                    ]
                ),
        }


        with open(
            checkpoint_paths[
                "manifest"
            ],
            "w",
            encoding="utf-8",
        ) as handle:

            json.dump(
                checkpoint_manifest,
                handle,
                indent=2,
                ensure_ascii=False,
            )


    all_candidate_metrics.append(
        candidate_metrics
    )


    all_family_winners.append(
        family_winners
    )


    all_outer_predictions.append(
        outer_predictions
    )


    all_winner_inner_oof.append(
        winner_inner_oof
    )


# ============================================================
# 29. COMBINE ALL FIVE OUTER FOLDS
# ============================================================

candidate_metrics_all = pd.concat(
    all_candidate_metrics,
    axis=0,
    ignore_index=True,
)


family_winners_all = pd.concat(
    all_family_winners,
    axis=0,
    ignore_index=True,
)


outer_predictions_all = pd.concat(
    all_outer_predictions,
    axis=0,
    ignore_index=True,
)


winner_inner_oof_all = pd.concat(
    all_winner_inner_oof,
    axis=0,
    ignore_index=True,
)


# ============================================================
# 30. GLOBAL COVERAGE QC
# ============================================================

if len(
    candidate_metrics_all
) != (
    120
    *
    5
    *
    2
):

    raise RuntimeError(
        "Expected 1200 candidate-summary rows."
    )


if len(
    family_winners_all
) != (
    10
    *
    5
    *
    2
):

    raise RuntimeError(
        "Expected 100 tuned family-winner rows."
    )


if (
    family_winners_all
    .groupby(
        [
            "Outer_Fold",
            "Stage",
            "Model",
        ]
    )
    .size()
    .ne(
        1
    )
    .any()
):

    raise RuntimeError(
        "Family-winner coverage is incomplete."
    )


# ============================================================
# 31. FAMILY-WISE TUNED PERFORMANCE SUMMARY
# ============================================================

summary_records = []


for (
    stage_name,
    model_name,
), group in family_winners_all.groupby(
    [
        "Stage",
        "Model",
    ],
    sort=False,
):

    ap = (
        group[
            "Outer_Weighted_AP"
        ]
        .to_numpy(
            dtype=float
        )
    )


    auc = (
        group[
            "Outer_Weighted_ROC_AUC"
        ]
        .to_numpy(
            dtype=float
        )
    )


    summary_records.append(
        {
            "Stage":
                stage_name,

            "Model":
                model_name,

            "Outer_Folds":
                int(
                    len(
                        group
                    )
                ),

            "Mean_Tuned_Weighted_AP":
                float(
                    np.mean(
                        ap
                    )
                ),

            "SD_Tuned_Weighted_AP":
                float(
                    np.std(
                        ap,
                        ddof=1,
                    )
                ),

            "Median_Tuned_Weighted_AP":
                float(
                    np.median(
                        ap
                    )
                ),

            "Min_Tuned_Weighted_AP":
                float(
                    np.min(
                        ap
                    )
                ),

            "Max_Tuned_Weighted_AP":
                float(
                    np.max(
                        ap
                    )
                ),

            "Mean_Tuned_Weighted_ROC_AUC":
                float(
                    np.mean(
                        auc
                    )
                ),

            "SD_Tuned_Weighted_ROC_AUC":
                float(
                    np.std(
                        auc,
                        ddof=1,
                    )
                ),

            "Median_Tuned_Weighted_ROC_AUC":
                float(
                    np.median(
                        auc
                    )
                ),

            "Min_Tuned_Weighted_ROC_AUC":
                float(
                    np.min(
                        auc
                    )
                ),

            "Max_Tuned_Weighted_ROC_AUC":
                float(
                    np.max(
                        auc
                    )
                ),
        }
    )


tuned_summary = pd.DataFrame(
    summary_records
)


# ============================================================
# 32. DEFAULT vs TUNED FOLD-BY-FOLD COMPARISON
# ============================================================

default_fold_metrics = pd.read_csv(
    STEP19A_FOLD_METRICS_FILE
)


default_for_merge = (
    default_fold_metrics[
        [
            "Outer_Fold",
            "Stage",
            "Model",
            "Weighted_AP",
            "Weighted_ROC_AUC",
        ]
    ]
    .rename(
        columns={
            "Weighted_AP":
                "Default_Weighted_AP",

            "Weighted_ROC_AUC":
                "Default_Weighted_ROC_AUC",
        }
    )
)


tuned_for_merge = (
    family_winners_all[
        [
            "Outer_Fold",
            "Stage",
            "Model",
            "Winning_Candidate",
            "Outer_Weighted_AP",
            "Outer_Weighted_ROC_AUC",
        ]
    ]
    .rename(
        columns={
            "Outer_Weighted_AP":
                "Tuned_Weighted_AP",

            "Outer_Weighted_ROC_AUC":
                "Tuned_Weighted_ROC_AUC",
        }
    )
)


default_tuned_fold_comparison = (
    default_for_merge
    .merge(
        tuned_for_merge,
        on=[
            "Outer_Fold",
            "Stage",
            "Model",
        ],
        how="inner",
        validate="one_to_one",
    )
)


if len(
    default_tuned_fold_comparison
) != 100:

    raise RuntimeError(
        "Default/tuned fold comparison does not contain 100 rows."
    )


default_tuned_fold_comparison[
    "Delta_AP_Tuned_minus_Default"
] = (
    default_tuned_fold_comparison[
        "Tuned_Weighted_AP"
    ]
    -
    default_tuned_fold_comparison[
        "Default_Weighted_AP"
    ]
)


default_tuned_fold_comparison[
    "Delta_ROC_AUC_Tuned_minus_Default"
] = (
    default_tuned_fold_comparison[
        "Tuned_Weighted_ROC_AUC"
    ]
    -
    default_tuned_fold_comparison[
        "Default_Weighted_ROC_AUC"
    ]
)


# ============================================================
# 33. DEFAULT vs TUNED SUMMARY
# ============================================================

comparison_summary_records = []


for (
    stage_name,
    model_name,
), group in (
    default_tuned_fold_comparison
    .groupby(
        [
            "Stage",
            "Model",
        ],
        sort=False,
    )
):

    comparison_summary_records.append(
        {
            "Stage":
                stage_name,

            "Model":
                model_name,

            "Mean_Default_AP":
                float(
                    group[
                        "Default_Weighted_AP"
                    ].mean()
                ),

            "Mean_Tuned_AP":
                float(
                    group[
                        "Tuned_Weighted_AP"
                    ].mean()
                ),

            "Mean_Delta_AP":
                float(
                    group[
                        "Delta_AP_Tuned_minus_Default"
                    ].mean()
                ),

            "Folds_AP_Improved":
                int(
                    (
                        group[
                            "Delta_AP_Tuned_minus_Default"
                        ]
                        >
                        0
                    ).sum()
                ),

            "Mean_Default_ROC_AUC":
                float(
                    group[
                        "Default_Weighted_ROC_AUC"
                    ].mean()
                ),

            "Mean_Tuned_ROC_AUC":
                float(
                    group[
                        "Tuned_Weighted_ROC_AUC"
                    ].mean()
                ),

            "Mean_Delta_ROC_AUC":
                float(
                    group[
                        "Delta_ROC_AUC_Tuned_minus_Default"
                    ].mean()
                ),

            "Folds_ROC_AUC_Improved":
                int(
                    (
                        group[
                            "Delta_ROC_AUC_Tuned_minus_Default"
                        ]
                        >
                        0
                    ).sum()
                ),
        }
    )


default_tuned_summary = pd.DataFrame(
    comparison_summary_records
)


# preserve model registry order for display

model_order_lookup = {
    model:
        index

    for index, model
    in enumerate(
        MODEL_ORDER
    )
}


default_tuned_summary[
    "__MODEL_ORDER"
] = (
    default_tuned_summary[
        "Model"
    ]
    .map(
        model_order_lookup
    )
)


default_tuned_summary[
    "__STAGE_ORDER"
] = (
    default_tuned_summary[
        "Stage"
    ]
    .map(
        {
            "Stage 1":
                0,

            "Stage 2":
                1,
        }
    )
)


default_tuned_summary = (
    default_tuned_summary
    .sort_values(
        [
            "__STAGE_ORDER",
            "__MODEL_ORDER",
        ]
    )
    .drop(
        columns=[
            "__STAGE_ORDER",
            "__MODEL_ORDER",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 34. CANDIDATE FAILURE SUMMARY
# ============================================================

candidate_failure_summary = (
    candidate_metrics_all
    .groupby(
        [
            "Stage",
            "Model",
        ],
        as_index=False,
    )
    .agg(
        Total_Candidate_Evaluations=(
            "Candidate_ID",
            "size",
        ),

        Successful_Candidate_Evaluations=(
            "Status",
            lambda x:
                int(
                    (
                        x
                        ==
                        "SUCCESS"
                    ).sum()
                ),
        ),

        Failed_Candidate_Evaluations=(
            "Status",
            lambda x:
                int(
                    (
                        x
                        ==
                        "FAILED"
                    ).sum()
                ),
        ),
    )
)


# ============================================================
# 35. NO OUTER-RESULT-DRIVEN MODEL ELIMINATION
# ============================================================

selection_boundary = {
    "all_10_families_tuned":
        True,

    "outer_test_metrics_used_for_family_tuning":
        False,

    "outer_test_metrics_used_to_modify_search_space":
        False,

    "outer_test_metrics_used_to_drop_model_family":
        False,

    "default_results_used_to_modify_search_space":
        False,

    "default_results_used_to_drop_model_family":
        False,

    "step19b_selects_final_model_family":
        False,

    "step19c_joint_selection_still_required":
        True,

    "step19c_joint_selection_source":
        (
            "inner-CV performance only"
        ),
}


# ============================================================
# 36. FINAL HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                (
                    sha256_file(
                        PRIMARY_FILE
                    )
                    ==
                    EXPECTED_PRIMARY_SHA256
                ),
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                (
                    sha256_file(
                        OUTER_SPLIT_FILE
                    )
                    ==
                    EXPECTED_OUTER_SHA256
                ),
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                (
                    sha256_file(
                        INNER_SPLIT_FILE
                    )
                    ==
                    EXPECTED_INNER_SHA256
                ),
        },

        {
            "Check":
                "Exactly 10 model families tuned",

            "PASS":
                (
                    family_winners_all[
                        "Model"
                    ]
                    .nunique()
                    ==
                    10
                ),
        },

        {
            "Check":
                "All five outer folds completed",

            "PASS":
                (
                    sorted(
                        family_winners_all[
                            "Outer_Fold"
                        ]
                        .unique()
                        .tolist()
                    )
                    ==
                    [
                        0,
                        1,
                        2,
                        3,
                        4,
                    ]
                ),
        },

        {
            "Check":
                "Both hierarchical stages tuned",

            "PASS":
                (
                    set(
                        family_winners_all[
                            "Stage"
                        ]
                    )
                    ==
                    {
                        "Stage 1",
                        "Stage 2",
                    }
                ),
        },

        {
            "Check":
                "Expected 1200 candidate summaries",

            "PASS":
                (
                    len(
                        candidate_metrics_all
                    )
                    ==
                    1200
                ),
        },

        {
            "Check":
                "Expected 100 family-specific winners",

            "PASS":
                (
                    len(
                        family_winners_all
                    )
                    ==
                    100
                ),
        },

        {
            "Check":
                "Each family has one winner per outer-fold/stage",

            "PASS":
                bool(
                    (
                        family_winners_all
                        .groupby(
                            [
                                "Outer_Fold",
                                "Stage",
                                "Model",
                            ]
                        )
                        .size()
                        ==
                        1
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All outer AP values finite",

            "PASS":
                bool(
                    np.isfinite(
                        family_winners_all[
                            "Outer_Weighted_AP"
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All outer ROC-AUC values finite",

            "PASS":
                bool(
                    np.isfinite(
                        family_winners_all[
                            "Outer_Weighted_ROC_AUC"
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Default vs tuned comparison has 100 rows",

            "PASS":
                (
                    len(
                        default_tuned_fold_comparison
                    )
                    ==
                    100
                ),
        },

        {
            "Check":
                "No family dropped after default benchmark",

            "PASS":
                (
                    selection_boundary[
                        "default_results_used_to_drop_model_family"
                    ]
                    is False
                ),
        },

        {
            "Check":
                "No family dropped after tuned outer performance",

            "PASS":
                (
                    selection_boundary[
                        "outer_test_metrics_used_to_drop_model_family"
                    ]
                    is False
                ),
        },

        {
            "Check":
                "Outer metrics do not modify tuning grid",

            "PASS":
                (
                    selection_boundary[
                        "outer_test_metrics_used_to_modify_search_space"
                    ]
                    is False
                ),
        },

        {
            "Check":
                "Step 19B does not select final model family",

            "PASS":
                (
                    selection_boundary[
                        "step19b_selects_final_model_family"
                    ]
                    is False
                ),
        },

        {
            "Check":
                "Calibration not performed in Step 19B",

            "PASS":
                True,
        },

        {
            "Check":
                "Threshold optimization not performed in Step 19B",

            "PASS":
                True,
        },

        {
            "Check":
                "Hierarchical fusion not performed in Step 19B",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "STEP 19B HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 37. SAVE COMBINED OUTPUTS
# ============================================================

CANDIDATE_METRICS_FILE = (
    DIRS["models"]
    / "STEP19B_TUNED_primary_all_candidate_inner_metrics.csv"
)

FAMILY_WINNERS_FILE = (
    DIRS["models"]
    / "STEP19B_TUNED_primary_family_winners.csv"
)

TUNED_SUMMARY_FILE = (
    DIRS["models"]
    / "STEP19B_TUNED_primary_5fold_summary.csv"
)

OUTER_PREDICTIONS_FILE = (
    DIRS["predictions"]
    / "STEP19B_TUNED_primary_family_outer_OOF_RESTRICTED.csv"
)

WINNER_INNER_OOF_FILE = (
    DIRS["calibration"]
    / "STEP19B_TUNED_primary_family_winner_inner_OOF_RESTRICTED.csv"
)

DEFAULT_TUNED_FOLD_FILE = (
    DIRS["models"]
    / "STEP19B_DEFAULT_vs_TUNED_fold_comparison.csv"
)

DEFAULT_TUNED_SUMMARY_FILE = (
    DIRS["models"]
    / "STEP19B_DEFAULT_vs_TUNED_summary.csv"
)

FAILURE_SUMMARY_FILE = (
    DIRS["qc"]
    / "STEP19B_TUNED_candidate_failure_summary.csv"
)

SELECTION_BOUNDARY_FILE = (
    DIRS["protocol"]
    / "STEP19B_TUNED_no_outer_selection_boundary.json"
)

HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP19B_TUNED_primary_hard_qc.csv"
)


candidate_metrics_all.to_csv(
    CANDIDATE_METRICS_FILE,
    index=False,
)

family_winners_all.to_csv(
    FAMILY_WINNERS_FILE,
    index=False,
)

tuned_summary.to_csv(
    TUNED_SUMMARY_FILE,
    index=False,
)

outer_predictions_all.to_csv(
    OUTER_PREDICTIONS_FILE,
    index=False,
)

winner_inner_oof_all.to_csv(
    WINNER_INNER_OOF_FILE,
    index=False,
)

default_tuned_fold_comparison.to_csv(
    DEFAULT_TUNED_FOLD_FILE,
    index=False,
)

default_tuned_summary.to_csv(
    DEFAULT_TUNED_SUMMARY_FILE,
    index=False,
)

candidate_failure_summary.to_csv(
    FAILURE_SUMMARY_FILE,
    index=False,
)

hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


with open(
    SELECTION_BOUNDARY_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        selection_boundary,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 38. FINAL MANIFEST
# ============================================================

final_manifest = {
    "step":
        "STEP 19B",

    "status":
        "ALL_MODEL_FAMILY_SPECIFIC_TUNING_COMPLETE",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "pipeline":
        PIPELINE_NAME,

    "benchmark":
        BENCHMARK_NAME,

    "model_family_count":
        10,

    "candidates_per_family":
        12,

    "outer_fold_count":
        5,

    "inner_fold_count":
        4,

    "stage_count":
        2,

    "expected_inner_base_model_fits":
        int(
            10
            *
            12
            *
            4
            *
            5
            *
            2
        ),

    "expected_outer_family_refits":
        int(
            10
            *
            5
            *
            2
        ),

    "default_results_used_to_drop_models":
        False,

    "outer_results_used_to_drop_models":
        False,

    "outer_results_used_for_tuning":
        False,

    "calibration_performed":
        False,

    "threshold_optimization_performed":
        False,

    "hierarchical_fusion_performed":
        False,

    "joint_final_family_selection_performed":
        False,

    "step19c_joint_selection_required":
        True,

    "primary_master_sha256":
        sha256_file(
            PRIMARY_FILE
        ),

    "outer_split_sha256":
        sha256_file(
            OUTER_SPLIT_FILE
        ),

    "inner_split_sha256":
        sha256_file(
            INNER_SPLIT_FILE
        ),

    "candidate_registry_sha256":
        CANDIDATE_REGISTRY_SHA256,

    "candidate_metrics_sha256":
        sha256_file(
            CANDIDATE_METRICS_FILE
        ),

    "family_winners_sha256":
        sha256_file(
            FAMILY_WINNERS_FILE
        ),

    "outer_predictions_sha256":
        sha256_file(
            OUTER_PREDICTIONS_FILE
        ),

    "winner_inner_oof_sha256":
        sha256_file(
            WINNER_INNER_OOF_FILE
        ),

    "default_tuned_summary_sha256":
        sha256_file(
            DEFAULT_TUNED_SUMMARY_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),
}


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP19B_TUNED_primary_execution_manifest.json"
)


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        final_manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 39. FINAL IMMUTABILITY CHECK
# ============================================================

if (
    sha256_file(
        PRIMARY_FILE
    )
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary master changed during Step 19B."
    )


if (
    sha256_file(
        OUTER_SPLIT_FILE
    )
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Outer split changed during Step 19B."
    )


if (
    sha256_file(
        INNER_SPLIT_FILE
    )
    !=
    EXPECTED_INNER_SHA256
):

    raise RuntimeError(
        "Inner split changed during Step 19B."
    )


# ============================================================
# 40. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 19B — ALL 10 MODEL FAMILIES TUNED"
)

print(
    "=" * 78
)


print(
    "\nCANDIDATE EXECUTION SUMMARY:"
)

display(
    candidate_failure_summary
)


print(
    "\nDEFAULT vs TUNED PERFORMANCE:"
)

display(
    default_tuned_summary
)


print(
    "\nTUNED FAMILY PERFORMANCE SUMMARY:"
)

display(
    tuned_summary
)


print(
    "\nFAMILY-SPECIFIC WINNERS BY OUTER FOLD:"
)

display(
    family_winners_all[
        [
            "Outer_Fold",
            "Stage",
            "Model",
            "Winning_Candidate",
            "Inner_Weighted_AP",
            "Inner_Weighted_ROC_AUC",
            "Outer_Weighted_AP",
            "Outer_Weighted_ROC_AUC",
            "Failed_Candidates",
        ]
    ]
    .sort_values(
        [
            "Stage",
            "Model",
            "Outer_Fold",
        ]
    )
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nIMPORTANT:"
)

print(
    "- All 10 families were tuned."
)

print(
    "- Default performance eliminated ZERO model families."
)

print(
    "- Tuned outer-test performance eliminated ZERO model families."
)

print(
    "- Each family used its frozen 12-candidate search space."
)

print(
    "- Hyperparameter selection occurred only in inner CV."
)

print(
    "- Outer folds were evaluation-only."
)

print(
    "- No calibration was performed."
)

print(
    "- No threshold was optimized."
)

print(
    "- No final joint model family was selected."
)

print(
    "- Step 19C will select the joint winner using INNER-CV results only."
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)

print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)

print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)

print(
    f"Candidate grid : {CANDIDATE_REGISTRY_SHA256}"
)


print(
    "=" * 78
)

print(
    "STATUS: STEP 19B TUNED BENCHMARK PASS"
)

print(
    "=" * 78
)


STEP 19B — FAMILY-SPECIFIC NESTED TUNING

Outer fold 0: executing tuning.

Outer 0 | Stage 1 | train N=4057 | test N=1006
  ElasticNet logistic regression: EN_005 | inner AP=0.217591 | outer AP=0.238911 | outer AUC=0.810200 | failures=0
  RBF SVM: SVM_005 | inner AP=0.139639 | outer AP=0.192642 | outer AUC=0.654235 | failures=0
  Random Forest: RF_001 | inner AP=0.190522 | outer AP=0.232622 | outer AUC=0.800984 | failures=0
  ExtraTrees: ET_001 | inner AP=0.183753 | outer AP=0.218259 | outer AUC=0.785087 | failures=0
  AdaBoost: ADA_005 | inner AP=0.205669 | outer AP=0.222750 | outer AUC=0.793298 | failures=0
  GradientBoosting: GB_006 | inner AP=0.205009 | outer AP=0.239811 | outer AUC=0.806486 | failures=0
  HistGradientBoosting: HGB_005 | inner AP=0.168613 | outer AP=0.222573 | outer AUC=0.785507 | failures=0
  XGBoost: XGB_009 | inner AP=0.186497 | outer AP=0.229769 | outer AUC=0.799163 | failures=0
  LightGBM: LGBM_005 | inner AP=0.167324 | outer AP=0.209481 | outer AUC=0.778400 

,Stage,Model,Total_Candidate_Evaluations,Successful_Candidate_Evaluations,Failed_Candidate_Evaluations
0,Stage 1,AdaBoost,60,60,0
1,Stage 1,CatBoost,60,60,0
2,Stage 1,ElasticNet logistic regression,60,60,0
3,Stage 1,ExtraTrees,60,60,0
4,Stage 1,GradientBoosting,60,60,0
5,Stage 1,HistGradientBoosting,60,60,0
6,Stage 1,LightGBM,60,60,0
7,Stage 1,RBF SVM,60,60,0
8,Stage 1,Random Forest,60,60,0
9,Stage 1,XGBoost,60,60,0



DEFAULT vs TUNED PERFORMANCE:


,Stage,Model,Mean_Default_AP,Mean_Tuned_AP,Mean_Delta_AP,Folds_AP_Improved,Mean_Default_ROC_AUC,Mean_Tuned_ROC_AUC,Mean_Delta_ROC_AUC,Folds_ROC_AUC_Improved
0,Stage 1,ElasticNet logistic regression,0.245824,0.242784,-0.003040,1,0.781347,0.781697,0.000350,2
1,Stage 1,RBF SVM,0.159142,0.168503,0.009360,5,0.638045,0.645612,0.007566,4
2,Stage 1,Random Forest,0.176568,0.223706,0.047138,5,0.722729,0.772791,0.050062,5
3,Stage 1,ExtraTrees,0.167323,0.215611,0.048288,5,0.713646,0.768445,0.054798,5
4,Stage 1,AdaBoost,0.223198,0.229394,0.006196,2,0.774285,0.776178,0.001893,2
5,Stage 1,GradientBoosting,0.211330,0.232469,0.021140,4,0.764130,0.775985,0.011855,5
6,Stage 1,HistGradientBoosting,0.182800,0.201176,0.018375,5,0.734848,0.748004,0.013157,4
7,Stage 1,XGBoost,0.158621,0.206855,0.048234,5,0.699006,0.749121,0.050115,5
8,Stage 1,LightGBM,0.176606,0.181664,0.005059,4,0.732030,0.730310,-0.001720,2
9,Stage 1,CatBoost,0.205284,0.203169,-0.002115,2,0.758317,0.761031,0.002713,3



TUNED FAMILY PERFORMANCE SUMMARY:


,Stage,Model,Outer_Folds,Mean_Tuned_Weighted_AP,SD_Tuned_Weighted_AP,Median_Tuned_Weighted_AP,Min_Tuned_Weighted_AP,Max_Tuned_Weighted_AP,Mean_Tuned_Weighted_ROC_AUC,SD_Tuned_Weighted_ROC_AUC,Median_Tuned_Weighted_ROC_AUC,Min_Tuned_Weighted_ROC_AUC,Max_Tuned_Weighted_ROC_AUC
0,Stage 1,ElasticNet logistic regression,5,0.242784,0.029226,0.243271,0.201224,0.283478,0.781697,0.026945,0.788575,0.740789,0.810200
1,Stage 1,RBF SVM,5,0.168503,0.027620,0.166810,0.125604,0.193157,0.645612,0.026939,0.635006,0.620104,0.688403
2,Stage 1,Random Forest,5,0.223706,0.011907,0.224404,0.205237,0.235440,0.772791,0.029137,0.782844,0.723940,0.800984
3,Stage 1,ExtraTrees,5,0.215611,0.016159,0.216607,0.194238,0.239011,0.768445,0.023898,0.777866,0.726325,0.785087
4,Stage 1,AdaBoost,5,0.229394,0.024616,0.229318,0.196890,0.265647,0.776178,0.027410,0.782831,0.732467,0.802858
5,Stage 1,GradientBoosting,5,0.232469,0.017725,0.239811,0.208713,0.249470,0.775985,0.028259,0.783463,0.732595,0.806486
6,Stage 1,HistGradientBoosting,5,0.201176,0.033842,0.204862,0.148840,0.237154,0.748004,0.036422,0.759944,0.695927,0.785507
7,Stage 1,XGBoost,5,0.206855,0.034581,0.226588,0.153134,0.233857,0.749121,0.046172,0.744455,0.678543,0.799163
8,Stage 1,LightGBM,5,0.181664,0.027476,0.195401,0.141869,0.209481,0.730310,0.035991,0.735411,0.678892,0.778400
9,Stage 1,CatBoost,5,0.203169,0.026055,0.209426,0.170714,0.237932,0.761031,0.033270,0.765431,0.705727,0.791499



FAMILY-SPECIFIC WINNERS BY OUTER FOLD:


,Outer_Fold,Stage,Model,Winning_Candidate,Inner_Weighted_AP,Inner_Weighted_ROC_AUC,Outer_Weighted_AP,Outer_Weighted_ROC_AUC,Failed_Candidates
4,0,Stage 1,AdaBoost,ADA_005,0.205669,0.759392,0.222750,0.793298,0
24,1,Stage 1,AdaBoost,ADA_006,0.218401,0.760328,0.196890,0.782831,0
44,2,Stage 1,AdaBoost,ADA_005,0.222292,0.762527,0.229318,0.769435,0
64,3,Stage 1,AdaBoost,ADA_005,0.196400,0.761251,0.265647,0.802858,0
84,4,Stage 1,AdaBoost,ADA_003,0.218173,0.786144,0.232366,0.732467,0
...,...,...,...,...,...,...,...,...,...
17,0,Stage 2,XGBoost,XGB_009,0.177502,0.721880,0.151624,0.719876,0
37,1,Stage 2,XGBoost,XGB_005,0.175662,0.722836,0.128864,0.703211,0
57,2,Stage 2,XGBoost,XGB_005,0.171222,0.720469,0.112634,0.700747,0
77,3,Stage 2,XGBoost,XGB_009,0.155788,0.720862,0.157792,0.704766,0



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,Exactly 10 model families tuned,True
4,All five outer folds completed,True
5,Both hierarchical stages tuned,True
6,Expected 1200 candidate summaries,True
7,Expected 100 family-specific winners,True
8,Each family has one winner per outer-fold/stage,True
9,All outer AP values finite,True



IMPORTANT:
- All 10 families were tuned.
- Default performance eliminated ZERO model families.
- Tuned outer-test performance eliminated ZERO model families.
- Each family used its frozen 12-candidate search space.
- Hyperparameter selection occurred only in inner CV.
- Outer folds were evaluation-only.
- No calibration was performed.
- No threshold was optimized.
- No final joint model family was selected.
- Step 19C will select the joint winner using INNER-CV results only.

INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Outer split    : 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
Inner split    : 2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633
Candidate grid : f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
STATUS: STEP 19B TUNED BENCHMARK PASS


In [4]:
# ============================================================
# STEP 19C — PRIMARY JOINT WINNER PIPELINE
#            CALIBRATION + THRESHOLD OPTIMIZATION
#            + HIERARCHICAL OUTER-OOF PREDICTION LOCK
# ============================================================
#
# IMPORTANT
# ---------
# This step DOES NOT evaluate final predictive performance.
#
# It only:
#
#   1. selects the Step-16 joint winner from INNER-CV results
#   2. fits the frozen weighted sigmoid calibrator
#   3. optimizes the frozen survey-weighted threshold
#   4. refits the selected stage winner on full outer training
#   5. scores untouched outer-test observations
#   6. performs hierarchical probability fusion
#   7. locks final primary outer-OOF predictions
#
# Outer-test outcomes are NOT used for:
#   - model selection
#   - hyperparameter tuning
#   - calibration fitting
#   - threshold optimization
#
# Final performance metrics will be calculated only AFTER these
# predictions are completely locked.
# ============================================================


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from pathlib import Path

import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit

from IPython.display import display


# ============================================================
# 2. REQUIRED STEP-19B HELPERS
# ============================================================
#
# These functions were created by the accepted Step-19B cell.
#
# If the kernel restarted:
# rerun Step 19B.
# Valid checkpoints will be loaded rather than refitted.
# ============================================================

required_helpers = [
    "build_representations",
    "instantiate_candidate",
    "fit_candidate",
    "model_representation",
    "positive_raw_score",
    "normalized_training_weight",
]


missing_helpers = [
    name
    for name in required_helpers
    if name not in globals()
]


if missing_helpers:

    raise RuntimeError(
        "Required Step-19B helper functions are not in memory:\n"
        f"{missing_helpers}\n\n"
        "If the kernel restarted, rerun the accepted Step-19B cell. "
        "Its valid checkpoints will be reused."
    )


# ============================================================
# 3. CONSTANTS
# ============================================================

SEED = 2026

PIPELINE_NAME = (
    "PRIMARY_SURVEY_WEIGHTED"
)

CALIBRATION_PROBABILITY_CLIP = 1e-6

THRESHOLD_PROBABILITY_CLIP = 1e-12

METRIC_TIE_TOLERANCE = 1e-12

REFIT_MATCH_TOLERANCE = 1e-10


# ============================================================
# 4. FROZEN INPUT FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


INNER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_inner_4fold_row_assignments_long.csv"
)


CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)


STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)


STEP17_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_evaluation_manifest.json"
)


STEP18_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_manifest.json"
)


STEP19B_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP19B_TUNED_primary_execution_manifest.json"
)


STEP19B_CANDIDATE_METRICS_FILE = (
    DIRS["models"]
    / "STEP19B_TUNED_primary_all_candidate_inner_metrics.csv"
)


STEP19B_FAMILY_WINNERS_FILE = (
    DIRS["models"]
    / "STEP19B_TUNED_primary_family_winners.csv"
)


STEP19B_OUTER_PREDICTIONS_FILE = (
    DIRS["predictions"]
    / "STEP19B_TUNED_primary_family_outer_OOF_RESTRICTED.csv"
)


STEP19B_WINNER_INNER_OOF_FILE = (
    DIRS["calibration"]
    / "STEP19B_TUNED_primary_family_winner_inner_OOF_RESTRICTED.csv"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_INNER_SHA256 = (
    "2b532ca3415e131e7a3555d9af76816d"
    "fa197cb825011c0d42692e58dea68633"
)


# ============================================================
# 5. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(alias)

        if key in lookup:

            return lookup[key]


    raise RuntimeError(
        "Required column could not be identified.\n"
        f"Aliases: {aliases}\n"
        f"Available: {list(dataframe.columns)}"
    )


def canonical_row_key(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(dtype=float),
            np.round(
                numeric.to_numpy(dtype=float)
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(dtype=float)
            ).astype(np.int64),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


# ============================================================
# 6. REQUIRED FILE CHECK
# ============================================================

required_files = [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    INNER_SPLIT_FILE,
    CANDIDATE_REGISTRY_FILE,
    STEP16_MANIFEST_FILE,
    STEP17_MANIFEST_FILE,
    STEP18_MANIFEST_FILE,
    STEP19B_MANIFEST_FILE,
    STEP19B_CANDIDATE_METRICS_FILE,
    STEP19B_FAMILY_WINNERS_FILE,
    STEP19B_OUTER_PREDICTIONS_FILE,
    STEP19B_WINNER_INNER_OOF_FILE,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )


# ============================================================
# 7. HASH / MANIFEST CHECK
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Outer-split hash mismatch."
    )


if sha256_file(INNER_SPLIT_FILE) != EXPECTED_INNER_SHA256:

    raise RuntimeError(
        "Inner-split hash mismatch."
    )


step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)


step17_manifest = read_json(
    STEP17_MANIFEST_FILE
)


step18_manifest = read_json(
    STEP18_MANIFEST_FILE
)


step19b_manifest = read_json(
    STEP19B_MANIFEST_FILE
)


if (
    step16_manifest.get("status")
    !=
    "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 16 is not frozen."
    )


if (
    step17_manifest.get("status")
    !=
    "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 17 is not frozen."
    )


if (
    step18_manifest.get("status")
    !=
    "THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 18 is not frozen."
    )


if (
    step19b_manifest.get("status")
    !=
    "ALL_MODEL_FAMILY_SPECIFIC_TUNING_COMPLETE"
):

    raise RuntimeError(
        "Step 19B is not complete."
    )


# ============================================================
# 8. LOAD FROZEN OUTPUTS
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)


candidate_registry = pd.read_csv(
    CANDIDATE_REGISTRY_FILE
)


candidate_metrics = pd.read_csv(
    STEP19B_CANDIDATE_METRICS_FILE
)


family_winners = pd.read_csv(
    STEP19B_FAMILY_WINNERS_FILE
)


family_outer_predictions = pd.read_csv(
    STEP19B_OUTER_PREDICTIONS_FILE
)


family_winner_inner_oof = pd.read_csv(
    STEP19B_WINNER_INNER_OOF_FILE
)


# ============================================================
# 9. ROW / FOLD IDENTIFICATION
# ============================================================

MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


master = master.copy()

outer_assignment = outer_assignment.copy()


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


family_outer_predictions[
    "_ROW_KEY"
] = canonical_row_key(
    family_outer_predictions[
        "_ROW_KEY"
    ]
)


family_winner_inner_oof[
    "_ROW_KEY"
] = canonical_row_key(
    family_winner_inner_oof[
        "_ROW_KEY"
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


outer_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__OUTER_FOLD"
] = master[
    "__ROW_KEY"
].map(
    outer_map
)


if master[
    "__OUTER_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing outer-fold assignment."
    )


master[
    "__OUTER_FOLD"
] = master[
    "__OUTER_FOLD"
].astype(int)


OUTER_FOLDS = [
    0,
    1,
    2,
    3,
    4,
]


# ============================================================
# 10. FROZEN PREDICTORS
# ============================================================

NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]


ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]


NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "division",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]


BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]


CATEGORICAL_PREDICTORS = (
    ORDERED_CATEGORICAL_PREDICTORS
    +
    NOMINAL_CATEGORICAL_PREDICTORS
    +
    BINARY_CATEGORICAL_PREDICTORS
)


PREDICTORS = (
    NUMERIC_PREDICTORS
    +
    CATEGORICAL_PREDICTORS
)


# ============================================================
# 11. EXACT STEP-16 JOINT SELECTION
# ============================================================

def select_joint_winner(
    outer_fold,
    stage_name,
):

    eligible = (
        candidate_metrics.loc[
            (
                candidate_metrics[
                    "Outer_Fold"
                ]
                ==
                outer_fold
            )
            &
            (
                candidate_metrics[
                    "Stage"
                ]
                ==
                stage_name
            )
            &
            (
                candidate_metrics[
                    "Status"
                ]
                ==
                "SUCCESS"
            )
        ]
        .copy()
    )


    if len(eligible) != 120:

        raise RuntimeError(
            f"Outer {outer_fold} {stage_name}: "
            f"expected 120 successful candidates; "
            f"observed {len(eligible)}."
        )


    best_ap = float(
        eligible[
            "Weighted_Inner_AP"
        ].max()
    )


    tied_ap = eligible.loc[
        np.isclose(
            eligible[
                "Weighted_Inner_AP"
            ],
            best_ap,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    best_auc = float(
        tied_ap[
            "Weighted_Inner_ROC_AUC"
        ].max()
    )


    tied_auc = tied_ap.loc[
        np.isclose(
            tied_ap[
                "Weighted_Inner_ROC_AUC"
            ],
            best_auc,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    winner = (
        tied_auc
        .sort_values(
            "Deterministic_Tie_Rank",
            ascending=True,
        )
        .iloc[0]
    )


    return winner


# ============================================================
# 12. VERIFY JOINT WINNER IS FAMILY WINNER
# ============================================================

def verify_family_winner(
    winner,
    outer_fold,
    stage_name,
):

    subset = family_winners.loc[
        (
            family_winners[
                "Outer_Fold"
            ]
            ==
            outer_fold
        )
        &
        (
            family_winners[
                "Stage"
            ]
            ==
            stage_name
        )
        &
        (
            family_winners[
                "Model"
            ]
            ==
            winner[
                "Model"
            ]
        )
    ]


    if len(subset) != 1:

        raise RuntimeError(
            "Could not identify corresponding family winner."
        )


    family_row = subset.iloc[0]


    if (
        str(
            family_row[
                "Winning_Candidate"
            ]
        )
        !=
        str(
            winner[
                "Candidate_ID"
            ]
        )
    ):

        raise RuntimeError(
            "Joint winner is not the corresponding "
            "family-specific winner."
        )


    return family_row


# ============================================================
# 13. CALIBRATION TRANSFORM
# ============================================================

def calibration_predictor(
    raw_score,
    model_name,
):

    raw_score = np.asarray(
        raw_score,
        dtype=float,
    )


    if model_name == "RBF SVM":

        return raw_score


    probability = np.clip(
        raw_score,
        CALIBRATION_PROBABILITY_CLIP,
        1.0
        -
        CALIBRATION_PROBABILITY_CLIP,
    )


    return np.log(
        probability
        /
        (
            1.0
            -
            probability
        )
    )


# ============================================================
# 14. FROZEN WEIGHTED SIGMOID CALIBRATOR
# ============================================================

def fit_weighted_sigmoid_calibrator(
    raw_score,
    y,
    sample_weight,
    model_name,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    weight = np.asarray(
        sample_weight,
        dtype=float,
    )


    if set(
        np.unique(
            y.astype(int)
        )
    ) != {
        0,
        1,
    }:

        raise RuntimeError(
            "Calibration sample lacks both classes."
        )


    weight = (
        weight
        /
        weight.mean()
    )


    z = calibration_predictor(
        raw_score,
        model_name,
    )


    def objective(
        theta,
    ):

        alpha = theta[0]

        beta = theta[1]


        eta = (
            alpha
            +
            beta * z
        )


        probability = expit(
            eta
        )


        loss = (
            np.sum(
                weight
                *
                (
                    np.logaddexp(
                        0.0,
                        eta,
                    )
                    -
                    y * eta
                )
            )
            /
            np.sum(
                weight
            )
        )


        residual = (
            probability
            -
            y
        )


        gradient = np.array(
            [
                (
                    np.sum(
                        weight
                        *
                        residual
                    )
                    /
                    np.sum(weight)
                ),

                (
                    np.sum(
                        weight
                        *
                        residual
                        *
                        z
                    )
                    /
                    np.sum(weight)
                ),
            ],
            dtype=float,
        )


        return (
            float(loss),
            gradient,
        )


    fit = minimize(
        objective,
        x0=np.array(
            [
                0.0,
                1.0,
            ],
            dtype=float,
        ),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter":
                5000,

            "ftol":
                1e-12,

            "gtol":
                1e-8,
        },
    )


    if not fit.success:

        raise RuntimeError(
            "Calibration optimization failed:\n"
            f"{fit.message}"
        )


    if not np.isfinite(
        fit.x
    ).all():

        raise RuntimeError(
            "Non-finite calibration coefficients."
        )


    alpha = float(
        fit.x[0]
    )


    beta = float(
        fit.x[1]
    )


    calibrated = expit(
        alpha
        +
        beta * z
    )


    return {
        "alpha":
            alpha,

        "beta":
            beta,

        "calibrated_probability":
            calibrated,

        "iterations":
            int(
                fit.nit
            ),
    }


def apply_calibrator(
    raw_score,
    model_name,
    alpha,
    beta,
):

    z = calibration_predictor(
        raw_score,
        model_name,
    )


    calibrated = expit(
        alpha
        +
        beta * z
    )


    if not np.isfinite(
        calibrated
    ).all():

        raise RuntimeError(
            "Non-finite calibrated probability."
        )


    return calibrated


# ============================================================
# 15. STEP-18 THRESHOLD OPTIMIZER
# ============================================================

def threshold_probabilities(
    probability,
):

    probability = np.asarray(
        probability,
        dtype=float,
    )


    if (
        (probability < 0)
        |
        (probability > 1)
    ).any():

        raise RuntimeError(
            "Threshold probability outside [0,1]."
        )


    return np.clip(
        probability,
        THRESHOLD_PROBABILITY_CLIP,
        1.0
        -
        THRESHOLD_PROBABILITY_CLIP,
    )


def threshold_candidates(
    probability,
):

    probability = threshold_probabilities(
        probability
    )


    unique_probability = np.unique(
        probability
    )


    if len(
        unique_probability
    ) == 1:

        return np.array(
            [
                0.0,
                1.0,
            ]
        )


    midpoint = (
        unique_probability[:-1]
        +
        (
            unique_probability[1:]
            -
            unique_probability[:-1]
        )
        /
        2.0
    )


    return np.unique(
        np.concatenate(
            [
                np.array(
                    [0.0]
                ),

                midpoint,

                np.array(
                    [1.0]
                ),
            ]
        )
    )


def threshold_metrics(
    y,
    probability,
    weight,
    threshold,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    probability = threshold_probabilities(
        probability
    )


    weight = np.asarray(
        weight,
        dtype=float,
    )


    prediction = (
        probability
        >=
        threshold
    ).astype(int)


    positive = (
        y
        ==
        1
    )


    negative = (
        y
        ==
        0
    )


    tp = weight[
        positive
        &
        (
            prediction
            ==
            1
        )
    ].sum()


    fn = weight[
        positive
        &
        (
            prediction
            ==
            0
        )
    ].sum()


    tn = weight[
        negative
        &
        (
            prediction
            ==
            0
        )
    ].sum()


    fp = weight[
        negative
        &
        (
            prediction
            ==
            1
        )
    ].sum()


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    balanced_accuracy = (
        sensitivity
        +
        specificity
    ) / 2.0


    return {
        "threshold":
            float(
                threshold
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "balance_floor":
            float(
                min(
                    sensitivity,
                    specificity,
                )
            ),
    }


def optimize_weighted_threshold(
    y_true,
    calibrated_probability,
    sample_weight,
):

    rows = []


    for threshold in threshold_candidates(
        calibrated_probability
    ):

        rows.append(
            threshold_metrics(
                y=y_true,
                probability=calibrated_probability,
                weight=sample_weight,
                threshold=threshold,
            )
        )


    results = pd.DataFrame(
        rows
    )


    best_ba = float(
        results[
            "balanced_accuracy"
        ].max()
    )


    tied = results.loc[
        np.isclose(
            results[
                "balanced_accuracy"
            ],
            best_ba,
            rtol=0.0,
            atol=1e-12,
        )
    ].copy()


    best_floor = float(
        tied[
            "balance_floor"
        ].max()
    )


    tied = tied.loc[
        np.isclose(
            tied[
                "balance_floor"
            ],
            best_floor,
            rtol=0.0,
            atol=1e-12,
        )
    ].copy()


    selected = (
        tied
        .sort_values(
            "threshold",
            ascending=True,
        )
        .iloc[0]
    )


    return {
        "selected_threshold":
            float(
                selected[
                    "threshold"
                ]
            ),

        "optimized_balanced_accuracy":
            float(
                selected[
                    "balanced_accuracy"
                ]
            ),

        "sensitivity":
            float(
                selected[
                    "sensitivity"
                ]
            ),

        "specificity":
            float(
                selected[
                    "specificity"
                ]
            ),
    }


# ============================================================
# 16. SIMPLE STAGE-TARGET CREATION
# ============================================================

def make_stage_train(
    dataframe,
    stage_name,
):

    data = dataframe.copy()


    if stage_name == "Stage 1":

        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            0
        ).astype(int)


    elif stage_name == "Stage 2":

        data = (
            data.loc[
                data[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
            ]
            .copy()
        )


        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            2
        ).astype(int)


    else:

        raise ValueError(
            f"Unknown stage: {stage_name}"
        )


    return data.reset_index(
        drop=True
    )


# ============================================================
# 17. REFIT SELECTED WINNER AND SCORE
# ============================================================

def refit_selected_winner(
    stage_train,
    scoring_data,
    candidate_id,
):

    registry_match = (
        candidate_registry.loc[
            candidate_registry[
                "Candidate_ID"
            ]
            .astype(str)
            ==
            str(
                candidate_id
            )
        ]
    )


    if len(
        registry_match
    ) != 1:

        raise RuntimeError(
            f"Could not uniquely identify candidate {candidate_id}."
        )


    candidate_row = (
        registry_match.iloc[0]
    )


    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    representations = build_representations(
        stage_train,
        scoring_data,
    )


    X_train, X_score = model_representation(
        representation_cache=representations,
        model_name=model_name,
        preprocessing_name=candidate_row[
            "Preprocessing"
        ],
    )


    estimator = instantiate_candidate(
        candidate_row
    )


    estimator = fit_candidate(
        estimator=estimator,
        model_name=model_name,
        X_train=X_train,
        y_train=stage_train[
            "__TARGET"
        ].to_numpy(
            dtype=int
        ),
        train_weight=normalized_training_weight(
            stage_train
        ),
    )


    raw_score = positive_raw_score(
        estimator=estimator,
        model_name=model_name,
        X=X_score,
    )


    return (
        raw_score,
        candidate_row,
    )


# ============================================================
# 18. EXECUTION
# ============================================================

joint_winner_rows = []

calibration_threshold_rows = []

outer_prediction_rows = []

refit_qc_rows = []


for outer_fold in OUTER_FOLDS:

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"OUTER FOLD {outer_fold}"
    )

    print(
        "-" * 78
    )


    outer_train = (
        master.loc[
            master[
                "__OUTER_FOLD"
            ]
            !=
            outer_fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    outer_test = (
        master.loc[
            master[
                "__OUTER_FOLD"
            ]
            ==
            outer_fold
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    stage_results = {}


    for stage_name in [
        "Stage 1",
        "Stage 2",
    ]:

        # ----------------------------------------------------
        # Joint winner from ALL 120 inner-CV candidates
        # ----------------------------------------------------

        winner = select_joint_winner(
            outer_fold=outer_fold,
            stage_name=stage_name,
        )


        family_row = verify_family_winner(
            winner=winner,
            outer_fold=outer_fold,
            stage_name=stage_name,
        )


        candidate_id = str(
            winner[
                "Candidate_ID"
            ]
        )


        model_name = str(
            winner[
                "Model"
            ]
        )


        print(
            f"{stage_name}: "
            f"{model_name} | {candidate_id} | "
            f"inner AP={float(winner['Weighted_Inner_AP']):.6f}"
        )


        # ----------------------------------------------------
        # Winner inner-OOF raw scores
        # ----------------------------------------------------

        inner_oof = (
            family_winner_inner_oof.loc[
                (
                    family_winner_inner_oof[
                        "Outer_Fold"
                    ]
                    ==
                    outer_fold
                )
                &
                (
                    family_winner_inner_oof[
                        "Stage"
                    ]
                    ==
                    stage_name
                )
                &
                (
                    family_winner_inner_oof[
                        "Model"
                    ]
                    ==
                    model_name
                )
                &
                (
                    family_winner_inner_oof[
                        "Winning_Candidate"
                    ]
                    .astype(str)
                    ==
                    candidate_id
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        stage_train = make_stage_train(
            outer_train,
            stage_name,
        )


        if (
            len(
                inner_oof
            )
            !=
            len(
                stage_train
            )
        ):

            raise RuntimeError(
                f"{outer_fold} {stage_name}: "
                "winner inner-OOF coverage mismatch."
            )


        if set(
            inner_oof[
                "_ROW_KEY"
            ]
        ) != set(
            stage_train[
                "__ROW_KEY"
            ]
        ):

            raise RuntimeError(
                f"{outer_fold} {stage_name}: "
                "winner inner-OOF row mismatch."
            )


        # ----------------------------------------------------
        # Weighted sigmoid calibration
        # ----------------------------------------------------

        calibration = fit_weighted_sigmoid_calibrator(
            raw_score=inner_oof[
                "Inner_OOF_Raw_Score"
            ].to_numpy(
                dtype=float
            ),
            y=inner_oof[
                "True_Target"
            ].to_numpy(
                dtype=int
            ),
            sample_weight=inner_oof[
                "SURVEY_WEIGHT"
            ].to_numpy(
                dtype=float
            ),
            model_name=model_name,
        )


        inner_calibrated = (
            calibration[
                "calibrated_probability"
            ]
        )


        # ----------------------------------------------------
        # Frozen Step-18 threshold optimization
        # ----------------------------------------------------

        threshold_result = optimize_weighted_threshold(
            y_true=inner_oof[
                "True_Target"
            ].to_numpy(
                dtype=int
            ),
            calibrated_probability=inner_calibrated,
            sample_weight=inner_oof[
                "SURVEY_WEIGHT"
            ].to_numpy(
                dtype=float
            ),
        )


        threshold = float(
            threshold_result[
                "selected_threshold"
            ]
        )


        # ----------------------------------------------------
        # Refit selected model on complete outer training
        #
        # BOTH stages score ALL outer-test observations.
        #
        # Stage 2 was trained only among true ANC users,
        # but q2 must be available for hierarchical fusion
        # among every row not classified as Stage-1 positive.
        # ----------------------------------------------------

        outer_raw_score, candidate_row = (
            refit_selected_winner(
                stage_train=stage_train,
                scoring_data=outer_test,
                candidate_id=candidate_id,
            )
        )


        outer_calibrated = apply_calibrator(
            raw_score=outer_raw_score,
            model_name=model_name,
            alpha=calibration[
                "alpha"
            ],
            beta=calibration[
                "beta"
            ],
        )


        # ----------------------------------------------------
        # Refit reproducibility QC against Step-19B predictions
        #
        # Stage 1:
        #   compare ALL outer-test rows
        #
        # Stage 2:
        #   compare TRUE ANC-user subset because Step 19B
        #   standalone Stage-2 evaluation only scored ANC users.
        # ----------------------------------------------------

        if stage_name == "Stage 1":

            comparison_mask = np.ones(
                len(
                    outer_test
                ),
                dtype=bool,
            )


        else:

            comparison_mask = (
                outer_test[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
                .to_numpy()
            )


        refit_compare = pd.DataFrame(
            {
                "_ROW_KEY":
                    outer_test.loc[
                        comparison_mask,
                        "__ROW_KEY",
                    ].to_numpy(),

                "Refit_Score":
                    outer_raw_score[
                        comparison_mask
                    ],
            }
        )


        saved_compare = (
            family_outer_predictions.loc[
                (
                    family_outer_predictions[
                        "Outer_Fold"
                    ]
                    ==
                    outer_fold
                )
                &
                (
                    family_outer_predictions[
                        "Stage"
                    ]
                    ==
                    stage_name
                )
                &
                (
                    family_outer_predictions[
                        "Model"
                    ]
                    ==
                    model_name
                )
                &
                (
                    family_outer_predictions[
                        "Winning_Candidate"
                    ]
                    .astype(str)
                    ==
                    candidate_id
                ),
                [
                    "_ROW_KEY",
                    "Raw_Discrimination_Score",
                ],
            ]
            .copy()
        )


        comparison = refit_compare.merge(
            saved_compare,
            on="_ROW_KEY",
            how="inner",
            validate="one_to_one",
        )


        if (
            len(
                comparison
            )
            !=
            comparison_mask.sum()
        ):

            raise RuntimeError(
                f"{outer_fold} {stage_name}: "
                "refit comparison coverage mismatch."
            )


        maximum_absolute_difference = float(
            np.max(
                np.abs(
                    comparison[
                        "Refit_Score"
                    ].to_numpy(
                        dtype=float
                    )
                    -
                    comparison[
                        "Raw_Discrimination_Score"
                    ].to_numpy(
                        dtype=float
                    )
                )
            )
        )


        if (
            maximum_absolute_difference
            >
            REFIT_MATCH_TOLERANCE
        ):

            raise RuntimeError(
                f"{outer_fold} {stage_name}: "
                "selected-winner refit does not reproduce "
                "Step-19B saved predictions.\n"
                f"Max absolute difference = "
                f"{maximum_absolute_difference}"
            )


        refit_qc_rows.append(
            {
                "Outer_Fold":
                    outer_fold,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate":
                    candidate_id,

                "Rows_Compared":
                    int(
                        len(
                            comparison
                        )
                    ),

                "Max_Absolute_Difference":
                    maximum_absolute_difference,

                "PASS":
                    (
                        maximum_absolute_difference
                        <=
                        REFIT_MATCH_TOLERANCE
                    ),
            }
        )


        joint_winner_rows.append(
            {
                "Outer_Fold":
                    outer_fold,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Inner_Weighted_AP":
                    float(
                        winner[
                            "Weighted_Inner_AP"
                        ]
                    ),

                "Inner_Weighted_ROC_AUC":
                    float(
                        winner[
                            "Weighted_Inner_ROC_AUC"
                        ]
                    ),

                "Joint_Winner_Verified_As_Family_Winner":
                    True,
            }
        )


        calibration_threshold_rows.append(
            {
                "Outer_Fold":
                    outer_fold,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Calibration_Alpha":
                    float(
                        calibration[
                            "alpha"
                        ]
                    ),

                "Calibration_Beta":
                    float(
                        calibration[
                            "beta"
                        ]
                    ),

                "Calibration_Iterations":
                    int(
                        calibration[
                            "iterations"
                        ]
                    ),

                "Optimized_Threshold":
                    threshold,

                "Inner_Threshold_Balanced_Accuracy":
                    float(
                        threshold_result[
                            "optimized_balanced_accuracy"
                        ]
                    ),

                "Inner_Threshold_Sensitivity":
                    float(
                        threshold_result[
                            "sensitivity"
                        ]
                    ),

                "Inner_Threshold_Specificity":
                    float(
                        threshold_result[
                            "specificity"
                        ]
                    ),
            }
        )


        stage_results[
            stage_name
        ] = {
            "model":
                model_name,

            "candidate":
                candidate_id,

            "raw":
                outer_raw_score,

            "calibrated":
                outer_calibrated,

            "threshold":
                threshold,
        }


    # ========================================================
    # 19. HIERARCHICAL FUSION
    # ========================================================

    q0 = np.asarray(
        stage_results[
            "Stage 1"
        ][
            "calibrated"
        ],
        dtype=float,
    )


    q2 = np.asarray(
        stage_results[
            "Stage 2"
        ][
            "calibrated"
        ],
        dtype=float,
    )


    t1 = float(
        stage_results[
            "Stage 1"
        ][
            "threshold"
        ]
    )


    t2 = float(
        stage_results[
            "Stage 2"
        ][
            "threshold"
        ]
    )


    p0 = q0


    p1 = (
        (1.0 - q0)
        *
        (1.0 - q2)
    )


    p2 = (
        (1.0 - q0)
        *
        q2
    )


    probability_matrix = np.column_stack(
        [
            p0,
            p1,
            p2,
        ]
    )


    if not np.isfinite(
        probability_matrix
    ).all():

        raise RuntimeError(
            f"Outer {outer_fold}: non-finite fused probability."
        )


    if not np.allclose(
        probability_matrix.sum(
            axis=1
        ),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"Outer {outer_fold}: probabilities do not sum to 1."
        )


    predicted_class = np.where(
        q0 >= t1,
        0,
        np.where(
            q2 >= t2,
            2,
            1,
        ),
    ).astype(int)


    outer_prediction_rows.append(
        pd.DataFrame(
            {
                "_ROW_KEY":
                    outer_test[
                        "__ROW_KEY"
                    ].to_numpy(),

                "Outer_Fold":
                    outer_fold,

                "ANC_CLASS_TRUE":
                    outer_test[
                        "ANC_CLASS"
                    ].to_numpy(
                        dtype=int
                    ),

                "SURVEY_WEIGHT":
                    outer_test[
                        "SURVEY_WEIGHT"
                    ].to_numpy(
                        dtype=float
                    ),

                "PSU":
                    outer_test[
                        "PSU"
                    ].to_numpy(),

                "STRATUM":
                    outer_test[
                        "STRATUM"
                    ].to_numpy(),

                "Stage1_Model":
                    stage_results[
                        "Stage 1"
                    ][
                        "model"
                    ],

                "Stage1_Candidate":
                    stage_results[
                        "Stage 1"
                    ][
                        "candidate"
                    ],

                "Stage1_Raw_Score":
                    stage_results[
                        "Stage 1"
                    ][
                        "raw"
                    ],

                "q0_CALIBRATED":
                    q0,

                "Stage1_Optimized_Threshold":
                    t1,

                "Stage2_Model":
                    stage_results[
                        "Stage 2"
                    ][
                        "model"
                    ],

                "Stage2_Candidate":
                    stage_results[
                        "Stage 2"
                    ][
                        "candidate"
                    ],

                "Stage2_Raw_Score":
                    stage_results[
                        "Stage 2"
                    ][
                        "raw"
                    ],

                "q2_CALIBRATED_CONDITIONAL":
                    q2,

                "Stage2_Optimized_Threshold":
                    t2,

                "P_NO_ANC":
                    p0,

                "P_ANC_1_7":
                    p1,

                "P_ANC_GE8":
                    p2,

                "P_SUM":
                    probability_matrix.sum(
                        axis=1
                    ),

                "PREDICTED_ANC_CLASS":
                    predicted_class,
            }
        )
    )


# ============================================================
# 20. COMBINE LOCKED OUTER-OOF PREDICTIONS
# ============================================================

joint_winners = pd.DataFrame(
    joint_winner_rows
)


calibration_thresholds = pd.DataFrame(
    calibration_threshold_rows
)


refit_qc = pd.DataFrame(
    refit_qc_rows
)


primary_outer_oof = pd.concat(
    outer_prediction_rows,
    axis=0,
    ignore_index=True,
)


# ============================================================
# 21. FINAL OOF COVERAGE QC
# ============================================================

if len(
    primary_outer_oof
) != 5063:

    raise RuntimeError(
        "Final primary OOF prediction count is not 5063."
    )


if primary_outer_oof[
    "_ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate row in final primary outer-OOF predictions."
    )


if (
    primary_outer_oof[
        "_ROW_KEY"
    ].nunique()
    !=
    5063
):

    raise RuntimeError(
        "Final primary OOF rows are not unique."
    )


if set(
    primary_outer_oof[
        "_ROW_KEY"
    ]
) != set(
    master[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Final primary OOF predictions do not exactly "
        "cover the modelling population."
    )


if not np.isfinite(
    primary_outer_oof[
        [
            "q0_CALIBRATED",
            "q2_CALIBRATED_CONDITIONAL",
            "P_NO_ANC",
            "P_ANC_1_7",
            "P_ANC_GE8",
            "P_SUM",
        ]
    ]
    .to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Non-finite final probability detected."
    )


if not np.allclose(
    primary_outer_oof[
        "P_SUM"
    ].to_numpy(
        dtype=float
    ),
    1.0,
    rtol=0.0,
    atol=1e-12,
):

    raise RuntimeError(
        "Final multiclass probabilities do not sum to one."
    )


# ============================================================
# 22. EXPECTED JOINT-WINNER COUNTS
# ============================================================

winner_count_table = (
    joint_winners
    .groupby(
        [
            "Stage",
            "Model",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "Outer_Fold_Wins"
        }
    )
)


# ============================================================
# 23. FINAL HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                (
                    sha256_file(
                        PRIMARY_FILE
                    )
                    ==
                    EXPECTED_PRIMARY_SHA256
                ),
        },

        {
            "Check":
                "Outer split hash unchanged",

            "PASS":
                (
                    sha256_file(
                        OUTER_SPLIT_FILE
                    )
                    ==
                    EXPECTED_OUTER_SHA256
                ),
        },

        {
            "Check":
                "Inner split hash unchanged",

            "PASS":
                (
                    sha256_file(
                        INNER_SPLIT_FILE
                    )
                    ==
                    EXPECTED_INNER_SHA256
                ),
        },

        {
            "Check":
                "10 joint winners selected",

            "PASS":
                (
                    len(
                        joint_winners
                    )
                    ==
                    10
                ),
        },

        {
            "Check":
                "One joint winner per outer-fold/stage",

            "PASS":
                bool(
                    (
                        joint_winners
                        .groupby(
                            [
                                "Outer_Fold",
                                "Stage",
                            ]
                        )
                        .size()
                        ==
                        1
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Every joint winner verified as family winner",

            "PASS":
                bool(
                    joint_winners[
                        "Joint_Winner_Verified_As_Family_Winner"
                    ].all()
                ),
        },

        {
            "Check":
                "All selected-winner refits reproduce Step-19B scores",

            "PASS":
                bool(
                    refit_qc[
                        "PASS"
                    ].all()
                ),
        },

        {
            "Check":
                "Calibration fitted for all 10 stage/fold winners",

            "PASS":
                (
                    len(
                        calibration_thresholds
                    )
                    ==
                    10
                ),
        },

        {
            "Check":
                "All optimized thresholds within [0,1]",

            "PASS":
                bool(
                    (
                        calibration_thresholds[
                            "Optimized_Threshold"
                        ]
                        .between(
                            0.0,
                            1.0,
                            inclusive="both",
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Exactly 5063 final outer-OOF rows",

            "PASS":
                (
                    len(
                        primary_outer_oof
                    )
                    ==
                    5063
                ),
        },

        {
            "Check":
                "Final outer-OOF rows unique",

            "PASS":
                not primary_outer_oof[
                    "_ROW_KEY"
                ]
                .duplicated()
                .any(),
        },

        {
            "Check":
                "Final outer-OOF coverage exact",

            "PASS":
                (
                    set(
                        primary_outer_oof[
                            "_ROW_KEY"
                        ]
                    )
                    ==
                    set(
                        master[
                            "__ROW_KEY"
                        ]
                    )
                ),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                bool(
                    np.isfinite(
                        primary_outer_oof[
                            [
                                "P_NO_ANC",
                                "P_ANC_1_7",
                                "P_ANC_GE8",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                bool(
                    np.allclose(
                        primary_outer_oof[
                            "P_SUM"
                        ],
                        1.0,
                        rtol=0.0,
                        atol=1e-12,
                    )
                ),
        },

        {
            "Check":
                "No outer outcome used for calibration fitting",

            "PASS":
                True,
        },

        {
            "Check":
                "No outer outcome used for threshold optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "No final performance metric calculated",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "STEP 19C HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 24. SAVE LOCKED OUTPUTS
# ============================================================

JOINT_WINNER_FILE = (
    DIRS["models"]
    / "STEP19C_PRIMARY_joint_winners_by_outer_stage.csv"
)


CALIBRATION_THRESHOLD_FILE = (
    DIRS["calibration"]
    / "STEP19C_PRIMARY_calibration_thresholds_by_outer_stage.csv"
)


REFIT_QC_FILE = (
    DIRS["qc"]
    / "STEP19C_PRIMARY_selected_winner_refit_qc.csv"
)


FINAL_OOF_FILE = (
    DIRS["predictions"]
    / "STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP19C_PRIMARY_hard_qc.csv"
)


joint_winners.to_csv(
    JOINT_WINNER_FILE,
    index=False,
)


calibration_thresholds.to_csv(
    CALIBRATION_THRESHOLD_FILE,
    index=False,
)


refit_qc.to_csv(
    REFIT_QC_FILE,
    index=False,
)


primary_outer_oof.to_csv(
    FINAL_OOF_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


# ============================================================
# 25. MANIFEST
# ============================================================

manifest = {
    "step":
        "STEP 19C",

    "status":
        (
            "PRIMARY_JOINT_SELECTED_CALIBRATED_"
            "THRESHOLD_OPTIMIZED_OOF_LOCKED"
        ),

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "pipeline":
        PIPELINE_NAME,

    "outer_fold_count":
        5,

    "stage_count":
        2,

    "joint_winner_count":
        10,

    "calibration_method":
        (
            "survey-weighted unregularized logistic sigmoid"
        ),

    "threshold_method":
        (
            "survey-weighted balanced-accuracy optimization"
        ),

    "threshold_values_fixed":
        False,

    "stage2_scored_on_all_outer_test_rows_for_fusion":
        True,

    "outer_test_used_for_model_selection":
        False,

    "outer_test_used_for_calibration":
        False,

    "outer_test_used_for_threshold_optimization":
        False,

    "final_performance_evaluated":
        False,

    "primary_master_sha256":
        sha256_file(
            PRIMARY_FILE
        ),

    "outer_split_sha256":
        sha256_file(
            OUTER_SPLIT_FILE
        ),

    "inner_split_sha256":
        sha256_file(
            INNER_SPLIT_FILE
        ),

    "joint_winner_sha256":
        sha256_file(
            JOINT_WINNER_FILE
        ),

    "calibration_threshold_sha256":
        sha256_file(
            CALIBRATION_THRESHOLD_FILE
        ),

    "final_outer_oof_sha256":
        sha256_file(
            FINAL_OOF_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP19C_PRIMARY_execution_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 26. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 19C — PRIMARY JOINT PIPELINE LOCKED"
)


print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)


display(
    joint_winners
    .sort_values(
        [
            "Stage",
            "Outer_Fold",
        ]
    )
)


print(
    "\nJOINT WINNER FREQUENCY:"
)


display(
    winner_count_table
)


print(
    "\nCALIBRATION + OPTIMIZED THRESHOLDS:"
)


display(
    calibration_thresholds
    .sort_values(
        [
            "Stage",
            "Outer_Fold",
        ]
    )
)


print(
    "\nSELECTED-WINNER REFIT REPRODUCIBILITY QC:"
)


display(
    refit_qc
)


print(
    "\nFINAL OUTER-OOF PREDICTION LOCK:"
)


print(
    f"Rows               : {len(primary_outer_oof)}"
)


print(
    f"Unique rows        : {primary_outer_oof['_ROW_KEY'].nunique()}"
)


print(
    "Probability sum QC : "
    f"{np.allclose(primary_outer_oof['P_SUM'], 1.0, atol=1e-12, rtol=0)}"
)


print(
    "Final performance  : NOT YET EVALUATED"
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)


print(
    f"Outer split    : {sha256_file(OUTER_SPLIT_FILE)}"
)


print(
    f"Inner split    : {sha256_file(INNER_SPLIT_FILE)}"
)


print(
    f"Locked OOF     : {sha256_file(FINAL_OOF_FILE)}"
)


print(
    "=" * 78
)


print(
    "STATUS: STEP 19C PRIMARY OOF PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


------------------------------------------------------------------------------
OUTER FOLD 0
------------------------------------------------------------------------------
Stage 1: ElasticNet logistic regression | EN_005 | inner AP=0.217591
Stage 2: GradientBoosting | GB_002 | inner AP=0.208294

------------------------------------------------------------------------------
OUTER FOLD 1
------------------------------------------------------------------------------
Stage 1: ElasticNet logistic regression | EN_004 | inner AP=0.238925
Stage 2: GradientBoosting | GB_003 | inner AP=0.214570

------------------------------------------------------------------------------
OUTER FOLD 2
------------------------------------------------------------------------------
Stage 1: ElasticNet logistic regression | EN_009 | inner AP=0.239664
Stage 2: AdaBoost | ADA_008 | inner AP=0.203855

------------------------------------------------------------------------------
OUTER FOLD 3
--------------------------

,Outer_Fold,Stage,Model,Candidate_ID,Preprocessing,Inner_Weighted_AP,Inner_Weighted_ROC_AUC,Joint_Winner_Verified_As_Family_Winner
0,0,Stage 1,ElasticNet logistic regression,EN_005,Scaled OHE,0.217591,0.766155,True
2,1,Stage 1,ElasticNet logistic regression,EN_004,Scaled OHE,0.238925,0.774269,True
4,2,Stage 1,ElasticNet logistic regression,EN_009,Scaled OHE,0.239664,0.780764,True
6,3,Stage 1,ElasticNet logistic regression,EN_010,Scaled OHE,0.219436,0.764884,True
8,4,Stage 1,ElasticNet logistic regression,EN_007,Scaled OHE,0.237613,0.787682,True
1,0,Stage 2,GradientBoosting,GB_002,Unscaled OHE,0.208294,0.735769,True
3,1,Stage 2,GradientBoosting,GB_003,Unscaled OHE,0.214570,0.734828,True
5,2,Stage 2,AdaBoost,ADA_008,Unscaled OHE,0.203855,0.734869,True
7,3,Stage 2,AdaBoost,ADA_003,Unscaled OHE,0.224952,0.746262,True
9,4,Stage 2,AdaBoost,ADA_007,Unscaled OHE,0.197436,0.750574,True



JOINT WINNER FREQUENCY:


,Stage,Model,Outer_Fold_Wins
0,Stage 1,ElasticNet logistic regression,5
1,Stage 2,AdaBoost,3
2,Stage 2,GradientBoosting,2



CALIBRATION + OPTIMIZED THRESHOLDS:


,Outer_Fold,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Inner_Threshold_Balanced_Accuracy,Inner_Threshold_Sensitivity,Inner_Threshold_Specificity
0,0,Stage 1,ElasticNet logistic regression,EN_005,0.097085,1.049504,11,0.088693,0.710173,0.678233,0.742114
2,1,Stage 1,ElasticNet logistic regression,EN_004,-0.030158,0.991219,10,0.075382,0.719548,0.770109,0.668986
4,2,Stage 1,ElasticNet logistic regression,EN_009,-0.204428,0.898139,6,0.065311,0.710500,0.823249,0.597750
6,3,Stage 1,ElasticNet logistic regression,EN_010,-0.373632,0.818994,10,0.067776,0.702173,0.789129,0.615218
8,4,Stage 1,ElasticNet logistic regression,EN_007,-0.284925,0.868880,6,0.069947,0.738155,0.842702,0.633608
1,0,Stage 2,GradientBoosting,GB_002,0.179947,1.066115,10,0.037448,0.681383,0.783876,0.578889
3,1,Stage 2,GradientBoosting,GB_003,-0.419003,0.832907,10,0.068807,0.683155,0.566015,0.800296
5,2,Stage 2,AdaBoost,ADA_008,-0.094425,2.434324,10,0.055573,0.704520,0.695599,0.713440
7,3,Stage 2,AdaBoost,ADA_003,-0.352196,4.061856,13,0.050849,0.680223,0.681911,0.678534
9,4,Stage 2,AdaBoost,ADA_007,0.771235,2.416000,12,0.052830,0.697823,0.645812,0.749835



SELECTED-WINNER REFIT REPRODUCIBILITY QC:


,Outer_Fold,Stage,Model,Candidate,Rows_Compared,Max_Absolute_Difference,PASS
0,0,Stage 1,ElasticNet logistic regression,EN_005,1006,9.887924e-17,True
1,0,Stage 2,GradientBoosting,GB_002,924,9.887924e-17,True
2,1,Stage 1,ElasticNet logistic regression,EN_004,1013,9.974660e-17,True
3,1,Stage 2,GradientBoosting,GB_003,936,9.714451e-17,True
4,2,Stage 1,ElasticNet logistic regression,EN_009,1005,9.887924e-17,True
5,2,Stage 2,AdaBoost,ADA_008,929,8.326673e-17,True
6,3,Stage 1,ElasticNet logistic regression,EN_010,1008,9.887924e-17,True
7,3,Stage 2,AdaBoost,ADA_003,923,8.326673e-17,True
8,4,Stage 1,ElasticNet logistic regression,EN_007,1031,9.974660e-17,True
9,4,Stage 2,AdaBoost,ADA_007,948,8.326673e-17,True



FINAL OUTER-OOF PREDICTION LOCK:
Rows               : 5063
Unique rows        : 5063
Probability sum QC : True
Final performance  : NOT YET EVALUATED

FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Outer split hash unchanged,True
2,Inner split hash unchanged,True
3,10 joint winners selected,True
4,One joint winner per outer-fold/stage,True
5,Every joint winner verified as family winner,True
6,All selected-winner refits reproduce Step-19B ...,True
7,Calibration fitted for all 10 stage/fold winners,True
8,"All optimized thresholds within [0,1]",True
9,Exactly 5063 final outer-OOF rows,True



INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Outer split    : 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
Inner split    : 2b532ca3415e131e7a3555d9af76816dfa197cb825011c0d42692e58dea68633
Locked OOF     : 6c4f4e21b09a54c286521682e9fb8ee13eda36ed94196d350fbb0622733a99bc
STATUS: STEP 19C PRIMARY OOF PREDICTIONS LOCKED


In [5]:
# ============================================================
# STEP 20 — FINAL PRIMARY PERFORMANCE EVALUATION
#           LOCKED OUTER-OOF PREDICTIONS
# ============================================================
#
# INPUT
# -----
# STEP19C locked primary outer-OOF predictions.
#
# NO:
#   model fitting
#   hyperparameter tuning
#   recalibration
#   threshold re-optimization
#   model re-selection
#
#
# EVALUATION POPULATIONS
# ----------------------
# Stage 1:
#   all N = 5063
#   positive = No ANC
#
# Stage 2:
#   true ANC users only N = 4660
#   positive = >=8 ANC visits
#
# Multiclass:
#   all N = 5063
#
#
# PRIMARY PROBABILITY METRICS
# ---------------------------
# Binary:
#   - weighted Average Precision
#   - weighted ROC-AUC
#   - weighted log loss
#   - weighted Brier score
#   - calibration-in-the-large
#   - calibration slope
#   - O/E ratio
#
# Multiclass:
#   - weighted multiclass log loss
#   - weighted multiclass Brier score
#   - classwise OVR AP / ROC-AUC
#   - macro AP / ROC-AUC
#   - classwise calibration
#
#
# CALIBRATION CURVE
# -----------------
# Survey-weighted restricted cubic spline on predicted log-odds.
#
# Prespecified knots:
#   weighted 5th, 35th, 65th, 95th percentiles.
#
# Grouped weighted calibration points:
#   supplementary only.
#
#
# THRESHOLD METRICS
# -----------------
# Fold-specific optimized thresholds from Step 19C are applied
# exactly as locked.
#
# No threshold is re-estimated here.
#
#
# UNCERTAINTY
# -----------
# Point estimates only.
# Confidence intervals will be obtained later using the frozen
# stratified PSU bootstrap procedure.
# ============================================================


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from pathlib import Path

import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.optimize import (
    brentq,
    minimize,
)

from scipy.special import (
    expit,
    logit,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from IPython.display import display


# ============================================================
# 2. FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


FINAL_OOF_FILE = (
    DIRS["predictions"]
    / "STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
)


STEP19C_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP19C_PRIMARY_execution_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_LOCKED_OOF_SHA256 = (
    "6c4f4e21b09a54c286521682e9fb8ee1"
    "3eda36ed94196d350fbb0622733a99bc"
)


PROBABILITY_CLIP = 1e-6


# ============================================================
# 3. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


def weighted_mean(
    values,
    weights,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    return float(
        np.sum(
            weights * values
        )
        /
        np.sum(weights)
    )


def weighted_quantile(
    values,
    weights,
    probabilities,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )


    order = np.argsort(
        values
    )


    x = values[
        order
    ]

    w = weights[
        order
    ]


    cumulative = (
        np.cumsum(w)
        -
        0.5 * w
    )


    cumulative = (
        cumulative
        /
        np.sum(w)
    )


    return np.interp(
        probabilities,
        cumulative,
        x,
        left=x[0],
        right=x[-1],
    )


# ============================================================
# 4. INPUT CHECKS
# ============================================================

for path in [
    PRIMARY_FILE,
    FINAL_OOF_FILE,
    STEP19C_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )


if (
    sha256_file(PRIMARY_FILE)
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary master hash mismatch."
    )


if (
    sha256_file(FINAL_OOF_FILE)
    !=
    EXPECTED_LOCKED_OOF_SHA256
):

    raise RuntimeError(
        "Locked Step-19C OOF hash mismatch."
    )


step19c_manifest = read_json(
    STEP19C_MANIFEST_FILE
)


if (
    step19c_manifest.get("status")
    !=
    "PRIMARY_JOINT_SELECTED_CALIBRATED_THRESHOLD_OPTIMIZED_OOF_LOCKED"
):

    raise RuntimeError(
        "Step 19C is not in the expected locked state."
    )


if (
    step19c_manifest.get(
        "final_performance_evaluated"
    )
    is not False
):

    raise RuntimeError(
        "Unexpected Step-19C performance-evaluation state."
    )


# ============================================================
# 5. LOAD LOCKED OOF PREDICTIONS
# ============================================================

pred = pd.read_csv(
    FINAL_OOF_FILE
)


if len(pred) != 5063:

    raise RuntimeError(
        f"Expected 5063 OOF rows; observed {len(pred)}."
    )


required_columns = [
    "_ROW_KEY",
    "Outer_Fold",
    "ANC_CLASS_TRUE",
    "SURVEY_WEIGHT",
    "PSU",
    "STRATUM",
    "Stage1_Raw_Score",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "Stage2_Raw_Score",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
    "P_NO_ANC",
    "P_ANC_1_7",
    "P_ANC_GE8",
    "P_SUM",
    "PREDICTED_ANC_CLASS",
]


missing_columns = [
    column
    for column in required_columns
    if column not in pred.columns
]


if missing_columns:

    raise RuntimeError(
        f"Locked OOF file missing columns:\n{missing_columns}"
    )


if pred[
    "_ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate locked OOF row."
    )


if set(
    pred[
        "Outer_Fold"
    ].unique()
) != {
    0,
    1,
    2,
    3,
    4,
}:

    raise RuntimeError(
        "Unexpected outer-fold coverage."
    )


if set(
    pred[
        "ANC_CLASS_TRUE"
    ].unique()
) != {
    0,
    1,
    2,
}:

    raise RuntimeError(
        "Unexpected ANC class values."
    )


weight = pred[
    "SURVEY_WEIGHT"
].to_numpy(
    dtype=float
)


if (
    not np.isfinite(
        weight
    ).all()
    or
    (weight <= 0).any()
):

    raise RuntimeError(
        "Invalid survey weights."
    )


probability_columns = [
    "q0_CALIBRATED",
    "q2_CALIBRATED_CONDITIONAL",
    "P_NO_ANC",
    "P_ANC_1_7",
    "P_ANC_GE8",
]


for column in probability_columns:

    values = pred[
        column
    ].to_numpy(
        dtype=float
    )


    if not np.isfinite(
        values
    ).all():

        raise RuntimeError(
            f"Non-finite probability in {column}."
        )


    if (
        (values < 0)
        |
        (values > 1)
    ).any():

        raise RuntimeError(
            f"Probability outside [0,1] in {column}."
        )


if not np.allclose(
    pred[
        "P_SUM"
    ].to_numpy(
        dtype=float
    ),
    1.0,
    rtol=0.0,
    atol=1e-12,
):

    raise RuntimeError(
        "Multiclass probabilities do not sum to one."
    )


# ============================================================
# 6. CALIBRATION-IN-THE-LARGE
# ============================================================

def calibration_in_the_large(
    y,
    probability,
    sample_weight,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    w = np.asarray(
        sample_weight,
        dtype=float,
    )


    lp = logit(
        p
    )


    def score(
        alpha,
    ):

        fitted = expit(
            alpha
            +
            lp
        )


        return float(
            np.sum(
                w
                *
                (
                    y
                    -
                    fitted
                )
            )
        )


    return float(
        brentq(
            score,
            -50.0,
            50.0,
        )
    )


# ============================================================
# 7. CALIBRATION SLOPE
# ============================================================

def calibration_slope_model(
    y,
    probability,
    sample_weight,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    w = np.asarray(
        sample_weight,
        dtype=float,
    )


    w = (
        w
        /
        np.mean(w)
    )


    lp = logit(
        p
    )


    def objective(
        theta,
    ):

        alpha = theta[0]

        beta = theta[1]


        eta = (
            alpha
            +
            beta * lp
        )


        loss = (
            np.sum(
                w
                *
                (
                    np.logaddexp(
                        0.0,
                        eta,
                    )
                    -
                    y * eta
                )
            )
            /
            np.sum(w)
        )


        residual = (
            expit(eta)
            -
            y
        )


        gradient = np.array(
            [
                np.sum(
                    w * residual
                )
                /
                np.sum(w),

                np.sum(
                    w
                    *
                    residual
                    *
                    lp
                )
                /
                np.sum(w),
            ],
            dtype=float,
        )


        return (
            float(loss),
            gradient,
        )


    fit = minimize(
        objective,
        x0=np.array(
            [
                0.0,
                1.0,
            ],
            dtype=float,
        ),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter":
                5000,

            "ftol":
                1e-12,

            "gtol":
                1e-8,
        },
    )


    if not fit.success:

        raise RuntimeError(
            "Calibration slope model failed:\n"
            f"{fit.message}"
        )


    return {
        "recalibration_intercept":
            float(
                fit.x[0]
            ),

        "calibration_slope":
            float(
                fit.x[1]
            ),
    }


# ============================================================
# 8. RESTRICTED CUBIC SPLINE BASIS
# ============================================================

def restricted_cubic_spline_basis(
    x,
    knots,
):

    x = np.asarray(
        x,
        dtype=float,
    )


    knots = np.asarray(
        knots,
        dtype=float,
    )


    if len(knots) != 4:

        raise ValueError(
            "Exactly four spline knots are required."
        )


    if not np.all(
        np.diff(knots)
        >
        0
    ):

        raise RuntimeError(
            "Spline knots are not strictly increasing."
        )


    def cube_positive(
        value,
    ):

        return (
            np.maximum(
                value,
                0.0,
            )
            **
            3
        )


    last = knots[-1]

    penultimate = knots[-2]


    denominator = (
        last
        -
        penultimate
    )


    scale = (
        last
        -
        knots[0]
    ) ** 2


    nonlinear_columns = []


    for knot in knots[:-2]:

        term = (
            cube_positive(
                x
                -
                knot
            )

            -

            cube_positive(
                x
                -
                penultimate
            )
            *
            (
                last
                -
                knot
            )
            /
            denominator

            +

            cube_positive(
                x
                -
                last
            )
            *
            (
                penultimate
                -
                knot
            )
            /
            denominator
        )


        nonlinear_columns.append(
            term
            /
            scale
        )


    return np.column_stack(
        [
            x,
            *nonlinear_columns,
        ]
    )


# ============================================================
# 9. FLEXIBLE WEIGHTED CALIBRATION CURVE
# ============================================================

def flexible_calibration_curve(
    y,
    probability,
    sample_weight,
    analysis_name,
    grid_points=200,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    w = np.asarray(
        sample_weight,
        dtype=float,
    )


    lp = logit(
        p
    )


    knots = weighted_quantile(
        values=lp,
        weights=w,
        probabilities=[
            0.05,
            0.35,
            0.65,
            0.95,
        ],
    )


    if not np.all(
        np.diff(knots)
        >
        0
    ):

        raise RuntimeError(
            f"{analysis_name}: spline knots not unique."
        )


    basis = restricted_cubic_spline_basis(
        lp,
        knots,
    )


    # weighted standardization for numerical stability

    basis_mean = np.array(
        [
            weighted_mean(
                basis[:, column],
                w,
            )
            for column in range(
                basis.shape[1]
            )
        ]
    )


    basis_sd = np.array(
        [
            np.sqrt(
                weighted_mean(
                    (
                        basis[:, column]
                        -
                        basis_mean[column]
                    ) ** 2,
                    w,
                )
            )
            for column in range(
                basis.shape[1]
            )
        ]
    )


    if (
        basis_sd
        <=
        1e-12
    ).any():

        raise RuntimeError(
            f"{analysis_name}: degenerate spline basis."
        )


    z = (
        basis
        -
        basis_mean
    ) / basis_sd


    design = np.column_stack(
        [
            np.ones(
                len(z)
            ),
            z,
        ]
    )


    normalized_weight = (
        w
        /
        np.mean(w)
    )


    def objective(
        beta,
    ):

        eta = (
            design
            @
            beta
        )


        fitted = expit(
            eta
        )


        loss = (
            np.sum(
                normalized_weight
                *
                (
                    np.logaddexp(
                        0.0,
                        eta,
                    )
                    -
                    y * eta
                )
            )
            /
            np.sum(
                normalized_weight
            )
        )


        gradient = (
            design.T
            @
            (
                normalized_weight
                *
                (
                    fitted
                    -
                    y
                )
            )
            /
            np.sum(
                normalized_weight
            )
        )


        return (
            float(loss),
            gradient,
        )


    initial_probability = (
        weighted_mean(
            y,
            w,
        )
    )


    initial_probability = np.clip(
        initial_probability,
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    initial_beta = np.zeros(
        design.shape[1],
        dtype=float,
    )


    initial_beta[0] = logit(
        initial_probability
    )


    fit = minimize(
        objective,
        x0=initial_beta,
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter":
                10000,

            "ftol":
                1e-12,

            "gtol":
                1e-8,
        },
    )


    if not fit.success:

        raise RuntimeError(
            f"{analysis_name}: spline calibration failed:\n"
            f"{fit.message}"
        )


    lp_grid = np.linspace(
        float(
            np.min(lp)
        ),
        float(
            np.max(lp)
        ),
        grid_points,
    )


    grid_basis = (
        restricted_cubic_spline_basis(
            lp_grid,
            knots,
        )
    )


    grid_z = (
        grid_basis
        -
        basis_mean
    ) / basis_sd


    grid_design = np.column_stack(
        [
            np.ones(
                grid_points
            ),
            grid_z,
        ]
    )


    observed_curve = expit(
        grid_design
        @
        fit.x
    )


    result = pd.DataFrame(
        {
            "Analysis":
                analysis_name,

            "Predicted_Probability":
                expit(
                    lp_grid
                ),

            "Smoothed_Observed_Probability":
                observed_curve,
        }
    )


    knot_table = pd.DataFrame(
        {
            "Analysis":
                analysis_name,

            "Knot_Number":
                [
                    1,
                    2,
                    3,
                    4,
                ],

            "Weighted_Quantile":
                [
                    0.05,
                    0.35,
                    0.65,
                    0.95,
                ],

            "Knot_Logit":
                knots,

            "Knot_Probability":
                expit(
                    knots
                ),
        }
    )


    return (
        result,
        knot_table,
    )


# ============================================================
# 10. GROUPED CALIBRATION POINTS — SUPPLEMENTARY
# ============================================================

def grouped_calibration_points(
    y,
    probability,
    sample_weight,
    analysis_name,
    groups=10,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    p = np.asarray(
        probability,
        dtype=float,
    )


    w = np.asarray(
        sample_weight,
        dtype=float,
    )


    cutpoints = weighted_quantile(
        values=p,
        weights=w,
        probabilities=np.linspace(
            0,
            1,
            groups + 1,
        ),
    )


    internal_cutpoints = np.unique(
        cutpoints[
            1:-1
        ]
    )


    group_id = (
        np.searchsorted(
            internal_cutpoints,
            p,
            side="right",
        )
        +
        1
    )


    rows = []


    for current_group in sorted(
        np.unique(
            group_id
        )
    ):

        mask = (
            group_id
            ==
            current_group
        )


        rows.append(
            {
                "Analysis":
                    analysis_name,

                "Group":
                    int(
                        current_group
                    ),

                "N":
                    int(
                        mask.sum()
                    ),

                "Weighted_Mean_Predicted":
                    weighted_mean(
                        p[
                            mask
                        ],
                        w[
                            mask
                        ],
                    ),

                "Weighted_Observed_Event_Rate":
                    weighted_mean(
                        y[
                            mask
                        ],
                        w[
                            mask
                        ],
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 11. BINARY PROBABILITY PERFORMANCE
# ============================================================

def binary_probability_metrics(
    y,
    probability,
    sample_weight,
    analysis_name,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    w = np.asarray(
        sample_weight,
        dtype=float,
    )


    if set(
        np.unique(y)
    ) != {
        0,
        1,
    }:

        raise RuntimeError(
            f"{analysis_name}: binary classes incomplete."
        )


    weighted_ap = float(
        average_precision_score(
            y,
            p,
            sample_weight=w,
        )
    )


    weighted_auc = float(
        roc_auc_score(
            y,
            p,
            sample_weight=w,
        )
    )


    log_loss = float(
        -np.sum(
            w
            *
            (
                y * np.log(p)
                +
                (
                    1
                    -
                    y
                )
                *
                np.log(
                    1.0
                    -
                    p
                )
            )
        )
        /
        np.sum(w)
    )


    brier = float(
        np.sum(
            w
            *
            (
                p
                -
                y
            ) ** 2
        )
        /
        np.sum(w)
    )


    observed_rate = weighted_mean(
        y,
        w,
    )


    predicted_rate = weighted_mean(
        p,
        w,
    )


    oe_ratio = float(
        np.sum(
            w * y
        )
        /
        np.sum(
            w * p
        )
    )


    citl = calibration_in_the_large(
        y=y,
        probability=p,
        sample_weight=w,
    )


    slope_result = calibration_slope_model(
        y=y,
        probability=p,
        sample_weight=w,
    )


    return {
        "Analysis":
            analysis_name,

        "N":
            int(
                len(y)
            ),

        "Positive_N":
            int(
                y.sum()
            ),

        "Weighted_Prevalence":
            observed_rate,

        "Weighted_Mean_Predicted":
            predicted_rate,

        "Weighted_Average_Precision":
            weighted_ap,

        "Weighted_ROC_AUC":
            weighted_auc,

        "Weighted_Log_Loss":
            log_loss,

        "Weighted_Brier_Score":
            brier,

        "Calibration_in_the_Large":
            citl,

        "Recalibration_Intercept":
            slope_result[
                "recalibration_intercept"
            ],

        "Calibration_Slope":
            slope_result[
                "calibration_slope"
            ],

        "Observed_to_Expected_Ratio":
            oe_ratio,
    }


# ============================================================
# 12. BINARY THRESHOLD METRICS
# ============================================================

def binary_threshold_metrics(
    y,
    prediction,
    sample_weight,
    analysis_name,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    prediction = np.asarray(
        prediction,
        dtype=int,
    )


    w = np.asarray(
        sample_weight,
        dtype=float,
    )


    positive = (
        y
        ==
        1
    )


    negative = (
        y
        ==
        0
    )


    tp = float(
        w[
            positive
            &
            (
                prediction
                ==
                1
            )
        ].sum()
    )


    fn = float(
        w[
            positive
            &
            (
                prediction
                ==
                0
            )
        ].sum()
    )


    tn = float(
        w[
            negative
            &
            (
                prediction
                ==
                0
            )
        ].sum()
    )


    fp = float(
        w[
            negative
            &
            (
                prediction
                ==
                1
            )
        ].sum()
    )


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    ppv = (
        tp
        /
        (
            tp
            +
            fp
        )
        if
        (
            tp
            +
            fp
        )
        >
        0
        else
        np.nan
    )


    npv = (
        tn
        /
        (
            tn
            +
            fn
        )
        if
        (
            tn
            +
            fn
        )
        >
        0
        else
        np.nan
    )


    f1 = (
        2.0
        *
        tp
        /
        (
            2.0 * tp
            +
            fp
            +
            fn
        )
        if
        (
            2.0 * tp
            +
            fp
            +
            fn
        )
        >
        0
        else
        np.nan
    )


    accuracy = (
        tp
        +
        tn
    ) / (
        tp
        +
        tn
        +
        fp
        +
        fn
    )


    return {
        "Analysis":
            analysis_name,

        "Sensitivity":
            float(
                sensitivity
            ),

        "Specificity":
            float(
                specificity
            ),

        "Balanced_Accuracy":
            float(
                (
                    sensitivity
                    +
                    specificity
                )
                /
                2.0
            ),

        "PPV":
            float(
                ppv
            ),

        "NPV":
            float(
                npv
            ),

        "F1":
            float(
                f1
            ),

        "Accuracy":
            float(
                accuracy
            ),

        "Weighted_TP":
            tp,

        "Weighted_FN":
            fn,

        "Weighted_TN":
            tn,

        "Weighted_FP":
            fp,
    }


# ============================================================
# 13. STAGE 1
# ============================================================

stage1_y = (
    pred[
        "ANC_CLASS_TRUE"
    ]
    .eq(0)
    .astype(int)
    .to_numpy()
)


stage1_p = pred[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


stage1_w = pred[
    "SURVEY_WEIGHT"
].to_numpy(
    dtype=float
)


stage1_probability_metrics = (
    binary_probability_metrics(
        y=stage1_y,
        probability=stage1_p,
        sample_weight=stage1_w,
        analysis_name="Stage 1: No ANC vs Any ANC",
    )
)


stage1_binary_prediction = (
    stage1_p
    >=
    pred[
        "Stage1_Optimized_Threshold"
    ].to_numpy(
        dtype=float
    )
).astype(int)


stage1_threshold_metrics = (
    binary_threshold_metrics(
        y=stage1_y,
        prediction=stage1_binary_prediction,
        sample_weight=stage1_w,
        analysis_name="Stage 1: No ANC vs Any ANC",
    )
)


# ============================================================
# 14. STAGE 2 — TRUE ANC USERS ONLY
# ============================================================

stage2_mask = (
    pred[
        "ANC_CLASS_TRUE"
    ]
    .isin(
        [
            1,
            2,
        ]
    )
    .to_numpy()
)


if int(
    stage2_mask.sum()
) != 4660:

    raise RuntimeError(
        "Stage-2 validation population is not 4660."
    )


stage2_y = (
    pred.loc[
        stage2_mask,
        "ANC_CLASS_TRUE",
    ]
    .eq(2)
    .astype(int)
    .to_numpy()
)


stage2_p = (
    pred.loc[
        stage2_mask,
        "q2_CALIBRATED_CONDITIONAL",
    ]
    .to_numpy(
        dtype=float
    )
)


stage2_w = (
    pred.loc[
        stage2_mask,
        "SURVEY_WEIGHT",
    ]
    .to_numpy(
        dtype=float
    )
)


stage2_probability_metrics = (
    binary_probability_metrics(
        y=stage2_y,
        probability=stage2_p,
        sample_weight=stage2_w,
        analysis_name="Stage 2: >=8 vs 1-7 among true ANC users",
    )
)


stage2_binary_prediction = (
    stage2_p
    >=
    pred.loc[
        stage2_mask,
        "Stage2_Optimized_Threshold",
    ]
    .to_numpy(
        dtype=float
    )
).astype(int)


stage2_threshold_metrics = (
    binary_threshold_metrics(
        y=stage2_y,
        prediction=stage2_binary_prediction,
        sample_weight=stage2_w,
        analysis_name="Stage 2: >=8 vs 1-7 among true ANC users",
    )
)


binary_probability_table = pd.DataFrame(
    [
        stage1_probability_metrics,
        stage2_probability_metrics,
    ]
)


binary_threshold_table = pd.DataFrame(
    [
        stage1_threshold_metrics,
        stage2_threshold_metrics,
    ]
)


# ============================================================
# 15. RAW WINNER DISCRIMINATION — FOLD-WISE ONLY
# ============================================================

raw_fold_rows = []


for outer_fold in [
    0,
    1,
    2,
    3,
    4,
]:

    fold = pred.loc[
        pred[
            "Outer_Fold"
        ]
        ==
        outer_fold
    ]


    # Stage 1

    y = (
        fold[
            "ANC_CLASS_TRUE"
        ]
        .eq(0)
        .astype(int)
        .to_numpy()
    )


    raw = fold[
        "Stage1_Raw_Score"
    ].to_numpy(
        dtype=float
    )


    w = fold[
        "SURVEY_WEIGHT"
    ].to_numpy(
        dtype=float
    )


    raw_fold_rows.append(
        {
            "Outer_Fold":
                outer_fold,

            "Stage":
                "Stage 1",

            "Weighted_AP":
                float(
                    average_precision_score(
                        y,
                        raw,
                        sample_weight=w,
                    )
                ),

            "Weighted_ROC_AUC":
                float(
                    roc_auc_score(
                        y,
                        raw,
                        sample_weight=w,
                    )
                ),
        }
    )


    # Stage 2

    anc_fold = fold.loc[
        fold[
            "ANC_CLASS_TRUE"
        ]
        .isin(
            [
                1,
                2,
            ]
        )
    ]


    y = (
        anc_fold[
            "ANC_CLASS_TRUE"
        ]
        .eq(2)
        .astype(int)
        .to_numpy()
    )


    raw = anc_fold[
        "Stage2_Raw_Score"
    ].to_numpy(
        dtype=float
    )


    w = anc_fold[
        "SURVEY_WEIGHT"
    ].to_numpy(
        dtype=float
    )


    raw_fold_rows.append(
        {
            "Outer_Fold":
                outer_fold,

            "Stage":
                "Stage 2",

            "Weighted_AP":
                float(
                    average_precision_score(
                        y,
                        raw,
                        sample_weight=w,
                    )
                ),

            "Weighted_ROC_AUC":
                float(
                    roc_auc_score(
                        y,
                        raw,
                        sample_weight=w,
                    )
                ),
        }
    )


raw_fold_discrimination = pd.DataFrame(
    raw_fold_rows
)


raw_fold_summary = (
    raw_fold_discrimination
    .groupby(
        "Stage",
        as_index=False,
    )
    .agg(
        Mean_Weighted_AP=(
            "Weighted_AP",
            "mean",
        ),

        SD_Weighted_AP=(
            "Weighted_AP",
            "std",
        ),

        Mean_Weighted_ROC_AUC=(
            "Weighted_ROC_AUC",
            "mean",
        ),

        SD_Weighted_ROC_AUC=(
            "Weighted_ROC_AUC",
            "std",
        ),
    )
)


# ============================================================
# 16. MULTICLASS PROBABILISTIC PERFORMANCE
# ============================================================

true_class = pred[
    "ANC_CLASS_TRUE"
].to_numpy(
    dtype=int
)


multiclass_probability = pred[
    [
        "P_NO_ANC",
        "P_ANC_1_7",
        "P_ANC_GE8",
    ]
].to_numpy(
    dtype=float
)


multiclass_weight = pred[
    "SURVEY_WEIGHT"
].to_numpy(
    dtype=float
)


probability_clipped = np.clip(
    multiclass_probability,
    1e-15,
    1.0,
)


true_probability = (
    probability_clipped[
        np.arange(
            len(pred)
        ),
        true_class,
    ]
)


multiclass_log_loss = float(
    -np.sum(
        multiclass_weight
        *
        np.log(
            true_probability
        )
    )
    /
    np.sum(
        multiclass_weight
    )
)


one_hot_true = np.eye(
    3
)[
    true_class
]


multiclass_brier = float(
    np.sum(
        multiclass_weight
        *
        np.sum(
            (
                multiclass_probability
                -
                one_hot_true
            ) ** 2,
            axis=1,
        )
    )
    /
    np.sum(
        multiclass_weight
    )
)


# ============================================================
# 17. MULTICLASS ONE-vs-REST METRICS
# ============================================================

class_names = {
    0:
        "No ANC",

    1:
        "1-7 visits",

    2:
        ">=8 visits",
}


class_probability_columns = {
    0:
        "P_NO_ANC",

    1:
        "P_ANC_1_7",

    2:
        "P_ANC_GE8",
}


classwise_rows = []


for class_id in [
    0,
    1,
    2,
]:

    y = (
        true_class
        ==
        class_id
    ).astype(int)


    p = pred[
        class_probability_columns[
            class_id
        ]
    ].to_numpy(
        dtype=float
    )


    metrics = binary_probability_metrics(
        y=y,
        probability=p,
        sample_weight=multiclass_weight,
        analysis_name=(
            f"Multiclass OVR: "
            f"{class_names[class_id]}"
        ),
    )


    metrics[
        "Class_ID"
    ] = class_id


    metrics[
        "Class"
    ] = class_names[
        class_id
    ]


    classwise_rows.append(
        metrics
    )


classwise_probability_table = pd.DataFrame(
    classwise_rows
)


macro_ap = float(
    classwise_probability_table[
        "Weighted_Average_Precision"
    ].mean()
)


macro_auc = float(
    classwise_probability_table[
        "Weighted_ROC_AUC"
    ].mean()
)


multiclass_summary = pd.DataFrame(
    [
        {
            "N":
                len(pred),

            "Weighted_Multiclass_Log_Loss":
                multiclass_log_loss,

            "Weighted_Multiclass_Brier_Score":
                multiclass_brier,

            "Macro_OVR_Average_Precision":
                macro_ap,

            "Macro_OVR_ROC_AUC":
                macro_auc,
        }
    ]
)


# ============================================================
# 18. MULTICLASS THRESHOLD CLASSIFICATION
# ============================================================

predicted_class = pred[
    "PREDICTED_ANC_CLASS"
].to_numpy(
    dtype=int
)


weighted_confusion = np.zeros(
    (
        3,
        3,
    ),
    dtype=float,
)


for true_value in [
    0,
    1,
    2,
]:

    for predicted_value in [
        0,
        1,
        2,
    ]:

        mask = (
            (true_class == true_value)
            &
            (
                predicted_class
                ==
                predicted_value
            )
        )


        weighted_confusion[
            true_value,
            predicted_value
        ] = multiclass_weight[
            mask
        ].sum()


multiclass_classification_rows = []


for class_id in [
    0,
    1,
    2,
]:

    tp = weighted_confusion[
        class_id,
        class_id
    ]


    fn = (
        weighted_confusion[
            class_id,
            :
        ].sum()
        -
        tp
    )


    fp = (
        weighted_confusion[
            :,
            class_id
        ].sum()
        -
        tp
    )


    precision = (
        tp
        /
        (
            tp
            +
            fp
        )
        if
        (
            tp
            +
            fp
        )
        >
        0
        else
        np.nan
    )


    recall = (
        tp
        /
        (
            tp
            +
            fn
        )
    )


    f1 = (
        2.0
        *
        precision
        *
        recall
        /
        (
            precision
            +
            recall
        )
        if
        (
            precision
            +
            recall
        )
        >
        0
        else
        np.nan
    )


    multiclass_classification_rows.append(
        {
            "Class_ID":
                class_id,

            "Class":
                class_names[
                    class_id
                ],

            "Precision":
                float(
                    precision
                ),

            "Recall":
                float(
                    recall
                ),

            "F1":
                float(
                    f1
                ),
        }
    )


multiclass_classification = pd.DataFrame(
    multiclass_classification_rows
)


weighted_overall_accuracy = float(
    np.trace(
        weighted_confusion
    )
    /
    weighted_confusion.sum()
)


multiclass_balanced_accuracy = float(
    multiclass_classification[
        "Recall"
    ].mean()
)


multiclass_macro_f1 = float(
    multiclass_classification[
        "F1"
    ].mean()
)


multiclass_threshold_summary = pd.DataFrame(
    [
        {
            "Weighted_Overall_Accuracy":
                weighted_overall_accuracy,

            "Macro_Recall_Balanced_Accuracy":
                multiclass_balanced_accuracy,

            "Macro_F1":
                multiclass_macro_f1,
        }
    ]
)


weighted_confusion_table = pd.DataFrame(
    weighted_confusion,
    index=[
        "True_No_ANC",
        "True_1_7",
        "True_GE8",
    ],
    columns=[
        "Pred_No_ANC",
        "Pred_1_7",
        "Pred_GE8",
    ],
)


# ============================================================
# 19. FLEXIBLE CALIBRATION CURVES
# ============================================================

curve_frames = []

knot_frames = []

grouped_frames = []


calibration_inputs = [
    (
        "Stage 1",
        stage1_y,
        stage1_p,
        stage1_w,
    ),

    (
        "Stage 2",
        stage2_y,
        stage2_p,
        stage2_w,
    ),

    (
        "Class 0: No ANC",
        (
            true_class
            ==
            0
        ).astype(int),
        pred[
            "P_NO_ANC"
        ].to_numpy(
            dtype=float
        ),
        multiclass_weight,
    ),

    (
        "Class 1: 1-7 visits",
        (
            true_class
            ==
            1
        ).astype(int),
        pred[
            "P_ANC_1_7"
        ].to_numpy(
            dtype=float
        ),
        multiclass_weight,
    ),

    (
        "Class 2: >=8 visits",
        (
            true_class
            ==
            2
        ).astype(int),
        pred[
            "P_ANC_GE8"
        ].to_numpy(
            dtype=float
        ),
        multiclass_weight,
    ),
]


for (
    analysis_name,
    y,
    p,
    w,
) in calibration_inputs:

    curve, knots = flexible_calibration_curve(
        y=y,
        probability=p,
        sample_weight=w,
        analysis_name=analysis_name,
    )


    groups = grouped_calibration_points(
        y=y,
        probability=p,
        sample_weight=w,
        analysis_name=analysis_name,
        groups=10,
    )


    curve_frames.append(
        curve
    )


    knot_frames.append(
        knots
    )


    grouped_frames.append(
        groups
    )


calibration_curve_data = pd.concat(
    curve_frames,
    ignore_index=True,
)


calibration_knots = pd.concat(
    knot_frames,
    ignore_index=True,
)


grouped_calibration_data = pd.concat(
    grouped_frames,
    ignore_index=True,
)


# ============================================================
# 20. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                (
                    sha256_file(
                        PRIMARY_FILE
                    )
                    ==
                    EXPECTED_PRIMARY_SHA256
                ),
        },

        {
            "Check":
                "Locked OOF hash unchanged",

            "PASS":
                (
                    sha256_file(
                        FINAL_OOF_FILE
                    )
                    ==
                    EXPECTED_LOCKED_OOF_SHA256
                ),
        },

        {
            "Check":
                "Exactly 5063 primary OOF rows",

            "PASS":
                len(pred)
                ==
                5063,
        },

        {
            "Check":
                "Stage 2 evaluation N = 4660",

            "PASS":
                int(
                    stage2_mask.sum()
                )
                ==
                4660,
        },

        {
            "Check":
                "Binary probability metrics finite",

            "PASS":
                bool(
                    np.isfinite(
                        binary_probability_table[
                            [
                                "Weighted_Average_Precision",
                                "Weighted_ROC_AUC",
                                "Weighted_Log_Loss",
                                "Weighted_Brier_Score",
                                "Calibration_in_the_Large",
                                "Recalibration_Intercept",
                                "Calibration_Slope",
                                "Observed_to_Expected_Ratio",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Multiclass probabilistic metrics finite",

            "PASS":
                bool(
                    np.isfinite(
                        multiclass_summary.to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Classwise OVR metrics finite",

            "PASS":
                bool(
                    np.isfinite(
                        classwise_probability_table[
                            [
                                "Weighted_Average_Precision",
                                "Weighted_ROC_AUC",
                                "Weighted_Log_Loss",
                                "Weighted_Brier_Score",
                                "Calibration_in_the_Large",
                                "Calibration_Slope",
                                "Observed_to_Expected_Ratio",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Flexible calibration curves finite",

            "PASS":
                bool(
                    np.isfinite(
                        calibration_curve_data[
                            [
                                "Predicted_Probability",
                                "Smoothed_Observed_Probability",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Four spline knots per calibration analysis",

            "PASS":
                bool(
                    (
                        calibration_knots
                        .groupby(
                            "Analysis"
                        )
                        .size()
                        ==
                        4
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Threshold metrics use locked predictions only",

            "PASS":
                True,
        },

        {
            "Check":
                "No threshold re-optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "No model fitting",

            "PASS":
                True,
        },

        {
            "Check":
                "No hyperparameter tuning",

            "PASS":
                True,
        },

        {
            "Check":
                "No model re-selection",

            "PASS":
                True,
        },

        {
            "Check":
                "No confidence intervals calculated yet",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "STEP 20 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 21. SAVE OUTPUTS
# ============================================================

BINARY_PROBABILITY_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_binary_probability_metrics.csv"
)


BINARY_THRESHOLD_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_binary_threshold_metrics.csv"
)


RAW_FOLD_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_raw_discrimination_by_outer_fold.csv"
)


RAW_FOLD_SUMMARY_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_raw_discrimination_summary.csv"
)


MULTICLASS_SUMMARY_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_probability_metrics.csv"
)


CLASSWISE_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_OVR_classwise_metrics.csv"
)


MULTICLASS_CLASSIFICATION_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_threshold_class_metrics.csv"
)


MULTICLASS_THRESHOLD_SUMMARY_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_threshold_summary.csv"
)


CONFUSION_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_weighted_confusion_matrix.csv"
)


CALIBRATION_CURVE_FILE = (
    DIRS["figures"]
    / "STEP20_PRIMARY_calibration_curve_data.csv"
)


CALIBRATION_KNOT_FILE = (
    DIRS["protocol"]
    / "STEP20_PRIMARY_calibration_spline_knots.csv"
)


GROUPED_CALIBRATION_FILE = (
    DIRS["figures"]
    / "STEP20_PRIMARY_grouped_calibration_points.csv"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP20_PRIMARY_hard_qc.csv"
)


binary_probability_table.to_csv(
    BINARY_PROBABILITY_FILE,
    index=False,
)


binary_threshold_table.to_csv(
    BINARY_THRESHOLD_FILE,
    index=False,
)


raw_fold_discrimination.to_csv(
    RAW_FOLD_FILE,
    index=False,
)


raw_fold_summary.to_csv(
    RAW_FOLD_SUMMARY_FILE,
    index=False,
)


multiclass_summary.to_csv(
    MULTICLASS_SUMMARY_FILE,
    index=False,
)


classwise_probability_table.to_csv(
    CLASSWISE_FILE,
    index=False,
)


multiclass_classification.to_csv(
    MULTICLASS_CLASSIFICATION_FILE,
    index=False,
)


multiclass_threshold_summary.to_csv(
    MULTICLASS_THRESHOLD_SUMMARY_FILE,
    index=False,
)


weighted_confusion_table.to_csv(
    CONFUSION_FILE,
)


calibration_curve_data.to_csv(
    CALIBRATION_CURVE_FILE,
    index=False,
)


calibration_knots.to_csv(
    CALIBRATION_KNOT_FILE,
    index=False,
)


grouped_calibration_data.to_csv(
    GROUPED_CALIBRATION_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


# ============================================================
# 22. MANIFEST
# ============================================================

manifest = {
    "step":
        "STEP 20",

    "status":
        "PRIMARY_LOCKED_OOF_POINT_ESTIMATES_COMPLETE",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "locked_oof_sha256":
        sha256_file(
            FINAL_OOF_FILE
        ),

    "stage1_n":
        5063,

    "stage2_n":
        4660,

    "multiclass_n":
        5063,

    "survey_weighted_evaluation":
        True,

    "pooled_discrimination_score":
        (
            "calibrated cross-fitted probability"
        ),

    "raw_selected_winner_discrimination":
        (
            "outer-fold-specific only; not pooled across folds"
        ),

    "calibration_in_the_large":
        (
            "weighted logistic intercept with predicted "
            "logit as offset"
        ),

    "calibration_slope":
        (
            "weighted logistic recalibration model"
        ),

    "flexible_calibration_curve":
        (
            "weighted restricted cubic spline on predicted log-odds"
        ),

    "spline_knots":
        (
            "weighted 5th, 35th, 65th, 95th percentiles"
        ),

    "hosmer_lemeshow":
        False,

    "threshold_reoptimized":
        False,

    "model_refitted":
        False,

    "hyperparameters_changed":
        False,

    "confidence_intervals_calculated":
        False,

    "binary_probability_metrics_sha256":
        sha256_file(
            BINARY_PROBABILITY_FILE
        ),

    "multiclass_metrics_sha256":
        sha256_file(
            MULTICLASS_SUMMARY_FILE
        ),

    "classwise_metrics_sha256":
        sha256_file(
            CLASSWISE_FILE
        ),

    "calibration_curve_data_sha256":
        sha256_file(
            CALIBRATION_CURVE_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP20_PRIMARY_evaluation_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 23. FINAL IMMUTABILITY CHECK
# ============================================================

if (
    sha256_file(
        FINAL_OOF_FILE
    )
    !=
    EXPECTED_LOCKED_OOF_SHA256
):

    raise RuntimeError(
        "Locked OOF predictions changed during Step 20."
    )


# ============================================================
# 24. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 20 — FINAL PRIMARY LOCKED-OOF POINT ESTIMATES"
)


print(
    "=" * 78
)


print(
    "\nBINARY PROBABILITY PERFORMANCE:"
)


display(
    binary_probability_table
)


print(
    "\nRAW SELECTED-WINNER DISCRIMINATION BY OUTER FOLD:"
)


display(
    raw_fold_discrimination
)


print(
    "\nRAW SELECTED-WINNER DISCRIMINATION SUMMARY:"
)


display(
    raw_fold_summary
)


print(
    "\nBINARY OPTIMIZED-THRESHOLD PERFORMANCE:"
)


display(
    binary_threshold_table
)


print(
    "\nMULTICLASS PROBABILISTIC PERFORMANCE:"
)


display(
    multiclass_summary
)


print(
    "\nMULTICLASS ONE-vs-REST CLASSWISE PERFORMANCE:"
)


display(
    classwise_probability_table
)


print(
    "\nMULTICLASS THRESHOLD CLASSIFICATION:"
)


display(
    multiclass_classification
)


print(
    "\nMULTICLASS THRESHOLD SUMMARY:"
)


display(
    multiclass_threshold_summary
)


print(
    "\nSURVEY-WEIGHTED CONFUSION MATRIX:"
)


display(
    weighted_confusion_table
)


print(
    "\nCALIBRATION SPLINE KNOTS:"
)


display(
    calibration_knots
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nIMPORTANT:"
)


print(
    "- Evaluation used ONLY locked cross-fitted outer-OOF predictions."
)


print(
    "- No model was refitted or reselected."
)


print(
    "- No hyperparameter was changed."
)


print(
    "- No calibration model was refitted."
)


print(
    "- No threshold was re-optimized."
)


print(
    "- Raw winner scores were evaluated fold-wise, not pooled across incompatible score scales."
)


print(
    "- Pooled final discrimination uses calibrated cross-fitted probabilities."
)


print(
    "- Confidence intervals are NOT calculated in this step."
)


print(
    "- Next: stratified PSU bootstrap uncertainty on these locked predictions."
)


print(
    "\nINTEGRITY:"
)


print(
    f"Locked OOF : {sha256_file(FINAL_OOF_FILE)}"
)


print(
    "=" * 78
)


print(
    "STATUS: STEP 20 PRIMARY POINT ESTIMATES PASS"
)


print(
    "=" * 78
)


STEP 20 — FINAL PRIMARY LOCKED-OOF POINT ESTIMATES

BINARY PROBABILITY PERFORMANCE:


,Analysis,N,Positive_N,Weighted_Prevalence,Weighted_Mean_Predicted,Weighted_Average_Precision,Weighted_ROC_AUC,Weighted_Log_Loss,Weighted_Brier_Score,Calibration_in_the_Large,Recalibration_Intercept,Calibration_Slope,Observed_to_Expected_Ratio
0,Stage 1: No ANC vs Any ANC,5063,403,0.079434,0.079278,0.235306,0.778288,0.239733,0.066997,0.002330,0.057187,1.026574,1.001958
1,Stage 2: >=8 vs 1-7 among true ANC users,4660,253,0.054901,0.051771,0.161610,0.735076,0.193557,0.049192,0.066214,-0.146403,0.917693,1.060461



RAW SELECTED-WINNER DISCRIMINATION BY OUTER FOLD:


,Outer_Fold,Stage,Weighted_AP,Weighted_ROC_AUC
0,0,Stage 1,0.238911,0.810200
1,0,Stage 2,0.184785,0.761003
2,1,Stage 1,0.201224,0.788575
3,1,Stage 2,0.145510,0.770436
4,2,Stage 1,0.247034,0.771128
5,2,Stage 2,0.132704,0.709334
6,3,Stage 1,0.283478,0.797796
7,3,Stage 2,0.150502,0.704229
8,4,Stage 1,0.243271,0.740789
9,4,Stage 2,0.218238,0.724220



RAW SELECTED-WINNER DISCRIMINATION SUMMARY:


,Stage,Mean_Weighted_AP,SD_Weighted_AP,Mean_Weighted_ROC_AUC,SD_Weighted_ROC_AUC
0,Stage 1,0.242784,0.029226,0.781697,0.026945
1,Stage 2,0.166348,0.034820,0.733845,0.030195


... (13 outputs left out)

,Analysis,Knot_Number,Weighted_Quantile,Knot_Logit,Knot_Probability
0,Stage 1,1,0.05,-4.883965,0.007510
1,Stage 1,2,0.35,-3.315054,0.035058
2,Stage 1,3,0.65,-2.424383,0.081332
3,Stage 1,4,0.95,-1.172466,0.236410
4,Stage 2,1,0.05,-4.466560,0.011356
5,Stage 2,2,0.35,-3.663063,0.025012
6,Stage 2,3,0.65,-3.075400,0.044133
7,Stage 2,4,0.95,-1.575003,0.171504
8,Class 0: No ANC,1,0.05,-4.883965,0.007510
9,Class 0: No ANC,2,0.35,-3.315054,0.035058



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Locked OOF hash unchanged,True
2,Exactly 5063 primary OOF rows,True
3,Stage 2 evaluation N = 4660,True
4,Binary probability metrics finite,True
5,Multiclass probabilistic metrics finite,True
6,Classwise OVR metrics finite,True
7,Flexible calibration curves finite,True
8,Four spline knots per calibration analysis,True
9,Threshold metrics use locked predictions only,True



IMPORTANT:
- Evaluation used ONLY locked cross-fitted outer-OOF predictions.
- No model was refitted or reselected.
- No hyperparameter was changed.
- No calibration model was refitted.
- No threshold was re-optimized.
- Raw winner scores were evaluated fold-wise, not pooled across incompatible score scales.
- Pooled final discrimination uses calibrated cross-fitted probabilities.
- Confidence intervals are NOT calculated in this step.
- Next: stratified PSU bootstrap uncertainty on these locked predictions.

INTEGRITY:
Locked OOF : 6c4f4e21b09a54c286521682e9fb8ee13eda36ed94196d350fbb0622733a99bc
STATUS: STEP 20 PRIMARY POINT ESTIMATES PASS


In [6]:
# ============================================================
# STEP 21 — SURVEY-DESIGN BOOTSTRAP UNCERTAINTY
#           LOCKED PRIMARY OUTER-OOF PREDICTIONS
# ============================================================
#
# PURPOSE
# -------
# Obtain 95% confidence intervals for the already-locked
# Step-20 primary performance estimates.
#
#
# DESIGN
# ------
# Rao-Wu style rescaled PSU bootstrap.
#
# Within each survey stratum h:
#
#   original PSU count = n_h
#
#   draw (n_h - 1) PSUs WITH replacement
#
#   bootstrap factor for PSU i:
#
#       f_hi = n_h / (n_h - 1) * multiplicity_hi
#
#   replicate participant weight:
#
#       w*_hij = original DHS weight_hij * f_hi
#
#
# This preserves:
#   - survey stratification
#   - PSU clustering
#   - original sampling weights
#
#
# IMPORTANT
# ---------
# Predictions are LOCKED.
#
# NO:
#   - model refitting
#   - hyperparameter tuning
#   - model-family reselection
#   - recalibration of predictions
#   - threshold re-optimization
#
# Calibration intercept/slope are PERFORMANCE STATISTICS and
# are re-estimated in each bootstrap replicate, exactly as one
# re-estimates AUC/Brier/etc. They do NOT modify predictions.
#
#
# INTERPRETATION
# --------------
# These intervals quantify uncertainty conditional on the
# locked cross-fitted prediction procedure and observed survey
# sample.
#
# They do NOT represent a full bootstrap repetition of the
# entire model-development/tuning process.
#
#
# BOOTSTRAP
# ---------
# Replicates = 2000
# Seed       = 2026
# CI         = percentile 95% CI
# ============================================================


# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from pathlib import Path

import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from scipy.special import (
    expit,
    logit,
)

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from IPython.display import display


# ============================================================
# 2. RESTORE DIRECTORIES IF NECESSARY
# ============================================================

if "DIRS" not in globals():

    ROOT = Path(
        r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
    )


    DIRS = {
        "root":
            ROOT,

        "processed":
            ROOT / "01_processed_data",

        "qc":
            ROOT / "02_quality_control",

        "protocol":
            ROOT / "03_protocol",

        "splits":
            ROOT / "04_cv_splits",

        "models":
            ROOT / "05_models",

        "predictions":
            ROOT / "06_predictions",

        "calibration":
            ROOT / "07_calibration",

        "bootstrap":
            ROOT / "08_bootstrap",

        "geographic":
            ROOT / "09_geographic_validation",

        "explainability":
            ROOT / "10_explainability",

        "sensitivity":
            ROOT / "11_sensitivity",

        "subgroup":
            ROOT / "12_subgroup_analysis",

        "tables":
            ROOT / "13_tables",

        "figures":
            ROOT / "14_figures",

        "logs":
            ROOT / "15_logs",

        "reporting":
            ROOT / "16_reporting",

        "archive":
            ROOT / "17_reproducibility_archive",
    }


# ============================================================
# 3. CONSTANTS
# ============================================================

BOOTSTRAP_REPLICATES = 2000

BOOTSTRAP_SEED = 2026

CI_LEVEL = 0.95

CI_LOWER_QUANTILE = 0.025

CI_UPPER_QUANTILE = 0.975

PROBABILITY_CLIP = 1e-6

NUMERICAL_TOLERANCE = 1e-10


# ============================================================
# 4. INPUT FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


FINAL_OOF_FILE = (
    DIRS["predictions"]
    / "STEP19C_PRIMARY_final_outer_OOF_predictions_RESTRICTED.csv"
)


STEP20_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP20_PRIMARY_evaluation_manifest.json"
)


STEP20_BINARY_PROBABILITY_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_binary_probability_metrics.csv"
)


STEP20_BINARY_THRESHOLD_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_binary_threshold_metrics.csv"
)


STEP20_MULTICLASS_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_probability_metrics.csv"
)


STEP20_CLASSWISE_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_OVR_classwise_metrics.csv"
)


STEP20_MULTICLASS_CLASS_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_threshold_class_metrics.csv"
)


STEP20_MULTICLASS_THRESHOLD_FILE = (
    DIRS["tables"]
    / "STEP20_PRIMARY_multiclass_threshold_summary.csv"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_LOCKED_OOF_SHA256 = (
    "6c4f4e21b09a54c286521682e9fb8ee1"
    "3eda36ed94196d350fbb0622733a99bc"
)


# ============================================================
# 5. UTILITIES
# ============================================================

def sha256_file(
    path,
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):

            digest.update(
                block
            )


    return digest.hexdigest()


def read_json(
    path,
):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(
            handle
        )


def weighted_mean(
    x,
    w,
):

    x = np.asarray(
        x,
        dtype=float,
    )


    w = np.asarray(
        w,
        dtype=float,
    )


    total = np.sum(
        w
    )


    if (
        not np.isfinite(total)
        or
        total <= 0
    ):

        return np.nan


    return float(
        np.sum(
            w * x
        )
        /
        total
    )


# ============================================================
# 6. INPUT / HASH CHECK
# ============================================================

required_files = [
    PRIMARY_FILE,
    FINAL_OOF_FILE,
    STEP20_MANIFEST_FILE,
    STEP20_BINARY_PROBABILITY_FILE,
    STEP20_BINARY_THRESHOLD_FILE,
    STEP20_MULTICLASS_FILE,
    STEP20_CLASSWISE_FILE,
    STEP20_MULTICLASS_CLASS_FILE,
    STEP20_MULTICLASS_THRESHOLD_FILE,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )


if (
    sha256_file(
        PRIMARY_FILE
    )
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    sha256_file(
        FINAL_OOF_FILE
    )
    !=
    EXPECTED_LOCKED_OOF_SHA256
):

    raise RuntimeError(
        "Locked OOF hash mismatch."
    )


step20_manifest = read_json(
    STEP20_MANIFEST_FILE
)


if (
    step20_manifest.get(
        "status"
    )
    !=
    "PRIMARY_LOCKED_OOF_POINT_ESTIMATES_COMPLETE"
):

    raise RuntimeError(
        "Step 20 is not complete/frozen."
    )


# ============================================================
# 7. LOAD LOCKED PREDICTIONS
# ============================================================

pred = pd.read_csv(
    FINAL_OOF_FILE
)


if len(
    pred
) != 5063:

    raise RuntimeError(
        "Locked OOF N is not 5063."
    )


required_prediction_columns = [
    "_ROW_KEY",
    "Outer_Fold",
    "ANC_CLASS_TRUE",
    "SURVEY_WEIGHT",
    "PSU",
    "STRATUM",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
    "P_NO_ANC",
    "P_ANC_1_7",
    "P_ANC_GE8",
    "PREDICTED_ANC_CLASS",
]


missing_prediction_columns = [
    column
    for column in required_prediction_columns
    if column not in pred.columns
]


if missing_prediction_columns:

    raise RuntimeError(
        "Locked prediction file missing columns:\n"
        f"{missing_prediction_columns}"
    )


# ============================================================
# 8. SURVEY DESIGN QC
# ============================================================

if pred[
    "_ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate participant row."
    )


if pred[
    [
        "STRATUM",
        "PSU",
    ]
].isna().any().any():

    raise RuntimeError(
        "Missing PSU/stratum."
    )


# Composite PSU key protects against possible repeated PSU codes
# across strata.

cluster_table = (
    pred[
        [
            "STRATUM",
            "PSU",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "STRATUM",
            "PSU",
        ]
    )
    .reset_index(
        drop=True
    )
)


cluster_table[
    "__CLUSTER_INDEX"
] = np.arange(
    len(
        cluster_table
    ),
    dtype=int,
)


if len(
    cluster_table
) != 672:

    raise RuntimeError(
        f"Expected 672 analytic PSUs; "
        f"observed {len(cluster_table)}."
    )


stratum_cluster_counts = (
    cluster_table
    .groupby(
        "STRATUM"
    )
    .size()
    .rename(
        "PSU_Count"
    )
    .reset_index()
)


if len(
    stratum_cluster_counts
) != 16:

    raise RuntimeError(
        "Expected 16 strata."
    )


if (
    stratum_cluster_counts[
        "PSU_Count"
    ]
    <=
    1
).any():

    raise RuntimeError(
        "Rao-Wu rescaled bootstrap requires "
        "more than one PSU per stratum."
    )


pred = (
    pred
    .assign(
        __ORIGINAL_ORDER=np.arange(
            len(pred)
        )
    )
    .merge(
        cluster_table,
        on=[
            "STRATUM",
            "PSU",
        ],
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        "__ORIGINAL_ORDER"
    )
    .reset_index(
        drop=True
    )
)


row_cluster_index = pred[
    "__CLUSTER_INDEX"
].to_numpy(
    dtype=int
)


if (
    row_cluster_index
    <
    0
).any():

    raise RuntimeError(
        "Invalid cluster index."
    )


# ============================================================
# 9. LOCKED ARRAYS
# ============================================================

base_weight = pred[
    "SURVEY_WEIGHT"
].to_numpy(
    dtype=float
)


if (
    not np.isfinite(
        base_weight
    ).all()
    or
    (
        base_weight
        <=
        0
    ).any()
):

    raise RuntimeError(
        "Invalid survey weights."
    )


true_class = pred[
    "ANC_CLASS_TRUE"
].to_numpy(
    dtype=int
)


# Stage 1

stage1_y = (
    true_class
    ==
    0
).astype(int)


stage1_p = pred[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


stage1_prediction = (
    stage1_p
    >=
    pred[
        "Stage1_Optimized_Threshold"
    ].to_numpy(
        dtype=float
    )
).astype(int)


# Stage 2

stage2_mask = np.isin(
    true_class,
    [
        1,
        2,
    ],
)


if int(
    stage2_mask.sum()
) != 4660:

    raise RuntimeError(
        "Stage-2 population is not 4660."
    )


stage2_y = (
    true_class[
        stage2_mask
    ]
    ==
    2
).astype(int)


stage2_p = (
    pred.loc[
        stage2_mask,
        "q2_CALIBRATED_CONDITIONAL",
    ]
    .to_numpy(
        dtype=float
    )
)


stage2_prediction = (
    stage2_p
    >=
    pred.loc[
        stage2_mask,
        "Stage2_Optimized_Threshold",
    ]
    .to_numpy(
        dtype=float
    )
).astype(int)


# Multiclass

multiclass_probability = pred[
    [
        "P_NO_ANC",
        "P_ANC_1_7",
        "P_ANC_GE8",
    ]
].to_numpy(
    dtype=float
)


locked_predicted_class = pred[
    "PREDICTED_ANC_CLASS"
].to_numpy(
    dtype=int
)


# ============================================================
# 10. SAFE WEIGHTED AP / AUC
# ============================================================

def weighted_ap_auc(
    y,
    score,
    w,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    score = np.asarray(
        score,
        dtype=float,
    )


    w = np.asarray(
        w,
        dtype=float,
    )


    positive_weight = float(
        w[
            y
            ==
            1
        ].sum()
    )


    negative_weight = float(
        w[
            y
            ==
            0
        ].sum()
    )


    if (
        positive_weight
        <=
        0
        or
        negative_weight
        <=
        0
    ):

        return (
            np.nan,
            np.nan,
        )


    ap = float(
        average_precision_score(
            y,
            score,
            sample_weight=w,
        )
    )


    auc = float(
        roc_auc_score(
            y,
            score,
            sample_weight=w,
        )
    )


    return (
        ap,
        auc,
    )


# ============================================================
# 11. CALIBRATION-IN-THE-LARGE
# ============================================================

def weighted_citl(
    y,
    probability,
    w,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    w = np.asarray(
        w,
        dtype=float,
    )


    if (
        w[
            y
            ==
            1
        ].sum()
        <=
        0
        or
        w[
            y
            ==
            0
        ].sum()
        <=
        0
    ):

        return np.nan


    lp = logit(
        p
    )


    alpha = 0.0


    for _ in range(
        100
    ):

        mu = expit(
            alpha
            +
            lp
        )


        score = float(
            np.sum(
                w
                *
                (
                    y
                    -
                    mu
                )
            )
        )


        information = float(
            np.sum(
                w
                *
                mu
                *
                (
                    1.0
                    -
                    mu
                )
            )
        )


        if (
            not np.isfinite(
                information
            )
            or
            information
            <=
            1e-15
        ):

            return np.nan


        step = (
            score
            /
            information
        )


        alpha += step


        if abs(
            step
        ) < 1e-12:

            return float(
                alpha
            )


    return float(
        alpha
    )


# ============================================================
# 12. WEIGHTED LOGISTIC RECALIBRATION INTERCEPT / SLOPE
# ============================================================

def weighted_calibration_model(
    y,
    probability,
    w,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    w = np.asarray(
        w,
        dtype=float,
    )


    if (
        w[
            y
            ==
            1
        ].sum()
        <=
        0
        or
        w[
            y
            ==
            0
        ].sum()
        <=
        0
    ):

        return (
            np.nan,
            np.nan,
        )


    lp = logit(
        p
    )


    design = np.column_stack(
        [
            np.ones(
                len(lp)
            ),
            lp,
        ]
    )


    theta = np.array(
        [
            0.0,
            1.0,
        ],
        dtype=float,
    )


    for _ in range(
        100
    ):

        eta = (
            design
            @
            theta
        )


        mu = expit(
            eta
        )


        variance = (
            mu
            *
            (
                1.0
                -
                mu
            )
        )


        score = (
            design.T
            @
            (
                w
                *
                (
                    y
                    -
                    mu
                )
            )
        )


        information = (
            design.T
            @
            (
                design
                *
                (
                    w
                    *
                    variance
                )[
                    :,
                    None
                ]
            )
        )


        if not np.isfinite(
            information
        ).all():

            return (
                np.nan,
                np.nan,
            )


        try:

            step = np.linalg.solve(
                information,
                score,
            )


        except np.linalg.LinAlgError:

            return (
                np.nan,
                np.nan,
            )


        if not np.isfinite(
            step
        ).all():

            return (
                np.nan,
                np.nan,
            )


        theta = (
            theta
            +
            step
        )


        if np.max(
            np.abs(
                step
            )
        ) < 1e-10:

            return (
                float(
                    theta[0]
                ),
                float(
                    theta[1]
                ),
            )


    return (
        float(
            theta[0]
        ),
        float(
            theta[1]
        ),
    )


# ============================================================
# 13. BINARY PROBABILITY METRICS
# ============================================================

def binary_probability_values(
    y,
    probability,
    w,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        PROBABILITY_CLIP,
        1.0
        -
        PROBABILITY_CLIP,
    )


    w = np.asarray(
        w,
        dtype=float,
    )


    ap, auc = weighted_ap_auc(
        y,
        p,
        w,
    )


    total_weight = float(
        w.sum()
    )


    if total_weight <= 0:

        return {
            key:
                np.nan

            for key in [
                "AP",
                "ROC_AUC",
                "Log_Loss",
                "Brier",
                "CITL",
                "Recalibration_Intercept",
                "Calibration_Slope",
                "O_E",
            ]
        }


    log_loss = float(
        -np.sum(
            w
            *
            (
                y
                *
                np.log(
                    p
                )

                +

                (
                    1
                    -
                    y
                )
                *
                np.log(
                    1.0
                    -
                    p
                )
            )
        )
        /
        total_weight
    )


    brier = float(
        np.sum(
            w
            *
            (
                p
                -
                y
            ) ** 2
        )
        /
        total_weight
    )


    citl = weighted_citl(
        y,
        p,
        w,
    )


    (
        recalibration_intercept,
        calibration_slope,
    ) = weighted_calibration_model(
        y,
        p,
        w,
    )


    expected = float(
        np.sum(
            w
            *
            p
        )
    )


    observed = float(
        np.sum(
            w
            *
            y
        )
    )


    oe = (
        observed
        /
        expected

        if expected
        >
        0

        else
        np.nan
    )


    return {
        "AP":
            ap,

        "ROC_AUC":
            auc,

        "Log_Loss":
            log_loss,

        "Brier":
            brier,

        "CITL":
            citl,

        "Recalibration_Intercept":
            recalibration_intercept,

        "Calibration_Slope":
            calibration_slope,

        "O_E":
            float(
                oe
            ),
    }


# ============================================================
# 14. BINARY THRESHOLD METRICS
# ============================================================

def binary_threshold_values(
    y,
    prediction,
    w,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    prediction = np.asarray(
        prediction,
        dtype=int,
    )


    w = np.asarray(
        w,
        dtype=float,
    )


    positive = (
        y
        ==
        1
    )


    negative = (
        y
        ==
        0
    )


    tp = float(
        w[
            positive
            &
            (
                prediction
                ==
                1
            )
        ].sum()
    )


    fn = float(
        w[
            positive
            &
            (
                prediction
                ==
                0
            )
        ].sum()
    )


    tn = float(
        w[
            negative
            &
            (
                prediction
                ==
                0
            )
        ].sum()
    )


    fp = float(
        w[
            negative
            &
            (
                prediction
                ==
                1
            )
        ].sum()
    )


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )

        if
        (
            tp
            +
            fn
        )
        >
        0

        else
        np.nan
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )

        if
        (
            tn
            +
            fp
        )
        >
        0

        else
        np.nan
    )


    ppv = (
        tp
        /
        (
            tp
            +
            fp
        )

        if
        (
            tp
            +
            fp
        )
        >
        0

        else
        np.nan
    )


    npv = (
        tn
        /
        (
            tn
            +
            fn
        )

        if
        (
            tn
            +
            fn
        )
        >
        0

        else
        np.nan
    )


    f1 = (
        2.0
        *
        tp
        /
        (
            2.0 * tp
            +
            fp
            +
            fn
        )

        if
        (
            2.0 * tp
            +
            fp
            +
            fn
        )
        >
        0

        else
        np.nan
    )


    accuracy = (
        (
            tp
            +
            tn
        )
        /
        (
            tp
            +
            tn
            +
            fp
            +
            fn
        )
    )


    balanced_accuracy = (
        (
            sensitivity
            +
            specificity
        )
        /
        2.0
    )


    return {
        "Sensitivity":
            float(
                sensitivity
            ),

        "Specificity":
            float(
                specificity
            ),

        "Balanced_Accuracy":
            float(
                balanced_accuracy
            ),

        "PPV":
            float(
                ppv
            ),

        "NPV":
            float(
                npv
            ),

        "F1":
            float(
                f1
            ),

        "Accuracy":
            float(
                accuracy
            ),
    }


# ============================================================
# 15. MULTICLASS METRICS
# ============================================================

def multiclass_values(
    true_class,
    probability,
    predicted_class,
    w,
):

    true_class = np.asarray(
        true_class,
        dtype=int,
    )


    probability = np.asarray(
        probability,
        dtype=float,
    )


    predicted_class = np.asarray(
        predicted_class,
        dtype=int,
    )


    w = np.asarray(
        w,
        dtype=float,
    )


    total_weight = float(
        np.sum(
            w
        )
    )


    if total_weight <= 0:

        raise RuntimeError(
            "Zero total bootstrap weight."
        )


    clipped = np.clip(
        probability,
        1e-15,
        1.0,
    )


    true_probability = clipped[
        np.arange(
            len(
                true_class
            )
        ),
        true_class,
    ]


    log_loss = float(
        -np.sum(
            w
            *
            np.log(
                true_probability
            )
        )
        /
        total_weight
    )


    one_hot = np.eye(
        3
    )[
        true_class
    ]


    brier = float(
        np.sum(
            w
            *
            np.sum(
                (
                    probability
                    -
                    one_hot
                ) ** 2,
                axis=1,
            )
        )
        /
        total_weight
    )


    class_probability_metrics = {}


    ap_values = []

    auc_values = []


    for class_id in [
        0,
        1,
        2,
    ]:

        y_class = (
            true_class
            ==
            class_id
        ).astype(int)


        class_metrics = (
            binary_probability_values(
                y_class,
                probability[
                    :,
                    class_id
                ],
                w,
            )
        )


        class_probability_metrics[
            class_id
        ] = class_metrics


        ap_values.append(
            class_metrics[
                "AP"
            ]
        )


        auc_values.append(
            class_metrics[
                "ROC_AUC"
            ]
        )


    # weighted confusion matrix

    confusion = np.zeros(
        (
            3,
            3,
        ),
        dtype=float,
    )


    np.add.at(
        confusion,
        (
            true_class,
            predicted_class,
        ),
        w,
    )


    class_threshold_metrics = {}


    recalls = []

    f1_values = []


    for class_id in [
        0,
        1,
        2,
    ]:

        tp = float(
            confusion[
                class_id,
                class_id
            ]
        )


        fn = float(
            confusion[
                class_id,
                :
            ].sum()
            -
            tp
        )


        fp = float(
            confusion[
                :,
                class_id
            ].sum()
            -
            tp
        )


        precision = (
            tp
            /
            (
                tp
                +
                fp
            )

            if
            (
                tp
                +
                fp
            )
            >
            0

            else
            np.nan
        )


        recall = (
            tp
            /
            (
                tp
                +
                fn
            )

            if
            (
                tp
                +
                fn
            )
            >
            0

            else
            np.nan
        )


        f1 = (
            2.0
            *
            precision
            *
            recall
            /
            (
                precision
                +
                recall
            )

            if
            (
                precision
                +
                recall
            )
            >
            0

            else
            np.nan
        )


        class_threshold_metrics[
            class_id
        ] = {
            "Precision":
                float(
                    precision
                ),

            "Recall":
                float(
                    recall
                ),

            "F1":
                float(
                    f1
                ),
        }


        recalls.append(
            recall
        )


        f1_values.append(
            f1
        )


    accuracy = float(
        np.trace(
            confusion
        )
        /
        confusion.sum()
    )


    return {
        "Log_Loss":
            log_loss,

        "Brier":
            brier,

        "Macro_AP":
            float(
                np.mean(
                    ap_values
                )
            ),

        "Macro_ROC_AUC":
            float(
                np.mean(
                    auc_values
                )
            ),

        "Accuracy":
            accuracy,

        "Balanced_Accuracy":
            float(
                np.mean(
                    recalls
                )
            ),

        "Macro_F1":
            float(
                np.mean(
                    f1_values
                )
            ),

        "class_probability_metrics":
            class_probability_metrics,

        "class_threshold_metrics":
            class_threshold_metrics,
    }


# ============================================================
# 16. ONE COMPLETE EVALUATION
# ============================================================

CLASS_PREFIX = {
    0:
        "Class0_No_ANC",

    1:
        "Class1_1_7",

    2:
        "Class2_GE8",
}


def evaluate_locked_predictions(
    replicate_weight,
):

    replicate_weight = np.asarray(
        replicate_weight,
        dtype=float,
    )


    result = {}


    # --------------------------------------------------------
    # Stage 1
    # --------------------------------------------------------

    stage1_prob = (
        binary_probability_values(
            stage1_y,
            stage1_p,
            replicate_weight,
        )
    )


    stage1_threshold = (
        binary_threshold_values(
            stage1_y,
            stage1_prediction,
            replicate_weight,
        )
    )


    for metric, value in (
        stage1_prob.items()
    ):

        result[
            f"Stage1.{metric}"
        ] = value


    for metric, value in (
        stage1_threshold.items()
    ):

        result[
            f"Stage1_Threshold.{metric}"
        ] = value


    # --------------------------------------------------------
    # Stage 2
    # --------------------------------------------------------

    stage2_weight = (
        replicate_weight[
            stage2_mask
        ]
    )


    stage2_prob = (
        binary_probability_values(
            stage2_y,
            stage2_p,
            stage2_weight,
        )
    )


    stage2_threshold = (
        binary_threshold_values(
            stage2_y,
            stage2_prediction,
            stage2_weight,
        )
    )


    for metric, value in (
        stage2_prob.items()
    ):

        result[
            f"Stage2.{metric}"
        ] = value


    for metric, value in (
        stage2_threshold.items()
    ):

        result[
            f"Stage2_Threshold.{metric}"
        ] = value


    # --------------------------------------------------------
    # Multiclass
    # --------------------------------------------------------

    multiclass = multiclass_values(
        true_class=true_class,
        probability=multiclass_probability,
        predicted_class=locked_predicted_class,
        w=replicate_weight,
    )


    for metric in [
        "Log_Loss",
        "Brier",
        "Macro_AP",
        "Macro_ROC_AUC",
        "Accuracy",
        "Balanced_Accuracy",
        "Macro_F1",
    ]:

        result[
            f"Multiclass.{metric}"
        ] = multiclass[
            metric
        ]


    # --------------------------------------------------------
    # Multiclass classwise OVR probability metrics
    # --------------------------------------------------------

    for class_id in [
        0,
        1,
        2,
    ]:

        prefix = (
            CLASS_PREFIX[
                class_id
            ]
        )


        probability_metrics = (
            multiclass[
                "class_probability_metrics"
            ][
                class_id
            ]
        )


        threshold_metrics = (
            multiclass[
                "class_threshold_metrics"
            ][
                class_id
            ]
        )


        for metric, value in (
            probability_metrics.items()
        ):

            result[
                f"{prefix}.{metric}"
            ] = value


        for metric, value in (
            threshold_metrics.items()
        ):

            result[
                f"{prefix}_Threshold.{metric}"
            ] = value


    return result


# ============================================================
# 17. RE-CALCULATE ORIGINAL POINT ESTIMATES
# ============================================================

point_metrics = evaluate_locked_predictions(
    base_weight
)


point_metric_series = pd.Series(
    point_metrics,
    name="Point_Estimate",
    dtype=float,
)


if not np.isfinite(
    point_metric_series
).all():

    raise RuntimeError(
        "Point-estimate recalculation produced non-finite values."
    )


# ============================================================
# 18. VERIFY AGAINST STEP-20 OUTPUTS
# ============================================================

step20_binary = pd.read_csv(
    STEP20_BINARY_PROBABILITY_FILE
)


step20_binary_threshold = pd.read_csv(
    STEP20_BINARY_THRESHOLD_FILE
)


step20_multiclass = pd.read_csv(
    STEP20_MULTICLASS_FILE
)


step20_classwise = pd.read_csv(
    STEP20_CLASSWISE_FILE
)


step20_multiclass_class = pd.read_csv(
    STEP20_MULTICLASS_CLASS_FILE
)


step20_multiclass_threshold = pd.read_csv(
    STEP20_MULTICLASS_THRESHOLD_FILE
)


expected_point_metrics = {}


# Stage 1 / Stage 2 probability metrics

for index, prefix in [
    (
        0,
        "Stage1",
    ),
    (
        1,
        "Stage2",
    ),
]:

    row = step20_binary.iloc[
        index
    ]


    expected_point_metrics.update(
        {
            f"{prefix}.AP":
                row[
                    "Weighted_Average_Precision"
                ],

            f"{prefix}.ROC_AUC":
                row[
                    "Weighted_ROC_AUC"
                ],

            f"{prefix}.Log_Loss":
                row[
                    "Weighted_Log_Loss"
                ],

            f"{prefix}.Brier":
                row[
                    "Weighted_Brier_Score"
                ],

            f"{prefix}.CITL":
                row[
                    "Calibration_in_the_Large"
                ],

            f"{prefix}.Recalibration_Intercept":
                row[
                    "Recalibration_Intercept"
                ],

            f"{prefix}.Calibration_Slope":
                row[
                    "Calibration_Slope"
                ],

            f"{prefix}.O_E":
                row[
                    "Observed_to_Expected_Ratio"
                ],
        }
    )


# Stage threshold metrics

for index, prefix in [
    (
        0,
        "Stage1_Threshold",
    ),
    (
        1,
        "Stage2_Threshold",
    ),
]:

    row = step20_binary_threshold.iloc[
        index
    ]


    expected_point_metrics.update(
        {
            f"{prefix}.Sensitivity":
                row[
                    "Sensitivity"
                ],

            f"{prefix}.Specificity":
                row[
                    "Specificity"
                ],

            f"{prefix}.Balanced_Accuracy":
                row[
                    "Balanced_Accuracy"
                ],

            f"{prefix}.PPV":
                row[
                    "PPV"
                ],

            f"{prefix}.NPV":
                row[
                    "NPV"
                ],

            f"{prefix}.F1":
                row[
                    "F1"
                ],

            f"{prefix}.Accuracy":
                row[
                    "Accuracy"
                ],
        }
    )


# Multiclass summary

mc_row = step20_multiclass.iloc[
    0
]


expected_point_metrics.update(
    {
        "Multiclass.Log_Loss":
            mc_row[
                "Weighted_Multiclass_Log_Loss"
            ],

        "Multiclass.Brier":
            mc_row[
                "Weighted_Multiclass_Brier_Score"
            ],

        "Multiclass.Macro_AP":
            mc_row[
                "Macro_OVR_Average_Precision"
            ],

        "Multiclass.Macro_ROC_AUC":
            mc_row[
                "Macro_OVR_ROC_AUC"
            ],
    }
)


mc_threshold_row = (
    step20_multiclass_threshold.iloc[
        0
    ]
)


expected_point_metrics.update(
    {
        "Multiclass.Accuracy":
            mc_threshold_row[
                "Weighted_Overall_Accuracy"
            ],

        "Multiclass.Balanced_Accuracy":
            mc_threshold_row[
                "Macro_Recall_Balanced_Accuracy"
            ],

        "Multiclass.Macro_F1":
            mc_threshold_row[
                "Macro_F1"
            ],
    }
)


# Classwise probability metrics

for class_id in [
    0,
    1,
    2,
]:

    prefix = (
        CLASS_PREFIX[
            class_id
        ]
    )


    row = (
        step20_classwise.loc[
            step20_classwise[
                "Class_ID"
            ]
            ==
            class_id
        ]
        .iloc[0]
    )


    expected_point_metrics.update(
        {
            f"{prefix}.AP":
                row[
                    "Weighted_Average_Precision"
                ],

            f"{prefix}.ROC_AUC":
                row[
                    "Weighted_ROC_AUC"
                ],

            f"{prefix}.Log_Loss":
                row[
                    "Weighted_Log_Loss"
                ],

            f"{prefix}.Brier":
                row[
                    "Weighted_Brier_Score"
                ],

            f"{prefix}.CITL":
                row[
                    "Calibration_in_the_Large"
                ],

            f"{prefix}.Recalibration_Intercept":
                row[
                    "Recalibration_Intercept"
                ],

            f"{prefix}.Calibration_Slope":
                row[
                    "Calibration_Slope"
                ],

            f"{prefix}.O_E":
                row[
                    "Observed_to_Expected_Ratio"
                ],
        }
    )


    threshold_row = (
        step20_multiclass_class.loc[
            step20_multiclass_class[
                "Class_ID"
            ]
            ==
            class_id
        ]
        .iloc[0]
    )


    expected_point_metrics.update(
        {
            f"{prefix}_Threshold.Precision":
                threshold_row[
                    "Precision"
                ],

            f"{prefix}_Threshold.Recall":
                threshold_row[
                    "Recall"
                ],

            f"{prefix}_Threshold.F1":
                threshold_row[
                    "F1"
                ],
        }
    )


point_match_rows = []


for metric, expected_value in (
    expected_point_metrics.items()
):

    observed_value = (
        point_metrics[
            metric
        ]
    )


    absolute_difference = abs(
        float(
            observed_value
        )
        -
        float(
            expected_value
        )
    )


    point_match_rows.append(
        {
            "Metric":
                metric,

            "Step20_Value":
                float(
                    expected_value
                ),

            "Step21_Recalculated_Value":
                float(
                    observed_value
                ),

            "Absolute_Difference":
                absolute_difference,

            "PASS":
                absolute_difference
                <=
                1e-6,
        }
    )


point_match_qc = pd.DataFrame(
    point_match_rows
)


if not point_match_qc[
    "PASS"
].all():

    failed = point_match_qc.loc[
        ~point_match_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "Step-20 point-estimate reproduction failed:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 19. GENERATE RAO-WU PSU BOOTSTRAP FACTORS
# ============================================================

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)


n_clusters = len(
    cluster_table
)


bootstrap_cluster_factor = np.zeros(
    (
        BOOTSTRAP_REPLICATES,
        n_clusters,
    ),
    dtype=np.float64,
)


for stratum_value, stratum_frame in (
    cluster_table
    .groupby(
        "STRATUM",
        sort=True,
    )
):

    cluster_indices = (
        stratum_frame[
            "__CLUSTER_INDEX"
        ]
        .to_numpy(
            dtype=int
        )
    )


    n_h = len(
        cluster_indices
    )


    if n_h <= 1:

        raise RuntimeError(
            f"Stratum {stratum_value} has <=1 PSU."
        )


    probability = np.full(
        n_h,
        1.0
        /
        n_h,
        dtype=float,
    )


    # Draw n_h - 1 PSUs with replacement.
    multiplicity = rng.multinomial(
        n=n_h - 1,
        pvals=probability,
        size=BOOTSTRAP_REPLICATES,
    )


    # Rao-Wu rescaling.
    factor = (
        multiplicity
        *
        (
            n_h
            /
            (
                n_h
                -
                1
            )
        )
    )


    bootstrap_cluster_factor[
        :,
        cluster_indices
    ] = factor


# ============================================================
# 20. BOOTSTRAP DESIGN QC
# ============================================================

design_qc_rows = []


for stratum_value, stratum_frame in (
    cluster_table
    .groupby(
        "STRATUM",
        sort=True,
    )
):

    cluster_indices = (
        stratum_frame[
            "__CLUSTER_INDEX"
        ]
        .to_numpy(
            dtype=int
        )
    )


    n_h = len(
        cluster_indices
    )


    factor_sum = (
        bootstrap_cluster_factor[
            :,
            cluster_indices
        ]
        .sum(
            axis=1
        )
    )


    design_qc_rows.append(
        {
            "STRATUM":
                stratum_value,

            "Original_PSU_Count":
                n_h,

            "Expected_Replicate_Factor_Sum":
                float(
                    n_h
                ),

            "Minimum_Replicate_Factor_Sum":
                float(
                    factor_sum.min()
                ),

            "Maximum_Replicate_Factor_Sum":
                float(
                    factor_sum.max()
                ),

            "PASS":
                bool(
                    np.allclose(
                        factor_sum,
                        float(
                            n_h
                        ),
                        rtol=0.0,
                        atol=1e-10,
                    )
                ),
        }
    )


bootstrap_design_qc = pd.DataFrame(
    design_qc_rows
)


if not bootstrap_design_qc[
    "PASS"
].all():

    raise RuntimeError(
        "Rao-Wu bootstrap factor QC failed."
    )


active_psu_counts = (
    bootstrap_cluster_factor
    >
    0
).sum(
    axis=1
)


# ============================================================
# 21. RUN 2000 DESIGN BOOTSTRAP REPLICATES
# ============================================================

bootstrap_rows = []


print(
    "\n"
    + "=" * 78
)


print(
    "STEP 21 — STRATIFIED PSU BOOTSTRAP"
)


print(
    "=" * 78
)


print(
    f"Replicates : {BOOTSTRAP_REPLICATES}"
)


print(
    f"Seed       : {BOOTSTRAP_SEED}"
)


print(
    f"Strata     : {len(stratum_cluster_counts)}"
)


print(
    f"PSUs       : {len(cluster_table)}"
)


for bootstrap_id in range(
    BOOTSTRAP_REPLICATES
):

    cluster_factor = (
        bootstrap_cluster_factor[
            bootstrap_id,
            :
        ]
    )


    replicate_weight = (
        base_weight
        *
        cluster_factor[
            row_cluster_index
        ]
    )


    metrics = evaluate_locked_predictions(
        replicate_weight
    )


    metrics[
        "Bootstrap_ID"
    ] = (
        bootstrap_id
        +
        1
    )


    bootstrap_rows.append(
        metrics
    )


    if (
        (
            bootstrap_id
            +
            1
        )
        %
        250
        ==
        0
    ):

        print(
            f"Completed "
            f"{bootstrap_id + 1}/"
            f"{BOOTSTRAP_REPLICATES} replicates"
        )


bootstrap_distribution = pd.DataFrame(
    bootstrap_rows
)


bootstrap_distribution = (
    bootstrap_distribution[
        [
            "Bootstrap_ID",
            *
            [
                column
                for column
                in bootstrap_distribution.columns
                if column
                !=
                "Bootstrap_ID"
            ],
        ]
    ]
)


# ============================================================
# 22. VALIDITY CHECK
# ============================================================

metric_columns = [
    column
    for column
    in bootstrap_distribution.columns
    if column
    !=
    "Bootstrap_ID"
]


validity_table = pd.DataFrame(
    {
        "Metric":
            metric_columns,

        "Valid_Replicates":
            [
                int(
                    np.isfinite(
                        bootstrap_distribution[
                            column
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    ).sum()
                )

                for column
                in metric_columns
            ],
    }
)


validity_table[
    "Total_Replicates"
] = BOOTSTRAP_REPLICATES


validity_table[
    "PASS"
] = (
    validity_table[
        "Valid_Replicates"
    ]
    ==
    BOOTSTRAP_REPLICATES
)


if not validity_table[
    "PASS"
].all():

    failed = validity_table.loc[
        ~validity_table[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "One or more bootstrap metrics were not valid "
        "in all 2000 design replicates:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 23. 95% PERCENTILE CONFIDENCE INTERVALS
# ============================================================

ci_rows = []


for metric in metric_columns:

    values = (
        bootstrap_distribution[
            metric
        ]
        .to_numpy(
            dtype=float
        )
    )


    point_estimate = float(
        point_metrics[
            metric
        ]
    )


    bootstrap_mean = float(
        np.mean(
            values
        )
    )


    bootstrap_se = float(
        np.std(
            values,
            ddof=1,
        )
    )


    lower = float(
        np.quantile(
            values,
            CI_LOWER_QUANTILE,
        )
    )


    upper = float(
        np.quantile(
            values,
            CI_UPPER_QUANTILE,
        )
    )


    analysis_name, metric_name = (
        metric.split(
            ".",
            1,
        )
    )


    ci_rows.append(
        {
            "Analysis":
                analysis_name,

            "Metric":
                metric_name,

            "Point_Estimate":
                point_estimate,

            "Bootstrap_Mean":
                bootstrap_mean,

            "Bootstrap_Bias":
                bootstrap_mean
                -
                point_estimate,

            "Bootstrap_SE":
                bootstrap_se,

            "CI_Lower_95":
                lower,

            "CI_Upper_95":
                upper,

            "Valid_Replicates":
                BOOTSTRAP_REPLICATES,

            "CI_Method":
                "Rao-Wu rescaled PSU bootstrap percentile",

            "Bootstrap_Replicates":
                BOOTSTRAP_REPLICATES,
        }
    )


bootstrap_ci = pd.DataFrame(
    ci_rows
)


# ============================================================
# 24. REPORTING ROLE
# ============================================================

def metric_role(
    analysis,
    metric,
):

    if (
        analysis
        in [
            "Stage1",
            "Stage2",
        ]
    ):

        if metric == "AP":

            return (
                "Primary discrimination"
            )


        if metric == "ROC_AUC":

            return (
                "Secondary discrimination"
            )


        if metric == "Log_Loss":

            return (
                "Primary probabilistic performance"
            )


        if metric == "Brier":

            return (
                "Secondary probabilistic performance"
            )


        if metric in [
            "CITL",
            "Calibration_Slope",
            "O_E",
            "Recalibration_Intercept",
        ]:

            return (
                "Calibration"
            )


    if analysis == "Multiclass":

        if metric == "Log_Loss":

            return (
                "Primary multiclass probabilistic performance"
            )


        if metric == "Brier":

            return (
                "Secondary multiclass probabilistic performance"
            )


        return (
            "Secondary multiclass summary"
        )


    if (
        "Threshold"
        in analysis
    ):

        return (
            "Secondary threshold-dependent performance"
        )


    if analysis.startswith(
        "Class"
    ):

        return (
            "Secondary classwise performance"
        )


    return (
        "Secondary"
    )


bootstrap_ci[
    "Role"
] = [
    metric_role(
        analysis=row[
            "Analysis"
        ],
        metric=row[
            "Metric"
        ],
    )

    for _, row
    in bootstrap_ci.iterrows()
]


# ============================================================
# 25. PRIMARY REPORTING TABLE
# ============================================================

primary_reporting_mask = (
    (
        bootstrap_ci[
            "Analysis"
        ]
        .isin(
            [
                "Stage1",
                "Stage2",
            ]
        )
        &
        bootstrap_ci[
            "Metric"
        ]
        .isin(
            [
                "AP",
                "ROC_AUC",
                "Log_Loss",
                "Brier",
                "CITL",
                "Calibration_Slope",
                "O_E",
            ]
        )
    )

    |

    (
        bootstrap_ci[
            "Analysis"
        ]
        ==
        "Multiclass"
    )
    &
    bootstrap_ci[
        "Metric"
    ]
    .isin(
        [
            "Log_Loss",
            "Brier",
            "Macro_AP",
            "Macro_ROC_AUC",
        ]
    )
)


primary_ci_table = (
    bootstrap_ci.loc[
        primary_reporting_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ============================================================
# 26. BOOTSTRAP DESIGN SUMMARY
# ============================================================

bootstrap_design_summary = pd.DataFrame(
    [
        {
            "Bootstrap_Method":
                "Rao-Wu rescaled stratified PSU bootstrap",

            "Replicates":
                BOOTSTRAP_REPLICATES,

            "Seed":
                BOOTSTRAP_SEED,

            "Survey_Strata":
                int(
                    len(
                        stratum_cluster_counts
                    )
                ),

            "Analytic_PSUs":
                int(
                    len(
                        cluster_table
                    )
                ),

            "Minimum_PSUs_per_Stratum":
                int(
                    stratum_cluster_counts[
                        "PSU_Count"
                    ].min()
                ),

            "Maximum_PSUs_per_Stratum":
                int(
                    stratum_cluster_counts[
                        "PSU_Count"
                    ].max()
                ),

            "Mean_Active_PSUs_per_Replicate":
                float(
                    active_psu_counts.mean()
                ),

            "Minimum_Active_PSUs":
                int(
                    active_psu_counts.min()
                ),

            "Maximum_Active_PSUs":
                int(
                    active_psu_counts.max()
                ),

            "Resample_Unit":
                "PSU within survey stratum",

            "Replicate_Weight":
                (
                    "Original DHS weight x "
                    "[n_h/(n_h-1)] x PSU multiplicity"
                ),

            "Model_Refitting":
                False,

            "Threshold_Reoptimization":
                False,

            "CI_Type":
                "Percentile",

            "CI_Level":
                CI_LEVEL,

            "Inference_Interpretation":
                (
                    "Survey-design uncertainty conditional on "
                    "locked cross-fitted predictions"
                ),
        }
    ]
)


# ============================================================
# 27. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                (
                    sha256_file(
                        PRIMARY_FILE
                    )
                    ==
                    EXPECTED_PRIMARY_SHA256
                ),
        },

        {
            "Check":
                "Locked OOF hash unchanged",

            "PASS":
                (
                    sha256_file(
                        FINAL_OOF_FILE
                    )
                    ==
                    EXPECTED_LOCKED_OOF_SHA256
                ),
        },

        {
            "Check":
                "Step 20 point estimates reproduced",

            "PASS":
                bool(
                    point_match_qc[
                        "PASS"
                    ].all()
                ),
        },

        {
            "Check":
                "Analytic PSU count = 672",

            "PASS":
                (
                    len(
                        cluster_table
                    )
                    ==
                    672
                ),
        },

        {
            "Check":
                "Survey strata count = 16",

            "PASS":
                (
                    len(
                        stratum_cluster_counts
                    )
                    ==
                    16
                ),
        },

        {
            "Check":
                "No singleton bootstrap stratum",

            "PASS":
                bool(
                    (
                        stratum_cluster_counts[
                            "PSU_Count"
                        ]
                        >
                        1
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Rao-Wu replicate factor sums correct",

            "PASS":
                bool(
                    bootstrap_design_qc[
                        "PASS"
                    ].all()
                ),
        },

        {
            "Check":
                "Exactly 2000 bootstrap replicates",

            "PASS":
                (
                    len(
                        bootstrap_distribution
                    )
                    ==
                    BOOTSTRAP_REPLICATES
                ),
        },

        {
            "Check":
                "All bootstrap metric values finite",

            "PASS":
                bool(
                    validity_table[
                        "PASS"
                    ].all()
                ),
        },

        {
            "Check":
                "All 95% confidence intervals finite",

            "PASS":
                bool(
                    np.isfinite(
                        bootstrap_ci[
                            [
                                "CI_Lower_95",
                                "CI_Upper_95",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "No participant-level resampling",

            "PASS":
                True,
        },

        {
            "Check":
                "PSU resampling performed within strata",

            "PASS":
                True,
        },

        {
            "Check":
                "Original DHS weights retained and rescaled",

            "PASS":
                True,
        },

        {
            "Check":
                "No model refitting",

            "PASS":
                True,
        },

        {
            "Check":
                "No model-family reselection",

            "PASS":
                True,
        },

        {
            "Check":
                "No hyperparameter retuning",

            "PASS":
                True,
        },

        {
            "Check":
                "No probability recalibration",

            "PASS":
                True,
        },

        {
            "Check":
                "No threshold re-optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "Bootstrap inference explicitly conditional",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "STEP 21 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 28. SAVE OUTPUTS
# ============================================================

BOOTSTRAP_DISTRIBUTION_FILE = (
    DIRS["bootstrap"]
    / "STEP21_PRIMARY_bootstrap_metric_distribution.csv"
)


BOOTSTRAP_CI_FILE = (
    DIRS["bootstrap"]
    / "STEP21_PRIMARY_all_metric_confidence_intervals.csv"
)


PRIMARY_CI_FILE = (
    DIRS["tables"]
    / "STEP21_PRIMARY_main_performance_with_95CI.csv"
)


DESIGN_SUMMARY_FILE = (
    DIRS["bootstrap"]
    / "STEP21_PRIMARY_bootstrap_design_summary.csv"
)


DESIGN_QC_FILE = (
    DIRS["qc"]
    / "STEP21_PRIMARY_bootstrap_design_qc.csv"
)


POINT_MATCH_FILE = (
    DIRS["qc"]
    / "STEP21_PRIMARY_step20_point_estimate_reproduction_qc.csv"
)


VALIDITY_FILE = (
    DIRS["qc"]
    / "STEP21_PRIMARY_bootstrap_metric_validity.csv"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP21_PRIMARY_hard_qc.csv"
)


bootstrap_distribution.to_csv(
    BOOTSTRAP_DISTRIBUTION_FILE,
    index=False,
)


bootstrap_ci.to_csv(
    BOOTSTRAP_CI_FILE,
    index=False,
)


primary_ci_table.to_csv(
    PRIMARY_CI_FILE,
    index=False,
)


bootstrap_design_summary.to_csv(
    DESIGN_SUMMARY_FILE,
    index=False,
)


bootstrap_design_qc.to_csv(
    DESIGN_QC_FILE,
    index=False,
)


point_match_qc.to_csv(
    POINT_MATCH_FILE,
    index=False,
)


validity_table.to_csv(
    VALIDITY_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


# ============================================================
# 29. MANIFEST
# ============================================================

manifest = {
    "step":
        "STEP 21",

    "status":
        "PRIMARY_SURVEY_DESIGN_BOOTSTRAP_UNCERTAINTY_COMPLETE",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "bootstrap_method":
        "Rao-Wu rescaled stratified PSU bootstrap",

    "bootstrap_replicates":
        BOOTSTRAP_REPLICATES,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "survey_strata":
        16,

    "analytic_psus":
        672,

    "resampling_unit":
        "PSU within survey stratum",

    "psus_drawn_per_stratum":
        "n_h - 1 with replacement",

    "replicate_weight_formula":
        (
            "original DHS survey weight x "
            "n_h/(n_h-1) x PSU bootstrap multiplicity"
        ),

    "ci_level":
        CI_LEVEL,

    "ci_method":
        "percentile",

    "model_refitting":
        False,

    "hyperparameter_tuning":
        False,

    "model_reselection":
        False,

    "probability_recalibration":
        False,

    "threshold_reoptimization":
        False,

    "interpretation":
        (
            "Survey-design uncertainty conditional on "
            "locked cross-fitted outer-OOF predictions"
        ),

    "locked_oof_sha256":
        sha256_file(
            FINAL_OOF_FILE
        ),

    "bootstrap_distribution_sha256":
        sha256_file(
            BOOTSTRAP_DISTRIBUTION_FILE
        ),

    "bootstrap_ci_sha256":
        sha256_file(
            BOOTSTRAP_CI_FILE
        ),

    "primary_ci_sha256":
        sha256_file(
            PRIMARY_CI_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP21_PRIMARY_bootstrap_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 30. FINAL IMMUTABILITY CHECK
# ============================================================

if (
    sha256_file(
        FINAL_OOF_FILE
    )
    !=
    EXPECTED_LOCKED_OOF_SHA256
):

    raise RuntimeError(
        "Locked OOF predictions changed during Step 21."
    )


# ============================================================
# 31. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 21 — SURVEY-DESIGN BOOTSTRAP UNCERTAINTY COMPLETE"
)


print(
    "=" * 78
)


print(
    "\nBOOTSTRAP DESIGN:"
)


display(
    bootstrap_design_summary
)


print(
    "\nSTEP-20 POINT-ESTIMATE REPRODUCTION QC:"
)


display(
    point_match_qc
)


print(
    "\nPRIMARY PERFORMANCE WITH 95% CI:"
)


display(
    primary_ci_table
)


print(
    "\nALL PERFORMANCE CONFIDENCE INTERVALS:"
)


display(
    bootstrap_ci
)


print(
    "\nBOOTSTRAP METRIC VALIDITY:"
)


display(
    validity_table
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nIMPORTANT:"
)


print(
    "- 2000 Rao-Wu rescaled stratified PSU bootstrap replicates were used."
)


print(
    "- PSUs, not individual participants, were resampled."
)


print(
    "- PSU resampling occurred separately within each survey stratum."
)


print(
    "- Original DHS sampling weights were retained and bootstrap-rescaled."
)


print(
    "- Locked cross-fitted predictions were never changed."
)


print(
    "- No model was refitted or retuned."
)


print(
    "- No threshold was re-optimized."
)


print(
    "- Intervals quantify survey-design uncertainty conditional on locked OOF predictions."
)


print(
    "\nINTEGRITY:"
)


print(
    f"Locked OOF : {sha256_file(FINAL_OOF_FILE)}"
)


print(
    "=" * 78
)


print(
    "STATUS: STEP 21 PRIMARY BOOTSTRAP CI PASS"
)


print(
    "=" * 78
)


STEP 21 — STRATIFIED PSU BOOTSTRAP
Replicates : 2000
Seed       : 2026
Strata     : 16
PSUs       : 672
Completed 250/2000 replicates
Completed 500/2000 replicates
Completed 750/2000 replicates
Completed 1000/2000 replicates
Completed 1250/2000 replicates
Completed 1500/2000 replicates
Completed 1750/2000 replicates
Completed 2000/2000 replicates

STEP 21 — SURVEY-DESIGN BOOTSTRAP UNCERTAINTY COMPLETE

BOOTSTRAP DESIGN:


,Bootstrap_Method,Replicates,Seed,Survey_Strata,Analytic_PSUs,Minimum_PSUs_per_Stratum,Maximum_PSUs_per_Stratum,Mean_Active_PSUs_per_Replicate,Minimum_Active_PSUs,Maximum_Active_PSUs,Resample_Unit,Replicate_Weight,Model_Refitting,Threshold_Reoptimization,CI_Type,CI_Level,Inference_Interpretation
0,Rao-Wu rescaled stratified PSU bootstrap,2000,2026,16,672,17,61,421.664,398,444,PSU within survey stratum,Original DHS weight x [n_h/(n_h-1)] x PSU mult...,False,False,Percentile,0.95,Survey-design uncertainty conditional on locke...



STEP-20 POINT-ESTIMATE REPRODUCTION QC:


,Metric,Step20_Value,Step21_Recalculated_Value,Absolute_Difference,PASS
0,Stage1.AP,0.235306,0.235306,5.551115e-17,True
1,Stage1.ROC_AUC,0.778288,0.778288,0.000000e+00,True
2,Stage1.Log_Loss,0.239733,0.239733,0.000000e+00,True
3,Stage1.Brier,0.066997,0.066997,8.326673e-17,True
4,Stage1.CITL,0.002330,0.002330,5.785303e-16,True
...,...,...,...,...,...
65,Class2_GE8.Calibration_Slope,0.940568,0.940568,1.094217e-08,True
66,Class2_GE8.O_E,1.066783,1.066783,0.000000e+00,True
67,Class2_GE8_Threshold.Precision,0.121679,0.121679,2.498002e-16,True
68,Class2_GE8_Threshold.Recall,0.593151,0.593151,1.110223e-16,True



PRIMARY PERFORMANCE WITH 95% CI:


,Analysis,Metric,Point_Estimate,Bootstrap_Mean,Bootstrap_Bias,Bootstrap_SE,CI_Lower_95,CI_Upper_95,Valid_Replicates,CI_Method,Bootstrap_Replicates,Role
0,Stage1,AP,0.235306,0.237493,0.002187,0.022720,0.194391,0.284181,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Primary discrimination
1,Stage1,ROC_AUC,0.778288,0.778072,-0.000216,0.012790,0.753040,0.802921,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Secondary discrimination
2,Stage1,Log_Loss,0.239733,0.239990,0.000257,0.011970,0.216949,0.263359,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Primary probabilistic performance
3,Stage1,Brier,0.066997,0.067082,0.000086,0.004146,0.059212,0.075314,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Secondary probabilistic performance
4,Stage1,CITL,0.002330,0.002513,0.000183,0.072904,-0.145873,0.141687,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Calibration
5,Stage1,Calibration_Slope,1.026574,1.027193,0.000619,0.065841,0.902065,1.159507,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Calibration
6,Stage1,O_E,1.001958,1.003649,0.001692,0.061268,0.884557,1.124849,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Calibration
7,Stage2,AP,0.161610,0.164724,0.003113,0.022909,0.122849,0.213053,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Primary discrimination
8,Stage2,ROC_AUC,0.735076,0.734399,-0.000678,0.019508,0.694635,0.771842,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Secondary discrimination
9,Stage2,Log_Loss,0.193557,0.193507,-0.000051,0.011717,0.171151,0.217497,2000,Rao-Wu rescaled PSU bootstrap percentile,2000,Primary probabilistic performance


... (3 outputs left out)

,Metric,Valid_Replicates,Total_Replicates,PASS
0,Stage1.AP,2000,2000,True
1,Stage1.ROC_AUC,2000,2000,True
2,Stage1.Log_Loss,2000,2000,True
3,Stage1.Brier,2000,2000,True
4,Stage1.CITL,2000,2000,True
...,...,...,...,...
65,Class2_GE8.Calibration_Slope,2000,2000,True
66,Class2_GE8.O_E,2000,2000,True
67,Class2_GE8_Threshold.Precision,2000,2000,True
68,Class2_GE8_Threshold.Recall,2000,2000,True



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Locked OOF hash unchanged,True
2,Step 20 point estimates reproduced,True
3,Analytic PSU count = 672,True
4,Survey strata count = 16,True
5,No singleton bootstrap stratum,True
6,Rao-Wu replicate factor sums correct,True
7,Exactly 2000 bootstrap replicates,True
8,All bootstrap metric values finite,True
9,All 95% confidence intervals finite,True



IMPORTANT:
- 2000 Rao-Wu rescaled stratified PSU bootstrap replicates were used.
- PSUs, not individual participants, were resampled.
- PSU resampling occurred separately within each survey stratum.
- Original DHS sampling weights were retained and bootstrap-rescaled.
- Locked cross-fitted predictions were never changed.
- No model was refitted or retuned.
- No threshold was re-optimized.
- Intervals quantify survey-design uncertainty conditional on locked OOF predictions.

INTEGRITY:
Locked OOF : 6c4f4e21b09a54c286521682e9fb8ee13eda36ed94196d350fbb0622733a99bc
STATUS: STEP 21 PRIMARY BOOTSTRAP CI PASS


In [7]:
# ============================================================
# STEP 22 — GEOGRAPHIC INTERNAL-EXTERNAL VALIDATION
#           METHODOLOGY / EXECUTION PROTOCOL FREEZE
# ============================================================
#
# NO MODEL IS FIT IN THIS STEP.
# NO GEOGRAPHIC PERFORMANCE IS INSPECTED.
#
# PURPOSE
# -------
# Prespecify an 8-division internal-external cross-validation
# (IECV) transportability stress test.
#
#
# EACH IECV ITERATION
# -------------------
# 1 Bangladesh division = completely held-out geographic test set
# Remaining 7 divisions = development population
#
# Within development population:
#
#   frozen Step-13 outer-fold labels
#       ↓
#   reused as 5-fold DEVELOPMENT CV
#       ↓
#   all 120 Step-16 candidates
#       ↓
#   joint model-family + hyperparameter selection
#       ↓
#   weighted sigmoid calibration from development-CV OOF
#       ↓
#   weighted balanced-accuracy threshold optimization
#       ↓
#   winner refit on all 7 development divisions
#       ↓
#   untouched held-out division
#
#
# GEOGRAPHIC PREDICTOR POLICY
# ---------------------------
# Primary model predictors = 15
#
# Geographic IECV predictors = 14
#
# 'division' is removed because it is the geographic hold-out
# variable itself and no effect for the unseen held-out division
# can legitimately be learned from the other seven divisions.
#
# All other predictors remain unchanged.
#
#
# IMPORTANT
# ---------
# This is INTERNAL-EXTERNAL geographic validation within BDHS
# 2022, NOT an independent external validation cohort.
#
# No geographic result may alter:
#   - predictor definitions
#   - model families
#   - hyperparameter grids
#   - preprocessing
#   - calibration method
#   - threshold optimization method
# ============================================================


from pathlib import Path

import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. RESTORE DIRECTORIES IF NECESSARY
# ============================================================

if "DIRS" not in globals():

    ROOT = Path(
        r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
    )

    DIRS = {
        "root":
            ROOT,

        "processed":
            ROOT / "01_processed_data",

        "qc":
            ROOT / "02_quality_control",

        "protocol":
            ROOT / "03_protocol",

        "splits":
            ROOT / "04_cv_splits",

        "models":
            ROOT / "05_models",

        "predictions":
            ROOT / "06_predictions",

        "calibration":
            ROOT / "07_calibration",

        "bootstrap":
            ROOT / "08_bootstrap",

        "geographic":
            ROOT / "09_geographic_validation",

        "explainability":
            ROOT / "10_explainability",

        "sensitivity":
            ROOT / "11_sensitivity",

        "subgroup":
            ROOT / "12_subgroup_analysis",

        "tables":
            ROOT / "13_tables",

        "figures":
            ROOT / "14_figures",

        "logs":
            ROOT / "15_logs",

        "reporting":
            ROOT / "16_reporting",

        "archive":
            ROOT / "17_reproducibility_archive",
    }


# ============================================================
# 2. CORE FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)


STEP16_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_selection_manifest.json"
)


STEP17_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP17_FINAL_calibration_evaluation_manifest.json"
)


STEP18_MANIFEST_FILE = (
    DIRS["protocol"]
    / "STEP18_FINAL_threshold_optimization_manifest.json"
)


STEP19C_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP19C_PRIMARY_execution_manifest.json"
)


STEP21_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP21_PRIMARY_bootstrap_manifest.json"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


# ============================================================
# 3. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column
        in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(
            alias
        )

        if key in lookup:

            return lookup[
                key
            ]


    raise RuntimeError(
        "Could not identify required column.\n"
        f"Aliases: {aliases}\n"
        f"Available: {list(dataframe.columns)}"
    )


def canonical_row_key(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(dtype=float),
            np.round(
                numeric.to_numpy(dtype=float)
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(dtype=float)
            ).astype(np.int64),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


# ============================================================
# 4. REQUIRED FILE / HASH CHECK
# ============================================================

required_files = [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    CANDIDATE_REGISTRY_FILE,
    STEP16_MANIFEST_FILE,
    STEP17_MANIFEST_FILE,
    STEP18_MANIFEST_FILE,
    STEP19C_MANIFEST_FILE,
    STEP21_MANIFEST_FILE,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )


if (
    sha256_file(PRIMARY_FILE)
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    sha256_file(OUTER_SPLIT_FILE)
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Step-13 outer-split hash mismatch."
    )


# ============================================================
# 5. UPSTREAM MANIFEST CHECK
# ============================================================

step16_manifest = read_json(
    STEP16_MANIFEST_FILE
)


step17_manifest = read_json(
    STEP17_MANIFEST_FILE
)


step18_manifest = read_json(
    STEP18_MANIFEST_FILE
)


step19c_manifest = read_json(
    STEP19C_MANIFEST_FILE
)


step21_manifest = read_json(
    STEP21_MANIFEST_FILE
)


expected_status = {
    "Step 16":
        (
            step16_manifest.get("status"),
            "JOINT_MODEL_HYPERPARAMETER_SELECTION_PROTOCOL_FROZEN",
        ),

    "Step 17":
        (
            step17_manifest.get("status"),
            "JOINT_WINNER_CALIBRATION_EVALUATION_PROTOCOL_FROZEN",
        ),

    "Step 18":
        (
            step18_manifest.get("status"),
            "THRESHOLD_OPTIMIZATION_PROTOCOL_FROZEN",
        ),

    "Step 19C":
        (
            step19c_manifest.get("status"),
            "PRIMARY_JOINT_SELECTED_CALIBRATED_THRESHOLD_OPTIMIZED_OOF_LOCKED",
        ),

    "Step 21":
        (
            step21_manifest.get("status"),
            "PRIMARY_SURVEY_DESIGN_BOOTSTRAP_UNCERTAINTY_COMPLETE",
        ),
}


for step_name, (
    observed,
    expected,
) in expected_status.items():

    if observed != expected:

        raise RuntimeError(
            f"{step_name} status mismatch.\n"
            f"Expected: {expected}\n"
            f"Observed: {observed}"
        )


# ============================================================
# 6. LOAD MASTER / FROZEN SPLITS
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)


candidate_registry = pd.read_csv(
    CANDIDATE_REGISTRY_FILE
)


if master.shape != (
    5063,
    23,
):

    raise RuntimeError(
        f"Unexpected master shape: {master.shape}"
    )


if len(
    candidate_registry
) != 120:

    raise RuntimeError(
        "Frozen candidate registry does not contain 120 candidates."
    )


# ============================================================
# 7. ROW / OUTER-FOLD MAPPING
# ============================================================

MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


master = master.copy()

outer_assignment = outer_assignment.copy()


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


if master[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate master row key."
    )


if outer_assignment[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Step-13 outer assignment."
    )


if set(
    master[
        "__ROW_KEY"
    ]
) != set(
    outer_assignment[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Step-13 outer split does not exactly cover master."
    )


outer_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__ROW_KEY"
].map(
    outer_map
)


if master[
    "__DEVELOPMENT_CV_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing Step-13 fold assignment."
    )


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__DEVELOPMENT_CV_FOLD"
].astype(int)


if sorted(
    master[
        "__DEVELOPMENT_CV_FOLD"
    ]
    .unique()
    .tolist()
) != [
    0,
    1,
    2,
    3,
    4,
]:

    raise RuntimeError(
        "Expected five frozen development-CV fold labels."
    )


# ============================================================
# 8. PRIMARY 15-PREDICTOR SET
# ============================================================

NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]


ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]


NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "division",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]


BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]


PRIMARY_PREDICTORS = (
    NUMERIC_PREDICTORS
    +
    ORDERED_CATEGORICAL_PREDICTORS
    +
    NOMINAL_CATEGORICAL_PREDICTORS
    +
    BINARY_CATEGORICAL_PREDICTORS
)


if len(
    PRIMARY_PREDICTORS
) != 15:

    raise RuntimeError(
        "Primary predictor count is not 15."
    )


# ============================================================
# 9. GEOGRAPHIC 14-PREDICTOR SET
# ============================================================

GEOGRAPHIC_PREDICTORS = [
    predictor

    for predictor in PRIMARY_PREDICTORS

    if predictor != "division"
]


GEOGRAPHIC_NUMERIC_PREDICTORS = (
    NUMERIC_PREDICTORS.copy()
)


GEOGRAPHIC_ORDERED_CATEGORICAL_PREDICTORS = (
    ORDERED_CATEGORICAL_PREDICTORS.copy()
)


GEOGRAPHIC_NOMINAL_CATEGORICAL_PREDICTORS = [
    predictor

    for predictor in NOMINAL_CATEGORICAL_PREDICTORS

    if predictor != "division"
]


GEOGRAPHIC_BINARY_CATEGORICAL_PREDICTORS = (
    BINARY_CATEGORICAL_PREDICTORS.copy()
)


GEOGRAPHIC_CATEGORICAL_PREDICTORS = (
    GEOGRAPHIC_ORDERED_CATEGORICAL_PREDICTORS
    +
    GEOGRAPHIC_NOMINAL_CATEGORICAL_PREDICTORS
    +
    GEOGRAPHIC_BINARY_CATEGORICAL_PREDICTORS
)


if len(
    GEOGRAPHIC_PREDICTORS
) != 14:

    raise RuntimeError(
        "Geographic predictor count is not 14."
    )


if "division" in GEOGRAPHIC_PREDICTORS:

    raise RuntimeError(
        "Division predictor was not removed."
    )


# ============================================================
# 10. BASIC ANALYTIC QC
# ============================================================

required_columns = (
    GEOGRAPHIC_PREDICTORS
    +
    [
        "division",
        "ANC_CLASS",
        "SURVEY_WEIGHT",
        "PSU",
        "STRATUM",
    ]
)


missing_columns = [
    column

    for column
    in required_columns

    if column not in master.columns
]


if missing_columns:

    raise RuntimeError(
        f"Master missing required columns:\n{missing_columns}"
    )


master[
    "ANC_CLASS"
] = pd.to_numeric(
    master[
        "ANC_CLASS"
    ],
    errors="raise",
).astype(int)


if set(
    master[
        "ANC_CLASS"
    ].unique()
) != {
    0,
    1,
    2,
}:

    raise RuntimeError(
        "ANC_CLASS is not exactly {0,1,2}."
    )


if master[
    "division"
].isna().any():

    raise RuntimeError(
        "Missing division."
    )


division_values = (
    master[
        "division"
    ]
    .drop_duplicates()
    .tolist()
)


division_values = sorted(
    division_values,
    key=lambda x:
        str(x),
)


if len(
    division_values
) != 8:

    raise RuntimeError(
        f"Expected 8 divisions; observed {len(division_values)}:\n"
        f"{division_values}"
    )


# ============================================================
# 11. PSU / STRATUM GEOGRAPHIC CONSISTENCY
# ============================================================

psu_division_count = (
    master
    .groupby(
        "PSU"
    )[
        "division"
    ]
    .nunique()
)


if (
    psu_division_count
    !=
    1
).any():

    raise RuntimeError(
        "At least one PSU occurs in more than one division."
    )


stratum_division_count = (
    master
    .groupby(
        "STRATUM"
    )[
        "division"
    ]
    .nunique()
)


if (
    stratum_division_count
    !=
    1
).any():

    raise RuntimeError(
        "At least one survey stratum occurs in more than one division."
    )


# ============================================================
# 12. HOLDOUT REGISTRY / DEVELOPMENT-CV QC
# ============================================================

holdout_rows = []

development_cv_rows = []

geographic_bootstrap_design_rows = []


for holdout_division in division_values:

    test = (
        master.loc[
            master[
                "division"
            ]
            ==
            holdout_division
        ]
        .copy()
    )


    development = (
        master.loc[
            master[
                "division"
            ]
            !=
            holdout_division
        ]
        .copy()
    )


    if len(
        test
    ) == 0:

        raise RuntimeError(
            f"Empty held-out division: {holdout_division}"
        )


    if len(
        development
    ) + len(
        test
    ) != len(
        master
    ):

        raise RuntimeError(
            "Geographic partition count mismatch."
        )


    if set(
        development[
            "__ROW_KEY"
        ]
    ) & set(
        test[
            "__ROW_KEY"
        ]
    ):

        raise RuntimeError(
            f"Participant overlap for {holdout_division}."
        )


    if holdout_division in set(
        development[
            "division"
        ]
    ):

        raise RuntimeError(
            f"Held-out division leaked into development: "
            f"{holdout_division}"
        )


    # --------------------------------------------------------
    # Held-out division outcome counts
    # --------------------------------------------------------

    test_stage1_positive = int(
        (
            test[
                "ANC_CLASS"
            ]
            ==
            0
        ).sum()
    )


    test_stage1_negative = int(
        (
            test[
                "ANC_CLASS"
            ]
            !=
            0
        ).sum()
    )


    test_anc = (
        test.loc[
            test[
                "ANC_CLASS"
            ]
            .isin(
                [
                    1,
                    2,
                ]
            )
        ]
    )


    test_stage2_positive = int(
        (
            test_anc[
                "ANC_CLASS"
            ]
            ==
            2
        ).sum()
    )


    test_stage2_negative = int(
        (
            test_anc[
                "ANC_CLASS"
            ]
            ==
            1
        ).sum()
    )


    # Development outcomes

    development_stage1_positive = int(
        (
            development[
                "ANC_CLASS"
            ]
            ==
            0
        ).sum()
    )


    development_stage1_negative = int(
        (
            development[
                "ANC_CLASS"
            ]
            !=
            0
        ).sum()
    )


    development_anc = (
        development.loc[
            development[
                "ANC_CLASS"
            ]
            .isin(
                [
                    1,
                    2,
                ]
            )
        ]
    )


    development_stage2_positive = int(
        (
            development_anc[
                "ANC_CLASS"
            ]
            ==
            2
        ).sum()
    )


    development_stage2_negative = int(
        (
            development_anc[
                "ANC_CLASS"
            ]
            ==
            1
        ).sum()
    )


    heldout_classes_valid = (
        test_stage1_positive
        >
        0
        and
        test_stage1_negative
        >
        0
        and
        test_stage2_positive
        >
        0
        and
        test_stage2_negative
        >
        0
    )


    development_classes_valid = (
        development_stage1_positive
        >
        0
        and
        development_stage1_negative
        >
        0
        and
        development_stage2_positive
        >
        0
        and
        development_stage2_negative
        >
        0
    )


    # --------------------------------------------------------
    # Frozen five-fold DEVELOPMENT CV
    # --------------------------------------------------------

    development_folds = sorted(
        development[
            "__DEVELOPMENT_CV_FOLD"
        ]
        .unique()
        .tolist()
    )


    if development_folds != [
        0,
        1,
        2,
        3,
        4,
    ]:

        raise RuntimeError(
            f"{holdout_division}: incomplete development-CV folds."
        )


    for cv_fold in development_folds:

        cv_validation = (
            development.loc[
                development[
                    "__DEVELOPMENT_CV_FOLD"
                ]
                ==
                cv_fold
            ]
            .copy()
        )


        cv_train = (
            development.loc[
                development[
                    "__DEVELOPMENT_CV_FOLD"
                ]
                !=
                cv_fold
            ]
            .copy()
        )


        psu_overlap = len(
            set(
                cv_train[
                    "PSU"
                ]
            )
            &
            set(
                cv_validation[
                    "PSU"
                ]
            )
        )


        # Stage 1 class QC

        stage1_train_classes = set(
            (
                cv_train[
                    "ANC_CLASS"
                ]
                ==
                0
            )
            .astype(int)
            .unique()
            .tolist()
        )


        stage1_validation_classes = set(
            (
                cv_validation[
                    "ANC_CLASS"
                ]
                ==
                0
            )
            .astype(int)
            .unique()
            .tolist()
        )


        # Stage 2 class QC

        cv_train_stage2 = (
            cv_train.loc[
                cv_train[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
            ]
        )


        cv_validation_stage2 = (
            cv_validation.loc[
                cv_validation[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
            ]
        )


        stage2_train_classes = set(
            (
                cv_train_stage2[
                    "ANC_CLASS"
                ]
                ==
                2
            )
            .astype(int)
            .unique()
            .tolist()
        )


        stage2_validation_classes = set(
            (
                cv_validation_stage2[
                    "ANC_CLASS"
                ]
                ==
                2
            )
            .astype(int)
            .unique()
            .tolist()
        )


        fold_pass = (
            psu_overlap
            ==
            0
            and
            stage1_train_classes
            ==
            {
                0,
                1,
            }
            and
            stage1_validation_classes
            ==
            {
                0,
                1,
            }
            and
            stage2_train_classes
            ==
            {
                0,
                1,
            }
            and
            stage2_validation_classes
            ==
            {
                0,
                1,
            }
        )


        development_cv_rows.append(
            {
                "Heldout_Division":
                    holdout_division,

                "Development_CV_Fold":
                    cv_fold,

                "Train_N":
                    int(
                        len(
                            cv_train
                        )
                    ),

                "Validation_N":
                    int(
                        len(
                            cv_validation
                        )
                    ),

                "Train_PSUs":
                    int(
                        cv_train[
                            "PSU"
                        ].nunique()
                    ),

                "Validation_PSUs":
                    int(
                        cv_validation[
                            "PSU"
                        ].nunique()
                    ),

                "PSU_Overlap":
                    int(
                        psu_overlap
                    ),

                "Stage1_Train_Both_Classes":
                    (
                        stage1_train_classes
                        ==
                        {
                            0,
                            1,
                        }
                    ),

                "Stage1_Validation_Both_Classes":
                    (
                        stage1_validation_classes
                        ==
                        {
                            0,
                            1,
                        }
                    ),

                "Stage2_Train_Both_Classes":
                    (
                        stage2_train_classes
                        ==
                        {
                            0,
                            1,
                        }
                    ),

                "Stage2_Validation_Both_Classes":
                    (
                        stage2_validation_classes
                        ==
                        {
                            0,
                            1,
                        }
                    ),

                "PASS":
                    bool(
                        fold_pass
                    ),
            }
        )


    # --------------------------------------------------------
    # Future held-out-division uncertainty design QC
    # --------------------------------------------------------

    test_stratum_psu = (
        test[
            [
                "STRATUM",
                "PSU",
            ]
        ]
        .drop_duplicates()
        .groupby(
            "STRATUM"
        )
        .size()
    )


    heldout_bootstrap_possible = bool(
        (
            test_stratum_psu
            >
            1
        )
        .all()
    )


    geographic_bootstrap_design_rows.append(
        {
            "Heldout_Division":
                holdout_division,

            "Heldout_Strata":
                int(
                    len(
                        test_stratum_psu
                    )
                ),

            "Minimum_PSUs_Per_Heldout_Stratum":
                int(
                    test_stratum_psu.min()
                ),

            "Maximum_PSUs_Per_Heldout_Stratum":
                int(
                    test_stratum_psu.max()
                ),

            "Rao_Wu_PSU_Bootstrap_Feasible":
                heldout_bootstrap_possible,
        }
    )


    holdout_rows.append(
        {
            "Heldout_Division":
                holdout_division,

            "Development_N":
                int(
                    len(
                        development
                    )
                ),

            "Heldout_N":
                int(
                    len(
                        test
                    )
                ),

            "Development_PSUs":
                int(
                    development[
                        "PSU"
                    ].nunique()
                ),

            "Heldout_PSUs":
                int(
                    test[
                        "PSU"
                    ].nunique()
                ),

            "Heldout_Strata":
                int(
                    test[
                        "STRATUM"
                    ].nunique()
                ),

            "Development_Stage1_NoANC":
                development_stage1_positive,

            "Development_Stage1_AnyANC":
                development_stage1_negative,

            "Heldout_Stage1_NoANC":
                test_stage1_positive,

            "Heldout_Stage1_AnyANC":
                test_stage1_negative,

            "Development_Stage2_GE8":
                development_stage2_positive,

            "Development_Stage2_1_7":
                development_stage2_negative,

            "Heldout_Stage2_GE8":
                test_stage2_positive,

            "Heldout_Stage2_1_7":
                test_stage2_negative,

            "Development_Both_Classes_Valid":
                development_classes_valid,

            "Heldout_Both_Classes_Valid":
                heldout_classes_valid,

            "Development_CV_Folds":
                5,

            "Division_Predictor_Removed":
                True,
        }
    )


holdout_registry = pd.DataFrame(
    holdout_rows
)


development_cv_qc = pd.DataFrame(
    development_cv_rows
)


geographic_bootstrap_design_qc = pd.DataFrame(
    geographic_bootstrap_design_rows
)


# ============================================================
# 13. CANDIDATE GRID QC
# ============================================================

model_family_count = (
    candidate_registry[
        "Model"
    ]
    .nunique()
)


candidate_count_by_family = (
    candidate_registry
    .groupby(
        "Model"
    )
    .size()
)


candidate_grid_qc = pd.DataFrame(
    [
        {
            "Check":
                "10 model families retained",

            "PASS":
                model_family_count
                ==
                10,
        },

        {
            "Check":
                "120 candidates retained",

            "PASS":
                len(
                    candidate_registry
                )
                ==
                120,
        },

        {
            "Check":
                "12 candidates per family retained",

            "PASS":
                bool(
                    (
                        candidate_count_by_family
                        ==
                        12
                    )
                    .all()
                ),
        },

        {
            "Check":
                "No geographic-result-driven grid modification",

            "PASS":
                True,
        },
    ]
)


# ============================================================
# 14. GEOGRAPHIC IECV PROTOCOL
# ============================================================

geographic_protocol = {
    "analysis":
        (
            "Geographic internal-external cross-validation "
            "transportability stress test"
        ),

    "independent_external_validation":
        False,

    "data_source":
        "BDHS 2022 analytic cohort",

    "geographic_cluster":
        "division",

    "geographic_cluster_count":
        8,

    "validation_scheme":
        "leave-one-division-out",

    "iterations":
        8,

    "primary_predictor_count":
        15,

    "geographic_predictor_count":
        14,

    "predictor_removed":
        "division",

    "reason_division_removed":
        (
            "Division defines the geographic hold-out cluster; "
            "the held-out division has no estimable development "
            "effect and geographic identity must not be used to "
            "circumvent transportability assessment."
        ),

    "other_predictors_changed":
        False,

    "feature_selection":
        False,

    "model_families":
        10,

    "joint_candidates":
        120,

    "candidate_grid_changed":
        False,

    "preprocessing":
        (
            "Same frozen model-specific Step-14 preprocessing, "
            "applied to the 14-predictor geographic set"
        ),

    "development_cv":
        (
            "Five frozen Step-13 PSU-exclusive fold labels "
            "restricted to the seven development divisions"
        ),

    "new_cv_split_generation":
        False,

    "development_cv_selection_primary":
        "survey-weighted average precision",

    "development_cv_selection_tie_1":
        "survey-weighted ROC-AUC",

    "development_cv_selection_tie_2":
        "frozen deterministic candidate rank",

    "primary_base_model_fitting":
        (
            "DHS survey weights normalized to mean 1 "
            "within each stage-specific training partition"
        ),

    "class_balancing":
        False,

    "stage1_development_population":
        "all participants in seven development divisions",

    "stage1_positive_class":
        "No ANC",

    "stage2_development_population":
        (
            "true ANC users only in seven development divisions"
        ),

    "stage2_positive_class":
        ">=8 reported ANC visits",

    "calibration":
        (
            "Frozen survey-weighted sigmoid calibration fitted "
            "only to pooled development-CV winner OOF scores"
        ),

    "threshold_optimization":
        (
            "Frozen survey-weighted balanced-accuracy "
            "optimization using calibrated development-CV OOF "
            "probabilities only"
        ),

    "heldout_outcome_used_for_model_selection":
        False,

    "heldout_outcome_used_for_hyperparameter_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold_optimization":
        False,

    "heldout_division_model_updating":
        False,

    "stage2_scoring_for_fusion":
        (
            "Stage-2 model trained on true ANC users but scores "
            "all participants in held-out division"
        ),

    "hierarchical_fusion":
        (
            "p0=q0; p1=(1-q0)(1-q2); p2=(1-q0)q2"
        ),

    "hierarchical_threshold_rule":
        (
            "Stage 1 priority: q0>=t1 -> No ANC; otherwise "
            "q2>=t2 -> >=8; otherwise 1-7"
        ),

    "primary_geographic_reporting":
        (
            "Division-specific discrimination, probabilistic "
            "performance, calibration, and threshold summaries"
        ),

    "secondary_geographic_reporting":
        (
            "Pooled leave-one-division-out cross-fitted "
            "predictions across all participants"
        ),

    "heterogeneity_reporting":
        (
            "Division-specific forest/table presentation and "
            "descriptive range/IQR; no performance-based ranking"
        ),

    "random_effects_meta_analysis":
        False,

    "random_effects_reason":
        (
            "Eight Bangladesh divisions are treated as the fixed "
            "administrative regions represented in BDHS 2022, "
            "rather than a random sample of centres/studies."
        ),

    "geographic_uncertainty_later":
        (
            "Held-out-division Rao-Wu PSU bootstrap within "
            "held-out survey strata on locked IECV predictions; "
            "no model retraining"
        ),

    "geographic_results_can_modify_primary_model":
        False,
}


# ============================================================
# 15. PREDICTOR POLICY TABLE
# ============================================================

predictor_policy_rows = []


for predictor in PRIMARY_PREDICTORS:

    predictor_policy_rows.append(
        {
            "Predictor":
                predictor,

            "Primary_Pipeline":
                "Included",

            "Geographic_IECV":
                (
                    "Excluded"
                    if predictor
                    ==
                    "division"
                    else
                    "Included"
                ),

            "Reason":
                (
                    "Geographic hold-out variable"
                    if predictor
                    ==
                    "division"
                    else
                    "Frozen predictor retained unchanged"
                ),
        }
    )


predictor_policy = pd.DataFrame(
    predictor_policy_rows
)


# ============================================================
# 16. METHODS PREVIEW
# ============================================================

methods_text = """
Geographic internal-external validation

Geographic transportability will be examined using leave-one-division-
out internal-external cross-validation across the eight administrative
divisions represented in the BDHS 2022 analytic cohort. This analysis
will be interpreted as an internal-external geographic validation and
stress test rather than as validation in an independent external
cohort.

For each iteration, observations from one division will be withheld
completely from model development and used only for geographic
evaluation. Model development will use observations from the remaining
seven divisions. Because division itself defines the geographic
hold-out unit, the division predictor will be removed from this
analysis, leaving the other 14 prespecified predictors unchanged.

Within each seven-division development set, the five previously frozen,
PSU-exclusive Step-13 fold assignments will be reused as development
cross-validation folds. No new resampling partitions will be generated
after inspection of predictive performance. All 120 prespecified joint
model-family and hyperparameter candidates will be evaluated using the
same model-specific preprocessing and survey-weighted fitting policy as
in the primary analysis. Candidate selection will be based first on
survey-weighted average precision, then survey-weighted ROC-AUC, with
the frozen deterministic candidate rank used for any remaining tie.

Following joint selection, probability calibration will use the
selected winner's pooled development-cross-validation out-of-fold
scores and the frozen survey-weighted sigmoid calibration procedure.
Stage-specific decision thresholds will subsequently be optimized from
the calibrated development-cross-validation predictions using the
prespecified survey-weighted balanced-accuracy criterion. No outcomes
from the held-out division will be used for model-family selection,
hyperparameter tuning, calibration, threshold selection, or model
updating.

Stage 1 will model no ANC versus any ANC in the full development
population. Stage 2 will model eight or more reported ANC visits versus
one to seven visits among true ANC users in the development data.
After refitting the selected models on the complete seven-division
development population, the Stage-2 model will generate conditional
probabilities for all participants in the held-out division to permit
hierarchical probability fusion.

Performance will be reported separately for each held-out division to
characterize geographic heterogeneity and transportability. Pooled
leave-one-division-out predictions will be treated as a secondary
overall geographic stress-test summary. No division will be ranked as
best or worst on the basis of predictive performance. Geographic
validation results will not be used to modify the frozen primary model
development procedure.
""".strip()


# ============================================================
# 17. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(
                    PRIMARY_FILE
                )
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                sha256_file(
                    OUTER_SPLIT_FILE
                )
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Exactly 8 divisions",

            "PASS":
                len(
                    division_values
                )
                ==
                8,
        },

        {
            "Check":
                "PSUs map to exactly one division",

            "PASS":
                bool(
                    (
                        psu_division_count
                        ==
                        1
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Strata map to exactly one division",

            "PASS":
                bool(
                    (
                        stratum_division_count
                        ==
                        1
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Geographic predictor count = 14",

            "PASS":
                len(
                    GEOGRAPHIC_PREDICTORS
                )
                ==
                14,
        },

        {
            "Check":
                "Division removed from geographic predictor set",

            "PASS":
                "division"
                not in
                GEOGRAPHIC_PREDICTORS,
        },

        {
            "Check":
                "No other predictor removed",

            "PASS":
                set(
                    GEOGRAPHIC_PREDICTORS
                )
                ==
                (
                    set(
                        PRIMARY_PREDICTORS
                    )
                    -
                    {
                        "division"
                    }
                ),
        },

        {
            "Check":
                "All held-out divisions have both Stage-1 classes",

            "PASS":
                bool(
                    (
                        holdout_registry[
                            "Heldout_Stage1_NoANC"
                        ]
                        >
                        0
                    )
                    .all()
                    and
                    (
                        holdout_registry[
                            "Heldout_Stage1_AnyANC"
                        ]
                        >
                        0
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All held-out divisions have both Stage-2 classes",

            "PASS":
                bool(
                    (
                        holdout_registry[
                            "Heldout_Stage2_GE8"
                        ]
                        >
                        0
                    )
                    .all()
                    and
                    (
                        holdout_registry[
                            "Heldout_Stage2_1_7"
                        ]
                        >
                        0
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All seven-division development sets have both classes",

            "PASS":
                bool(
                    holdout_registry[
                        "Development_Both_Classes_Valid"
                    ].all()
                ),
        },

        {
            "Check":
                "All 40 development-CV fold checks pass",

            "PASS":
                (
                    len(
                        development_cv_qc
                    )
                    ==
                    40
                    and
                    bool(
                        development_cv_qc[
                            "PASS"
                        ].all()
                    )
                ),
        },

        {
            "Check":
                "No PSU overlap in development CV",

            "PASS":
                bool(
                    (
                        development_cv_qc[
                            "PSU_Overlap"
                        ]
                        ==
                        0
                    )
                    .all()
                ),
        },

        {
            "Check":
                "All held-out divisions support later PSU bootstrap",

            "PASS":
                bool(
                    geographic_bootstrap_design_qc[
                        "Rao_Wu_PSU_Bootstrap_Feasible"
                    ].all()
                ),
        },

        {
            "Check":
                "All 10 model families retained",

            "PASS":
                model_family_count
                ==
                10,
        },

        {
            "Check":
                "All 120 candidates retained",

            "PASS":
                len(
                    candidate_registry
                )
                ==
                120,
        },

        {
            "Check":
                "No new CV split generated",

            "PASS":
                True,
        },

        {
            "Check":
                "Held-out outcomes cannot tune/select models",

            "PASS":
                (
                    geographic_protocol[
                        "heldout_outcome_used_for_model_selection"
                    ]
                    is False
                    and
                    geographic_protocol[
                        "heldout_outcome_used_for_hyperparameter_tuning"
                    ]
                    is False
                ),
        },

        {
            "Check":
                "Held-out outcomes cannot calibrate/optimize thresholds",

            "PASS":
                (
                    geographic_protocol[
                        "heldout_outcome_used_for_calibration"
                    ]
                    is False
                    and
                    geographic_protocol[
                        "heldout_outcome_used_for_threshold_optimization"
                    ]
                    is False
                ),
        },

        {
            "Check":
                "No held-out model updating",

            "PASS":
                geographic_protocol[
                    "heldout_division_model_updating"
                ]
                is False,
        },

        {
            "Check":
                "No geographic model fitted in Step 22",

            "PASS":
                True,
        },

        {
            "Check":
                "No geographic performance inspected in Step 22",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "STEP 22 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 18. SAVE PROTOCOL OUTPUTS
# ============================================================

HOLDOUT_REGISTRY_FILE = (
    DIRS["geographic"]
    / "STEP22_GEOGRAPHIC_holdout_division_registry.csv"
)


DEVELOPMENT_CV_QC_FILE = (
    DIRS["qc"]
    / "STEP22_GEOGRAPHIC_development_CV_qc.csv"
)


BOOTSTRAP_DESIGN_QC_FILE = (
    DIRS["qc"]
    / "STEP22_GEOGRAPHIC_future_bootstrap_design_qc.csv"
)


CANDIDATE_GRID_QC_FILE = (
    DIRS["qc"]
    / "STEP22_GEOGRAPHIC_candidate_grid_qc.csv"
)


PREDICTOR_POLICY_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_predictor_policy.csv"
)


PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_IECV_protocol.json"
)


METHODS_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_manuscript_methods.txt"
)


HARD_QC_FILE = (
    DIRS["qc"]
    / "STEP22_GEOGRAPHIC_hard_qc.csv"
)


holdout_registry.to_csv(
    HOLDOUT_REGISTRY_FILE,
    index=False,
)


development_cv_qc.to_csv(
    DEVELOPMENT_CV_QC_FILE,
    index=False,
)


geographic_bootstrap_design_qc.to_csv(
    BOOTSTRAP_DESIGN_QC_FILE,
    index=False,
)


candidate_grid_qc.to_csv(
    CANDIDATE_GRID_QC_FILE,
    index=False,
)


predictor_policy.to_csv(
    PREDICTOR_POLICY_FILE,
    index=False,
)


hard_qc.to_csv(
    HARD_QC_FILE,
    index=False,
)


with open(
    PROTOCOL_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        geographic_protocol,
        handle,
        indent=2,
        ensure_ascii=False,
    )


with open(
    METHODS_FILE,
    "w",
    encoding="utf-8",
) as handle:

    handle.write(
        methods_text
        +
        "\n"
    )


# ============================================================
# 19. MANIFEST
# ============================================================

manifest = {
    "step":
        "STEP 22",

    "status":
        "GEOGRAPHIC_IECV_PROTOCOL_FROZEN",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "validation_type":
        (
            "geographic internal-external cross-validation"
        ),

    "independent_external_validation":
        False,

    "heldout_unit":
        "division",

    "division_count":
        8,

    "predictor_count":
        14,

    "division_predictor_removed":
        True,

    "model_family_count":
        10,

    "joint_candidate_count":
        120,

    "development_cv_folds":
        5,

    "development_cv_source":
        (
            "frozen Step-13 PSU-exclusive outer-fold assignments "
            "restricted to seven development divisions"
        ),

    "new_splits_generated":
        False,

    "calibration_method_changed":
        False,

    "threshold_method_changed":
        False,

    "heldout_outcome_used_during_development":
        False,

    "geographic_performance_inspected":
        False,

    "model_fitted":
        False,

    "primary_master_sha256":
        sha256_file(
            PRIMARY_FILE
        ),

    "step13_outer_split_sha256":
        sha256_file(
            OUTER_SPLIT_FILE
        ),

    "candidate_registry_sha256":
        sha256_file(
            CANDIDATE_REGISTRY_FILE
        ),

    "holdout_registry_sha256":
        sha256_file(
            HOLDOUT_REGISTRY_FILE
        ),

    "development_cv_qc_sha256":
        sha256_file(
            DEVELOPMENT_CV_QC_FILE
        ),

    "predictor_policy_sha256":
        sha256_file(
            PREDICTOR_POLICY_FILE
        ),

    "protocol_sha256":
        sha256_file(
            PROTOCOL_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            HARD_QC_FILE
        ),
}


MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP22_GEOGRAPHIC_IECV_protocol_manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 20. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 22 — GEOGRAPHIC IECV PROTOCOL FREEZE"
)


print(
    "=" * 78
)


print(
    "\nHELD-OUT DIVISION REGISTRY:"
)


display(
    holdout_registry
)


print(
    "\n14-PREDICTOR GEOGRAPHIC POLICY:"
)


display(
    predictor_policy
)


print(
    "\nDEVELOPMENT-CV QC:"
)


display(
    development_cv_qc
)


print(
    "\nHELD-OUT DIVISION BOOTSTRAP FEASIBILITY:"
)


display(
    geographic_bootstrap_design_qc
)


print(
    "\nCANDIDATE GRID QC:"
)


display(
    candidate_grid_qc
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nMETHODS PREVIEW:"
)


print(
    methods_text
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)


print(
    f"Step-13 folds  : {sha256_file(OUTER_SPLIT_FILE)}"
)


print(
    f"Candidate grid : {sha256_file(CANDIDATE_REGISTRY_FILE)}"
)


print(
    "=" * 78
)


print(
    "STATUS: STEP 22 GEOGRAPHIC IECV PROTOCOL PASS"
)


print(
    "=" * 78
)


STEP 22 — GEOGRAPHIC IECV PROTOCOL FREEZE

HELD-OUT DIVISION REGISTRY:


,Heldout_Division,Development_N,Heldout_N,Development_PSUs,Heldout_PSUs,Heldout_Strata,Development_Stage1_NoANC,Development_Stage1_AnyANC,Heldout_Stage1_NoANC,Heldout_Stage1_AnyANC,Development_Stage2_GE8,Development_Stage2_1_7,Heldout_Stage2_GE8,Heldout_Stage2_1_7,Development_Both_Classes_Valid,Heldout_Both_Classes_Valid,Development_CV_Folds,Division_Predictor_Removed
0,Barishal,4519,544,601,71,2,363,4156,40,504,224,3932,29,475,True,True,5,True
1,Chattogram,4183,880,579,93,2,329,3854,74,806,217,3637,36,770,True,True,5,True
2,Dhaka,4310,753,567,105,2,360,3950,43,710,184,3766,69,641,True,True,5,True
3,Khulna,4489,574,586,86,2,380,4109,23,551,212,3897,41,510,True,True,5,True
4,Mymensingh,4443,620,597,75,2,334,4109,69,551,237,3872,16,535,True,True,5,True
5,Rajshahi,4551,512,584,88,2,377,4174,26,486,235,3939,18,468,True,True,5,True
6,Rangpur,4471,592,588,84,2,354,4117,49,543,232,3885,21,522,True,True,5,True
7,Sylhet,4475,588,602,70,2,324,4151,79,509,230,3921,23,486,True,True,5,True



14-PREDICTOR GEOGRAPHIC POLICY:


,Predictor,Primary_Pipeline,Geographic_IECV,Reason
0,maternal_age_at_index_birth_years,Included,Included,Frozen predictor retained unchanged
1,prior_live_births,Included,Included,Frozen predictor retained unchanged
2,household_size_members,Included,Included,Frozen predictor retained unchanged
3,maternal_education,Included,Included,Frozen predictor retained unchanged
4,wealth_quintile,Included,Included,Frozen predictor retained unchanged
5,partner_education_primary_final,Included,Included,Frozen predictor retained unchanged
6,division,Included,Excluded,Geographic hold-out variable
7,pregnancy_intention,Included,Included,Frozen predictor retained unchanged
8,healthcare_decision_primary_final,Included,Included,Frozen predictor retained unchanged
9,residence,Included,Included,Frozen predictor retained unchanged



DEVELOPMENT-CV QC:


,Heldout_Division,Development_CV_Fold,Train_N,Validation_N,Train_PSUs,Validation_PSUs,PSU_Overlap,Stage1_Train_Both_Classes,Stage1_Validation_Both_Classes,Stage2_Train_Both_Classes,Stage2_Validation_Both_Classes,PASS
0,Barishal,0,3620,899,483,118,0,True,True,True,True,True
1,Barishal,1,3614,905,478,123,0,True,True,True,True,True
2,Barishal,2,3624,895,484,117,0,True,True,True,True,True
3,Barishal,3,3623,896,483,118,0,True,True,True,True,True
4,Barishal,4,3595,924,476,125,0,True,True,True,True,True
5,Chattogram,0,3346,837,465,114,0,True,True,True,True,True
6,Chattogram,1,3349,834,463,116,0,True,True,True,True,True
7,Chattogram,2,3351,832,465,114,0,True,True,True,True,True
8,Chattogram,3,3351,832,463,116,0,True,True,True,True,True
9,Chattogram,4,3335,848,460,119,0,True,True,True,True,True


... (3 outputs left out)

,Check,PASS
0,10 model families retained,True
1,120 candidates retained,True
2,12 candidates per family retained,True
3,No geographic-result-driven grid modification,True



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Exactly 8 divisions,True
3,PSUs map to exactly one division,True
4,Strata map to exactly one division,True
5,Geographic predictor count = 14,True
6,Division removed from geographic predictor set,True
7,No other predictor removed,True
8,All held-out divisions have both Stage-1 classes,True
9,All held-out divisions have both Stage-2 classes,True



METHODS PREVIEW:
Geographic internal-external validation

Geographic transportability will be examined using leave-one-division-
out internal-external cross-validation across the eight administrative
divisions represented in the BDHS 2022 analytic cohort. This analysis
will be interpreted as an internal-external geographic validation and
stress test rather than as validation in an independent external
cohort.

For each iteration, observations from one division will be withheld
completely from model development and used only for geographic
evaluation. Model development will use observations from the remaining
seven divisions. Because division itself defines the geographic
hold-out unit, the division predictor will be removed from this
analysis, leaving the other 14 prespecified predictors unchanged.

Within each seven-division development set, the five previously frozen,
PSU-exclusive Step-13 fold assignments will be reused as development
cross-validation folds. No new resampling parti

In [1]:
# ============================================================
# STEP 23 — LAPTOP-OPTIMIZED GEOGRAPHIC IECV
#           ONE DIVISION AT A TIME
#           FAMILY + STAGE CHECKPOINTING
# ============================================================
#
# IMPORTANT
# ---------
# - Scientific protocol is UNCHANGED.
# - Same frozen 120 candidates.
# - Same 5 frozen PSU-exclusive CV folds.
# - Same survey-weighted fitting/selection.
# - Same calibration.
# - Same threshold optimization.
# - Same hierarchical fusion.
#
# ONLY execution is changed:
#   * ONE held-out division per run
#   * CPU threads limited to 2
#   * model-family checkpoints
#   * stage checkpoints
#
# NO held-out performance metric is calculated here.
# ============================================================


# ============================================================
# 0. SELECT ONLY ONE DIVISION
# ============================================================

TARGET_DIVISION = "Barishal"

# Later change ONLY this line:
#
# "Barishal"
# "Chattogram"
# "Dhaka"
# "Khulna"
# "Mymensingh"
# "Rajshahi"
# "Rangpur"
# "Sylhet"


# ============================================================
# 1. LIMIT CPU / BLAS THREADS
# ============================================================

import os

EXECUTION_THREADS = 2

os.environ["OMP_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["MKL_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(EXECUTION_THREADS)


# ============================================================
# 2. IMPORTS
# ============================================================

from pathlib import Path

import ast
import datetime as dt
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit

from sklearn.exceptions import ConvergenceWarning

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from IPython.display import display


# ============================================================
# 3. PROJECT DIRECTORIES
# ============================================================

ROOT = Path(
    r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
)


DIRS = {
    "root":
        ROOT,

    "processed":
        ROOT / "01_processed_data",

    "qc":
        ROOT / "02_quality_control",

    "protocol":
        ROOT / "03_protocol",

    "splits":
        ROOT / "04_cv_splits",

    "models":
        ROOT / "05_models",

    "predictions":
        ROOT / "06_predictions",

    "calibration":
        ROOT / "07_calibration",

    "bootstrap":
        ROOT / "08_bootstrap",

    "geographic":
        ROOT / "09_geographic_validation",

    "explainability":
        ROOT / "10_explainability",

    "sensitivity":
        ROOT / "11_sensitivity",

    "subgroup":
        ROOT / "12_subgroup_analysis",

    "tables":
        ROOT / "13_tables",

    "figures":
        ROOT / "14_figures",

    "logs":
        ROOT / "15_logs",

    "reporting":
        ROOT / "16_reporting",

    "archive":
        ROOT / "17_reproducibility_archive",
}


for path in DIRS.values():

    if isinstance(path, Path):

        path.mkdir(
            parents=True,
            exist_ok=True,
        )


# ============================================================
# 4. CONSTANTS
# ============================================================

SEED = 2026

EXECUTION_SCHEMA_VERSION = (
    "STEP23_GEOGRAPHIC_LAPTOP_v2"
)

METRIC_TIE_TOLERANCE = 1e-12

CALIBRATION_PROBABILITY_CLIP = 1e-6

THRESHOLD_PROBABILITY_CLIP = 1e-12


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_CANDIDATE_SHA256 = (
    "f95dc93431eced694114fe8ed071a2878"
    "e2572fac53aa3b9fd9c6cb8df87c697"
)


VALID_DIVISIONS = [
    "Barishal",
    "Chattogram",
    "Dhaka",
    "Khulna",
    "Mymensingh",
    "Rajshahi",
    "Rangpur",
    "Sylhet",
]


if TARGET_DIVISION not in VALID_DIVISIONS:

    raise ValueError(
        f"TARGET_DIVISION must be one of:\n"
        f"{VALID_DIVISIONS}"
    )


# ============================================================
# 5. FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)


STEP22_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_IECV_protocol.json"
)


STEP22_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP22_GEOGRAPHIC_IECV_protocol_manifest.json"
)


STEP22_HOLDOUT_FILE = (
    DIRS["geographic"]
    / "STEP22_GEOGRAPHIC_holdout_division_registry.csv"
)


# ============================================================
# 6. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):

            digest.update(block)

    return digest.hexdigest()


def sha256_text(text):

    return hashlib.sha256(
        str(text).encode(
            "utf-8"
        )
    ).hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def safe_name(value):

    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(">=", "GE")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column
        in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(alias)

        if key in lookup:

            return lookup[key]


    raise RuntimeError(
        "Required column could not be identified.\n"
        f"Aliases: {aliases}\n"
        f"Available columns:\n"
        f"{list(dataframe.columns)}"
    )


def canonical_row_key(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(
                dtype=float
            ),
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ).astype(
                np.int64
            ),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


def parse_parameter_dictionary(value):

    if isinstance(
        value,
        dict,
    ):

        return value.copy()


    if pd.isna(value):

        return {}


    text = str(value).strip()


    if text == "":

        return {}


    try:

        parsed = json.loads(text)

    except json.JSONDecodeError:

        parsed = ast.literal_eval(text)


    if not isinstance(
        parsed,
        dict,
    ):

        raise RuntimeError(
            "Candidate parameter field did not decode "
            "to a dictionary."
        )


    return parsed


def row_key_hash(dataframe):

    values = (
        dataframe[
            "__ROW_KEY"
        ]
        .astype(str)
        .tolist()
    )

    return sha256_text(
        "|".join(values)
    )


# ============================================================
# 7. REQUIRED FILE CHECK
# ============================================================

required_files = [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    CANDIDATE_REGISTRY_FILE,
    STEP22_PROTOCOL_FILE,
    STEP22_MANIFEST_FILE,
    STEP22_HOLDOUT_FILE,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n{path}"
        )


# ============================================================
# 8. IMMUTABILITY CHECK
# ============================================================

if (
    sha256_file(PRIMARY_FILE)
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    sha256_file(OUTER_SPLIT_FILE)
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Step-13 outer split hash mismatch."
    )


if (
    sha256_file(
        CANDIDATE_REGISTRY_FILE
    )
    !=
    EXPECTED_CANDIDATE_SHA256
):

    raise RuntimeError(
        "Step-16 candidate registry hash mismatch."
    )


step22_manifest = read_json(
    STEP22_MANIFEST_FILE
)


if (
    step22_manifest.get("status")
    !=
    "GEOGRAPHIC_IECV_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 22 geographic protocol is not frozen."
    )


STEP22_PROTOCOL_SHA256 = (
    sha256_file(
        STEP22_PROTOCOL_FILE
    )
)


# ============================================================
# 9. LOAD DATA
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)


candidate_registry = pd.read_csv(
    CANDIDATE_REGISTRY_FILE
)


holdout_registry = pd.read_csv(
    STEP22_HOLDOUT_FILE
)


if master.shape != (
    5063,
    23,
):

    raise RuntimeError(
        f"Unexpected master shape: {master.shape}"
    )


if len(
    candidate_registry
) != 120:

    raise RuntimeError(
        "Candidate registry must contain 120 candidates."
    )


# ============================================================
# 10. CANDIDATE REGISTRY QC
# ============================================================

required_candidate_columns = {
    "Candidate_ID",
    "Model",
    "Preprocessing",
    "Deterministic_Tie_Rank",
    "Tuned_Params_JSON",
    "Fixed_Params_JSON",
}


missing_candidate_columns = (
    required_candidate_columns
    -
    set(
        candidate_registry.columns
    )
)


if missing_candidate_columns:

    raise RuntimeError(
        "Candidate registry missing columns:\n"
        f"{sorted(missing_candidate_columns)}"
    )


candidate_count_by_family = (
    candidate_registry
    .groupby("Model")
    .size()
)


if (
    len(
        candidate_count_by_family
    )
    !=
    10
):

    raise RuntimeError(
        "Expected exactly 10 model families."
    )


if (
    candidate_count_by_family
    !=
    12
).any():

    raise RuntimeError(
        "Each family must contain exactly 12 candidates."
    )


MODEL_ORDER = (
    candidate_registry
    .groupby(
        "Model",
        as_index=False,
    )[
        "Deterministic_Tie_Rank"
    ]
    .min()
    .sort_values(
        "Deterministic_Tie_Rank"
    )[
        "Model"
    ]
    .tolist()
)


# ============================================================
# 11. ROW / FOLD MAPPING
# ============================================================

MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


master = master.copy()

outer_assignment = outer_assignment.copy()


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


if master[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate master row key."
    )


if outer_assignment[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate outer-fold row key."
    )


if set(
    master[
        "__ROW_KEY"
    ]
) != set(
    outer_assignment[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Outer split does not exactly cover master."
    )


fold_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__ROW_KEY"
].map(
    fold_map
)


if master[
    "__DEVELOPMENT_CV_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing development-CV assignment."
    )


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__DEVELOPMENT_CV_FOLD"
].astype(int)


if sorted(
    master[
        "__DEVELOPMENT_CV_FOLD"
    ]
    .unique()
    .tolist()
) != [
    0,
    1,
    2,
    3,
    4,
]:

    raise RuntimeError(
        "Expected five frozen CV folds."
    )


# ============================================================
# 12. GEOGRAPHIC 14-PREDICTOR SET
# ============================================================

GEO_NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]


GEO_ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]


GEO_NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]


GEO_BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]


GEO_CATEGORICAL_PREDICTORS = (
    GEO_ORDERED_CATEGORICAL_PREDICTORS
    +
    GEO_NOMINAL_CATEGORICAL_PREDICTORS
    +
    GEO_BINARY_CATEGORICAL_PREDICTORS
)


GEO_PREDICTORS = (
    GEO_NUMERIC_PREDICTORS
    +
    GEO_CATEGORICAL_PREDICTORS
)


if len(
    GEO_PREDICTORS
) != 14:

    raise RuntimeError(
        "Geographic predictor count is not 14."
    )


if "division" in GEO_PREDICTORS:

    raise RuntimeError(
        "Division leaked into geographic predictor set."
    )


# ============================================================
# 13. OUTCOME / WEIGHT QC
# ============================================================

master[
    "ANC_CLASS"
] = pd.to_numeric(
    master[
        "ANC_CLASS"
    ],
    errors="raise",
).astype(int)


if set(
    master[
        "ANC_CLASS"
    ].unique()
) != {
    0,
    1,
    2,
}:

    raise RuntimeError(
        "ANC_CLASS must equal {0,1,2}."
    )


if (
    not np.isfinite(
        master[
            "SURVEY_WEIGHT"
        ].to_numpy(
            dtype=float
        )
    ).all()
):

    raise RuntimeError(
        "Non-finite survey weight."
    )


if (
    master[
        "SURVEY_WEIGHT"
    ]
    .to_numpy(
        dtype=float
    )
    <=
    0
).any():

    raise RuntimeError(
        "Survey weight must be positive."
    )


if TARGET_DIVISION not in set(
    master[
        "division"
    ].astype(str)
):

    raise RuntimeError(
        f"{TARGET_DIVISION} not found in master."
    )


# ============================================================
# 14. DEVELOPMENT / HOLDOUT
# ============================================================

development = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        !=
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


heldout = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        ==
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if len(
    development
) + len(
    heldout
) != 5063:

    raise RuntimeError(
        "Development + held-out count mismatch."
    )


if (
    set(
        development[
            "__ROW_KEY"
        ]
    )
    &
    set(
        heldout[
            "__ROW_KEY"
        ]
    )
):

    raise RuntimeError(
        "Development / held-out overlap."
    )


if TARGET_DIVISION in set(
    development[
        "division"
    ].astype(str)
):

    raise RuntimeError(
        "Held-out division leaked into development."
    )


EXPECTED_HELDOUT_N = len(
    heldout
)


print(
    "\n"
    + "=" * 78
)

print(
    "STEP 23 — LAPTOP-OPTIMIZED GEOGRAPHIC IECV"
)

print(
    "=" * 78
)

print(
    f"Held-out division : {TARGET_DIVISION}"
)

print(
    f"Development N     : {len(development)}"
)

print(
    f"Held-out N        : {len(heldout)}"
)

print(
    f"CPU threads       : {EXECUTION_THREADS}"
)

print(
    "Division predictor: EXCLUDED"
)


# ============================================================
# 15. TRAINING WEIGHTS
# ============================================================

def normalized_training_weight(
    dataframe,
):

    weight = (
        dataframe[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    normalized = (
        weight
        /
        np.mean(weight)
    )


    if not np.isclose(
        np.mean(normalized),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Training-weight normalization failed."
        )


    return normalized


# ============================================================
# 16. STAGE DATA
# ============================================================

def prepare_stage(
    dataframe,
    stage_name,
):

    data = dataframe.copy()


    if stage_name == "Stage 1":

        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            0
        ).astype(int)


    elif stage_name == "Stage 2":

        data = (
            data.loc[
                data[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
            ]
            .copy()
        )


        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            2
        ).astype(int)


    else:

        raise ValueError(
            f"Unknown stage: {stage_name}"
        )


    return data.reset_index(
        drop=True
    )


# ============================================================
# 17. MODEL-SPECIFIC PREPROCESSING
# ============================================================

def build_representations(
    training_data,
    scoring_data,
):

    training_numeric = (
        training_data[
            GEO_NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    scoring_numeric = (
        scoring_data[
            GEO_NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    encoder = OneHotEncoder(
        drop=None,
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float64,
    )


    training_categorical = (
        training_data[
            GEO_CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    scoring_categorical = (
        scoring_data[
            GEO_CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    training_ohe = (
        encoder.fit_transform(
            training_categorical
        )
    )


    scoring_ohe = (
        encoder.transform(
            scoring_categorical
        )
    )


    training_unscaled = np.hstack(
        [
            training_numeric,
            training_ohe,
        ]
    )


    scoring_unscaled = np.hstack(
        [
            scoring_numeric,
            scoring_ohe,
        ]
    )


    scaler = StandardScaler()


    training_scaled_numeric = (
        scaler.fit_transform(
            training_numeric
        )
    )


    scoring_scaled_numeric = (
        scaler.transform(
            scoring_numeric
        )
    )


    training_scaled = np.hstack(
        [
            training_scaled_numeric,
            training_ohe,
        ]
    )


    scoring_scaled = np.hstack(
        [
            scoring_scaled_numeric,
            scoring_ohe,
        ]
    )


    # --------------------------------------------------------
    # Native categorical
    # --------------------------------------------------------

    training_native = (
        training_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    scoring_native = (
        scoring_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    unseen_counts = {}


    for column in GEO_NUMERIC_PREDICTORS:

        training_native[
            column
        ] = pd.to_numeric(
            training_native[
                column
            ],
            errors="raise",
        ).astype(float)


        scoring_native[
            column
        ] = pd.to_numeric(
            scoring_native[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in GEO_CATEGORICAL_PREDICTORS:

        train_values = (
            training_native[
                column
            ]
            .astype(str)
        )


        scoring_values = (
            scoring_native[
                column
            ]
            .astype(str)
        )


        categories = list(
            pd.unique(
                train_values
            )
        )


        unseen_counts[
            column
        ] = int(
            (
                ~scoring_values.isin(
                    categories
                )
            )
            .sum()
        )


        training_native[
            column
        ] = pd.Categorical(
            train_values,
            categories=categories,
            ordered=False,
        )


        scoring_native[
            column
        ] = pd.Categorical(
            scoring_values,
            categories=categories,
            ordered=False,
        )


    # --------------------------------------------------------
    # CatBoost native
    # --------------------------------------------------------

    training_catboost = (
        training_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    scoring_catboost = (
        scoring_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    for column in GEO_NUMERIC_PREDICTORS:

        training_catboost[
            column
        ] = pd.to_numeric(
            training_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


        scoring_catboost[
            column
        ] = pd.to_numeric(
            scoring_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in GEO_CATEGORICAL_PREDICTORS:

        training_catboost[
            column
        ] = (
            training_catboost[
                column
            ]
            .astype(str)
        )


        scoring_catboost[
            column
        ] = (
            scoring_catboost[
                column
            ]
            .astype(str)
        )


    return {
        "Scaled OHE":
            (
                training_scaled,
                scoring_scaled,
            ),

        "Unscaled OHE":
            (
                training_unscaled,
                scoring_unscaled,
            ),

        "Native categorical":
            (
                training_native,
                scoring_native,
            ),

        "CatBoost native":
            (
                training_catboost,
                scoring_catboost,
            ),

        "__UNSEEN_COUNTS":
            unseen_counts,
    }


# ============================================================
# 18. MODEL CONSTRUCTOR
# ============================================================

def instantiate_candidate(
    candidate_row,
):

    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    fixed_params = (
        parse_parameter_dictionary(
            candidate_row[
                "Fixed_Params_JSON"
            ]
        )
    )


    tuned_params = (
        parse_parameter_dictionary(
            candidate_row[
                "Tuned_Params_JSON"
            ]
        )
    )


    params = {
        **fixed_params,
        **tuned_params,
    }


    # --------------------------------------------------------
    # Execution-only thread overrides.
    # These DO NOT alter scientific hyperparameters.
    # --------------------------------------------------------

    if model_name in [
        "Random Forest",
        "ExtraTrees",
        "XGBoost",
        "LightGBM",
    ]:

        params[
            "n_jobs"
        ] = EXECUTION_THREADS


    if model_name == "CatBoost":

        params[
            "thread_count"
        ] = EXECUTION_THREADS


    # --------------------------------------------------------
    # Models
    # --------------------------------------------------------

    if model_name == "ElasticNet logistic regression":

        return LogisticRegression(
            **params
        )


    if model_name == "RBF SVM":

        probability_value = params.pop(
            "probability",
            False,
        )


        if probability_value not in [
            False,
            None,
        ]:

            raise RuntimeError(
                "Frozen SVM unexpectedly requests probability=True."
            )


        return SVC(
            **params
        )


    if model_name == "Random Forest":

        return RandomForestClassifier(
            **params
        )


    if model_name == "ExtraTrees":

        return ExtraTreesClassifier(
            **params
        )


    if model_name == "AdaBoost":

        depth_aliases = {
            "estimatormaxdepth",
            "baseestimatormaxdepth",
            "stumpmaxdepth",
            "decisiontreemaxdepth",
            "maxdepth",
        }


        depth_keys = [
            key

            for key
            in list(
                params.keys()
            )

            if normalize_name(
                key
            )
            in depth_aliases
        ]


        if len(
            depth_keys
        ) != 1:

            raise RuntimeError(
                "Could not identify AdaBoost tree depth.\n"
                f"{params}"
            )


        depth = int(
            params.pop(
                depth_keys[0]
            )
        )


        if (
            "estimator"
            in params
            and
            not hasattr(
                params[
                    "estimator"
                ],
                "fit",
            )
        ):

            params.pop(
                "estimator"
            )


        tree = DecisionTreeClassifier(
            max_depth=depth,
            random_state=SEED,
        )


        return AdaBoostClassifier(
            estimator=tree,
            **params,
        )


    if model_name == "GradientBoosting":

        return GradientBoostingClassifier(
            **params
        )


    if model_name == "HistGradientBoosting":

        return HistGradientBoostingClassifier(
            **params
        )


    if model_name == "XGBoost":

        return XGBClassifier(
            **params
        )


    if model_name == "LightGBM":

        return LGBMClassifier(
            **params
        )


    if model_name == "CatBoost":

        return CatBoostClassifier(
            **params
        )


    raise RuntimeError(
        f"Unknown model family: {model_name}"
    )


# ============================================================
# 19. FIT MODEL
# ============================================================

def fit_candidate(
    estimator,
    model_name,
    X_train,
    y_train,
    training_weight,
):

    with warnings.catch_warnings():

        warnings.simplefilter(
            "error",
            ConvergenceWarning,
        )


        if model_name == "CatBoost":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
                cat_features=GEO_CATEGORICAL_PREDICTORS,
                verbose=False,
            )


        elif model_name == "LightGBM":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
                categorical_feature=GEO_CATEGORICAL_PREDICTORS,
            )


        else:

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
            )


    return estimator


# ============================================================
# 20. POSITIVE-CLASS RAW SCORE
# ============================================================

def positive_score(
    estimator,
    model_name,
    X,
):

    classes = np.asarray(
        estimator.classes_
    )


    if set(
        classes.tolist()
    ) != {
        0,
        1,
    }:

        raise RuntimeError(
            f"{model_name}: unexpected classes {classes}"
        )


    if model_name == "RBF SVM":

        score = np.asarray(
            estimator.decision_function(
                X
            ),
            dtype=float,
        ).reshape(-1)


    else:

        probability = np.asarray(
            estimator.predict_proba(
                X
            ),
            dtype=float,
        )


        positive_index = int(
            np.where(
                classes
                ==
                1
            )[0][0]
        )


        score = probability[
            :,
            positive_index
        ]


    if not np.isfinite(
        score
    ).all():

        raise RuntimeError(
            f"{model_name}: non-finite raw score."
        )


    return score


# ============================================================
# 21. REPRESENTATION LOOKUP
# ============================================================

def get_representation(
    representations,
    model_name,
    preprocessing_name,
):

    if model_name == "CatBoost":

        return representations[
            "CatBoost native"
        ]


    if (
        preprocessing_name
        not in representations
    ):

        raise RuntimeError(
            f"Unknown preprocessing: {preprocessing_name}"
        )


    return representations[
        preprocessing_name
    ]


# ============================================================
# 22. BUILD FIVE-FOLD DEVELOPMENT CACHE
# ============================================================

def build_cv_cache(
    stage_development,
):

    cache = {}


    folds = sorted(
        stage_development[
            "__DEVELOPMENT_CV_FOLD"
        ]
        .unique()
        .tolist()
    )


    if folds != [
        0,
        1,
        2,
        3,
        4,
    ]:

        raise RuntimeError(
            f"Expected folds 0-4, observed {folds}"
        )


    for fold in folds:

        validation_mask = (
            stage_development[
                "__DEVELOPMENT_CV_FOLD"
            ]
            ==
            fold
        )


        train_positions = np.where(
            ~validation_mask.to_numpy()
        )[0]


        validation_positions = np.where(
            validation_mask.to_numpy()
        )[0]


        train = (
            stage_development.iloc[
                train_positions
            ]
            .copy()
        )


        validation = (
            stage_development.iloc[
                validation_positions
            ]
            .copy()
        )


        y_train = (
            train[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        y_validation = (
            validation[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        if set(
            np.unique(
                y_train
            )
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"Fold {fold}: training lacks both classes."
            )


        if set(
            np.unique(
                y_validation
            )
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"Fold {fold}: validation lacks both classes."
            )


        cache[
            fold
        ] = {
            "train_positions":
                train_positions,

            "validation_positions":
                validation_positions,

            "y_train":
                y_train,

            "y_validation":
                y_validation,

            "training_weight":
                normalized_training_weight(
                    train
                ),

            "validation_weight":
                validation[
                    "SURVEY_WEIGHT"
                ]
                .to_numpy(
                    dtype=float
                ),

            "representations":
                build_representations(
                    train,
                    validation,
                ),
        }


    return cache


# ============================================================
# 23. FAMILY CHECKPOINT PATHS
# ============================================================

def family_paths(
    stage_name,
    model_name,
):

    division_slug = safe_name(
        TARGET_DIVISION
    )

    stage_slug = safe_name(
        stage_name
    )

    model_slug = safe_name(
        model_name
    )


    prefix = (
        f"STEP23_GEOGRAPHIC_{division_slug}_"
        f"{stage_slug}_{model_slug}"
    )


    return {
        "metrics":
            (
                DIRS["geographic"]
                /
                f"{prefix}_candidate_metrics.csv"
            ),

        "oof":
            (
                DIRS["geographic"]
                /
                f"{prefix}_candidate_OOF_scores.npz"
            ),

        "manifest":
            (
                DIRS["logs"]
                /
                f"{prefix}_family_checkpoint.json"
            ),
    }


# ============================================================
# 24. FAMILY CHECKPOINT VALIDITY
# ============================================================

def family_checkpoint_valid(
    stage_name,
    model_name,
    stage_development,
):

    paths = family_paths(
        stage_name,
        model_name,
    )


    if not all(
        path.exists()
        for path in paths.values()
    ):

        return False


    try:

        manifest = read_json(
            paths[
                "manifest"
            ]
        )

    except Exception:

        return False


    required = {
        "status":
            "STEP23_FAMILY_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "model":
            model_name,

        "stage_n":
            int(
                len(
                    stage_development
                )
            ),

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),

        "primary_master_sha256":
            EXPECTED_PRIMARY_SHA256,

        "candidate_registry_sha256":
            EXPECTED_CANDIDATE_SHA256,
    }


    for key, expected in required.items():

        if manifest.get(
            key
        ) != expected:

            return False


    if (
        manifest.get(
            "metrics_sha256"
        )
        !=
        sha256_file(
            paths[
                "metrics"
            ]
        )
    ):

        return False


    if (
        manifest.get(
            "oof_sha256"
        )
        !=
        sha256_file(
            paths[
                "oof"
            ]
        )
    ):

        return False


    return True


# ============================================================
# 25. RUN / LOAD ONE MODEL FAMILY
# ============================================================

def evaluate_family(
    stage_name,
    stage_development,
    cv_cache,
    model_name,
):

    paths = family_paths(
        stage_name,
        model_name,
    )


    # --------------------------------------------------------
    # Load checkpoint
    # --------------------------------------------------------

    if family_checkpoint_valid(
        stage_name,
        model_name,
        stage_development,
    ):

        metrics = pd.read_csv(
            paths[
                "metrics"
            ]
        )


        loaded = np.load(
            paths[
                "oof"
            ],
            allow_pickle=False,
        )


        ids = (
            loaded[
                "candidate_ids"
            ]
            .astype(str)
            .tolist()
        )


        score_matrix = loaded[
            "scores"
        ]


        if score_matrix.shape != (
            12,
            len(
                stage_development
            ),
        ):

            raise RuntimeError(
                f"Invalid OOF checkpoint shape for "
                f"{stage_name} / {model_name}."
            )


        score_lookup = {
            candidate_id:
                score_matrix[
                    index
                ].copy()

            for index, candidate_id
            in enumerate(ids)
        }


        print(
            f"    {model_name}: checkpoint loaded"
        )


        return (
            metrics,
            score_lookup,
        )


    # --------------------------------------------------------
    # Fresh family execution
    # --------------------------------------------------------

    family_candidates = (
        candidate_registry.loc[
            candidate_registry[
                "Model"
            ]
            ==
            model_name
        ]
        .copy()
        .sort_values(
            "Deterministic_Tie_Rank"
        )
        .reset_index(
            drop=True
        )
    )


    if len(
        family_candidates
    ) != 12:

        raise RuntimeError(
            f"{model_name}: expected 12 candidates."
        )


    n = len(
        stage_development
    )


    y_all = (
        stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        )
    )


    weight_all = (
        stage_development[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    rows = []

    score_lookup = {}


    for _, candidate_row in (
        family_candidates.iterrows()
    ):

        candidate_id = str(
            candidate_row[
                "Candidate_ID"
            ]
        )


        oof_score = np.full(
            n,
            np.nan,
            dtype=float,
        )


        try:

            for fold in [
                0,
                1,
                2,
                3,
                4,
            ]:

                fold_info = (
                    cv_cache[
                        fold
                    ]
                )


                X_train, X_validation = (
                    get_representation(
                        representations=fold_info[
                            "representations"
                        ],
                        model_name=model_name,
                        preprocessing_name=candidate_row[
                            "Preprocessing"
                        ],
                    )
                )


                estimator = (
                    instantiate_candidate(
                        candidate_row
                    )
                )


                estimator = (
                    fit_candidate(
                        estimator=estimator,
                        model_name=model_name,
                        X_train=X_train,
                        y_train=fold_info[
                            "y_train"
                        ],
                        training_weight=fold_info[
                            "training_weight"
                        ],
                    )
                )


                score = positive_score(
                    estimator=estimator,
                    model_name=model_name,
                    X=X_validation,
                )


                oof_score[
                    fold_info[
                        "validation_positions"
                    ]
                ] = score


            if not np.isfinite(
                oof_score
            ).all():

                raise RuntimeError(
                    "Incomplete OOF prediction."
                )


            weighted_ap = float(
                average_precision_score(
                    y_all,
                    oof_score,
                    sample_weight=weight_all,
                )
            )


            weighted_auc = float(
                roc_auc_score(
                    y_all,
                    oof_score,
                    sample_weight=weight_all,
                )
            )


            status = "SUCCESS"

            error_message = None


            score_lookup[
                candidate_id
            ] = oof_score


        except Exception as exc:

            weighted_ap = np.nan

            weighted_auc = np.nan

            status = "FAILED"

            error_message = (
                f"{type(exc).__name__}: {exc}"
            )


        rows.append(
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Deterministic_Tie_Rank":
                    int(
                        candidate_row[
                            "Deterministic_Tie_Rank"
                        ]
                    ),

                "Development_CV_Weighted_AP":
                    weighted_ap,

                "Development_CV_Weighted_ROC_AUC":
                    weighted_auc,

                "Status":
                    status,

                "Error":
                    error_message,
            }
        )


    metrics = pd.DataFrame(
        rows
    )


    failed = metrics.loc[
        metrics[
            "Status"
        ]
        !=
        "SUCCESS"
    ]


    if len(
        failed
    ) > 0:

        raise RuntimeError(
            f"\nMODEL FAMILY FAILURE\n"
            f"Division: {TARGET_DIVISION}\n"
            f"Stage: {stage_name}\n"
            f"Model: {model_name}\n\n"
            f"{failed[['Candidate_ID','Error']].to_string(index=False)}"
        )


    if len(
        score_lookup
    ) != 12:

        raise RuntimeError(
            f"{model_name}: expected 12 successful OOF scores."
        )


    # --------------------------------------------------------
    # Save family checkpoint
    # --------------------------------------------------------

    ordered_ids = (
        family_candidates[
            "Candidate_ID"
        ]
        .astype(str)
        .tolist()
    )


    score_matrix = np.vstack(
        [
            score_lookup[
                candidate_id
            ]

            for candidate_id
            in ordered_ids
        ]
    )


    metrics.to_csv(
        paths[
            "metrics"
        ],
        index=False,
    )


    np.savez_compressed(
        paths[
            "oof"
        ],
        candidate_ids=np.asarray(
            ordered_ids,
            dtype="U64",
        ),
        scores=score_matrix,
    )


    manifest = {
        "status":
            "STEP23_FAMILY_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "execution_time":
            dt.datetime.now().isoformat(
                timespec="seconds"
            ),

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "model":
            model_name,

        "stage_n":
            int(
                len(
                    stage_development
                )
            ),

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),

        "primary_master_sha256":
            EXPECTED_PRIMARY_SHA256,

        "candidate_registry_sha256":
            EXPECTED_CANDIDATE_SHA256,

        "execution_threads":
            EXECUTION_THREADS,

        "scientific_hyperparameters_changed":
            False,

        "metrics_sha256":
            sha256_file(
                paths[
                    "metrics"
                ]
            ),

        "oof_sha256":
            sha256_file(
                paths[
                    "oof"
                ]
            ),
    }


    with open(
        paths[
            "manifest"
        ],
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            manifest,
            handle,
            indent=2,
            ensure_ascii=False,
        )


    best_family_ap = float(
        metrics[
            "Development_CV_Weighted_AP"
        ].max()
    )


    print(
        f"    {model_name}: complete"
        f" | best development-CV AP="
        f"{best_family_ap:.6f}"
    )


    return (
        metrics,
        score_lookup,
    )


# ============================================================
# 26. CALIBRATION
# ============================================================

def calibration_predictor(
    raw_score,
    model_name,
):

    raw_score = np.asarray(
        raw_score,
        dtype=float,
    )


    if model_name == "RBF SVM":

        return raw_score


    p = np.clip(
        raw_score,
        CALIBRATION_PROBABILITY_CLIP,
        1.0
        -
        CALIBRATION_PROBABILITY_CLIP,
    )


    return np.log(
        p
        /
        (
            1.0
            -
            p
        )
    )


def fit_calibrator(
    raw_score,
    y,
    weight,
    model_name,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    weight = np.asarray(
        weight,
        dtype=float,
    )


    normalized_weight = (
        weight
        /
        np.mean(weight)
    )


    z = calibration_predictor(
        raw_score,
        model_name,
    )


    def objective(theta):

        alpha = theta[0]

        beta = theta[1]


        eta = (
            alpha
            +
            beta * z
        )


        loss = (
            np.sum(
                normalized_weight
                *
                (
                    np.logaddexp(
                        0.0,
                        eta,
                    )
                    -
                    y * eta
                )
            )
            /
            np.sum(
                normalized_weight
            )
        )


        residual = (
            expit(eta)
            -
            y
        )


        gradient = np.array(
            [
                np.sum(
                    normalized_weight
                    *
                    residual
                )
                /
                np.sum(
                    normalized_weight
                ),

                np.sum(
                    normalized_weight
                    *
                    residual
                    *
                    z
                )
                /
                np.sum(
                    normalized_weight
                ),
            ],
            dtype=float,
        )


        return (
            float(loss),
            gradient,
        )


    fit = minimize(
        objective,
        x0=np.array(
            [
                0.0,
                1.0,
            ],
            dtype=float,
        ),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter":
                5000,

            "ftol":
                1e-12,

            "gtol":
                1e-8,
        },
    )


    if not fit.success:

        raise RuntimeError(
            "Calibration optimization failed:\n"
            f"{fit.message}"
        )


    alpha = float(
        fit.x[0]
    )


    beta = float(
        fit.x[1]
    )


    calibrated = expit(
        alpha
        +
        beta
        *
        z
    )


    return {
        "alpha":
            alpha,

        "beta":
            beta,

        "iterations":
            int(
                fit.nit
            ),

        "calibrated":
            calibrated,
    }


def apply_calibrator(
    raw_score,
    model_name,
    alpha,
    beta,
):

    z = calibration_predictor(
        raw_score,
        model_name,
    )


    probability = expit(
        alpha
        +
        beta
        *
        z
    )


    if not np.isfinite(
        probability
    ).all():

        raise RuntimeError(
            "Non-finite calibrated probability."
        )


    return probability


# ============================================================
# 27. THRESHOLD OPTIMIZATION
# ============================================================

def threshold_candidates(
    probability,
):

    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        THRESHOLD_PROBABILITY_CLIP,
        1.0
        -
        THRESHOLD_PROBABILITY_CLIP,
    )


    unique_p = np.unique(
        p
    )


    if len(
        unique_p
    ) == 1:

        return np.array(
            [
                0.0,
                1.0,
            ]
        )


    midpoint = (
        unique_p[:-1]
        +
        (
            unique_p[1:]
            -
            unique_p[:-1]
        )
        /
        2.0
    )


    return np.unique(
        np.concatenate(
            [
                np.array(
                    [0.0]
                ),

                midpoint,

                np.array(
                    [1.0]
                ),
            ]
        )
    )


def threshold_metric(
    y,
    probability,
    weight,
    threshold,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    probability = np.asarray(
        probability,
        dtype=float,
    )


    weight = np.asarray(
        weight,
        dtype=float,
    )


    prediction = (
        probability
        >=
        threshold
    ).astype(int)


    positive = (
        y
        ==
        1
    )


    negative = (
        y
        ==
        0
    )


    tp = weight[
        positive
        &
        (
            prediction
            ==
            1
        )
    ].sum()


    fn = weight[
        positive
        &
        (
            prediction
            ==
            0
        )
    ].sum()


    tn = weight[
        negative
        &
        (
            prediction
            ==
            0
        )
    ].sum()


    fp = weight[
        negative
        &
        (
            prediction
            ==
            1
        )
    ].sum()


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    balanced_accuracy = (
        sensitivity
        +
        specificity
    ) / 2.0


    return {
        "threshold":
            float(
                threshold
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "balance_floor":
            float(
                min(
                    sensitivity,
                    specificity,
                )
            ),
    }


def optimize_threshold(
    y,
    probability,
    weight,
):

    rows = [
        threshold_metric(
            y=y,
            probability=probability,
            weight=weight,
            threshold=threshold,
        )

        for threshold
        in threshold_candidates(
            probability
        )
    ]


    result = pd.DataFrame(
        rows
    )


    best_ba = float(
        result[
            "balanced_accuracy"
        ].max()
    )


    tied = result.loc[
        np.isclose(
            result[
                "balanced_accuracy"
            ],
            best_ba,
            rtol=0.0,
            atol=1e-12,
        )
    ].copy()


    best_floor = float(
        tied[
            "balance_floor"
        ].max()
    )


    tied = tied.loc[
        np.isclose(
            tied[
                "balance_floor"
            ],
            best_floor,
            rtol=0.0,
            atol=1e-12,
        )
    ].copy()


    selected = (
        tied
        .sort_values(
            "threshold"
        )
        .iloc[0]
    )


    return {
        key:
            float(
                selected[
                    key
                ]
            )

        for key in [
            "threshold",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "balance_floor",
        ]
    }


# ============================================================
# 28. REFIT WINNER
# ============================================================

def refit_winner(
    stage_development,
    candidate_row,
):

    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    representations = (
        build_representations(
            stage_development,
            heldout,
        )
    )


    X_train, X_heldout = (
        get_representation(
            representations=representations,
            model_name=model_name,
            preprocessing_name=candidate_row[
                "Preprocessing"
            ],
        )
    )


    estimator = (
        instantiate_candidate(
            candidate_row
        )
    )


    estimator = (
        fit_candidate(
            estimator=estimator,
            model_name=model_name,
            X_train=X_train,
            y_train=stage_development[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            ),
            training_weight=normalized_training_weight(
                stage_development
            ),
        )
    )


    score = positive_score(
        estimator=estimator,
        model_name=model_name,
        X=X_heldout,
    )


    return (
        score,
        representations[
            "__UNSEEN_COUNTS"
        ],
    )


# ============================================================
# 29. STAGE CHECKPOINT PATHS
# ============================================================

def stage_paths(
    stage_name,
):

    division_slug = safe_name(
        TARGET_DIVISION
    )


    stage_slug = safe_name(
        stage_name
    )


    prefix = (
        f"STEP23_GEOGRAPHIC_{division_slug}_"
        f"{stage_slug}"
    )


    return {
        "candidate_metrics":
            (
                DIRS["geographic"]
                /
                f"{prefix}_all_candidate_metrics.csv"
            ),

        "winner":
            (
                DIRS["geographic"]
                /
                f"{prefix}_winner.csv"
            ),

        "calibration":
            (
                DIRS["geographic"]
                /
                f"{prefix}_calibration_threshold.csv"
            ),

        "development_oof":
            (
                DIRS["geographic"]
                /
                f"{prefix}_winner_development_OOF_RESTRICTED.csv"
            ),

        "heldout_stage":
            (
                DIRS["geographic"]
                /
                f"{prefix}_heldout_stage_predictions_RESTRICTED.csv"
            ),

        "unseen":
            (
                DIRS["qc"]
                /
                f"{prefix}_unseen_category_qc.csv"
            ),

        "manifest":
            (
                DIRS["logs"]
                /
                f"{prefix}_stage_checkpoint.json"
            ),
    }


# ============================================================
# 30. STAGE CHECKPOINT VALIDITY
# ============================================================

def stage_checkpoint_valid(
    stage_name,
    stage_development,
):

    paths = stage_paths(
        stage_name
    )


    if not all(
        path.exists()
        for path in paths.values()
    ):

        return False


    try:

        manifest = read_json(
            paths[
                "manifest"
            ]
        )

    except Exception:

        return False


    required = {
        "status":
            "STEP23_STAGE_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "stage_n":
            int(
                len(
                    stage_development
                )
            ),

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),

        "primary_master_sha256":
            EXPECTED_PRIMARY_SHA256,

        "candidate_registry_sha256":
            EXPECTED_CANDIDATE_SHA256,
    }


    for key, expected in required.items():

        if manifest.get(
            key
        ) != expected:

            return False


    file_map = {
        "candidate_metrics_sha256":
            "candidate_metrics",

        "winner_sha256":
            "winner",

        "calibration_sha256":
            "calibration",

        "development_oof_sha256":
            "development_oof",

        "heldout_stage_sha256":
            "heldout_stage",

        "unseen_sha256":
            "unseen",
    }


    for hash_key, path_key in (
        file_map.items()
    ):

        if (
            manifest.get(
                hash_key
            )
            !=
            sha256_file(
                paths[
                    path_key
                ]
            )
        ):

            return False


    return True


# ============================================================
# 31. RUN / LOAD ONE STAGE
# ============================================================

def execute_stage(
    stage_name,
):

    stage_development = (
        prepare_stage(
            development,
            stage_name,
        )
    )


    paths = stage_paths(
        stage_name
    )


    # --------------------------------------------------------
    # Load completed stage
    # --------------------------------------------------------

    if stage_checkpoint_valid(
        stage_name,
        stage_development,
    ):

        print(
            f"\n{stage_name}: "
            "valid stage checkpoint found — loading."
        )


        return {
            "candidate_metrics":
                pd.read_csv(
                    paths[
                        "candidate_metrics"
                    ]
                ),

            "winner":
                pd.read_csv(
                    paths[
                        "winner"
                    ]
                ),

            "calibration":
                pd.read_csv(
                    paths[
                        "calibration"
                    ]
                ),

            "development_oof":
                pd.read_csv(
                    paths[
                        "development_oof"
                    ]
                ),

            "heldout_stage":
                pd.read_csv(
                    paths[
                        "heldout_stage"
                    ]
                ),

            "unseen":
                pd.read_csv(
                    paths[
                        "unseen"
                    ]
                ),
        }


    print(
        "\n"
        + "-" * 78
    )

    print(
        f"{TARGET_DIVISION} | {stage_name}"
    )

    print(
        f"Development N = {len(stage_development)}"
    )

    print(
        f"Held-out scoring N = {len(heldout)}"
    )

    print(
        "-" * 78
    )


    # --------------------------------------------------------
    # Build preprocessing once for each CV fold
    # --------------------------------------------------------

    cv_cache = build_cv_cache(
        stage_development
    )


    family_metric_frames = []

    all_oof_scores = {}


    # --------------------------------------------------------
    # Each family checkpointed independently
    # --------------------------------------------------------

    for family_number, model_name in enumerate(
        MODEL_ORDER,
        start=1,
    ):

        print(
            f"  [{family_number}/10] {model_name}"
        )


        family_metrics, family_scores = (
            evaluate_family(
                stage_name=stage_name,
                stage_development=stage_development,
                cv_cache=cv_cache,
                model_name=model_name,
            )
        )


        family_metric_frames.append(
            family_metrics
        )


        all_oof_scores.update(
            family_scores
        )


    candidate_metrics = pd.concat(
        family_metric_frames,
        ignore_index=True,
    )


    if len(
        candidate_metrics
    ) != 120:

        raise RuntimeError(
            f"{stage_name}: expected 120 candidate rows."
        )


    if (
        candidate_metrics[
            "Status"
        ]
        !=
        "SUCCESS"
    ).any():

        raise RuntimeError(
            f"{stage_name}: candidate failure detected."
        )


    if len(
        all_oof_scores
    ) != 120:

        raise RuntimeError(
            f"{stage_name}: expected 120 OOF score vectors."
        )


    # ========================================================
    # 32. JOINT WINNER
    # ========================================================

    best_ap = float(
        candidate_metrics[
            "Development_CV_Weighted_AP"
        ].max()
    )


    tied_ap = candidate_metrics.loc[
        np.isclose(
            candidate_metrics[
                "Development_CV_Weighted_AP"
            ],
            best_ap,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    best_auc = float(
        tied_ap[
            "Development_CV_Weighted_ROC_AUC"
        ].max()
    )


    tied_auc = tied_ap.loc[
        np.isclose(
            tied_ap[
                "Development_CV_Weighted_ROC_AUC"
            ],
            best_auc,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    winner = (
        tied_auc
        .sort_values(
            "Deterministic_Tie_Rank"
        )
        .iloc[0]
    )


    candidate_id = str(
        winner[
            "Candidate_ID"
        ]
    )


    model_name = str(
        winner[
            "Model"
        ]
    )


    registry_match = (
        candidate_registry.loc[
            candidate_registry[
                "Candidate_ID"
            ]
            .astype(str)
            ==
            candidate_id
        ]
    )


    if len(
        registry_match
    ) != 1:

        raise RuntimeError(
            "Could not uniquely identify joint winner."
        )


    candidate_row = (
        registry_match.iloc[0]
    )


    winner_oof = (
        all_oof_scores[
            candidate_id
        ]
    )


    print(
        f"\n  JOINT WINNER: "
        f"{model_name} | {candidate_id}"
    )

    print(
        f"  Development-CV AP  = "
        f"{float(winner['Development_CV_Weighted_AP']):.6f}"
    )

    print(
        f"  Development-CV AUC = "
        f"{float(winner['Development_CV_Weighted_ROC_AUC']):.6f}"
    )


    # ========================================================
    # 33. CALIBRATION
    # ========================================================

    calibration = fit_calibrator(
        raw_score=winner_oof,
        y=stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        ),
        weight=stage_development[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        ),
        model_name=model_name,
    )


    development_calibrated = (
        calibration[
            "calibrated"
        ]
    )


    # ========================================================
    # 34. THRESHOLD
    # ========================================================

    threshold_result = optimize_threshold(
        y=stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        ),
        probability=development_calibrated,
        weight=stage_development[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        ),
    )


    threshold = float(
        threshold_result[
            "threshold"
        ]
    )


    print(
        f"  Optimized threshold = "
        f"{threshold:.6f}"
    )


    # ========================================================
    # 35. FINAL REFIT + HELDOUT SCORE
    # ========================================================

    (
        heldout_raw_score,
        unseen_counts,
    ) = refit_winner(
        stage_development=stage_development,
        candidate_row=candidate_row,
    )


    heldout_probability = apply_calibrator(
        raw_score=heldout_raw_score,
        model_name=model_name,
        alpha=calibration[
            "alpha"
        ],
        beta=calibration[
            "beta"
        ],
    )


    # ========================================================
    # 36. SAVE STAGE OUTPUTS
    # ========================================================

    winner_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Development_CV_Weighted_AP":
                    float(
                        winner[
                            "Development_CV_Weighted_AP"
                        ]
                    ),

                "Development_CV_Weighted_ROC_AUC":
                    float(
                        winner[
                            "Development_CV_Weighted_ROC_AUC"
                        ]
                    ),

                "Tuned_Params_JSON":
                    candidate_row[
                        "Tuned_Params_JSON"
                    ],

                "Fixed_Params_JSON":
                    candidate_row[
                        "Fixed_Params_JSON"
                    ],

                "Execution_Threads":
                    EXECUTION_THREADS,
            }
        ]
    )


    calibration_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Calibration_Alpha":
                    float(
                        calibration[
                            "alpha"
                        ]
                    ),

                "Calibration_Beta":
                    float(
                        calibration[
                            "beta"
                        ]
                    ),

                "Calibration_Iterations":
                    int(
                        calibration[
                            "iterations"
                        ]
                    ),

                "Optimized_Threshold":
                    threshold,

                "Development_Threshold_Balanced_Accuracy":
                    float(
                        threshold_result[
                            "balanced_accuracy"
                        ]
                    ),

                "Development_Threshold_Sensitivity":
                    float(
                        threshold_result[
                            "sensitivity"
                        ]
                    ),

                "Development_Threshold_Specificity":
                    float(
                        threshold_result[
                            "specificity"
                        ]
                    ),
            }
        ]
    )


    development_oof_df = pd.DataFrame(
        {
            "_ROW_KEY":
                stage_development[
                    "__ROW_KEY"
                ].to_numpy(),

            "Heldout_Division":
                TARGET_DIVISION,

            "Development_CV_Fold":
                stage_development[
                    "__DEVELOPMENT_CV_FOLD"
                ]
                .to_numpy(
                    dtype=int
                ),

            "Stage":
                stage_name,

            "Model":
                model_name,

            "Candidate_ID":
                candidate_id,

            "True_Target":
                stage_development[
                    "__TARGET"
                ]
                .to_numpy(
                    dtype=int
                ),

            "SURVEY_WEIGHT":
                stage_development[
                    "SURVEY_WEIGHT"
                ]
                .to_numpy(
                    dtype=float
                ),

            "Development_OOF_Raw_Score":
                winner_oof,

            "Development_OOF_Calibrated_Probability":
                development_calibrated,
        }
    )


    heldout_stage_df = pd.DataFrame(
        {
            "_ROW_KEY":
                heldout[
                    "__ROW_KEY"
                ]
                .to_numpy(),

            "Heldout_Division":
                TARGET_DIVISION,

            "Stage":
                stage_name,

            "Model":
                model_name,

            "Candidate_ID":
                candidate_id,

            "Heldout_Raw_Score":
                heldout_raw_score,

            "Heldout_Calibrated_Probability":
                heldout_probability,

            "Optimized_Threshold":
                threshold,
        }
    )


    unseen_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Predictor":
                    predictor,

                "Heldout_Unseen_Category_Rows":
                    int(
                        count
                    ),
            }

            for predictor, count
            in unseen_counts.items()
        ]
    )


    candidate_metrics.to_csv(
        paths[
            "candidate_metrics"
        ],
        index=False,
    )


    winner_df.to_csv(
        paths[
            "winner"
        ],
        index=False,
    )


    calibration_df.to_csv(
        paths[
            "calibration"
        ],
        index=False,
    )


    development_oof_df.to_csv(
        paths[
            "development_oof"
        ],
        index=False,
    )


    heldout_stage_df.to_csv(
        paths[
            "heldout_stage"
        ],
        index=False,
    )


    unseen_df.to_csv(
        paths[
            "unseen"
        ],
        index=False,
    )


    manifest = {
        "status":
            "STEP23_STAGE_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "execution_time":
            dt.datetime.now().isoformat(
                timespec="seconds"
            ),

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "stage_n":
            int(
                len(
                    stage_development
                )
            ),

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),

        "primary_master_sha256":
            EXPECTED_PRIMARY_SHA256,

        "candidate_registry_sha256":
            EXPECTED_CANDIDATE_SHA256,

        "execution_threads":
            EXECUTION_THREADS,

        "scientific_hyperparameters_changed":
            False,

        "candidate_metrics_sha256":
            sha256_file(
                paths[
                    "candidate_metrics"
                ]
            ),

        "winner_sha256":
            sha256_file(
                paths[
                    "winner"
                ]
            ),

        "calibration_sha256":
            sha256_file(
                paths[
                    "calibration"
                ]
            ),

        "development_oof_sha256":
            sha256_file(
                paths[
                    "development_oof"
                ]
            ),

        "heldout_stage_sha256":
            sha256_file(
                paths[
                    "heldout_stage"
                ]
            ),

        "unseen_sha256":
            sha256_file(
                paths[
                    "unseen"
                ]
            ),

        "heldout_performance_calculated":
            False,
    }


    with open(
        paths[
            "manifest"
        ],
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            manifest,
            handle,
            indent=2,
            ensure_ascii=False,
        )


    print(
        f"  {stage_name} CHECKPOINT SAVED."
    )


    return {
        "candidate_metrics":
            candidate_metrics,

        "winner":
            winner_df,

        "calibration":
            calibration_df,

        "development_oof":
            development_oof_df,

        "heldout_stage":
            heldout_stage_df,

        "unseen":
            unseen_df,
    }


# ============================================================
# 37. RUN BOTH STAGES
# ============================================================

stage1_result = execute_stage(
    "Stage 1"
)


stage2_result = execute_stage(
    "Stage 2"
)


# ============================================================
# 38. COMBINE STAGE 1 + STAGE 2
# ============================================================

stage1_heldout = (
    stage1_result[
        "heldout_stage"
    ]
    .copy()
)


stage2_heldout = (
    stage2_result[
        "heldout_stage"
    ]
    .copy()
)


if len(
    stage1_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-1 held-out prediction count mismatch."
    )


if len(
    stage2_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-2 held-out prediction count mismatch."
    )


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "ANC_CLASS":
                "ANC_CLASS_TRUE",
        }
    )
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked[
    "Heldout_Division"
] = TARGET_DIVISION


q0 = locked[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(
    dtype=float
)


t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(
    dtype=float
)


t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(
    dtype=float
)


locked[
    "P_NO_ANC"
] = q0


locked[
    "P_ANC_1_7"
] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)


locked[
    "P_ANC_GE8"
] = (
    (1.0 - q0)
    *
    q2
)


locked[
    "P_SUM"
] = (
    locked[
        "P_NO_ANC"
    ]
    +
    locked[
        "P_ANC_1_7"
    ]
    +
    locked[
        "P_ANC_GE8"
    ]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0
    >=
    t1,
    0,
    np.where(
        q2
        >=
        t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 39. DIVISION-LEVEL QC
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result[
            "candidate_metrics"
        ],
        stage2_result[
            "candidate_metrics"
        ],
    ],
    ignore_index=True,
)


winners_all = pd.concat(
    [
        stage1_result[
            "winner"
        ],
        stage2_result[
            "winner"
        ],
    ],
    ignore_index=True,
)


calibration_all = pd.concat(
    [
        stage1_result[
            "calibration"
        ],
        stage2_result[
            "calibration"
        ],
    ],
    ignore_index=True,
)


development_oof_all = pd.concat(
    [
        stage1_result[
            "development_oof"
        ],
        stage2_result[
            "development_oof"
        ],
    ],
    ignore_index=True,
)


unseen_all = pd.concat(
    [
        stage1_result[
            "unseen"
        ],
        stage2_result[
            "unseen"
        ],
    ],
    ignore_index=True,
)


if len(
    candidate_metrics_all
) != 240:

    raise RuntimeError(
        "Expected 240 candidate summaries "
        "for this division."
    )


if (
    candidate_metrics_all[
        "Status"
    ]
    !=
    "SUCCESS"
).any():

    raise RuntimeError(
        "At least one candidate failed."
    )


if len(
    winners_all
) != 2:

    raise RuntimeError(
        "Expected exactly two stage winners."
    )


if len(
    calibration_all
) != 2:

    raise RuntimeError(
        "Expected exactly two calibration/threshold rows."
    )


if len(
    locked
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Held-out locked prediction count mismatch."
    )


if locked[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate held-out prediction row."
    )


if not np.isfinite(
    locked[
        [
            "q0_CALIBRATED",
            "q2_CALIBRATED_CONDITIONAL",
            "P_NO_ANC",
            "P_ANC_1_7",
            "P_ANC_GE8",
            "P_SUM",
        ]
    ]
    .to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Non-finite final geographic probability."
    )


if not np.allclose(
    locked[
        "P_SUM"
    ],
    1.0,
    rtol=0.0,
    atol=1e-12,
):

    raise RuntimeError(
        "Hierarchical probabilities do not sum to one."
    )


# ============================================================
# 40. FINAL DIVISION OUTPUT PATHS
# ============================================================

division_slug = safe_name(
    TARGET_DIVISION
)


prefix = (
    f"STEP23_GEOGRAPHIC_holdout_{division_slug}"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)


FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)


FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)


FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)


FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)


FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)


FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


# ============================================================
# 41. FINAL HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(
                    PRIMARY_FILE
                )
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                sha256_file(
                    OUTER_SPLIT_FILE
                )
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                sha256_file(
                    CANDIDATE_REGISTRY_FILE
                )
                ==
                EXPECTED_CANDIDATE_SHA256,
        },

        {
            "Check":
                "Division predictor excluded",

            "PASS":
                "division"
                not in
                GEO_PREDICTORS,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                len(
                    candidate_metrics_all
                )
                ==
                240,
        },

        {
            "Check":
                "All 240 candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    ).all()
                ),
        },

        {
            "Check":
                "Exactly two joint stage winners",

            "PASS":
                len(
                    winners_all
                )
                ==
                2,
        },

        {
            "Check":
                "Both stages completed",

            "PASS":
                set(
                    winners_all[
                        "Stage"
                    ]
                )
                ==
                {
                    "Stage 1",
                    "Stage 2",
                },
        },

        {
            "Check":
                "Expected held-out N recovered",

            "PASS":
                len(
                    locked
                )
                ==
                EXPECTED_HELDOUT_N,
        },

        {
            "Check":
                "Held-out rows unique",

            "PASS":
                not locked[
                    "__ROW_KEY"
                ]
                .duplicated()
                .any(),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                bool(
                    np.isfinite(
                        locked[
                            [
                                "P_NO_ANC",
                                "P_ANC_1_7",
                                "P_ANC_GE8",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                bool(
                    np.allclose(
                        locked[
                            "P_SUM"
                        ],
                        1.0,
                        rtol=0.0,
                        atol=1e-12,
                    )
                ),
        },

        {
            "Check":
                "Execution thread limit = 2",

            "PASS":
                EXECUTION_THREADS
                ==
                2,
        },

        {
            "Check":
                "Scientific hyperparameters unchanged",

            "PASS":
                True,
        },

        {
            "Check":
                "Held-out outcomes not used for tuning",

            "PASS":
                True,
        },

        {
            "Check":
                "Held-out outcomes not used for calibration",

            "PASS":
                True,
        },

        {
            "Check":
                "Held-out outcomes not used for threshold optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "No held-out model updating",

            "PASS":
                True,
        },

        {
            "Check":
                "No held-out performance metric calculated",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "STEP 23 DIVISION HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 42. SAVE FINAL DIVISION LOCK
# ============================================================

candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)


winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)


calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)


development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)


locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)


unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)


hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


final_manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        int(
            len(
                development
            )
        ),

    "heldout_n":
        int(
            len(
                heldout
            )
        ),

    "execution_threads":
        EXECUTION_THREADS,

    "scientific_hyperparameters_changed":
        False,

    "division_predictor_used":
        False,

    "candidate_count_per_stage":
        120,

    "development_cv_folds":
        5,

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "step22_protocol_sha256":
        STEP22_PROTOCOL_SHA256,

    "candidate_metrics_sha256":
        sha256_file(
            FINAL_CANDIDATE_FILE
        ),

    "winners_sha256":
        sha256_file(
            FINAL_WINNER_FILE
        ),

    "calibration_threshold_sha256":
        sha256_file(
            FINAL_CALIBRATION_FILE
        ),

    "winner_development_oof_sha256":
        sha256_file(
            FINAL_DEVELOPMENT_OOF_FILE
        ),

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "unseen_category_qc_sha256":
        sha256_file(
            FINAL_UNSEEN_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        final_manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 43. NEXT DIVISION
# ============================================================

current_index = VALID_DIVISIONS.index(
    TARGET_DIVISION
)


if current_index < (
    len(
        VALID_DIVISIONS
    )
    -
    1
):

    next_division = (
        VALID_DIVISIONS[
            current_index
            +
            1
        ]
    )

else:

    next_division = None


# ============================================================
# 44. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    f"STEP 23 — {TARGET_DIVISION} GEOGRAPHIC IECV LOCKED"
)


print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)


display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
            "Execution_Threads",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)


display(
    calibration_all
)


print(
    "\nUNSEEN CATEGORY QC:"
)


display(
    unseen_all
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nLOCKED PREDICTIONS:"
)


print(
    f"Held-out division : {TARGET_DIVISION}"
)


print(
    f"Rows              : {len(locked)}"
)


print(
    f"Unique rows       : {locked['_ROW_KEY'].nunique()}"
)


print(
    "Probability sum  : "
    f"{np.allclose(locked['P_SUM'], 1.0, rtol=0, atol=1e-12)}"
)


print(
    "Held-out performance: NOT CALCULATED"
)


print(
    "\nCHECKPOINT POLICY:"
)


print(
    "- Each completed model family is checkpointed."
)


print(
    "- Each completed stage is checkpointed."
)


print(
    "- Kernel restart will NOT repeat valid completed checkpoints."
)


print(
    "- CPU-intensive models use only 2 execution threads."
)


print(
    "- Scientific hyperparameters remain unchanged."
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)


print(
    f"Step-13 folds  : {sha256_file(OUTER_SPLIT_FILE)}"
)


print(
    f"Candidate grid : {sha256_file(CANDIDATE_REGISTRY_FILE)}"
)


print(
    f"Locked division: {sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    + "=" * 78
)


print(
    f"STATUS: STEP 23 {TARGET_DIVISION.upper()} IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


if next_division is not None:

    print(
        "\nNEXT RUN:"
    )

    print(
        f'TARGET_DIVISION = "{next_division}"'
    )

else:

    print(
        "\nALL 8 DIVISIONS SHOULD NOW BE COMPLETE."
    )

    print(
        "Next step: combine all division locks and run Step 24 evaluation."
    )


STEP 23 — LAPTOP-OPTIMIZED GEOGRAPHIC IECV
Held-out division : Barishal
Development N     : 4519
Held-out N        : 544
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Barishal | Stage 1
Development N = 4519
Held-out scoring N = 544
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.240458
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.154189
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.205837
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.205141
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.217292
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.225552
  [7/10] HistGradientBoosting
    HistGradientBoosting: complete | best developme

KeyError: '_ROW_KEY'

In [2]:
# ============================================================
# STEP 23 — BARISHAL RECOVERY / FINALIZATION ONLY
# ============================================================
#
# PURPOSE
# -------
# Recover already-completed Barishal Stage-1 and Stage-2
# checkpoints after the final merge KeyError.
#
# THIS CELL DOES NOT:
#   - fit any model
#   - tune any hyperparameter
#   - recalibrate anything
#   - re-optimize any threshold
#   - calculate held-out performance
#
# It ONLY:
#   - verifies saved stage checkpoints
#   - fixes _ROW_KEY / __ROW_KEY merge naming
#   - performs frozen hierarchical fusion
#   - writes final Barishal locked prediction files
# ============================================================


from pathlib import Path

import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. PROJECT
# ============================================================

ROOT = Path(
    r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
)


DIRS = {
    "processed":
        ROOT / "01_processed_data",

    "qc":
        ROOT / "02_quality_control",

    "protocol":
        ROOT / "03_protocol",

    "splits":
        ROOT / "04_cv_splits",

    "geographic":
        ROOT / "09_geographic_validation",

    "logs":
        ROOT / "15_logs",
}


TARGET_DIVISION = "Barishal"

EXECUTION_THREADS = 2

EXECUTION_SCHEMA_VERSION = (
    "STEP23_GEOGRAPHIC_LAPTOP_v2"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_CANDIDATE_SHA256 = (
    "f95dc93431eced694114fe8ed071a2878"
    "e2572fac53aa3b9fd9c6cb8df87c697"
)


# ============================================================
# 2. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):

            digest.update(block)

    return digest.hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(handle)


def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column
        in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(
            alias
        )

        if key in lookup:

            return lookup[
                key
            ]


    raise RuntimeError(
        "Required column not found.\n"
        f"Aliases: {aliases}\n"
        f"Available: {list(dataframe.columns)}"
    )


def canonical_row_key(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(
                dtype=float
            ),
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ).astype(
                np.int64
            ),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


# ============================================================
# 3. CORE FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)


STEP22_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_IECV_protocol.json"
)


# ============================================================
# 4. HASH CHECK
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Step-13 split hash mismatch."
    )


if sha256_file(
    CANDIDATE_REGISTRY_FILE
) != EXPECTED_CANDIDATE_SHA256:

    raise RuntimeError(
        "Candidate-registry hash mismatch."
    )


STEP22_PROTOCOL_SHA256 = sha256_file(
    STEP22_PROTOCOL_FILE
)


# ============================================================
# 5. LOAD MASTER AND RECREATE ROW KEY
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


heldout = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        ==
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


if len(
    heldout
) != 544:

    raise RuntimeError(
        f"Expected Barishal N=544; observed {len(heldout)}."
    )


if heldout[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Barishal master row key."
    )


# ============================================================
# 6. STAGE CHECKPOINT PATHS
# ============================================================

def stage_files(stage_number):

    stage_slug = (
        f"Stage_{stage_number}"
    )


    prefix = (
        f"STEP23_GEOGRAPHIC_"
        f"{TARGET_DIVISION}_"
        f"{stage_slug}"
    )


    return {
        "candidate_metrics":
            (
                DIRS["geographic"]
                /
                f"{prefix}_all_candidate_metrics.csv"
            ),

        "winner":
            (
                DIRS["geographic"]
                /
                f"{prefix}_winner.csv"
            ),

        "calibration":
            (
                DIRS["geographic"]
                /
                f"{prefix}_calibration_threshold.csv"
            ),

        "development_oof":
            (
                DIRS["geographic"]
                /
                f"{prefix}_winner_development_OOF_RESTRICTED.csv"
            ),

        "heldout_stage":
            (
                DIRS["geographic"]
                /
                f"{prefix}_heldout_stage_predictions_RESTRICTED.csv"
            ),

        "unseen":
            (
                DIRS["qc"]
                /
                f"{prefix}_unseen_category_qc.csv"
            ),

        "manifest":
            (
                DIRS["logs"]
                /
                f"{prefix}_stage_checkpoint.json"
            ),
    }


stage1_files = stage_files(
    1
)


stage2_files = stage_files(
    2
)


# ============================================================
# 7. VERIFY BOTH SAVED STAGES
# ============================================================

def verify_stage_checkpoint(
    stage_name,
    paths,
    expected_development_n,
):

    for path in paths.values():

        if not path.exists():

            raise FileNotFoundError(
                f"Missing {stage_name} checkpoint:\n{path}"
            )


    manifest = read_json(
        paths[
            "manifest"
        ]
    )


    if (
        manifest.get(
            "status"
        )
        !=
        "STEP23_STAGE_COMPLETE"
    ):

        raise RuntimeError(
            f"{stage_name} checkpoint not complete."
        )


    if (
        manifest.get(
            "execution_schema_version"
        )
        !=
        EXECUTION_SCHEMA_VERSION
    ):

        raise RuntimeError(
            f"{stage_name} schema mismatch."
        )


    if (
        manifest.get(
            "heldout_division"
        )
        !=
        TARGET_DIVISION
    ):

        raise RuntimeError(
            f"{stage_name} held-out division mismatch."
        )


    if (
        int(
            manifest.get(
                "stage_n"
            )
        )
        !=
        expected_development_n
    ):

        raise RuntimeError(
            f"{stage_name} development-N mismatch."
        )


    hash_pairs = {
        "candidate_metrics_sha256":
            "candidate_metrics",

        "winner_sha256":
            "winner",

        "calibration_sha256":
            "calibration",

        "development_oof_sha256":
            "development_oof",

        "heldout_stage_sha256":
            "heldout_stage",

        "unseen_sha256":
            "unseen",
    }


    for hash_key, path_key in (
        hash_pairs.items()
    ):

        observed_hash = sha256_file(
            paths[
                path_key
            ]
        )


        expected_hash = manifest.get(
            hash_key
        )


        if observed_hash != expected_hash:

            raise RuntimeError(
                f"{stage_name}: checkpoint hash mismatch "
                f"for {path_key}."
            )


    if (
        manifest.get(
            "heldout_performance_calculated"
        )
        is not False
    ):

        raise RuntimeError(
            f"{stage_name}: unexpected performance state."
        )


    return manifest


stage1_manifest = verify_stage_checkpoint(
    stage_name="Stage 1",
    paths=stage1_files,
    expected_development_n=4519,
)


stage2_manifest = verify_stage_checkpoint(
    stage_name="Stage 2",
    paths=stage2_files,
    expected_development_n=4156,
)


print(
    "Stage 1 checkpoint: VERIFIED"
)


print(
    "Stage 2 checkpoint: VERIFIED"
)


# ============================================================
# 8. LOAD SAVED STAGE OUTPUTS
# ============================================================

stage1_candidate_metrics = pd.read_csv(
    stage1_files[
        "candidate_metrics"
    ]
)


stage2_candidate_metrics = pd.read_csv(
    stage2_files[
        "candidate_metrics"
    ]
)


stage1_winner = pd.read_csv(
    stage1_files[
        "winner"
    ]
)


stage2_winner = pd.read_csv(
    stage2_files[
        "winner"
    ]
)


stage1_calibration = pd.read_csv(
    stage1_files[
        "calibration"
    ]
)


stage2_calibration = pd.read_csv(
    stage2_files[
        "calibration"
    ]
)


stage1_development_oof = pd.read_csv(
    stage1_files[
        "development_oof"
    ]
)


stage2_development_oof = pd.read_csv(
    stage2_files[
        "development_oof"
    ]
)


stage1_heldout = pd.read_csv(
    stage1_files[
        "heldout_stage"
    ]
)


stage2_heldout = pd.read_csv(
    stage2_files[
        "heldout_stage"
    ]
)


stage1_unseen = pd.read_csv(
    stage1_files[
        "unseen"
    ]
)


stage2_unseen = pd.read_csv(
    stage2_files[
        "unseen"
    ]
)


# ============================================================
# 9. SAVED-STAGE QC
# ============================================================

if len(
    stage1_candidate_metrics
) != 120:

    raise RuntimeError(
        "Stage 1 candidate count is not 120."
    )


if len(
    stage2_candidate_metrics
) != 120:

    raise RuntimeError(
        "Stage 2 candidate count is not 120."
    )


if (
    stage1_candidate_metrics[
        "Status"
    ]
    !=
    "SUCCESS"
).any():

    raise RuntimeError(
        "Stage 1 contains failed candidate."
    )


if (
    stage2_candidate_metrics[
        "Status"
    ]
    !=
    "SUCCESS"
).any():

    raise RuntimeError(
        "Stage 2 contains failed candidate."
    )


if len(
    stage1_winner
) != 1:

    raise RuntimeError(
        "Stage 1 winner count is not 1."
    )


if len(
    stage2_winner
) != 1:

    raise RuntimeError(
        "Stage 2 winner count is not 1."
    )


if len(
    stage1_heldout
) != 544:

    raise RuntimeError(
        "Stage 1 held-out predictions are not N=544."
    )


if len(
    stage2_heldout
) != 544:

    raise RuntimeError(
        "Stage 2 held-out predictions are not N=544."
    )


if stage1_heldout[
    "_ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Stage-1 held-out row key."
    )


if stage2_heldout[
    "_ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Stage-2 held-out row key."
    )


# ============================================================
# 10. FIXED ROW-KEY MERGE
# ============================================================
#
# Original bug:
#
# master / heldout:
#     __ROW_KEY
#
# saved stage predictions:
#     _ROW_KEY
#
# We standardize the held-out base to _ROW_KEY BEFORE merge.
# ============================================================

heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY":
                "_ROW_KEY",

            "ANC_CLASS":
                "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked[
    "Heldout_Division"
] = TARGET_DIVISION


# ============================================================
# 11. MERGE COVERAGE QC
# ============================================================

if len(
    locked
) != 544:

    raise RuntimeError(
        f"Expected 544 merged rows; observed {len(locked)}."
    )


if locked[
    "_ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate row after fixed merge."
    )


required_prediction_columns = [
    "Stage1_Raw_Score",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "Stage2_Raw_Score",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
]


if locked[
    required_prediction_columns
].isna().any().any():

    raise RuntimeError(
        "Missing prediction after row-key merge."
    )


# ============================================================
# 12. FROZEN HIERARCHICAL FUSION
# ============================================================

q0 = locked[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(
    dtype=float
)


t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(
    dtype=float
)


t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(
    dtype=float
)


locked[
    "P_NO_ANC"
] = q0


locked[
    "P_ANC_1_7"
] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)


locked[
    "P_ANC_GE8"
] = (
    (1.0 - q0)
    *
    q2
)


locked[
    "P_SUM"
] = (
    locked[
        "P_NO_ANC"
    ]
    +
    locked[
        "P_ANC_1_7"
    ]
    +
    locked[
        "P_ANC_GE8"
    ]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0 >= t1,
    0,
    np.where(
        q2 >= t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 13. COMBINE SAVED METADATA
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_candidate_metrics,
        stage2_candidate_metrics,
    ],
    ignore_index=True,
)


winners_all = pd.concat(
    [
        stage1_winner,
        stage2_winner,
    ],
    ignore_index=True,
)


calibration_all = pd.concat(
    [
        stage1_calibration,
        stage2_calibration,
    ],
    ignore_index=True,
)


development_oof_all = pd.concat(
    [
        stage1_development_oof,
        stage2_development_oof,
    ],
    ignore_index=True,
)


unseen_all = pd.concat(
    [
        stage1_unseen,
        stage2_unseen,
    ],
    ignore_index=True,
)


# ============================================================
# 14. FINAL HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                (
                    sha256_file(
                        PRIMARY_FILE
                    )
                    ==
                    EXPECTED_PRIMARY_SHA256
                ),
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                (
                    sha256_file(
                        OUTER_SPLIT_FILE
                    )
                    ==
                    EXPECTED_OUTER_SHA256
                ),
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                (
                    sha256_file(
                        CANDIDATE_REGISTRY_FILE
                    )
                    ==
                    EXPECTED_CANDIDATE_SHA256
                ),
        },

        {
            "Check":
                "Stage 1 checkpoint verified",

            "PASS":
                True,
        },

        {
            "Check":
                "Stage 2 checkpoint verified",

            "PASS":
                True,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                (
                    len(
                        candidate_metrics_all
                    )
                    ==
                    240
                ),
        },

        {
            "Check":
                "All 240 candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Exactly two joint winners",

            "PASS":
                (
                    len(
                        winners_all
                    )
                    ==
                    2
                ),
        },

        {
            "Check":
                "Stage 1 winner = ElasticNet EN_012",

            "PASS":
                (
                    str(
                        stage1_winner.iloc[
                            0
                        ][
                            "Model"
                        ]
                    )
                    ==
                    "ElasticNet logistic regression"
                    and
                    str(
                        stage1_winner.iloc[
                            0
                        ][
                            "Candidate_ID"
                        ]
                    )
                    ==
                    "EN_012"
                ),
        },

        {
            "Check":
                "Stage 2 winner = ElasticNet EN_001",

            "PASS":
                (
                    str(
                        stage2_winner.iloc[
                            0
                        ][
                            "Model"
                        ]
                    )
                    ==
                    "ElasticNet logistic regression"
                    and
                    str(
                        stage2_winner.iloc[
                            0
                        ][
                            "Candidate_ID"
                        ]
                    )
                    ==
                    "EN_001"
                ),
        },

        {
            "Check":
                "Exactly 544 held-out predictions",

            "PASS":
                (
                    len(
                        locked
                    )
                    ==
                    544
                ),
        },

        {
            "Check":
                "Held-out rows unique",

            "PASS":
                (
                    locked[
                        "_ROW_KEY"
                    ]
                    .nunique()
                    ==
                    544
                ),
        },

        {
            "Check":
                "Stage merge complete with no missing predictions",

            "PASS":
                (
                    not locked[
                        required_prediction_columns
                    ]
                    .isna()
                    .any()
                    .any()
                ),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                bool(
                    np.isfinite(
                        locked[
                            [
                                "P_NO_ANC",
                                "P_ANC_1_7",
                                "P_ANC_GE8",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                bool(
                    np.allclose(
                        locked[
                            "P_SUM"
                        ],
                        1.0,
                        rtol=0.0,
                        atol=1e-12,
                    )
                ),
        },

        {
            "Check":
                "No model refitting in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No hyperparameter retuning in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No recalibration in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No threshold re-optimization in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No held-out performance calculated",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "BARISHAL RECOVERY HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 15. FINAL BARISHAL FILES
# ============================================================

prefix = (
    "STEP23_GEOGRAPHIC_holdout_Barishal"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)


FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)


FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)


FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)


FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)


FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)


FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)


winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)


calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)


development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)


locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)


unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)


hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 16. FINAL MANIFEST
# ============================================================

manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        4519,

    "heldout_n":
        544,

    "stage1_development_n":
        4519,

    "stage2_development_n":
        4156,

    "execution_threads":
        EXECUTION_THREADS,

    "scientific_hyperparameters_changed":
        False,

    "division_predictor_used":
        False,

    "candidate_count_per_stage":
        120,

    "total_candidate_summaries":
        240,

    "development_cv_folds":
        5,

    "stage1_model":
        str(
            stage1_winner.iloc[
                0
            ][
                "Model"
            ]
        ),

    "stage1_candidate":
        str(
            stage1_winner.iloc[
                0
            ][
                "Candidate_ID"
            ]
        ),

    "stage2_model":
        str(
            stage2_winner.iloc[
                0
            ][
                "Model"
            ]
        ),

    "stage2_candidate":
        str(
            stage2_winner.iloc[
                0
            ][
                "Candidate_ID"
            ]
        ),

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_performance_calculated":
        False,

    "recovery_model_refitting":
        False,

    "recovery_retuning":
        False,

    "recovery_recalibration":
        False,

    "recovery_threshold_reoptimization":
        False,

    "merge_bug_fixed":
        (
            "__ROW_KEY in master standardized to "
            "_ROW_KEY before stage merges"
        ),

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "step22_protocol_sha256":
        STEP22_PROTOCOL_SHA256,

    "candidate_metrics_sha256":
        sha256_file(
            FINAL_CANDIDATE_FILE
        ),

    "winners_sha256":
        sha256_file(
            FINAL_WINNER_FILE
        ),

    "calibration_threshold_sha256":
        sha256_file(
            FINAL_CALIBRATION_FILE
        ),

    "winner_development_oof_sha256":
        sha256_file(
            FINAL_DEVELOPMENT_OOF_FILE
        ),

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "unseen_category_qc_sha256":
        sha256_file(
            FINAL_UNSEEN_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 17. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — BARISHAL RECOVERY / FINAL LOCK"
)


print(
    "=" * 78
)


print(
    "\nSAVED STAGE WINNERS:"
)


display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)


display(
    calibration_all
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nLOCKED BARISHAL PREDICTIONS:"
)


print(
    f"Rows             : {len(locked)}"
)


print(
    f"Unique rows      : {locked['_ROW_KEY'].nunique()}"
)


print(
    "Probability sum : "
    f"{np.allclose(locked['P_SUM'], 1.0, rtol=0, atol=1e-12)}"
)


print(
    "Model refitting : NO"
)


print(
    "Retuning        : NO"
)


print(
    "Recalibration   : NO"
)


print(
    "Threshold redo  : NO"
)


print(
    "Heldout metrics : NOT CALCULATED"
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : {sha256_file(PRIMARY_FILE)}"
)


print(
    f"Candidate grid : {sha256_file(CANDIDATE_REGISTRY_FILE)}"
)


print(
    f"Locked Barishal: {sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    + "=" * 78
)


print(
    "STATUS: STEP 23 BARISHAL IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    '\nNEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Chattogram"'
)

Stage 1 checkpoint: VERIFIED
Stage 2 checkpoint: VERIFIED

STEP 23 — BARISHAL RECOVERY / FINAL LOCK

SAVED STAGE WINNERS:


,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC
0,Barishal,Stage 1,ElasticNet logistic regression,EN_012,0.240458,0.778259
1,Barishal,Stage 2,ElasticNet logistic regression,EN_001,0.174462,0.725977



CALIBRATION + THRESHOLD:


,Heldout_Division,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Development_Threshold_Balanced_Accuracy,Development_Threshold_Sensitivity,Development_Threshold_Specificity
0,Barishal,Stage 1,ElasticNet logistic regression,EN_012,-0.260888,0.872131,6,0.073825,0.718581,0.791048,0.646113
1,Barishal,Stage 2,ElasticNet logistic regression,EN_001,2.993492,2.113844,12,0.046626,0.680611,0.709976,0.651246



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Stage 1 checkpoint verified,True
4,Stage 2 checkpoint verified,True
5,Exactly 240 candidate summaries,True
6,All 240 candidates successful,True
7,Exactly two joint winners,True
8,Stage 1 winner = ElasticNet EN_012,True
9,Stage 2 winner = ElasticNet EN_001,True



LOCKED BARISHAL PREDICTIONS:
Rows             : 544
Unique rows      : 544
Probability sum : True
Model refitting : NO
Retuning        : NO
Recalibration   : NO
Threshold redo  : NO
Heldout metrics : NOT CALCULATED

INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Candidate grid : f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Barishal: 5ce4bc6b8c2acf613b04634b4e281ff300c814576081eca80dab9879a3c84d06

STATUS: STEP 23 BARISHAL IECV PREDICTIONS LOCKED

NEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Chattogram"


In [3]:
# ============================================================
# STEP 23 — CHATTOGRAM IECV
#           CONTINUATION FROM EXISTING LAPTOP-OPTIMIZED SESSION
#           FIXED FINAL ROW-KEY MERGE
# ============================================================
#
# DO NOT RESTART KERNEL BEFORE RUNNING THIS CELL.
#
# Scientific protocol unchanged.
# Existing family/stage checkpoint functions are reused.
#
# Expected:
#   Held-out Chattogram N = 880
#   Development N         = 4183
#   Stage-2 development N = 3854
#
# NO HELD-OUT PERFORMANCE IS CALCULATED.
# ============================================================


from pathlib import Path

import datetime as dt
import hashlib
import json

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. REQUIRED FUNCTIONS FROM THE LAPTOP-OPTIMIZED STEP 23 CELL
# ============================================================

required_helpers = [
    "execute_stage",
    "stage_paths",
    "stage_checkpoint_valid",
    "evaluate_family",
    "build_cv_cache",
    "instantiate_candidate",
    "fit_candidate",
    "positive_score",
    "prepare_stage",
    "row_key_hash",
    "safe_name",
]


missing_helpers = [
    name
    for name in required_helpers
    if name not in globals()
]


if missing_helpers:

    raise RuntimeError(
        "Required Step-23 functions are not in memory:\n"
        f"{missing_helpers}\n\n"
        "Do NOT continue with this cell after a kernel restart. "
        "The laptop-optimized Step-23 setup cell must first be restored."
    )


# ============================================================
# 2. TARGET DIVISION
# ============================================================

TARGET_DIVISION = "Chattogram"

EXECUTION_THREADS = 2

EXECUTION_SCHEMA_VERSION = (
    "STEP23_GEOGRAPHIC_LAPTOP_v2"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_CANDIDATE_SHA256 = (
    "f95dc93431eced694114fe8ed071a2878"
    "e2572fac53aa3b9fd9c6cb8df87c697"
)


# ============================================================
# 3. RESTORE COMPLETE PROJECT PATHS
# ============================================================

ROOT = Path(
    r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
)


DIRS = {
    "root":
        ROOT,

    "processed":
        ROOT / "01_processed_data",

    "qc":
        ROOT / "02_quality_control",

    "protocol":
        ROOT / "03_protocol",

    "splits":
        ROOT / "04_cv_splits",

    "models":
        ROOT / "05_models",

    "predictions":
        ROOT / "06_predictions",

    "calibration":
        ROOT / "07_calibration",

    "bootstrap":
        ROOT / "08_bootstrap",

    "geographic":
        ROOT / "09_geographic_validation",

    "explainability":
        ROOT / "10_explainability",

    "sensitivity":
        ROOT / "11_sensitivity",

    "subgroup":
        ROOT / "12_subgroup_analysis",

    "tables":
        ROOT / "13_tables",

    "figures":
        ROOT / "14_figures",

    "logs":
        ROOT / "15_logs",

    "reporting":
        ROOT / "16_reporting",

    "archive":
        ROOT / "17_reproducibility_archive",
}


# ============================================================
# 4. INPUT FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)


STEP22_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_IECV_protocol.json"
)


# ============================================================
# 5. CORE HASH QC
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master hash mismatch."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Step-13 split hash mismatch."
    )


if sha256_file(
    CANDIDATE_REGISTRY_FILE
) != EXPECTED_CANDIDATE_SHA256:

    raise RuntimeError(
        "Candidate-registry hash mismatch."
    )


STEP22_PROTOCOL_SHA256 = sha256_file(
    STEP22_PROTOCOL_FILE
)


# ============================================================
# 6. RELOAD MASTER / SPLITS / CANDIDATE REGISTRY
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)


candidate_registry = pd.read_csv(
    CANDIDATE_REGISTRY_FILE
)


if master.shape != (
    5063,
    23,
):

    raise RuntimeError(
        f"Unexpected master shape: {master.shape}"
    )


if len(
    candidate_registry
) != 120:

    raise RuntimeError(
        "Candidate registry is not N=120."
    )


# ============================================================
# 7. RECREATE ROW KEY + FROZEN DEVELOPMENT FOLDS
# ============================================================

MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


master = master.copy()

outer_assignment = outer_assignment.copy()


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


if master[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate master row key."
    )


if outer_assignment[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate frozen split row key."
    )


if set(
    master[
        "__ROW_KEY"
    ]
) != set(
    outer_assignment[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Frozen Step-13 split does not exactly cover master."
    )


fold_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__ROW_KEY"
].map(
    fold_map
)


if master[
    "__DEVELOPMENT_CV_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing frozen development-CV fold."
    )


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__DEVELOPMENT_CV_FOLD"
].astype(int)


# ============================================================
# 8. REBUILD MODEL ORDER
# ============================================================

MODEL_ORDER = (
    candidate_registry
    .groupby(
        "Model",
        as_index=False,
    )[
        "Deterministic_Tie_Rank"
    ]
    .min()
    .sort_values(
        "Deterministic_Tie_Rank"
    )[
        "Model"
    ]
    .tolist()
)


if len(
    MODEL_ORDER
) != 10:

    raise RuntimeError(
        "Expected 10 model families."
    )


# ============================================================
# 9. CHATTOGRAM DEVELOPMENT / HOLDOUT
# ============================================================

development = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        !=
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


heldout = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        ==
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


EXPECTED_HELDOUT_N = 880


if len(
    development
) != 4183:

    raise RuntimeError(
        f"Expected Chattogram development N=4183; "
        f"observed {len(development)}."
    )


if len(
    heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        f"Expected Chattogram heldout N=880; "
        f"observed {len(heldout)}."
    )


if (
    set(
        development[
            "__ROW_KEY"
        ]
    )
    &
    set(
        heldout[
            "__ROW_KEY"
        ]
    )
):

    raise RuntimeError(
        "Development / held-out overlap."
    )


stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin(
        [
            1,
            2,
        ]
    )
    .sum()
)


if stage2_development_n != 3854:

    raise RuntimeError(
        f"Expected Stage-2 development N=3854; "
        f"observed {stage2_development_n}."
    )


print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — CHATTOGRAM GEOGRAPHIC IECV"
)


print(
    "=" * 78
)


print(
    f"Held-out division : {TARGET_DIVISION}"
)


print(
    f"Development N     : {len(development)}"
)


print(
    f"Stage-2 dev N     : {stage2_development_n}"
)


print(
    f"Held-out N        : {len(heldout)}"
)


print(
    f"CPU threads       : {EXECUTION_THREADS}"
)


print(
    "Division predictor: EXCLUDED"
)


# ============================================================
# 10. EXECUTE / LOAD STAGE 1
# ============================================================

stage1_result = execute_stage(
    "Stage 1"
)


# ============================================================
# 11. EXECUTE / LOAD STAGE 2
# ============================================================

stage2_result = execute_stage(
    "Stage 2"
)


# ============================================================
# 12. GET SAVED HELD-OUT STAGE PREDICTIONS
# ============================================================

stage1_heldout = (
    stage1_result[
        "heldout_stage"
    ]
    .copy()
)


stage2_heldout = (
    stage2_result[
        "heldout_stage"
    ]
    .copy()
)


if len(
    stage1_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-1 heldout prediction count mismatch."
    )


if len(
    stage2_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-2 heldout prediction count mismatch."
    )


# ============================================================
# 13. FIXED ROW-KEY MERGE
# ============================================================
#
# IMPORTANT FIX:
#
# master uses:
#       __ROW_KEY
#
# saved stage prediction uses:
#       _ROW_KEY
#
# Standardize base table to _ROW_KEY before merging.
# ============================================================

heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY":
                "_ROW_KEY",

            "ANC_CLASS":
                "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked[
    "Heldout_Division"
] = TARGET_DIVISION


required_prediction_columns = [
    "Stage1_Raw_Score",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "Stage2_Raw_Score",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
]


if locked[
    required_prediction_columns
].isna().any().any():

    raise RuntimeError(
        "Missing values after corrected stage merge."
    )


# ============================================================
# 14. HIERARCHICAL FUSION
# ============================================================

q0 = locked[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(
    dtype=float
)


t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(
    dtype=float
)


t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(
    dtype=float
)


locked[
    "P_NO_ANC"
] = q0


locked[
    "P_ANC_1_7"
] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)


locked[
    "P_ANC_GE8"
] = (
    (1.0 - q0)
    *
    q2
)


locked[
    "P_SUM"
] = (
    locked[
        "P_NO_ANC"
    ]
    +
    locked[
        "P_ANC_1_7"
    ]
    +
    locked[
        "P_ANC_GE8"
    ]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0 >= t1,
    0,
    np.where(
        q2 >= t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 15. COMBINE STAGE METADATA
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result[
            "candidate_metrics"
        ],
        stage2_result[
            "candidate_metrics"
        ],
    ],
    ignore_index=True,
)


winners_all = pd.concat(
    [
        stage1_result[
            "winner"
        ],
        stage2_result[
            "winner"
        ],
    ],
    ignore_index=True,
)


calibration_all = pd.concat(
    [
        stage1_result[
            "calibration"
        ],
        stage2_result[
            "calibration"
        ],
    ],
    ignore_index=True,
)


development_oof_all = pd.concat(
    [
        stage1_result[
            "development_oof"
        ],
        stage2_result[
            "development_oof"
        ],
    ],
    ignore_index=True,
)


unseen_all = pd.concat(
    [
        stage1_result[
            "unseen"
        ],
        stage2_result[
            "unseen"
        ],
    ],
    ignore_index=True,
)


# ============================================================
# 16. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(
                    PRIMARY_FILE
                )
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                sha256_file(
                    OUTER_SPLIT_FILE
                )
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                sha256_file(
                    CANDIDATE_REGISTRY_FILE
                )
                ==
                EXPECTED_CANDIDATE_SHA256,
        },

        {
            "Check":
                "Chattogram development N = 4183",

            "PASS":
                len(
                    development
                )
                ==
                4183,
        },

        {
            "Check":
                "Chattogram heldout N = 880",

            "PASS":
                len(
                    heldout
                )
                ==
                880,
        },

        {
            "Check":
                "Stage-2 development N = 3854",

            "PASS":
                stage2_development_n
                ==
                3854,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                len(
                    candidate_metrics_all
                )
                ==
                240,
        },

        {
            "Check":
                "All 240 candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Exactly two joint winners",

            "PASS":
                len(
                    winners_all
                )
                ==
                2,
        },

        {
            "Check":
                "Both stages represented",

            "PASS":
                set(
                    winners_all[
                        "Stage"
                    ]
                )
                ==
                {
                    "Stage 1",
                    "Stage 2",
                },
        },

        {
            "Check":
                "Exactly 880 final heldout predictions",

            "PASS":
                len(
                    locked
                )
                ==
                880,
        },

        {
            "Check":
                "Heldout rows unique",

            "PASS":
                locked[
                    "_ROW_KEY"
                ].nunique()
                ==
                880,
        },

        {
            "Check":
                "Stage merge complete",

            "PASS":
                not locked[
                    required_prediction_columns
                ]
                .isna()
                .any()
                .any(),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                bool(
                    np.isfinite(
                        locked[
                            [
                                "P_NO_ANC",
                                "P_ANC_1_7",
                                "P_ANC_GE8",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                bool(
                    np.allclose(
                        locked[
                            "P_SUM"
                        ],
                        1.0,
                        rtol=0.0,
                        atol=1e-12,
                    )
                ),
        },

        {
            "Check":
                "Division predictor excluded",

            "PASS":
                "division"
                not in
                GEO_PREDICTORS,
        },

        {
            "Check":
                "Execution threads = 2",

            "PASS":
                EXECUTION_THREADS
                ==
                2,
        },

        {
            "Check":
                "Scientific hyperparameters unchanged",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for selection",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for calibration",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for threshold optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "No heldout performance calculated",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]


    raise RuntimeError(
        "CHATTOGRAM STEP-23 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 17. FINAL CHATTOGRAM OUTPUTS
# ============================================================

prefix = (
    "STEP23_GEOGRAPHIC_holdout_Chattogram"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)


FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)


FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)


FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)


FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)


FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)


FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)


winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)


calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)


development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)


locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)


unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)


hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 18. FINAL MANIFEST
# ============================================================

manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        int(
            len(
                development
            )
        ),

    "heldout_n":
        int(
            len(
                heldout
            )
        ),

    "stage1_development_n":
        int(
            len(
                development
            )
        ),

    "stage2_development_n":
        int(
            stage2_development_n
        ),

    "execution_threads":
        EXECUTION_THREADS,

    "scientific_hyperparameters_changed":
        False,

    "division_predictor_used":
        False,

    "candidate_count_per_stage":
        120,

    "total_candidate_summaries":
        240,

    "development_cv_folds":
        5,

    "stage1_model":
        str(
            winners_all.loc[
                winners_all[
                    "Stage"
                ]
                ==
                "Stage 1",
                "Model",
            ].iloc[0]
        ),

    "stage1_candidate":
        str(
            winners_all.loc[
                winners_all[
                    "Stage"
                ]
                ==
                "Stage 1",
                "Candidate_ID",
            ].iloc[0]
        ),

    "stage2_model":
        str(
            winners_all.loc[
                winners_all[
                    "Stage"
                ]
                ==
                "Stage 2",
                "Model",
            ].iloc[0]
        ),

    "stage2_candidate":
        str(
            winners_all.loc[
                winners_all[
                    "Stage"
                ]
                ==
                "Stage 2",
                "Candidate_ID",
            ].iloc[0]
        ),

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "step22_protocol_sha256":
        STEP22_PROTOCOL_SHA256,

    "candidate_metrics_sha256":
        sha256_file(
            FINAL_CANDIDATE_FILE
        ),

    "winners_sha256":
        sha256_file(
            FINAL_WINNER_FILE
        ),

    "calibration_threshold_sha256":
        sha256_file(
            FINAL_CALIBRATION_FILE
        ),

    "winner_development_oof_sha256":
        sha256_file(
            FINAL_DEVELOPMENT_OOF_FILE
        ),

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "unseen_category_qc_sha256":
        sha256_file(
            FINAL_UNSEEN_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 19. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — CHATTOGRAM IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)


display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
            "Execution_Threads",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)


display(
    calibration_all
)


print(
    "\nUNSEEN CATEGORY QC:"
)


display(
    unseen_all
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nLOCKED CHATTOGRAM PREDICTIONS:"
)


print(
    f"Rows             : {len(locked)}"
)


print(
    f"Unique rows      : {locked['_ROW_KEY'].nunique()}"
)


print(
    "Probability sum : "
    f"{np.allclose(locked['P_SUM'], 1.0, rtol=0, atol=1e-12)}"
)


print(
    "Heldout metrics : NOT CALCULATED"
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master    : {sha256_file(PRIMARY_FILE)}"
)


print(
    f"Candidate registry: {sha256_file(CANDIDATE_REGISTRY_FILE)}"
)


print(
    f"Locked Chattogram : {sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    + "=" * 78
)


print(
    "STATUS: STEP 23 CHATTOGRAM IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    '\nNEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Dhaka"'
)


STEP 23 — CHATTOGRAM GEOGRAPHIC IECV
Held-out division : Chattogram
Development N     : 4183
Stage-2 dev N     : 3854
Held-out N        : 880
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Chattogram | Stage 1
Development N = 4183
Held-out scoring N = 880
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.232141
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.150883
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.200886
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.193123
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.217118
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.223340
  [7/10] HistGradientBoosting
    HistGradientBoosting: co

,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC,Execution_Threads
0,Chattogram,Stage 1,ElasticNet logistic regression,EN_005,0.232141,0.779764,2
1,Chattogram,Stage 2,ElasticNet logistic regression,EN_001,0.188025,0.732498,2



CALIBRATION + THRESHOLD:


,Heldout_Division,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Development_Threshold_Balanced_Accuracy,Development_Threshold_Sensitivity,Development_Threshold_Specificity
0,Chattogram,Stage 1,ElasticNet logistic regression,EN_005,0.204908,1.095608,10,0.079522,0.719274,0.729335,0.709214
1,Chattogram,Stage 2,ElasticNet logistic regression,EN_001,2.976227,2.132474,13,0.070043,0.690428,0.598703,0.782153



UNSEEN CATEGORY QC:


,Heldout_Division,Stage,Model,Predictor,Heldout_Unseen_Category_Rows
0,Chattogram,Stage 1,ElasticNet logistic regression,maternal_education,0
1,Chattogram,Stage 1,ElasticNet logistic regression,wealth_quintile,0
2,Chattogram,Stage 1,ElasticNet logistic regression,partner_education_primary_final,0
3,Chattogram,Stage 1,ElasticNet logistic regression,pregnancy_intention,0
4,Chattogram,Stage 1,ElasticNet logistic regression,healthcare_decision_primary_final,0
5,Chattogram,Stage 1,ElasticNet logistic regression,residence,0
6,Chattogram,Stage 1,ElasticNet logistic regression,weekly_media_exposure,0
7,Chattogram,Stage 1,ElasticNet logistic regression,working_status,0
8,Chattogram,Stage 1,ElasticNet logistic regression,religion_final,0
9,Chattogram,Stage 1,ElasticNet logistic regression,household_head_sex,0



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Chattogram development N = 4183,True
4,Chattogram heldout N = 880,True
5,Stage-2 development N = 3854,True
6,Exactly 240 candidate summaries,True
7,All 240 candidates successful,True
8,Exactly two joint winners,True
9,Both stages represented,True



LOCKED CHATTOGRAM PREDICTIONS:
Rows             : 880
Unique rows      : 880
Probability sum : True
Heldout metrics : NOT CALCULATED

INTEGRITY:
Primary master    : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Candidate registry: f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Chattogram : 2d96c20b1269c93a94687c223db063c1b5066cb938714a8923d7e65463b0a076

STATUS: STEP 23 CHATTOGRAM IECV PREDICTIONS LOCKED

NEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Dhaka"


In [4]:
# ============================================================
# STEP 23 — DHAKA IECV
#           CONTINUATION + CORRECTED FINAL LOCK
# ============================================================

from pathlib import Path
import datetime as dt
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. REQUIRED EXISTING STEP-23 HELPERS
# ============================================================

required_helpers = [
    "execute_stage",
    "sha256_file",
    "safe_name",
]

missing_helpers = [
    x for x in required_helpers
    if x not in globals()
]

if missing_helpers:
    raise RuntimeError(
        "Step-23 helper functions missing:\n"
        f"{missing_helpers}\n\n"
        "Kernel was likely restarted. "
        "Restore the laptop-optimized Step-23 setup first."
    )


# ============================================================
# 2. DHAKA SETTINGS
# ============================================================

TARGET_DIVISION = "Dhaka"

EXPECTED_DEVELOPMENT_N = 4310
EXPECTED_STAGE2_DEVELOPMENT_N = 3950
EXPECTED_HELDOUT_N = 753

EXECUTION_THREADS = 2

EXECUTION_SCHEMA_VERSION = (
    "STEP23_GEOGRAPHIC_LAPTOP_v2"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)

EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)

EXPECTED_CANDIDATE_SHA256 = (
    "f95dc93431eced694114fe8ed071a2878"
    "e2572fac53aa3b9fd9c6cb8df87c697"
)


# ============================================================
# 3. VERIFY CURRENT MASTER IS READY
# ============================================================

required_master_columns = [
    "__ROW_KEY",
    "__DEVELOPMENT_CV_FOLD",
    "division",
    "ANC_CLASS",
    "SURVEY_WEIGHT",
    "PSU",
    "STRATUM",
]

missing_master = [
    c for c in required_master_columns
    if c not in master.columns
]

if missing_master:
    raise RuntimeError(
        "Current master is missing required prepared columns:\n"
        f"{missing_master}"
    )


if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError(
        "Primary master hash mismatch."
    )

if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:
    raise RuntimeError(
        "Step-13 split hash mismatch."
    )

if (
    sha256_file(CANDIDATE_REGISTRY_FILE)
    != EXPECTED_CANDIDATE_SHA256
):
    raise RuntimeError(
        "Candidate registry hash mismatch."
    )


# ============================================================
# 4. DHAKA DEVELOPMENT / HELDOUT
# ============================================================

development = (
    master.loc[
        master["division"].astype(str)
        != TARGET_DIVISION
    ]
    .copy()
    .reset_index(drop=True)
)

heldout = (
    master.loc[
        master["division"].astype(str)
        == TARGET_DIVISION
    ]
    .copy()
    .reset_index(drop=True)
)


stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin([1, 2])
    .sum()
)


if len(development) != EXPECTED_DEVELOPMENT_N:
    raise RuntimeError(
        f"Development N mismatch: "
        f"{len(development)}"
    )

if len(heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        f"Heldout N mismatch: {len(heldout)}"
    )

if (
    stage2_development_n
    != EXPECTED_STAGE2_DEVELOPMENT_N
):
    raise RuntimeError(
        "Stage-2 development N mismatch: "
        f"{stage2_development_n}"
    )

if (
    set(development["__ROW_KEY"])
    &
    set(heldout["__ROW_KEY"])
):
    raise RuntimeError(
        "Development / heldout overlap."
    )


print(
    "\n"
    + "=" * 78
)

print(
    "STEP 23 — DHAKA GEOGRAPHIC IECV"
)

print(
    "=" * 78
)

print(
    f"Held-out division : {TARGET_DIVISION}"
)

print(
    f"Development N     : {len(development)}"
)

print(
    f"Stage-2 dev N     : {stage2_development_n}"
)

print(
    f"Held-out N        : {len(heldout)}"
)

print(
    f"CPU threads       : {EXECUTION_THREADS}"
)

print(
    "Division predictor: EXCLUDED"
)


# ============================================================
# 5. RUN / LOAD BOTH STAGES
# ============================================================

stage1_result = execute_stage(
    "Stage 1"
)

stage2_result = execute_stage(
    "Stage 2"
)


# ============================================================
# 6. HELDOUT STAGE OUTPUTS
# ============================================================

stage1_heldout = (
    stage1_result[
        "heldout_stage"
    ]
    .copy()
)

stage2_heldout = (
    stage2_result[
        "heldout_stage"
    ]
    .copy()
)


if len(stage1_heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        "Stage-1 heldout N mismatch."
    )

if len(stage2_heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        "Stage-2 heldout N mismatch."
    )


# ============================================================
# 7. CORRECTED ROW-KEY MERGE
# ============================================================

heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY":
                "_ROW_KEY",

            "ANC_CLASS":
                "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked[
    "Heldout_Division"
] = TARGET_DIVISION


required_prediction_columns = [
    "Stage1_Raw_Score",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "Stage2_Raw_Score",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
]


if (
    locked[
        required_prediction_columns
    ]
    .isna()
    .any()
    .any()
):
    raise RuntimeError(
        "Missing prediction after merge."
    )


# ============================================================
# 8. HIERARCHICAL FUSION
# ============================================================

q0 = locked[
    "q0_CALIBRATED"
].to_numpy(dtype=float)

q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(dtype=float)

t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(dtype=float)

t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(dtype=float)


locked[
    "P_NO_ANC"
] = q0

locked[
    "P_ANC_1_7"
] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)

locked[
    "P_ANC_GE8"
] = (
    (1.0 - q0)
    *
    q2
)

locked[
    "P_SUM"
] = (
    locked[
        "P_NO_ANC"
    ]
    +
    locked[
        "P_ANC_1_7"
    ]
    +
    locked[
        "P_ANC_GE8"
    ]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0 >= t1,
    0,
    np.where(
        q2 >= t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 9. COMBINE METADATA
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result[
            "candidate_metrics"
        ],
        stage2_result[
            "candidate_metrics"
        ],
    ],
    ignore_index=True,
)


winners_all = pd.concat(
    [
        stage1_result[
            "winner"
        ],
        stage2_result[
            "winner"
        ],
    ],
    ignore_index=True,
)


calibration_all = pd.concat(
    [
        stage1_result[
            "calibration"
        ],
        stage2_result[
            "calibration"
        ],
    ],
    ignore_index=True,
)


development_oof_all = pd.concat(
    [
        stage1_result[
            "development_oof"
        ],
        stage2_result[
            "development_oof"
        ],
    ],
    ignore_index=True,
)


unseen_all = pd.concat(
    [
        stage1_result[
            "unseen"
        ],
        stage2_result[
            "unseen"
        ],
    ],
    ignore_index=True,
)


# ============================================================
# 10. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(
                    PRIMARY_FILE
                )
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                sha256_file(
                    OUTER_SPLIT_FILE
                )
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                sha256_file(
                    CANDIDATE_REGISTRY_FILE
                )
                ==
                EXPECTED_CANDIDATE_SHA256,
        },

        {
            "Check":
                "Dhaka development N = 4310",

            "PASS":
                len(development)
                ==
                4310,
        },

        {
            "Check":
                "Dhaka heldout N = 753",

            "PASS":
                len(heldout)
                ==
                753,
        },

        {
            "Check":
                "Stage-2 development N = 3950",

            "PASS":
                stage2_development_n
                ==
                3950,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                len(
                    candidate_metrics_all
                )
                ==
                240,
        },

        {
            "Check":
                "All 240 candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Exactly two joint winners",

            "PASS":
                len(
                    winners_all
                )
                ==
                2,
        },

        {
            "Check":
                "Both stages represented",

            "PASS":
                set(
                    winners_all[
                        "Stage"
                    ]
                )
                ==
                {
                    "Stage 1",
                    "Stage 2",
                },
        },

        {
            "Check":
                "Exactly 753 heldout predictions",

            "PASS":
                len(
                    locked
                )
                ==
                753,
        },

        {
            "Check":
                "Heldout rows unique",

            "PASS":
                locked[
                    "_ROW_KEY"
                ].nunique()
                ==
                753,
        },

        {
            "Check":
                "Stage merge complete",

            "PASS":
                not locked[
                    required_prediction_columns
                ]
                .isna()
                .any()
                .any(),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                bool(
                    np.isfinite(
                        locked[
                            [
                                "P_NO_ANC",
                                "P_ANC_1_7",
                                "P_ANC_GE8",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                bool(
                    np.allclose(
                        locked[
                            "P_SUM"
                        ],
                        1.0,
                        rtol=0.0,
                        atol=1e-12,
                    )
                ),
        },

        {
            "Check":
                "Division predictor excluded",

            "PASS":
                "division"
                not in
                GEO_PREDICTORS,
        },

        {
            "Check":
                "Execution threads = 2",

            "PASS":
                EXECUTION_THREADS
                ==
                2,
        },

        {
            "Check":
                "Scientific hyperparameters unchanged",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for selection",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for calibration",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for threshold optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "No heldout performance calculated",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "DHAKA STEP-23 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 11. SAVE FINAL DHAKA LOCK
# ============================================================

division_slug = safe_name(
    TARGET_DIVISION
)

prefix = (
    f"STEP23_GEOGRAPHIC_holdout_"
    f"{division_slug}"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)

FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)

FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)

FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)

FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)

FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)

FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)

FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)

winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)

calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)

development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)

locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)

unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)

hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 12. MANIFEST
# ============================================================

stage1_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 1"
    ]
    .iloc[0]
)

stage2_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 2"
    ]
    .iloc[0]
)


manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        int(
            len(development)
        ),

    "heldout_n":
        int(
            len(heldout)
        ),

    "stage1_development_n":
        int(
            len(development)
        ),

    "stage2_development_n":
        int(
            stage2_development_n
        ),

    "execution_threads":
        EXECUTION_THREADS,

    "scientific_hyperparameters_changed":
        False,

    "division_predictor_used":
        False,

    "candidate_count_per_stage":
        120,

    "total_candidate_summaries":
        240,

    "development_cv_folds":
        5,

    "stage1_model":
        str(
            stage1_row[
                "Model"
            ]
        ),

    "stage1_candidate":
        str(
            stage1_row[
                "Candidate_ID"
            ]
        ),

    "stage2_model":
        str(
            stage2_row[
                "Model"
            ]
        ),

    "stage2_candidate":
        str(
            stage2_row[
                "Candidate_ID"
            ]
        ),

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "step22_protocol_sha256":
        STEP22_PROTOCOL_SHA256,

    "candidate_metrics_sha256":
        sha256_file(
            FINAL_CANDIDATE_FILE
        ),

    "winners_sha256":
        sha256_file(
            FINAL_WINNER_FILE
        ),

    "calibration_threshold_sha256":
        sha256_file(
            FINAL_CALIBRATION_FILE
        ),

    "winner_development_oof_sha256":
        sha256_file(
            FINAL_DEVELOPMENT_OOF_FILE
        ),

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "unseen_category_qc_sha256":
        sha256_file(
            FINAL_UNSEEN_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)

print(
    "STEP 23 — DHAKA IECV PREDICTIONS LOCKED"
)

print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)

display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
            "Execution_Threads",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)

display(
    calibration_all
)


print(
    "\nUNSEEN CATEGORY QC:"
)

display(
    unseen_all
)


print(
    "\nFINAL HARD QC:"
)

display(
    hard_qc
)


print(
    "\nLOCKED DHAKA PREDICTIONS:"
)

print(
    f"Rows             : {len(locked)}"
)

print(
    f"Unique rows      : "
    f"{locked['_ROW_KEY'].nunique()}"
)

print(
    "Probability sum : "
    f"{np.allclose(locked['P_SUM'], 1.0, rtol=0, atol=1e-12)}"
)

print(
    "Heldout metrics : NOT CALCULATED"
)


print(
    "\nINTEGRITY:"
)

print(
    f"Primary master : "
    f"{sha256_file(PRIMARY_FILE)}"
)

print(
    f"Candidate grid : "
    f"{sha256_file(CANDIDATE_REGISTRY_FILE)}"
)

print(
    f"Locked Dhaka   : "
    f"{sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    + "=" * 78
)

print(
    "STATUS: STEP 23 DHAKA IECV PREDICTIONS LOCKED"
)

print(
    "=" * 78
)

print(
    '\nNEXT DIVISION AFTER AUDIT: '
    'TARGET_DIVISION = "Khulna"'
)


STEP 23 — DHAKA GEOGRAPHIC IECV
Held-out division : Dhaka
Development N     : 4310
Stage-2 dev N     : 3950
Held-out N        : 753
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Dhaka | Stage 1
Development N = 4310
Held-out scoring N = 753
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.244853
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.177526
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.226389
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.223561
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.241969
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.244236
  [7/10] HistGradientBoosting
    HistGradientBoosting: complete | best d

,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC,Execution_Threads
0,Dhaka,Stage 1,ElasticNet logistic regression,EN_011,0.244853,0.772135,2
1,Dhaka,Stage 2,AdaBoost,ADA_007,0.114987,0.680813,2



CALIBRATION + THRESHOLD:


,Heldout_Division,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Development_Threshold_Balanced_Accuracy,Development_Threshold_Sensitivity,Development_Threshold_Specificity
0,Dhaka,Stage 1,ElasticNet logistic regression,EN_011,-0.294317,0.849717,6,0.077022,0.714294,0.811413,0.617176
1,Dhaka,Stage 2,AdaBoost,ADA_007,0.866068,2.473268,12,0.042703,0.661533,0.528140,0.794926



UNSEEN CATEGORY QC:


,Heldout_Division,Stage,Model,Predictor,Heldout_Unseen_Category_Rows
0,Dhaka,Stage 1,ElasticNet logistic regression,maternal_education,0
1,Dhaka,Stage 1,ElasticNet logistic regression,wealth_quintile,0
2,Dhaka,Stage 1,ElasticNet logistic regression,partner_education_primary_final,0
3,Dhaka,Stage 1,ElasticNet logistic regression,pregnancy_intention,0
4,Dhaka,Stage 1,ElasticNet logistic regression,healthcare_decision_primary_final,0
5,Dhaka,Stage 1,ElasticNet logistic regression,residence,0
6,Dhaka,Stage 1,ElasticNet logistic regression,weekly_media_exposure,0
7,Dhaka,Stage 1,ElasticNet logistic regression,working_status,0
8,Dhaka,Stage 1,ElasticNet logistic regression,religion_final,0
9,Dhaka,Stage 1,ElasticNet logistic regression,household_head_sex,0



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Dhaka development N = 4310,True
4,Dhaka heldout N = 753,True
5,Stage-2 development N = 3950,True
6,Exactly 240 candidate summaries,True
7,All 240 candidates successful,True
8,Exactly two joint winners,True
9,Both stages represented,True



LOCKED DHAKA PREDICTIONS:
Rows             : 753
Unique rows      : 753
Probability sum : True
Heldout metrics : NOT CALCULATED

INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Candidate grid : f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Dhaka   : e6b27bdadf1028e1253bdf919aff36e1c51c40ee0f2734a8eafe5f29bb5387cc

STATUS: STEP 23 DHAKA IECV PREDICTIONS LOCKED

NEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Khulna"


In [5]:
# ============================================================
# STEP 23 — KHULNA IECV
#           CONTINUATION + CORRECTED FINAL LOCK
# ============================================================

from pathlib import Path
import datetime as dt
import json
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. REQUIRED EXISTING HELPERS
# ============================================================

required_helpers = [
    "execute_stage",
    "sha256_file",
    "safe_name",
]

missing_helpers = [
    name
    for name in required_helpers
    if name not in globals()
]

if missing_helpers:

    raise RuntimeError(
        "Required Step-23 helper functions missing:\n"
        f"{missing_helpers}\n\n"
        "Kernel was likely restarted. "
        "Restore the laptop-optimized Step-23 setup first."
    )


# ============================================================
# 2. KHULNA SETTINGS
# ============================================================

TARGET_DIVISION = "Khulna"

EXPECTED_DEVELOPMENT_N = 4489

EXPECTED_STAGE2_DEVELOPMENT_N = 4109

EXPECTED_HELDOUT_N = 574

EXECUTION_THREADS = 2

EXECUTION_SCHEMA_VERSION = (
    "STEP23_GEOGRAPHIC_LAPTOP_v2"
)


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_CANDIDATE_SHA256 = (
    "f95dc93431eced694114fe8ed071a2878"
    "e2572fac53aa3b9fd9c6cb8df87c697"
)


# ============================================================
# 3. CORE HASH QC
# ============================================================

if sha256_file(
    PRIMARY_FILE
) != EXPECTED_PRIMARY_SHA256:

    raise RuntimeError(
        "Primary master hash mismatch."
    )


if sha256_file(
    OUTER_SPLIT_FILE
) != EXPECTED_OUTER_SHA256:

    raise RuntimeError(
        "Step-13 split hash mismatch."
    )


if sha256_file(
    CANDIDATE_REGISTRY_FILE
) != EXPECTED_CANDIDATE_SHA256:

    raise RuntimeError(
        "Candidate registry hash mismatch."
    )


# ============================================================
# 4. KHULNA DEVELOPMENT / HELDOUT
# ============================================================

development = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        !=
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


heldout = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        ==
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin(
        [
            1,
            2,
        ]
    )
    .sum()
)


if len(
    development
) != EXPECTED_DEVELOPMENT_N:

    raise RuntimeError(
        f"Expected development N="
        f"{EXPECTED_DEVELOPMENT_N}; "
        f"observed {len(development)}."
    )


if len(
    heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        f"Expected heldout N="
        f"{EXPECTED_HELDOUT_N}; "
        f"observed {len(heldout)}."
    )


if (
    stage2_development_n
    !=
    EXPECTED_STAGE2_DEVELOPMENT_N
):

    raise RuntimeError(
        f"Expected Stage-2 development N="
        f"{EXPECTED_STAGE2_DEVELOPMENT_N}; "
        f"observed {stage2_development_n}."
    )


if (
    set(
        development[
            "__ROW_KEY"
        ]
    )
    &
    set(
        heldout[
            "__ROW_KEY"
        ]
    )
):

    raise RuntimeError(
        "Development / heldout overlap."
    )


print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — KHULNA GEOGRAPHIC IECV"
)


print(
    "=" * 78
)


print(
    f"Held-out division : {TARGET_DIVISION}"
)


print(
    f"Development N     : {len(development)}"
)


print(
    f"Stage-2 dev N     : {stage2_development_n}"
)


print(
    f"Held-out N        : {len(heldout)}"
)


print(
    f"CPU threads       : {EXECUTION_THREADS}"
)


print(
    "Division predictor: EXCLUDED"
)


# ============================================================
# 5. RUN / LOAD BOTH STAGES
# ============================================================

stage1_result = execute_stage(
    "Stage 1"
)


stage2_result = execute_stage(
    "Stage 2"
)


# ============================================================
# 6. LOAD HELDOUT STAGE OUTPUTS
# ============================================================

stage1_heldout = (
    stage1_result[
        "heldout_stage"
    ]
    .copy()
)


stage2_heldout = (
    stage2_result[
        "heldout_stage"
    ]
    .copy()
)


if len(
    stage1_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-1 heldout N mismatch."
    )


if len(
    stage2_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-2 heldout N mismatch."
    )


# ============================================================
# 7. CORRECTED ROW-KEY MERGE
# ============================================================

heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY":
                "_ROW_KEY",

            "ANC_CLASS":
                "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked[
    "Heldout_Division"
] = TARGET_DIVISION


required_prediction_columns = [
    "Stage1_Raw_Score",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "Stage2_Raw_Score",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
]


if (
    locked[
        required_prediction_columns
    ]
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "Missing prediction after merge."
    )


# ============================================================
# 8. HIERARCHICAL FUSION
# ============================================================

q0 = locked[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(
    dtype=float
)


t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(
    dtype=float
)


t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(
    dtype=float
)


locked[
    "P_NO_ANC"
] = q0


locked[
    "P_ANC_1_7"
] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)


locked[
    "P_ANC_GE8"
] = (
    (1.0 - q0)
    *
    q2
)


locked[
    "P_SUM"
] = (
    locked[
        "P_NO_ANC"
    ]
    +
    locked[
        "P_ANC_1_7"
    ]
    +
    locked[
        "P_ANC_GE8"
    ]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0 >= t1,
    0,
    np.where(
        q2 >= t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 9. COMBINE METADATA
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result[
            "candidate_metrics"
        ],
        stage2_result[
            "candidate_metrics"
        ],
    ],
    ignore_index=True,
)


winners_all = pd.concat(
    [
        stage1_result[
            "winner"
        ],
        stage2_result[
            "winner"
        ],
    ],
    ignore_index=True,
)


calibration_all = pd.concat(
    [
        stage1_result[
            "calibration"
        ],
        stage2_result[
            "calibration"
        ],
    ],
    ignore_index=True,
)


development_oof_all = pd.concat(
    [
        stage1_result[
            "development_oof"
        ],
        stage2_result[
            "development_oof"
        ],
    ],
    ignore_index=True,
)


unseen_all = pd.concat(
    [
        stage1_result[
            "unseen"
        ],
        stage2_result[
            "unseen"
        ],
    ],
    ignore_index=True,
)


# ============================================================
# 10. HARD QC
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(
                    PRIMARY_FILE
                )
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                sha256_file(
                    OUTER_SPLIT_FILE
                )
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                sha256_file(
                    CANDIDATE_REGISTRY_FILE
                )
                ==
                EXPECTED_CANDIDATE_SHA256,
        },

        {
            "Check":
                "Khulna development N = 4489",

            "PASS":
                len(
                    development
                )
                ==
                4489,
        },

        {
            "Check":
                "Khulna heldout N = 574",

            "PASS":
                len(
                    heldout
                )
                ==
                574,
        },

        {
            "Check":
                "Stage-2 development N = 4109",

            "PASS":
                stage2_development_n
                ==
                4109,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                len(
                    candidate_metrics_all
                )
                ==
                240,
        },

        {
            "Check":
                "All 240 candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Exactly two joint winners",

            "PASS":
                len(
                    winners_all
                )
                ==
                2,
        },

        {
            "Check":
                "Both stages represented",

            "PASS":
                set(
                    winners_all[
                        "Stage"
                    ]
                )
                ==
                {
                    "Stage 1",
                    "Stage 2",
                },
        },

        {
            "Check":
                "Exactly 574 heldout predictions",

            "PASS":
                len(
                    locked
                )
                ==
                574,
        },

        {
            "Check":
                "Heldout rows unique",

            "PASS":
                locked[
                    "_ROW_KEY"
                ]
                .nunique()
                ==
                574,
        },

        {
            "Check":
                "Stage merge complete",

            "PASS":
                not locked[
                    required_prediction_columns
                ]
                .isna()
                .any()
                .any(),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                bool(
                    np.isfinite(
                        locked[
                            [
                                "P_NO_ANC",
                                "P_ANC_1_7",
                                "P_ANC_GE8",
                            ]
                        ]
                        .to_numpy(
                            dtype=float
                        )
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                bool(
                    np.allclose(
                        locked[
                            "P_SUM"
                        ],
                        1.0,
                        rtol=0.0,
                        atol=1e-12,
                    )
                ),
        },

        {
            "Check":
                "Division predictor excluded",

            "PASS":
                "division"
                not in
                GEO_PREDICTORS,
        },

        {
            "Check":
                "Execution threads = 2",

            "PASS":
                EXECUTION_THREADS
                ==
                2,
        },

        {
            "Check":
                "Scientific hyperparameters unchanged",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for selection",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for calibration",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for threshold optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "No heldout performance calculated",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "KHULNA STEP-23 HARD QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 11. SAVE FINAL KHULNA LOCK
# ============================================================

division_slug = safe_name(
    TARGET_DIVISION
)


prefix = (
    f"STEP23_GEOGRAPHIC_holdout_"
    f"{division_slug}"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)


FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)


FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)


FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)


FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)


FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)


FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)


winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)


calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)


development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)


locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)


unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)


hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 12. MANIFEST
# ============================================================

stage1_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 1"
    ]
    .iloc[0]
)


stage2_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 2"
    ]
    .iloc[0]
)


manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        int(
            len(
                development
            )
        ),

    "heldout_n":
        int(
            len(
                heldout
            )
        ),

    "stage1_development_n":
        int(
            len(
                development
            )
        ),

    "stage2_development_n":
        int(
            stage2_development_n
        ),

    "execution_threads":
        EXECUTION_THREADS,

    "scientific_hyperparameters_changed":
        False,

    "division_predictor_used":
        False,

    "candidate_count_per_stage":
        120,

    "total_candidate_summaries":
        240,

    "development_cv_folds":
        5,

    "stage1_model":
        str(
            stage1_row[
                "Model"
            ]
        ),

    "stage1_candidate":
        str(
            stage1_row[
                "Candidate_ID"
            ]
        ),

    "stage2_model":
        str(
            stage2_row[
                "Model"
            ]
        ),

    "stage2_candidate":
        str(
            stage2_row[
                "Candidate_ID"
            ]
        ),

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "step22_protocol_sha256":
        STEP22_PROTOCOL_SHA256,

    "candidate_metrics_sha256":
        sha256_file(
            FINAL_CANDIDATE_FILE
        ),

    "winners_sha256":
        sha256_file(
            FINAL_WINNER_FILE
        ),

    "calibration_threshold_sha256":
        sha256_file(
            FINAL_CALIBRATION_FILE
        ),

    "winner_development_oof_sha256":
        sha256_file(
            FINAL_DEVELOPMENT_OOF_FILE
        ),

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "unseen_category_qc_sha256":
        sha256_file(
            FINAL_UNSEEN_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — KHULNA IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)


display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
            "Execution_Threads",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)


display(
    calibration_all
)


print(
    "\nUNSEEN CATEGORY QC:"
)


display(
    unseen_all
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nLOCKED KHULNA PREDICTIONS:"
)


print(
    f"Rows             : {len(locked)}"
)


print(
    f"Unique rows      : "
    f"{locked['_ROW_KEY'].nunique()}"
)


print(
    "Probability sum : "
    f"{np.allclose(locked['P_SUM'], 1.0, rtol=0, atol=1e-12)}"
)


print(
    "Heldout metrics : NOT CALCULATED"
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master : "
    f"{sha256_file(PRIMARY_FILE)}"
)


print(
    f"Candidate grid : "
    f"{sha256_file(CANDIDATE_REGISTRY_FILE)}"
)


print(
    f"Locked Khulna  : "
    f"{sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    + "=" * 78
)


print(
    "STATUS: STEP 23 KHULNA IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    '\nNEXT DIVISION AFTER AUDIT: '
    'TARGET_DIVISION = "Mymensingh"'
)


STEP 23 — KHULNA GEOGRAPHIC IECV
Held-out division : Khulna
Development N     : 4489
Stage-2 dev N     : 4109
Held-out N        : 574
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Khulna | Stage 1
Development N = 4489
Held-out scoring N = 574
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.246720
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.168048
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.216316
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.210141
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.223965
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.236248
  [7/10] HistGradientBoosting
    HistGradientBoosting: complete | bes

,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC,Execution_Threads
0,Khulna,Stage 1,ElasticNet logistic regression,EN_011,0.246720,0.776230,2
1,Khulna,Stage 2,ElasticNet logistic regression,EN_001,0.170946,0.736701,2



CALIBRATION + THRESHOLD:


,Heldout_Division,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Development_Threshold_Balanced_Accuracy,Development_Threshold_Sensitivity,Development_Threshold_Specificity
0,Khulna,Stage 1,ElasticNet logistic regression,EN_011,-0.258180,0.870191,6,0.080882,0.711561,0.775020,0.648103
1,Khulna,Stage 2,ElasticNet logistic regression,EN_001,3.237476,2.186458,13,0.044405,0.690395,0.731195,0.649595



UNSEEN CATEGORY QC:


,Heldout_Division,Stage,Model,Predictor,Heldout_Unseen_Category_Rows
0,Khulna,Stage 1,ElasticNet logistic regression,maternal_education,0
1,Khulna,Stage 1,ElasticNet logistic regression,wealth_quintile,0
2,Khulna,Stage 1,ElasticNet logistic regression,partner_education_primary_final,0
3,Khulna,Stage 1,ElasticNet logistic regression,pregnancy_intention,0
4,Khulna,Stage 1,ElasticNet logistic regression,healthcare_decision_primary_final,0
5,Khulna,Stage 1,ElasticNet logistic regression,residence,0
6,Khulna,Stage 1,ElasticNet logistic regression,weekly_media_exposure,0
7,Khulna,Stage 1,ElasticNet logistic regression,working_status,0
8,Khulna,Stage 1,ElasticNet logistic regression,religion_final,0
9,Khulna,Stage 1,ElasticNet logistic regression,household_head_sex,0



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Khulna development N = 4489,True
4,Khulna heldout N = 574,True
5,Stage-2 development N = 4109,True
6,Exactly 240 candidate summaries,True
7,All 240 candidates successful,True
8,Exactly two joint winners,True
9,Both stages represented,True



LOCKED KHULNA PREDICTIONS:
Rows             : 574
Unique rows      : 574
Probability sum : True
Heldout metrics : NOT CALCULATED

INTEGRITY:
Primary master : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Candidate grid : f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Khulna  : abce108f86cf30a053838d2b53abed51bb27eaa52d79a36b8ccc98abce921969

STATUS: STEP 23 KHULNA IECV PREDICTIONS LOCKED

NEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Mymensingh"


In [1]:
# ============================================================
# STEP 23 — RESTART-SAFE GEOGRAPHIC IECV RESUME
# ONE DIVISION AT A TIME
# FAMILY + STAGE CHECKPOINTING
# FIXED FINAL ROW-KEY MERGE
# ============================================================
#
# COMPLETED SO FAR:
#   Barishal
#   Chattogram
#   Dhaka
#   Khulna
#
# CURRENT TARGET:
#   Mymensingh
#
# Scientific protocol remains unchanged.
# NO held-out performance is calculated here.
# ============================================================


# ============================================================
# 0. TARGET DIVISION
# ============================================================

TARGET_DIVISION = "Mymensingh"

# Later use:
# TARGET_DIVISION = "Rajshahi"
# TARGET_DIVISION = "Rangpur"
# TARGET_DIVISION = "Sylhet"


# ============================================================
# 1. THREAD CONTROL
# ============================================================

import os

EXECUTION_THREADS = 2

os.environ["OMP_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["MKL_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(EXECUTION_THREADS)


# ============================================================
# 2. IMPORTS
# ============================================================

from pathlib import Path

import ast
import datetime as dt
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit

from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from IPython.display import display


# ============================================================
# 3. PROJECT
# ============================================================

ROOT = Path(
    r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
)

DIRS = {
    "root":
        ROOT,

    "processed":
        ROOT / "01_processed_data",

    "qc":
        ROOT / "02_quality_control",

    "protocol":
        ROOT / "03_protocol",

    "splits":
        ROOT / "04_cv_splits",

    "geographic":
        ROOT / "09_geographic_validation",

    "logs":
        ROOT / "15_logs",
}


for path in DIRS.values():

    if isinstance(path, Path):

        path.mkdir(
            parents=True,
            exist_ok=True,
        )


# ============================================================
# 4. CONSTANTS
# ============================================================

SEED = 2026

EXECUTION_SCHEMA_VERSION = (
    "STEP23_GEOGRAPHIC_LAPTOP_v2"
)

METRIC_TIE_TOLERANCE = 1e-12

CALIBRATION_PROBABILITY_CLIP = 1e-6

THRESHOLD_PROBABILITY_CLIP = 1e-12


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)

EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)

EXPECTED_CANDIDATE_SHA256 = (
    "f95dc93431eced694114fe8ed071a2878"
    "e2572fac53aa3b9fd9c6cb8df87c697"
)


DIVISION_EXPECTATIONS = {

    "Barishal": {
        "development":
            4519,
        "stage2":
            4156,
        "heldout":
            544,
    },

    "Chattogram": {
        "development":
            4183,
        "stage2":
            3854,
        "heldout":
            880,
    },

    "Dhaka": {
        "development":
            4310,
        "stage2":
            3950,
        "heldout":
            753,
    },

    "Khulna": {
        "development":
            4489,
        "stage2":
            4109,
        "heldout":
            574,
    },

    "Mymensingh": {
        "development":
            4443,
        "stage2":
            4109,
        "heldout":
            620,
    },

    "Rajshahi": {
        "development":
            4551,
        "stage2":
            4174,
        "heldout":
            512,
    },

    "Rangpur": {
        "development":
            4471,
        "stage2":
            4117,
        "heldout":
            592,
    },

    "Sylhet": {
        "development":
            4475,
        "stage2":
            4151,
        "heldout":
            588,
    },
}


if TARGET_DIVISION not in DIVISION_EXPECTATIONS:

    raise ValueError(
        f"Invalid TARGET_DIVISION: "
        f"{TARGET_DIVISION}"
    )


EXPECTED_DEVELOPMENT_N = (
    DIVISION_EXPECTATIONS[
        TARGET_DIVISION
    ][
        "development"
    ]
)

EXPECTED_STAGE2_DEVELOPMENT_N = (
    DIVISION_EXPECTATIONS[
        TARGET_DIVISION
    ][
        "stage2"
    ]
)

EXPECTED_HELDOUT_N = (
    DIVISION_EXPECTATIONS[
        TARGET_DIVISION
    ][
        "heldout"
    ]
)


# ============================================================
# 5. FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)

OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)

CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)

STEP22_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_IECV_protocol.json"
)

STEP22_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP22_GEOGRAPHIC_IECV_protocol_manifest.json"
)


# ============================================================
# 6. UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda:
                handle.read(
                    1024 * 1024
                ),
            b"",
        ):

            digest.update(
                block
            )

    return digest.hexdigest()


def sha256_text(text):

    return hashlib.sha256(
        str(text).encode(
            "utf-8"
        )
    ).hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(
            handle
        )


def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def safe_name(value):

    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(">=", "GE")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column
        in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(
            alias
        )

        if key in lookup:

            return lookup[
                key
            ]


    raise RuntimeError(
        "Required column not found.\n"
        f"Aliases: {aliases}\n"
        f"Available: {list(dataframe.columns)}"
    )


def canonical_row_key(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(
                dtype=float
            ),
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ).astype(
                np.int64
            ),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


def parse_parameter_dictionary(value):

    if isinstance(
        value,
        dict,
    ):

        return value.copy()


    if pd.isna(
        value
    ):

        return {}


    text = str(
        value
    ).strip()


    if text == "":

        return {}


    try:

        parsed = json.loads(
            text
        )

    except json.JSONDecodeError:

        parsed = ast.literal_eval(
            text
        )


    if not isinstance(
        parsed,
        dict,
    ):

        raise RuntimeError(
            "Parameter field did not decode "
            "to a dictionary."
        )


    return parsed


def row_key_hash(
    dataframe,
):

    return sha256_text(
        "|".join(
            dataframe[
                "__ROW_KEY"
            ]
            .astype(str)
            .tolist()
        )
    )


# ============================================================
# 7. INPUT / HASH CHECK
# ============================================================

for path in [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    CANDIDATE_REGISTRY_FILE,
    STEP22_PROTOCOL_FILE,
    STEP22_MANIFEST_FILE,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing required file:\n"
            f"{path}"
        )


if (
    sha256_file(
        PRIMARY_FILE
    )
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    sha256_file(
        OUTER_SPLIT_FILE
    )
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Step-13 split hash mismatch."
    )


if (
    sha256_file(
        CANDIDATE_REGISTRY_FILE
    )
    !=
    EXPECTED_CANDIDATE_SHA256
):

    raise RuntimeError(
        "Candidate-registry hash mismatch."
    )


step22_manifest = read_json(
    STEP22_MANIFEST_FILE
)


if (
    step22_manifest.get(
        "status"
    )
    !=
    "GEOGRAPHIC_IECV_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 22 protocol is not frozen."
    )


STEP22_PROTOCOL_SHA256 = (
    sha256_file(
        STEP22_PROTOCOL_FILE
    )
)


# ============================================================
# 8. LOAD DATA
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)

outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)

candidate_registry = pd.read_csv(
    CANDIDATE_REGISTRY_FILE
)


if master.shape != (
    5063,
    23,
):

    raise RuntimeError(
        f"Unexpected master shape: "
        f"{master.shape}"
    )


if len(
    candidate_registry
) != 120:

    raise RuntimeError(
        "Candidate registry must contain 120 candidates."
    )


required_candidate_columns = {
    "Candidate_ID",
    "Model",
    "Preprocessing",
    "Deterministic_Tie_Rank",
    "Tuned_Params_JSON",
    "Fixed_Params_JSON",
}


missing_candidate_columns = (
    required_candidate_columns
    -
    set(
        candidate_registry.columns
    )
)


if missing_candidate_columns:

    raise RuntimeError(
        "Candidate registry missing columns:\n"
        f"{sorted(missing_candidate_columns)}"
    )


# ============================================================
# 9. ROW/FOLD RECONSTRUCTION
# ============================================================

MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)

OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)

OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


master = master.copy()

outer_assignment = (
    outer_assignment.copy()
)


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


if master[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate master row key."
    )


if outer_assignment[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate fold row key."
    )


if set(
    master[
        "__ROW_KEY"
    ]
) != set(
    outer_assignment[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Frozen split does not exactly cover master."
    )


fold_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__ROW_KEY"
].map(
    fold_map
)


if master[
    "__DEVELOPMENT_CV_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing development-CV fold."
    )


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__DEVELOPMENT_CV_FOLD"
].astype(int)


# ============================================================
# 10. PREDICTORS
# ============================================================

GEO_NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]


GEO_ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]


GEO_NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]


GEO_BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]


GEO_CATEGORICAL_PREDICTORS = (
    GEO_ORDERED_CATEGORICAL_PREDICTORS
    +
    GEO_NOMINAL_CATEGORICAL_PREDICTORS
    +
    GEO_BINARY_CATEGORICAL_PREDICTORS
)


GEO_PREDICTORS = (
    GEO_NUMERIC_PREDICTORS
    +
    GEO_CATEGORICAL_PREDICTORS
)


if len(
    GEO_PREDICTORS
) != 14:

    raise RuntimeError(
        "Geographic predictor count != 14."
    )


if "division" in GEO_PREDICTORS:

    raise RuntimeError(
        "Division leaked into predictor set."
    )


# ============================================================
# 11. MODEL ORDER
# ============================================================

candidate_counts = (
    candidate_registry
    .groupby(
        "Model"
    )
    .size()
)


if (
    len(
        candidate_counts
    )
    != 10
    or
    (
        candidate_counts
        !=
        12
    ).any()
):

    raise RuntimeError(
        "Expected 10 families x 12 candidates."
    )


MODEL_ORDER = (
    candidate_registry
    .groupby(
        "Model",
        as_index=False,
    )[
        "Deterministic_Tie_Rank"
    ]
    .min()
    .sort_values(
        "Deterministic_Tie_Rank"
    )[
        "Model"
    ]
    .tolist()
)


# ============================================================
# 12. DEVELOPMENT / HELDOUT
# ============================================================

master[
    "ANC_CLASS"
] = pd.to_numeric(
    master[
        "ANC_CLASS"
    ],
    errors="raise",
).astype(int)


development = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        !=
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


heldout = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        ==
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin(
        [
            1,
            2,
        ]
    )
    .sum()
)


if len(
    development
) != EXPECTED_DEVELOPMENT_N:

    raise RuntimeError(
        f"Development N mismatch: "
        f"{len(development)}"
    )


if len(
    heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        f"Heldout N mismatch: "
        f"{len(heldout)}"
    )


if (
    stage2_development_n
    !=
    EXPECTED_STAGE2_DEVELOPMENT_N
):

    raise RuntimeError(
        f"Stage-2 development N mismatch: "
        f"{stage2_development_n}"
    )


if (
    set(
        development[
            "__ROW_KEY"
        ]
    )
    &
    set(
        heldout[
            "__ROW_KEY"
        ]
    )
):

    raise RuntimeError(
        "Development/heldout overlap."
    )


# ============================================================
# 13. WEIGHTS / STAGE DATA
# ============================================================

def normalized_training_weight(
    dataframe,
):

    weight = (
        dataframe[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    normalized = (
        weight
        /
        np.mean(
            weight
        )
    )


    if not np.isclose(
        normalized.mean(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Weight normalization failed."
        )


    return normalized


def prepare_stage(
    dataframe,
    stage_name,
):

    data = dataframe.copy()


    if stage_name == "Stage 1":

        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            0
        ).astype(int)


    elif stage_name == "Stage 2":

        data = (
            data.loc[
                data[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
            ]
            .copy()
        )


        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            2
        ).astype(int)


    else:

        raise ValueError(
            stage_name
        )


    return data.reset_index(
        drop=True
    )


# ============================================================
# 14. PREPROCESSING
# ============================================================

def build_representations(
    training_data,
    scoring_data,
):

    train_num = (
        training_data[
            GEO_NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    score_num = (
        scoring_data[
            GEO_NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    train_cat = (
        training_data[
            GEO_CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    score_cat = (
        scoring_data[
            GEO_CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop=None,
        sparse_output=False,
        dtype=np.float64,
    )


    train_ohe = encoder.fit_transform(
        train_cat
    )


    score_ohe = encoder.transform(
        score_cat
    )


    train_unscaled = np.hstack(
        [
            train_num,
            train_ohe,
        ]
    )


    score_unscaled = np.hstack(
        [
            score_num,
            score_ohe,
        ]
    )


    scaler = StandardScaler()


    train_scaled_num = (
        scaler.fit_transform(
            train_num
        )
    )


    score_scaled_num = (
        scaler.transform(
            score_num
        )
    )


    train_scaled = np.hstack(
        [
            train_scaled_num,
            train_ohe,
        ]
    )


    score_scaled = np.hstack(
        [
            score_scaled_num,
            score_ohe,
        ]
    )


    train_native = (
        training_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    score_native = (
        scoring_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    unseen_counts = {}


    for column in GEO_NUMERIC_PREDICTORS:

        train_native[
            column
        ] = pd.to_numeric(
            train_native[
                column
            ],
            errors="raise",
        ).astype(float)


        score_native[
            column
        ] = pd.to_numeric(
            score_native[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in GEO_CATEGORICAL_PREDICTORS:

        train_values = (
            train_native[
                column
            ]
            .astype(str)
        )


        score_values = (
            score_native[
                column
            ]
            .astype(str)
        )


        categories = list(
            pd.unique(
                train_values
            )
        )


        unseen_counts[
            column
        ] = int(
            (
                ~score_values.isin(
                    categories
                )
            )
            .sum()
        )


        train_native[
            column
        ] = pd.Categorical(
            train_values,
            categories=categories,
        )


        score_native[
            column
        ] = pd.Categorical(
            score_values,
            categories=categories,
        )


    train_catboost = (
        training_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    score_catboost = (
        scoring_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    for column in GEO_NUMERIC_PREDICTORS:

        train_catboost[
            column
        ] = pd.to_numeric(
            train_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


        score_catboost[
            column
        ] = pd.to_numeric(
            score_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in GEO_CATEGORICAL_PREDICTORS:

        train_catboost[
            column
        ] = train_catboost[
            column
        ].astype(str)


        score_catboost[
            column
        ] = score_catboost[
            column
        ].astype(str)


    return {
        "Scaled OHE":
            (
                train_scaled,
                score_scaled,
            ),

        "Unscaled OHE":
            (
                train_unscaled,
                score_unscaled,
            ),

        "Native categorical":
            (
                train_native,
                score_native,
            ),

        "CatBoost native":
            (
                train_catboost,
                score_catboost,
            ),

        "__UNSEEN_COUNTS":
            unseen_counts,
    }


# ============================================================
# 15. MODEL CONSTRUCTION
# ============================================================

def instantiate_candidate(
    candidate_row,
):

    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    params = {
        **parse_parameter_dictionary(
            candidate_row[
                "Fixed_Params_JSON"
            ]
        ),

        **parse_parameter_dictionary(
            candidate_row[
                "Tuned_Params_JSON"
            ]
        ),
    }


    if model_name in [
        "Random Forest",
        "ExtraTrees",
        "XGBoost",
        "LightGBM",
    ]:

        params[
            "n_jobs"
        ] = EXECUTION_THREADS


    if model_name == "CatBoost":

        params[
            "thread_count"
        ] = EXECUTION_THREADS


    if model_name == (
        "ElasticNet logistic regression"
    ):

        return LogisticRegression(
            **params
        )


    if model_name == "RBF SVM":

        params.pop(
            "probability",
            None,
        )


        return SVC(
            **params
        )


    if model_name == "Random Forest":

        return RandomForestClassifier(
            **params
        )


    if model_name == "ExtraTrees":

        return ExtraTreesClassifier(
            **params
        )


    if model_name == "AdaBoost":

        aliases = {
            "estimatormaxdepth",
            "baseestimatormaxdepth",
            "stumpmaxdepth",
            "decisiontreemaxdepth",
            "maxdepth",
        }


        depth_keys = [
            key

            for key
            in list(
                params.keys()
            )

            if normalize_name(
                key
            )
            in aliases
        ]


        if len(
            depth_keys
        ) != 1:

            raise RuntimeError(
                "AdaBoost depth parameter "
                "could not be uniquely identified."
            )


        depth = int(
            params.pop(
                depth_keys[0]
            )
        )


        if (
            "estimator"
            in params
            and
            not hasattr(
                params[
                    "estimator"
                ],
                "fit",
            )
        ):

            params.pop(
                "estimator"
            )


        return AdaBoostClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=depth,
                random_state=SEED,
            ),
            **params,
        )


    if model_name == "GradientBoosting":

        return GradientBoostingClassifier(
            **params
        )


    if model_name == "HistGradientBoosting":

        return HistGradientBoostingClassifier(
            **params
        )


    if model_name == "XGBoost":

        return XGBClassifier(
            **params
        )


    if model_name == "LightGBM":

        return LGBMClassifier(
            **params
        )


    if model_name == "CatBoost":

        return CatBoostClassifier(
            **params
        )


    raise RuntimeError(
        f"Unknown model: {model_name}"
    )


def fit_candidate(
    estimator,
    model_name,
    X_train,
    y_train,
    training_weight,
):

    with warnings.catch_warnings():

        warnings.simplefilter(
            "error",
            ConvergenceWarning,
        )


        if model_name == "CatBoost":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
                cat_features=GEO_CATEGORICAL_PREDICTORS,
                verbose=False,
            )


        elif model_name == "LightGBM":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
                categorical_feature=GEO_CATEGORICAL_PREDICTORS,
            )


        else:

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
            )


    return estimator


def positive_score(
    estimator,
    model_name,
    X,
):

    if model_name == "RBF SVM":

        score = np.asarray(
            estimator.decision_function(
                X
            ),
            dtype=float,
        ).reshape(-1)


    else:

        classes = np.asarray(
            estimator.classes_
        )


        positive_index = int(
            np.where(
                classes
                ==
                1
            )[0][0]
        )


        score = np.asarray(
            estimator.predict_proba(
                X
            ),
            dtype=float,
        )[
            :,
            positive_index
        ]


    if not np.isfinite(
        score
    ).all():

        raise RuntimeError(
            "Non-finite raw score."
        )


    return score


def get_representation(
    representations,
    model_name,
    preprocessing,
):

    if model_name == "CatBoost":

        return representations[
            "CatBoost native"
        ]


    return representations[
        preprocessing
    ]


# ============================================================
# 16. CV CACHE
# ============================================================

def build_cv_cache(
    stage_development,
):

    cache = {}


    for fold in [
        0,
        1,
        2,
        3,
        4,
    ]:

        validation_mask = (
            stage_development[
                "__DEVELOPMENT_CV_FOLD"
            ]
            ==
            fold
        )


        train_pos = np.where(
            ~validation_mask.to_numpy()
        )[0]


        validation_pos = np.where(
            validation_mask.to_numpy()
        )[0]


        train = (
            stage_development.iloc[
                train_pos
            ]
            .copy()
        )


        validation = (
            stage_development.iloc[
                validation_pos
            ]
            .copy()
        )


        cache[
            fold
        ] = {
            "validation_positions":
                validation_pos,

            "y_train":
                train[
                    "__TARGET"
                ]
                .to_numpy(
                    dtype=int
                ),

            "training_weight":
                normalized_training_weight(
                    train
                ),

            "representations":
                build_representations(
                    train,
                    validation,
                ),
        }


    return cache


# ============================================================
# 17. FAMILY CHECKPOINTS
# ============================================================

def family_paths(
    stage_name,
    model_name,
):

    prefix = (
        f"STEP23_GEOGRAPHIC_"
        f"{safe_name(TARGET_DIVISION)}_"
        f"{safe_name(stage_name)}_"
        f"{safe_name(model_name)}"
    )


    return {
        "metrics":
            DIRS["geographic"]
            /
            f"{prefix}_candidate_metrics.csv",

        "oof":
            DIRS["geographic"]
            /
            f"{prefix}_candidate_OOF_scores.npz",

        "manifest":
            DIRS["logs"]
            /
            f"{prefix}_family_checkpoint.json",
    }


def family_checkpoint_valid(
    stage_name,
    model_name,
    stage_development,
):

    paths = family_paths(
        stage_name,
        model_name,
    )


    if not all(
        path.exists()
        for path in paths.values()
    ):

        return False


    try:

        manifest = read_json(
            paths[
                "manifest"
            ]
        )

    except Exception:

        return False


    return (
        manifest.get(
            "status"
        )
        ==
        "STEP23_FAMILY_COMPLETE"
        and
        manifest.get(
            "execution_schema_version"
        )
        ==
        EXECUTION_SCHEMA_VERSION
        and
        manifest.get(
            "heldout_division"
        )
        ==
        TARGET_DIVISION
        and
        manifest.get(
            "stage"
        )
        ==
        stage_name
        and
        manifest.get(
            "model"
        )
        ==
        model_name
        and
        manifest.get(
            "stage_row_key_sha256"
        )
        ==
        row_key_hash(
            stage_development
        )
        and
        manifest.get(
            "metrics_sha256"
        )
        ==
        sha256_file(
            paths[
                "metrics"
            ]
        )
        and
        manifest.get(
            "oof_sha256"
        )
        ==
        sha256_file(
            paths[
                "oof"
            ]
        )
    )


def evaluate_family(
    stage_name,
    stage_development,
    cv_cache,
    model_name,
):

    paths = family_paths(
        stage_name,
        model_name,
    )


    if family_checkpoint_valid(
        stage_name,
        model_name,
        stage_development,
    ):

        metrics = pd.read_csv(
            paths[
                "metrics"
            ]
        )


        loaded = np.load(
            paths[
                "oof"
            ],
            allow_pickle=False,
        )


        ids = loaded[
            "candidate_ids"
        ].astype(str)


        scores = loaded[
            "scores"
        ]


        print(
            f"    {model_name}: checkpoint loaded"
        )


        return (
            metrics,
            {
                candidate_id:
                    scores[
                        i
                    ].copy()

                for i, candidate_id
                in enumerate(ids)
            },
        )


    family_candidates = (
        candidate_registry.loc[
            candidate_registry[
                "Model"
            ]
            ==
            model_name
        ]
        .copy()
        .sort_values(
            "Deterministic_Tie_Rank"
        )
    )


    n = len(
        stage_development
    )


    y_all = (
        stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        )
    )


    w_all = (
        stage_development[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    rows = []

    score_lookup = {}


    for _, candidate_row in (
        family_candidates.iterrows()
    ):

        candidate_id = str(
            candidate_row[
                "Candidate_ID"
            ]
        )


        oof = np.full(
            n,
            np.nan,
        )


        for fold in [
            0,
            1,
            2,
            3,
            4,
        ]:

            info = cv_cache[
                fold
            ]


            X_train, X_valid = (
                get_representation(
                    info[
                        "representations"
                    ],
                    model_name,
                    candidate_row[
                        "Preprocessing"
                    ],
                )
            )


            estimator = instantiate_candidate(
                candidate_row
            )


            estimator = fit_candidate(
                estimator,
                model_name,
                X_train,
                info[
                    "y_train"
                ],
                info[
                    "training_weight"
                ],
            )


            oof[
                info[
                    "validation_positions"
                ]
            ] = positive_score(
                estimator,
                model_name,
                X_valid,
            )


        if not np.isfinite(
            oof
        ).all():

            raise RuntimeError(
                f"Incomplete OOF: "
                f"{candidate_id}"
            )


        ap = float(
            average_precision_score(
                y_all,
                oof,
                sample_weight=w_all,
            )
        )


        auc = float(
            roc_auc_score(
                y_all,
                oof,
                sample_weight=w_all,
            )
        )


        rows.append(
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Deterministic_Tie_Rank":
                    int(
                        candidate_row[
                            "Deterministic_Tie_Rank"
                        ]
                    ),

                "Development_CV_Weighted_AP":
                    ap,

                "Development_CV_Weighted_ROC_AUC":
                    auc,

                "Status":
                    "SUCCESS",

                "Error":
                    None,
            }
        )


        score_lookup[
            candidate_id
        ] = oof


    metrics = pd.DataFrame(
        rows
    )


    ordered_ids = (
        family_candidates[
            "Candidate_ID"
        ]
        .astype(str)
        .tolist()
    )


    metrics.to_csv(
        paths[
            "metrics"
        ],
        index=False,
    )


    np.savez_compressed(
        paths[
            "oof"
        ],
        candidate_ids=np.asarray(
            ordered_ids,
            dtype="U64",
        ),
        scores=np.vstack(
            [
                score_lookup[
                    candidate_id
                ]

                for candidate_id
                in ordered_ids
            ]
        ),
    )


    manifest = {
        "status":
            "STEP23_FAMILY_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "model":
            model_name,

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),

        "metrics_sha256":
            sha256_file(
                paths[
                    "metrics"
                ]
            ),

        "oof_sha256":
            sha256_file(
                paths[
                    "oof"
                ]
            ),
    }


    with open(
        paths[
            "manifest"
        ],
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            manifest,
            handle,
            indent=2,
        )


    print(
        f"    {model_name}: complete "
        f"| best development-CV AP="
        f"{metrics['Development_CV_Weighted_AP'].max():.6f}"
    )


    return (
        metrics,
        score_lookup,
    )


# ============================================================
# 18. CALIBRATION / THRESHOLD
# ============================================================

def calibration_predictor(
    raw_score,
    model_name,
):

    raw_score = np.asarray(
        raw_score,
        dtype=float,
    )


    if model_name == "RBF SVM":

        return raw_score


    p = np.clip(
        raw_score,
        CALIBRATION_PROBABILITY_CLIP,
        1
        -
        CALIBRATION_PROBABILITY_CLIP,
    )


    return np.log(
        p
        /
        (
            1 - p
        )
    )


def fit_calibrator(
    raw_score,
    y,
    weight,
    model_name,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    weight = np.asarray(
        weight,
        dtype=float,
    )


    weight = (
        weight
        /
        weight.mean()
    )


    z = calibration_predictor(
        raw_score,
        model_name,
    )


    def objective(theta):

        alpha, beta = theta

        eta = (
            alpha
            +
            beta * z
        )


        residual = (
            expit(
                eta
            )
            -
            y
        )


        loss = (
            np.sum(
                weight
                *
                (
                    np.logaddexp(
                        0,
                        eta,
                    )
                    -
                    y * eta
                )
            )
            /
            np.sum(
                weight
            )
        )


        grad = np.array(
            [
                np.sum(
                    weight
                    *
                    residual
                )
                /
                np.sum(
                    weight
                ),

                np.sum(
                    weight
                    *
                    residual
                    *
                    z
                )
                /
                np.sum(
                    weight
                ),
            ]
        )


        return (
            float(
                loss
            ),
            grad,
        )


    fit = minimize(
        objective,
        x0=np.array(
            [
                0.0,
                1.0,
            ]
        ),
        method="L-BFGS-B",
        jac=True,
    )


    if not fit.success:

        raise RuntimeError(
            f"Calibration failed: "
            f"{fit.message}"
        )


    alpha = float(
        fit.x[
            0
        ]
    )


    beta = float(
        fit.x[
            1
        ]
    )


    return {
        "alpha":
            alpha,

        "beta":
            beta,

        "iterations":
            int(
                fit.nit
            ),

        "calibrated":
            expit(
                alpha
                +
                beta * z
            ),
    }


def apply_calibrator(
    raw_score,
    model_name,
    alpha,
    beta,
):

    return expit(
        alpha
        +
        beta
        *
        calibration_predictor(
            raw_score,
            model_name,
        )
    )


def optimize_threshold(
    y,
    probability,
    weight,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        THRESHOLD_PROBABILITY_CLIP,
        1
        -
        THRESHOLD_PROBABILITY_CLIP,
    )


    weight = np.asarray(
        weight,
        dtype=float,
    )


    unique_p = np.unique(
        p
    )


    candidates = np.unique(
        np.concatenate(
            [
                [0.0],

                (
                    unique_p[:-1]
                    +
                    unique_p[1:]
                )
                /
                2.0,

                [1.0],
            ]
        )
    )


    rows = []


    for threshold in candidates:

        pred = (
            p
            >=
            threshold
        ).astype(int)


        positive = (
            y
            ==
            1
        )


        negative = (
            y
            ==
            0
        )


        tp = weight[
            positive
            &
            (
                pred
                ==
                1
            )
        ].sum()


        fn = weight[
            positive
            &
            (
                pred
                ==
                0
            )
        ].sum()


        tn = weight[
            negative
            &
            (
                pred
                ==
                0
            )
        ].sum()


        fp = weight[
            negative
            &
            (
                pred
                ==
                1
            )
        ].sum()


        sens = (
            tp
            /
            (
                tp
                +
                fn
            )
        )


        spec = (
            tn
            /
            (
                tn
                +
                fp
            )
        )


        rows.append(
            {
                "threshold":
                    threshold,

                "balanced_accuracy":
                    (
                        sens
                        +
                        spec
                    )
                    /
                    2,

                "sensitivity":
                    sens,

                "specificity":
                    spec,

                "balance_floor":
                    min(
                        sens,
                        spec,
                    ),
            }
        )


    result = pd.DataFrame(
        rows
    )


    best_ba = result[
        "balanced_accuracy"
    ].max()


    tied = result.loc[
        np.isclose(
            result[
                "balanced_accuracy"
            ],
            best_ba,
            atol=1e-12,
            rtol=0,
        )
    ]


    best_floor = tied[
        "balance_floor"
    ].max()


    tied = tied.loc[
        np.isclose(
            tied[
                "balance_floor"
            ],
            best_floor,
            atol=1e-12,
            rtol=0,
        )
    ]


    return (
        tied
        .sort_values(
            "threshold"
        )
        .iloc[0]
        .to_dict()
    )


# ============================================================
# 19. REFIT
# ============================================================

def refit_winner(
    stage_development,
    candidate_row,
):

    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    representations = (
        build_representations(
            stage_development,
            heldout,
        )
    )


    X_train, X_holdout = (
        get_representation(
            representations,
            model_name,
            candidate_row[
                "Preprocessing"
            ],
        )
    )


    estimator = instantiate_candidate(
        candidate_row
    )


    estimator = fit_candidate(
        estimator,
        model_name,
        X_train,
        stage_development[
            "__TARGET"
        ].to_numpy(
            dtype=int
        ),
        normalized_training_weight(
            stage_development
        ),
    )


    return (
        positive_score(
            estimator,
            model_name,
            X_holdout,
        ),

        representations[
            "__UNSEEN_COUNTS"
        ],
    )


# ============================================================
# 20. STAGE CHECKPOINT
# ============================================================

def stage_paths(
    stage_name,
):

    prefix = (
        f"STEP23_GEOGRAPHIC_"
        f"{safe_name(TARGET_DIVISION)}_"
        f"{safe_name(stage_name)}"
    )


    return {
        "candidate_metrics":
            DIRS["geographic"]
            /
            f"{prefix}_all_candidate_metrics.csv",

        "winner":
            DIRS["geographic"]
            /
            f"{prefix}_winner.csv",

        "calibration":
            DIRS["geographic"]
            /
            f"{prefix}_calibration_threshold.csv",

        "development_oof":
            DIRS["geographic"]
            /
            f"{prefix}_winner_development_OOF_RESTRICTED.csv",

        "heldout_stage":
            DIRS["geographic"]
            /
            f"{prefix}_heldout_stage_predictions_RESTRICTED.csv",

        "unseen":
            DIRS["qc"]
            /
            f"{prefix}_unseen_category_qc.csv",

        "manifest":
            DIRS["logs"]
            /
            f"{prefix}_stage_checkpoint.json",
    }


def stage_checkpoint_valid(
    stage_name,
    stage_development,
):

    paths = stage_paths(
        stage_name
    )


    if not all(
        path.exists()
        for path in paths.values()
    ):

        return False


    try:

        manifest = read_json(
            paths[
                "manifest"
            ]
        )

    except Exception:

        return False


    return (
        manifest.get(
            "status"
        )
        ==
        "STEP23_STAGE_COMPLETE"
        and
        manifest.get(
            "execution_schema_version"
        )
        ==
        EXECUTION_SCHEMA_VERSION
        and
        manifest.get(
            "heldout_division"
        )
        ==
        TARGET_DIVISION
        and
        manifest.get(
            "stage"
        )
        ==
        stage_name
        and
        manifest.get(
            "stage_row_key_sha256"
        )
        ==
        row_key_hash(
            stage_development
        )
    )


def execute_stage(
    stage_name,
):

    stage_development = (
        prepare_stage(
            development,
            stage_name,
        )
    )


    paths = stage_paths(
        stage_name
    )


    if stage_checkpoint_valid(
        stage_name,
        stage_development,
    ):

        print(
            f"\n{stage_name}: "
            "completed checkpoint loaded."
        )


        return {
            "candidate_metrics":
                pd.read_csv(
                    paths[
                        "candidate_metrics"
                    ]
                ),

            "winner":
                pd.read_csv(
                    paths[
                        "winner"
                    ]
                ),

            "calibration":
                pd.read_csv(
                    paths[
                        "calibration"
                    ]
                ),

            "development_oof":
                pd.read_csv(
                    paths[
                        "development_oof"
                    ]
                ),

            "heldout_stage":
                pd.read_csv(
                    paths[
                        "heldout_stage"
                    ]
                ),

            "unseen":
                pd.read_csv(
                    paths[
                        "unseen"
                    ]
                ),
        }


    print(
        "\n"
        +
        "-" * 78
    )


    print(
        f"{TARGET_DIVISION} | "
        f"{stage_name}"
    )


    print(
        f"Development N = "
        f"{len(stage_development)}"
    )


    print(
        f"Heldout scoring N = "
        f"{len(heldout)}"
    )


    print(
        "-" * 78
    )


    cv_cache = build_cv_cache(
        stage_development
    )


    all_metrics = []

    all_scores = {}


    for i, model_name in enumerate(
        MODEL_ORDER,
        start=1,
    ):

        print(
            f"  [{i}/10] "
            f"{model_name}"
        )


        metrics, scores = (
            evaluate_family(
                stage_name,
                stage_development,
                cv_cache,
                model_name,
            )
        )


        all_metrics.append(
            metrics
        )


        all_scores.update(
            scores
        )


    candidate_metrics = pd.concat(
        all_metrics,
        ignore_index=True,
    )


    best_ap = candidate_metrics[
        "Development_CV_Weighted_AP"
    ].max()


    tied = candidate_metrics.loc[
        np.isclose(
            candidate_metrics[
                "Development_CV_Weighted_AP"
            ],
            best_ap,
            atol=1e-12,
            rtol=0,
        )
    ]


    best_auc = tied[
        "Development_CV_Weighted_ROC_AUC"
    ].max()


    winner = (
        tied.loc[
            np.isclose(
                tied[
                    "Development_CV_Weighted_ROC_AUC"
                ],
                best_auc,
                atol=1e-12,
                rtol=0,
            )
        ]
        .sort_values(
            "Deterministic_Tie_Rank"
        )
        .iloc[0]
    )


    candidate_id = str(
        winner[
            "Candidate_ID"
        ]
    )


    model_name = str(
        winner[
            "Model"
        ]
    )


    candidate_row = (
        candidate_registry.loc[
            candidate_registry[
                "Candidate_ID"
            ]
            .astype(str)
            ==
            candidate_id
        ]
        .iloc[0]
    )


    winner_oof = all_scores[
        candidate_id
    ]


    print(
        f"\n  JOINT WINNER: "
        f"{model_name} | "
        f"{candidate_id}"
    )


    print(
        f"  Development-CV AP  = "
        f"{winner['Development_CV_Weighted_AP']:.6f}"
    )


    print(
        f"  Development-CV AUC = "
        f"{winner['Development_CV_Weighted_ROC_AUC']:.6f}"
    )


    calibration = fit_calibrator(
        winner_oof,
        stage_development[
            "__TARGET"
        ].to_numpy(
            dtype=int
        ),
        stage_development[
            "SURVEY_WEIGHT"
        ].to_numpy(
            dtype=float
        ),
        model_name,
    )


    threshold_result = optimize_threshold(
        stage_development[
            "__TARGET"
        ].to_numpy(
            dtype=int
        ),
        calibration[
            "calibrated"
        ],
        stage_development[
            "SURVEY_WEIGHT"
        ].to_numpy(
            dtype=float
        ),
    )


    threshold = float(
        threshold_result[
            "threshold"
        ]
    )


    print(
        f"  Optimized threshold = "
        f"{threshold:.6f}"
    )


    (
        heldout_raw,
        unseen_counts,
    ) = refit_winner(
        stage_development,
        candidate_row,
    )


    heldout_calibrated = (
        apply_calibrator(
            heldout_raw,
            model_name,
            calibration[
                "alpha"
            ],
            calibration[
                "beta"
            ],
        )
    )


    winner_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Development_CV_Weighted_AP":
                    float(
                        winner[
                            "Development_CV_Weighted_AP"
                        ]
                    ),

                "Development_CV_Weighted_ROC_AUC":
                    float(
                        winner[
                            "Development_CV_Weighted_ROC_AUC"
                        ]
                    ),

                "Execution_Threads":
                    EXECUTION_THREADS,
            }
        ]
    )


    calibration_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Calibration_Alpha":
                    calibration[
                        "alpha"
                    ],

                "Calibration_Beta":
                    calibration[
                        "beta"
                    ],

                "Calibration_Iterations":
                    calibration[
                        "iterations"
                    ],

                "Optimized_Threshold":
                    threshold,

                "Development_Threshold_Balanced_Accuracy":
                    threshold_result[
                        "balanced_accuracy"
                    ],

                "Development_Threshold_Sensitivity":
                    threshold_result[
                        "sensitivity"
                    ],

                "Development_Threshold_Specificity":
                    threshold_result[
                        "specificity"
                    ],
            }
        ]
    )


    development_oof_df = pd.DataFrame(
        {
            "_ROW_KEY":
                stage_development[
                    "__ROW_KEY"
                ],

            "Heldout_Division":
                TARGET_DIVISION,

            "Development_CV_Fold":
                stage_development[
                    "__DEVELOPMENT_CV_FOLD"
                ],

            "Stage":
                stage_name,

            "Model":
                model_name,

            "Candidate_ID":
                candidate_id,

            "True_Target":
                stage_development[
                    "__TARGET"
                ],

            "SURVEY_WEIGHT":
                stage_development[
                    "SURVEY_WEIGHT"
                ],

            "Development_OOF_Raw_Score":
                winner_oof,

            "Development_OOF_Calibrated_Probability":
                calibration[
                    "calibrated"
                ],
        }
    )


    heldout_stage_df = pd.DataFrame(
        {
            "_ROW_KEY":
                heldout[
                    "__ROW_KEY"
                ],

            "Heldout_Division":
                TARGET_DIVISION,

            "Stage":
                stage_name,

            "Model":
                model_name,

            "Candidate_ID":
                candidate_id,

            "Heldout_Raw_Score":
                heldout_raw,

            "Heldout_Calibrated_Probability":
                heldout_calibrated,

            "Optimized_Threshold":
                threshold,
        }
    )


    unseen_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Predictor":
                    predictor,

                "Heldout_Unseen_Category_Rows":
                    count,
            }

            for predictor, count
            in unseen_counts.items()
        ]
    )


    candidate_metrics.to_csv(
        paths[
            "candidate_metrics"
        ],
        index=False,
    )


    winner_df.to_csv(
        paths[
            "winner"
        ],
        index=False,
    )


    calibration_df.to_csv(
        paths[
            "calibration"
        ],
        index=False,
    )


    development_oof_df.to_csv(
        paths[
            "development_oof"
        ],
        index=False,
    )


    heldout_stage_df.to_csv(
        paths[
            "heldout_stage"
        ],
        index=False,
    )


    unseen_df.to_csv(
        paths[
            "unseen"
        ],
        index=False,
    )


    manifest = {
        "status":
            "STEP23_STAGE_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),
    }


    with open(
        paths[
            "manifest"
        ],
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            manifest,
            handle,
            indent=2,
        )


    print(
        f"  {stage_name} CHECKPOINT SAVED."
    )


    return {
        "candidate_metrics":
            candidate_metrics,

        "winner":
            winner_df,

        "calibration":
            calibration_df,

        "development_oof":
            development_oof_df,

        "heldout_stage":
            heldout_stage_df,

        "unseen":
            unseen_df,
    }


# ============================================================
# 21. RUN STAGES
# ============================================================

print(
    "\n"
    +
    "=" * 78
)


print(
    f"STEP 23 — "
    f"{TARGET_DIVISION.upper()} "
    f"GEOGRAPHIC IECV"
)


print(
    "=" * 78
)


print(
    f"Held-out division : "
    f"{TARGET_DIVISION}"
)


print(
    f"Development N     : "
    f"{len(development)}"
)


print(
    f"Stage-2 dev N     : "
    f"{stage2_development_n}"
)


print(
    f"Held-out N        : "
    f"{len(heldout)}"
)


print(
    f"CPU threads       : "
    f"{EXECUTION_THREADS}"
)


print(
    "Division predictor: EXCLUDED"
)


stage1_result = execute_stage(
    "Stage 1"
)


stage2_result = execute_stage(
    "Stage 2"
)


# ============================================================
# 22. CORRECTED FINAL MERGE
# ============================================================

stage1_heldout = (
    stage1_result[
        "heldout_stage"
    ]
    .copy()
)


stage2_heldout = (
    stage2_result[
        "heldout_stage"
    ]
    .copy()
)


heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY":
                "_ROW_KEY",

            "ANC_CLASS":
                "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked[
    "Heldout_Division"
] = TARGET_DIVISION


q0 = locked[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(
    dtype=float
)


t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(
    dtype=float
)


t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(
    dtype=float
)


locked[
    "P_NO_ANC"
] = q0


locked[
    "P_ANC_1_7"
] = (
    (1 - q0)
    *
    (1 - q2)
)


locked[
    "P_ANC_GE8"
] = (
    (1 - q0)
    *
    q2
)


locked[
    "P_SUM"
] = (
    locked[
        "P_NO_ANC"
    ]
    +
    locked[
        "P_ANC_1_7"
    ]
    +
    locked[
        "P_ANC_GE8"
    ]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0 >= t1,
    0,
    np.where(
        q2 >= t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 23. COMBINE / QC
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result[
            "candidate_metrics"
        ],
        stage2_result[
            "candidate_metrics"
        ],
    ],
    ignore_index=True,
)


winners_all = pd.concat(
    [
        stage1_result[
            "winner"
        ],
        stage2_result[
            "winner"
        ],
    ],
    ignore_index=True,
)


calibration_all = pd.concat(
    [
        stage1_result[
            "calibration"
        ],
        stage2_result[
            "calibration"
        ],
    ],
    ignore_index=True,
)


development_oof_all = pd.concat(
    [
        stage1_result[
            "development_oof"
        ],
        stage2_result[
            "development_oof"
        ],
    ],
    ignore_index=True,
)


unseen_all = pd.concat(
    [
        stage1_result[
            "unseen"
        ],
        stage2_result[
            "unseen"
        ],
    ],
    ignore_index=True,
)


hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                sha256_file(
                    PRIMARY_FILE
                )
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                sha256_file(
                    CANDIDATE_REGISTRY_FILE
                )
                ==
                EXPECTED_CANDIDATE_SHA256,
        },

        {
            "Check":
                "Development N correct",

            "PASS":
                len(
                    development
                )
                ==
                EXPECTED_DEVELOPMENT_N,
        },

        {
            "Check":
                "Stage-2 development N correct",

            "PASS":
                stage2_development_n
                ==
                EXPECTED_STAGE2_DEVELOPMENT_N,
        },

        {
            "Check":
                "Heldout N correct",

            "PASS":
                len(
                    heldout
                )
                ==
                EXPECTED_HELDOUT_N,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                len(
                    candidate_metrics_all
                )
                ==
                240,
        },

        {
            "Check":
                "All candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    ).all()
                ),
        },

        {
            "Check":
                "Exactly two stage winners",

            "PASS":
                len(
                    winners_all
                )
                ==
                2,
        },

        {
            "Check":
                "Heldout rows unique",

            "PASS":
                locked[
                    "_ROW_KEY"
                ].nunique()
                ==
                EXPECTED_HELDOUT_N,
        },

        {
            "Check":
                "Predictions complete",

            "PASS":
                not locked[
                    [
                        "q0_CALIBRATED",
                        "q2_CALIBRATED_CONDITIONAL",
                    ]
                ]
                .isna()
                .any()
                .any(),
        },

        {
            "Check":
                "Probabilities finite",

            "PASS":
                bool(
                    np.isfinite(
                        locked[
                            [
                                "P_NO_ANC",
                                "P_ANC_1_7",
                                "P_ANC_GE8",
                            ]
                        ]
                    ).all()
                ),
        },

        {
            "Check":
                "Probabilities sum to one",

            "PASS":
                bool(
                    np.allclose(
                        locked[
                            "P_SUM"
                        ],
                        1.0,
                        atol=1e-12,
                        rtol=0,
                    )
                ),
        },

        {
            "Check":
                "Division predictor excluded",

            "PASS":
                "division"
                not in
                GEO_PREDICTORS,
        },

        {
            "Check":
                "No heldout performance calculated",

            "PASS":
                True,
        },
    ]
)


if not hard_qc[
    "PASS"
].all():

    raise RuntimeError(
        "FINAL QC FAILED:\n"
        +
        hard_qc.loc[
            ~hard_qc[
                "PASS"
            ]
        ]
        .to_string(
            index=False
        )
    )


# ============================================================
# 24. SAVE FINAL DIVISION LOCK
# ============================================================

division_slug = safe_name(
    TARGET_DIVISION
)


prefix = (
    f"STEP23_GEOGRAPHIC_holdout_"
    f"{division_slug}"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)


FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)


FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)


FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)


FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)


FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)


FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)


winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)


calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)


development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)


locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)


unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)


hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        len(
            development
        ),

    "stage2_development_n":
        stage2_development_n,

    "heldout_n":
        len(
            heldout
        ),

    "scientific_hyperparameters_changed":
        False,

    "division_predictor_used":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
    )


# ============================================================
# 25. REPORT
# ============================================================

print(
    "\n"
    +
    "=" * 78
)


print(
    f"STEP 23 — "
    f"{TARGET_DIVISION.upper()} "
    f"IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)


display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)


display(
    calibration_all
)


print(
    "\nUNSEEN CATEGORY QC:"
)


display(
    unseen_all
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    f"\nRows             : "
    f"{len(locked)}"
)


print(
    f"Unique rows      : "
    f"{locked['_ROW_KEY'].nunique()}"
)


print(
    "Probability sum : "
    f"{np.allclose(locked['P_SUM'], 1.0, atol=1e-12, rtol=0)}"
)


print(
    "Heldout metrics : NOT CALCULATED"
)


print(
    f"\nLocked hash      : "
    f"{sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    +
    "=" * 78
)


print(
    f"STATUS: STEP 23 "
    f"{TARGET_DIVISION.upper()} "
    f"IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


STEP 23 — MYMENSINGH GEOGRAPHIC IECV
Held-out division : Mymensingh
Development N     : 4443
Stage-2 dev N     : 4109
Held-out N        : 620
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Mymensingh | Stage 1
Development N = 4443
Heldout scoring N = 620
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.237773
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.164114
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.208926
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.203982
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.207258
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.221818
  [7/10] HistGradientBoosting
    HistGradientBoosting: com

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [2]:
# ============================================================
# STEP 23 — MYMENSINGH QC BUG RECOVERY + FINAL LOCK
# NO MODEL REFIT / NO RETUNING / NO RECALIBRATION
# ============================================================

import datetime as dt
import json
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. SAFETY CHECK
# ============================================================

TARGET_DIVISION = "Mymensingh"

EXPECTED_DEVELOPMENT_N = 4443
EXPECTED_STAGE2_DEVELOPMENT_N = 4109
EXPECTED_HELDOUT_N = 620


required_objects = [
    "locked",
    "development",
    "heldout",
    "candidate_metrics_all",
    "winners_all",
    "calibration_all",
    "development_oof_all",
    "unseen_all",
    "DIRS",
    "sha256_file",
    "PRIMARY_FILE",
    "OUTER_SPLIT_FILE",
    "CANDIDATE_REGISTRY_FILE",
    "EXPECTED_PRIMARY_SHA256",
    "EXPECTED_OUTER_SHA256",
    "EXPECTED_CANDIDATE_SHA256",
]


missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]


if missing_objects:

    raise RuntimeError(
        "Required objects are missing from memory:\n"
        f"{missing_objects}\n\n"
        "This recovery cell assumes the kernel has NOT been restarted "
        "after the Mymensingh run."
    )


# ============================================================
# 2. CONFIRM WE ARE FINALIZING MYMENSINGH
# ============================================================

if len(development) != EXPECTED_DEVELOPMENT_N:

    raise RuntimeError(
        f"Development N mismatch: {len(development)}"
    )


if len(heldout) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        f"Heldout N mismatch: {len(heldout)}"
    )


stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin([1, 2])
    .sum()
)


if (
    stage2_development_n
    != EXPECTED_STAGE2_DEVELOPMENT_N
):

    raise RuntimeError(
        "Stage-2 development N mismatch: "
        f"{stage2_development_n}"
    )


if (
    not (
        locked[
            "Heldout_Division"
        ]
        .astype(str)
        ==
        TARGET_DIVISION
    ).all()
):

    raise RuntimeError(
        "Current locked object is not Mymensingh."
    )


# ============================================================
# 3. REQUIRED PREDICTION COLUMNS
# ============================================================

required_prediction_columns = [
    "Stage1_Raw_Score",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "Stage2_Raw_Score",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
]


missing_columns = [
    column
    for column in required_prediction_columns
    if column not in locked.columns
]


if missing_columns:

    raise RuntimeError(
        "Missing prediction columns:\n"
        f"{missing_columns}"
    )


# ============================================================
# 4. CORRECT FINITE-PROBABILITY CHECK
# ============================================================
#
# Previous buggy form:
#
# bool(np.isfinite(DataFrame).all())
#
# .all() returned a pandas Series.
#
# Correct form:
# convert to NumPy array, then reduce to ONE boolean.
# ============================================================

probability_matrix = (
    locked[
        [
            "P_NO_ANC",
            "P_ANC_1_7",
            "P_ANC_GE8",
        ]
    ]
    .to_numpy(
        dtype=float
    )
)


probabilities_finite = bool(
    np.isfinite(
        probability_matrix
    ).all()
)


probabilities_sum_to_one = bool(
    np.allclose(
        locked[
            "P_SUM"
        ]
        .to_numpy(
            dtype=float
        ),
        1.0,
        atol=1e-12,
        rtol=0.0,
    )
)


# ============================================================
# 5. FINAL HARD QC — CORRECTED
# ============================================================

hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                (
                    sha256_file(
                        PRIMARY_FILE
                    )
                    ==
                    EXPECTED_PRIMARY_SHA256
                ),
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                (
                    sha256_file(
                        OUTER_SPLIT_FILE
                    )
                    ==
                    EXPECTED_OUTER_SHA256
                ),
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                (
                    sha256_file(
                        CANDIDATE_REGISTRY_FILE
                    )
                    ==
                    EXPECTED_CANDIDATE_SHA256
                ),
        },

        {
            "Check":
                "Mymensingh development N = 4443",

            "PASS":
                (
                    len(development)
                    ==
                    4443
                ),
        },

        {
            "Check":
                "Stage-2 development N = 4109",

            "PASS":
                (
                    stage2_development_n
                    ==
                    4109
                ),
        },

        {
            "Check":
                "Mymensingh heldout N = 620",

            "PASS":
                (
                    len(heldout)
                    ==
                    620
                ),
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                (
                    len(
                        candidate_metrics_all
                    )
                    ==
                    240
                ),
        },

        {
            "Check":
                "All 240 candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Exactly two stage winners",

            "PASS":
                (
                    len(
                        winners_all
                    )
                    ==
                    2
                ),
        },

        {
            "Check":
                "Both stages represented",

            "PASS":
                (
                    set(
                        winners_all[
                            "Stage"
                        ]
                    )
                    ==
                    {
                        "Stage 1",
                        "Stage 2",
                    }
                ),
        },

        {
            "Check":
                "Stage 1 winner = ElasticNet EN_009",

            "PASS":
                bool(
                    (
                        winners_all.loc[
                            winners_all[
                                "Stage"
                            ]
                            ==
                            "Stage 1",
                            "Model",
                        ]
                        .iloc[0]
                        ==
                        "ElasticNet logistic regression"
                    )
                    and
                    (
                        str(
                            winners_all.loc[
                                winners_all[
                                    "Stage"
                                ]
                                ==
                                "Stage 1",
                                "Candidate_ID",
                            ]
                            .iloc[0]
                        )
                        ==
                        "EN_009"
                    )
                ),
        },

        {
            "Check":
                "Stage 2 winner = ElasticNet EN_001",

            "PASS":
                bool(
                    (
                        winners_all.loc[
                            winners_all[
                                "Stage"
                            ]
                            ==
                            "Stage 2",
                            "Model",
                        ]
                        .iloc[0]
                        ==
                        "ElasticNet logistic regression"
                    )
                    and
                    (
                        str(
                            winners_all.loc[
                                winners_all[
                                    "Stage"
                                ]
                                ==
                                "Stage 2",
                                "Candidate_ID",
                            ]
                            .iloc[0]
                        )
                        ==
                        "EN_001"
                    )
                ),
        },

        {
            "Check":
                "Exactly 620 heldout predictions",

            "PASS":
                (
                    len(locked)
                    ==
                    620
                ),
        },

        {
            "Check":
                "Heldout rows unique",

            "PASS":
                (
                    locked[
                        "_ROW_KEY"
                    ]
                    .nunique()
                    ==
                    620
                ),
        },

        {
            "Check":
                "Prediction merge complete",

            "PASS":
                bool(
                    ~locked[
                        required_prediction_columns
                    ]
                    .isna()
                    .any()
                    .any()
                ),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                probabilities_finite,
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                probabilities_sum_to_one,
        },

        {
            "Check":
                "Division predictor excluded",

            "PASS":
                (
                    "division"
                    not in
                    GEO_PREDICTORS
                ),
        },

        {
            "Check":
                "No model refitting in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No hyperparameter retuning in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No recalibration in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No threshold re-optimization in recovery",

            "PASS":
                True,
        },

        {
            "Check":
                "No heldout performance calculated",

            "PASS":
                True,
        },
    ]
)


if not bool(
    hard_qc[
        "PASS"
    ]
    .all()
):

    failed = hard_qc.loc[
        ~hard_qc[
            "PASS"
        ]
    ]

    raise RuntimeError(
        "MYMENSINGH FINAL QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 6. FINAL FILE PATHS
# ============================================================

prefix = (
    "STEP23_GEOGRAPHIC_holdout_Mymensingh"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)


FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)


FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)


FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)


FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)


FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)


FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


# ============================================================
# 7. SAVE FINAL MYMENSINGH LOCK
# ============================================================

candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)


winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)


calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)


development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)


locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)


unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)


hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 8. FINAL MANIFEST
# ============================================================

stage1_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 1"
    ]
    .iloc[0]
)


stage2_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 2"
    ]
    .iloc[0]
)


manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        4443,

    "stage2_development_n":
        4109,

    "heldout_n":
        620,

    "stage1_model":
        str(
            stage1_row[
                "Model"
            ]
        ),

    "stage1_candidate":
        str(
            stage1_row[
                "Candidate_ID"
            ]
        ),

    "stage2_model":
        str(
            stage2_row[
                "Model"
            ]
        ),

    "stage2_candidate":
        str(
            stage2_row[
                "Candidate_ID"
            ]
        ),

    "scientific_hyperparameters_changed":
        False,

    "model_refit_during_recovery":
        False,

    "retuning_during_recovery":
        False,

    "recalibration_during_recovery":
        False,

    "threshold_reoptimization_during_recovery":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),

    "recovery_note":
        (
            "Finalization repeated only because the prior "
            "QC expression used bool(np.isfinite(DataFrame).all()), "
            "which returned an ambiguous pandas Series. "
            "No scientific analysis was repeated."
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 9. REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — MYMENSINGH RECOVERY / FINAL LOCK"
)


print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)


display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)


display(
    calibration_all
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nLOCKED MYMENSINGH PREDICTIONS:"
)


print(
    f"Rows             : {len(locked)}"
)


print(
    f"Unique rows      : "
    f"{locked['_ROW_KEY'].nunique()}"
)


print(
    f"Probabilities finite : "
    f"{probabilities_finite}"
)


print(
    f"Probability sum      : "
    f"{probabilities_sum_to_one}"
)


print(
    "Model refitting      : NO"
)


print(
    "Retuning             : NO"
)


print(
    "Recalibration        : NO"
)


print(
    "Threshold redo       : NO"
)


print(
    "Heldout metrics      : NOT CALCULATED"
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master    : "
    f"{sha256_file(PRIMARY_FILE)}"
)


print(
    f"Candidate registry: "
    f"{sha256_file(CANDIDATE_REGISTRY_FILE)}"
)


print(
    f"Locked Mymensingh : "
    f"{sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    + "=" * 78
)


print(
    "STATUS: STEP 23 MYMENSINGH IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    '\nNEXT DIVISION AFTER AUDIT: '
    'TARGET_DIVISION = "Rajshahi"'
)


STEP 23 — MYMENSINGH RECOVERY / FINAL LOCK

JOINT WINNERS:


,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC
0,Mymensingh,Stage 1,ElasticNet logistic regression,EN_009,0.237773,0.776155
1,Mymensingh,Stage 2,ElasticNet logistic regression,EN_001,0.181624,0.727775



CALIBRATION + THRESHOLD:


,Heldout_Division,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Development_Threshold_Balanced_Accuracy,Development_Threshold_Sensitivity,Development_Threshold_Specificity
0,Mymensingh,Stage 1,ElasticNet logistic regression,EN_009,-0.195214,0.907530,5,0.062762,0.708604,0.813701,0.603507
1,Mymensingh,Stage 2,ElasticNet logistic regression,EN_001,2.864958,2.079305,11,0.045949,0.682464,0.733548,0.631379



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Mymensingh development N = 4443,True
4,Stage-2 development N = 4109,True
5,Mymensingh heldout N = 620,True
6,Exactly 240 candidate summaries,True
7,All 240 candidates successful,True
8,Exactly two stage winners,True
9,Both stages represented,True



LOCKED MYMENSINGH PREDICTIONS:
Rows             : 620
Unique rows      : 620
Probabilities finite : True
Probability sum      : True
Model refitting      : NO
Retuning             : NO
Recalibration        : NO
Threshold redo       : NO
Heldout metrics      : NOT CALCULATED

INTEGRITY:
Primary master    : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Candidate registry: f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Mymensingh : 9601dd4d19d372d9d80901de53b72235a6479c1d09e03f70e8e6a684d7c41ddb

STATUS: STEP 23 MYMENSINGH IECV PREDICTIONS LOCKED

NEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Rajshahi"


In [3]:
# ============================================================
# STEP 23 — RAJSHAHI IECV CONTINUATION
# Reuses restart-safe Step-23 functions already in memory
# ============================================================

import datetime as dt
import json
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. CHECK REQUIRED FUNCTIONS
# ============================================================

required_objects = [
    "master",
    "execute_stage",
    "sha256_file",
    "safe_name",
    "DIRS",
    "PRIMARY_FILE",
    "OUTER_SPLIT_FILE",
    "CANDIDATE_REGISTRY_FILE",
    "EXPECTED_PRIMARY_SHA256",
    "EXPECTED_OUTER_SHA256",
    "EXPECTED_CANDIDATE_SHA256",
    "GEO_PREDICTORS",
]

missing = [
    x for x in required_objects
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Required Step-23 objects/functions missing:\n"
        f"{missing}\n\n"
        "Kernel was likely restarted. "
        "Run the restart-safe Step-23 setup cell first."
    )


# ============================================================
# 2. TARGET
# ============================================================

TARGET_DIVISION = "Rajshahi"

EXPECTED_DEVELOPMENT_N = 4551
EXPECTED_STAGE2_DEVELOPMENT_N = 4174
EXPECTED_HELDOUT_N = 512

EXECUTION_THREADS = 2


# ============================================================
# 3. INTEGRITY
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError("Primary master hash mismatch.")

if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:
    raise RuntimeError("Step-13 split hash mismatch.")

if (
    sha256_file(CANDIDATE_REGISTRY_FILE)
    != EXPECTED_CANDIDATE_SHA256
):
    raise RuntimeError("Candidate registry hash mismatch.")


# ============================================================
# 4. REBUILD RAJSHAHI DEVELOPMENT / HOLDOUT
# ============================================================

development = (
    master.loc[
        master["division"].astype(str)
        != TARGET_DIVISION
    ]
    .copy()
    .reset_index(drop=True)
)

heldout = (
    master.loc[
        master["division"].astype(str)
        == TARGET_DIVISION
    ]
    .copy()
    .reset_index(drop=True)
)


stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin([1, 2])
    .sum()
)


if len(development) != EXPECTED_DEVELOPMENT_N:
    raise RuntimeError(
        f"Development N mismatch: {len(development)}"
    )

if len(heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        f"Heldout N mismatch: {len(heldout)}"
    )

if (
    stage2_development_n
    != EXPECTED_STAGE2_DEVELOPMENT_N
):
    raise RuntimeError(
        "Stage-2 development N mismatch: "
        f"{stage2_development_n}"
    )

if (
    set(development["__ROW_KEY"])
    &
    set(heldout["__ROW_KEY"])
):
    raise RuntimeError(
        "Development / heldout overlap."
    )


print("\n" + "=" * 78)
print("STEP 23 — RAJSHAHI GEOGRAPHIC IECV")
print("=" * 78)

print(f"Held-out division : {TARGET_DIVISION}")
print(f"Development N     : {len(development)}")
print(f"Stage-2 dev N     : {stage2_development_n}")
print(f"Held-out N        : {len(heldout)}")
print(f"CPU threads       : {EXECUTION_THREADS}")
print("Division predictor: EXCLUDED")


# ============================================================
# 5. RUN / LOAD STAGES
# ============================================================

stage1_result = execute_stage(
    "Stage 1"
)

stage2_result = execute_stage(
    "Stage 2"
)


# ============================================================
# 6. STAGE HELDOUT PREDICTIONS
# ============================================================

stage1_heldout = (
    stage1_result[
        "heldout_stage"
    ]
    .copy()
)

stage2_heldout = (
    stage2_result[
        "heldout_stage"
    ]
    .copy()
)


if len(stage1_heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        "Stage-1 heldout prediction N mismatch."
    )

if len(stage2_heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        "Stage-2 heldout prediction N mismatch."
    )


# ============================================================
# 7. CORRECT ROW-KEY MERGE
# ============================================================

heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY": "_ROW_KEY",
            "ANC_CLASS": "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked["Heldout_Division"] = (
    TARGET_DIVISION
)


# ============================================================
# 8. HIERARCHICAL FUSION
# ============================================================

q0 = locked[
    "q0_CALIBRATED"
].to_numpy(dtype=float)

q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(dtype=float)

t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(dtype=float)

t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(dtype=float)


locked["P_NO_ANC"] = q0

locked["P_ANC_1_7"] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)

locked["P_ANC_GE8"] = (
    (1.0 - q0)
    *
    q2
)

locked["P_SUM"] = (
    locked["P_NO_ANC"]
    +
    locked["P_ANC_1_7"]
    +
    locked["P_ANC_GE8"]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0 >= t1,
    0,
    np.where(
        q2 >= t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 9. COMBINE METADATA
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result["candidate_metrics"],
        stage2_result["candidate_metrics"],
    ],
    ignore_index=True,
)

winners_all = pd.concat(
    [
        stage1_result["winner"],
        stage2_result["winner"],
    ],
    ignore_index=True,
)

calibration_all = pd.concat(
    [
        stage1_result["calibration"],
        stage2_result["calibration"],
    ],
    ignore_index=True,
)

development_oof_all = pd.concat(
    [
        stage1_result["development_oof"],
        stage2_result["development_oof"],
    ],
    ignore_index=True,
)

unseen_all = pd.concat(
    [
        stage1_result["unseen"],
        stage2_result["unseen"],
    ],
    ignore_index=True,
)


# ============================================================
# 10. CORRECT HARD QC
# ============================================================

probability_matrix = (
    locked[
        [
            "P_NO_ANC",
            "P_ANC_1_7",
            "P_ANC_GE8",
        ]
    ]
    .to_numpy(dtype=float)
)


probabilities_finite = bool(
    np.isfinite(
        probability_matrix
    ).all()
)


probabilities_sum_to_one = bool(
    np.allclose(
        locked[
            "P_SUM"
        ]
        .to_numpy(dtype=float),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",
            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Step-13 split hash unchanged",
            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Candidate registry hash unchanged",
            "PASS":
                sha256_file(
                    CANDIDATE_REGISTRY_FILE
                )
                ==
                EXPECTED_CANDIDATE_SHA256,
        },

        {
            "Check":
                "Rajshahi development N = 4551",
            "PASS":
                len(development)
                ==
                4551,
        },

        {
            "Check":
                "Stage-2 development N = 4174",
            "PASS":
                stage2_development_n
                ==
                4174,
        },

        {
            "Check":
                "Rajshahi heldout N = 512",
            "PASS":
                len(heldout)
                ==
                512,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",
            "PASS":
                len(candidate_metrics_all)
                ==
                240,
        },

        {
            "Check":
                "All 240 candidates successful",
            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    ).all()
                ),
        },

        {
            "Check":
                "Exactly two stage winners",
            "PASS":
                len(winners_all)
                ==
                2,
        },

        {
            "Check":
                "Both stages represented",
            "PASS":
                set(
                    winners_all["Stage"]
                )
                ==
                {
                    "Stage 1",
                    "Stage 2",
                },
        },

        {
            "Check":
                "Exactly 512 heldout predictions",
            "PASS":
                len(locked)
                ==
                512,
        },

        {
            "Check":
                "Heldout rows unique",
            "PASS":
                locked[
                    "_ROW_KEY"
                ].nunique()
                ==
                512,
        },

        {
            "Check":
                "Prediction merge complete",
            "PASS":
                bool(
                    ~locked[
                        [
                            "q0_CALIBRATED",
                            "q2_CALIBRATED_CONDITIONAL",
                        ]
                    ]
                    .isna()
                    .any()
                    .any()
                ),
        },

        {
            "Check":
                "Fused probabilities finite",
            "PASS":
                probabilities_finite,
        },

        {
            "Check":
                "Fused probabilities sum to one",
            "PASS":
                probabilities_sum_to_one,
        },

        {
            "Check":
                "Division predictor excluded",
            "PASS":
                "division"
                not in
                GEO_PREDICTORS,
        },

        {
            "Check":
                "Scientific hyperparameters unchanged",
            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for selection",
            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for calibration",
            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for threshold optimization",
            "PASS":
                True,
        },

        {
            "Check":
                "No heldout performance calculated",
            "PASS":
                True,
        },
    ]
)


if not bool(
    hard_qc[
        "PASS"
    ].all()
):
    raise RuntimeError(
        "RAJSHAHI FINAL QC FAILED:\n"
        +
        hard_qc.loc[
            ~hard_qc["PASS"]
        ]
        .to_string(index=False)
    )


# ============================================================
# 11. SAVE FINAL RAJSHAHI LOCK
# ============================================================

prefix = (
    "STEP23_GEOGRAPHIC_holdout_Rajshahi"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)

FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)

FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)

FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)

FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)

FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)

FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)

FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)

winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)

calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)

development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)

locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)

unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)

hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 12. MANIFEST
# ============================================================

stage1_row = (
    winners_all.loc[
        winners_all["Stage"]
        ==
        "Stage 1"
    ]
    .iloc[0]
)

stage2_row = (
    winners_all.loc[
        winners_all["Stage"]
        ==
        "Stage 2"
    ]
    .iloc[0]
)


manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        len(development),

    "stage2_development_n":
        stage2_development_n,

    "heldout_n":
        len(heldout),

    "stage1_model":
        str(stage1_row["Model"]),

    "stage1_candidate":
        str(stage1_row["Candidate_ID"]),

    "stage2_model":
        str(stage2_row["Model"]),

    "stage2_candidate":
        str(stage2_row["Candidate_ID"]),

    "scientific_hyperparameters_changed":
        False,

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. REPORT
# ============================================================

print("\n" + "=" * 78)

print(
    "STEP 23 — RAJSHAHI IECV PREDICTIONS LOCKED"
)

print("=" * 78)


print("\nJOINT WINNERS:")

display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
        ]
    ]
)


print("\nCALIBRATION + THRESHOLD:")

display(
    calibration_all
)


print("\nUNSEEN CATEGORY QC:")

display(
    unseen_all
)


print("\nFINAL HARD QC:")

display(
    hard_qc
)


print("\nLOCKED RAJSHAHI PREDICTIONS:")

print(
    f"Rows             : {len(locked)}"
)

print(
    f"Unique rows      : "
    f"{locked['_ROW_KEY'].nunique()}"
)

print(
    f"Probabilities finite : "
    f"{probabilities_finite}"
)

print(
    f"Probability sum      : "
    f"{probabilities_sum_to_one}"
)

print(
    "Heldout metrics      : NOT CALCULATED"
)


print("\nINTEGRITY:")

print(
    f"Primary master    : "
    f"{sha256_file(PRIMARY_FILE)}"
)

print(
    f"Candidate registry: "
    f"{sha256_file(CANDIDATE_REGISTRY_FILE)}"
)

print(
    f"Locked Rajshahi   : "
    f"{sha256_file(FINAL_PREDICTION_FILE)}"
)


print("\n" + "=" * 78)

print(
    "STATUS: STEP 23 RAJSHAHI IECV PREDICTIONS LOCKED"
)

print("=" * 78)

print(
    '\nNEXT DIVISION AFTER AUDIT: '
    'TARGET_DIVISION = "Rangpur"'
)


STEP 23 — RAJSHAHI GEOGRAPHIC IECV
Held-out division : Rajshahi
Development N     : 4551
Stage-2 dev N     : 4174
Held-out N        : 512
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Rajshahi | Stage 1
Development N = 4551
Heldout scoring N = 512
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.242419
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.163674
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.219393
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.217587
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.223289
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.230729
  [7/10] HistGradientBoosting
    HistGradientBoosting: complete 

,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC
0,Rajshahi,Stage 1,ElasticNet logistic regression,EN_012,0.242419,0.781181
1,Rajshahi,Stage 2,ElasticNet logistic regression,EN_001,0.186933,0.734571



CALIBRATION + THRESHOLD:


,Heldout_Division,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Development_Threshold_Balanced_Accuracy,Development_Threshold_Sensitivity,Development_Threshold_Specificity
0,Rajshahi,Stage 1,ElasticNet logistic regression,EN_012,-0.249417,0.873961,5,0.081158,0.717976,0.778522,0.657431
1,Rajshahi,Stage 2,ElasticNet logistic regression,EN_001,2.924163,2.105741,11,0.045234,0.689358,0.748025,0.630691



UNSEEN CATEGORY QC:


,Heldout_Division,Stage,Model,Predictor,Heldout_Unseen_Category_Rows
0,Rajshahi,Stage 1,ElasticNet logistic regression,maternal_education,0
1,Rajshahi,Stage 1,ElasticNet logistic regression,wealth_quintile,0
2,Rajshahi,Stage 1,ElasticNet logistic regression,partner_education_primary_final,0
3,Rajshahi,Stage 1,ElasticNet logistic regression,pregnancy_intention,0
4,Rajshahi,Stage 1,ElasticNet logistic regression,healthcare_decision_primary_final,0
5,Rajshahi,Stage 1,ElasticNet logistic regression,residence,0
6,Rajshahi,Stage 1,ElasticNet logistic regression,weekly_media_exposure,0
7,Rajshahi,Stage 1,ElasticNet logistic regression,working_status,0
8,Rajshahi,Stage 1,ElasticNet logistic regression,religion_final,0
9,Rajshahi,Stage 1,ElasticNet logistic regression,household_head_sex,0



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Rajshahi development N = 4551,True
4,Stage-2 development N = 4174,True
5,Rajshahi heldout N = 512,True
6,Exactly 240 candidate summaries,True
7,All 240 candidates successful,True
8,Exactly two stage winners,True
9,Both stages represented,True



LOCKED RAJSHAHI PREDICTIONS:
Rows             : 512
Unique rows      : 512
Probabilities finite : True
Probability sum      : True
Heldout metrics      : NOT CALCULATED

INTEGRITY:
Primary master    : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Candidate registry: f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Rajshahi   : 4faac36296acf6e40a0724fed766538d82e4a4baca0d55c04eab15b23f6340e4

STATUS: STEP 23 RAJSHAHI IECV PREDICTIONS LOCKED

NEXT DIVISION AFTER AUDIT: TARGET_DIVISION = "Rangpur"


In [4]:
# ============================================================
# STEP 23 — RANGPUR IECV CONTINUATION + FINAL LOCK
# ============================================================

import datetime as dt
import json
import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 1. REQUIRED EXISTING OBJECTS
# ============================================================

required_objects = [
    "master",
    "execute_stage",
    "sha256_file",
    "DIRS",
    "PRIMARY_FILE",
    "OUTER_SPLIT_FILE",
    "CANDIDATE_REGISTRY_FILE",
    "EXPECTED_PRIMARY_SHA256",
    "EXPECTED_OUTER_SHA256",
    "EXPECTED_CANDIDATE_SHA256",
    "GEO_PREDICTORS",
]

missing = [
    x for x in required_objects
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Required Step-23 objects/functions missing:\n"
        f"{missing}\n\n"
        "Kernel was likely restarted. "
        "Run the restart-safe Step-23 setup cell first."
    )


# ============================================================
# 2. TARGET
# ============================================================

TARGET_DIVISION = "Rangpur"

EXPECTED_DEVELOPMENT_N = 4471
EXPECTED_STAGE2_DEVELOPMENT_N = 4117
EXPECTED_HELDOUT_N = 592

EXECUTION_THREADS = 2


# ============================================================
# 3. INTEGRITY
# ============================================================

if sha256_file(PRIMARY_FILE) != EXPECTED_PRIMARY_SHA256:
    raise RuntimeError("Primary master hash mismatch.")

if sha256_file(OUTER_SPLIT_FILE) != EXPECTED_OUTER_SHA256:
    raise RuntimeError("Step-13 split hash mismatch.")

if (
    sha256_file(CANDIDATE_REGISTRY_FILE)
    != EXPECTED_CANDIDATE_SHA256
):
    raise RuntimeError("Candidate registry hash mismatch.")


# ============================================================
# 4. DEVELOPMENT / HOLDOUT
# ============================================================

development = (
    master.loc[
        master["division"].astype(str)
        != TARGET_DIVISION
    ]
    .copy()
    .reset_index(drop=True)
)

heldout = (
    master.loc[
        master["division"].astype(str)
        == TARGET_DIVISION
    ]
    .copy()
    .reset_index(drop=True)
)

stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin([1, 2])
    .sum()
)


if len(development) != EXPECTED_DEVELOPMENT_N:
    raise RuntimeError(
        f"Development N mismatch: {len(development)}"
    )

if len(heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        f"Heldout N mismatch: {len(heldout)}"
    )

if (
    stage2_development_n
    != EXPECTED_STAGE2_DEVELOPMENT_N
):
    raise RuntimeError(
        "Stage-2 development N mismatch: "
        f"{stage2_development_n}"
    )

if (
    set(development["__ROW_KEY"])
    &
    set(heldout["__ROW_KEY"])
):
    raise RuntimeError(
        "Development / heldout overlap."
    )


print("\n" + "=" * 78)
print("STEP 23 — RANGPUR GEOGRAPHIC IECV")
print("=" * 78)

print(f"Held-out division : {TARGET_DIVISION}")
print(f"Development N     : {len(development)}")
print(f"Stage-2 dev N     : {stage2_development_n}")
print(f"Held-out N        : {len(heldout)}")
print(f"CPU threads       : {EXECUTION_THREADS}")
print("Division predictor: EXCLUDED")


# ============================================================
# 5. RUN / LOAD BOTH STAGES
# ============================================================

stage1_result = execute_stage("Stage 1")
stage2_result = execute_stage("Stage 2")


# ============================================================
# 6. HELDOUT STAGE PREDICTIONS
# ============================================================

stage1_heldout = (
    stage1_result["heldout_stage"].copy()
)

stage2_heldout = (
    stage2_result["heldout_stage"].copy()
)


if len(stage1_heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        "Stage-1 heldout prediction N mismatch."
    )

if len(stage2_heldout) != EXPECTED_HELDOUT_N:
    raise RuntimeError(
        "Stage-2 heldout prediction N mismatch."
    )


# ============================================================
# 7. CORRECTED ROW-KEY MERGE
# ============================================================

heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY": "_ROW_KEY",
            "ANC_CLASS": "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model": "Stage1_Model",
            "Candidate_ID": "Stage1_Candidate",
            "Heldout_Raw_Score": "Stage1_Raw_Score",
            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",
            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model": "Stage2_Model",
            "Candidate_ID": "Stage2_Candidate",
            "Heldout_Raw_Score": "Stage2_Raw_Score",
            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",
            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked["Heldout_Division"] = TARGET_DIVISION


# ============================================================
# 8. HIERARCHICAL FUSION
# ============================================================

q0 = locked[
    "q0_CALIBRATED"
].to_numpy(dtype=float)

q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(dtype=float)

t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(dtype=float)

t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(dtype=float)


locked["P_NO_ANC"] = q0

locked["P_ANC_1_7"] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)

locked["P_ANC_GE8"] = (
    (1.0 - q0)
    *
    q2
)

locked["P_SUM"] = (
    locked["P_NO_ANC"]
    +
    locked["P_ANC_1_7"]
    +
    locked["P_ANC_GE8"]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0 >= t1,
    0,
    np.where(
        q2 >= t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 9. COMBINE METADATA
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result["candidate_metrics"],
        stage2_result["candidate_metrics"],
    ],
    ignore_index=True,
)

winners_all = pd.concat(
    [
        stage1_result["winner"],
        stage2_result["winner"],
    ],
    ignore_index=True,
)

calibration_all = pd.concat(
    [
        stage1_result["calibration"],
        stage2_result["calibration"],
    ],
    ignore_index=True,
)

development_oof_all = pd.concat(
    [
        stage1_result["development_oof"],
        stage2_result["development_oof"],
    ],
    ignore_index=True,
)

unseen_all = pd.concat(
    [
        stage1_result["unseen"],
        stage2_result["unseen"],
    ],
    ignore_index=True,
)


# ============================================================
# 10. CORRECT HARD QC
# ============================================================

probability_matrix = (
    locked[
        [
            "P_NO_ANC",
            "P_ANC_1_7",
            "P_ANC_GE8",
        ]
    ]
    .to_numpy(dtype=float)
)


probabilities_finite = bool(
    np.isfinite(
        probability_matrix
    ).all()
)


probabilities_sum_to_one = bool(
    np.allclose(
        locked[
            "P_SUM"
        ]
        .to_numpy(dtype=float),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",
            "PASS":
                sha256_file(PRIMARY_FILE)
                ==
                EXPECTED_PRIMARY_SHA256,
        },

        {
            "Check":
                "Step-13 split hash unchanged",
            "PASS":
                sha256_file(OUTER_SPLIT_FILE)
                ==
                EXPECTED_OUTER_SHA256,
        },

        {
            "Check":
                "Candidate registry hash unchanged",
            "PASS":
                sha256_file(
                    CANDIDATE_REGISTRY_FILE
                )
                ==
                EXPECTED_CANDIDATE_SHA256,
        },

        {
            "Check":
                "Rangpur development N = 4471",
            "PASS":
                len(development)
                ==
                4471,
        },

        {
            "Check":
                "Stage-2 development N = 4117",
            "PASS":
                stage2_development_n
                ==
                4117,
        },

        {
            "Check":
                "Rangpur heldout N = 592",
            "PASS":
                len(heldout)
                ==
                592,
        },

        {
            "Check":
                "Exactly 240 candidate summaries",
            "PASS":
                len(candidate_metrics_all)
                ==
                240,
        },

        {
            "Check":
                "All 240 candidates successful",
            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    ).all()
                ),
        },

        {
            "Check":
                "Exactly two stage winners",
            "PASS":
                len(winners_all)
                ==
                2,
        },

        {
            "Check":
                "Both stages represented",
            "PASS":
                set(
                    winners_all["Stage"]
                )
                ==
                {
                    "Stage 1",
                    "Stage 2",
                },
        },

        {
            "Check":
                "Exactly 592 heldout predictions",
            "PASS":
                len(locked)
                ==
                592,
        },

        {
            "Check":
                "Heldout rows unique",
            "PASS":
                locked[
                    "_ROW_KEY"
                ].nunique()
                ==
                592,
        },

        {
            "Check":
                "Prediction merge complete",
            "PASS":
                bool(
                    ~locked[
                        [
                            "q0_CALIBRATED",
                            "q2_CALIBRATED_CONDITIONAL",
                        ]
                    ]
                    .isna()
                    .any()
                    .any()
                ),
        },

        {
            "Check":
                "Fused probabilities finite",
            "PASS":
                probabilities_finite,
        },

        {
            "Check":
                "Fused probabilities sum to one",
            "PASS":
                probabilities_sum_to_one,
        },

        {
            "Check":
                "Division predictor excluded",
            "PASS":
                "division"
                not in
                GEO_PREDICTORS,
        },

        {
            "Check":
                "Scientific hyperparameters unchanged",
            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for selection",
            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for calibration",
            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for threshold optimization",
            "PASS":
                True,
        },

        {
            "Check":
                "No heldout performance calculated",
            "PASS":
                True,
        },
    ]
)


if not bool(
    hard_qc["PASS"].all()
):
    raise RuntimeError(
        "RANGPUR FINAL QC FAILED:\n"
        +
        hard_qc.loc[
            ~hard_qc["PASS"]
        ]
        .to_string(index=False)
    )


# ============================================================
# 11. SAVE FINAL RANGPUR LOCK
# ============================================================

prefix = (
    "STEP23_GEOGRAPHIC_holdout_Rangpur"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)

FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)

FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)

FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)

FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)

FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)

FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)

FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)

winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)

calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)

development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)

locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)

unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)

hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 12. MANIFEST
# ============================================================

stage1_row = (
    winners_all.loc[
        winners_all["Stage"]
        ==
        "Stage 1"
    ]
    .iloc[0]
)

stage2_row = (
    winners_all.loc[
        winners_all["Stage"]
        ==
        "Stage 2"
    ]
    .iloc[0]
)


manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        len(development),

    "stage2_development_n":
        stage2_development_n,

    "heldout_n":
        len(heldout),

    "stage1_model":
        str(stage1_row["Model"]),

    "stage1_candidate":
        str(stage1_row["Candidate_ID"]),

    "stage2_model":
        str(stage2_row["Model"]),

    "stage2_candidate":
        str(stage2_row["Candidate_ID"]),

    "scientific_hyperparameters_changed":
        False,

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 13. REPORT
# ============================================================

print("\n" + "=" * 78)

print(
    "STEP 23 — RANGPUR IECV PREDICTIONS LOCKED"
)

print("=" * 78)


print("\nJOINT WINNERS:")

display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
        ]
    ]
)


print("\nCALIBRATION + THRESHOLD:")

display(
    calibration_all
)


print("\nUNSEEN CATEGORY QC:")

display(
    unseen_all
)


print("\nFINAL HARD QC:")

display(
    hard_qc
)


print("\nLOCKED RANGPUR PREDICTIONS:")

print(
    f"Rows             : {len(locked)}"
)

print(
    f"Unique rows      : "
    f"{locked['_ROW_KEY'].nunique()}"
)

print(
    f"Probabilities finite : "
    f"{probabilities_finite}"
)

print(
    f"Probability sum      : "
    f"{probabilities_sum_to_one}"
)

print(
    "Heldout metrics      : NOT CALCULATED"
)


print("\nINTEGRITY:")

print(
    f"Primary master    : "
    f"{sha256_file(PRIMARY_FILE)}"
)

print(
    f"Candidate registry: "
    f"{sha256_file(CANDIDATE_REGISTRY_FILE)}"
)

print(
    f"Locked Rangpur    : "
    f"{sha256_file(FINAL_PREDICTION_FILE)}"
)


print("\n" + "=" * 78)

print(
    "STATUS: STEP 23 RANGPUR IECV PREDICTIONS LOCKED"
)

print("=" * 78)

print(
    '\nFINAL DIVISION AFTER AUDIT: '
    'TARGET_DIVISION = "Sylhet"'
)


STEP 23 — RANGPUR GEOGRAPHIC IECV
Held-out division : Rangpur
Development N     : 4471
Stage-2 dev N     : 4117
Held-out N        : 592
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Rangpur | Stage 1
Development N = 4471
Heldout scoring N = 592
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.228720
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.148354
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.198777
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.194914
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.206677
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.213242
  [7/10] HistGradientBoosting
    HistGradientBoosting: complete | b

E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.paral

    AdaBoost: complete | best development-CV AP=0.164239
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.159071
  [7/10] HistGradientBoosting


E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
E:\Python\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.paral

    HistGradientBoosting: complete | best development-CV AP=0.122000
  [8/10] XGBoost
    XGBoost: complete | best development-CV AP=0.124999
  [9/10] LightGBM
    LightGBM: complete | best development-CV AP=0.105133
  [10/10] CatBoost
    CatBoost: complete | best development-CV AP=0.130304

  JOINT WINNER: ElasticNet logistic regression | EN_001
  Development-CV AP  = 0.184105
  Development-CV AUC = 0.732472
  Optimized threshold = 0.049560
  Stage 2 CHECKPOINT SAVED.

STEP 23 — RANGPUR IECV PREDICTIONS LOCKED

JOINT WINNERS:


,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC
0,Rangpur,Stage 1,ElasticNet logistic regression,EN_005,0.228720,0.787699
1,Rangpur,Stage 2,ElasticNet logistic regression,EN_001,0.184105,0.732472


... (3 outputs left out)

,Heldout_Division,Stage,Model,Predictor,Heldout_Unseen_Category_Rows
0,Rangpur,Stage 1,ElasticNet logistic regression,maternal_education,0
1,Rangpur,Stage 1,ElasticNet logistic regression,wealth_quintile,0
2,Rangpur,Stage 1,ElasticNet logistic regression,partner_education_primary_final,0
3,Rangpur,Stage 1,ElasticNet logistic regression,pregnancy_intention,0
4,Rangpur,Stage 1,ElasticNet logistic regression,healthcare_decision_primary_final,0
5,Rangpur,Stage 1,ElasticNet logistic regression,residence,0
6,Rangpur,Stage 1,ElasticNet logistic regression,weekly_media_exposure,0
7,Rangpur,Stage 1,ElasticNet logistic regression,working_status,0
8,Rangpur,Stage 1,ElasticNet logistic regression,religion_final,0
9,Rangpur,Stage 1,ElasticNet logistic regression,household_head_sex,0



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Rangpur development N = 4471,True
4,Stage-2 development N = 4117,True
5,Rangpur heldout N = 592,True
6,Exactly 240 candidate summaries,True
7,All 240 candidates successful,True
8,Exactly two stage winners,True
9,Both stages represented,True



LOCKED RANGPUR PREDICTIONS:
Rows             : 592
Unique rows      : 592
Probabilities finite : True
Probability sum      : True
Heldout metrics      : NOT CALCULATED

INTEGRITY:
Primary master    : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Candidate registry: f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Rangpur    : bbda9b981c8e69f0be5701042681dfe6c4c2056fdcae68611b1da19357cbdbfb

STATUS: STEP 23 RANGPUR IECV PREDICTIONS LOCKED

FINAL DIVISION AFTER AUDIT: TARGET_DIVISION = "Sylhet"


In [1]:
# ============================================================
# STEP 23 — SYLHET GEOGRAPHIC IECV
# RESTART-SAFE STANDALONE VERSION
# FAMILY + STAGE CHECKPOINTING
# LAPTOP OPTIMIZED: 2 CPU THREADS
# CORRECTED ROW-KEY MERGE + CORRECTED FINAL QC
# ============================================================
#
# HELD-OUT DIVISION:
#   SYLHET
#
# EXPECTED:
#   Development N     = 4475
#   Stage-2 dev N     = 4151
#   Held-out N        = 588
#
# SCIENTIFIC PROTOCOL:
#   - Division completely held out
#   - Division predictor excluded
#   - Remaining 14 predictors used
#   - Frozen Step-13 5-fold PSU-exclusive folds reused
#   - All 120 candidates evaluated independently per stage
#   - Survey-weighted model fitting
#   - Weighted AP primary selection metric
#   - Weighted ROC-AUC tie-breaker
#   - Deterministic candidate-rank final tie-breaker
#   - Winner calibrated using development OOF predictions only
#   - Threshold optimized using development OOF only
#   - Final selected model refit on full 7-division development data
#   - Held-out Sylhet only scored after all development decisions frozen
#   - NO held-out performance calculated in this cell
#
# ============================================================


# ============================================================
# 0. TARGET DIVISION
# ============================================================

TARGET_DIVISION = "Sylhet"


# ============================================================
# 1. EXECUTION THREAD CONTROL
# ============================================================

import os

EXECUTION_THREADS = 2

os.environ["OMP_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["MKL_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(EXECUTION_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(EXECUTION_THREADS)


# ============================================================
# 2. IMPORTS
# ============================================================

from pathlib import Path

import ast
import datetime as dt
import hashlib
import json
import warnings

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import expit

from sklearn.exceptions import ConvergenceWarning

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

from IPython.display import display


# ============================================================
# 3. PROJECT DIRECTORIES
# ============================================================

ROOT = Path(
    r"E:\BDHS_ANC_Q1_FINAL_REBUILD_2026"
)


DIRS = {
    "root":
        ROOT,

    "processed":
        ROOT / "01_processed_data",

    "qc":
        ROOT / "02_quality_control",

    "protocol":
        ROOT / "03_protocol",

    "splits":
        ROOT / "04_cv_splits",

    "models":
        ROOT / "05_models",

    "predictions":
        ROOT / "06_predictions",

    "calibration":
        ROOT / "07_calibration",

    "bootstrap":
        ROOT / "08_bootstrap",

    "geographic":
        ROOT / "09_geographic_validation",

    "explainability":
        ROOT / "10_explainability",

    "sensitivity":
        ROOT / "11_sensitivity",

    "subgroup":
        ROOT / "12_subgroup_analysis",

    "tables":
        ROOT / "13_tables",

    "figures":
        ROOT / "14_figures",

    "logs":
        ROOT / "15_logs",

    "reporting":
        ROOT / "16_reporting",

    "archive":
        ROOT / "17_reproducibility_archive",
}


for path in DIRS.values():

    if isinstance(path, Path):

        path.mkdir(
            parents=True,
            exist_ok=True,
        )


# ============================================================
# 4. CONSTANTS
# ============================================================

SEED = 2026


EXECUTION_SCHEMA_VERSION = (
    "STEP23_GEOGRAPHIC_LAPTOP_v2"
)


METRIC_TIE_TOLERANCE = 1e-12


CALIBRATION_PROBABILITY_CLIP = 1e-6


THRESHOLD_PROBABILITY_CLIP = 1e-12


EXPECTED_PRIMARY_SHA256 = (
    "0cf7e22a0f8d1f2c4554e96ac56ef6d"
    "ec6170f9067596519b0dcd1978132283a"
)


EXPECTED_OUTER_SHA256 = (
    "258dac5ff253a96895d89ddb80ac892b7"
    "6ef965d043c04cf94972994a82ab372"
)


EXPECTED_CANDIDATE_SHA256 = (
    "f95dc93431eced694114fe8ed071a2878"
    "e2572fac53aa3b9fd9c6cb8df87c697"
)


EXPECTED_DEVELOPMENT_N = 4475


EXPECTED_STAGE2_DEVELOPMENT_N = 4151


EXPECTED_HELDOUT_N = 588


# ============================================================
# 5. INPUT FILES
# ============================================================

PRIMARY_FILE = (
    DIRS["processed"]
    / "FINAL_PRIMARY_model_master_LITERATURE_LOCKED.csv"
)


OUTER_SPLIT_FILE = (
    DIRS["splits"]
    / "STEP13_FROZEN_outer_5fold_row_assignments.csv"
)


CANDIDATE_REGISTRY_FILE = (
    DIRS["protocol"]
    / "STEP16_FINAL_hyperparameter_candidate_registry.csv"
)


STEP22_PROTOCOL_FILE = (
    DIRS["protocol"]
    / "STEP22_GEOGRAPHIC_IECV_protocol.json"
)


STEP22_MANIFEST_FILE = (
    DIRS["logs"]
    / "STEP22_GEOGRAPHIC_IECV_protocol_manifest.json"
)


# ============================================================
# 6. BASIC UTILITIES
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        for block in iter(
            lambda:
                handle.read(
                    1024 * 1024
                ),
            b"",
        ):

            digest.update(
                block
            )

    return digest.hexdigest()


def sha256_text(text):

    return hashlib.sha256(
        str(text).encode(
            "utf-8"
        )
    ).hexdigest()


def read_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        return json.load(
            handle
        )


def normalize_name(value):

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def safe_name(value):

    return (
        str(value)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(">=", "GE")
    )


def find_column(
    dataframe,
    aliases,
):

    lookup = {
        normalize_name(column):
            column

        for column
        in dataframe.columns
    }


    for alias in aliases:

        key = normalize_name(
            alias
        )

        if key in lookup:

            return lookup[
                key
            ]


    raise RuntimeError(
        "Required column could not be identified.\n"
        f"Aliases: {aliases}\n"
        f"Available columns:\n"
        f"{list(dataframe.columns)}"
    )


def canonical_row_key(series):

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )


    if (
        numeric.notna().all()
        and
        np.allclose(
            numeric.to_numpy(
                dtype=float
            ),
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ),
        )
    ):

        return pd.Series(
            np.round(
                numeric.to_numpy(
                    dtype=float
                )
            ).astype(
                np.int64
            ),
            index=series.index,
        )


    return (
        series
        .astype(str)
        .str.strip()
    )


def parse_parameter_dictionary(value):

    if isinstance(
        value,
        dict,
    ):

        return value.copy()


    if pd.isna(
        value
    ):

        return {}


    text = str(
        value
    ).strip()


    if text == "":

        return {}


    try:

        parsed = json.loads(
            text
        )


    except json.JSONDecodeError:

        parsed = ast.literal_eval(
            text
        )


    if not isinstance(
        parsed,
        dict,
    ):

        raise RuntimeError(
            "Candidate parameter field did not decode "
            "to a dictionary."
        )


    return parsed


def row_key_hash(
    dataframe,
):

    values = (
        dataframe[
            "__ROW_KEY"
        ]
        .astype(str)
        .tolist()
    )


    return sha256_text(
        "|".join(
            values
        )
    )


# ============================================================
# 7. REQUIRED FILE CHECK
# ============================================================

required_files = [
    PRIMARY_FILE,
    OUTER_SPLIT_FILE,
    CANDIDATE_REGISTRY_FILE,
    STEP22_PROTOCOL_FILE,
    STEP22_MANIFEST_FILE,
]


for path in required_files:

    if not path.exists():

        raise FileNotFoundError(
            f"Required file missing:\n"
            f"{path}"
        )


# ============================================================
# 8. IMMUTABILITY / HASH CHECK
# ============================================================

if (
    sha256_file(
        PRIMARY_FILE
    )
    !=
    EXPECTED_PRIMARY_SHA256
):

    raise RuntimeError(
        "Primary-master hash mismatch."
    )


if (
    sha256_file(
        OUTER_SPLIT_FILE
    )
    !=
    EXPECTED_OUTER_SHA256
):

    raise RuntimeError(
        "Step-13 outer split hash mismatch."
    )


if (
    sha256_file(
        CANDIDATE_REGISTRY_FILE
    )
    !=
    EXPECTED_CANDIDATE_SHA256
):

    raise RuntimeError(
        "Step-16 candidate registry hash mismatch."
    )


step22_manifest = read_json(
    STEP22_MANIFEST_FILE
)


if (
    step22_manifest.get(
        "status"
    )
    !=
    "GEOGRAPHIC_IECV_PROTOCOL_FROZEN"
):

    raise RuntimeError(
        "Step 22 geographic IECV protocol is not frozen."
    )


STEP22_PROTOCOL_SHA256 = (
    sha256_file(
        STEP22_PROTOCOL_FILE
    )
)


# ============================================================
# 9. LOAD CORE DATA
# ============================================================

master = pd.read_csv(
    PRIMARY_FILE
)


outer_assignment = pd.read_csv(
    OUTER_SPLIT_FILE
)


candidate_registry = pd.read_csv(
    CANDIDATE_REGISTRY_FILE
)


if master.shape != (
    5063,
    23,
):

    raise RuntimeError(
        f"Unexpected master shape: "
        f"{master.shape}"
    )


if len(
    candidate_registry
) != 120:

    raise RuntimeError(
        "Candidate registry must contain exactly 120 candidates."
    )


# ============================================================
# 10. CANDIDATE REGISTRY QC
# ============================================================

required_candidate_columns = {
    "Candidate_ID",
    "Model",
    "Preprocessing",
    "Deterministic_Tie_Rank",
    "Tuned_Params_JSON",
    "Fixed_Params_JSON",
}


missing_candidate_columns = (
    required_candidate_columns
    -
    set(
        candidate_registry.columns
    )
)


if missing_candidate_columns:

    raise RuntimeError(
        "Candidate registry missing columns:\n"
        f"{sorted(missing_candidate_columns)}"
    )


candidate_counts = (
    candidate_registry
    .groupby(
        "Model"
    )
    .size()
)


if len(
    candidate_counts
) != 10:

    raise RuntimeError(
        "Expected exactly 10 model families."
    )


if (
    candidate_counts
    !=
    12
).any():

    raise RuntimeError(
        "Each model family must contain exactly 12 candidates."
    )


MODEL_ORDER = (
    candidate_registry
    .groupby(
        "Model",
        as_index=False,
    )[
        "Deterministic_Tie_Rank"
    ]
    .min()
    .sort_values(
        "Deterministic_Tie_Rank"
    )[
        "Model"
    ]
    .tolist()
)


# ============================================================
# 11. ROW-ID + FROZEN FOLD RECONSTRUCTION
# ============================================================

MASTER_ROW_COL = find_column(
    master,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_ROW_COL = find_column(
    outer_assignment,
    [
        "_ROW_ID",
        "ROW_ID",
        "row_id",
    ],
)


OUTER_FOLD_COL = find_column(
    outer_assignment,
    [
        "OUTER_FOLD",
        "outer_fold",
        "OUTER_FOLD_ID",
        "outer_fold_id",
    ],
)


master = master.copy()


outer_assignment = (
    outer_assignment.copy()
)


master[
    "__ROW_KEY"
] = canonical_row_key(
    master[
        MASTER_ROW_COL
    ]
)


outer_assignment[
    "__ROW_KEY"
] = canonical_row_key(
    outer_assignment[
        OUTER_ROW_COL
    ]
)


outer_assignment[
    OUTER_FOLD_COL
] = pd.to_numeric(
    outer_assignment[
        OUTER_FOLD_COL
    ],
    errors="raise",
).astype(int)


if master[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate master row key."
    )


if outer_assignment[
    "__ROW_KEY"
].duplicated().any():

    raise RuntimeError(
        "Duplicate Step-13 fold row key."
    )


if set(
    master[
        "__ROW_KEY"
    ]
) != set(
    outer_assignment[
        "__ROW_KEY"
    ]
):

    raise RuntimeError(
        "Frozen Step-13 split does not exactly cover primary master."
    )


fold_map = (
    outer_assignment
    .set_index(
        "__ROW_KEY"
    )[
        OUTER_FOLD_COL
    ]
)


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__ROW_KEY"
].map(
    fold_map
)


if master[
    "__DEVELOPMENT_CV_FOLD"
].isna().any():

    raise RuntimeError(
        "Missing development-CV fold assignment."
    )


master[
    "__DEVELOPMENT_CV_FOLD"
] = master[
    "__DEVELOPMENT_CV_FOLD"
].astype(int)


if sorted(
    master[
        "__DEVELOPMENT_CV_FOLD"
    ]
    .unique()
    .tolist()
) != [
    0,
    1,
    2,
    3,
    4,
]:

    raise RuntimeError(
        "Expected frozen development folds 0-4."
    )


# ============================================================
# 12. GEOGRAPHIC 14-PREDICTOR SET
# ============================================================

GEO_NUMERIC_PREDICTORS = [
    "maternal_age_at_index_birth_years",
    "prior_live_births",
    "household_size_members",
]


GEO_ORDERED_CATEGORICAL_PREDICTORS = [
    "maternal_education",
    "wealth_quintile",
]


GEO_NOMINAL_CATEGORICAL_PREDICTORS = [
    "partner_education_primary_final",
    "pregnancy_intention",
    "healthcare_decision_primary_final",
]


GEO_BINARY_CATEGORICAL_PREDICTORS = [
    "residence",
    "weekly_media_exposure",
    "working_status",
    "religion_final",
    "household_head_sex",
    "distance_barrier",
]


GEO_CATEGORICAL_PREDICTORS = (
    GEO_ORDERED_CATEGORICAL_PREDICTORS
    +
    GEO_NOMINAL_CATEGORICAL_PREDICTORS
    +
    GEO_BINARY_CATEGORICAL_PREDICTORS
)


GEO_PREDICTORS = (
    GEO_NUMERIC_PREDICTORS
    +
    GEO_CATEGORICAL_PREDICTORS
)


if len(
    GEO_PREDICTORS
) != 14:

    raise RuntimeError(
        "Geographic predictor count is not 14."
    )


if "division" in GEO_PREDICTORS:

    raise RuntimeError(
        "Division predictor leaked into IECV predictor set."
    )


# ============================================================
# 13. OUTCOME / WEIGHT QC
# ============================================================

master[
    "ANC_CLASS"
] = pd.to_numeric(
    master[
        "ANC_CLASS"
    ],
    errors="raise",
).astype(int)


if set(
    master[
        "ANC_CLASS"
    ].unique()
) != {
    0,
    1,
    2,
}:

    raise RuntimeError(
        "ANC_CLASS must contain exactly {0,1,2}."
    )


survey_weight = (
    master[
        "SURVEY_WEIGHT"
    ]
    .to_numpy(
        dtype=float
    )
)


if not np.isfinite(
    survey_weight
).all():

    raise RuntimeError(
        "Non-finite survey weight detected."
    )


if (
    survey_weight
    <=
    0
).any():

    raise RuntimeError(
        "Survey weights must all be positive."
    )


# ============================================================
# 14. SYLHET DEVELOPMENT / HOLDOUT
# ============================================================

development = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        !=
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


heldout = (
    master.loc[
        master[
            "division"
        ]
        .astype(str)
        ==
        TARGET_DIVISION
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


stage2_development_n = int(
    development[
        "ANC_CLASS"
    ]
    .isin(
        [
            1,
            2,
        ]
    )
    .sum()
)


if len(
    development
) != EXPECTED_DEVELOPMENT_N:

    raise RuntimeError(
        f"Expected Sylhet development N="
        f"{EXPECTED_DEVELOPMENT_N}; "
        f"observed {len(development)}."
    )


if (
    stage2_development_n
    !=
    EXPECTED_STAGE2_DEVELOPMENT_N
):

    raise RuntimeError(
        f"Expected Sylhet Stage-2 development N="
        f"{EXPECTED_STAGE2_DEVELOPMENT_N}; "
        f"observed {stage2_development_n}."
    )


if len(
    heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        f"Expected Sylhet heldout N="
        f"{EXPECTED_HELDOUT_N}; "
        f"observed {len(heldout)}."
    )


if (
    set(
        development[
            "__ROW_KEY"
        ]
    )
    &
    set(
        heldout[
            "__ROW_KEY"
        ]
    )
):

    raise RuntimeError(
        "Development / heldout row overlap detected."
    )


if TARGET_DIVISION in set(
    development[
        "division"
    ].astype(str)
):

    raise RuntimeError(
        "Sylhet leaked into development dataset."
    )


# ============================================================
# 15. TRAINING WEIGHT
# ============================================================

def normalized_training_weight(
    dataframe,
):

    weight = (
        dataframe[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    normalized = (
        weight
        /
        np.mean(
            weight
        )
    )


    if not np.isclose(
        np.mean(
            normalized
        ),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Training-weight normalization failed."
        )


    return normalized


# ============================================================
# 16. PREPARE STAGE DATA
# ============================================================

def prepare_stage(
    dataframe,
    stage_name,
):

    data = dataframe.copy()


    if stage_name == "Stage 1":

        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            0
        ).astype(int)


    elif stage_name == "Stage 2":

        data = (
            data.loc[
                data[
                    "ANC_CLASS"
                ]
                .isin(
                    [
                        1,
                        2,
                    ]
                )
            ]
            .copy()
        )


        data[
            "__TARGET"
        ] = (
            data[
                "ANC_CLASS"
            ]
            ==
            2
        ).astype(int)


    else:

        raise ValueError(
            f"Unknown stage: "
            f"{stage_name}"
        )


    return data.reset_index(
        drop=True
    )


# ============================================================
# 17. PREPROCESSING
# ============================================================

def build_representations(
    training_data,
    scoring_data,
):

    training_numeric = (
        training_data[
            GEO_NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    scoring_numeric = (
        scoring_data[
            GEO_NUMERIC_PREDICTORS
        ]
        .astype(float)
        .to_numpy()
    )


    training_categorical = (
        training_data[
            GEO_CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    scoring_categorical = (
        scoring_data[
            GEO_CATEGORICAL_PREDICTORS
        ]
        .astype(str)
    )


    # --------------------------------------------------------
    # OHE
    # --------------------------------------------------------

    encoder = OneHotEncoder(
        drop=None,
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float64,
    )


    training_ohe = (
        encoder.fit_transform(
            training_categorical
        )
    )


    scoring_ohe = (
        encoder.transform(
            scoring_categorical
        )
    )


    training_unscaled = np.hstack(
        [
            training_numeric,
            training_ohe,
        ]
    )


    scoring_unscaled = np.hstack(
        [
            scoring_numeric,
            scoring_ohe,
        ]
    )


    # --------------------------------------------------------
    # Scaled numeric + OHE
    # --------------------------------------------------------

    scaler = StandardScaler()


    training_scaled_numeric = (
        scaler.fit_transform(
            training_numeric
        )
    )


    scoring_scaled_numeric = (
        scaler.transform(
            scoring_numeric
        )
    )


    training_scaled = np.hstack(
        [
            training_scaled_numeric,
            training_ohe,
        ]
    )


    scoring_scaled = np.hstack(
        [
            scoring_scaled_numeric,
            scoring_ohe,
        ]
    )


    # --------------------------------------------------------
    # Native categorical
    # --------------------------------------------------------

    training_native = (
        training_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    scoring_native = (
        scoring_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    unseen_counts = {}


    for column in GEO_NUMERIC_PREDICTORS:

        training_native[
            column
        ] = pd.to_numeric(
            training_native[
                column
            ],
            errors="raise",
        ).astype(float)


        scoring_native[
            column
        ] = pd.to_numeric(
            scoring_native[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in GEO_CATEGORICAL_PREDICTORS:

        train_values = (
            training_native[
                column
            ]
            .astype(str)
        )


        scoring_values = (
            scoring_native[
                column
            ]
            .astype(str)
        )


        categories = list(
            pd.unique(
                train_values
            )
        )


        unseen_counts[
            column
        ] = int(
            (
                ~scoring_values.isin(
                    categories
                )
            )
            .sum()
        )


        training_native[
            column
        ] = pd.Categorical(
            train_values,
            categories=categories,
            ordered=False,
        )


        scoring_native[
            column
        ] = pd.Categorical(
            scoring_values,
            categories=categories,
            ordered=False,
        )


    # --------------------------------------------------------
    # CatBoost-native dataframe
    # --------------------------------------------------------

    training_catboost = (
        training_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    scoring_catboost = (
        scoring_data[
            GEO_PREDICTORS
        ]
        .copy()
    )


    for column in GEO_NUMERIC_PREDICTORS:

        training_catboost[
            column
        ] = pd.to_numeric(
            training_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


        scoring_catboost[
            column
        ] = pd.to_numeric(
            scoring_catboost[
                column
            ],
            errors="raise",
        ).astype(float)


    for column in GEO_CATEGORICAL_PREDICTORS:

        training_catboost[
            column
        ] = (
            training_catboost[
                column
            ]
            .astype(str)
        )


        scoring_catboost[
            column
        ] = (
            scoring_catboost[
                column
            ]
            .astype(str)
        )


    return {
        "Scaled OHE":
            (
                training_scaled,
                scoring_scaled,
            ),

        "Unscaled OHE":
            (
                training_unscaled,
                scoring_unscaled,
            ),

        "Native categorical":
            (
                training_native,
                scoring_native,
            ),

        "CatBoost native":
            (
                training_catboost,
                scoring_catboost,
            ),

        "__UNSEEN_COUNTS":
            unseen_counts,
    }


# ============================================================
# 18. MODEL INSTANTIATION
# ============================================================

def instantiate_candidate(
    candidate_row,
):

    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    fixed_params = (
        parse_parameter_dictionary(
            candidate_row[
                "Fixed_Params_JSON"
            ]
        )
    )


    tuned_params = (
        parse_parameter_dictionary(
            candidate_row[
                "Tuned_Params_JSON"
            ]
        )
    )


    params = {
        **fixed_params,
        **tuned_params,
    }


    # --------------------------------------------------------
    # EXECUTION-ONLY THREAD LIMIT
    # --------------------------------------------------------

    if model_name in [
        "Random Forest",
        "ExtraTrees",
        "XGBoost",
        "LightGBM",
    ]:

        params[
            "n_jobs"
        ] = EXECUTION_THREADS


    if model_name == "CatBoost":

        params[
            "thread_count"
        ] = EXECUTION_THREADS


    # --------------------------------------------------------
    # MODELS
    # --------------------------------------------------------

    if (
        model_name
        ==
        "ElasticNet logistic regression"
    ):

        return LogisticRegression(
            **params
        )


    if model_name == "RBF SVM":

        probability_value = params.pop(
            "probability",
            False,
        )


        if probability_value not in [
            False,
            None,
        ]:

            raise RuntimeError(
                "Frozen SVM unexpectedly requests probability=True."
            )


        return SVC(
            **params
        )


    if model_name == "Random Forest":

        return RandomForestClassifier(
            **params
        )


    if model_name == "ExtraTrees":

        return ExtraTreesClassifier(
            **params
        )


    if model_name == "AdaBoost":

        depth_aliases = {
            "estimatormaxdepth",
            "baseestimatormaxdepth",
            "stumpmaxdepth",
            "decisiontreemaxdepth",
            "maxdepth",
        }


        depth_keys = [
            key

            for key
            in list(
                params.keys()
            )

            if normalize_name(
                key
            )
            in depth_aliases
        ]


        if len(
            depth_keys
        ) != 1:

            raise RuntimeError(
                "Could not uniquely identify "
                "AdaBoost tree-depth parameter.\n"
                f"{params}"
            )


        depth = int(
            params.pop(
                depth_keys[
                    0
                ]
            )
        )


        if (
            "estimator"
            in params
            and
            not hasattr(
                params[
                    "estimator"
                ],
                "fit",
            )
        ):

            params.pop(
                "estimator"
            )


        tree = DecisionTreeClassifier(
            max_depth=depth,
            random_state=SEED,
        )


        return AdaBoostClassifier(
            estimator=tree,
            **params,
        )


    if model_name == "GradientBoosting":

        return GradientBoostingClassifier(
            **params
        )


    if model_name == "HistGradientBoosting":

        return HistGradientBoostingClassifier(
            **params
        )


    if model_name == "XGBoost":

        return XGBClassifier(
            **params
        )


    if model_name == "LightGBM":

        return LGBMClassifier(
            **params
        )


    if model_name == "CatBoost":

        return CatBoostClassifier(
            **params
        )


    raise RuntimeError(
        f"Unknown model family: "
        f"{model_name}"
    )


# ============================================================
# 19. FIT MODEL
# ============================================================

def fit_candidate(
    estimator,
    model_name,
    X_train,
    y_train,
    training_weight,
):

    with warnings.catch_warnings():

        warnings.simplefilter(
            "error",
            ConvergenceWarning,
        )


        if model_name == "CatBoost":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
                cat_features=GEO_CATEGORICAL_PREDICTORS,
                verbose=False,
            )


        elif model_name == "LightGBM":

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
                categorical_feature=GEO_CATEGORICAL_PREDICTORS,
            )


        else:

            estimator.fit(
                X_train,
                y_train,
                sample_weight=training_weight,
            )


    return estimator


# ============================================================
# 20. RAW POSITIVE SCORE
# ============================================================

def positive_score(
    estimator,
    model_name,
    X,
):

    classes = np.asarray(
        estimator.classes_
    )


    if set(
        classes.tolist()
    ) != {
        0,
        1,
    }:

        raise RuntimeError(
            f"{model_name}: unexpected classes "
            f"{classes}"
        )


    if model_name == "RBF SVM":

        score = np.asarray(
            estimator.decision_function(
                X
            ),
            dtype=float,
        ).reshape(-1)


    else:

        probability = np.asarray(
            estimator.predict_proba(
                X
            ),
            dtype=float,
        )


        positive_index = int(
            np.where(
                classes
                ==
                1
            )[0][0]
        )


        score = probability[
            :,
            positive_index
        ]


    if not np.isfinite(
        score
    ).all():

        raise RuntimeError(
            f"{model_name}: non-finite raw score."
        )


    return score


# ============================================================
# 21. REPRESENTATION LOOKUP
# ============================================================

def get_representation(
    representations,
    model_name,
    preprocessing_name,
):

    if model_name == "CatBoost":

        return representations[
            "CatBoost native"
        ]


    if (
        preprocessing_name
        not in representations
    ):

        raise RuntimeError(
            f"Unknown preprocessing: "
            f"{preprocessing_name}"
        )


    return representations[
        preprocessing_name
    ]


# ============================================================
# 22. BUILD 5-FOLD DEVELOPMENT CACHE
# ============================================================

def build_cv_cache(
    stage_development,
):

    cache = {}


    folds = sorted(
        stage_development[
            "__DEVELOPMENT_CV_FOLD"
        ]
        .unique()
        .tolist()
    )


    if folds != [
        0,
        1,
        2,
        3,
        4,
    ]:

        raise RuntimeError(
            f"Expected folds 0-4; "
            f"observed {folds}"
        )


    for fold in folds:

        validation_mask = (
            stage_development[
                "__DEVELOPMENT_CV_FOLD"
            ]
            ==
            fold
        )


        train_positions = np.where(
            ~validation_mask.to_numpy()
        )[0]


        validation_positions = np.where(
            validation_mask.to_numpy()
        )[0]


        train = (
            stage_development.iloc[
                train_positions
            ]
            .copy()
        )


        validation = (
            stage_development.iloc[
                validation_positions
            ]
            .copy()
        )


        y_train = (
            train[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        y_validation = (
            validation[
                "__TARGET"
            ]
            .to_numpy(
                dtype=int
            )
        )


        if set(
            np.unique(
                y_train
            )
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"Fold {fold}: training lacks both classes."
            )


        if set(
            np.unique(
                y_validation
            )
        ) != {
            0,
            1,
        }:

            raise RuntimeError(
                f"Fold {fold}: validation lacks both classes."
            )


        cache[
            fold
        ] = {
            "train_positions":
                train_positions,

            "validation_positions":
                validation_positions,

            "y_train":
                y_train,

            "y_validation":
                y_validation,

            "training_weight":
                normalized_training_weight(
                    train
                ),

            "validation_weight":
                validation[
                    "SURVEY_WEIGHT"
                ]
                .to_numpy(
                    dtype=float
                ),

            "representations":
                build_representations(
                    train,
                    validation,
                ),
        }


    return cache


# ============================================================
# 23. FAMILY CHECKPOINT PATHS
# ============================================================

def family_paths(
    stage_name,
    model_name,
):

    prefix = (
        f"STEP23_GEOGRAPHIC_"
        f"{safe_name(TARGET_DIVISION)}_"
        f"{safe_name(stage_name)}_"
        f"{safe_name(model_name)}"
    )


    return {
        "metrics":
            (
                DIRS["geographic"]
                /
                f"{prefix}_candidate_metrics.csv"
            ),

        "oof":
            (
                DIRS["geographic"]
                /
                f"{prefix}_candidate_OOF_scores.npz"
            ),

        "manifest":
            (
                DIRS["logs"]
                /
                f"{prefix}_family_checkpoint.json"
            ),
    }


# ============================================================
# 24. FAMILY CHECKPOINT VALIDITY
# ============================================================

def family_checkpoint_valid(
    stage_name,
    model_name,
    stage_development,
):

    paths = family_paths(
        stage_name,
        model_name,
    )


    if not all(
        path.exists()
        for path in paths.values()
    ):

        return False


    try:

        manifest = read_json(
            paths[
                "manifest"
            ]
        )


        if (
            manifest.get(
                "status"
            )
            !=
            "STEP23_FAMILY_COMPLETE"
        ):

            return False


        if (
            manifest.get(
                "execution_schema_version"
            )
            !=
            EXECUTION_SCHEMA_VERSION
        ):

            return False


        if (
            manifest.get(
                "heldout_division"
            )
            !=
            TARGET_DIVISION
        ):

            return False


        if (
            manifest.get(
                "stage"
            )
            !=
            stage_name
        ):

            return False


        if (
            manifest.get(
                "model"
            )
            !=
            model_name
        ):

            return False


        if (
            manifest.get(
                "stage_row_key_sha256"
            )
            !=
            row_key_hash(
                stage_development
            )
        ):

            return False


        if (
            manifest.get(
                "metrics_sha256"
            )
            !=
            sha256_file(
                paths[
                    "metrics"
                ]
            )
        ):

            return False


        if (
            manifest.get(
                "oof_sha256"
            )
            !=
            sha256_file(
                paths[
                    "oof"
                ]
            )
        ):

            return False


    except Exception:

        return False


    return True


# ============================================================
# 25. RUN / LOAD ONE MODEL FAMILY
# ============================================================

def evaluate_family(
    stage_name,
    stage_development,
    cv_cache,
    model_name,
):

    paths = family_paths(
        stage_name,
        model_name,
    )


    # --------------------------------------------------------
    # LOAD VALID CHECKPOINT
    # --------------------------------------------------------

    if family_checkpoint_valid(
        stage_name,
        model_name,
        stage_development,
    ):

        metrics = pd.read_csv(
            paths[
                "metrics"
            ]
        )


        loaded = np.load(
            paths[
                "oof"
            ],
            allow_pickle=False,
        )


        ids = (
            loaded[
                "candidate_ids"
            ]
            .astype(str)
            .tolist()
        )


        score_matrix = (
            loaded[
                "scores"
            ]
        )


        if score_matrix.shape != (
            12,
            len(
                stage_development
            ),
        ):

            raise RuntimeError(
                f"Invalid OOF checkpoint shape: "
                f"{stage_name} / {model_name}"
            )


        score_lookup = {
            candidate_id:
                score_matrix[
                    index
                ].copy()

            for index, candidate_id
            in enumerate(
                ids
            )
        }


        print(
            f"    {model_name}: "
            f"checkpoint loaded"
        )


        return (
            metrics,
            score_lookup,
        )


    # --------------------------------------------------------
    # FRESH FAMILY RUN
    # --------------------------------------------------------

    family_candidates = (
        candidate_registry.loc[
            candidate_registry[
                "Model"
            ]
            ==
            model_name
        ]
        .copy()
        .sort_values(
            "Deterministic_Tie_Rank"
        )
        .reset_index(
            drop=True
        )
    )


    if len(
        family_candidates
    ) != 12:

        raise RuntimeError(
            f"{model_name}: expected exactly 12 candidates."
        )


    n = len(
        stage_development
    )


    y_all = (
        stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        )
    )


    weight_all = (
        stage_development[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        )
    )


    rows = []


    score_lookup = {}


    for _, candidate_row in (
        family_candidates.iterrows()
    ):

        candidate_id = str(
            candidate_row[
                "Candidate_ID"
            ]
        )


        oof_score = np.full(
            n,
            np.nan,
            dtype=float,
        )


        try:

            for fold in [
                0,
                1,
                2,
                3,
                4,
            ]:

                fold_info = (
                    cv_cache[
                        fold
                    ]
                )


                X_train, X_validation = (
                    get_representation(
                        representations=fold_info[
                            "representations"
                        ],
                        model_name=model_name,
                        preprocessing_name=candidate_row[
                            "Preprocessing"
                        ],
                    )
                )


                estimator = (
                    instantiate_candidate(
                        candidate_row
                    )
                )


                estimator = fit_candidate(
                    estimator=estimator,
                    model_name=model_name,
                    X_train=X_train,
                    y_train=fold_info[
                        "y_train"
                    ],
                    training_weight=fold_info[
                        "training_weight"
                    ],
                )


                score = positive_score(
                    estimator=estimator,
                    model_name=model_name,
                    X=X_validation,
                )


                oof_score[
                    fold_info[
                        "validation_positions"
                    ]
                ] = score


            if not np.isfinite(
                oof_score
            ).all():

                raise RuntimeError(
                    "Incomplete OOF prediction."
                )


            weighted_ap = float(
                average_precision_score(
                    y_all,
                    oof_score,
                    sample_weight=weight_all,
                )
            )


            weighted_auc = float(
                roc_auc_score(
                    y_all,
                    oof_score,
                    sample_weight=weight_all,
                )
            )


            status = "SUCCESS"


            error_message = None


            score_lookup[
                candidate_id
            ] = oof_score


        except Exception as exc:

            weighted_ap = np.nan


            weighted_auc = np.nan


            status = "FAILED"


            error_message = (
                f"{type(exc).__name__}: "
                f"{exc}"
            )


        rows.append(
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Deterministic_Tie_Rank":
                    int(
                        candidate_row[
                            "Deterministic_Tie_Rank"
                        ]
                    ),

                "Development_CV_Weighted_AP":
                    weighted_ap,

                "Development_CV_Weighted_ROC_AUC":
                    weighted_auc,

                "Status":
                    status,

                "Error":
                    error_message,
            }
        )


    metrics = pd.DataFrame(
        rows
    )


    failed = metrics.loc[
        metrics[
            "Status"
        ]
        !=
        "SUCCESS"
    ]


    if len(
        failed
    ) > 0:

        raise RuntimeError(
            "\nMODEL FAMILY FAILURE\n"
            f"Division: {TARGET_DIVISION}\n"
            f"Stage: {stage_name}\n"
            f"Model: {model_name}\n\n"
            f"{failed[['Candidate_ID','Error']].to_string(index=False)}"
        )


    if len(
        score_lookup
    ) != 12:

        raise RuntimeError(
            f"{model_name}: expected 12 successful OOF vectors."
        )


    # --------------------------------------------------------
    # SAVE FAMILY CHECKPOINT
    # --------------------------------------------------------

    ordered_ids = (
        family_candidates[
            "Candidate_ID"
        ]
        .astype(str)
        .tolist()
    )


    score_matrix = np.vstack(
        [
            score_lookup[
                candidate_id
            ]

            for candidate_id
            in ordered_ids
        ]
    )


    metrics.to_csv(
        paths[
            "metrics"
        ],
        index=False,
    )


    np.savez_compressed(
        paths[
            "oof"
        ],
        candidate_ids=np.asarray(
            ordered_ids,
            dtype="U64",
        ),
        scores=score_matrix,
    )


    family_manifest = {
        "status":
            "STEP23_FAMILY_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "execution_time":
            dt.datetime.now().isoformat(
                timespec="seconds"
            ),

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "model":
            model_name,

        "stage_n":
            int(
                len(
                    stage_development
                )
            ),

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),

        "primary_master_sha256":
            EXPECTED_PRIMARY_SHA256,

        "candidate_registry_sha256":
            EXPECTED_CANDIDATE_SHA256,

        "execution_threads":
            EXECUTION_THREADS,

        "scientific_hyperparameters_changed":
            False,

        "metrics_sha256":
            sha256_file(
                paths[
                    "metrics"
                ]
            ),

        "oof_sha256":
            sha256_file(
                paths[
                    "oof"
                ]
            ),
    }


    with open(
        paths[
            "manifest"
        ],
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            family_manifest,
            handle,
            indent=2,
            ensure_ascii=False,
        )


    print(
        f"    {model_name}: complete"
        f" | best development-CV AP="
        f"{metrics['Development_CV_Weighted_AP'].max():.6f}"
    )


    return (
        metrics,
        score_lookup,
    )


# ============================================================
# 26. CALIBRATION PREDICTOR
# ============================================================

def calibration_predictor(
    raw_score,
    model_name,
):

    raw_score = np.asarray(
        raw_score,
        dtype=float,
    )


    if model_name == "RBF SVM":

        return raw_score


    p = np.clip(
        raw_score,
        CALIBRATION_PROBABILITY_CLIP,
        1.0
        -
        CALIBRATION_PROBABILITY_CLIP,
    )


    return np.log(
        p
        /
        (
            1.0
            -
            p
        )
    )


# ============================================================
# 27. FIT WEIGHTED SIGMOID CALIBRATOR
# ============================================================

def fit_calibrator(
    raw_score,
    y,
    weight,
    model_name,
):

    y = np.asarray(
        y,
        dtype=float,
    )


    weight = np.asarray(
        weight,
        dtype=float,
    )


    normalized_weight = (
        weight
        /
        np.mean(
            weight
        )
    )


    z = calibration_predictor(
        raw_score,
        model_name,
    )


    def objective(theta):

        alpha = theta[
            0
        ]


        beta = theta[
            1
        ]


        eta = (
            alpha
            +
            beta * z
        )


        denominator = np.sum(
            normalized_weight
        )


        loss = (
            np.sum(
                normalized_weight
                *
                (
                    np.logaddexp(
                        0.0,
                        eta,
                    )
                    -
                    y * eta
                )
            )
            /
            denominator
        )


        residual = (
            expit(
                eta
            )
            -
            y
        )


        gradient = np.array(
            [
                np.sum(
                    normalized_weight
                    *
                    residual
                )
                /
                denominator,

                np.sum(
                    normalized_weight
                    *
                    residual
                    *
                    z
                )
                /
                denominator,
            ],
            dtype=float,
        )


        return (
            float(
                loss
            ),
            gradient,
        )


    fit = minimize(
        objective,
        x0=np.array(
            [
                0.0,
                1.0,
            ],
            dtype=float,
        ),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter":
                5000,

            "ftol":
                1e-12,

            "gtol":
                1e-8,
        },
    )


    if not fit.success:

        raise RuntimeError(
            "Calibration optimization failed:\n"
            f"{fit.message}"
        )


    alpha = float(
        fit.x[
            0
        ]
    )


    beta = float(
        fit.x[
            1
        ]
    )


    calibrated = expit(
        alpha
        +
        beta * z
    )


    if not np.isfinite(
        calibrated
    ).all():

        raise RuntimeError(
            "Non-finite development calibrated probability."
        )


    return {
        "alpha":
            alpha,

        "beta":
            beta,

        "iterations":
            int(
                fit.nit
            ),

        "calibrated":
            calibrated,
    }


def apply_calibrator(
    raw_score,
    model_name,
    alpha,
    beta,
):

    z = calibration_predictor(
        raw_score,
        model_name,
    )


    probability = expit(
        alpha
        +
        beta * z
    )


    if not np.isfinite(
        probability
    ).all():

        raise RuntimeError(
            "Non-finite calibrated heldout probability."
        )


    return probability


# ============================================================
# 28. THRESHOLD OPTIMIZATION
# ============================================================

def threshold_candidates(
    probability,
):

    p = np.clip(
        np.asarray(
            probability,
            dtype=float,
        ),
        THRESHOLD_PROBABILITY_CLIP,
        1.0
        -
        THRESHOLD_PROBABILITY_CLIP,
    )


    unique_p = np.unique(
        p
    )


    if len(
        unique_p
    ) == 1:

        return np.array(
            [
                0.0,
                1.0,
            ]
        )


    midpoint = (
        unique_p[:-1]
        +
        (
            unique_p[1:]
            -
            unique_p[:-1]
        )
        /
        2.0
    )


    return np.unique(
        np.concatenate(
            [
                np.array(
                    [
                        0.0
                    ]
                ),

                midpoint,

                np.array(
                    [
                        1.0
                    ]
                ),
            ]
        )
    )


def threshold_metric(
    y,
    probability,
    weight,
    threshold,
):

    y = np.asarray(
        y,
        dtype=int,
    )


    probability = np.asarray(
        probability,
        dtype=float,
    )


    weight = np.asarray(
        weight,
        dtype=float,
    )


    prediction = (
        probability
        >=
        threshold
    ).astype(int)


    positive = (
        y
        ==
        1
    )


    negative = (
        y
        ==
        0
    )


    tp = weight[
        positive
        &
        (
            prediction
            ==
            1
        )
    ].sum()


    fn = weight[
        positive
        &
        (
            prediction
            ==
            0
        )
    ].sum()


    tn = weight[
        negative
        &
        (
            prediction
            ==
            0
        )
    ].sum()


    fp = weight[
        negative
        &
        (
            prediction
            ==
            1
        )
    ].sum()


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
    )


    balanced_accuracy = (
        sensitivity
        +
        specificity
    ) / 2.0


    return {
        "threshold":
            float(
                threshold
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "balance_floor":
            float(
                min(
                    sensitivity,
                    specificity,
                )
            ),
    }


def optimize_threshold(
    y,
    probability,
    weight,
):

    rows = [
        threshold_metric(
            y=y,
            probability=probability,
            weight=weight,
            threshold=threshold,
        )

        for threshold
        in threshold_candidates(
            probability
        )
    ]


    result = pd.DataFrame(
        rows
    )


    best_ba = float(
        result[
            "balanced_accuracy"
        ].max()
    )


    tied = result.loc[
        np.isclose(
            result[
                "balanced_accuracy"
            ],
            best_ba,
            rtol=0.0,
            atol=1e-12,
        )
    ].copy()


    best_floor = float(
        tied[
            "balance_floor"
        ].max()
    )


    tied = tied.loc[
        np.isclose(
            tied[
                "balance_floor"
            ],
            best_floor,
            rtol=0.0,
            atol=1e-12,
        )
    ].copy()


    selected = (
        tied
        .sort_values(
            "threshold"
        )
        .iloc[
            0
        ]
    )


    return {
        key:
            float(
                selected[
                    key
                ]
            )

        for key in [
            "threshold",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
            "balance_floor",
        ]
    }


# ============================================================
# 29. FINAL WINNER REFIT
# ============================================================

def refit_winner(
    stage_development,
    candidate_row,
):

    model_name = str(
        candidate_row[
            "Model"
        ]
    )


    representations = (
        build_representations(
            stage_development,
            heldout,
        )
    )


    X_train, X_heldout = (
        get_representation(
            representations=representations,
            model_name=model_name,
            preprocessing_name=candidate_row[
                "Preprocessing"
            ],
        )
    )


    estimator = (
        instantiate_candidate(
            candidate_row
        )
    )


    estimator = fit_candidate(
        estimator=estimator,
        model_name=model_name,
        X_train=X_train,
        y_train=stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        ),
        training_weight=normalized_training_weight(
            stage_development
        ),
    )


    score = positive_score(
        estimator=estimator,
        model_name=model_name,
        X=X_heldout,
    )


    return (
        score,
        representations[
            "__UNSEEN_COUNTS"
        ],
    )


# ============================================================
# 30. STAGE CHECKPOINT PATHS
# ============================================================

def stage_paths(
    stage_name,
):

    prefix = (
        f"STEP23_GEOGRAPHIC_"
        f"{safe_name(TARGET_DIVISION)}_"
        f"{safe_name(stage_name)}"
    )


    return {
        "candidate_metrics":
            (
                DIRS["geographic"]
                /
                f"{prefix}_all_candidate_metrics.csv"
            ),

        "winner":
            (
                DIRS["geographic"]
                /
                f"{prefix}_winner.csv"
            ),

        "calibration":
            (
                DIRS["geographic"]
                /
                f"{prefix}_calibration_threshold.csv"
            ),

        "development_oof":
            (
                DIRS["geographic"]
                /
                f"{prefix}_winner_development_OOF_RESTRICTED.csv"
            ),

        "heldout_stage":
            (
                DIRS["geographic"]
                /
                f"{prefix}_heldout_stage_predictions_RESTRICTED.csv"
            ),

        "unseen":
            (
                DIRS["qc"]
                /
                f"{prefix}_unseen_category_qc.csv"
            ),

        "manifest":
            (
                DIRS["logs"]
                /
                f"{prefix}_stage_checkpoint.json"
            ),
    }


# ============================================================
# 31. STAGE CHECKPOINT VALIDITY
# ============================================================

def stage_checkpoint_valid(
    stage_name,
    stage_development,
):

    paths = stage_paths(
        stage_name
    )


    if not all(
        path.exists()
        for path in paths.values()
    ):

        return False


    try:

        manifest = read_json(
            paths[
                "manifest"
            ]
        )


        if (
            manifest.get(
                "status"
            )
            !=
            "STEP23_STAGE_COMPLETE"
        ):

            return False


        if (
            manifest.get(
                "execution_schema_version"
            )
            !=
            EXECUTION_SCHEMA_VERSION
        ):

            return False


        if (
            manifest.get(
                "heldout_division"
            )
            !=
            TARGET_DIVISION
        ):

            return False


        if (
            manifest.get(
                "stage"
            )
            !=
            stage_name
        ):

            return False


        if (
            manifest.get(
                "stage_row_key_sha256"
            )
            !=
            row_key_hash(
                stage_development
            )
        ):

            return False


        hash_map = {
            "candidate_metrics_sha256":
                "candidate_metrics",

            "winner_sha256":
                "winner",

            "calibration_sha256":
                "calibration",

            "development_oof_sha256":
                "development_oof",

            "heldout_stage_sha256":
                "heldout_stage",

            "unseen_sha256":
                "unseen",
        }


        for hash_key, path_key in (
            hash_map.items()
        ):

            if (
                manifest.get(
                    hash_key
                )
                !=
                sha256_file(
                    paths[
                        path_key
                    ]
                )
            ):

                return False


    except Exception:

        return False


    return True


# ============================================================
# 32. EXECUTE ONE STAGE
# ============================================================

def execute_stage(
    stage_name,
):

    stage_development = (
        prepare_stage(
            development,
            stage_name,
        )
    )


    paths = stage_paths(
        stage_name
    )


    # --------------------------------------------------------
    # LOAD COMPLETED STAGE CHECKPOINT
    # --------------------------------------------------------

    if stage_checkpoint_valid(
        stage_name,
        stage_development,
    ):

        print(
            f"\n{stage_name}: "
            f"valid completed checkpoint loaded."
        )


        return {
            "candidate_metrics":
                pd.read_csv(
                    paths[
                        "candidate_metrics"
                    ]
                ),

            "winner":
                pd.read_csv(
                    paths[
                        "winner"
                    ]
                ),

            "calibration":
                pd.read_csv(
                    paths[
                        "calibration"
                    ]
                ),

            "development_oof":
                pd.read_csv(
                    paths[
                        "development_oof"
                    ]
                ),

            "heldout_stage":
                pd.read_csv(
                    paths[
                        "heldout_stage"
                    ]
                ),

            "unseen":
                pd.read_csv(
                    paths[
                        "unseen"
                    ]
                ),
        }


    print(
        "\n"
        + "-" * 78
    )


    print(
        f"{TARGET_DIVISION} | "
        f"{stage_name}"
    )


    print(
        f"Development N = "
        f"{len(stage_development)}"
    )


    print(
        f"Heldout scoring N = "
        f"{len(heldout)}"
    )


    print(
        "-" * 78
    )


    # --------------------------------------------------------
    # BUILD FOLD-SPECIFIC PREPROCESSING CACHE
    # --------------------------------------------------------

    cv_cache = build_cv_cache(
        stage_development
    )


    family_metric_frames = []


    all_oof_scores = {}


    # --------------------------------------------------------
    # ALL 10 FAMILIES
    # --------------------------------------------------------

    for family_number, model_name in enumerate(
        MODEL_ORDER,
        start=1,
    ):

        print(
            f"  [{family_number}/10] "
            f"{model_name}"
        )


        family_metrics, family_scores = (
            evaluate_family(
                stage_name=stage_name,
                stage_development=stage_development,
                cv_cache=cv_cache,
                model_name=model_name,
            )
        )


        family_metric_frames.append(
            family_metrics
        )


        all_oof_scores.update(
            family_scores
        )


    candidate_metrics = pd.concat(
        family_metric_frames,
        ignore_index=True,
    )


    if len(
        candidate_metrics
    ) != 120:

        raise RuntimeError(
            f"{stage_name}: expected 120 candidate rows."
        )


    if (
        candidate_metrics[
            "Status"
        ]
        !=
        "SUCCESS"
    ).any():

        raise RuntimeError(
            f"{stage_name}: at least one candidate failed."
        )


    if len(
        all_oof_scores
    ) != 120:

        raise RuntimeError(
            f"{stage_name}: expected 120 OOF score vectors."
        )


    # ========================================================
    # 33. JOINT WINNER
    # ========================================================

    best_ap = float(
        candidate_metrics[
            "Development_CV_Weighted_AP"
        ].max()
    )


    tied_ap = candidate_metrics.loc[
        np.isclose(
            candidate_metrics[
                "Development_CV_Weighted_AP"
            ],
            best_ap,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    best_auc = float(
        tied_ap[
            "Development_CV_Weighted_ROC_AUC"
        ].max()
    )


    tied_auc = tied_ap.loc[
        np.isclose(
            tied_ap[
                "Development_CV_Weighted_ROC_AUC"
            ],
            best_auc,
            rtol=0.0,
            atol=METRIC_TIE_TOLERANCE,
        )
    ].copy()


    winner = (
        tied_auc
        .sort_values(
            "Deterministic_Tie_Rank"
        )
        .iloc[
            0
        ]
    )


    candidate_id = str(
        winner[
            "Candidate_ID"
        ]
    )


    model_name = str(
        winner[
            "Model"
        ]
    )


    registry_match = (
        candidate_registry.loc[
            candidate_registry[
                "Candidate_ID"
            ]
            .astype(str)
            ==
            candidate_id
        ]
    )


    if len(
        registry_match
    ) != 1:

        raise RuntimeError(
            "Could not uniquely identify selected candidate."
        )


    candidate_row = (
        registry_match.iloc[
            0
        ]
    )


    winner_oof = (
        all_oof_scores[
            candidate_id
        ]
    )


    print(
        f"\n  JOINT WINNER: "
        f"{model_name} | "
        f"{candidate_id}"
    )


    print(
        f"  Development-CV AP  = "
        f"{float(winner['Development_CV_Weighted_AP']):.6f}"
    )


    print(
        f"  Development-CV AUC = "
        f"{float(winner['Development_CV_Weighted_ROC_AUC']):.6f}"
    )


    # ========================================================
    # 34. DEVELOPMENT-OOF CALIBRATION
    # ========================================================

    calibration = fit_calibrator(
        raw_score=winner_oof,
        y=stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        ),
        weight=stage_development[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        ),
        model_name=model_name,
    )


    development_calibrated = (
        calibration[
            "calibrated"
        ]
    )


    # ========================================================
    # 35. DEVELOPMENT-OOF THRESHOLD
    # ========================================================

    threshold_result = optimize_threshold(
        y=stage_development[
            "__TARGET"
        ]
        .to_numpy(
            dtype=int
        ),
        probability=development_calibrated,
        weight=stage_development[
            "SURVEY_WEIGHT"
        ]
        .to_numpy(
            dtype=float
        ),
    )


    threshold = float(
        threshold_result[
            "threshold"
        ]
    )


    print(
        f"  Optimized threshold = "
        f"{threshold:.6f}"
    )


    # ========================================================
    # 36. FULL DEVELOPMENT REFIT + SYLHET SCORE
    # ========================================================

    (
        heldout_raw_score,
        unseen_counts,
    ) = refit_winner(
        stage_development=stage_development,
        candidate_row=candidate_row,
    )


    heldout_probability = (
        apply_calibrator(
            raw_score=heldout_raw_score,
            model_name=model_name,
            alpha=calibration[
                "alpha"
            ],
            beta=calibration[
                "beta"
            ],
        )
    )


    # ========================================================
    # 37. BUILD STAGE OUTPUTS
    # ========================================================

    winner_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Preprocessing":
                    candidate_row[
                        "Preprocessing"
                    ],

                "Development_CV_Weighted_AP":
                    float(
                        winner[
                            "Development_CV_Weighted_AP"
                        ]
                    ),

                "Development_CV_Weighted_ROC_AUC":
                    float(
                        winner[
                            "Development_CV_Weighted_ROC_AUC"
                        ]
                    ),

                "Execution_Threads":
                    EXECUTION_THREADS,
            }
        ]
    )


    calibration_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Candidate_ID":
                    candidate_id,

                "Calibration_Alpha":
                    float(
                        calibration[
                            "alpha"
                        ]
                    ),

                "Calibration_Beta":
                    float(
                        calibration[
                            "beta"
                        ]
                    ),

                "Calibration_Iterations":
                    int(
                        calibration[
                            "iterations"
                        ]
                    ),

                "Optimized_Threshold":
                    threshold,

                "Development_Threshold_Balanced_Accuracy":
                    float(
                        threshold_result[
                            "balanced_accuracy"
                        ]
                    ),

                "Development_Threshold_Sensitivity":
                    float(
                        threshold_result[
                            "sensitivity"
                        ]
                    ),

                "Development_Threshold_Specificity":
                    float(
                        threshold_result[
                            "specificity"
                        ]
                    ),
            }
        ]
    )


    development_oof_df = pd.DataFrame(
        {
            "_ROW_KEY":
                stage_development[
                    "__ROW_KEY"
                ]
                .to_numpy(),

            "Heldout_Division":
                TARGET_DIVISION,

            "Development_CV_Fold":
                stage_development[
                    "__DEVELOPMENT_CV_FOLD"
                ]
                .to_numpy(
                    dtype=int
                ),

            "Stage":
                stage_name,

            "Model":
                model_name,

            "Candidate_ID":
                candidate_id,

            "True_Target":
                stage_development[
                    "__TARGET"
                ]
                .to_numpy(
                    dtype=int
                ),

            "SURVEY_WEIGHT":
                stage_development[
                    "SURVEY_WEIGHT"
                ]
                .to_numpy(
                    dtype=float
                ),

            "Development_OOF_Raw_Score":
                winner_oof,

            "Development_OOF_Calibrated_Probability":
                development_calibrated,
        }
    )


    heldout_stage_df = pd.DataFrame(
        {
            "_ROW_KEY":
                heldout[
                    "__ROW_KEY"
                ]
                .to_numpy(),

            "Heldout_Division":
                TARGET_DIVISION,

            "Stage":
                stage_name,

            "Model":
                model_name,

            "Candidate_ID":
                candidate_id,

            "Heldout_Raw_Score":
                heldout_raw_score,

            "Heldout_Calibrated_Probability":
                heldout_probability,

            "Optimized_Threshold":
                threshold,
        }
    )


    unseen_df = pd.DataFrame(
        [
            {
                "Heldout_Division":
                    TARGET_DIVISION,

                "Stage":
                    stage_name,

                "Model":
                    model_name,

                "Predictor":
                    predictor,

                "Heldout_Unseen_Category_Rows":
                    int(
                        count
                    ),
            }

            for predictor, count
            in unseen_counts.items()
        ]
    )


    # ========================================================
    # 38. SAVE STAGE FILES
    # ========================================================

    candidate_metrics.to_csv(
        paths[
            "candidate_metrics"
        ],
        index=False,
    )


    winner_df.to_csv(
        paths[
            "winner"
        ],
        index=False,
    )


    calibration_df.to_csv(
        paths[
            "calibration"
        ],
        index=False,
    )


    development_oof_df.to_csv(
        paths[
            "development_oof"
        ],
        index=False,
    )


    heldout_stage_df.to_csv(
        paths[
            "heldout_stage"
        ],
        index=False,
    )


    unseen_df.to_csv(
        paths[
            "unseen"
        ],
        index=False,
    )


    stage_manifest = {
        "status":
            "STEP23_STAGE_COMPLETE",

        "execution_schema_version":
            EXECUTION_SCHEMA_VERSION,

        "execution_time":
            dt.datetime.now().isoformat(
                timespec="seconds"
            ),

        "heldout_division":
            TARGET_DIVISION,

        "stage":
            stage_name,

        "stage_n":
            int(
                len(
                    stage_development
                )
            ),

        "stage_row_key_sha256":
            row_key_hash(
                stage_development
            ),

        "primary_master_sha256":
            EXPECTED_PRIMARY_SHA256,

        "candidate_registry_sha256":
            EXPECTED_CANDIDATE_SHA256,

        "execution_threads":
            EXECUTION_THREADS,

        "scientific_hyperparameters_changed":
            False,

        "heldout_performance_calculated":
            False,

        "candidate_metrics_sha256":
            sha256_file(
                paths[
                    "candidate_metrics"
                ]
            ),

        "winner_sha256":
            sha256_file(
                paths[
                    "winner"
                ]
            ),

        "calibration_sha256":
            sha256_file(
                paths[
                    "calibration"
                ]
            ),

        "development_oof_sha256":
            sha256_file(
                paths[
                    "development_oof"
                ]
            ),

        "heldout_stage_sha256":
            sha256_file(
                paths[
                    "heldout_stage"
                ]
            ),

        "unseen_sha256":
            sha256_file(
                paths[
                    "unseen"
                ]
            ),
    }


    with open(
        paths[
            "manifest"
        ],
        "w",
        encoding="utf-8",
    ) as handle:

        json.dump(
            stage_manifest,
            handle,
            indent=2,
            ensure_ascii=False,
        )


    print(
        f"  {stage_name} CHECKPOINT SAVED."
    )


    return {
        "candidate_metrics":
            candidate_metrics,

        "winner":
            winner_df,

        "calibration":
            calibration_df,

        "development_oof":
            development_oof_df,

        "heldout_stage":
            heldout_stage_df,

        "unseen":
            unseen_df,
    }


# ============================================================
# 39. START REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — SYLHET GEOGRAPHIC IECV"
)


print(
    "=" * 78
)


print(
    f"Held-out division : "
    f"{TARGET_DIVISION}"
)


print(
    f"Development N     : "
    f"{len(development)}"
)


print(
    f"Stage-2 dev N     : "
    f"{stage2_development_n}"
)


print(
    f"Held-out N        : "
    f"{len(heldout)}"
)


print(
    f"CPU threads       : "
    f"{EXECUTION_THREADS}"
)


print(
    "Division predictor: EXCLUDED"
)


# ============================================================
# 40. RUN STAGE 1 + STAGE 2
# ============================================================

stage1_result = execute_stage(
    "Stage 1"
)


stage2_result = execute_stage(
    "Stage 2"
)


# ============================================================
# 41. LOAD STAGE HELDOUT PREDICTIONS
# ============================================================

stage1_heldout = (
    stage1_result[
        "heldout_stage"
    ]
    .copy()
)


stage2_heldout = (
    stage2_result[
        "heldout_stage"
    ]
    .copy()
)


if len(
    stage1_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-1 Sylhet prediction count mismatch."
    )


if len(
    stage2_heldout
) != EXPECTED_HELDOUT_N:

    raise RuntimeError(
        "Stage-2 Sylhet prediction count mismatch."
    )


# ============================================================
# 42. CORRECTED ROW-KEY MERGE
# ============================================================

heldout_base = (
    heldout[
        [
            "__ROW_KEY",
            "ANC_CLASS",
            "SURVEY_WEIGHT",
            "PSU",
            "STRATUM",
        ]
    ]
    .rename(
        columns={
            "__ROW_KEY":
                "_ROW_KEY",

            "ANC_CLASS":
                "ANC_CLASS_TRUE",
        }
    )
)


stage1_columns = (
    stage1_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage1_Model",

            "Candidate_ID":
                "Stage1_Candidate",

            "Heldout_Raw_Score":
                "Stage1_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q0_CALIBRATED",

            "Optimized_Threshold":
                "Stage1_Optimized_Threshold",
        }
    )
)


stage2_columns = (
    stage2_heldout[
        [
            "_ROW_KEY",
            "Model",
            "Candidate_ID",
            "Heldout_Raw_Score",
            "Heldout_Calibrated_Probability",
            "Optimized_Threshold",
        ]
    ]
    .rename(
        columns={
            "Model":
                "Stage2_Model",

            "Candidate_ID":
                "Stage2_Candidate",

            "Heldout_Raw_Score":
                "Stage2_Raw_Score",

            "Heldout_Calibrated_Probability":
                "q2_CALIBRATED_CONDITIONAL",

            "Optimized_Threshold":
                "Stage2_Optimized_Threshold",
        }
    )
)


locked = (
    heldout_base
    .merge(
        stage1_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
    .merge(
        stage2_columns,
        on="_ROW_KEY",
        how="left",
        validate="one_to_one",
    )
)


locked[
    "Heldout_Division"
] = TARGET_DIVISION


required_prediction_columns = [
    "Stage1_Raw_Score",
    "q0_CALIBRATED",
    "Stage1_Optimized_Threshold",
    "Stage2_Raw_Score",
    "q2_CALIBRATED_CONDITIONAL",
    "Stage2_Optimized_Threshold",
]


if (
    locked[
        required_prediction_columns
    ]
    .isna()
    .any()
    .any()
):

    raise RuntimeError(
        "Missing Sylhet prediction after stage merge."
    )


# ============================================================
# 43. HIERARCHICAL FUSION
# ============================================================

q0 = locked[
    "q0_CALIBRATED"
].to_numpy(
    dtype=float
)


q2 = locked[
    "q2_CALIBRATED_CONDITIONAL"
].to_numpy(
    dtype=float
)


t1 = locked[
    "Stage1_Optimized_Threshold"
].to_numpy(
    dtype=float
)


t2 = locked[
    "Stage2_Optimized_Threshold"
].to_numpy(
    dtype=float
)


locked[
    "P_NO_ANC"
] = q0


locked[
    "P_ANC_1_7"
] = (
    (1.0 - q0)
    *
    (1.0 - q2)
)


locked[
    "P_ANC_GE8"
] = (
    (1.0 - q0)
    *
    q2
)


locked[
    "P_SUM"
] = (
    locked[
        "P_NO_ANC"
    ]
    +
    locked[
        "P_ANC_1_7"
    ]
    +
    locked[
        "P_ANC_GE8"
    ]
)


locked[
    "PREDICTED_ANC_CLASS"
] = np.where(
    q0
    >=
    t1,
    0,
    np.where(
        q2
        >=
        t2,
        2,
        1,
    ),
).astype(int)


# ============================================================
# 44. COMBINE STAGE METADATA
# ============================================================

candidate_metrics_all = pd.concat(
    [
        stage1_result[
            "candidate_metrics"
        ],
        stage2_result[
            "candidate_metrics"
        ],
    ],
    ignore_index=True,
)


winners_all = pd.concat(
    [
        stage1_result[
            "winner"
        ],
        stage2_result[
            "winner"
        ],
    ],
    ignore_index=True,
)


calibration_all = pd.concat(
    [
        stage1_result[
            "calibration"
        ],
        stage2_result[
            "calibration"
        ],
    ],
    ignore_index=True,
)


development_oof_all = pd.concat(
    [
        stage1_result[
            "development_oof"
        ],
        stage2_result[
            "development_oof"
        ],
    ],
    ignore_index=True,
)


unseen_all = pd.concat(
    [
        stage1_result[
            "unseen"
        ],
        stage2_result[
            "unseen"
        ],
    ],
    ignore_index=True,
)


# ============================================================
# 45. CORRECTED FINAL QC
# ============================================================

probability_matrix = (
    locked[
        [
            "P_NO_ANC",
            "P_ANC_1_7",
            "P_ANC_GE8",
        ]
    ]
    .to_numpy(
        dtype=float
    )
)


probabilities_finite = bool(
    np.isfinite(
        probability_matrix
    ).all()
)


probabilities_sum_to_one = bool(
    np.allclose(
        locked[
            "P_SUM"
        ]
        .to_numpy(
            dtype=float
        ),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )
)


hard_qc = pd.DataFrame(
    [
        {
            "Check":
                "Primary master hash unchanged",

            "PASS":
                (
                    sha256_file(
                        PRIMARY_FILE
                    )
                    ==
                    EXPECTED_PRIMARY_SHA256
                ),
        },

        {
            "Check":
                "Step-13 split hash unchanged",

            "PASS":
                (
                    sha256_file(
                        OUTER_SPLIT_FILE
                    )
                    ==
                    EXPECTED_OUTER_SHA256
                ),
        },

        {
            "Check":
                "Candidate registry hash unchanged",

            "PASS":
                (
                    sha256_file(
                        CANDIDATE_REGISTRY_FILE
                    )
                    ==
                    EXPECTED_CANDIDATE_SHA256
                ),
        },

        {
            "Check":
                "Sylhet development N = 4475",

            "PASS":
                (
                    len(
                        development
                    )
                    ==
                    4475
                ),
        },

        {
            "Check":
                "Stage-2 development N = 4151",

            "PASS":
                (
                    stage2_development_n
                    ==
                    4151
                ),
        },

        {
            "Check":
                "Sylhet heldout N = 588",

            "PASS":
                (
                    len(
                        heldout
                    )
                    ==
                    588
                ),
        },

        {
            "Check":
                "Exactly 240 candidate summaries",

            "PASS":
                (
                    len(
                        candidate_metrics_all
                    )
                    ==
                    240
                ),
        },

        {
            "Check":
                "All 240 candidates successful",

            "PASS":
                bool(
                    (
                        candidate_metrics_all[
                            "Status"
                        ]
                        ==
                        "SUCCESS"
                    )
                    .all()
                ),
        },

        {
            "Check":
                "Exactly two stage winners",

            "PASS":
                (
                    len(
                        winners_all
                    )
                    ==
                    2
                ),
        },

        {
            "Check":
                "Both stages represented",

            "PASS":
                (
                    set(
                        winners_all[
                            "Stage"
                        ]
                    )
                    ==
                    {
                        "Stage 1",
                        "Stage 2",
                    }
                ),
        },

        {
            "Check":
                "Exactly 588 heldout predictions",

            "PASS":
                (
                    len(
                        locked
                    )
                    ==
                    588
                ),
        },

        {
            "Check":
                "Heldout rows unique",

            "PASS":
                (
                    locked[
                        "_ROW_KEY"
                    ]
                    .nunique()
                    ==
                    588
                ),
        },

        {
            "Check":
                "Prediction merge complete",

            "PASS":
                bool(
                    ~locked[
                        required_prediction_columns
                    ]
                    .isna()
                    .any()
                    .any()
                ),
        },

        {
            "Check":
                "Fused probabilities finite",

            "PASS":
                probabilities_finite,
        },

        {
            "Check":
                "Fused probabilities sum to one",

            "PASS":
                probabilities_sum_to_one,
        },

        {
            "Check":
                "Division predictor excluded",

            "PASS":
                (
                    "division"
                    not in
                    GEO_PREDICTORS
                ),
        },

        {
            "Check":
                "Execution threads = 2",

            "PASS":
                (
                    EXECUTION_THREADS
                    ==
                    2
                ),
        },

        {
            "Check":
                "Scientific hyperparameters unchanged",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for selection",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for calibration",

            "PASS":
                True,
        },

        {
            "Check":
                "Heldout outcome not used for threshold optimization",

            "PASS":
                True,
        },

        {
            "Check":
                "No heldout performance calculated",

            "PASS":
                True,
        },
    ]
)


if not bool(
    hard_qc[
        "PASS"
    ]
    .all()
):

    failed = (
        hard_qc.loc[
            ~hard_qc[
                "PASS"
            ]
        ]
    )


    raise RuntimeError(
        "SYLHET STEP-23 FINAL QC FAILED:\n"
        f"{failed.to_string(index=False)}"
    )


# ============================================================
# 46. FINAL SYLHET OUTPUT PATHS
# ============================================================

prefix = (
    "STEP23_GEOGRAPHIC_holdout_Sylhet"
)


FINAL_CANDIDATE_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_candidate_development_CV_metrics.csv"
)


FINAL_WINNER_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_joint_winners.csv"
)


FINAL_CALIBRATION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_calibration_threshold.csv"
)


FINAL_DEVELOPMENT_OOF_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_winner_development_OOF_RESTRICTED.csv"
)


FINAL_PREDICTION_FILE = (
    DIRS["geographic"]
    /
    f"{prefix}_heldout_predictions_RESTRICTED.csv"
)


FINAL_UNSEEN_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_unseen_category_qc.csv"
)


FINAL_QC_FILE = (
    DIRS["qc"]
    /
    f"{prefix}_hard_qc.csv"
)


FINAL_MANIFEST_FILE = (
    DIRS["logs"]
    /
    f"{prefix}_checkpoint_manifest.json"
)


# ============================================================
# 47. SAVE FINAL SYLHET LOCK
# ============================================================

candidate_metrics_all.to_csv(
    FINAL_CANDIDATE_FILE,
    index=False,
)


winners_all.to_csv(
    FINAL_WINNER_FILE,
    index=False,
)


calibration_all.to_csv(
    FINAL_CALIBRATION_FILE,
    index=False,
)


development_oof_all.to_csv(
    FINAL_DEVELOPMENT_OOF_FILE,
    index=False,
)


locked.to_csv(
    FINAL_PREDICTION_FILE,
    index=False,
)


unseen_all.to_csv(
    FINAL_UNSEEN_FILE,
    index=False,
)


hard_qc.to_csv(
    FINAL_QC_FILE,
    index=False,
)


# ============================================================
# 48. FINAL SYLHET MANIFEST
# ============================================================

stage1_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 1"
    ]
    .iloc[
        0
    ]
)


stage2_row = (
    winners_all.loc[
        winners_all[
            "Stage"
        ]
        ==
        "Stage 2"
    ]
    .iloc[
        0
    ]
)


final_manifest = {
    "status":
        "STEP23_GEOGRAPHIC_DIVISION_LOCKED",

    "execution_schema_version":
        EXECUTION_SCHEMA_VERSION,

    "execution_time":
        dt.datetime.now().isoformat(
            timespec="seconds"
        ),

    "heldout_division":
        TARGET_DIVISION,

    "development_n":
        int(
            len(
                development
            )
        ),

    "stage1_development_n":
        int(
            len(
                development
            )
        ),

    "stage2_development_n":
        int(
            stage2_development_n
        ),

    "heldout_n":
        int(
            len(
                heldout
            )
        ),

    "execution_threads":
        EXECUTION_THREADS,

    "scientific_hyperparameters_changed":
        False,

    "division_predictor_used":
        False,

    "candidate_count_per_stage":
        120,

    "total_candidate_summaries":
        240,

    "development_cv_folds":
        5,

    "stage1_model":
        str(
            stage1_row[
                "Model"
            ]
        ),

    "stage1_candidate":
        str(
            stage1_row[
                "Candidate_ID"
            ]
        ),

    "stage2_model":
        str(
            stage2_row[
                "Model"
            ]
        ),

    "stage2_candidate":
        str(
            stage2_row[
                "Candidate_ID"
            ]
        ),

    "heldout_outcome_used_for_tuning":
        False,

    "heldout_outcome_used_for_calibration":
        False,

    "heldout_outcome_used_for_threshold":
        False,

    "heldout_model_updating":
        False,

    "heldout_performance_calculated":
        False,

    "primary_master_sha256":
        EXPECTED_PRIMARY_SHA256,

    "step13_split_sha256":
        EXPECTED_OUTER_SHA256,

    "candidate_registry_sha256":
        EXPECTED_CANDIDATE_SHA256,

    "step22_protocol_sha256":
        STEP22_PROTOCOL_SHA256,

    "candidate_metrics_sha256":
        sha256_file(
            FINAL_CANDIDATE_FILE
        ),

    "winners_sha256":
        sha256_file(
            FINAL_WINNER_FILE
        ),

    "calibration_threshold_sha256":
        sha256_file(
            FINAL_CALIBRATION_FILE
        ),

    "winner_development_oof_sha256":
        sha256_file(
            FINAL_DEVELOPMENT_OOF_FILE
        ),

    "heldout_predictions_sha256":
        sha256_file(
            FINAL_PREDICTION_FILE
        ),

    "unseen_category_qc_sha256":
        sha256_file(
            FINAL_UNSEEN_FILE
        ),

    "hard_qc_sha256":
        sha256_file(
            FINAL_QC_FILE
        ),
}


with open(
    FINAL_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        final_manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# ============================================================
# 49. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 78
)


print(
    "STEP 23 — SYLHET IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    "\nJOINT WINNERS:"
)


display(
    winners_all[
        [
            "Heldout_Division",
            "Stage",
            "Model",
            "Candidate_ID",
            "Development_CV_Weighted_AP",
            "Development_CV_Weighted_ROC_AUC",
            "Execution_Threads",
        ]
    ]
)


print(
    "\nCALIBRATION + THRESHOLD:"
)


display(
    calibration_all
)


print(
    "\nUNSEEN CATEGORY QC:"
)


display(
    unseen_all
)


print(
    "\nFINAL HARD QC:"
)


display(
    hard_qc
)


print(
    "\nLOCKED SYLHET PREDICTIONS:"
)


print(
    f"Rows                 : "
    f"{len(locked)}"
)


print(
    f"Unique rows          : "
    f"{locked['_ROW_KEY'].nunique()}"
)


print(
    f"Probabilities finite : "
    f"{probabilities_finite}"
)


print(
    f"Probability sum      : "
    f"{probabilities_sum_to_one}"
)


print(
    "Heldout metrics      : NOT CALCULATED"
)


print(
    "\nINTEGRITY:"
)


print(
    f"Primary master       : "
    f"{sha256_file(PRIMARY_FILE)}"
)


print(
    f"Step-13 folds        : "
    f"{sha256_file(OUTER_SPLIT_FILE)}"
)


print(
    f"Candidate registry   : "
    f"{sha256_file(CANDIDATE_REGISTRY_FILE)}"
)


print(
    f"Locked Sylhet        : "
    f"{sha256_file(FINAL_PREDICTION_FILE)}"
)


print(
    "\n"
    + "=" * 78
)


print(
    "STATUS: STEP 23 SYLHET IECV PREDICTIONS LOCKED"
)


print(
    "=" * 78
)


print(
    "\nALL 8 DIVISION-SPECIFIC IECV RUNS SHOULD NOW BE COMPLETE."
)


print(
    "NEXT: combine all 8 locked heldout prediction files, "
    "verify exact N=5063 coverage, and create final Step-23 pooled lock."
)


STEP 23 — SYLHET GEOGRAPHIC IECV
Held-out division : Sylhet
Development N     : 4475
Stage-2 dev N     : 4151
Held-out N        : 588
CPU threads       : 2
Division predictor: EXCLUDED

------------------------------------------------------------------------------
Sylhet | Stage 1
Development N = 4475
Heldout scoring N = 588
------------------------------------------------------------------------------
  [1/10] ElasticNet logistic regression
    ElasticNet logistic regression: complete | best development-CV AP=0.220233
  [2/10] RBF SVM
    RBF SVM: complete | best development-CV AP=0.138541
  [3/10] Random Forest
    Random Forest: complete | best development-CV AP=0.189526
  [4/10] ExtraTrees
    ExtraTrees: complete | best development-CV AP=0.185543
  [5/10] AdaBoost
    AdaBoost: complete | best development-CV AP=0.204379
  [6/10] GradientBoosting
    GradientBoosting: complete | best development-CV AP=0.211422
  [7/10] HistGradientBoosting
    HistGradientBoosting: complete | best

,Heldout_Division,Stage,Model,Candidate_ID,Development_CV_Weighted_AP,Development_CV_Weighted_ROC_AUC,Execution_Threads
0,Sylhet,Stage 1,ElasticNet logistic regression,EN_009,0.220233,0.771699,2
1,Sylhet,Stage 2,ElasticNet logistic regression,EN_001,0.175030,0.729588,2



CALIBRATION + THRESHOLD:


,Heldout_Division,Stage,Model,Candidate_ID,Calibration_Alpha,Calibration_Beta,Calibration_Iterations,Optimized_Threshold,Development_Threshold_Balanced_Accuracy,Development_Threshold_Sensitivity,Development_Threshold_Specificity
0,Sylhet,Stage 1,ElasticNet logistic regression,EN_009,-0.233449,0.889934,6,0.075417,0.714836,0.763759,0.665912
1,Sylhet,Stage 2,ElasticNet logistic regression,EN_001,2.987576,2.116142,12,0.065238,0.684606,0.595100,0.774112



UNSEEN CATEGORY QC:


,Heldout_Division,Stage,Model,Predictor,Heldout_Unseen_Category_Rows
0,Sylhet,Stage 1,ElasticNet logistic regression,maternal_education,0
1,Sylhet,Stage 1,ElasticNet logistic regression,wealth_quintile,0
2,Sylhet,Stage 1,ElasticNet logistic regression,partner_education_primary_final,0
3,Sylhet,Stage 1,ElasticNet logistic regression,pregnancy_intention,0
4,Sylhet,Stage 1,ElasticNet logistic regression,healthcare_decision_primary_final,0
5,Sylhet,Stage 1,ElasticNet logistic regression,residence,0
6,Sylhet,Stage 1,ElasticNet logistic regression,weekly_media_exposure,0
7,Sylhet,Stage 1,ElasticNet logistic regression,working_status,0
8,Sylhet,Stage 1,ElasticNet logistic regression,religion_final,0
9,Sylhet,Stage 1,ElasticNet logistic regression,household_head_sex,0



FINAL HARD QC:


,Check,PASS
0,Primary master hash unchanged,True
1,Step-13 split hash unchanged,True
2,Candidate registry hash unchanged,True
3,Sylhet development N = 4475,True
4,Stage-2 development N = 4151,True
5,Sylhet heldout N = 588,True
6,Exactly 240 candidate summaries,True
7,All 240 candidates successful,True
8,Exactly two stage winners,True
9,Both stages represented,True



LOCKED SYLHET PREDICTIONS:
Rows                 : 588
Unique rows          : 588
Probabilities finite : True
Probability sum      : True
Heldout metrics      : NOT CALCULATED

INTEGRITY:
Primary master       : 0cf7e22a0f8d1f2c4554e96ac56ef6dec6170f9067596519b0dcd1978132283a
Step-13 folds        : 258dac5ff253a96895d89ddb80ac892b76ef965d043c04cf94972994a82ab372
Candidate registry   : f95dc93431eced694114fe8ed071a2878e2572fac53aa3b9fd9c6cb8df87c697
Locked Sylhet        : 656d819b32080b10cf197b51cf9523e57bc9f104a15104577b0fb3ba2e5285bd

STATUS: STEP 23 SYLHET IECV PREDICTIONS LOCKED

ALL 8 DIVISION-SPECIFIC IECV RUNS SHOULD NOW BE COMPLETE.
NEXT: combine all 8 locked heldout prediction files, verify exact N=5063 coverage, and create final Step-23 pooled lock.
